# Project 2: basic well logs and core measurements
Self-contained notebook: the supplied LAS and core table are embedded below, so the original files and Excel/LAS readers are not required. Dependencies: numpy, pandas, matplotlib.

Source files: HA_2_reader_compatible.las and HA2_core_data.xls, sheet conventional_core_analysis. Assignment analysis interval: 5400-6300 ft. These are exploratory plots, not a completed lithology or saturation interpretation.

## Reading the core table
- 34 primary horizontal samples; 23 vertical companion rows end in V. V rows are not extra porosity/saturation samples.
- kh and kv are horizontal and vertical permeability. Klink. columns are Klinkenberg-corrected values; Air columns are retained in the embedded raw table but not used in these plots.
- Porosity and permeability are supplied at 800 and 3500 psia confining pressure. Show both; do not assume either pressure is proven to equal in-situ effective stress.
- Use log_depth for plotting. The table already applies +4 ft for cores 1/4, +7 ft for cores 2/3, and +3 ft for core 5. No additional alignment is asserted.
- >9999 values remain censored lower limits, marked with arrows/triangles; they are not exact observations.
- Core So and Sw do not sum to 100% (sums: 56.1-98.6%). The file does not document the deficit. Do not renormalize or equate the deficit with in-situ gas saturation.

## Important log limits
- DT24 is labeled Differential TEMP. in the LAS, not sonic transit time. No sonic porosity is calculated.
- No formation-pressure measurements are supplied, so that example track cannot be recreated. The core saturation track is included instead.
- Display units follow LAS metadata, including PE: BARN. CNCF PU is shown as porosity units (percent), without dividing by 100 for this plot.
- Null -9999 values become NaN. Resistivity must be positive for log axes. No additional quality filtering is imposed: caliper and density correction should guide the next interpretation step. RTBR has nonpositive values and is not plotted.
- Density/neutron scales are independent, not matched for a gas-crossover test. Spectral and caliper overlay ranges are printed explicitly.
- The whole 5400-6300 ft interval is shown even where no cores exist. No interpolation between isolated core samples is used.

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, LogLocator

logs = pd.read_json(io.StringIO('{"columns":["DEPT","BRIT","CAL","CHT","CIPD","CNCF","DT24","GR","K","KTH","PE","RIPD","RIPM","RSFL","RTBR","RTHK","RTHU","RUK","RWAD","SP","TEN","TH","U","ZCOR","ZDEN"],"index":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,402,403,404,405,406,407,408,409,410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436,437,438,439,440,441,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,470,471,472,473,474,475,476,477,478,479,480,481,482,483,484,485,486,487,488,489,490,491,492,493,494,495,496,497,498,499,500,501,502,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,521,522,523,524,525,526,527,528,529,530,531,532,533,534,535,536,537,538,539,540,541,542,543,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,563,564,565,566,567,568,569,570,571,572,573,574,575,576,577,578,579,580,581,582,583,584,585,586,587,588,589,590,591,592,593,594,595,596,597,598,599,600,601,602,603,604,605,606,607,608,609,610,611,612,613,614,615,616,617,618,619,620,621,622,623,624,625,626,627,628,629,630,631,632,633,634,635,636,637,638,639,640,641,642,643,644,645,646,647,648,649,650,651,652,653,654,655,656,657,658,659,660,661,662,663,664,665,666,667,668,669,670,671,672,673,674,675,676,677,678,679,680,681,682,683,684,685,686,687,688,689,690,691,692,693,694,695,696,697,698,699,700,701,702,703,704,705,706,707,708,709,710,711,712,713,714,715,716,717,718,719,720,721,722,723,724,725,726,727,728,729,730,731,732,733,734,735,736,737,738,739,740,741,742,743,744,745,746,747,748,749,750,751,752,753,754,755,756,757,758,759,760,761,762,763,764,765,766,767,768,769,770,771,772,773,774,775,776,777,778,779,780,781,782,783,784,785,786,787,788,789,790,791,792,793,794,795,796,797,798,799,800,801,802,803,804,805,806,807,808,809,810,811,812,813,814,815,816,817,818,819,820,821,822,823,824,825,826,827,828,829,830,831,832,833,834,835,836,837,838,839,840,841,842,843,844,845,846,847,848,849,850,851,852,853,854,855,856,857,858,859,860,861,862,863,864,865,866,867,868,869,870,871,872,873,874,875,876,877,878,879,880,881,882,883,884,885,886,887,888,889,890,891,892,893,894,895,896,897,898,899,900,901,902,903,904,905,906,907,908,909,910,911,912,913,914,915,916,917,918,919,920,921,922,923,924,925,926,927,928,929,930,931,932,933,934,935,936,937,938,939,940,941,942,943,944,945,946,947,948,949,950,951,952,953,954,955,956,957,958,959,960,961,962,963,964,965,966,967,968,969,970,971,972,973,974,975,976,977,978,979,980,981,982,983,984,985,986,987,988,989,990,991,992,993,994,995,996,997,998,999,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026,1027,1028,1029,1030,1031,1032,1033,1034,1035,1036,1037,1038,1039,1040,1041,1042,1043,1044,1045,1046,1047,1048,1049,1050,1051,1052,1053,1054,1055,1056,1057,1058,1059,1060,1061,1062,1063,1064,1065,1066,1067,1068,1069,1070,1071,1072,1073,1074,1075,1076,1077,1078,1079,1080,1081,1082,1083,1084,1085,1086,1087,1088,1089,1090,1091,1092,1093,1094,1095,1096,1097,1098,1099,1100,1101,1102,1103,1104,1105,1106,1107,1108,1109,1110,1111,1112,1113,1114,1115,1116,1117,1118,1119,1120,1121,1122,1123,1124,1125,1126,1127,1128,1129,1130,1131,1132,1133,1134,1135,1136,1137,1138,1139,1140,1141,1142,1143,1144,1145,1146,1147,1148,1149,1150,1151,1152,1153,1154,1155,1156,1157,1158,1159,1160,1161,1162,1163,1164,1165,1166,1167,1168,1169,1170,1171,1172,1173,1174,1175,1176,1177,1178,1179,1180,1181,1182,1183,1184,1185,1186,1187,1188,1189,1190,1191,1192,1193,1194,1195,1196,1197,1198,1199,1200,1201,1202,1203,1204,1205,1206,1207,1208,1209,1210,1211,1212,1213,1214,1215,1216,1217,1218,1219,1220,1221,1222,1223,1224,1225,1226,1227,1228,1229,1230,1231,1232,1233,1234,1235,1236,1237,1238,1239,1240,1241,1242,1243,1244,1245,1246,1247,1248,1249,1250,1251,1252,1253,1254,1255,1256,1257,1258,1259,1260,1261,1262,1263,1264,1265,1266,1267,1268,1269,1270,1271,1272,1273,1274,1275,1276,1277,1278,1279,1280,1281,1282,1283,1284,1285,1286,1287,1288,1289,1290,1291,1292,1293,1294,1295,1296,1297,1298,1299,1300,1301,1302,1303,1304,1305,1306,1307,1308,1309,1310,1311,1312,1313,1314,1315,1316,1317,1318,1319,1320,1321,1322,1323,1324,1325,1326,1327,1328,1329,1330,1331,1332,1333,1334,1335,1336,1337,1338,1339,1340,1341,1342,1343,1344,1345,1346,1347,1348,1349,1350,1351,1352,1353,1354,1355,1356,1357,1358,1359,1360,1361,1362,1363,1364,1365,1366,1367,1368,1369,1370,1371,1372,1373,1374,1375,1376,1377,1378,1379,1380,1381,1382,1383,1384,1385,1386,1387,1388,1389,1390,1391,1392,1393,1394,1395,1396,1397,1398,1399,1400,1401,1402,1403,1404,1405,1406,1407,1408,1409,1410,1411,1412,1413,1414,1415,1416,1417,1418,1419,1420,1421,1422,1423,1424,1425,1426,1427,1428,1429,1430,1431,1432,1433,1434,1435,1436,1437,1438,1439,1440,1441,1442,1443,1444,1445,1446,1447,1448,1449,1450,1451,1452,1453,1454,1455,1456,1457,1458,1459,1460,1461,1462,1463,1464,1465,1466,1467,1468,1469,1470,1471,1472,1473,1474,1475,1476,1477,1478,1479,1480,1481,1482,1483,1484,1485,1486,1487,1488,1489,1490,1491,1492,1493,1494,1495,1496,1497,1498,1499,1500,1501,1502,1503,1504,1505,1506,1507,1508,1509,1510,1511,1512,1513,1514,1515,1516,1517,1518,1519,1520,1521,1522,1523,1524,1525,1526,1527,1528,1529,1530,1531,1532,1533,1534,1535,1536,1537,1538,1539,1540,1541,1542,1543,1544,1545,1546,1547,1548,1549,1550,1551,1552,1553,1554,1555,1556,1557,1558,1559,1560,1561,1562,1563,1564,1565,1566,1567,1568,1569,1570,1571,1572,1573,1574,1575,1576,1577,1578,1579,1580,1581,1582,1583,1584,1585,1586,1587,1588,1589,1590,1591,1592,1593,1594,1595,1596,1597,1598,1599,1600,1601,1602,1603,1604,1605,1606,1607,1608,1609,1610,1611,1612,1613,1614,1615,1616,1617,1618,1619,1620,1621,1622,1623,1624,1625,1626,1627,1628,1629,1630,1631,1632,1633,1634,1635,1636,1637,1638,1639,1640,1641,1642,1643,1644,1645,1646,1647,1648,1649,1650,1651,1652,1653,1654,1655,1656,1657,1658,1659,1660,1661,1662,1663,1664,1665,1666,1667,1668,1669,1670,1671,1672,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682,1683,1684,1685,1686,1687,1688,1689,1690,1691,1692,1693,1694,1695,1696,1697,1698,1699,1700,1701,1702,1703,1704,1705,1706,1707,1708,1709,1710,1711,1712,1713,1714,1715,1716,1717,1718,1719,1720,1721,1722,1723,1724,1725,1726,1727,1728,1729,1730,1731,1732,1733,1734,1735,1736,1737,1738,1739,1740,1741,1742,1743,1744,1745,1746,1747,1748,1749,1750,1751,1752,1753,1754,1755,1756,1757,1758,1759,1760,1761,1762,1763,1764,1765,1766,1767,1768,1769,1770,1771,1772,1773,1774,1775,1776,1777,1778,1779,1780,1781,1782,1783,1784,1785,1786,1787,1788,1789,1790,1791,1792,1793,1794,1795,1796,1797,1798,1799,1800,1801,1802,1803,1804,1805,1806,1807,1808,1809,1810,1811,1812,1813,1814,1815,1816,1817,1818,1819,1820,1821,1822,1823,1824,1825,1826,1827,1828,1829,1830,1831,1832,1833,1834,1835,1836,1837,1838,1839,1840,1841,1842,1843,1844,1845,1846,1847,1848,1849,1850,1851,1852,1853,1854,1855,1856,1857,1858,1859,1860,1861,1862,1863,1864,1865,1866,1867,1868,1869,1870,1871,1872,1873,1874,1875,1876,1877,1878,1879,1880,1881,1882,1883,1884,1885,1886,1887,1888,1889,1890,1891,1892,1893,1894,1895,1896,1897,1898,1899,1900,1901,1902,1903,1904,1905,1906,1907,1908,1909,1910,1911,1912,1913,1914,1915,1916,1917,1918,1919,1920,1921,1922,1923,1924,1925,1926,1927,1928,1929,1930,1931,1932,1933,1934,1935,1936,1937,1938,1939,1940,1941,1942,1943,1944,1945,1946,1947,1948,1949,1950,1951,1952,1953,1954,1955,1956,1957,1958,1959,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,1975,1976,1977,1978,1979,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057,2058,2059,2060,2061,2062,2063,2064,2065,2066,2067,2068,2069,2070,2071,2072,2073,2074,2075,2076,2077,2078,2079,2080,2081,2082,2083,2084,2085,2086,2087,2088,2089,2090,2091,2092,2093,2094,2095,2096,2097,2098,2099,2100,2101,2102,2103,2104,2105,2106,2107,2108,2109,2110,2111,2112,2113,2114,2115,2116,2117,2118,2119,2120,2121,2122,2123,2124,2125,2126,2127,2128,2129,2130,2131,2132,2133,2134,2135,2136,2137,2138,2139,2140,2141,2142,2143,2144,2145,2146,2147,2148,2149,2150,2151,2152,2153,2154,2155,2156,2157,2158,2159,2160,2161,2162,2163,2164,2165,2166,2167,2168,2169,2170,2171,2172,2173,2174,2175,2176,2177,2178,2179,2180,2181,2182,2183,2184,2185,2186,2187,2188,2189,2190,2191,2192,2193,2194,2195,2196,2197,2198,2199,2200,2201,2202,2203,2204,2205,2206,2207,2208,2209,2210,2211,2212,2213,2214,2215,2216,2217,2218,2219,2220,2221,2222,2223,2224,2225,2226,2227,2228,2229,2230,2231,2232,2233,2234,2235,2236,2237,2238,2239,2240,2241,2242,2243,2244,2245,2246,2247,2248,2249,2250,2251,2252,2253,2254,2255,2256,2257,2258,2259,2260,2261,2262,2263,2264,2265,2266,2267,2268,2269,2270,2271,2272,2273,2274,2275,2276,2277,2278,2279,2280,2281,2282,2283,2284,2285,2286,2287,2288,2289,2290,2291,2292,2293,2294,2295,2296,2297,2298,2299,2300,2301,2302,2303,2304,2305,2306,2307,2308,2309,2310,2311,2312,2313,2314,2315,2316,2317,2318,2319,2320,2321,2322,2323,2324,2325,2326,2327,2328,2329,2330,2331,2332,2333,2334,2335,2336,2337,2338,2339,2340,2341,2342,2343,2344,2345,2346,2347,2348,2349,2350,2351,2352,2353,2354,2355,2356,2357,2358,2359,2360,2361,2362,2363,2364,2365,2366,2367,2368,2369,2370,2371,2372,2373,2374,2375,2376,2377,2378,2379,2380,2381,2382,2383,2384,2385,2386,2387,2388,2389,2390,2391,2392,2393,2394,2395,2396,2397,2398,2399,2400,2401,2402,2403,2404,2405,2406,2407,2408,2409,2410,2411,2412,2413,2414,2415,2416,2417,2418,2419,2420,2421,2422,2423,2424,2425,2426,2427,2428,2429,2430,2431,2432,2433,2434,2435,2436,2437,2438,2439,2440,2441,2442,2443,2444,2445,2446,2447,2448,2449,2450,2451,2452,2453,2454,2455,2456,2457,2458,2459,2460,2461,2462,2463,2464,2465,2466,2467,2468,2469,2470,2471,2472,2473,2474,2475,2476,2477,2478,2479,2480,2481,2482,2483,2484,2485,2486,2487,2488,2489,2490,2491,2492,2493,2494,2495,2496,2497,2498,2499,2500,2501,2502,2503,2504,2505,2506,2507,2508,2509,2510,2511,2512,2513,2514,2515,2516,2517,2518,2519,2520,2521,2522,2523,2524,2525,2526,2527,2528,2529,2530,2531,2532,2533,2534,2535,2536,2537,2538,2539,2540,2541,2542,2543,2544,2545,2546,2547,2548,2549,2550,2551,2552,2553,2554,2555,2556,2557,2558,2559,2560,2561,2562,2563,2564,2565,2566,2567,2568,2569,2570,2571,2572,2573,2574,2575,2576,2577,2578,2579,2580,2581,2582,2583,2584,2585,2586,2587,2588,2589,2590,2591,2592,2593,2594,2595,2596,2597,2598,2599,2600,2601,2602,2603,2604,2605,2606,2607,2608,2609,2610,2611,2612,2613,2614,2615,2616,2617,2618,2619,2620,2621,2622,2623,2624,2625,2626,2627,2628,2629,2630,2631,2632,2633,2634,2635,2636,2637,2638,2639,2640,2641,2642,2643,2644,2645,2646,2647,2648,2649,2650,2651,2652,2653,2654,2655,2656,2657,2658,2659,2660,2661,2662,2663,2664,2665,2666,2667,2668,2669,2670,2671,2672,2673,2674,2675,2676,2677,2678,2679,2680,2681,2682,2683,2684,2685,2686,2687,2688,2689,2690,2691,2692,2693,2694,2695,2696,2697,2698,2699,2700,2701,2702,2703,2704,2705,2706,2707,2708,2709,2710,2711,2712,2713,2714,2715,2716,2717,2718,2719,2720,2721,2722,2723,2724,2725,2726,2727,2728,2729,2730,2731,2732,2733,2734,2735,2736,2737,2738,2739,2740,2741,2742,2743,2744,2745,2746,2747,2748,2749,2750,2751,2752,2753,2754,2755,2756,2757,2758,2759,2760,2761,2762,2763,2764,2765,2766,2767,2768,2769,2770,2771,2772,2773,2774,2775,2776,2777,2778,2779,2780,2781,2782,2783,2784,2785,2786,2787,2788,2789,2790,2791,2792,2793,2794,2795,2796,2797,2798,2799,2800,2801,2802,2803,2804,2805,2806,2807,2808,2809,2810,2811,2812,2813,2814,2815,2816,2817,2818,2819,2820,2821,2822,2823,2824,2825,2826,2827,2828,2829,2830,2831,2832,2833,2834,2835,2836,2837,2838,2839,2840,2841,2842,2843,2844,2845,2846,2847,2848,2849,2850,2851,2852,2853,2854,2855,2856,2857,2858,2859,2860,2861,2862,2863,2864,2865,2866,2867,2868,2869,2870,2871,2872,2873,2874,2875,2876,2877,2878,2879,2880,2881,2882,2883,2884,2885,2886,2887,2888,2889,2890,2891,2892,2893,2894,2895,2896,2897,2898,2899,2900,2901,2902,2903,2904,2905,2906,2907,2908,2909,2910,2911,2912,2913,2914,2915,2916,2917,2918,2919,2920,2921,2922,2923,2924,2925,2926,2927,2928,2929,2930,2931,2932,2933,2934,2935,2936,2937,2938,2939,2940,2941,2942,2943,2944,2945,2946,2947,2948,2949,2950,2951,2952,2953,2954,2955,2956,2957,2958,2959,2960,2961,2962,2963,2964,2965,2966,2967,2968,2969,2970,2971,2972,2973,2974,2975,2976,2977,2978,2979,2980,2981,2982,2983,2984,2985,2986,2987,2988,2989,2990,2991,2992,2993,2994,2995,2996,2997,2998,2999,3000,3001,3002,3003,3004,3005,3006,3007,3008,3009,3010,3011,3012,3013,3014,3015,3016,3017,3018,3019,3020,3021,3022,3023,3024,3025,3026,3027,3028,3029,3030,3031,3032,3033,3034,3035,3036,3037,3038,3039,3040,3041,3042,3043,3044,3045,3046,3047,3048,3049,3050,3051,3052,3053,3054,3055,3056,3057,3058,3059,3060,3061,3062,3063,3064,3065,3066,3067,3068,3069,3070,3071,3072,3073,3074,3075,3076,3077,3078,3079,3080,3081,3082,3083,3084,3085,3086,3087,3088,3089,3090,3091,3092,3093,3094,3095,3096,3097,3098,3099,3100,3101,3102,3103,3104,3105,3106,3107,3108,3109,3110,3111,3112,3113,3114,3115,3116,3117,3118,3119,3120,3121,3122,3123,3124,3125,3126,3127,3128,3129,3130,3131,3132,3133,3134,3135,3136,3137,3138,3139,3140,3141,3142,3143,3144,3145,3146,3147,3148,3149,3150,3151,3152,3153,3154,3155,3156,3157,3158,3159,3160,3161,3162,3163,3164,3165,3166,3167,3168,3169,3170,3171,3172,3173,3174,3175,3176,3177,3178,3179,3180,3181,3182,3183,3184,3185,3186,3187,3188,3189,3190,3191,3192,3193,3194,3195,3196,3197,3198,3199,3200,3201,3202,3203,3204,3205,3206,3207,3208,3209,3210,3211,3212,3213,3214,3215,3216,3217,3218,3219,3220,3221,3222,3223,3224,3225,3226,3227,3228,3229,3230,3231,3232,3233,3234,3235,3236,3237,3238,3239,3240,3241,3242,3243,3244,3245,3246,3247,3248,3249,3250,3251,3252,3253,3254,3255,3256,3257,3258,3259,3260,3261,3262,3263,3264,3265,3266,3267,3268,3269,3270,3271,3272,3273,3274,3275,3276,3277,3278,3279,3280,3281,3282,3283,3284,3285,3286,3287,3288,3289,3290,3291,3292,3293,3294,3295,3296,3297,3298,3299,3300,3301,3302,3303,3304,3305,3306,3307,3308,3309,3310,3311,3312,3313,3314,3315,3316,3317,3318,3319,3320,3321,3322,3323,3324,3325,3326,3327,3328,3329,3330,3331,3332,3333,3334,3335,3336,3337,3338,3339,3340,3341,3342,3343,3344,3345,3346,3347,3348,3349,3350,3351,3352,3353,3354,3355,3356,3357,3358,3359,3360,3361,3362,3363,3364,3365,3366,3367,3368,3369,3370,3371,3372,3373,3374,3375,3376,3377,3378,3379,3380,3381,3382,3383,3384,3385,3386,3387,3388,3389,3390,3391,3392,3393,3394,3395,3396,3397,3398,3399,3400,3401,3402,3403,3404,3405,3406,3407,3408,3409,3410,3411,3412,3413,3414,3415,3416,3417,3418,3419,3420,3421,3422,3423,3424,3425,3426,3427,3428,3429,3430,3431,3432,3433,3434,3435,3436,3437,3438,3439,3440,3441,3442,3443,3444,3445,3446,3447,3448,3449,3450,3451,3452,3453,3454,3455,3456,3457,3458,3459,3460,3461,3462,3463,3464,3465,3466,3467,3468,3469,3470,3471,3472,3473,3474,3475,3476,3477,3478,3479,3480,3481,3482,3483,3484,3485,3486,3487,3488,3489,3490,3491,3492,3493,3494,3495,3496,3497,3498,3499,3500,3501,3502,3503,3504,3505,3506,3507,3508,3509,3510,3511,3512,3513,3514,3515,3516,3517,3518,3519,3520,3521,3522,3523,3524,3525,3526,3527,3528,3529,3530,3531,3532,3533,3534,3535,3536,3537,3538,3539,3540,3541,3542,3543,3544,3545,3546,3547,3548,3549,3550,3551,3552,3553,3554,3555,3556,3557,3558,3559,3560,3561,3562,3563,3564,3565,3566,3567,3568,3569,3570,3571,3572,3573,3574,3575,3576,3577,3578,3579,3580,3581,3582,3583,3584,3585,3586,3587,3588,3589,3590,3591,3592,3593,3594,3595,3596,3597,3598,3599,3600,3601,3602,3603,3604,3605,3606,3607,3608,3609,3610,3611,3612,3613,3614,3615,3616,3617,3618,3619,3620,3621,3622,3623,3624,3625,3626,3627,3628,3629,3630,3631,3632,3633,3634,3635,3636,3637,3638,3639,3640,3641,3642,3643,3644,3645,3646,3647,3648,3649,3650,3651,3652,3653,3654,3655,3656,3657,3658,3659,3660,3661,3662,3663,3664,3665,3666,3667,3668,3669,3670,3671,3672,3673,3674,3675,3676,3677,3678,3679,3680,3681,3682,3683,3684,3685,3686,3687,3688,3689,3690,3691,3692,3693,3694,3695,3696,3697,3698,3699,3700,3701,3702,3703,3704,3705,3706,3707,3708,3709,3710,3711,3712,3713,3714,3715,3716,3717,3718,3719,3720,3721,3722,3723,3724,3725,3726,3727,3728,3729,3730,3731,3732,3733,3734,3735,3736,3737,3738,3739,3740,3741,3742,3743,3744,3745,3746,3747,3748,3749,3750,3751,3752,3753,3754,3755,3756,3757,3758,3759,3760,3761,3762,3763,3764,3765,3766,3767,3768,3769,3770,3771,3772,3773,3774,3775,3776,3777,3778,3779,3780,3781,3782,3783,3784,3785,3786,3787,3788,3789,3790,3791,3792,3793,3794,3795,3796,3797,3798,3799,3800,3801,3802,3803,3804,3805,3806,3807,3808,3809,3810,3811,3812,3813,3814,3815,3816,3817,3818,3819,3820,3821,3822,3823,3824,3825,3826,3827,3828,3829,3830,3831,3832,3833,3834,3835,3836,3837,3838,3839,3840,3841,3842,3843,3844,3845,3846,3847,3848,3849,3850,3851,3852,3853,3854,3855,3856,3857,3858,3859,3860,3861,3862,3863,3864,3865,3866,3867,3868,3869,3870,3871,3872,3873,3874,3875,3876,3877,3878,3879,3880,3881,3882,3883,3884,3885,3886,3887,3888,3889,3890,3891,3892,3893,3894,3895,3896,3897,3898,3899,3900,3901,3902,3903,3904,3905,3906,3907,3908,3909,3910,3911,3912,3913,3914,3915,3916,3917,3918,3919,3920,3921,3922,3923,3924,3925,3926,3927,3928,3929,3930,3931,3932,3933,3934,3935,3936,3937,3938,3939,3940,3941,3942,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952,3953,3954,3955,3956,3957,3958,3959,3960,3961,3962,3963,3964,3965,3966,3967,3968,3969,3970,3971,3972,3973,3974,3975,3976,3977,3978,3979,3980,3981,3982,3983,3984,3985,3986,3987,3988,3989,3990,3991,3992,3993,3994,3995,3996,3997,3998,3999,4000,4001,4002,4003,4004,4005,4006,4007,4008,4009,4010,4011,4012,4013,4014,4015,4016,4017,4018,4019,4020,4021,4022,4023,4024,4025,4026,4027,4028,4029,4030,4031,4032,4033,4034,4035,4036,4037,4038,4039,4040,4041,4042,4043,4044,4045,4046,4047,4048,4049,4050,4051,4052,4053,4054,4055,4056,4057,4058,4059,4060,4061,4062,4063,4064,4065,4066,4067,4068,4069,4070,4071,4072,4073,4074,4075,4076,4077,4078,4079,4080,4081,4082,4083,4084,4085,4086,4087,4088,4089,4090,4091,4092,4093,4094,4095,4096,4097,4098,4099,4100,4101,4102,4103,4104,4105,4106,4107,4108,4109,4110,4111,4112,4113,4114,4115,4116,4117,4118,4119,4120,4121,4122,4123,4124,4125,4126,4127,4128,4129,4130,4131,4132,4133,4134,4135,4136,4137,4138,4139,4140,4141,4142,4143,4144,4145,4146,4147,4148,4149,4150,4151,4152,4153,4154,4155,4156,4157,4158,4159,4160,4161,4162,4163,4164,4165,4166,4167,4168,4169,4170,4171,4172,4173,4174,4175,4176,4177,4178,4179,4180,4181,4182,4183,4184,4185,4186,4187,4188,4189,4190,4191,4192,4193,4194,4195,4196,4197,4198,4199,4200,4201,4202,4203,4204,4205,4206,4207,4208,4209,4210,4211,4212,4213,4214,4215,4216,4217,4218,4219,4220,4221,4222,4223,4224,4225,4226,4227,4228,4229,4230,4231,4232,4233,4234,4235,4236,4237,4238,4239,4240,4241,4242,4243,4244,4245,4246,4247,4248,4249,4250,4251,4252,4253,4254,4255,4256,4257,4258,4259,4260,4261,4262,4263,4264,4265,4266,4267,4268,4269,4270,4271,4272,4273,4274,4275,4276,4277,4278,4279,4280,4281,4282,4283,4284,4285,4286,4287,4288,4289,4290,4291,4292,4293,4294,4295,4296,4297,4298,4299,4300,4301,4302,4303,4304,4305,4306,4307,4308,4309,4310,4311,4312,4313,4314,4315,4316,4317,4318,4319,4320,4321,4322,4323,4324,4325,4326,4327,4328,4329,4330,4331,4332,4333,4334,4335,4336,4337,4338,4339,4340,4341,4342,4343,4344,4345,4346,4347,4348,4349,4350,4351,4352,4353,4354,4355,4356,4357,4358,4359,4360,4361,4362,4363,4364,4365,4366,4367,4368,4369,4370,4371,4372,4373,4374,4375,4376,4377,4378,4379,4380,4381,4382,4383,4384,4385,4386,4387,4388,4389,4390,4391,4392,4393,4394,4395,4396,4397,4398,4399,4400,4401,4402,4403,4404,4405,4406,4407,4408,4409,4410,4411,4412,4413,4414,4415,4416,4417,4418,4419,4420,4421,4422,4423,4424,4425,4426,4427,4428,4429,4430,4431,4432,4433,4434,4435,4436,4437,4438,4439,4440,4441,4442,4443,4444,4445,4446,4447,4448,4449,4450,4451,4452,4453,4454,4455,4456,4457,4458,4459,4460,4461,4462,4463,4464,4465,4466,4467,4468,4469,4470,4471,4472,4473,4474,4475,4476,4477,4478,4479,4480,4481,4482,4483,4484,4485,4486,4487,4488,4489,4490,4491,4492,4493,4494,4495,4496,4497,4498,4499,4500,4501,4502,4503,4504,4505,4506,4507,4508,4509,4510,4511,4512,4513,4514,4515,4516,4517,4518,4519,4520,4521,4522,4523,4524,4525,4526,4527,4528,4529,4530,4531,4532,4533,4534,4535,4536,4537,4538,4539,4540,4541,4542,4543,4544,4545,4546,4547,4548,4549,4550,4551,4552,4553,4554,4555,4556,4557,4558,4559,4560,4561,4562,4563,4564,4565,4566,4567,4568,4569,4570,4571,4572,4573,4574,4575,4576,4577,4578,4579,4580,4581,4582,4583,4584,4585,4586,4587,4588,4589,4590,4591,4592,4593,4594,4595,4596,4597,4598,4599,4600,4601,4602,4603,4604,4605,4606,4607,4608,4609,4610,4611,4612,4613,4614,4615,4616,4617,4618,4619,4620,4621,4622,4623,4624,4625,4626,4627,4628,4629,4630,4631,4632,4633,4634,4635,4636,4637,4638,4639,4640,4641,4642,4643,4644,4645,4646,4647,4648,4649,4650,4651,4652,4653,4654,4655,4656,4657,4658,4659,4660,4661,4662,4663,4664,4665,4666,4667,4668,4669,4670,4671,4672,4673,4674,4675,4676,4677,4678,4679,4680,4681,4682,4683,4684,4685,4686,4687,4688,4689,4690,4691,4692,4693,4694,4695,4696,4697,4698,4699,4700,4701,4702,4703,4704,4705,4706,4707,4708,4709,4710,4711,4712,4713,4714,4715,4716,4717,4718,4719,4720,4721,4722,4723,4724,4725,4726,4727,4728,4729,4730,4731,4732,4733,4734,4735,4736,4737,4738,4739,4740,4741,4742,4743,4744,4745,4746,4747,4748,4749,4750,4751,4752,4753,4754,4755,4756,4757,4758,4759,4760,4761,4762,4763,4764,4765,4766,4767,4768,4769,4770,4771,4772,4773,4774,4775,4776,4777,4778,4779,4780,4781,4782,4783,4784,4785,4786,4787],"data":[[3984.0,null,null,1963.898,944.759,null,143.34,101.276,0.0,0.0,null,1.058,0.985,1.226,null,0.0,0.0,0.0,null,-6.083,-9.712,0.0,0.0,null,null],[3984.5,null,null,1964.437,939.586,null,142.806,100.533,0.0,0.0,null,1.064,0.989,1.204,null,0.0,0.0,0.0,null,-6.182,-6.165,0.0,0.0,null,null],[3985.0,null,null,1966.241,936.66,null,141.86,99.838,0.0,0.0,null,1.068,0.996,1.184,null,0.0,0.0,0.0,null,-6.213,-1.915,0.0,0.0,null,null],[3985.5,null,null,1965.033,936.135,null,140.633,100.939,0.0,0.0,null,1.068,1.001,1.174,null,0.0,0.0,0.0,null,-6.164,-6.951,0.0,0.0,null,null],[3986.0,null,null,1961.27,938.968,null,139.494,102.609,0.0,0.0,null,1.065,1.0,1.179,null,0.0,0.0,0.0,null,-6.121,-15.43,0.0,0.0,null,null],[3986.5,null,null,1962.907,945.324,null,138.786,102.877,0.0,0.0,null,1.058,0.989,1.211,null,0.0,0.0,0.0,null,-6.134,-18.697,0.0,0.0,null,null],[3987.0,null,null,1971.434,953.016,null,138.542,102.522,0.0,0.0,null,1.049,0.975,1.258,null,0.0,0.0,0.0,null,-6.107,-19.109,0.0,0.0,null,null],[3987.5,null,null,1974.884,960.07,null,138.548,103.402,0.0,0.0,null,1.042,0.963,1.223,null,0.0,0.0,0.0,null,-5.947,-17.537,0.0,0.0,null,null],[3988.0,null,null,1969.016,966.386,null,138.657,105.576,0.0,0.0,null,1.035,0.956,1.139,null,0.0,0.0,0.0,null,-5.77,-15.818,0.0,0.0,null,null],[3988.5,null,null,1962.109,971.634,null,138.988,107.265,0.0,0.0,null,1.029,0.952,1.106,null,0.0,0.0,0.0,null,-5.735,-17.995,0.0,0.0,null,null],[3989.0,null,null,1958.977,975.073,null,139.675,107.589,0.0,0.0,null,1.026,0.95,1.134,null,0.0,0.0,0.0,null,-5.837,-18.211,0.0,0.0,null,null],[3989.5,null,null,1958.348,975.311,null,140.553,107.306,0.0,0.0,null,1.025,0.951,1.144,null,0.0,0.0,0.0,null,-5.957,-14.896,0.0,0.0,null,null],[3990.0,null,null,1963.207,970.952,null,141.248,107.954,0.0,0.0,null,1.03,0.958,1.13,null,0.0,0.0,0.0,null,-6.065,-14.907,0.0,0.0,null,null],[3990.5,null,null,1974.673,961.748,null,141.518,109.953,0.0,0.0,null,1.04,0.969,1.125,null,0.0,0.0,0.0,null,-6.204,-19.119,0.0,0.0,null,null],[3991.0,null,null,1985.426,951.533,null,141.441,111.8,0.0,0.0,null,1.051,0.981,1.14,null,0.0,0.0,0.0,null,-6.369,-23.033,0.0,0.0,null,null],[3991.5,null,null,1990.668,943.718,null,141.254,111.865,0.0,0.0,null,1.06,0.992,1.183,null,0.0,0.0,0.0,null,-6.499,-26.048,0.0,0.0,null,null],[3992.0,null,null,1992.346,937.491,null,141.085,111.314,0.0,0.0,null,1.067,1.003,1.224,null,0.0,0.0,0.0,null,-6.575,-26.152,0.0,0.0,null,null],[3992.5,null,null,1990.627,930.181,null,140.881,112.039,0.0,0.0,null,1.075,1.01,1.237,null,0.0,0.0,0.0,null,-6.657,-20.905,0.0,0.0,null,null],[3993.0,null,null,1985.136,923.022,null,140.594,114.168,0.0,0.0,null,1.083,1.012,1.226,null,0.0,0.0,0.0,null,-6.769,-12.737,0.0,0.0,null,null],[3993.5,null,null,1979.222,917.537,null,140.278,115.297,0.0,0.0,null,1.09,1.011,1.226,null,0.0,0.0,0.0,null,-6.858,-7.554,0.0,0.0,null,null],[3994.0,null,null,1975.267,913.832,null,139.962,113.868,0.0,0.0,null,1.094,1.008,1.228,null,0.0,0.0,0.0,null,-6.891,-4.995,0.0,0.0,null,null],[3994.5,null,null,1974.48,911.113,null,139.61,111.861,0.0,0.0,null,1.098,1.007,1.201,null,0.0,0.0,0.0,null,-6.888,-1.466,0.0,0.0,null,null],[3995.0,null,null,1974.731,909.835,null,139.196,110.919,0.0,0.0,null,1.099,1.009,1.168,null,0.0,0.0,0.0,null,-6.86,2.334,0.0,0.0,null,null],[3995.5,null,null,1973.352,909.103,null,138.715,110.069,0.0,0.0,null,1.1,1.016,1.151,null,0.0,0.0,0.0,null,-6.863,0.212,0.0,0.0,null,null],[3996.0,null,null,1972.476,906.609,null,138.151,109.271,0.0,0.0,null,1.103,1.027,1.164,null,0.0,0.0,0.0,null,-6.958,-6.83,0.0,0.0,null,null],[3996.5,null,null,1973.958,900.311,null,137.531,110.348,0.0,0.0,null,1.111,1.041,1.197,null,0.0,0.0,0.0,null,-7.119,-13.905,0.0,0.0,null,null],[3997.0,null,null,1975.988,889.943,null,136.968,112.665,0.0,0.0,null,1.124,1.056,1.24,null,0.0,0.0,0.0,null,-7.325,-17.235,0.0,0.0,null,null],[3997.5,null,null,1975.803,876.423,null,136.599,113.901,0.0,0.0,null,1.141,1.07,1.277,null,0.0,0.0,0.0,null,-7.571,-16.021,0.0,0.0,null,null],[3998.0,null,null,1973.713,861.37,null,136.441,113.227,0.0,0.0,null,1.161,1.087,1.276,null,0.0,0.0,0.0,null,-7.796,-14.529,0.0,0.0,null,null],[3998.5,null,null,1971.844,846.749,null,136.297,110.603,0.0,0.0,null,1.181,1.104,1.277,null,0.0,0.0,0.0,null,-7.957,-18.521,0.0,0.0,null,null],[3999.0,null,null,1969.469,833.481,null,135.907,107.317,0.0,0.0,null,1.2,1.121,1.32,null,0.0,0.0,0.0,null,-8.126,-25.892,0.0,0.0,null,null],[3999.5,null,null,1966.063,821.884,null,135.264,105.882,0.0,0.0,null,1.217,1.137,1.377,null,0.0,0.0,0.0,null,-8.322,-29.251,0.0,0.0,null,null],[4000.0,null,null,1967.423,812.674,null,134.626,106.538,0.0,0.0,null,1.231,1.154,1.389,null,0.0,0.0,0.0,null,-8.472,-28.405,0.0,0.0,null,null],[4000.5,null,null,1978.003,806.381,null,134.246,106.9,0.0,0.0,null,1.24,1.171,1.366,null,0.0,0.0,0.0,null,-8.54,-25.994,0.0,0.0,null,null],[4001.0,null,null,1985.965,801.598,null,134.143,108.244,0.0,0.0,null,1.248,1.183,1.369,null,0.0,0.0,0.0,null,-8.542,-24.894,0.0,0.0,null,null],[4001.5,null,null,1980.219,795.414,null,134.057,110.635,0.0,0.0,null,1.257,1.188,1.423,null,0.0,0.0,0.0,null,-8.536,-27.623,0.0,0.0,null,null],[4002.0,null,null,1969.424,787.165,null,133.637,111.462,0.0,0.0,null,1.27,1.192,1.462,null,0.0,0.0,0.0,null,-8.609,-33.043,0.0,0.0,null,null],[4002.5,null,null,1967.49,777.142,null,132.811,111.689,0.0,0.0,null,1.287,1.204,1.435,null,0.0,0.0,0.0,null,-8.712,-35.238,0.0,0.0,null,null],[4003.0,null,null,1968.313,768.126,null,131.917,113.223,0.0,0.0,null,1.302,1.226,1.367,null,0.0,0.0,0.0,null,-8.655,-31.326,0.0,0.0,null,null],[4003.5,null,null,1964.982,765.445,null,131.385,113.24,0.0,0.0,null,1.306,1.245,1.32,null,0.0,0.0,0.0,null,-8.449,-26.685,0.0,0.0,null,null],[4004.0,null,null,1961.667,772.174,null,131.388,109.906,0.0,0.0,null,1.295,1.243,1.402,null,0.0,0.0,0.0,null,-8.452,-25.566,0.0,0.0,null,null],[4004.5,null,null,1960.194,785.557,null,131.736,105.704,0.0,0.0,null,1.273,1.213,1.64,null,0.0,0.0,0.0,null,-8.728,-25.344,0.0,0.0,null,null],[4005.0,null,null,1963.916,802.632,null,132.071,103.646,0.0,0.0,null,1.246,1.167,1.772,null,0.0,0.0,0.0,null,-8.741,-25.165,0.0,0.0,null,null],[4005.5,null,null,1976.748,822.418,null,132.275,105.869,0.0,0.0,null,1.216,1.121,1.574,null,0.0,0.0,0.0,null,-8.162,-27.664,0.0,0.0,null,null],[4006.0,null,null,1989.07,844.269,null,132.625,111.782,0.0,0.0,null,1.184,1.079,1.325,null,0.0,0.0,0.0,null,-7.396,-30.765,0.0,0.0,null,null],[4006.5,null,null,1995.979,868.336,null,133.437,116.763,0.0,0.0,null,1.152,1.043,1.203,null,0.0,0.0,0.0,null,-6.923,-29.172,0.0,0.0,null,null],[4007.0,null,null,2002.537,894.046,null,134.677,117.47,0.0,0.0,null,1.119,1.015,1.14,null,0.0,0.0,0.0,null,-6.743,-22.698,0.0,0.0,null,null],[4007.5,null,null,2006.193,915.975,null,135.971,116.253,0.0,0.0,null,1.092,0.998,1.103,null,0.0,0.0,0.0,null,-6.7,-17.492,0.0,0.0,null,null],[4008.0,null,null,2000.146,929.618,null,136.916,115.59,0.0,0.0,null,1.076,0.991,1.091,null,0.0,0.0,0.0,null,-6.688,-14.545,0.0,0.0,null,null],[4008.5,null,null,1991.299,934.352,null,137.4,114.469,0.0,0.0,null,1.07,0.992,1.085,null,0.0,0.0,0.0,null,-6.685,-10.877,0.0,0.0,null,null],[4009.0,null,null,1985.031,931.416,null,137.627,112.426,0.0,0.0,null,1.074,1.0,1.091,null,0.0,0.0,0.0,null,-6.723,-8.807,0.0,0.0,null,null],[4009.5,null,null,1980.609,921.739,null,137.82,110.576,0.0,0.0,null,1.085,1.012,1.135,null,0.0,0.0,0.0,null,-6.831,-13.271,0.0,0.0,null,null],[4010.0,null,null,1977.201,909.34,null,138.041,108.72,0.0,0.0,null,1.1,1.027,1.207,null,0.0,0.0,0.0,null,-6.939,-18.675,0.0,0.0,null,null],[4010.5,null,null,1977.68,896.495,null,138.258,107.175,0.0,0.0,null,1.115,1.041,1.257,null,0.0,0.0,0.0,null,-6.993,-16.966,0.0,0.0,null,null],[4011.0,null,null,1980.49,884.407,null,138.354,107.261,0.0,0.0,null,1.131,1.054,1.259,null,0.0,0.0,0.0,null,-7.044,-6.925,0.0,0.0,null,null],[4011.5,null,null,1978.473,874.343,null,138.143,107.865,0.0,0.0,null,1.144,1.063,1.24,null,0.0,0.0,0.0,null,-7.135,2.699,0.0,0.0,null,null],[4012.0,null,null,1968.744,868.122,null,137.502,107.425,0.0,0.0,null,1.152,1.067,1.234,null,0.0,0.0,0.0,null,-7.243,4.698,0.0,0.0,null,null],[4012.5,null,null,1958.189,865.194,null,136.54,107.302,0.0,0.0,null,1.156,1.068,1.249,null,0.0,0.0,0.0,null,-7.338,2.267,0.0,0.0,null,null],[4013.0,null,null,1957.083,862.942,null,135.619,109.097,0.0,0.0,null,1.159,1.07,1.242,null,0.0,0.0,0.0,null,-7.431,0.602,0.0,0.0,null,null],[4013.5,null,null,1966.988,859.31,null,135.126,112.388,0.0,0.0,null,1.164,1.076,1.206,null,0.0,0.0,0.0,null,-7.502,-1.754,0.0,0.0,null,null],[4014.0,null,null,1976.695,854.587,null,135.076,116.199,0.0,0.0,null,1.17,1.088,1.199,null,0.0,0.0,0.0,null,-7.511,-5.667,0.0,0.0,null,null],[4014.5,null,null,1976.403,849.564,null,135.073,119.206,0.0,0.0,null,1.177,1.1,1.242,null,0.0,0.0,0.0,null,-7.482,-12.668,0.0,0.0,null,null],[4015.0,null,null,1969.955,842.921,null,134.823,119.73,0.0,0.0,null,1.186,1.11,1.329,null,0.0,0.0,0.0,null,-7.428,-20.682,0.0,0.0,null,null],[4015.5,null,null,1966.373,834.163,null,134.53,119.199,0.0,0.0,null,1.199,1.119,1.396,null,0.0,0.0,0.0,null,-7.339,-23.631,0.0,0.0,null,null],[4016.0,null,null,1968.432,824.849,null,134.699,118.953,0.0,0.0,null,1.212,1.133,1.377,null,0.0,0.0,0.0,null,-7.247,-22.891,0.0,0.0,null,null],[4016.5,null,null,1972.262,815.453,null,135.557,117.13,0.0,0.0,null,1.226,1.147,1.331,null,0.0,0.0,0.0,null,-7.169,-23.591,0.0,0.0,null,null],[4017.0,null,null,1977.616,807.131,null,136.716,113.073,0.0,0.0,null,1.239,1.156,1.326,null,0.0,0.0,0.0,null,-7.068,-21.829,0.0,0.0,null,null],[4017.5,null,null,1982.207,801.904,null,137.531,109.906,0.0,0.0,null,1.247,1.157,1.338,null,0.0,0.0,0.0,null,-6.937,-13.199,0.0,0.0,null,null],[4018.0,null,null,1981.26,801.495,null,137.77,108.962,0.0,0.0,null,1.248,1.151,1.324,null,0.0,0.0,0.0,null,-6.854,-5.002,0.0,0.0,null,null],[4018.5,null,null,1976.556,804.156,null,137.777,109.33,0.0,0.0,null,1.244,1.141,1.306,null,0.0,0.0,0.0,null,-6.891,-3.296,0.0,0.0,null,null],[4019.0,null,null,1972.288,808.097,null,137.977,110.793,0.0,0.0,null,1.237,1.13,1.32,null,0.0,0.0,0.0,null,-7.038,-4.107,0.0,0.0,null,null],[4019.5,null,null,1967.377,811.207,null,138.395,112.063,0.0,0.0,null,1.233,1.122,1.304,null,0.0,0.0,0.0,null,-7.191,-4.474,0.0,0.0,null,null],[4020.0,null,null,1965.88,811.845,null,138.661,112.865,0.0,0.0,null,1.232,1.124,1.232,null,0.0,0.0,0.0,null,-7.261,-7.861,0.0,0.0,null,null],[4020.5,null,null,1972.751,808.786,null,138.375,114.952,0.0,0.0,null,1.236,1.137,1.167,null,0.0,0.0,0.0,null,-7.282,-10.673,0.0,0.0,null,null],[4021.0,null,null,1981.456,803.607,null,137.423,117.802,0.0,0.0,null,1.244,1.155,1.189,null,0.0,0.0,0.0,null,-7.375,-8.832,0.0,0.0,null,null],[4021.5,null,null,1985.137,796.812,null,136.021,118.341,0.0,0.0,null,1.255,1.172,1.296,null,0.0,0.0,0.0,null,-7.566,-5.387,0.0,0.0,null,null],[4022.0,null,null,1982.374,786.481,null,134.538,116.354,0.0,0.0,null,1.271,1.191,1.355,null,0.0,0.0,0.0,null,-7.754,-1.626,0.0,0.0,null,null],[4022.5,null,null,1977.695,773.451,null,133.276,114.495,0.0,0.0,null,1.293,1.215,1.334,null,0.0,0.0,0.0,null,-7.886,1.886,0.0,0.0,null,null],[4023.0,null,null,1976.753,763.452,null,132.384,114.056,0.0,0.0,null,1.31,1.235,1.321,null,0.0,0.0,0.0,null,-8.021,1.488,0.0,0.0,null,null],[4023.5,null,null,1979.946,759.96,null,131.865,113.908,0.0,0.0,null,1.316,1.239,1.412,null,0.0,0.0,0.0,null,-8.192,-3.692,0.0,0.0,null,null],[4024.0,null,null,1982.177,760.45,null,131.66,112.579,0.0,0.0,null,1.315,1.227,1.552,null,0.0,0.0,0.0,null,-8.311,-9.103,0.0,0.0,null,null],[4024.5,null,null,1983.183,761.735,null,131.781,110.788,0.0,0.0,null,1.313,1.209,1.552,null,0.0,0.0,0.0,null,-8.22,-9.839,0.0,0.0,null,null],[4025.0,null,null,1983.677,764.818,null,132.303,110.418,0.0,0.0,null,1.308,1.196,1.429,null,0.0,0.0,0.0,null,-7.92,-8.817,0.0,0.0,null,null],[4025.5,null,null,1980.404,771.706,null,133.157,110.986,0.0,0.0,null,1.296,1.183,1.33,null,0.0,0.0,0.0,null,-7.614,-13.262,0.0,0.0,null,null],[4026.0,null,null,1973.249,784.114,null,134.073,110.601,0.0,0.0,null,1.275,1.164,1.336,null,0.0,0.0,0.0,null,-7.443,-18.448,0.0,0.0,null,null],[4026.5,null,null,1968.865,798.743,null,134.75,109.559,0.0,0.0,null,1.252,1.144,1.335,null,0.0,0.0,0.0,null,-7.349,-14.423,0.0,0.0,null,null],[4027.0,null,null,1970.744,809.762,null,134.982,109.17,0.0,0.0,null,1.235,1.128,1.284,null,0.0,0.0,0.0,null,-7.243,-4.063,0.0,0.0,null,null],[4027.5,null,null,1974.924,815.902,null,134.794,109.319,0.0,0.0,null,1.226,1.12,1.212,null,0.0,0.0,0.0,null,-7.128,2.153,0.0,0.0,null,null],[4028.0,null,null,1977.473,820.098,null,134.414,109.284,0.0,0.0,null,1.219,1.118,1.173,null,0.0,0.0,0.0,null,-7.08,-0.204,0.0,0.0,null,null],[4028.5,null,null,1980.882,821.996,null,134.053,110.312,0.0,0.0,null,1.217,1.119,1.193,null,0.0,0.0,0.0,null,-7.128,-2.005,0.0,0.0,null,null],[4029.0,null,null,1987.195,821.769,null,133.822,110.974,0.0,0.0,null,1.217,1.121,1.244,null,0.0,0.0,0.0,null,-7.221,1.149,0.0,0.0,null,null],[4029.5,null,null,1992.628,820.435,null,133.78,109.05,0.0,0.0,null,1.219,1.123,1.265,null,0.0,0.0,0.0,null,-7.317,0.757,0.0,0.0,null,null],[4030.0,null,null,1989.162,816.828,null,133.94,105.348,0.0,0.0,null,1.224,1.13,1.267,null,0.0,0.0,0.0,null,-7.408,-8.256,0.0,0.0,null,null],[4030.5,null,null,1977.296,810.079,null,134.259,103.322,0.0,0.0,null,1.234,1.141,1.293,null,0.0,0.0,0.0,null,-7.489,-16.553,0.0,0.0,null,null],[4031.0,null,null,1968.427,801.886,null,134.659,103.155,0.0,0.0,null,1.247,1.155,1.308,null,0.0,0.0,0.0,null,-7.575,-14.589,0.0,0.0,null,null],[4031.5,null,null,1970.789,791.269,null,134.994,102.443,0.0,0.0,null,1.264,1.173,1.298,null,0.0,0.0,0.0,null,-7.667,-4.61,0.0,0.0,null,null],[4032.0,null,null,1978.444,777.741,null,135.103,101.423,0.0,0.0,null,1.286,1.193,1.302,null,0.0,0.0,0.0,null,-7.763,1.592,0.0,0.0,null,null],[4032.5,null,null,1981.582,764.582,null,134.917,101.072,0.0,0.0,null,1.308,1.213,1.345,null,0.0,0.0,0.0,null,-7.884,1.634,0.0,0.0,null,null],[4033.0,null,null,1975.164,754.907,null,134.463,100.911,0.0,0.0,null,1.325,1.232,1.401,null,0.0,0.0,0.0,null,-8.016,3.767,0.0,0.0,null,null],[4033.5,null,null,1965.481,747.201,null,133.871,100.805,0.0,0.0,null,1.338,1.25,1.432,null,0.0,0.0,0.0,null,-8.119,9.242,0.0,0.0,null,null],[4034.0,null,null,1964.48,740.76,null,133.311,101.314,0.0,0.0,null,1.35,1.265,1.445,null,0.0,0.0,0.0,null,-8.169,11.701,0.0,0.0,null,null],[4034.5,null,null,1971.791,736.469,null,132.851,102.285,0.0,0.0,null,1.358,1.274,1.466,null,0.0,0.0,0.0,null,-8.187,9.917,0.0,0.0,null,null],[4035.0,null,null,1974.348,734.395,null,132.433,102.828,0.0,0.0,null,1.362,1.277,1.492,null,0.0,0.0,0.0,null,-8.207,5.681,0.0,0.0,null,null],[4035.5,null,null,1970.996,733.686,null,132.009,102.808,0.0,0.0,null,1.363,1.275,1.488,null,0.0,0.0,0.0,null,-8.232,0.303,0.0,0.0,null,null],[4036.0,null,null,1968.579,735.947,null,131.617,102.315,0.0,0.0,null,1.359,1.265,1.472,null,0.0,0.0,0.0,null,-8.247,-4.3,0.0,0.0,null,null],[4036.5,null,null,1968.849,742.104,null,131.345,101.594,0.0,0.0,null,1.348,1.25,1.453,null,0.0,0.0,0.0,null,-8.236,-8.292,0.0,0.0,null,null],[4037.0,null,null,1972.506,750.375,null,131.263,100.878,0.0,0.0,null,1.333,1.236,1.428,null,0.0,0.0,0.0,null,-8.133,-12.105,0.0,0.0,null,null],[4037.5,null,null,1975.635,758.679,null,131.343,100.927,0.0,0.0,null,1.318,1.22,1.397,null,0.0,0.0,0.0,null,-7.9,-12.739,0.0,0.0,null,null],[4038.0,null,null,1973.1,768.512,null,131.481,101.505,0.0,0.0,null,1.301,1.202,1.38,null,0.0,0.0,0.0,null,-7.619,-9.738,0.0,0.0,null,null],[4038.5,null,null,1967.489,781.675,null,131.604,100.892,0.0,0.0,null,1.279,1.181,1.401,null,0.0,0.0,0.0,null,-7.399,-9.009,0.0,0.0,null,null],[4039.0,null,null,1965.838,797.709,null,131.714,99.512,0.0,0.0,null,1.254,1.16,1.403,null,0.0,0.0,0.0,null,-7.238,-13.066,0.0,0.0,null,null],[4039.5,null,null,1969.129,813.653,null,131.85,99.522,0.0,0.0,null,1.229,1.139,1.372,null,0.0,0.0,0.0,null,-7.092,-14.503,0.0,0.0,null,null],[4040.0,null,null,1972.623,827.54,null,132.058,102.142,0.0,0.0,null,1.208,1.119,1.319,null,0.0,0.0,0.0,null,-6.956,-8.99,0.0,0.0,null,null],[4040.5,null,null,1971.834,839.314,null,132.323,106.168,0.0,0.0,null,1.191,1.104,1.273,null,0.0,0.0,0.0,null,-6.824,-3.229,0.0,0.0,null,null],[4041.0,null,null,1970.173,850.354,null,132.551,110.376,0.0,0.0,null,1.176,1.091,1.247,null,0.0,0.0,0.0,null,-6.662,-6.057,0.0,0.0,null,null],[4041.5,null,null,1967.269,861.998,null,132.684,114.309,0.0,0.0,null,1.16,1.079,1.256,null,0.0,0.0,0.0,null,-6.519,-12.955,0.0,0.0,null,null],[4042.0,null,null,1960.811,873.046,null,132.771,116.532,0.0,0.0,null,1.145,1.067,1.283,null,0.0,0.0,0.0,null,-6.511,-15.44,0.0,0.0,null,null],[4042.5,null,null,1958.007,880.703,null,132.923,115.856,0.0,0.0,null,1.135,1.056,1.296,null,0.0,0.0,0.0,null,-6.598,-13.313,0.0,0.0,null,null],[4043.0,null,null,1965.134,885.858,null,133.24,114.382,0.0,0.0,null,1.129,1.046,1.294,null,0.0,0.0,0.0,null,-6.626,-13.486,0.0,0.0,null,null],[4043.5,null,null,1970.182,891.665,null,133.73,114.665,0.0,0.0,null,1.121,1.035,1.284,null,0.0,0.0,0.0,null,-6.562,-17.751,0.0,0.0,null,null],[4044.0,null,null,1967.5,899.21,null,134.274,115.845,0.0,0.0,null,1.112,1.025,1.258,null,0.0,0.0,0.0,null,-6.502,-19.763,0.0,0.0,null,null],[4044.5,null,null,1967.704,908.658,null,134.771,116.463,0.0,0.0,null,1.101,1.012,1.239,null,0.0,0.0,0.0,null,-6.483,-17.174,0.0,0.0,null,null],[4045.0,null,null,1974.88,919.485,null,135.235,116.068,0.0,0.0,null,1.088,0.996,1.228,null,0.0,0.0,0.0,null,-6.521,-16.086,0.0,0.0,null,null],[4045.5,null,null,1973.994,931.003,null,135.735,113.67,0.0,0.0,null,1.074,0.982,1.196,null,0.0,0.0,0.0,null,-6.626,-19.462,0.0,0.0,null,null],[4046.0,null,null,1962.036,943.428,null,136.304,110.498,0.0,0.0,null,1.06,0.972,1.147,null,0.0,0.0,0.0,null,-6.768,-21.384,0.0,0.0,null,null],[4046.5,null,null,1953.749,955.014,null,136.866,109.554,0.0,0.0,null,1.047,0.966,1.116,null,0.0,0.0,0.0,null,-6.906,-19.429,0.0,0.0,null,null],[4047.0,null,null,1955.527,961.234,null,137.225,110.079,0.0,0.0,null,1.04,0.962,1.114,null,0.0,0.0,0.0,null,-7.014,-15.636,0.0,0.0,null,null],[4047.5,null,null,1962.09,961.397,null,137.233,109.327,0.0,0.0,null,1.04,0.963,1.109,null,0.0,0.0,0.0,null,-7.08,-11.466,0.0,0.0,null,null],[4048.0,null,null,1966.59,955.229,null,136.977,108.378,0.0,0.0,null,1.047,0.972,1.099,null,0.0,0.0,0.0,null,-7.186,-8.904,0.0,0.0,null,null],[4048.5,null,null,1965.033,940.346,null,136.664,109.247,0.0,0.0,null,1.063,0.99,1.103,null,0.0,0.0,0.0,null,-7.379,-8.635,0.0,0.0,null,null],[4049.0,null,null,1956.968,919.812,null,136.444,109.628,0.0,0.0,null,1.087,1.016,1.114,null,0.0,0.0,0.0,null,-7.621,-7.444,0.0,0.0,null,null],[4049.5,null,null,1948.118,899.21,null,136.288,108.857,0.0,0.0,null,1.112,1.044,1.158,null,0.0,0.0,0.0,null,-7.98,-5.008,0.0,0.0,null,null],[4050.0,null,null,1945.161,878.462,null,136.017,109.427,0.0,0.0,null,1.138,1.072,1.253,null,0.0,0.0,0.0,null,-8.483,-2.033,0.0,0.0,null,null],[4050.5,null,null,1951.372,855.862,null,135.56,111.063,0.0,0.0,null,1.168,1.102,1.354,null,0.0,0.0,0.0,null,-8.926,2.258,0.0,0.0,null,null],[4051.0,null,null,1963.232,832.745,null,135.1,112.218,0.0,0.0,null,1.201,1.134,1.411,null,0.0,0.0,0.0,null,-9.206,0.821,0.0,0.0,null,null],[4051.5,null,null,1972.629,811.741,null,134.921,112.433,0.0,0.0,null,1.232,1.166,1.398,null,0.0,0.0,0.0,null,-9.519,-8.971,0.0,0.0,null,null],[4052.0,null,null,1976.061,796.873,null,135.072,110.906,0.0,0.0,null,1.255,1.191,1.396,null,0.0,0.0,0.0,null,-9.96,-17.283,0.0,0.0,null,null],[4052.5,null,null,1974.563,791.848,null,135.28,109.259,0.0,0.0,null,1.263,1.203,1.417,null,0.0,0.0,0.0,null,-10.448,-17.858,0.0,0.0,null,null],[4053.0,null,null,1969.189,793.255,null,135.215,110.676,0.0,0.0,null,1.261,1.202,1.434,null,0.0,0.0,0.0,null,-10.983,-17.016,0.0,0.0,null,null],[4053.5,null,null,1960.921,793.703,null,134.809,114.115,0.0,0.0,null,1.26,1.195,1.404,null,0.0,0.0,0.0,null,-11.616,-18.614,0.0,0.0,null,null],[4054.0,null,null,1955.366,788.874,null,134.184,116.606,0.0,0.0,null,1.268,1.196,1.32,null,0.0,0.0,0.0,null,-12.396,-19.211,0.0,0.0,null,null],[4054.5,null,null,1957.157,779.957,null,133.296,116.916,0.0,0.0,null,1.282,1.216,1.218,null,0.0,0.0,0.0,null,-13.409,-15.153,0.0,0.0,null,null],[4055.0,null,null,1962.331,768.019,null,131.91,115.062,0.0,0.0,null,1.302,1.26,1.179,null,0.0,0.0,0.0,null,-14.632,-7.226,0.0,0.0,null,null],[4055.5,null,null,1965.063,755.542,null,129.931,111.895,0.0,0.0,null,1.324,1.323,1.178,null,0.0,0.0,0.0,null,-15.762,-1.359,0.0,0.0,null,null],[4056.0,null,null,1963.478,745.253,null,127.673,106.311,0.0,0.0,null,1.342,1.391,1.173,null,0.0,0.0,0.0,null,-16.874,-3.174,0.0,0.0,null,null],[4056.5,null,null,1958.523,734.447,null,125.672,94.994,0.0,0.0,null,1.362,1.435,1.477,null,0.0,0.0,0.0,null,-18.993,-10.714,0.0,0.0,null,null],[4057.0,null,null,1951.309,720.449,null,124.172,80.421,0.0,0.0,null,1.388,1.451,2.418,null,0.0,0.0,0.0,null,-22.295,-18.857,0.0,0.0,null,null],[4057.5,null,null,1947.386,706.119,null,122.913,71.137,0.0,0.0,null,1.416,1.458,3.033,null,0.0,0.0,0.0,null,-24.773,-24.747,0.0,0.0,null,null],[4058.0,null,null,1951.896,692.099,null,121.637,70.519,0.0,0.0,null,1.445,1.472,2.467,null,0.0,0.0,0.0,null,-24.842,-26.935,0.0,0.0,null,null],[4058.5,null,null,1957.951,679.516,null,120.705,74.094,0.0,0.0,null,1.472,1.491,1.893,null,0.0,0.0,0.0,null,-23.197,-26.586,0.0,0.0,null,null],[4059.0,null,null,1958.55,676.027,null,120.825,76.712,0.0,0.0,null,1.479,1.496,1.768,null,0.0,0.0,0.0,null,-21.136,-26.758,0.0,0.0,null,null],[4059.5,null,null,1955.328,687.027,null,122.217,76.209,0.0,0.0,null,1.456,1.464,1.927,null,0.0,0.0,0.0,null,-19.633,-24.982,0.0,0.0,null,null],[4060.0,null,null,1953.305,708.094,null,124.248,74.427,0.0,0.0,null,1.412,1.395,2.285,null,0.0,0.0,0.0,null,-18.871,-18.509,0.0,0.0,null,null],[4060.5,null,null,1955.758,731.261,null,126.013,76.817,0.0,0.0,null,1.367,1.319,2.505,null,0.0,0.0,0.0,null,-17.57,-14.861,0.0,0.0,null,null],[4061.0,null,null,1964.377,751.779,null,127.216,85.427,0.0,0.0,null,1.33,1.261,1.906,null,0.0,0.0,0.0,null,-14.735,-17.687,0.0,0.0,null,null],[4061.5,null,null,1970.385,771.167,null,128.346,94.267,0.0,0.0,null,1.297,1.216,1.323,null,0.0,0.0,0.0,null,-11.493,-21.561,0.0,0.0,null,null],[4062.0,null,null,1970.691,792.484,null,129.892,98.492,0.0,0.0,null,1.262,1.173,1.201,null,0.0,0.0,0.0,null,-9.298,-23.852,0.0,0.0,null,null],[4062.5,null,null,1974.003,816.399,null,131.654,101.547,0.0,0.0,null,1.225,1.133,1.27,null,0.0,0.0,0.0,null,-8.275,-23.488,0.0,0.0,null,null],[4063.0,null,null,1979.482,840.227,null,133.057,106.74,0.0,0.0,null,1.19,1.102,1.232,null,0.0,0.0,0.0,null,-7.855,-21.811,0.0,0.0,null,null],[4063.5,null,null,1976.266,860.126,null,133.814,111.169,0.0,0.0,null,1.163,1.079,1.141,null,0.0,0.0,0.0,null,-7.66,-23.311,0.0,0.0,null,null],[4064.0,null,null,1963.435,874.115,null,134.169,112.483,0.0,0.0,null,1.144,1.061,1.139,null,0.0,0.0,0.0,null,-7.546,-26.772,0.0,0.0,null,null],[4064.5,null,null,1948.795,883.474,null,134.616,113.056,0.0,0.0,null,1.132,1.048,1.196,null,0.0,0.0,0.0,null,-7.429,-25.328,0.0,0.0,null,null],[4065.0,null,null,1940.202,888.823,null,135.426,113.934,0.0,0.0,null,1.125,1.04,1.215,null,0.0,0.0,0.0,null,-7.332,-20.469,0.0,0.0,null,null],[4065.5,null,null,1941.614,891.399,null,136.464,113.971,0.0,0.0,null,1.122,1.037,1.217,null,0.0,0.0,0.0,null,-7.307,-15.777,0.0,0.0,null,null],[4066.0,null,null,1949.6,891.998,null,137.447,113.46,0.0,0.0,null,1.121,1.034,1.215,null,0.0,0.0,0.0,null,-7.33,-6.901,0.0,0.0,null,null],[4066.5,null,null,1956.288,890.752,null,138.215,113.772,0.0,0.0,null,1.123,1.031,1.215,null,0.0,0.0,0.0,null,-7.376,3.892,0.0,0.0,null,null],[4067.0,null,null,1956.302,888.984,null,138.74,114.111,0.0,0.0,null,1.125,1.032,1.196,null,0.0,0.0,0.0,null,-7.467,5.625,0.0,0.0,null,null],[4067.5,null,null,1949.156,888.884,null,139.092,113.177,0.0,0.0,null,1.125,1.036,1.174,null,0.0,0.0,0.0,null,-7.625,-2.138,0.0,0.0,null,null],[4068.0,null,null,1943.066,889.436,null,139.332,111.416,0.0,0.0,null,1.124,1.041,1.198,null,0.0,0.0,0.0,null,-7.844,-8.591,0.0,0.0,null,null],[4068.5,null,null,1947.603,889.598,null,139.487,110.095,0.0,0.0,null,1.124,1.041,1.227,null,0.0,0.0,0.0,null,-8.076,-12.287,0.0,0.0,null,null],[4069.0,null,null,1959.409,889.217,null,139.549,109.734,0.0,0.0,null,1.125,1.039,1.224,null,0.0,0.0,0.0,null,-8.222,-14.655,0.0,0.0,null,null],[4069.5,null,null,1967.607,888.592,null,139.475,109.756,0.0,0.0,null,1.125,1.038,1.206,null,0.0,0.0,0.0,null,-8.248,-15.273,0.0,0.0,null,null],[4070.0,null,null,1964.301,888.952,null,139.238,109.632,0.0,0.0,null,1.125,1.038,1.218,null,0.0,0.0,0.0,null,-8.246,-11.283,0.0,0.0,null,null],[4070.5,null,null,1954.771,890.136,null,138.828,108.599,0.0,0.0,null,1.123,1.038,1.23,null,0.0,0.0,0.0,null,-8.26,-3.805,0.0,0.0,null,null],[4071.0,null,null,1949.037,889.965,null,138.25,107.802,0.0,0.0,null,1.124,1.039,1.228,null,0.0,0.0,0.0,null,-8.238,-0.296,0.0,0.0,null,null],[4071.5,null,null,1950.913,888.635,null,137.568,108.282,0.0,0.0,null,1.125,1.042,1.221,null,0.0,0.0,0.0,null,-8.147,-6.309,0.0,0.0,null,null],[4072.0,null,null,1955.57,886.265,null,136.914,108.778,0.0,0.0,null,1.128,1.046,1.233,null,0.0,0.0,0.0,null,-8.061,-14.903,0.0,0.0,null,null],[4072.5,null,null,1958.991,880.797,null,136.33,107.901,0.0,0.0,null,1.135,1.053,1.24,null,0.0,0.0,0.0,null,-8.03,-21.785,0.0,0.0,null,null],[4073.0,null,null,1959.321,871.568,null,135.765,107.109,0.0,0.0,null,1.147,1.066,1.205,null,0.0,0.0,0.0,null,-8.001,-26.96,0.0,0.0,null,null],[4073.5,null,null,1958.42,859.332,null,135.256,107.807,0.0,0.0,null,1.164,1.084,1.198,null,0.0,0.0,0.0,null,-7.953,-25.91,0.0,0.0,null,null],[4074.0,null,null,1961.294,846.053,null,134.935,109.746,0.0,0.0,null,1.182,1.103,1.239,null,0.0,0.0,0.0,null,-7.955,-20.798,0.0,0.0,null,null],[4074.5,null,null,1965.192,834.12,null,134.891,112.596,0.0,0.0,null,1.199,1.119,1.283,null,0.0,0.0,0.0,null,-8.041,-20.186,0.0,0.0,null,null],[4075.0,null,null,1964.544,823.926,null,135.072,114.417,0.0,0.0,null,1.214,1.13,1.299,null,0.0,0.0,0.0,null,-8.147,-23.873,0.0,0.0,null,null],[4075.5,null,null,1964.257,813.947,null,135.299,114.121,0.0,0.0,null,1.229,1.139,1.3,null,0.0,0.0,0.0,null,-8.226,-24.756,0.0,0.0,null,null],[4076.0,null,null,1970.524,804.813,null,135.441,113.111,0.0,0.0,null,1.243,1.147,1.313,null,0.0,0.0,0.0,null,-8.317,-24.591,0.0,0.0,null,null],[4076.5,null,null,1976.069,797.642,null,135.529,112.769,0.0,0.0,null,1.254,1.157,1.328,null,0.0,0.0,0.0,null,-8.428,-26.467,0.0,0.0,null,null],[4077.0,null,null,1971.572,790.602,null,135.615,112.842,0.0,0.0,null,1.265,1.169,1.339,null,0.0,0.0,0.0,null,-8.51,-23.658,0.0,0.0,null,null],[4077.5,null,null,1963.633,782.037,null,135.65,113.243,0.0,0.0,null,1.279,1.184,1.324,null,0.0,0.0,0.0,null,-8.556,-14.814,0.0,0.0,null,null],[4078.0,null,null,1960.254,772.371,null,135.589,113.664,0.0,0.0,null,1.295,1.201,1.296,null,0.0,0.0,0.0,null,-8.658,-10.778,0.0,0.0,null,null],[4078.5,null,null,1959.875,763.418,null,135.507,114.517,0.0,0.0,null,1.31,1.218,1.293,null,0.0,0.0,0.0,null,-8.823,-14.415,0.0,0.0,null,null],[4079.0,null,null,1961.154,755.214,null,135.482,116.563,0.0,0.0,null,1.324,1.231,1.334,null,0.0,0.0,0.0,null,-8.971,-17.237,0.0,0.0,null,null],[4079.5,null,null,1965.542,747.588,null,135.479,118.924,0.0,0.0,null,1.338,1.243,1.387,null,0.0,0.0,0.0,null,-9.074,-16.397,0.0,0.0,null,null],[4080.0,null,null,1971.091,740.197,null,135.411,119.38,0.0,0.0,null,1.351,1.256,1.406,null,0.0,0.0,0.0,null,-9.184,-14.82,0.0,0.0,null,null],[4080.5,null,null,1975.637,733.675,null,135.278,117.687,0.0,0.0,null,1.363,1.265,1.41,null,0.0,0.0,0.0,null,-9.317,-16.139,0.0,0.0,null,null],[4081.0,null,null,1976.778,729.484,null,135.192,115.846,0.0,0.0,null,1.371,1.264,1.438,null,0.0,0.0,0.0,null,-9.429,-17.98,0.0,0.0,null,null],[4081.5,null,null,1970.651,729.768,null,135.192,115.569,0.0,0.0,null,1.37,1.257,1.452,null,0.0,0.0,0.0,null,-9.464,-13.945,0.0,0.0,null,null],[4082.0,null,null,1963.959,734.121,null,135.151,116.491,0.0,0.0,null,1.362,1.246,1.439,null,0.0,0.0,0.0,null,-9.393,-7.177,0.0,0.0,null,null],[4082.5,null,null,1965.435,738.823,null,134.925,116.944,0.0,0.0,null,1.354,1.236,1.406,null,0.0,0.0,0.0,null,-9.199,-7.411,0.0,0.0,null,null],[4083.0,null,null,1970.383,741.29,null,134.565,116.863,0.0,0.0,null,1.349,1.229,1.353,null,0.0,0.0,0.0,null,-8.896,-13.497,0.0,0.0,null,null],[4083.5,null,null,1970.839,744.106,null,134.273,117.45,0.0,0.0,null,1.344,1.228,1.307,null,0.0,0.0,0.0,null,-8.559,-17.8,0.0,0.0,null,null],[4084.0,null,null,1967.952,749.263,null,134.135,119.503,0.0,0.0,null,1.335,1.23,1.315,null,0.0,0.0,0.0,null,-8.267,-19.441,0.0,0.0,null,null],[4084.5,null,null,1964.945,755.26,null,134.04,120.775,0.0,0.0,null,1.324,1.229,1.365,null,0.0,0.0,0.0,null,-8.091,-19.637,0.0,0.0,null,null],[4085.0,null,null,1963.518,759.827,null,133.838,119.829,0.0,0.0,null,1.316,1.226,1.407,null,0.0,0.0,0.0,null,-8.071,-14.779,0.0,0.0,null,null],[4085.5,null,null,1963.319,763.221,null,133.537,118.635,0.0,0.0,null,1.31,1.221,1.395,null,0.0,0.0,0.0,null,-8.128,-5.589,0.0,0.0,null,null],[4086.0,null,null,1960.592,767.59,null,133.301,118.274,0.0,0.0,null,1.303,1.212,1.383,null,0.0,0.0,0.0,null,-8.138,1.938,0.0,0.0,null,null],[4086.5,null,null,1955.874,775.419,null,133.313,117.801,0.0,0.0,null,1.29,1.197,1.392,null,0.0,0.0,0.0,null,-8.094,5.711,0.0,0.0,null,null],[4087.0,null,null,1957.359,787.037,null,133.619,116.992,0.0,0.0,null,1.271,1.171,1.429,null,0.0,0.0,0.0,null,-8.006,6.745,0.0,0.0,null,null],[4087.5,null,null,1963.149,802.437,null,134.084,115.245,0.0,0.0,null,1.246,1.139,1.437,null,0.0,0.0,0.0,null,-7.822,4.335,0.0,0.0,null,null],[4088.0,null,null,1967.135,820.793,null,134.524,112.402,0.0,0.0,null,1.218,1.108,1.349,null,0.0,0.0,0.0,null,-7.555,-1.474,0.0,0.0,null,null],[4088.5,null,null,1971.818,839.099,null,134.828,110.727,0.0,0.0,null,1.192,1.084,1.218,null,0.0,0.0,0.0,null,-7.279,-7.608,0.0,0.0,null,null],[4089.0,null,null,1979.034,855.047,null,135.024,110.866,0.0,0.0,null,1.17,1.068,1.148,null,0.0,0.0,0.0,null,-7.06,-10.382,0.0,0.0,null,null],[4089.5,null,null,1981.329,868.635,null,135.243,111.075,0.0,0.0,null,1.151,1.055,1.166,null,0.0,0.0,0.0,null,-6.936,-8.644,0.0,0.0,null,null],[4090.0,null,null,1975.594,880.347,null,135.539,110.765,0.0,0.0,null,1.136,1.044,1.218,null,0.0,0.0,0.0,null,-6.877,-6.238,0.0,0.0,null,null],[4090.5,null,null,1966.561,890.355,null,135.811,110.59,0.0,0.0,null,1.123,1.036,1.224,null,0.0,0.0,0.0,null,-6.816,-6.244,0.0,0.0,null,null],[4091.0,null,null,1958.551,897.259,null,135.87,109.635,0.0,0.0,null,1.115,1.032,1.177,null,0.0,0.0,0.0,null,-6.754,-7.151,0.0,0.0,null,null],[4091.5,null,null,1953.83,899.965,null,135.6,108.056,0.0,0.0,null,1.111,1.031,1.154,null,0.0,0.0,0.0,null,-6.711,-6.497,0.0,0.0,null,null],[4092.0,null,null,1952.164,898.642,null,135.081,106.526,0.0,0.0,null,1.113,1.029,1.173,null,0.0,0.0,0.0,null,-6.653,-5.622,0.0,0.0,null,null],[4092.5,null,null,1951.902,895.813,null,134.551,105.411,0.0,0.0,null,1.116,1.029,1.168,null,0.0,0.0,0.0,null,-6.539,-7.385,0.0,0.0,null,null],[4093.0,null,null,1951.083,892.548,null,134.195,104.934,0.0,0.0,null,1.12,1.034,1.15,null,0.0,0.0,0.0,null,-6.409,-10.262,0.0,0.0,null,null],[4093.5,null,null,1950.657,890.311,null,134.025,105.861,0.0,0.0,null,1.123,1.042,1.171,null,0.0,0.0,0.0,null,-6.34,-12.06,0.0,0.0,null,null],[4094.0,null,null,1951.573,888.849,null,133.969,107.032,0.0,0.0,null,1.125,1.047,1.23,null,0.0,0.0,0.0,null,-6.355,-14.334,0.0,0.0,null,null],[4094.5,null,null,1952.593,888.462,null,134.012,106.797,0.0,0.0,null,1.126,1.05,1.271,null,0.0,0.0,0.0,null,-6.379,-17.461,0.0,0.0,null,null],[4095.0,null,null,1953.134,888.947,null,134.276,104.55,0.0,0.0,null,1.125,1.049,1.284,null,0.0,0.0,0.0,null,-6.379,-13.784,0.0,0.0,null,null],[4095.5,null,null,1952.257,890.11,null,134.929,102.565,0.0,0.0,null,1.123,1.044,1.284,null,0.0,0.0,0.0,null,-6.366,-3.061,0.0,0.0,null,null],[4096.0,null,null,1947.052,891.224,null,135.945,103.104,0.0,0.0,null,1.122,1.036,1.273,null,0.0,0.0,0.0,null,-6.338,1.218,0.0,0.0,null,null],[4096.5,null,null,1939.994,894.952,null,137.017,104.454,0.0,0.0,null,1.117,1.029,1.223,null,0.0,0.0,0.0,null,-6.314,-8.005,0.0,0.0,null,null],[4097.0,null,null,1937.733,904.067,null,137.773,103.418,0.0,0.0,null,1.106,1.021,1.185,null,0.0,0.0,0.0,null,-6.327,-17.535,0.0,0.0,null,null],[4097.5,null,null,1942.471,918.27,null,137.993,100.939,0.0,0.0,null,1.089,1.01,1.191,null,0.0,0.0,0.0,null,-6.352,-17.751,0.0,0.0,null,null],[4098.0,null,null,1949.981,931.588,null,137.727,100.823,0.0,0.0,null,1.073,0.997,1.198,null,0.0,0.0,0.0,null,-6.339,-17.201,0.0,0.0,null,null],[4098.5,null,null,1952.596,940.052,null,137.24,103.377,0.0,0.0,null,1.064,0.986,1.184,null,0.0,0.0,0.0,null,-6.299,-20.314,0.0,0.0,null,null],[4099.0,null,null,1949.293,946.047,null,136.8,106.707,0.0,0.0,null,1.057,0.979,1.149,null,0.0,0.0,0.0,null,-6.262,-23.068,0.0,0.0,null,null],[4099.5,null,null,1949.89,953.352,null,136.541,108.009,0.0,0.0,null,1.049,0.975,1.133,null,0.0,0.0,0.0,null,-6.218,-24.057,0.0,0.0,null,null],[4100.0,null,null,1956.327,959.407,null,136.495,105.722,0.0,0.0,null,1.042,0.971,1.147,null,0.0,0.0,0.0,null,-6.159,-18.116,0.0,0.0,null,null],[4100.5,null,null,1956.902,961.704,null,136.682,101.988,0.0,0.0,null,1.04,0.966,1.166,null,0.0,0.0,0.0,null,-6.129,-6.304,0.0,0.0,null,null],[4101.0,null,null,1948.852,961.151,null,137.107,100.59,0.0,0.0,null,1.04,0.963,1.163,null,0.0,0.0,0.0,null,-6.149,2.117,0.0,0.0,null,null],[4101.5,null,null,1940.598,961.08,null,137.77,101.716,0.0,0.0,null,1.04,0.964,1.138,null,0.0,0.0,0.0,null,-6.163,7.902,0.0,0.0,null,null],[4102.0,null,null,1937.982,962.747,null,138.621,103.185,0.0,0.0,null,1.039,0.969,1.11,null,0.0,0.0,0.0,null,-6.133,15.305,0.0,0.0,null,null],[4102.5,null,null,1938.832,962.604,null,139.504,104.103,0.0,0.0,null,1.039,0.971,1.104,null,0.0,0.0,0.0,null,-6.103,16.825,0.0,0.0,null,null],[4103.0,null,null,1940.223,958.582,null,140.241,104.309,0.0,0.0,null,1.043,0.969,1.144,null,0.0,0.0,0.0,null,-6.135,9.821,0.0,0.0,null,null],[4103.5,null,null,1945.138,955.74,null,140.718,104.637,0.0,0.0,null,1.046,0.967,1.199,null,0.0,0.0,0.0,null,-6.206,3.127,0.0,0.0,null,null],[4104.0,null,null,1954.826,956.5,null,140.958,105.539,0.0,0.0,null,1.045,0.967,1.22,null,0.0,0.0,0.0,null,-6.26,-0.227,0.0,0.0,null,null],[4104.5,null,null,1960.023,958.798,null,141.076,106.309,0.0,0.0,null,1.043,0.965,1.18,null,0.0,0.0,0.0,null,-6.272,-2.55,0.0,0.0,null,null],[4105.0,null,null,1954.597,963.266,null,141.214,105.929,0.0,0.0,null,1.038,0.96,1.135,null,0.0,0.0,0.0,null,-6.247,-4.535,0.0,0.0,null,null],[4105.5,null,null,1948.445,971.394,null,141.405,104.222,0.0,0.0,null,1.029,0.954,1.123,null,0.0,0.0,0.0,null,-6.182,-3.807,0.0,0.0,null,null],[4106.0,null,null,1949.997,980.507,null,141.555,102.071,0.0,0.0,null,1.02,0.947,1.13,null,0.0,0.0,0.0,null,-6.086,-1.692,0.0,0.0,null,null],[4106.5,null,null,1952.323,987.927,null,141.573,100.704,0.0,0.0,null,1.012,0.939,1.145,null,0.0,0.0,0.0,null,-6.01,-4.085,0.0,0.0,null,null],[4107.0,null,null,1948.053,993.571,null,141.457,100.638,0.0,0.0,null,1.006,0.932,1.147,null,0.0,0.0,0.0,null,-5.976,-13.549,0.0,0.0,null,null],[4107.5,null,null,1942.293,999.588,null,141.304,101.158,0.0,0.0,null,1.0,0.928,1.129,null,0.0,0.0,0.0,null,-5.938,-22.445,0.0,0.0,null,null],[4108.0,null,null,1941.702,1006.872,null,141.241,101.473,0.0,0.0,null,0.993,0.923,1.116,null,0.0,0.0,0.0,null,-5.856,-21.176,0.0,0.0,null,null],[4108.5,null,null,1943.78,1015.124,null,141.284,101.829,0.0,0.0,null,0.985,0.916,1.112,null,0.0,0.0,0.0,null,-5.754,-11.449,0.0,0.0,null,null],[4109.0,null,null,1943.75,1023.018,null,141.309,102.497,0.0,0.0,null,0.977,0.909,1.111,null,0.0,0.0,0.0,null,-5.665,-5.542,0.0,0.0,null,null],[4109.5,null,null,1945.835,1028.506,null,141.18,102.047,0.0,0.0,null,0.972,0.906,1.09,null,0.0,0.0,0.0,null,-5.598,-9.065,0.0,0.0,null,null],[4110.0,null,null,1954.149,1030.807,null,140.856,100.092,0.0,0.0,null,0.97,0.907,1.059,null,0.0,0.0,0.0,null,-5.559,-15.168,0.0,0.0,null,null],[4110.5,null,null,1960.058,1030.736,null,140.394,98.46,0.0,0.0,null,0.97,0.909,1.049,null,0.0,0.0,0.0,null,-5.613,-15.692,0.0,0.0,null,null],[4111.0,null,null,1958.123,1028.039,null,139.917,97.734,0.0,0.0,null,0.973,0.909,1.055,null,0.0,0.0,0.0,null,-5.784,-12.166,0.0,0.0,null,null],[4111.5,null,null,1954.612,1020.259,null,139.496,97.522,0.0,0.0,null,0.98,0.912,1.061,null,0.0,0.0,0.0,null,-5.984,-7.983,0.0,0.0,null,null],[4112.0,null,null,1956.512,1006.946,null,139.098,98.197,0.0,0.0,null,0.993,0.922,1.042,null,0.0,0.0,0.0,null,-6.139,-2.421,0.0,0.0,null,null],[4112.5,null,null,1964.41,991.065,null,138.685,100.515,0.0,0.0,null,1.009,0.941,1.002,null,0.0,0.0,0.0,null,-6.312,1.019,0.0,0.0,null,null],[4113.0,null,null,1973.374,974.672,null,138.284,102.84,0.0,0.0,null,1.026,0.965,1.015,null,0.0,0.0,0.0,null,-6.56,-1.301,0.0,0.0,null,null],[4113.5,null,null,1974.993,957.377,null,137.993,103.196,0.0,0.0,null,1.045,0.987,1.136,null,0.0,0.0,0.0,null,-6.873,-5.3,0.0,0.0,null,null],[4114.0,null,null,1973.086,938.594,null,137.955,103.38,0.0,0.0,null,1.065,1.005,1.269,null,0.0,0.0,0.0,null,-7.212,-8.326,0.0,0.0,null,null],[4114.5,null,null,1972.798,919.98,null,138.205,105.733,0.0,0.0,null,1.087,1.02,1.296,null,0.0,0.0,0.0,null,-7.5,-11.727,0.0,0.0,null,null],[4115.0,null,null,1973.152,903.271,null,138.587,107.149,0.0,0.0,null,1.107,1.033,1.266,null,0.0,0.0,0.0,null,-7.693,-10.252,0.0,0.0,null,null],[4115.5,null,null,1972.991,890.452,null,138.91,106.218,0.0,0.0,null,1.123,1.043,1.262,null,0.0,0.0,0.0,null,-7.823,-1.304,0.0,0.0,null,null],[4116.0,null,null,1974.486,881.524,null,139.114,105.64,0.0,0.0,null,1.134,1.05,1.263,null,0.0,0.0,0.0,null,-7.911,7.414,0.0,0.0,null,null],[4116.5,null,null,1974.696,876.825,null,139.23,106.401,0.0,0.0,null,1.14,1.055,1.241,null,0.0,0.0,0.0,null,-7.948,7.88,0.0,0.0,null,null],[4117.0,null,null,1973.857,875.626,null,139.281,106.952,0.0,0.0,null,1.142,1.058,1.24,null,0.0,0.0,0.0,null,-7.958,0.731,0.0,0.0,null,null],[4117.5,null,null,1973.321,875.17,null,139.233,108.056,0.0,0.0,null,1.143,1.059,1.247,null,0.0,0.0,0.0,null,-7.975,-4.547,0.0,0.0,null,null],[4118.0,null,null,1970.392,873.597,null,139.024,109.325,0.0,0.0,null,1.145,1.06,1.216,null,0.0,0.0,0.0,null,-7.991,-2.618,0.0,0.0,null,null],[4118.5,null,null,1965.019,871.818,null,138.696,108.697,0.0,0.0,null,1.147,1.06,1.227,null,0.0,0.0,0.0,null,-7.987,1.678,0.0,0.0,null,null],[4119.0,null,null,1962.689,870.699,null,138.399,106.958,0.0,0.0,null,1.149,1.059,1.26,null,0.0,0.0,0.0,null,-7.947,2.927,0.0,0.0,null,null],[4119.5,null,null,1965.779,870.484,null,138.217,106.19,0.0,0.0,null,1.149,1.06,1.263,null,0.0,0.0,0.0,null,-7.855,2.593,0.0,0.0,null,null],[4120.0,null,null,1966.593,870.51,null,138.055,107.47,0.0,0.0,null,1.149,1.06,1.226,null,0.0,0.0,0.0,null,-7.741,1.42,0.0,0.0,null,null],[4120.5,null,null,1962.747,871.852,null,137.773,109.009,0.0,0.0,null,1.147,1.06,1.196,null,0.0,0.0,0.0,null,-7.681,-2.545,0.0,0.0,null,null],[4121.0,null,null,1961.308,873.894,null,137.354,108.587,0.0,0.0,null,1.144,1.061,1.194,null,0.0,0.0,0.0,null,-7.741,-7.781,0.0,0.0,null,null],[4121.5,null,null,1964.534,871.99,null,136.941,106.521,0.0,0.0,null,1.147,1.066,1.178,null,0.0,0.0,0.0,null,-7.92,-9.393,0.0,0.0,null,null],[4122.0,null,null,1968.937,860.992,null,136.734,106.253,0.0,0.0,null,1.161,1.076,1.174,null,0.0,0.0,0.0,null,-8.134,-6.581,0.0,0.0,null,null],[4122.5,null,null,1970.655,843.232,null,136.738,108.519,0.0,0.0,null,1.186,1.096,1.187,null,0.0,0.0,0.0,null,-8.314,-3.258,0.0,0.0,null,null],[4123.0,null,null,1968.633,823.523,null,136.7,110.824,0.0,0.0,null,1.214,1.127,1.189,null,0.0,0.0,0.0,null,-8.454,-1.083,0.0,0.0,null,null],[4123.5,null,null,1963.301,805.065,null,136.382,111.635,0.0,0.0,null,1.242,1.165,1.197,null,0.0,0.0,0.0,null,-8.574,2.447,0.0,0.0,null,null],[4124.0,null,null,1956.148,788.873,null,135.828,111.829,0.0,0.0,null,1.268,1.201,1.266,null,0.0,0.0,0.0,null,-8.726,8.808,0.0,0.0,null,null],[4124.5,null,null,1950.223,775.391,null,135.302,112.841,0.0,0.0,null,1.29,1.223,1.424,null,0.0,0.0,0.0,null,-8.945,12.949,0.0,0.0,null,null],[4125.0,null,null,1948.997,763.748,null,134.988,113.293,0.0,0.0,null,1.309,1.23,1.592,null,0.0,0.0,0.0,null,-9.145,7.496,0.0,0.0,null,null],[4125.5,null,null,1950.055,755.283,null,134.845,112.536,0.0,0.0,null,1.324,1.229,1.6,null,0.0,0.0,0.0,null,-9.2,-2.89,0.0,0.0,null,null],[4126.0,null,null,1949.948,752.165,null,134.712,111.928,0.0,0.0,null,1.329,1.224,1.486,null,0.0,0.0,0.0,null,-9.102,-6.218,0.0,0.0,null,null],[4126.5,null,null,1951.092,755.313,null,134.577,112.739,0.0,0.0,null,1.324,1.213,1.38,null,0.0,0.0,0.0,null,-8.928,-0.983,0.0,0.0,null,null],[4127.0,null,null,1953.282,764.151,null,134.653,115.064,0.0,0.0,null,1.309,1.197,1.309,null,0.0,0.0,0.0,null,-8.759,3.983,0.0,0.0,null,null],[4127.5,null,null,1953.603,775.725,null,135.095,117.766,0.0,0.0,null,1.289,1.179,1.259,null,0.0,0.0,0.0,null,-8.621,2.472,0.0,0.0,null,null],[4128.0,null,null,1951.71,785.344,null,135.786,118.849,0.0,0.0,null,1.273,1.165,1.247,null,0.0,0.0,0.0,null,-8.48,-5.693,0.0,0.0,null,null],[4128.5,null,null,1950.567,794.233,null,136.428,117.977,0.0,0.0,null,1.259,1.155,1.256,null,0.0,0.0,0.0,null,-8.27,-12.008,0.0,0.0,null,null],[4129.0,null,null,1954.071,804.498,null,136.804,115.902,0.0,0.0,null,1.243,1.145,1.255,null,0.0,0.0,0.0,null,-8.013,-8.504,0.0,0.0,null,null],[4129.5,null,null,1959.244,812.414,null,136.843,113.477,0.0,0.0,null,1.231,1.137,1.277,null,0.0,0.0,0.0,null,-7.82,-2.152,0.0,0.0,null,null],[4130.0,null,null,1957.376,814.026,null,136.572,111.379,0.0,0.0,null,1.228,1.131,1.284,null,0.0,0.0,0.0,null,-7.723,-3.235,0.0,0.0,null,null],[4130.5,null,null,1949.327,812.87,null,136.079,110.497,0.0,0.0,null,1.23,1.128,1.246,null,0.0,0.0,0.0,null,-7.658,-6.974,0.0,0.0,null,null],[4131.0,null,null,1943.695,814.25,null,135.526,110.698,0.0,0.0,null,1.228,1.129,1.199,null,0.0,0.0,0.0,null,-7.621,-8.173,0.0,0.0,null,null],[4131.5,null,null,1943.509,817.695,null,135.105,111.351,0.0,0.0,null,1.223,1.125,1.222,null,0.0,0.0,0.0,null,-7.635,-6.159,0.0,0.0,null,null],[4132.0,null,null,1946.776,819.057,null,134.867,111.639,0.0,0.0,null,1.221,1.117,1.276,null,0.0,0.0,0.0,null,-7.624,-2.256,0.0,0.0,null,null],[4132.5,null,null,1950.287,817.596,null,134.728,112.37,0.0,0.0,null,1.223,1.112,1.268,null,0.0,0.0,0.0,null,-7.546,-1.24,0.0,0.0,null,null],[4133.0,null,null,1951.586,816.007,null,134.658,114.0,0.0,0.0,null,1.225,1.118,1.208,null,0.0,0.0,0.0,null,-7.468,-5.921,0.0,0.0,null,null],[4133.5,null,null,1952.948,815.667,null,134.801,113.841,0.0,0.0,null,1.226,1.131,1.204,null,0.0,0.0,0.0,null,-7.48,-10.341,0.0,0.0,null,null],[4134.0,null,null,1954.965,815.085,null,135.255,110.839,0.0,0.0,null,1.227,1.141,1.277,null,0.0,0.0,0.0,null,-7.571,-9.944,0.0,0.0,null,null],[4134.5,null,null,1955.424,812.96,null,135.827,107.453,0.0,0.0,null,1.23,1.144,1.329,null,0.0,0.0,0.0,null,-7.671,-6.097,0.0,0.0,null,null],[4135.0,null,null,1956.292,809.611,null,136.108,105.204,0.0,0.0,null,1.235,1.144,1.33,null,0.0,0.0,0.0,null,-7.745,-3.565,0.0,0.0,null,null],[4135.5,null,null,1957.721,806.425,null,135.843,104.265,0.0,0.0,null,1.24,1.146,1.319,null,0.0,0.0,0.0,null,-7.822,-2.735,0.0,0.0,null,null],[4136.0,null,null,1956.101,805.755,null,135.202,104.939,0.0,0.0,null,1.241,1.146,1.309,null,0.0,0.0,0.0,null,-7.902,-1.972,0.0,0.0,null,null],[4136.5,null,null,1952.193,807.174,null,134.549,106.381,0.0,0.0,null,1.239,1.146,1.282,null,0.0,0.0,0.0,null,-7.931,-5.687,0.0,0.0,null,null],[4137.0,null,null,1952.719,808.35,null,134.062,107.382,0.0,0.0,null,1.237,1.143,1.267,null,0.0,0.0,0.0,null,-7.927,-13.685,0.0,0.0,null,null],[4137.5,null,null,1961.123,808.499,null,133.613,107.703,0.0,0.0,null,1.237,1.14,1.286,null,0.0,0.0,0.0,null,-7.951,-14.573,0.0,0.0,null,null],[4138.0,null,null,1972.6,809.803,null,133.018,107.408,0.0,0.0,null,1.235,1.137,1.306,null,0.0,0.0,0.0,null,-7.991,-7.424,0.0,0.0,null,null],[4138.5,null,null,1978.646,812.205,null,132.311,107.472,0.0,0.0,null,1.231,1.136,1.303,null,0.0,0.0,0.0,null,-7.987,-4.881,0.0,0.0,null,null],[4139.0,null,null,1979.299,813.901,null,131.75,107.626,0.0,0.0,null,1.229,1.135,1.287,null,0.0,0.0,0.0,null,-7.968,-7.28,0.0,0.0,null,null],[4139.5,null,null,1978.602,815.887,null,131.58,106.175,0.0,0.0,null,1.226,1.133,1.276,null,0.0,0.0,0.0,null,-7.957,-4.169,0.0,0.0,null,null],[4140.0,null,null,1973.935,819.997,null,131.792,105.617,0.0,0.0,null,1.22,1.127,1.284,null,0.0,0.0,0.0,null,-7.911,1.191,0.0,0.0,null,null],[4140.5,null,null,1963.544,825.284,null,132.203,107.367,0.0,0.0,null,1.212,1.12,1.311,null,0.0,0.0,0.0,null,-7.785,-3.443,0.0,0.0,null,null],[4141.0,null,null,1957.084,830.314,null,132.685,107.649,0.0,0.0,null,1.204,1.111,1.33,null,0.0,0.0,0.0,null,-7.612,-13.275,0.0,0.0,null,null],[4141.5,null,null,1963.078,835.005,null,133.2,104.16,0.0,0.0,null,1.198,1.103,1.303,null,0.0,0.0,0.0,null,-7.451,-17.413,0.0,0.0,null,null],[4142.0,null,null,1971.316,839.134,null,133.709,100.695,0.0,0.0,null,1.192,1.097,1.271,null,0.0,0.0,0.0,null,-7.341,-15.612,0.0,0.0,null,null],[4142.5,null,null,1968.31,842.05,null,134.134,100.754,0.0,0.0,null,1.188,1.095,1.267,null,0.0,0.0,0.0,null,-7.276,-10.894,0.0,0.0,null,null],[4143.0,null,null,1960.265,844.333,null,134.372,103.91,0.0,0.0,null,1.184,1.095,1.275,null,0.0,0.0,0.0,null,-7.247,-6.343,0.0,0.0,null,null],[4143.5,null,null,1959.089,846.027,null,134.383,106.512,0.0,0.0,null,1.182,1.098,1.267,null,0.0,0.0,0.0,null,-7.246,-7.358,0.0,0.0,null,null],[4144.0,null,null,1961.251,846.439,null,134.248,106.032,0.0,0.0,null,1.181,1.1,1.263,null,0.0,0.0,0.0,null,-7.285,-14.978,0.0,0.0,null,null],[4144.5,null,null,1962.277,845.649,null,134.077,103.99,0.0,0.0,null,1.183,1.102,1.284,null,0.0,0.0,0.0,null,-7.381,-21.943,0.0,0.0,null,null],[4145.0,null,null,1966.155,844.795,null,133.959,102.397,0.0,0.0,null,1.184,1.102,1.301,null,0.0,0.0,0.0,null,-7.513,-22.379,0.0,0.0,null,null],[4145.5,null,null,1972.845,844.486,null,134.016,102.179,0.0,0.0,null,1.184,1.101,1.283,null,0.0,0.0,0.0,null,-7.622,-15.901,0.0,0.0,null,null],[4146.0,null,null,1977.461,846.949,null,134.369,102.656,0.0,0.0,null,1.181,1.096,1.272,null,0.0,0.0,0.0,null,-7.732,-7.344,0.0,0.0,null,null],[4146.5,null,null,1979.799,853.036,null,135.009,102.812,0.0,0.0,null,1.172,1.088,1.266,null,0.0,0.0,0.0,null,-7.876,-1.702,0.0,0.0,null,null],[4147.0,null,null,1983.177,858.929,null,135.775,102.426,0.0,0.0,null,1.164,1.079,1.266,null,0.0,0.0,0.0,null,-8.046,0.816,0.0,0.0,null,null],[4147.5,null,null,1984.859,860.933,null,136.435,103.137,0.0,0.0,null,1.162,1.073,1.26,null,0.0,0.0,0.0,null,-8.227,0.448,0.0,0.0,null,null],[4148.0,null,null,1982.768,861.134,null,136.837,105.572,0.0,0.0,null,1.161,1.072,1.244,null,0.0,0.0,0.0,null,-8.416,-3.446,0.0,0.0,null,null],[4148.5,null,null,1980.45,861.213,null,136.987,107.476,0.0,0.0,null,1.161,1.075,1.237,null,0.0,0.0,0.0,null,-8.561,-9.771,0.0,0.0,null,null],[4149.0,null,null,1979.851,860.071,null,136.912,107.547,0.0,0.0,null,1.163,1.081,1.253,null,0.0,0.0,0.0,null,-8.636,-12.698,0.0,0.0,null,null],[4149.5,null,null,1979.649,856.596,null,136.589,106.998,0.0,0.0,null,1.167,1.087,1.293,null,0.0,0.0,0.0,null,-8.671,-10.334,0.0,0.0,null,null],[4150.0,null,null,1977.853,851.466,null,136.032,107.086,0.0,0.0,null,1.174,1.091,1.326,null,0.0,0.0,0.0,null,-8.721,-6.592,0.0,0.0,null,null],[4150.5,null,null,1974.99,846.098,null,135.337,107.793,0.0,0.0,null,1.182,1.095,1.333,null,0.0,0.0,0.0,null,-8.821,-5.261,0.0,0.0,null,null],[4151.0,null,null,1974.811,841.496,null,134.646,109.058,0.0,0.0,null,1.188,1.1,1.311,null,0.0,0.0,0.0,null,-8.907,-7.525,0.0,0.0,null,null],[4151.5,null,null,1976.473,836.306,null,134.047,110.334,0.0,0.0,null,1.196,1.106,1.296,null,0.0,0.0,0.0,null,-8.917,-12.583,0.0,0.0,null,null],[4152.0,null,null,1972.008,830.51,null,133.541,111.585,0.0,0.0,null,1.204,1.112,1.289,null,0.0,0.0,0.0,null,-8.883,-18.022,0.0,0.0,null,null],[4152.5,null,null,1960.116,826.559,null,133.091,112.733,0.0,0.0,null,1.21,1.121,1.277,null,0.0,0.0,0.0,null,-8.894,-18.68,0.0,0.0,null,null],[4153.0,null,null,1953.151,824.365,null,132.685,113.473,0.0,0.0,null,1.213,1.131,1.273,null,0.0,0.0,0.0,null,-9.002,-14.025,0.0,0.0,null,null],[4153.5,null,null,1957.227,820.821,null,132.308,114.523,0.0,0.0,null,1.218,1.14,1.291,null,0.0,0.0,0.0,null,-9.177,-10.483,0.0,0.0,null,null],[4154.0,null,null,1963.776,815.544,null,131.932,116.121,0.0,0.0,null,1.226,1.146,1.337,null,0.0,0.0,0.0,null,-9.301,-10.611,0.0,0.0,null,null],[4154.5,null,null,1962.932,810.558,null,131.586,117.058,0.0,0.0,null,1.234,1.151,1.345,null,0.0,0.0,0.0,null,-9.335,-12.329,0.0,0.0,null,null],[4155.0,null,null,1960.167,806.836,null,131.371,116.909,0.0,0.0,null,1.239,1.156,1.337,null,0.0,0.0,0.0,null,-9.337,-12.791,0.0,0.0,null,null],[4155.5,null,null,1963.5,804.044,null,131.431,115.976,0.0,0.0,null,1.244,1.158,1.352,null,0.0,0.0,0.0,null,-9.33,-8.131,0.0,0.0,null,null],[4156.0,null,null,1968.462,802.319,null,131.856,115.022,0.0,0.0,null,1.246,1.156,1.368,null,0.0,0.0,0.0,null,-9.311,0.796,0.0,0.0,null,null],[4156.5,null,null,1968.669,802.541,null,132.579,114.869,0.0,0.0,null,1.246,1.152,1.353,null,0.0,0.0,0.0,null,-9.328,3.682,0.0,0.0,null,null],[4157.0,null,null,1965.322,805.856,null,133.391,115.009,0.0,0.0,null,1.241,1.15,1.331,null,0.0,0.0,0.0,null,-9.392,1.784,0.0,0.0,null,null],[4157.5,null,null,1962.599,810.304,null,134.083,115.23,0.0,0.0,null,1.234,1.149,1.322,null,0.0,0.0,0.0,null,-9.483,3.837,0.0,0.0,null,null],[4158.0,null,null,1958.955,813.234,null,134.584,117.217,0.0,0.0,null,1.23,1.146,1.333,null,0.0,0.0,0.0,null,-9.628,6.893,0.0,0.0,null,null],[4158.5,null,null,1956.008,814.02,null,134.909,120.683,0.0,0.0,null,1.228,1.144,1.331,null,0.0,0.0,0.0,null,-9.786,2.645,0.0,0.0,null,null],[4159.0,null,null,1959.213,812.989,null,135.055,122.934,0.0,0.0,null,1.23,1.145,1.319,null,0.0,0.0,0.0,null,-9.855,-7.75,0.0,0.0,null,null],[4159.5,null,null,1963.925,810.941,null,134.939,121.59,0.0,0.0,null,1.233,1.147,1.314,null,0.0,0.0,0.0,null,-9.873,-18.884,0.0,0.0,null,null],[4160.0,null,null,1962.311,809.78,null,134.489,118.96,0.0,0.0,null,1.235,1.147,1.328,null,0.0,0.0,0.0,null,-9.972,-24.902,0.0,0.0,null,null],[4160.5,null,null,1958.47,809.589,null,133.787,118.655,0.0,0.0,null,1.235,1.146,1.346,null,0.0,0.0,0.0,null,-10.123,-23.003,0.0,0.0,null,null],[4161.0,null,null,1958.98,810.06,null,133.074,120.076,0.0,0.0,null,1.234,1.149,1.339,null,0.0,0.0,0.0,null,-10.213,-16.702,0.0,0.0,null,null],[4161.5,null,null,1960.725,811.302,null,132.616,119.908,0.0,0.0,null,1.233,1.152,1.329,null,0.0,0.0,0.0,null,-10.261,-12.69,0.0,0.0,null,null],[4162.0,null,null,1961.901,812.055,null,132.535,118.601,0.0,0.0,null,1.231,1.151,1.344,null,0.0,0.0,0.0,null,-10.325,-15.236,0.0,0.0,null,null],[4162.5,null,null,1964.167,810.801,null,132.788,118.214,0.0,0.0,null,1.233,1.15,1.343,null,0.0,0.0,0.0,null,-10.335,-23.434,0.0,0.0,null,null],[4163.0,null,null,1968.137,808.626,null,133.265,117.81,0.0,0.0,null,1.237,1.151,1.312,null,0.0,0.0,0.0,null,-10.193,-27.032,0.0,0.0,null,null],[4163.5,null,null,1975.051,804.432,null,133.812,116.527,0.0,0.0,null,1.243,1.154,1.283,null,0.0,0.0,0.0,null,-9.954,-20.261,0.0,0.0,null,null],[4164.0,null,null,1982.153,797.6,null,134.219,116.181,0.0,0.0,null,1.254,1.162,1.282,null,0.0,0.0,0.0,null,-9.763,-9.829,0.0,0.0,null,null],[4164.5,null,null,1982.804,790.08,null,134.299,118.151,0.0,0.0,null,1.266,1.177,1.306,null,0.0,0.0,0.0,null,-9.661,-5.337,0.0,0.0,null,null],[4165.0,null,null,1973.039,785.988,null,134.043,121.238,0.0,0.0,null,1.272,1.196,1.328,null,0.0,0.0,0.0,null,-9.557,-10.265,0.0,0.0,null,null],[4165.5,null,null,1959.289,786.387,null,133.639,122.341,0.0,0.0,null,1.272,1.206,1.351,null,0.0,0.0,0.0,null,-9.373,-18.386,0.0,0.0,null,null],[4166.0,null,null,1954.994,790.155,null,133.309,119.505,0.0,0.0,null,1.266,1.2,1.433,null,0.0,0.0,0.0,null,-9.146,-21.946,0.0,0.0,null,null],[4166.5,null,null,1960.804,794.578,null,133.123,113.608,0.0,0.0,null,1.259,1.178,1.568,null,0.0,0.0,0.0,null,-8.897,-19.694,0.0,0.0,null,null],[4167.0,null,null,1963.453,797.988,null,132.962,108.657,0.0,0.0,null,1.253,1.153,1.565,null,0.0,0.0,0.0,null,-8.623,-16.43,0.0,0.0,null,null],[4167.5,null,null,1959.301,801.634,null,132.758,107.485,0.0,0.0,null,1.247,1.136,1.378,null,0.0,0.0,0.0,null,-8.38,-14.49,0.0,0.0,null,null],[4168.0,null,null,1959.301,809.678,null,132.635,109.138,0.0,0.0,null,1.235,1.124,1.217,null,0.0,0.0,0.0,null,-8.217,-15.026,0.0,0.0,null,null],[4168.5,null,null,1967.067,824.156,null,132.78,110.272,0.0,0.0,null,1.213,1.109,1.193,null,0.0,0.0,0.0,null,-8.115,-15.104,0.0,0.0,null,null],[4169.0,null,null,1977.583,842.505,null,133.25,110.452,0.0,0.0,null,1.187,1.092,1.231,null,0.0,0.0,0.0,null,-8.085,-10.941,0.0,0.0,null,null],[4169.5,null,null,1983.305,859.233,null,133.882,110.692,0.0,0.0,null,1.164,1.076,1.228,null,0.0,0.0,0.0,null,-8.098,-7.514,0.0,0.0,null,null],[4170.0,null,null,1978.917,870.24,null,134.428,110.33,0.0,0.0,null,1.149,1.062,1.218,null,0.0,0.0,0.0,null,-8.073,-8.619,0.0,0.0,null,null],[4170.5,null,null,1970.817,876.48,null,134.758,109.219,0.0,0.0,null,1.141,1.051,1.214,null,0.0,0.0,0.0,null,-7.993,-7.797,0.0,0.0,null,null],[4171.0,null,null,1971.404,881.814,null,134.938,109.566,0.0,0.0,null,1.134,1.043,1.215,null,0.0,0.0,0.0,null,-7.9,-1.233,0.0,0.0,null,null],[4171.5,null,null,1977.541,886.911,null,135.154,111.676,0.0,0.0,null,1.128,1.036,1.2,null,0.0,0.0,0.0,null,-7.83,1.661,0.0,0.0,null,null],[4172.0,null,null,1978.865,891.086,null,135.565,112.854,0.0,0.0,null,1.122,1.029,1.18,null,0.0,0.0,0.0,null,-7.804,-3.82,0.0,0.0,null,null],[4172.5,null,null,1976.909,896.12,null,136.2,111.635,0.0,0.0,null,1.116,1.023,1.173,null,0.0,0.0,0.0,null,-7.762,-9.725,0.0,0.0,null,null],[4173.0,null,null,1978.753,902.487,null,136.91,109.526,0.0,0.0,null,1.108,1.02,1.156,null,0.0,0.0,0.0,null,-7.686,-11.929,0.0,0.0,null,null],[4173.5,null,null,1978.52,905.943,null,137.475,108.516,0.0,0.0,null,1.104,1.02,1.139,null,0.0,0.0,0.0,null,-7.671,-12.539,0.0,0.0,null,null],[4174.0,null,null,1971.702,904.923,null,137.762,108.523,0.0,0.0,null,1.105,1.023,1.147,null,0.0,0.0,0.0,null,-7.756,-10.904,0.0,0.0,null,null],[4174.5,null,null,1968.351,898.972,null,137.798,108.924,0.0,0.0,null,1.112,1.029,1.178,null,0.0,0.0,0.0,null,-7.866,-5.974,0.0,0.0,null,null],[4175.0,null,null,1972.647,887.337,null,137.709,109.518,0.0,0.0,null,1.127,1.045,1.161,null,0.0,0.0,0.0,null,-7.945,0.144,0.0,0.0,null,null],[4175.5,null,null,1975.835,871.114,null,137.534,110.029,0.0,0.0,null,1.148,1.069,1.118,null,0.0,0.0,0.0,null,-8.037,5.581,0.0,0.0,null,null],[4176.0,null,null,1976.316,854.461,null,137.222,109.635,0.0,0.0,null,1.17,1.094,1.146,null,0.0,0.0,0.0,null,-8.231,4.964,0.0,0.0,null,null],[4176.5,null,null,1978.625,838.47,null,136.745,109.107,0.0,0.0,null,1.193,1.113,1.24,null,0.0,0.0,0.0,null,-8.534,-3.066,0.0,0.0,null,null],[4177.0,null,null,1977.824,824.159,null,136.177,111.219,0.0,0.0,null,1.213,1.13,1.336,null,0.0,0.0,0.0,null,-8.856,-8.88,0.0,0.0,null,null],[4177.5,null,null,1969.495,811.282,null,135.666,115.12,0.0,0.0,null,1.233,1.149,1.373,null,0.0,0.0,0.0,null,-9.108,-7.583,0.0,0.0,null,null],[4178.0,null,null,1959.191,799.009,null,135.288,116.344,0.0,0.0,null,1.252,1.166,1.376,null,0.0,0.0,0.0,null,-9.251,-5.31,0.0,0.0,null,null],[4178.5,null,null,1953.805,787.816,null,134.984,115.435,0.0,0.0,null,1.269,1.178,1.387,null,0.0,0.0,0.0,null,-9.317,-5.399,0.0,0.0,null,null],[4179.0,null,null,1954.367,781.151,null,134.617,116.436,0.0,0.0,null,1.28,1.182,1.391,null,0.0,0.0,0.0,null,-9.373,-5.941,0.0,0.0,null,null],[4179.5,null,null,1957.677,780.737,null,134.146,117.771,0.0,0.0,null,1.281,1.177,1.392,null,0.0,0.0,0.0,null,-9.404,-6.31,0.0,0.0,null,null],[4180.0,null,null,1958.37,783.972,null,133.689,117.349,0.0,0.0,null,1.276,1.169,1.373,null,0.0,0.0,0.0,null,-9.377,-6.424,0.0,0.0,null,null],[4180.5,null,null,1957.053,786.896,null,133.41,116.662,0.0,0.0,null,1.271,1.161,1.325,null,0.0,0.0,0.0,null,-9.34,-6.147,0.0,0.0,null,null],[4181.0,null,null,1958.198,789.745,null,133.438,116.777,0.0,0.0,null,1.266,1.156,1.271,null,0.0,0.0,0.0,null,-9.328,-9.084,0.0,0.0,null,null],[4181.5,null,null,1959.176,793.187,null,133.793,116.979,0.0,0.0,null,1.261,1.155,1.255,null,0.0,0.0,0.0,null,-9.348,-12.599,0.0,0.0,null,null],[4182.0,null,null,1957.632,796.843,null,134.41,116.115,0.0,0.0,null,1.255,1.155,1.279,null,0.0,0.0,0.0,null,-9.44,-10.594,0.0,0.0,null,null],[4182.5,null,null,1956.796,800.834,null,135.14,114.884,0.0,0.0,null,1.249,1.155,1.286,null,0.0,0.0,0.0,null,-9.573,-5.882,0.0,0.0,null,null],[4183.0,null,null,1955.932,803.805,null,135.844,114.359,0.0,0.0,null,1.244,1.155,1.296,null,0.0,0.0,0.0,null,-9.638,-2.66,0.0,0.0,null,null],[4183.5,null,null,1950.735,802.93,null,136.442,113.618,0.0,0.0,null,1.245,1.154,1.314,null,0.0,0.0,0.0,null,-9.665,1.45,0.0,0.0,null,null],[4184.0,null,null,1944.635,799.425,null,136.884,112.843,0.0,0.0,null,1.251,1.157,1.334,null,0.0,0.0,0.0,null,-9.742,6.546,0.0,0.0,null,null],[4184.5,null,null,1944.378,795.804,null,137.147,114.326,0.0,0.0,null,1.257,1.161,1.334,null,0.0,0.0,0.0,null,-9.846,7.398,0.0,0.0,null,null],[4185.0,null,null,1951.744,792.17,null,137.136,117.032,0.0,0.0,null,1.262,1.167,1.303,null,0.0,0.0,0.0,null,-9.938,3.504,0.0,0.0,null,null],[4185.5,null,null,1960.331,789.301,null,136.761,117.561,0.0,0.0,null,1.267,1.171,1.287,null,0.0,0.0,0.0,null,-10.048,-1.939,0.0,0.0,null,null],[4186.0,null,null,1964.305,788.597,null,136.07,116.544,0.0,0.0,null,1.268,1.172,1.303,null,0.0,0.0,0.0,null,-10.16,-6.856,0.0,0.0,null,null],[4186.5,null,null,1965.703,787.846,null,135.266,117.399,0.0,0.0,null,1.269,1.171,1.336,null,0.0,0.0,0.0,null,-10.203,-9.595,0.0,0.0,null,null],[4187.0,null,null,1968.06,784.824,null,134.564,119.958,0.0,0.0,null,1.274,1.174,1.344,null,0.0,0.0,0.0,null,-10.183,-8.762,0.0,0.0,null,null],[4187.5,null,null,1971.528,779.784,null,134.04,121.18,0.0,0.0,null,1.282,1.18,1.322,null,0.0,0.0,0.0,null,-10.177,-7.386,0.0,0.0,null,null],[4188.0,null,null,1970.124,773.994,null,133.589,120.852,0.0,0.0,null,1.292,1.189,1.309,null,0.0,0.0,0.0,null,-10.234,-10.511,0.0,0.0,null,null],[4188.5,null,null,1962.816,768.317,null,133.027,119.944,0.0,0.0,null,1.302,1.201,1.321,null,0.0,0.0,0.0,null,-10.338,-14.833,0.0,0.0,null,null],[4189.0,null,null,1956.392,763.235,null,132.289,118.333,0.0,0.0,null,1.31,1.215,1.348,null,0.0,0.0,0.0,null,-10.457,-15.76,0.0,0.0,null,null],[4189.5,null,null,1956.375,756.169,null,131.427,116.881,0.0,0.0,null,1.322,1.231,1.363,null,0.0,0.0,0.0,null,-10.578,-15.114,0.0,0.0,null,null],[4190.0,null,null,1959.937,744.712,null,130.494,116.333,0.0,0.0,null,1.343,1.251,1.385,null,0.0,0.0,0.0,null,-10.724,-14.83,0.0,0.0,null,null],[4190.5,null,null,1963.632,729.066,null,129.572,116.519,0.0,0.0,null,1.372,1.279,1.415,null,0.0,0.0,0.0,null,-10.873,-15.917,0.0,0.0,null,null],[4191.0,null,null,1964.07,714.399,null,128.793,117.039,0.0,0.0,null,1.4,1.311,1.433,null,0.0,0.0,0.0,null,-10.952,-20.869,0.0,0.0,null,null],[4191.5,null,null,1963.696,706.061,null,128.3,117.183,0.0,0.0,null,1.416,1.336,1.448,null,0.0,0.0,0.0,null,-10.945,-26.604,0.0,0.0,null,null],[4192.0,null,null,1966.864,705.264,null,128.194,116.85,0.0,0.0,null,1.418,1.341,1.492,null,0.0,0.0,0.0,null,-10.938,-27.519,0.0,0.0,null,null],[4192.5,null,null,1970.637,709.708,null,128.504,117.384,0.0,0.0,null,1.409,1.32,1.581,null,0.0,0.0,0.0,null,-10.971,-22.956,0.0,0.0,null,null],[4193.0,null,null,1972.994,717.239,null,129.215,118.215,0.0,0.0,null,1.394,1.286,1.643,null,0.0,0.0,0.0,null,-10.956,-18.22,0.0,0.0,null,null],[4193.5,null,null,1976.217,726.2,null,130.264,116.887,0.0,0.0,null,1.377,1.249,1.575,null,0.0,0.0,0.0,null,-10.767,-18.58,0.0,0.0,null,null],[4194.0,null,null,1976.761,737.79,null,131.507,113.592,0.0,0.0,null,1.355,1.219,1.403,null,0.0,0.0,0.0,null,-10.437,-20.986,0.0,0.0,null,null],[4194.5,null,null,1969.437,752.715,null,132.721,111.727,0.0,0.0,null,1.329,1.194,1.279,null,0.0,0.0,0.0,null,-10.15,-17.313,0.0,0.0,null,null],[4195.0,null,null,1960.044,769.894,null,133.741,112.093,0.0,0.0,null,1.299,1.175,1.242,null,0.0,0.0,0.0,null,-10.006,-11.609,0.0,0.0,null,null],[4195.5,null,null,1956.446,787.548,null,134.595,112.46,0.0,0.0,null,1.27,1.159,1.26,null,0.0,0.0,0.0,null,-9.939,-13.883,0.0,0.0,null,null],[4196.0,null,null,1958.282,801.922,null,135.398,112.434,0.0,0.0,null,1.247,1.148,1.289,null,0.0,0.0,0.0,null,-9.873,-22.6,0.0,0.0,null,null],[4196.5,null,null,1958.968,809.254,null,136.162,112.361,0.0,0.0,null,1.236,1.141,1.306,null,0.0,0.0,0.0,null,-9.769,-29.952,0.0,0.0,null,null],[4197.0,null,null,1955.847,811.419,null,136.71,111.624,0.0,0.0,null,1.232,1.136,1.316,null,0.0,0.0,0.0,null,-9.648,-32.192,0.0,0.0,null,null],[4197.5,null,null,1953.212,811.762,null,136.871,110.274,0.0,0.0,null,1.232,1.131,1.306,null,0.0,0.0,0.0,null,-9.605,-31.13,0.0,0.0,null,null],[4198.0,null,null,1955.069,810.612,null,136.694,110.309,0.0,0.0,null,1.234,1.131,1.281,null,0.0,0.0,0.0,null,-9.679,-29.262,0.0,0.0,null,null],[4198.5,null,null,1957.777,809.027,null,136.442,113.361,0.0,0.0,null,1.236,1.134,1.267,null,0.0,0.0,0.0,null,-9.756,-24.344,0.0,0.0,null,null],[4199.0,null,null,1959.813,809.905,null,136.328,116.623,0.0,0.0,null,1.235,1.136,1.273,null,0.0,0.0,0.0,null,-9.748,-16.431,0.0,0.0,null,null],[4199.5,null,null,1963.065,813.598,null,136.262,116.054,0.0,0.0,null,1.229,1.133,1.318,null,0.0,0.0,0.0,null,-9.671,-13.788,0.0,0.0,null,null],[4200.0,null,null,1963.976,817.494,null,135.977,114.226,0.0,0.0,null,1.223,1.128,1.359,null,0.0,0.0,0.0,null,-9.56,-18.43,0.0,0.0,null,null],[4200.5,null,null,1962.064,819.167,null,135.416,114.499,0.0,0.0,null,1.221,1.126,1.346,null,0.0,0.0,0.0,null,-9.472,-21.318,0.0,0.0,null,null],[4201.0,null,null,1964.557,819.087,null,134.849,114.69,0.0,0.0,null,1.221,1.126,1.302,null,0.0,0.0,0.0,null,-9.491,-19.585,0.0,0.0,null,null],[4201.5,null,null,1969.164,818.806,null,134.585,113.236,0.0,0.0,null,1.221,1.126,1.267,null,0.0,0.0,0.0,null,-9.595,-20.528,0.0,0.0,null,null],[4202.0,null,null,1968.195,818.803,null,134.632,112.289,0.0,0.0,null,1.221,1.124,1.267,null,0.0,0.0,0.0,null,-9.629,-26.278,0.0,0.0,null,null],[4202.5,null,null,1964.937,820.392,null,134.7,113.493,0.0,0.0,null,1.219,1.121,1.285,null,0.0,0.0,0.0,null,-9.538,-30.677,0.0,0.0,null,null],[4203.0,null,null,1966.518,826.488,null,134.516,116.121,0.0,0.0,null,1.21,1.117,1.293,null,0.0,0.0,0.0,null,-9.448,-30.12,0.0,0.0,null,null],[4203.5,null,null,1970.015,838.945,null,134.115,117.068,0.0,0.0,null,1.192,1.107,1.286,null,0.0,0.0,0.0,null,-9.435,-28.286,0.0,0.0,null,null],[4204.0,null,null,1972.088,856.023,null,133.757,114.227,0.0,0.0,null,1.168,1.086,1.317,null,0.0,0.0,0.0,null,-9.388,-28.246,0.0,0.0,null,null],[4204.5,null,null,1971.962,873.375,null,133.654,110.282,0.0,0.0,null,1.145,1.054,1.371,null,0.0,0.0,0.0,null,-9.256,-24.733,0.0,0.0,null,null],[4205.0,null,null,1970.242,887.253,null,133.836,109.437,0.0,0.0,null,1.127,1.021,1.349,null,0.0,0.0,0.0,null,-9.101,-17.707,0.0,0.0,null,null],[4205.5,null,null,1969.245,898.156,null,134.195,111.399,0.0,0.0,null,1.113,0.998,1.202,null,0.0,0.0,0.0,null,-8.949,-15.408,0.0,0.0,null,null],[4206.0,null,null,1970.09,910.908,null,134.605,113.983,0.0,0.0,null,1.098,0.986,1.061,null,0.0,0.0,0.0,null,-8.8,-18.033,0.0,0.0,null,null],[4206.5,null,null,1968.025,926.354,null,135.011,115.523,0.0,0.0,null,1.08,0.98,1.033,null,0.0,0.0,0.0,null,-8.703,-18.667,0.0,0.0,null,null],[4207.0,null,null,1960.925,938.817,null,135.438,114.599,0.0,0.0,null,1.065,0.975,1.087,null,0.0,0.0,0.0,null,-8.692,-15.579,0.0,0.0,null,null],[4207.5,null,null,1957.102,943.699,null,135.899,112.434,0.0,0.0,null,1.06,0.975,1.126,null,0.0,0.0,0.0,null,-8.736,-8.392,0.0,0.0,null,null],[4208.0,null,null,1962.834,942.314,null,136.307,111.782,0.0,0.0,null,1.061,0.981,1.133,null,0.0,0.0,0.0,null,-8.737,-2.851,0.0,0.0,null,null],[4208.5,null,null,1971.341,935.903,null,136.482,112.582,0.0,0.0,null,1.068,0.991,1.164,null,0.0,0.0,0.0,null,-8.667,-4.147,0.0,0.0,null,null],[4209.0,null,null,1976.205,923.94,null,136.268,112.923,0.0,0.0,null,1.082,1.003,1.204,null,0.0,0.0,0.0,null,-8.604,-6.739,0.0,0.0,null,null],[4209.5,null,null,1979.064,907.259,null,135.71,111.694,0.0,0.0,null,1.102,1.017,1.197,null,0.0,0.0,0.0,null,-8.635,-7.2,0.0,0.0,null,null],[4210.0,null,null,1978.248,888.982,null,135.08,108.762,0.0,0.0,null,1.125,1.037,1.163,null,0.0,0.0,0.0,null,-8.73,-6.398,0.0,0.0,null,null],[4210.5,null,null,1972.317,872.42,null,134.636,105.532,0.0,0.0,null,1.146,1.059,1.184,null,0.0,0.0,0.0,null,-8.846,-3.691,0.0,0.0,null,null],[4211.0,null,null,1964.494,859.678,null,134.412,104.526,0.0,0.0,null,1.163,1.078,1.229,null,0.0,0.0,0.0,null,-9.008,-0.545,0.0,0.0,null,null],[4211.5,null,null,1958.29,850.33,null,134.265,105.335,0.0,0.0,null,1.176,1.091,1.249,null,0.0,0.0,0.0,null,-9.218,-0.72,0.0,0.0,null,null],[4212.0,null,null,1955.802,843.016,null,134.075,105.887,0.0,0.0,null,1.186,1.101,1.254,null,0.0,0.0,0.0,null,-9.416,-2.039,0.0,0.0,null,null],[4212.5,null,null,1956.354,836.881,null,133.844,106.418,0.0,0.0,null,1.195,1.107,1.28,null,0.0,0.0,0.0,null,-9.57,-3.822,0.0,0.0,null,null],[4213.0,null,null,1954.968,832.472,null,133.648,107.518,0.0,0.0,null,1.201,1.112,1.303,null,0.0,0.0,0.0,null,-9.704,-7.387,0.0,0.0,null,null],[4213.5,null,null,1953.983,828.934,null,133.528,107.161,0.0,0.0,null,1.206,1.116,1.3,null,0.0,0.0,0.0,null,-9.812,-11.014,0.0,0.0,null,null],[4214.0,null,null,1958.335,824.359,null,133.484,105.7,0.0,0.0,null,1.213,1.121,1.28,null,0.0,0.0,0.0,null,-9.902,-9.966,0.0,0.0,null,null],[4214.5,null,null,1962.443,818.924,null,133.518,105.467,0.0,0.0,null,1.221,1.127,1.256,null,0.0,0.0,0.0,null,-10.024,-5.852,0.0,0.0,null,null],[4215.0,null,null,1962.634,815.203,null,133.601,106.486,0.0,0.0,null,1.227,1.134,1.26,null,0.0,0.0,0.0,null,-10.219,-6.077,0.0,0.0,null,null],[4215.5,null,null,1963.418,812.75,null,133.674,107.279,0.0,0.0,null,1.23,1.139,1.295,null,0.0,0.0,0.0,null,-10.43,-9.442,0.0,0.0,null,null],[4216.0,null,null,1965.495,808.691,null,133.703,108.595,0.0,0.0,null,1.237,1.145,1.342,null,0.0,0.0,0.0,null,-10.59,-10.194,0.0,0.0,null,null],[4216.5,null,null,1963.273,801.211,null,133.75,109.469,0.0,0.0,null,1.248,1.151,1.366,null,0.0,0.0,0.0,null,-10.714,-7.601,0.0,0.0,null,null],[4217.0,null,null,1956.944,791.884,null,133.958,107.984,0.0,0.0,null,1.263,1.162,1.352,null,0.0,0.0,0.0,null,-10.826,-4.725,0.0,0.0,null,null],[4217.5,null,null,1952.524,783.672,null,134.398,104.691,0.0,0.0,null,1.276,1.178,1.333,null,0.0,0.0,0.0,null,-10.869,-4.256,0.0,0.0,null,null],[4218.0,null,null,1952.444,779.493,null,134.94,102.926,0.0,0.0,null,1.283,1.196,1.332,null,0.0,0.0,0.0,null,-10.799,-6.301,0.0,0.0,null,null],[4218.5,null,null,1955.87,778.937,null,135.315,103.742,0.0,0.0,null,1.284,1.205,1.377,null,0.0,0.0,0.0,null,-10.691,-11.061,0.0,0.0,null,null],[4219.0,null,null,1958.92,779.361,null,135.343,104.839,0.0,0.0,null,1.283,1.202,1.444,null,0.0,0.0,0.0,null,-10.626,-16.691,0.0,0.0,null,null],[4219.5,null,null,1958.106,778.605,null,135.094,105.442,0.0,0.0,null,1.284,1.191,1.454,null,0.0,0.0,0.0,null,-10.596,-19.895,0.0,0.0,null,null],[4220.0,null,null,1956.544,777.73,null,134.845,107.42,0.0,0.0,null,1.286,1.18,1.383,null,0.0,0.0,0.0,null,-10.531,-19.834,0.0,0.0,null,null],[4220.5,null,null,1958.429,777.587,null,134.805,110.347,0.0,0.0,null,1.286,1.176,1.3,null,0.0,0.0,0.0,null,-10.366,-20.533,0.0,0.0,null,null],[4221.0,null,null,1960.26,778.884,null,134.9,110.835,0.0,0.0,null,1.284,1.177,1.267,null,0.0,0.0,0.0,null,-10.129,-21.342,0.0,0.0,null,null],[4221.5,null,null,1958.779,782.32,null,134.904,108.579,0.0,0.0,null,1.278,1.174,1.308,null,0.0,0.0,0.0,null,-9.939,-19.544,0.0,0.0,null,null],[4222.0,null,null,1955.667,787.144,null,134.741,107.28,0.0,0.0,null,1.27,1.164,1.402,null,0.0,0.0,0.0,null,-9.822,-17.942,0.0,0.0,null,null],[4222.5,null,null,1956.109,790.19,null,134.579,109.694,0.0,0.0,null,1.266,1.155,1.433,null,0.0,0.0,0.0,null,-9.68,-18.608,0.0,0.0,null,null],[4223.0,null,null,1959.804,790.061,null,134.665,112.568,0.0,0.0,null,1.266,1.149,1.384,null,0.0,0.0,0.0,null,-9.494,-20.397,0.0,0.0,null,null],[4223.5,null,null,1964.341,788.761,null,135.062,112.555,0.0,0.0,null,1.268,1.147,1.319,null,0.0,0.0,0.0,null,-9.373,-20.504,0.0,0.0,null,null],[4224.0,null,null,1969.095,789.287,null,135.581,111.653,0.0,0.0,null,1.267,1.15,1.286,null,0.0,0.0,0.0,null,-9.349,-17.015,0.0,0.0,null,null],[4224.5,null,null,1973.3,792.078,null,135.974,112.83,0.0,0.0,null,1.263,1.154,1.292,null,0.0,0.0,0.0,null,-9.359,-14.694,0.0,0.0,null,null],[4225.0,null,null,1973.535,796.088,null,136.133,113.916,0.0,0.0,null,1.256,1.155,1.319,null,0.0,0.0,0.0,null,-9.347,-18.856,0.0,0.0,null,null],[4225.5,null,null,1970.726,801.207,null,136.096,111.869,0.0,0.0,null,1.248,1.15,1.336,null,0.0,0.0,0.0,null,-9.31,-25.741,0.0,0.0,null,null],[4226.0,null,null,1968.384,806.316,null,135.908,108.733,0.0,0.0,null,1.24,1.142,1.338,null,0.0,0.0,0.0,null,-9.287,-29.019,0.0,0.0,null,null],[4226.5,null,null,1968.43,808.498,null,135.559,108.282,0.0,0.0,null,1.237,1.138,1.309,null,0.0,0.0,0.0,null,-9.338,-27.835,0.0,0.0,null,null],[4227.0,null,null,1971.347,806.29,null,135.021,109.977,0.0,0.0,null,1.24,1.14,1.278,null,0.0,0.0,0.0,null,-9.43,-24.513,0.0,0.0,null,null],[4227.5,null,null,1971.669,802.481,null,134.328,110.327,0.0,0.0,null,1.246,1.148,1.261,null,0.0,0.0,0.0,null,-9.493,-22.751,0.0,0.0,null,null],[4228.0,null,null,1966.532,800.598,null,133.586,109.747,0.0,0.0,null,1.249,1.159,1.271,null,0.0,0.0,0.0,null,-9.592,-23.393,0.0,0.0,null,null],[4228.5,null,null,1962.227,800.786,null,132.911,109.562,0.0,0.0,null,1.249,1.168,1.332,null,0.0,0.0,0.0,null,-9.819,-22.819,0.0,0.0,null,null],[4229.0,null,null,1965.737,799.85,null,132.379,109.365,0.0,0.0,null,1.25,1.173,1.39,null,0.0,0.0,0.0,null,-10.073,-22.482,0.0,0.0,null,null],[4229.5,null,null,1968.829,796.539,null,132.087,109.039,0.0,0.0,null,1.255,1.174,1.417,null,0.0,0.0,0.0,null,-10.17,-23.272,0.0,0.0,null,null],[4230.0,null,null,1963.138,791.967,null,132.2,108.996,0.0,0.0,null,1.263,1.173,1.404,null,0.0,0.0,0.0,null,-10.078,-22.276,0.0,0.0,null,null],[4230.5,null,null,1959.612,786.927,null,132.837,108.607,0.0,0.0,null,1.271,1.169,1.366,null,0.0,0.0,0.0,null,-9.87,-20.354,0.0,0.0,null,null],[4231.0,null,null,1969.745,785.393,null,133.886,108.665,0.0,0.0,null,1.273,1.166,1.328,null,0.0,0.0,0.0,null,-9.647,-21.452,0.0,0.0,null,null],[4231.5,null,null,1978.822,792.261,null,135.015,109.941,0.0,0.0,null,1.262,1.159,1.309,null,0.0,0.0,0.0,null,-9.469,-23.758,0.0,0.0,null,null],[4232.0,null,null,1977.27,804.786,null,135.908,110.989,0.0,0.0,null,1.243,1.147,1.331,null,0.0,0.0,0.0,null,-9.318,-21.544,0.0,0.0,null,null],[4232.5,null,null,1973.971,816.396,null,136.55,111.414,0.0,0.0,null,1.225,1.131,1.353,null,0.0,0.0,0.0,null,-9.169,-12.659,0.0,0.0,null,null],[4233.0,null,null,1971.406,827.714,null,137.167,111.714,0.0,0.0,null,1.208,1.113,1.347,null,0.0,0.0,0.0,null,-9.019,-3.096,0.0,0.0,null,null],[4233.5,null,null,1966.295,840.791,null,137.856,112.089,0.0,0.0,null,1.189,1.093,1.313,null,0.0,0.0,0.0,null,-8.864,0.594,0.0,0.0,null,null],[4234.0,null,null,1964.362,854.822,null,138.383,112.617,0.0,0.0,null,1.17,1.072,1.287,null,0.0,0.0,0.0,null,-8.674,1.23,0.0,0.0,null,null],[4234.5,null,null,1965.335,868.417,null,138.421,114.039,0.0,0.0,null,1.152,1.051,1.263,null,0.0,0.0,0.0,null,-8.47,0.99,0.0,0.0,null,null],[4235.0,null,null,1965.018,881.788,null,137.958,116.116,0.0,0.0,null,1.134,1.032,1.215,null,0.0,0.0,0.0,null,-8.311,-4.137,0.0,0.0,null,null],[4235.5,null,null,1965.611,894.643,null,137.38,118.441,0.0,0.0,null,1.118,1.019,1.148,null,0.0,0.0,0.0,null,-8.246,-13.468,0.0,0.0,null,null],[4236.0,null,null,1967.692,906.666,null,137.106,119.089,0.0,0.0,null,1.103,1.01,1.121,null,0.0,0.0,0.0,null,-8.264,-16.593,0.0,0.0,null,null],[4236.5,null,null,1966.9,915.09,null,137.149,116.836,0.0,0.0,null,1.093,1.004,1.143,null,0.0,0.0,0.0,null,-8.312,-10.19,0.0,0.0,null,null],[4237.0,null,null,1963.74,918.799,null,137.142,113.586,0.0,0.0,null,1.088,1.002,1.16,null,0.0,0.0,0.0,null,-8.387,-6.816,0.0,0.0,null,null],[4237.5,null,null,1962.434,917.372,null,136.801,112.67,0.0,0.0,null,1.09,1.007,1.168,null,0.0,0.0,0.0,null,-8.548,-13.891,0.0,0.0,null,null],[4238.0,null,null,1966.663,910.345,null,136.253,114.554,0.0,0.0,null,1.098,1.02,1.176,null,0.0,0.0,0.0,null,-8.766,-19.764,0.0,0.0,null,null],[4238.5,null,null,1974.881,899.072,null,135.894,117.208,0.0,0.0,null,1.112,1.037,1.198,null,0.0,0.0,0.0,null,-8.895,-17.308,0.0,0.0,null,null],[4239.0,null,null,1980.596,886.75,null,135.969,118.213,0.0,0.0,null,1.128,1.055,1.226,null,0.0,0.0,0.0,null,-8.924,-14.207,0.0,0.0,null,null],[4239.5,null,null,1980.188,874.469,null,136.358,117.798,0.0,0.0,null,1.144,1.071,1.252,null,0.0,0.0,0.0,null,-8.971,-13.034,0.0,0.0,null,null],[4240.0,null,null,1977.404,862.425,null,136.75,118.869,0.0,0.0,null,1.16,1.085,1.291,null,0.0,0.0,0.0,null,-9.084,-9.626,0.0,0.0,null,null],[4240.5,null,null,1977.662,850.926,null,136.967,121.462,0.0,0.0,null,1.175,1.096,1.306,null,0.0,0.0,0.0,null,-9.21,-5.434,0.0,0.0,null,null],[4241.0,null,null,1978.24,839.965,null,137.118,122.375,0.0,0.0,null,1.191,1.106,1.302,null,0.0,0.0,0.0,null,-9.301,-3.875,0.0,0.0,null,null],[4241.5,null,null,1973.754,829.683,null,137.394,121.2,0.0,0.0,null,1.205,1.117,1.291,null,0.0,0.0,0.0,null,-9.349,-2.148,0.0,0.0,null,null],[4242.0,null,null,1967.22,820.9,null,137.744,119.648,0.0,0.0,null,1.218,1.124,1.3,null,0.0,0.0,0.0,null,-9.36,1.929,0.0,0.0,null,null],[4242.5,null,null,1965.027,814.462,null,137.899,118.064,0.0,0.0,null,1.228,1.13,1.317,null,0.0,0.0,0.0,null,-9.331,3.439,0.0,0.0,null,null],[4243.0,null,null,1965.381,811.597,null,137.634,117.417,0.0,0.0,null,1.232,1.136,1.315,null,0.0,0.0,0.0,null,-9.264,1.261,0.0,0.0,null,null],[4243.5,null,null,1964.135,812.163,null,136.982,118.739,0.0,0.0,null,1.231,1.138,1.313,null,0.0,0.0,0.0,null,-9.207,-0.02,0.0,0.0,null,null],[4244.0,null,null,1960.93,814.28,null,136.196,119.92,0.0,0.0,null,1.228,1.138,1.322,null,0.0,0.0,0.0,null,-9.17,0.843,0.0,0.0,null,null],[4244.5,null,null,1958.778,813.803,null,135.508,119.507,0.0,0.0,null,1.229,1.136,1.337,null,0.0,0.0,0.0,null,-9.129,0.697,0.0,0.0,null,null],[4245.0,null,null,1961.767,811.134,null,134.937,119.432,0.0,0.0,null,1.233,1.135,1.308,null,0.0,0.0,0.0,null,-9.099,-2.71,0.0,0.0,null,null],[4245.5,null,null,1970.323,808.761,null,134.393,120.68,0.0,0.0,null,1.236,1.14,1.239,null,0.0,0.0,0.0,null,-9.071,-5.47,0.0,0.0,null,null],[4246.0,null,null,1979.748,804.485,null,133.908,120.783,0.0,0.0,null,1.243,1.152,1.186,null,0.0,0.0,0.0,null,-8.983,-3.379,0.0,0.0,null,null],[4246.5,null,null,1985.358,795.048,null,133.562,117.761,0.0,0.0,null,1.258,1.171,1.184,null,0.0,0.0,0.0,null,-8.916,-1.645,0.0,0.0,null,null],[4247.0,null,null,1984.281,782.609,null,133.313,114.186,0.0,0.0,null,1.278,1.196,1.259,null,0.0,0.0,0.0,null,-9.031,-8.165,0.0,0.0,null,null],[4247.5,null,null,1980.451,769.078,null,133.004,113.534,0.0,0.0,null,1.3,1.222,1.358,null,0.0,0.0,0.0,null,-9.255,-18.8,0.0,0.0,null,null],[4248.0,null,null,1975.321,754.689,null,132.494,115.836,0.0,0.0,null,1.325,1.248,1.427,null,0.0,0.0,0.0,null,-9.385,-22.299,0.0,0.0,null,null],[4248.5,null,null,1970.616,741.093,null,131.796,118.273,0.0,0.0,null,1.349,1.271,1.452,null,0.0,0.0,0.0,null,-9.405,-17.633,0.0,0.0,null,null],[4249.0,null,null,1971.914,731.055,null,131.068,119.388,0.0,0.0,null,1.368,1.284,1.489,null,0.0,0.0,0.0,null,-9.377,-13.968,0.0,0.0,null,null],[4249.5,null,null,1977.406,728.486,null,130.498,119.606,0.0,0.0,null,1.373,1.286,1.532,null,0.0,0.0,0.0,null,-9.313,-16.226,0.0,0.0,null,null],[4250.0,null,null,1977.269,735.593,null,130.205,119.793,0.0,0.0,null,1.359,1.27,1.538,null,0.0,0.0,0.0,null,-9.257,-17.226,0.0,0.0,null,null],[4250.5,null,null,1975.016,751.285,null,130.23,119.713,0.0,0.0,null,1.331,1.236,1.512,null,0.0,0.0,0.0,null,-9.238,-13.874,0.0,0.0,null,null],[4251.0,null,null,1978.541,774.173,null,130.56,118.68,0.0,0.0,null,1.292,1.19,1.466,null,0.0,0.0,0.0,null,-9.205,-11.334,0.0,0.0,null,null],[4251.5,null,null,1980.334,799.839,null,131.114,116.992,0.0,0.0,null,1.25,1.144,1.367,null,0.0,0.0,0.0,null,-9.095,-12.451,0.0,0.0,null,null],[4252.0,null,null,1976.752,822.005,null,131.785,114.124,0.0,0.0,null,1.217,1.108,1.246,null,0.0,0.0,0.0,null,-8.911,-15.065,0.0,0.0,null,null],[4252.5,null,null,1973.456,837.26,null,132.491,109.612,0.0,0.0,null,1.194,1.087,1.167,null,0.0,0.0,0.0,null,-8.73,-17.688,0.0,0.0,null,null],[4253.0,null,null,1971.907,846.267,null,133.122,106.189,0.0,0.0,null,1.182,1.082,1.124,null,0.0,0.0,0.0,null,-8.646,-21.328,0.0,0.0,null,null],[4253.5,null,null,1966.276,848.816,null,133.478,105.122,0.0,0.0,null,1.178,1.092,1.106,null,0.0,0.0,0.0,null,-8.672,-21.697,0.0,0.0,null,null],[4254.0,null,null,1960.873,844.777,null,133.357,103.607,0.0,0.0,null,1.184,1.112,1.117,null,0.0,0.0,0.0,null,-8.78,-16.806,0.0,0.0,null,null],[4254.5,null,null,1958.989,834.439,null,132.748,100.84,0.0,0.0,null,1.198,1.134,1.181,null,0.0,0.0,0.0,null,-8.987,-14.11,0.0,0.0,null,null],[4255.0,null,null,1957.504,819.519,null,131.882,99.56,0.0,0.0,null,1.22,1.156,1.322,null,0.0,0.0,0.0,null,-9.293,-17.79,0.0,0.0,null,null],[4255.5,null,null,1957.9,803.034,null,131.067,100.114,0.0,0.0,null,1.245,1.177,1.459,null,0.0,0.0,0.0,null,-9.579,-20.769,0.0,0.0,null,null],[4256.0,null,null,1962.103,788.621,null,130.456,100.906,0.0,0.0,null,1.268,1.196,1.495,null,0.0,0.0,0.0,null,-9.77,-19.483,0.0,0.0,null,null],[4256.5,null,null,1964.409,779.625,null,129.976,100.817,0.0,0.0,null,1.283,1.205,1.468,null,0.0,0.0,0.0,null,-9.876,-17.656,0.0,0.0,null,null],[4257.0,null,null,1962.273,777.793,null,129.543,99.254,0.0,0.0,null,1.286,1.201,1.469,null,0.0,0.0,0.0,null,-9.947,-16.735,0.0,0.0,null,null],[4257.5,null,null,1959.063,781.768,null,129.239,97.357,0.0,0.0,null,1.279,1.189,1.475,null,0.0,0.0,0.0,null,-10.051,-15.231,0.0,0.0,null,null],[4258.0,null,null,1956.932,789.239,null,129.219,96.91,0.0,0.0,null,1.267,1.174,1.431,null,0.0,0.0,0.0,null,-10.178,-11.782,0.0,0.0,null,null],[4258.5,null,null,1956.276,797.772,null,129.531,98.491,0.0,0.0,null,1.253,1.159,1.351,null,0.0,0.0,0.0,null,-10.219,-5.146,0.0,0.0,null,null],[4259.0,null,null,1955.48,805.661,null,130.071,101.069,0.0,0.0,null,1.241,1.15,1.282,null,0.0,0.0,0.0,null,-10.174,0.902,0.0,0.0,null,null],[4259.5,null,null,1956.423,812.302,null,130.693,103.191,0.0,0.0,null,1.231,1.143,1.253,null,0.0,0.0,0.0,null,-10.146,0.538,0.0,0.0,null,null],[4260.0,null,null,1960.469,817.709,null,131.325,104.368,0.0,0.0,null,1.223,1.14,1.262,null,0.0,0.0,0.0,null,-10.164,-9.06,0.0,0.0,null,null],[4260.5,null,null,1965.334,820.334,null,131.92,105.117,0.0,0.0,null,1.219,1.14,1.26,null,0.0,0.0,0.0,null,-10.198,-19.685,0.0,0.0,null,null],[4261.0,null,null,1967.024,819.587,null,132.426,105.603,0.0,0.0,null,1.22,1.144,1.266,null,0.0,0.0,0.0,null,-10.192,-20.546,0.0,0.0,null,null],[4261.5,null,null,1971.451,814.111,null,132.83,106.016,0.0,0.0,null,1.228,1.149,1.321,null,0.0,0.0,0.0,null,-10.149,-15.585,0.0,0.0,null,null],[4262.0,null,null,1982.09,803.776,null,133.202,107.125,0.0,0.0,null,1.244,1.156,1.358,null,0.0,0.0,0.0,null,-10.244,-16.516,0.0,0.0,null,null],[4262.5,null,null,1988.548,791.782,null,133.655,109.358,0.0,0.0,null,1.263,1.169,1.339,null,0.0,0.0,0.0,null,-10.519,-21.024,0.0,0.0,null,null],[4263.0,null,null,1981.579,781.986,null,134.212,111.669,0.0,0.0,null,1.279,1.187,1.316,null,0.0,0.0,0.0,null,-10.794,-19.269,0.0,0.0,null,null],[4263.5,null,null,1972.397,774.007,null,134.715,112.764,0.0,0.0,null,1.292,1.2,1.349,null,0.0,0.0,0.0,null,-10.954,-9.804,0.0,0.0,null,null],[4264.0,null,null,1970.667,767.833,null,134.925,111.966,0.0,0.0,null,1.302,1.203,1.424,null,0.0,0.0,0.0,null,-11.053,0.541,0.0,0.0,null,null],[4264.5,null,null,1968.585,763.613,null,134.792,109.494,0.0,0.0,null,1.31,1.202,1.475,null,0.0,0.0,0.0,null,-11.118,2.303,0.0,0.0,null,null],[4265.0,null,null,1962.296,760.069,null,134.484,106.999,0.0,0.0,null,1.316,1.203,1.457,null,0.0,0.0,0.0,null,-11.179,-5.179,0.0,0.0,null,null],[4265.5,null,null,1959.542,755.939,null,134.183,105.944,0.0,0.0,null,1.323,1.208,1.386,null,0.0,0.0,0.0,null,-11.282,-10.617,0.0,0.0,null,null],[4266.0,null,null,1963.677,753.763,null,133.935,106.641,0.0,0.0,null,1.327,1.21,1.345,null,0.0,0.0,0.0,null,-11.401,-6.612,0.0,0.0,null,null],[4266.5,null,null,1966.168,753.958,null,133.616,107.874,0.0,0.0,null,1.326,1.208,1.329,null,0.0,0.0,0.0,null,-11.5,-1.164,0.0,0.0,null,null],[4267.0,null,null,1962.666,752.119,null,133.103,108.183,0.0,0.0,null,1.33,1.212,1.321,null,0.0,0.0,0.0,null,-11.599,-1.514,0.0,0.0,null,null],[4267.5,null,null,1959.155,744.973,null,132.415,107.378,0.0,0.0,null,1.342,1.23,1.313,null,0.0,0.0,0.0,null,-11.72,-2.054,0.0,0.0,null,null],[4268.0,null,null,1964.594,732.932,null,131.688,106.637,0.0,0.0,null,1.364,1.265,1.274,null,0.0,0.0,0.0,null,-11.844,0.306,0.0,0.0,null,null],[4268.5,null,null,1975.748,717.607,null,130.989,106.085,0.0,0.0,null,1.394,1.31,1.267,null,0.0,0.0,0.0,null,-12.027,-2.348,0.0,0.0,null,null],[4269.0,null,null,1981.303,701.779,null,130.118,105.557,0.0,0.0,null,1.425,1.36,1.376,null,0.0,0.0,0.0,null,-12.428,-8.11,0.0,0.0,null,null],[4269.5,null,null,1980.925,684.985,null,128.728,105.588,0.0,0.0,null,1.46,1.409,1.54,null,0.0,0.0,0.0,null,-13.006,-7.501,0.0,0.0,null,null],[4270.0,null,null,1981.596,665.482,null,126.672,105.931,0.0,0.0,null,1.503,1.46,1.612,null,0.0,0.0,0.0,null,-13.466,-3.213,0.0,0.0,null,null],[4270.5,null,null,1982.328,646.681,null,124.287,104.796,0.0,0.0,null,1.546,1.51,1.589,null,0.0,0.0,0.0,null,-13.687,-2.386,0.0,0.0,null,null],[4271.0,null,null,1981.382,635.335,null,122.186,100.578,0.0,0.0,null,1.574,1.546,1.647,null,0.0,0.0,0.0,null,-13.937,-1.786,0.0,0.0,null,null],[4271.5,null,null,1981.917,631.922,null,120.805,94.022,0.0,0.0,null,1.582,1.552,1.848,null,0.0,0.0,0.0,null,-14.516,0.033,0.0,0.0,null,null],[4272.0,null,null,1980.938,633.248,null,120.197,88.689,0.0,0.0,null,1.579,1.532,2.109,null,0.0,0.0,0.0,null,-15.334,-2.755,0.0,0.0,null,null],[4272.5,null,null,1980.215,638.477,null,120.206,86.092,0.0,0.0,null,1.566,1.497,2.212,null,0.0,0.0,0.0,null,-15.839,-10.485,0.0,0.0,null,null],[4273.0,null,null,1979.94,647.639,null,120.803,85.555,0.0,0.0,null,1.544,1.462,1.998,null,0.0,0.0,0.0,null,-15.507,-17.276,0.0,0.0,null,null],[4273.5,null,null,1973.766,658.9,null,122.051,85.951,0.0,0.0,null,1.518,1.426,1.734,null,0.0,0.0,0.0,null,-14.491,-18.919,0.0,0.0,null,null],[4274.0,null,null,1966.614,670.101,null,123.798,86.842,0.0,0.0,null,1.492,1.39,1.588,null,0.0,0.0,0.0,null,-13.479,-16.538,0.0,0.0,null,null],[4274.5,null,null,1968.107,681.164,null,125.589,90.325,0.0,0.0,null,1.468,1.361,1.497,null,0.0,0.0,0.0,null,-12.856,-13.988,0.0,0.0,null,null],[4275.0,null,null,1972.311,691.144,null,126.895,96.575,0.0,0.0,null,1.447,1.345,1.395,null,0.0,0.0,0.0,null,-12.51,-14.746,0.0,0.0,null,null],[4275.5,null,null,1974.055,699.466,null,127.475,100.505,0.0,0.0,null,1.43,1.335,1.336,null,0.0,0.0,0.0,null,-12.245,-16.146,0.0,0.0,null,null],[4276.0,null,null,1977.558,708.44,null,127.496,100.609,0.0,0.0,null,1.412,1.324,1.354,null,0.0,0.0,0.0,null,-12.013,-16.019,0.0,0.0,null,null],[4276.5,null,null,1980.345,717.559,null,127.328,100.679,0.0,0.0,null,1.394,1.31,1.394,null,0.0,0.0,0.0,null,-11.793,-16.915,0.0,0.0,null,null],[4277.0,null,null,1979.459,724.538,null,127.234,101.338,0.0,0.0,null,1.38,1.292,1.429,null,0.0,0.0,0.0,null,-11.639,-19.202,0.0,0.0,null,null],[4277.5,null,null,1976.498,730.597,null,127.28,101.337,0.0,0.0,null,1.369,1.273,1.445,null,0.0,0.0,0.0,null,-11.527,-19.709,0.0,0.0,null,null],[4278.0,null,null,1971.914,738.117,null,127.423,101.495,0.0,0.0,null,1.355,1.254,1.429,null,0.0,0.0,0.0,null,-11.371,-20.658,0.0,0.0,null,null],[4278.5,null,null,1967.525,743.498,null,127.601,101.77,0.0,0.0,null,1.345,1.241,1.389,null,0.0,0.0,0.0,null,-11.218,-24.974,0.0,0.0,null,null],[4279.0,null,null,1966.21,744.399,null,127.807,101.684,0.0,0.0,null,1.343,1.235,1.344,null,0.0,0.0,0.0,null,-11.175,-28.523,0.0,0.0,null,null],[4279.5,null,null,1967.158,742.872,null,128.031,102.07,0.0,0.0,null,1.346,1.235,1.337,null,0.0,0.0,0.0,null,-11.201,-25.932,0.0,0.0,null,null],[4280.0,null,null,1970.806,739.781,null,128.128,102.66,0.0,0.0,null,1.352,1.244,1.354,null,0.0,0.0,0.0,null,-11.228,-18.569,0.0,0.0,null,null],[4280.5,null,null,1978.095,732.394,null,127.871,103.52,0.0,0.0,null,1.365,1.268,1.367,null,0.0,0.0,0.0,null,-11.171,-13.383,0.0,0.0,null,null],[4281.0,null,null,1984.958,721.962,null,127.165,104.666,0.0,0.0,null,1.385,1.3,1.365,null,0.0,0.0,0.0,null,-10.999,-11.817,0.0,0.0,null,null],[4281.5,null,null,1989.425,712.929,null,126.2,105.013,0.0,0.0,null,1.403,1.327,1.403,null,0.0,0.0,0.0,null,-10.935,-11.003,0.0,0.0,null,null],[4282.0,null,null,1990.439,707.825,null,125.331,103.438,0.0,0.0,null,1.413,1.34,1.572,null,0.0,0.0,0.0,null,-11.115,-8.995,0.0,0.0,null,null],[4282.5,null,null,1987.071,706.122,null,124.787,101.836,0.0,0.0,null,1.416,1.335,1.754,null,0.0,0.0,0.0,null,-11.267,-8.721,0.0,0.0,null,null],[4283.0,null,null,1983.167,706.691,null,124.572,103.321,0.0,0.0,null,1.415,1.317,1.739,null,0.0,0.0,0.0,null,-11.078,-13.083,0.0,0.0,null,null],[4283.5,null,null,1983.485,709.604,null,124.617,108.486,0.0,0.0,null,1.409,1.297,1.547,null,0.0,0.0,0.0,null,-10.594,-19.383,0.0,0.0,null,null],[4284.0,null,null,1983.628,719.568,null,124.942,114.688,0.0,0.0,null,1.39,1.274,1.42,null,0.0,0.0,0.0,null,-10.052,-22.499,0.0,0.0,null,null],[4284.5,null,null,1981.956,739.34,null,125.594,118.631,0.0,0.0,null,1.353,1.241,1.418,null,0.0,0.0,0.0,null,-9.654,-21.801,0.0,0.0,null,null],[4285.0,null,null,1982.127,763.644,null,126.5,119.401,0.0,0.0,null,1.31,1.201,1.431,null,0.0,0.0,0.0,null,-9.406,-18.166,0.0,0.0,null,null],[4285.5,null,null,1981.141,785.195,null,127.429,118.899,0.0,0.0,null,1.274,1.167,1.381,null,0.0,0.0,0.0,null,-9.155,-13.63,0.0,0.0,null,null],[4286.0,null,null,1973.048,802.075,null,128.139,119.144,0.0,0.0,null,1.247,1.145,1.279,null,0.0,0.0,0.0,null,-8.824,-12.898,0.0,0.0,null,null],[4286.5,null,null,1962.016,814.847,null,128.578,119.409,0.0,0.0,null,1.227,1.132,1.223,null,0.0,0.0,0.0,null,-8.502,-13.532,0.0,0.0,null,null],[4287.0,null,null,1955.727,824.531,null,128.843,118.392,0.0,0.0,null,1.213,1.122,1.23,null,0.0,0.0,0.0,null,-8.306,-11.416,0.0,0.0,null,null],[4287.5,null,null,1954.54,832.456,null,129.02,117.064,0.0,0.0,null,1.201,1.111,1.271,null,0.0,0.0,0.0,null,-8.225,-8.416,0.0,0.0,null,null],[4288.0,null,null,1957.034,839.839,null,129.151,117.157,0.0,0.0,null,1.191,1.1,1.316,null,0.0,0.0,0.0,null,-8.196,-5.274,0.0,0.0,null,null],[4288.5,null,null,1959.651,844.416,null,129.251,117.992,0.0,0.0,null,1.184,1.095,1.303,null,0.0,0.0,0.0,null,-8.16,-1.934,0.0,0.0,null,null],[4289.0,null,null,1957.291,844.896,null,129.321,118.603,0.0,0.0,null,1.184,1.095,1.267,null,0.0,0.0,0.0,null,-8.104,-3.575,0.0,0.0,null,null],[4289.5,null,null,1953.279,844.202,null,129.357,118.288,0.0,0.0,null,1.185,1.097,1.248,null,0.0,0.0,0.0,null,-8.069,-11.255,0.0,0.0,null,null],[4290.0,null,null,1953.803,845.722,null,129.36,117.406,0.0,0.0,null,1.182,1.097,1.265,null,0.0,0.0,0.0,null,-8.075,-17.058,0.0,0.0,null,null],[4290.5,null,null,1956.629,847.772,null,129.315,115.895,0.0,0.0,null,1.18,1.093,1.295,null,0.0,0.0,0.0,null,-8.092,-16.989,0.0,0.0,null,null],[4291.0,null,null,1957.594,849.732,null,129.227,114.418,0.0,0.0,null,1.177,1.089,1.296,null,0.0,0.0,0.0,null,-8.126,-15.882,0.0,0.0,null,null],[4291.5,null,null,1958.649,851.862,null,129.157,114.644,0.0,0.0,null,1.174,1.087,1.282,null,0.0,0.0,0.0,null,-8.245,-18.323,0.0,0.0,null,null],[4292.0,null,null,1959.321,853.868,null,129.137,115.548,0.0,0.0,null,1.171,1.086,1.287,null,0.0,0.0,0.0,null,-8.437,-22.618,0.0,0.0,null,null],[4292.5,null,null,1957.379,855.49,null,129.174,115.269,0.0,0.0,null,1.169,1.086,1.301,null,0.0,0.0,0.0,null,-8.615,-24.228,0.0,0.0,null,null],[4293.0,null,null,1956.311,857.686,null,129.28,115.826,0.0,0.0,null,1.166,1.086,1.301,null,0.0,0.0,0.0,null,-8.737,-22.761,0.0,0.0,null,null],[4293.5,null,null,1960.245,860.322,null,129.446,118.233,0.0,0.0,null,1.162,1.084,1.292,null,0.0,0.0,0.0,null,-8.814,-21.208,0.0,0.0,null,null],[4294.0,null,null,1965.179,863.61,null,129.69,119.071,0.0,0.0,null,1.158,1.08,1.299,null,0.0,0.0,0.0,null,-8.868,-18.562,0.0,0.0,null,null],[4294.5,null,null,1966.088,868.071,null,130.047,117.854,0.0,0.0,null,1.152,1.076,1.303,null,0.0,0.0,0.0,null,-8.888,-15.313,0.0,0.0,null,null],[4295.0,null,null,1964.587,872.418,null,130.485,117.396,0.0,0.0,null,1.146,1.071,1.299,null,0.0,0.0,0.0,null,-8.855,-16.29,0.0,0.0,null,null],[4295.5,null,null,1964.644,874.529,null,130.891,118.015,0.0,0.0,null,1.143,1.066,1.29,null,0.0,0.0,0.0,null,-8.794,-17.16,0.0,0.0,null,null],[4296.0,null,null,1963.839,875.659,null,131.225,118.232,0.0,0.0,null,1.142,1.06,1.273,null,0.0,0.0,0.0,null,-8.785,-12.538,0.0,0.0,null,null],[4296.5,null,null,1961.961,878.08,null,131.583,118.209,0.0,0.0,null,1.139,1.055,1.27,null,0.0,0.0,0.0,null,-8.847,-8.524,0.0,0.0,null,null],[4297.0,null,null,1966.141,882.419,null,132.111,118.045,0.0,0.0,null,1.133,1.052,1.277,null,0.0,0.0,0.0,null,-8.905,-10.121,0.0,0.0,null,null],[4297.5,null,null,1974.941,887.208,null,132.863,116.897,0.0,0.0,null,1.127,1.049,1.276,null,0.0,0.0,0.0,null,-8.915,-11.109,0.0,0.0,null,null],[4298.0,null,null,1976.988,891.705,null,133.659,115.656,0.0,0.0,null,1.121,1.044,1.268,null,0.0,0.0,0.0,null,-8.91,-8.787,0.0,0.0,null,null],[4298.5,null,null,1971.358,896.463,null,134.245,115.978,0.0,0.0,null,1.115,1.04,1.275,null,0.0,0.0,0.0,null,-8.928,-7.187,0.0,0.0,null,null],[4299.0,null,null,1968.354,901.072,null,134.536,116.369,0.0,0.0,null,1.11,1.037,1.293,null,0.0,0.0,0.0,null,-8.981,-7.13,0.0,0.0,null,null],[4299.5,null,null,1973.679,904.396,null,134.626,115.727,0.0,0.0,null,1.106,1.032,1.284,null,0.0,0.0,0.0,null,-9.048,-7.914,0.0,0.0,null,null],[4300.0,null,null,1984.187,908.039,null,134.627,115.615,0.0,0.0,null,1.101,1.025,1.255,null,0.0,0.0,0.0,null,-9.072,-9.314,0.0,0.0,null,null],[4300.5,null,null,1993.312,913.313,null,134.591,116.384,0.0,0.0,null,1.095,1.017,1.243,null,0.0,0.0,0.0,null,-9.031,-9.665,0.0,0.0,null,null],[4301.0,null,null,1994.731,919.111,null,134.556,116.944,0.0,0.0,null,1.088,1.009,1.25,null,0.0,0.0,0.0,null,-8.956,-9.642,0.0,0.0,null,null],[4301.5,null,null,1986.445,925.141,null,134.575,116.327,0.0,0.0,null,1.081,1.001,1.252,null,0.0,0.0,0.0,null,-8.866,-13.822,0.0,0.0,null,null],[4302.0,null,null,1974.095,931.609,null,134.702,114.857,0.0,0.0,null,1.073,0.994,1.228,null,0.0,0.0,0.0,null,-8.741,-19.976,0.0,0.0,null,null],[4302.5,null,null,1969.904,936.862,null,134.892,114.297,0.0,0.0,null,1.067,0.991,1.216,null,0.0,0.0,0.0,null,-8.605,-19.434,0.0,0.0,null,null],[4303.0,null,null,1975.692,940.691,null,135.059,114.975,0.0,0.0,null,1.063,0.99,1.215,null,0.0,0.0,0.0,null,-8.517,-11.493,0.0,0.0,null,null],[4303.5,null,null,1979.296,943.043,null,135.225,114.667,0.0,0.0,null,1.06,0.991,1.219,null,0.0,0.0,0.0,null,-8.508,-5.101,0.0,0.0,null,null],[4304.0,null,null,1973.28,942.016,null,135.478,112.853,0.0,0.0,null,1.062,0.993,1.219,null,0.0,0.0,0.0,null,-8.522,-2.505,0.0,0.0,null,null],[4304.5,null,null,1964.474,936.98,null,135.768,111.342,0.0,0.0,null,1.067,0.997,1.227,null,0.0,0.0,0.0,null,-8.49,-1.914,0.0,0.0,null,null],[4305.0,null,null,1961.219,930.974,null,135.924,111.293,0.0,0.0,null,1.074,1.0,1.244,null,0.0,0.0,0.0,null,-8.423,-5.124,0.0,0.0,null,null],[4305.5,null,null,1961.096,927.314,null,135.872,111.869,0.0,0.0,null,1.078,1.001,1.26,null,0.0,0.0,0.0,null,-8.378,-11.279,0.0,0.0,null,null],[4306.0,null,null,1960.665,927.407,null,135.759,111.698,0.0,0.0,null,1.078,1.002,1.256,null,0.0,0.0,0.0,null,-8.381,-17.056,0.0,0.0,null,null],[4306.5,null,null,1962.071,929.48,null,135.862,110.581,0.0,0.0,null,1.076,1.0,1.234,null,0.0,0.0,0.0,null,-8.417,-20.583,0.0,0.0,null,null],[4307.0,null,null,1966.035,931.767,null,136.311,108.808,0.0,0.0,null,1.073,0.997,1.227,null,0.0,0.0,0.0,null,-8.482,-20.515,0.0,0.0,null,null],[4307.5,null,null,1969.465,934.445,null,136.919,107.258,0.0,0.0,null,1.07,0.993,1.224,null,0.0,0.0,0.0,null,-8.541,-19.029,0.0,0.0,null,null],[4308.0,null,null,1973.223,940.231,null,137.38,108.259,0.0,0.0,null,1.064,0.989,1.206,null,0.0,0.0,0.0,null,-8.561,-20.469,0.0,0.0,null,null],[4308.5,null,null,1974.43,949.26,null,137.546,110.734,0.0,0.0,null,1.053,0.986,1.174,null,0.0,0.0,0.0,null,-8.586,-21.058,0.0,0.0,null,null],[4309.0,null,null,1972.983,959.299,null,137.439,110.823,0.0,0.0,null,1.042,0.979,1.163,null,0.0,0.0,0.0,null,-8.66,-18.644,0.0,0.0,null,null],[4309.5,null,null,1976.595,967.106,null,137.202,108.634,0.0,0.0,null,1.034,0.97,1.186,null,0.0,0.0,0.0,null,-8.777,-17.859,0.0,0.0,null,null],[4310.0,null,null,1984.548,969.655,null,136.979,107.336,0.0,0.0,null,1.031,0.963,1.189,null,0.0,0.0,0.0,null,-8.936,-18.03,0.0,0.0,null,null],[4310.5,null,null,1984.971,963.369,null,136.773,107.86,0.0,0.0,null,1.038,0.964,1.167,null,0.0,0.0,0.0,null,-9.129,-15.303,0.0,0.0,null,null],[4311.0,null,null,1982.234,947.761,null,136.474,109.338,0.0,0.0,null,1.055,0.979,1.125,null,0.0,0.0,0.0,null,-9.288,-14.14,0.0,0.0,null,null],[4311.5,null,null,1989.822,923.754,null,135.97,111.24,0.0,0.0,null,1.083,1.009,1.082,null,0.0,0.0,0.0,null,-9.313,-19.57,0.0,0.0,null,null],[4312.0,null,null,2002.499,892.151,null,135.233,111.142,0.0,0.0,null,1.121,1.053,1.094,null,0.0,0.0,0.0,null,-9.355,-27.158,0.0,0.0,null,null],[4312.5,null,null,2008.28,856.46,null,134.335,108.623,0.0,0.0,null,1.168,1.111,1.192,null,0.0,0.0,0.0,null,-9.725,-30.567,0.0,0.0,null,null],[4313.0,null,null,2009.727,821.939,null,133.358,105.692,0.0,0.0,null,1.217,1.176,1.349,null,0.0,0.0,0.0,null,-10.372,-29.249,0.0,0.0,null,null],[4313.5,null,null,2009.617,789.777,null,132.213,104.388,0.0,0.0,null,1.266,1.239,1.526,null,0.0,0.0,0.0,null,-10.899,-25.517,0.0,0.0,null,null],[4314.0,null,null,2002.25,758.342,null,130.745,105.513,0.0,0.0,null,1.319,1.286,1.71,null,0.0,0.0,0.0,null,-11.094,-23.649,0.0,0.0,null,null],[4314.5,null,null,1990.099,730.235,null,129.05,109.57,0.0,0.0,null,1.369,1.317,1.785,null,0.0,0.0,0.0,null,-10.986,-24.519,0.0,0.0,null,null],[4315.0,null,null,1981.806,711.691,null,127.56,114.973,0.0,0.0,null,1.405,1.334,1.723,null,0.0,0.0,0.0,null,-10.646,-24.53,0.0,0.0,null,null],[4315.5,null,null,1981.64,707.363,null,126.818,118.278,0.0,0.0,null,1.414,1.332,1.634,null,0.0,0.0,0.0,null,-10.239,-22.386,0.0,0.0,null,null],[4316.0,null,null,1990.334,718.012,null,127.075,118.977,0.0,0.0,null,1.393,1.306,1.564,null,0.0,0.0,0.0,null,-9.901,-22.434,0.0,0.0,null,null],[4316.5,null,null,2000.371,738.955,null,128.1,118.704,0.0,0.0,null,1.353,1.259,1.54,null,0.0,0.0,0.0,null,-9.615,-26.381,0.0,0.0,null,null],[4317.0,null,null,2000.939,762.88,null,129.467,117.261,0.0,0.0,null,1.311,1.207,1.514,null,0.0,0.0,0.0,null,-9.229,-29.315,0.0,0.0,null,null],[4317.5,null,null,1996.928,784.428,null,130.956,114.367,0.0,0.0,null,1.275,1.169,1.386,null,0.0,0.0,0.0,null,-8.611,-25.329,0.0,0.0,null,null],[4318.0,null,null,1995.258,804.042,null,132.589,113.106,0.0,0.0,null,1.244,1.143,1.233,null,0.0,0.0,0.0,null,-7.912,-19.2,0.0,0.0,null,null],[4318.5,null,null,1992.503,823.216,null,134.289,114.414,0.0,0.0,null,1.215,1.12,1.159,null,0.0,0.0,0.0,null,-7.404,-14.957,0.0,0.0,null,null],[4319.0,null,null,1989.201,841.768,null,135.717,115.064,0.0,0.0,null,1.188,1.094,1.206,null,0.0,0.0,0.0,null,-7.145,-11.62,0.0,0.0,null,null],[4319.5,null,null,1991.392,858.786,null,136.464,113.712,0.0,0.0,null,1.164,1.068,1.277,null,0.0,0.0,0.0,null,-7.051,-9.502,0.0,0.0,null,null],[4320.0,null,null,1997.552,874.921,null,136.395,112.448,0.0,0.0,null,1.143,1.046,1.277,null,0.0,0.0,0.0,null,-7.06,-11.296,0.0,0.0,null,null],[4320.5,null,null,2003.206,890.056,null,135.766,112.376,0.0,0.0,null,1.124,1.029,1.21,null,0.0,0.0,0.0,null,-7.079,-13.431,0.0,0.0,null,null],[4321.0,null,null,2006.541,904.395,null,134.979,112.042,0.0,0.0,null,1.106,1.017,1.134,null,0.0,0.0,0.0,null,-7.006,-14.76,0.0,0.0,null,null],[4321.5,null,null,2003.352,917.879,null,134.307,111.059,0.0,0.0,null,1.089,1.005,1.129,null,0.0,0.0,0.0,null,-6.873,-14.174,0.0,0.0,null,null],[4322.0,null,null,1994.708,930.32,null,133.881,110.696,0.0,0.0,null,1.075,0.993,1.157,null,0.0,0.0,0.0,null,-6.876,-11.629,0.0,0.0,null,null],[4322.5,null,null,1987.794,940.47,null,133.809,111.281,0.0,0.0,null,1.063,0.983,1.156,null,0.0,0.0,0.0,null,-7.067,-11.487,0.0,0.0,null,null],[4323.0,null,null,1983.39,946.94,null,134.084,111.6,0.0,0.0,null,1.056,0.976,1.125,null,0.0,0.0,0.0,null,-7.242,-16.559,0.0,0.0,null,null],[4323.5,null,null,1973.86,949.051,null,134.512,111.499,0.0,0.0,null,1.054,0.975,1.091,null,0.0,0.0,0.0,null,-7.271,-19.744,0.0,0.0,null,null],[4324.0,null,null,1960.724,947.868,null,134.853,111.287,0.0,0.0,null,1.055,0.979,1.101,null,0.0,0.0,0.0,null,-7.278,-17.092,0.0,0.0,null,null],[4324.5,null,null,1953.763,945.494,null,135.003,111.147,0.0,0.0,null,1.058,0.986,1.123,null,0.0,0.0,0.0,null,-7.326,-16.931,0.0,0.0,null,null],[4325.0,null,null,1954.626,944.832,null,135.017,111.36,0.0,0.0,null,1.058,0.992,1.132,null,0.0,0.0,0.0,null,-7.385,-22.351,0.0,0.0,null,null],[4325.5,null,null,1956.659,946.741,null,134.919,111.448,0.0,0.0,null,1.056,0.995,1.139,null,0.0,0.0,0.0,null,-7.428,-21.46,0.0,0.0,null,null],[4326.0,null,null,1958.781,948.495,null,134.596,111.255,0.0,0.0,null,1.054,0.992,1.174,null,0.0,0.0,0.0,null,-7.47,-11.427,0.0,0.0,null,null],[4326.5,null,null,1964.266,947.229,null,134.04,111.419,0.0,0.0,null,1.056,0.988,1.203,null,0.0,0.0,0.0,null,-7.508,-6.603,0.0,0.0,null,null],[4327.0,null,null,1973.669,941.634,null,133.476,113.468,0.0,0.0,null,1.062,0.987,1.17,null,0.0,0.0,0.0,null,-7.539,-8.381,0.0,0.0,null,null],[4327.5,null,null,1983.181,933.391,null,133.134,117.125,0.0,0.0,null,1.071,0.993,1.103,null,0.0,0.0,0.0,null,-7.595,-9.347,0.0,0.0,null,null],[4328.0,null,null,1990.418,924.335,null,133.009,118.582,0.0,0.0,null,1.082,1.006,1.06,null,0.0,0.0,0.0,null,-7.698,-11.717,0.0,0.0,null,null],[4328.5,null,null,1992.892,912.49,null,132.916,115.728,0.0,0.0,null,1.096,1.025,1.088,null,0.0,0.0,0.0,null,-7.825,-15.946,0.0,0.0,null,null],[4329.0,null,null,1988.7,897.075,null,132.739,111.821,0.0,0.0,null,1.115,1.047,1.149,null,0.0,0.0,0.0,null,-7.969,-16.179,0.0,0.0,null,null],[4329.5,null,null,1981.286,881.089,null,132.587,109.128,0.0,0.0,null,1.135,1.07,1.194,null,0.0,0.0,0.0,null,-8.116,-14.907,0.0,0.0,null,null],[4330.0,null,null,1975.544,865.285,null,132.595,106.548,0.0,0.0,null,1.156,1.09,1.237,null,0.0,0.0,0.0,null,-8.231,-14.848,0.0,0.0,null,null],[4330.5,null,null,1972.318,849.181,null,132.642,105.325,0.0,0.0,null,1.178,1.105,1.307,null,0.0,0.0,0.0,null,-8.308,-13.204,0.0,0.0,null,null],[4331.0,null,null,1970.116,835.515,null,132.522,107.424,0.0,0.0,null,1.197,1.116,1.35,null,0.0,0.0,0.0,null,-8.379,-11.66,0.0,0.0,null,null],[4331.5,null,null,1971.945,825.94,null,132.174,110.42,0.0,0.0,null,1.211,1.124,1.323,null,0.0,0.0,0.0,null,-8.465,-13.804,0.0,0.0,null,null],[4332.0,null,null,1976.11,820.746,null,131.659,112.61,0.0,0.0,null,1.218,1.129,1.328,null,0.0,0.0,0.0,null,-8.598,-16.976,0.0,0.0,null,null],[4332.5,null,null,1976.271,818.683,null,131.074,114.637,0.0,0.0,null,1.221,1.133,1.346,null,0.0,0.0,0.0,null,-8.72,-17.024,0.0,0.0,null,null],[4333.0,null,null,1972.868,815.609,null,130.447,117.937,0.0,0.0,null,1.226,1.138,1.319,null,0.0,0.0,0.0,null,-8.725,-11.839,0.0,0.0,null,null],[4333.5,null,null,1972.75,810.007,null,129.757,122.548,0.0,0.0,null,1.235,1.146,1.248,null,0.0,0.0,0.0,null,-8.582,-4.566,0.0,0.0,null,null],[4334.0,null,null,1976.6,804.568,null,129.065,126.015,0.0,0.0,null,1.243,1.158,1.214,null,0.0,0.0,0.0,null,-8.409,-2.147,0.0,0.0,null,null],[4334.5,null,null,1980.731,800.067,null,128.483,124.949,0.0,0.0,null,1.25,1.17,1.219,null,0.0,0.0,0.0,null,-8.306,-5.81,0.0,0.0,null,null],[4335.0,null,null,1981.271,795.231,null,128.028,121.405,0.0,0.0,null,1.257,1.18,1.253,null,0.0,0.0,0.0,null,-8.292,-8.463,0.0,0.0,null,null],[4335.5,null,null,1978.249,790.666,null,127.714,119.243,0.0,0.0,null,1.265,1.19,1.295,null,0.0,0.0,0.0,null,-8.318,-6.577,0.0,0.0,null,null],[4336.0,null,null,1977.128,783.905,null,127.69,117.518,0.0,0.0,null,1.276,1.198,1.349,null,0.0,0.0,0.0,null,-8.331,-6.669,0.0,0.0,null,null],[4336.5,null,null,1979.645,772.434,null,128.102,115.122,0.0,0.0,null,1.295,1.209,1.401,null,0.0,0.0,0.0,null,-8.329,-12.763,0.0,0.0,null,null],[4337.0,null,null,1979.847,759.463,null,128.958,114.09,0.0,0.0,null,1.317,1.221,1.393,null,0.0,0.0,0.0,null,-8.357,-18.697,0.0,0.0,null,null],[4337.5,null,null,1977.893,750.535,null,130.058,115.158,0.0,0.0,null,1.332,1.232,1.356,null,0.0,0.0,0.0,null,-8.406,-19.448,0.0,0.0,null,null],[4338.0,null,null,1978.059,747.181,null,131.067,116.092,0.0,0.0,null,1.338,1.237,1.349,null,0.0,0.0,0.0,null,-8.419,-17.923,0.0,0.0,null,null],[4338.5,null,null,1976.219,748.557,null,131.746,115.34,0.0,0.0,null,1.336,1.236,1.389,null,0.0,0.0,0.0,null,-8.417,-18.485,0.0,0.0,null,null],[4339.0,null,null,1970.328,752.465,null,132.068,114.207,0.0,0.0,null,1.329,1.228,1.425,null,0.0,0.0,0.0,null,-8.469,-22.307,0.0,0.0,null,null],[4339.5,null,null,1967.754,757.592,null,132.132,114.048,0.0,0.0,null,1.32,1.218,1.414,null,0.0,0.0,0.0,null,-8.564,-25.333,0.0,0.0,null,null],[4340.0,null,null,1971.675,764.887,null,132.007,114.35,0.0,0.0,null,1.307,1.209,1.377,null,0.0,0.0,0.0,null,-8.597,-21.841,0.0,0.0,null,null],[4340.5,null,null,1975.298,774.846,null,131.658,114.957,0.0,0.0,null,1.291,1.199,1.331,null,0.0,0.0,0.0,null,-8.521,-12.853,0.0,0.0,null,null],[4341.0,null,null,1973.972,782.802,null,131.012,115.805,0.0,0.0,null,1.277,1.19,1.299,null,0.0,0.0,0.0,null,-8.464,-4.677,0.0,0.0,null,null],[4341.5,null,null,1969.55,783.359,null,130.126,115.374,0.0,0.0,null,1.277,1.186,1.267,null,0.0,0.0,0.0,null,-8.587,-2.366,0.0,0.0,null,null],[4342.0,null,null,1966.3,776.422,null,129.247,113.54,0.0,0.0,null,1.288,1.194,1.233,null,0.0,0.0,0.0,null,-8.827,-3.232,0.0,0.0,null,null],[4342.5,null,null,1971.058,766.257,null,128.573,112.938,0.0,0.0,null,1.305,1.219,1.22,null,0.0,0.0,0.0,null,-9.05,-3.113,0.0,0.0,null,null],[4343.0,null,null,1980.738,756.006,null,127.986,113.977,0.0,0.0,null,1.323,1.255,1.235,null,0.0,0.0,0.0,null,-9.26,-6.004,0.0,0.0,null,null],[4343.5,null,null,1983.778,747.007,null,127.154,113.229,0.0,0.0,null,1.339,1.288,1.304,null,0.0,0.0,0.0,null,-9.579,-8.977,0.0,0.0,null,null],[4344.0,null,null,1980.633,738.986,null,125.903,107.966,0.0,0.0,null,1.353,1.307,1.5,null,0.0,0.0,0.0,null,-10.02,-6.206,0.0,0.0,null,null],[4344.5,null,null,1981.365,730.929,null,124.366,100.735,0.0,0.0,null,1.368,1.322,1.67,null,0.0,0.0,0.0,null,-10.376,-2.666,0.0,0.0,null,null],[4345.0,null,null,1986.659,723.382,null,122.817,96.679,0.0,0.0,null,1.382,1.338,1.626,null,0.0,0.0,0.0,null,-10.451,-5.32,0.0,0.0,null,null],[4345.5,null,null,1988.744,716.99,null,121.415,98.057,0.0,0.0,null,1.395,1.352,1.54,null,0.0,0.0,0.0,null,-10.409,-6.119,0.0,0.0,null,null],[4346.0,null,null,1985.406,713.581,null,120.139,102.289,0.0,0.0,null,1.401,1.357,1.54,null,0.0,0.0,0.0,null,-10.435,-2.544,0.0,0.0,null,null],[4346.5,null,null,1980.91,715.367,null,118.94,104.841,0.0,0.0,null,1.398,1.356,1.572,null,0.0,0.0,0.0,null,-10.436,-1.722,0.0,0.0,null,null],[4347.0,null,null,1980.196,718.786,null,117.919,103.49,0.0,0.0,null,1.391,1.354,1.562,null,0.0,0.0,0.0,null,-10.398,-5.475,0.0,0.0,null,null],[4347.5,null,null,1982.93,718.188,null,117.216,99.952,0.0,0.0,null,1.392,1.351,1.606,null,0.0,0.0,0.0,null,-10.443,-8.838,0.0,0.0,null,null],[4348.0,null,null,1984.848,712.949,null,116.787,98.174,0.0,0.0,null,1.403,1.357,1.628,null,0.0,0.0,0.0,null,-10.434,-11.551,0.0,0.0,null,null],[4348.5,null,null,1985.781,705.468,null,116.516,99.473,0.0,0.0,null,1.417,1.377,1.523,null,0.0,0.0,0.0,null,-10.168,-16.638,0.0,0.0,null,null],[4349.0,null,null,1987.271,697.606,null,116.431,100.992,0.0,0.0,null,1.433,1.403,1.473,null,0.0,0.0,0.0,null,-9.866,-21.022,0.0,0.0,null,null],[4349.5,null,null,1989.11,693.153,null,116.686,99.608,0.0,0.0,null,1.443,1.416,1.583,null,0.0,0.0,0.0,null,-9.844,-23.612,0.0,0.0,null,null],[4350.0,null,null,1991.086,694.267,null,117.337,96.764,0.0,0.0,null,1.44,1.403,1.754,null,0.0,0.0,0.0,null,-10.017,-24.88,0.0,0.0,null,null],[4350.5,null,null,1996.086,700.1,null,118.238,95.504,0.0,0.0,null,1.428,1.367,1.84,null,0.0,0.0,0.0,null,-10.135,-22.693,0.0,0.0,null,null],[4351.0,null,null,2000.103,708.32,null,119.209,96.28,0.0,0.0,null,1.412,1.32,1.759,null,0.0,0.0,0.0,null,-10.15,-18.859,0.0,0.0,null,null],[4351.5,null,null,1995.981,717.884,null,120.256,98.333,0.0,0.0,null,1.393,1.279,1.582,null,0.0,0.0,0.0,null,-10.026,-18.92,0.0,0.0,null,null],[4352.0,null,null,1988.271,728.36,null,121.532,102.306,0.0,0.0,null,1.373,1.252,1.386,null,0.0,0.0,0.0,null,-9.697,-22.073,0.0,0.0,null,null],[4352.5,null,null,1988.554,740.974,null,123.002,107.051,0.0,0.0,null,1.35,1.239,1.22,null,0.0,0.0,0.0,null,-9.406,-24.499,0.0,0.0,null,null],[4353.0,null,null,1991.547,756.417,null,124.362,109.181,0.0,0.0,null,1.322,1.23,1.188,null,0.0,0.0,0.0,null,-9.459,-25.157,0.0,0.0,null,null],[4353.5,null,null,1985.792,771.713,null,125.32,108.408,0.0,0.0,null,1.296,1.217,1.251,null,0.0,0.0,0.0,null,-9.784,-25.274,0.0,0.0,null,null],[4354.0,null,null,1973.215,779.176,null,125.774,106.964,0.0,0.0,null,1.283,1.205,1.324,null,0.0,0.0,0.0,null,-10.069,-26.247,0.0,0.0,null,null],[4354.5,null,null,1963.124,777.135,null,125.797,105.838,0.0,0.0,null,1.287,1.202,1.326,null,0.0,0.0,0.0,null,-10.199,-25.531,0.0,0.0,null,null],[4355.0,null,null,1957.109,768.846,null,125.509,104.671,0.0,0.0,null,1.301,1.213,1.261,null,0.0,0.0,0.0,null,-10.249,-21.547,0.0,0.0,null,null],[4355.5,null,null,1951.396,757.467,null,125.011,103.349,0.0,0.0,null,1.32,1.236,1.248,null,0.0,0.0,0.0,null,-10.299,-17.406,0.0,0.0,null,null],[4356.0,null,null,1944.341,745.109,null,124.412,102.789,0.0,0.0,null,1.342,1.266,1.293,null,0.0,0.0,0.0,null,-10.367,-16.216,0.0,0.0,null,null],[4356.5,null,null,1939.256,733.873,null,123.853,103.801,0.0,0.0,null,1.363,1.295,1.359,null,0.0,0.0,0.0,null,-10.472,-11.091,0.0,0.0,null,null],[4357.0,null,null,1939.66,721.327,null,123.322,104.971,0.0,0.0,null,1.386,1.323,1.413,null,0.0,0.0,0.0,null,-10.658,-1.211,0.0,0.0,null,null],[4357.5,null,null,1944.178,705.229,null,122.513,104.323,0.0,0.0,null,1.418,1.35,1.492,null,0.0,0.0,0.0,null,-10.928,2.04,0.0,0.0,null,null],[4358.0,null,null,1949.875,685.323,null,121.134,102.427,0.0,0.0,null,1.459,1.383,1.552,null,0.0,0.0,0.0,null,-11.16,-5.932,0.0,0.0,null,null],[4358.5,null,null,1956.438,663.94,null,119.369,101.633,0.0,0.0,null,1.506,1.432,1.497,null,0.0,0.0,0.0,null,-11.168,-11.459,0.0,0.0,null,null],[4359.0,null,null,1963.415,647.626,null,117.858,101.668,0.0,0.0,null,1.544,1.49,1.4,null,0.0,0.0,0.0,null,-10.952,-12.065,0.0,0.0,null,null],[4359.5,null,null,1967.876,644.242,null,117.185,99.588,0.0,0.0,null,1.552,1.52,1.503,null,0.0,0.0,0.0,null,-10.825,-15.204,0.0,0.0,null,null],[4360.0,null,null,1968.647,654.553,null,117.392,95.153,0.0,0.0,null,1.528,1.492,1.955,null,0.0,0.0,0.0,null,-11.035,-20.492,0.0,0.0,null,null],[4360.5,null,null,1966.439,674.988,null,118.114,92.597,0.0,0.0,null,1.482,1.417,2.406,null,0.0,0.0,0.0,null,-11.279,-21.658,0.0,0.0,null,null],[4361.0,null,null,1962.674,702.513,null,119.099,95.516,0.0,0.0,null,1.423,1.324,2.272,null,0.0,0.0,0.0,null,-11.027,-19.724,0.0,0.0,null,null],[4361.5,null,null,1961.38,734.849,null,120.503,103.687,0.0,0.0,null,1.361,1.234,1.765,null,0.0,0.0,0.0,null,-10.245,-20.121,0.0,0.0,null,null],[4362.0,null,null,1964.354,771.686,null,122.564,113.146,0.0,0.0,null,1.296,1.157,1.38,null,0.0,0.0,0.0,null,-9.292,-24.091,0.0,0.0,null,null],[4362.5,null,null,1967.977,813.328,null,125.068,119.198,0.0,0.0,null,1.23,1.096,1.138,null,0.0,0.0,0.0,null,-8.463,-25.273,0.0,0.0,null,null],[4363.0,null,null,1970.388,855.559,null,127.388,120.205,0.0,0.0,null,1.169,1.051,0.991,null,0.0,0.0,0.0,null,-7.896,-17.135,0.0,0.0,null,null],[4363.5,null,null,1974.342,892.023,null,128.972,118.135,0.0,0.0,null,1.121,1.02,0.959,null,0.0,0.0,0.0,null,-7.632,-7.152,0.0,0.0,null,null],[4364.0,null,null,1980.541,918.076,null,129.734,115.695,0.0,0.0,null,1.089,1.005,0.991,null,0.0,0.0,0.0,null,-7.53,-4.475,0.0,0.0,null,null],[4364.5,null,null,1988.336,931.546,null,130.008,114.157,0.0,0.0,null,1.073,1.004,1.028,null,0.0,0.0,0.0,null,-7.495,-5.661,0.0,0.0,null,null],[4365.0,null,null,1992.191,931.406,null,130.194,114.051,0.0,0.0,null,1.074,1.013,1.07,null,0.0,0.0,0.0,null,-7.576,-5.748,0.0,0.0,null,null],[4365.5,null,null,1986.819,920.547,null,130.473,113.987,0.0,0.0,null,1.086,1.025,1.129,null,0.0,0.0,0.0,null,-7.781,-6.085,0.0,0.0,null,null],[4366.0,null,null,1978.129,902.47,null,130.79,112.617,0.0,0.0,null,1.108,1.039,1.206,null,0.0,0.0,0.0,null,-8.039,-4.86,0.0,0.0,null,null],[4366.5,null,null,1977.594,881.057,null,131.035,109.949,0.0,0.0,null,1.135,1.057,1.25,null,0.0,0.0,0.0,null,-8.274,-2.367,0.0,0.0,null,null],[4367.0,null,null,1981.062,857.548,null,131.114,107.625,0.0,0.0,null,1.166,1.083,1.231,null,0.0,0.0,0.0,null,-8.455,-2.754,0.0,0.0,null,null],[4367.5,null,null,1975.995,832.133,null,130.985,106.848,0.0,0.0,null,1.202,1.118,1.212,null,0.0,0.0,0.0,null,-8.636,-2.405,0.0,0.0,null,null],[4368.0,null,null,1967.468,806.323,null,130.714,107.877,0.0,0.0,null,1.24,1.16,1.216,null,0.0,0.0,0.0,null,-8.871,-4.14,0.0,0.0,null,null],[4368.5,null,null,1969.391,783.11,null,130.398,108.965,0.0,0.0,null,1.277,1.202,1.26,null,0.0,0.0,0.0,null,-9.114,-11.542,0.0,0.0,null,null],[4369.0,null,null,1979.522,764.757,null,130.039,108.228,0.0,0.0,null,1.308,1.236,1.367,null,0.0,0.0,0.0,null,-9.326,-16.58,0.0,0.0,null,null],[4369.5,null,null,1983.306,751.061,null,129.516,106.394,0.0,0.0,null,1.331,1.259,1.521,null,0.0,0.0,0.0,null,-9.562,-14.363,0.0,0.0,null,null],[4370.0,null,null,1975.075,739.503,null,128.748,106.854,0.0,0.0,null,1.352,1.271,1.618,null,0.0,0.0,0.0,null,-9.802,-12.323,0.0,0.0,null,null],[4370.5,null,null,1961.643,729.691,null,127.835,109.877,0.0,0.0,null,1.37,1.278,1.575,null,0.0,0.0,0.0,null,-9.948,-10.497,0.0,0.0,null,null],[4371.0,null,null,1953.162,724.307,null,127.051,112.325,0.0,0.0,null,1.381,1.28,1.488,null,0.0,0.0,0.0,null,-10.005,-7.137,0.0,0.0,null,null],[4371.5,null,null,1953.906,724.643,null,126.669,112.788,0.0,0.0,null,1.38,1.275,1.463,null,0.0,0.0,0.0,null,-10.062,-6.521,0.0,0.0,null,null],[4372.0,null,null,1962.869,730.581,null,126.725,113.004,0.0,0.0,null,1.369,1.261,1.448,null,0.0,0.0,0.0,null,-10.138,-8.666,0.0,0.0,null,null],[4372.5,null,null,1972.734,742.123,null,127.001,113.956,0.0,0.0,null,1.347,1.241,1.411,null,0.0,0.0,0.0,null,-10.174,-12.213,0.0,0.0,null,null],[4373.0,null,null,1976.854,757.282,null,127.237,114.2,0.0,0.0,null,1.321,1.22,1.389,null,0.0,0.0,0.0,null,-10.156,-16.84,0.0,0.0,null,null],[4373.5,null,null,1975.919,770.591,null,127.35,113.175,0.0,0.0,null,1.298,1.202,1.393,null,0.0,0.0,0.0,null,-10.136,-18.759,0.0,0.0,null,null],[4374.0,null,null,1975.514,777.926,null,127.471,112.896,0.0,0.0,null,1.285,1.189,1.36,null,0.0,0.0,0.0,null,-10.168,-18.268,0.0,0.0,null,null],[4374.5,null,null,1978.752,781.538,null,127.78,114.615,0.0,0.0,null,1.28,1.183,1.308,null,0.0,0.0,0.0,null,-10.27,-18.584,0.0,0.0,null,null],[4375.0,null,null,1982.894,787.213,null,128.251,116.115,0.0,0.0,null,1.27,1.179,1.299,null,0.0,0.0,0.0,null,-10.482,-15.967,0.0,0.0,null,null],[4375.5,null,null,1983.994,798.077,null,128.632,116.894,0.0,0.0,null,1.253,1.173,1.311,null,0.0,0.0,0.0,null,-10.812,-9.569,0.0,0.0,null,null],[4376.0,null,null,1983.17,809.754,null,128.779,117.864,0.0,0.0,null,1.235,1.164,1.315,null,0.0,0.0,0.0,null,-11.2,-3.359,0.0,0.0,null,null],[4376.5,null,null,1983.474,815.863,null,128.847,117.938,0.0,0.0,null,1.226,1.157,1.28,null,0.0,0.0,0.0,null,-11.675,1.622,0.0,0.0,null,null],[4377.0,null,null,1985.878,811.828,null,128.989,116.389,0.0,0.0,null,1.232,1.164,1.225,null,0.0,0.0,0.0,null,-12.398,3.234,0.0,0.0,null,null],[4377.5,null,null,1996.336,796.446,null,128.938,114.653,0.0,0.0,null,1.256,1.194,1.175,null,0.0,0.0,0.0,null,-13.522,0.173,0.0,0.0,null,null],[4378.0,null,null,2006.279,771.566,null,128.122,113.628,0.0,0.0,null,1.296,1.254,1.134,null,0.0,0.0,0.0,null,-15.004,-1.822,0.0,0.0,null,null],[4378.5,null,null,2001.621,743.967,null,126.215,112.054,0.0,0.0,null,1.344,1.338,1.07,null,0.0,0.0,0.0,null,-16.607,-0.984,0.0,0.0,null,null],[4379.0,null,null,1989.297,719.506,null,123.507,107.448,0.0,0.0,null,1.39,1.437,1.042,null,0.0,0.0,0.0,null,-18.424,-3.142,0.0,0.0,null,null],[4379.5,null,null,1986.19,698.905,null,120.639,98.179,0.0,0.0,null,1.431,1.527,1.302,null,0.0,0.0,0.0,null,-21.21,-6.087,0.0,0.0,null,null],[4380.0,null,null,1989.737,680.967,null,117.978,85.337,0.0,0.0,null,1.468,1.588,2.088,null,0.0,0.0,0.0,null,-25.161,-5.103,0.0,0.0,null,null],[4380.5,null,null,1989.248,666.017,null,115.439,72.81,0.0,0.0,null,1.501,1.622,3.071,null,0.0,0.0,0.0,null,-29.073,-6.09,0.0,0.0,null,null],[4381.0,null,null,1986.33,655.866,null,113.004,65.221,0.0,0.0,null,1.525,1.634,3.57,null,0.0,0.0,0.0,null,-31.665,-14.414,0.0,0.0,null,null],[4381.5,null,null,1987.229,653.952,null,111.161,63.773,0.0,0.0,null,1.529,1.618,3.614,null,0.0,0.0,0.0,null,-32.69,-23.359,0.0,0.0,null,null],[4382.0,null,null,1988.546,662.228,null,110.544,67.678,0.0,0.0,null,1.51,1.576,3.433,null,0.0,0.0,0.0,null,-32.492,-27.141,0.0,0.0,null,null],[4382.5,null,null,1984.595,678.407,null,111.272,76.681,0.0,0.0,null,1.474,1.519,3.097,null,0.0,0.0,0.0,null,-31.522,-27.501,0.0,0.0,null,null],[4383.0,null,null,1978.683,697.617,null,112.788,88.988,0.0,0.0,null,1.433,1.464,2.572,null,0.0,0.0,0.0,null,-30.117,-24.983,0.0,0.0,null,null],[4383.5,null,null,1975.723,718.202,null,114.382,100.692,0.0,0.0,null,1.392,1.424,1.923,null,0.0,0.0,0.0,null,-28.155,-23.392,0.0,0.0,null,null],[4384.0,null,null,1975.023,739.99,null,115.751,108.383,0.0,0.0,null,1.351,1.399,1.43,null,0.0,0.0,0.0,null,-25.787,-27.257,0.0,0.0,null,null],[4384.5,null,null,1974.214,761.46,null,116.923,109.599,0.0,0.0,null,1.313,1.376,1.285,null,0.0,0.0,0.0,null,-24.359,-32.075,0.0,0.0,null,null],[4385.0,null,null,1972.739,779.339,null,117.83,104.614,0.0,0.0,null,1.283,1.347,1.47,null,0.0,0.0,0.0,null,-25.183,-32.593,0.0,0.0,null,null],[4385.5,null,null,1970.081,790.595,null,118.228,98.273,0.0,0.0,null,1.265,1.324,1.74,null,0.0,0.0,0.0,null,-27.521,-31.147,0.0,0.0,null,null],[4386.0,null,null,1966.88,794.269,null,118.009,94.49,0.0,0.0,null,1.259,1.32,1.687,null,0.0,0.0,0.0,null,-29.376,-29.607,0.0,0.0,null,null],[4386.5,null,null,1964.347,792.575,null,117.364,92.63,0.0,0.0,null,1.262,1.332,1.478,null,0.0,0.0,0.0,null,-30.224,-27.215,0.0,0.0,null,null],[4387.0,null,null,1963.627,789.161,null,116.632,89.273,0.0,0.0,null,1.267,1.345,1.49,null,0.0,0.0,0.0,null,-31.144,-26.911,0.0,0.0,null,null],[4387.5,null,null,1965.243,786.86,null,116.035,83.568,0.0,0.0,null,1.271,1.351,1.862,null,0.0,0.0,0.0,null,-32.878,-28.896,0.0,0.0,null,null],[4388.0,null,null,1969.231,785.375,null,115.554,77.828,0.0,0.0,null,1.273,1.353,2.36,null,0.0,0.0,0.0,null,-34.959,-28.242,0.0,0.0,null,null],[4388.5,null,null,1973.79,781.974,null,115.099,74.351,0.0,0.0,null,1.279,1.358,2.403,null,0.0,0.0,0.0,null,-36.378,-23.965,0.0,0.0,null,null],[4389.0,null,null,1976.105,776.626,null,114.693,73.28,0.0,0.0,null,1.288,1.365,2.086,null,0.0,0.0,0.0,null,-36.899,-21.274,0.0,0.0,null,null],[4389.5,null,null,1974.901,772.883,null,114.446,73.348,0.0,0.0,null,1.294,1.37,1.94,null,0.0,0.0,0.0,null,-37.441,-22.79,0.0,0.0,null,null],[4390.0,null,null,1971.906,772.755,null,114.396,72.696,0.0,0.0,null,1.294,1.369,2.135,null,0.0,0.0,0.0,null,-38.684,-24.143,0.0,0.0,null,null],[4390.5,null,null,1968.817,775.227,null,114.458,71.04,0.0,0.0,null,1.29,1.37,2.403,null,0.0,0.0,0.0,null,-40.11,-23.25,0.0,0.0,null,null],[4391.0,null,null,1966.895,778.298,null,114.518,70.069,0.0,0.0,null,1.285,1.374,2.451,null,0.0,0.0,0.0,null,-40.934,-23.474,0.0,0.0,null,null],[4391.5,null,null,1968.744,779.174,null,114.515,69.984,0.0,0.0,null,1.283,1.382,2.393,null,0.0,0.0,0.0,null,-40.971,-23.151,0.0,0.0,null,null],[4392.0,null,null,1973.566,774.367,null,114.439,69.887,0.0,0.0,null,1.291,1.396,2.339,null,0.0,0.0,0.0,null,-40.43,-17.057,0.0,0.0,null,null],[4392.5,null,null,1973.949,764.176,null,114.302,71.003,0.0,0.0,null,1.309,1.416,2.307,null,0.0,0.0,0.0,null,-39.593,-11.454,0.0,0.0,null,null],[4393.0,null,null,1969.836,752.672,null,114.131,74.803,0.0,0.0,null,1.329,1.434,2.338,null,0.0,0.0,0.0,null,-38.603,-14.677,0.0,0.0,null,null],[4393.5,null,null,1969.504,741.663,null,113.946,78.802,0.0,0.0,null,1.348,1.448,2.297,null,0.0,0.0,0.0,null,-37.392,-22.705,0.0,0.0,null,null],[4394.0,null,null,1973.657,730.901,null,113.748,79.83,0.0,0.0,null,1.368,1.463,2.099,null,0.0,0.0,0.0,null,-35.983,-24.192,0.0,0.0,null,null],[4394.5,null,null,1977.519,721.484,null,113.506,78.575,0.0,0.0,null,1.386,1.475,1.958,null,0.0,0.0,0.0,null,-34.791,-14.887,0.0,0.0,null,null],[4395.0,null,null,1980.801,714.104,null,113.175,77.529,0.0,0.0,null,1.4,1.488,2.035,null,0.0,0.0,0.0,null,-34.187,-1.255,0.0,0.0,null,null],[4395.5,null,null,1982.647,706.884,null,112.747,76.492,0.0,0.0,null,1.415,1.505,2.249,null,0.0,0.0,0.0,null,-33.899,5.432,0.0,0.0,null,null],[4396.0,null,null,1979.283,699.846,null,112.301,75.536,0.0,0.0,null,1.429,1.519,2.433,null,0.0,0.0,0.0,null,-33.386,1.298,0.0,0.0,null,null],[4396.5,null,null,1973.863,694.781,null,111.962,75.84,0.0,0.0,null,1.439,1.522,2.552,null,0.0,0.0,0.0,null,-32.449,-9.825,0.0,0.0,null,null],[4397.0,null,null,1974.499,690.93,null,111.864,76.85,0.0,0.0,null,1.447,1.518,2.624,null,0.0,0.0,0.0,null,-31.079,-20.241,0.0,0.0,null,null],[4397.5,null,null,1979.452,686.7,null,112.101,77.857,0.0,0.0,null,1.456,1.51,2.583,null,0.0,0.0,0.0,null,-29.112,-21.847,0.0,0.0,null,null],[4398.0,null,null,1981.368,683.393,null,112.683,79.613,0.0,0.0,null,1.463,1.495,2.415,null,0.0,0.0,0.0,null,-26.473,-16.952,0.0,0.0,null,null],[4398.5,null,null,1979.68,682.638,null,113.593,81.39,0.0,0.0,null,1.465,1.475,2.12,null,0.0,0.0,0.0,null,-23.445,-15.86,0.0,0.0,null,null],[4399.0,null,null,1975.308,684.5,null,114.823,82.731,0.0,0.0,null,1.461,1.455,1.845,null,0.0,0.0,0.0,null,-20.352,-21.861,0.0,0.0,null,null],[4399.5,null,null,1967.772,689.38,null,116.336,85.11,0.0,0.0,null,1.451,1.432,1.674,null,0.0,0.0,0.0,null,-17.363,-26.944,0.0,0.0,null,null],[4400.0,null,null,1961.46,697.539,null,118.024,88.518,0.0,0.0,null,1.434,1.398,1.563,null,0.0,0.0,0.0,null,-14.785,-26.935,0.0,0.0,null,null],[4400.5,null,null,1961.934,707.901,null,119.715,91.141,0.0,0.0,null,1.413,1.359,1.498,null,0.0,0.0,0.0,null,-12.946,-26.573,0.0,0.0,null,null],[4401.0,null,null,1966.925,719.277,null,121.203,92.841,0.0,0.0,null,1.39,1.325,1.459,null,0.0,0.0,0.0,null,-11.794,-28.564,0.0,0.0,null,null],[4401.5,null,null,1971.586,729.42,null,122.338,94.3,0.0,0.0,null,1.371,1.3,1.391,null,0.0,0.0,0.0,null,-10.996,-28.328,0.0,0.0,null,null],[4402.0,null,null,1976.051,736.88,null,123.11,95.401,0.0,0.0,null,1.357,1.283,1.323,null,0.0,0.0,0.0,null,-10.356,-25.145,0.0,0.0,null,null],[4402.5,null,null,1982.362,743.607,null,123.677,96.717,0.0,0.0,null,1.345,1.271,1.306,null,0.0,0.0,0.0,null,-9.877,-20.987,0.0,0.0,null,null],[4403.0,null,null,1988.923,752.304,null,124.229,98.205,0.0,0.0,null,1.329,1.256,1.336,null,0.0,0.0,0.0,null,-9.619,-16.946,0.0,0.0,null,null],[4403.5,null,null,1993.313,761.84,null,124.841,99.231,0.0,0.0,null,1.313,1.237,1.371,null,0.0,0.0,0.0,null,-9.506,-16.853,0.0,0.0,null,null],[4404.0,null,null,1997.375,770.358,null,125.471,100.284,0.0,0.0,null,1.298,1.217,1.392,null,0.0,0.0,0.0,null,-9.412,-18.651,0.0,0.0,null,null],[4404.5,null,null,2000.14,777.699,null,126.036,101.643,0.0,0.0,null,1.286,1.197,1.4,null,0.0,0.0,0.0,null,-9.312,-14.489,0.0,0.0,null,null],[4405.0,null,null,2002.883,784.086,null,126.476,102.846,0.0,0.0,null,1.275,1.181,1.367,null,0.0,0.0,0.0,null,-9.25,-9.471,0.0,0.0,null,null],[4405.5,null,null,2013.043,788.224,null,126.795,104.359,0.0,0.0,null,1.269,1.169,1.332,null,0.0,0.0,0.0,null,-9.207,-12.025,0.0,0.0,null,null],[4406.0,null,null,2024.502,791.275,null,127.055,106.243,0.0,0.0,null,1.264,1.161,1.308,null,0.0,0.0,0.0,null,-9.137,-13.003,0.0,0.0,null,null],[4406.5,null,null,2020.764,794.96,null,127.273,107.627,0.0,0.0,null,1.258,1.158,1.267,null,0.0,0.0,0.0,null,-9.05,-4.201,0.0,0.0,null,null],[4407.0,null,null,2004.042,798.864,null,127.404,108.617,0.0,0.0,null,1.252,1.158,1.242,null,0.0,0.0,0.0,null,-9.015,4.75,0.0,0.0,null,null],[4407.5,null,null,1987.44,800.955,null,127.419,109.107,0.0,0.0,null,1.249,1.159,1.263,null,0.0,0.0,0.0,null,-9.115,6.592,0.0,0.0,null,null],[4408.0,null,null,1975.009,799.146,null,127.362,108.491,0.0,0.0,null,1.251,1.162,1.311,null,0.0,0.0,0.0,null,-9.314,7.171,0.0,0.0,null,null],[4408.5,null,null,1968.829,791.614,null,127.349,107.672,0.0,0.0,null,1.263,1.169,1.326,null,0.0,0.0,0.0,null,-9.489,5.983,0.0,0.0,null,null],[4409.0,null,null,1968.319,781.303,null,127.442,107.879,0.0,0.0,null,1.28,1.184,1.318,null,0.0,0.0,0.0,null,-9.617,-1.571,0.0,0.0,null,null],[4409.5,null,null,1967.925,772.218,null,127.512,108.369,0.0,0.0,null,1.295,1.202,1.328,null,0.0,0.0,0.0,null,-9.735,-10.766,0.0,0.0,null,null],[4410.0,null,null,1967.822,765.662,null,127.345,108.041,0.0,0.0,null,1.306,1.218,1.358,null,0.0,0.0,0.0,null,-9.816,-15.513,0.0,0.0,null,null],[4410.5,null,null,1974.107,759.809,null,126.851,106.634,0.0,0.0,null,1.316,1.232,1.389,null,0.0,0.0,0.0,null,-9.841,-13.192,0.0,0.0,null,null],[4411.0,null,null,1985.443,753.015,null,126.144,105.275,0.0,0.0,null,1.328,1.249,1.401,null,0.0,0.0,0.0,null,-9.829,-3.352,0.0,0.0,null,null],[4411.5,null,null,1993.582,744.308,null,125.368,105.384,0.0,0.0,null,1.344,1.267,1.387,null,0.0,0.0,0.0,null,-9.789,4.037,0.0,0.0,null,null],[4412.0,null,null,1993.991,734.622,null,124.527,106.372,0.0,0.0,null,1.361,1.283,1.373,null,0.0,0.0,0.0,null,-9.74,2.212,0.0,0.0,null,null],[4412.5,null,null,1989.366,724.975,null,123.558,105.996,0.0,0.0,null,1.379,1.297,1.396,null,0.0,0.0,0.0,null,-9.693,-0.1,0.0,0.0,null,null],[4413.0,null,null,1989.286,714.721,null,122.474,103.994,0.0,0.0,null,1.399,1.315,1.415,null,0.0,0.0,0.0,null,-9.635,-0.189,0.0,0.0,null,null],[4413.5,null,null,1997.405,701.311,null,121.296,102.232,0.0,0.0,null,1.426,1.343,1.386,null,0.0,0.0,0.0,null,-9.621,-2.546,0.0,0.0,null,null],[4414.0,null,null,2003.449,683.411,null,119.966,101.434,0.0,0.0,null,1.463,1.386,1.365,null,0.0,0.0,0.0,null,-9.804,-3.791,0.0,0.0,null,null],[4414.5,null,null,2002.585,662.678,null,118.413,101.005,0.0,0.0,null,1.509,1.442,1.424,null,0.0,0.0,0.0,null,-10.187,0.331,0.0,0.0,null,null],[4415.0,null,null,2003.181,642.562,null,116.739,99.889,0.0,0.0,null,1.556,1.508,1.535,null,0.0,0.0,0.0,null,-10.565,5.593,0.0,0.0,null,null],[4415.5,null,null,2001.964,626.821,null,115.261,97.147,0.0,0.0,null,1.595,1.567,1.662,null,0.0,0.0,0.0,null,-10.857,9.822,0.0,0.0,null,null],[4416.0,null,null,1991.522,617.01,null,114.287,92.612,0.0,0.0,null,1.621,1.597,1.841,null,0.0,0.0,0.0,null,-11.13,12.014,0.0,0.0,null,null],[4416.5,null,null,1981.478,611.35,null,113.831,88.476,0.0,0.0,null,1.636,1.593,2.102,null,0.0,0.0,0.0,null,-11.315,8.448,0.0,0.0,null,null],[4417.0,null,null,1978.758,608.74,null,113.709,87.047,0.0,0.0,null,1.643,1.568,2.187,null,0.0,0.0,0.0,null,-11.246,1.048,0.0,0.0,null,null],[4417.5,null,null,1976.107,609.24,null,113.922,88.551,0.0,0.0,null,1.641,1.542,1.927,null,0.0,0.0,0.0,null,-10.962,-3.371,0.0,0.0,null,null],[4418.0,null,null,1968.314,612.498,null,114.665,92.136,0.0,0.0,null,1.633,1.522,1.643,null,0.0,0.0,0.0,null,-10.7,-7.562,0.0,0.0,null,null],[4418.5,null,null,1959.905,619.334,null,116.005,96.813,0.0,0.0,null,1.615,1.505,1.515,null,0.0,0.0,0.0,null,-10.626,-17.92,0.0,0.0,null,null],[4419.0,null,null,1957.338,631.358,null,117.667,99.63,0.0,0.0,null,1.584,1.487,1.498,null,0.0,0.0,0.0,null,-10.671,-28.949,0.0,0.0,null,null],[4419.5,null,null,1965.164,646.113,null,119.145,98.348,0.0,0.0,null,1.548,1.467,1.507,null,0.0,0.0,0.0,null,-10.703,-31.916,0.0,0.0,null,null],[4420.0,null,null,1976.364,658.527,null,120.105,95.111,0.0,0.0,null,1.519,1.449,1.519,null,0.0,0.0,0.0,null,-10.73,-29.293,0.0,0.0,null,null],[4420.5,null,null,1979.849,663.528,null,120.591,93.911,0.0,0.0,null,1.507,1.436,1.535,null,0.0,0.0,0.0,null,-10.811,-26.616,0.0,0.0,null,null],[4421.0,null,null,1976.223,659.788,null,120.703,95.135,0.0,0.0,null,1.516,1.434,1.53,null,0.0,0.0,0.0,null,-10.94,-24.043,0.0,0.0,null,null],[4421.5,null,null,1973.778,651.3,null,120.248,95.935,0.0,0.0,null,1.535,1.45,1.516,null,0.0,0.0,0.0,null,-11.075,-19.207,0.0,0.0,null,null],[4422.0,null,null,1974.392,642.513,null,119.073,94.662,0.0,0.0,null,1.556,1.484,1.495,null,0.0,0.0,0.0,null,-11.162,-11.485,0.0,0.0,null,null],[4422.5,null,null,1973.584,637.159,null,117.578,92.399,0.0,0.0,null,1.569,1.522,1.494,null,0.0,0.0,0.0,null,-11.101,-3.595,0.0,0.0,null,null],[4423.0,null,null,1970.191,641.933,null,116.627,90.587,0.0,0.0,null,1.558,1.537,1.564,null,0.0,0.0,0.0,null,-10.854,1.339,0.0,0.0,null,null],[4423.5,null,null,1967.281,659.774,null,116.829,88.398,0.0,0.0,null,1.516,1.498,1.802,null,0.0,0.0,0.0,null,-10.683,0.617,0.0,0.0,null,null],[4424.0,null,null,1968.369,685.36,null,118.012,85.526,0.0,0.0,null,1.459,1.406,2.267,null,0.0,0.0,0.0,null,-10.831,-4.676,0.0,0.0,null,null],[4424.5,null,null,1974.928,712.628,null,119.56,84.163,0.0,0.0,null,1.403,1.298,2.523,null,0.0,0.0,0.0,null,-10.856,-11.001,0.0,0.0,null,null],[4425.0,null,null,1983.072,741.121,null,121.264,87.621,0.0,0.0,null,1.349,1.206,2.001,null,0.0,0.0,0.0,null,-10.156,-15.367,0.0,0.0,null,null],[4425.5,null,null,1985.846,773.211,null,123.514,96.205,0.0,0.0,null,1.293,1.144,1.284,null,0.0,0.0,0.0,null,-8.93,-17.576,0.0,0.0,null,null],[4426.0,null,null,1980.308,809.702,null,126.579,106.308,0.0,0.0,null,1.235,1.104,0.99,null,0.0,0.0,0.0,null,-7.855,-17.176,0.0,0.0,null,null],[4426.5,null,null,1970.116,847.488,null,129.976,112.932,0.0,0.0,null,1.18,1.073,0.977,null,0.0,0.0,0.0,null,-7.231,-13.048,0.0,0.0,null,null],[4427.0,null,null,1960.801,882.002,null,132.665,114.558,0.0,0.0,null,1.134,1.049,1.055,null,0.0,0.0,0.0,null,-6.99,-8.955,0.0,0.0,null,null],[4427.5,null,null,1954.636,907.552,null,133.941,114.424,0.0,0.0,null,1.102,1.032,1.117,null,0.0,0.0,0.0,null,-6.986,-10.024,0.0,0.0,null,null],[4428.0,null,null,1951.295,920.436,null,134.035,114.791,0.0,0.0,null,1.086,1.024,1.125,null,0.0,0.0,0.0,null,-7.067,-14.635,0.0,0.0,null,null],[4428.5,null,null,1948.55,921.473,null,133.653,115.476,0.0,0.0,null,1.085,1.023,1.101,null,0.0,0.0,0.0,null,-7.122,-16.469,0.0,0.0,null,null],[4429.0,null,null,1946.508,913.845,null,133.173,115.696,0.0,0.0,null,1.094,1.028,1.089,null,0.0,0.0,0.0,null,-7.141,-13.665,0.0,0.0,null,null],[4429.5,null,null,1945.23,899.521,null,132.477,116.242,0.0,0.0,null,1.112,1.04,1.105,null,0.0,0.0,0.0,null,-7.203,-10.034,0.0,0.0,null,null],[4430.0,null,null,1945.435,880.778,null,131.356,117.495,0.0,0.0,null,1.135,1.061,1.133,null,0.0,0.0,0.0,null,-7.337,-10.619,0.0,0.0,null,null],[4430.5,null,null,1946.44,859.515,null,129.883,118.861,0.0,0.0,null,1.163,1.09,1.16,null,0.0,0.0,0.0,null,-7.511,-13.539,0.0,0.0,null,null],[4431.0,null,null,1945.629,839.448,null,128.496,120.018,0.0,0.0,null,1.191,1.123,1.214,null,0.0,0.0,0.0,null,-7.7,-13.035,0.0,0.0,null,null],[4431.5,null,null,1943.919,826.913,null,127.646,121.112,0.0,0.0,null,1.209,1.153,1.254,null,0.0,0.0,0.0,null,-7.824,-10.887,0.0,0.0,null,null],[4432.0,null,null,1946.983,823.539,null,127.411,121.42,0.0,0.0,null,1.214,1.167,1.311,null,0.0,0.0,0.0,null,-7.83,-12.248,0.0,0.0,null,null],[4432.5,null,null,1958.628,823.037,null,127.549,119.566,0.0,0.0,null,1.215,1.155,1.515,null,0.0,0.0,0.0,null,-7.835,-14.228,0.0,0.0,null,null],[4433.0,null,null,1971.959,821.253,null,127.708,115.885,0.0,0.0,null,1.218,1.13,1.664,null,0.0,0.0,0.0,null,-7.849,-14.04,0.0,0.0,null,null],[4433.5,null,null,1975.891,818.557,null,127.708,111.489,0.0,0.0,null,1.222,1.114,1.488,null,0.0,0.0,0.0,null,-7.707,-16.344,0.0,0.0,null,null],[4434.0,null,null,1970.023,816.501,null,127.723,107.163,0.0,0.0,null,1.225,1.111,1.18,null,0.0,0.0,0.0,null,-7.434,-20.235,0.0,0.0,null,null],[4434.5,null,null,1965.781,821.052,null,128.021,105.169,0.0,0.0,null,1.218,1.113,1.048,null,0.0,0.0,0.0,null,-7.253,-19.217,0.0,0.0,null,null],[4435.0,null,null,1968.976,838.542,null,128.646,106.058,0.0,0.0,null,1.193,1.106,1.109,null,0.0,0.0,0.0,null,-7.276,-14.281,0.0,0.0,null,null],[4435.5,null,null,1974.792,864.658,null,129.497,107.175,0.0,0.0,null,1.157,1.087,1.223,null,0.0,0.0,0.0,null,-7.532,-12.171,0.0,0.0,null,null],[4436.0,null,null,1979.635,886.737,null,130.479,107.375,0.0,0.0,null,1.128,1.063,1.302,null,0.0,0.0,0.0,null,-8.0,-14.37,0.0,0.0,null,null],[4436.5,null,null,1985.948,895.405,null,131.369,108.0,0.0,0.0,null,1.117,1.046,1.266,null,0.0,0.0,0.0,null,-8.609,-18.587,0.0,0.0,null,null],[4437.0,null,null,1995.956,887.554,null,131.795,108.547,0.0,0.0,null,1.127,1.048,1.12,null,0.0,0.0,0.0,null,-9.446,-20.32,0.0,0.0,null,null],[4437.5,null,null,2006.222,865.674,null,131.365,109.035,0.0,0.0,null,1.155,1.078,0.975,null,0.0,0.0,0.0,null,-10.705,-14.791,0.0,0.0,null,null],[4438.0,null,null,2006.528,838.296,null,129.779,109.952,0.0,0.0,null,1.193,1.142,0.879,null,0.0,0.0,0.0,null,-12.394,-8.582,0.0,0.0,null,null],[4438.5,null,null,1996.601,811.655,null,127.06,107.726,0.0,0.0,null,1.232,1.235,0.847,null,0.0,0.0,0.0,null,-14.517,-9.131,0.0,0.0,null,null],[4439.0,null,null,1987.506,785.649,null,123.533,98.926,0.0,0.0,null,1.273,1.344,0.945,null,0.0,0.0,0.0,null,-17.391,-12.218,0.0,0.0,null,null],[4439.5,null,null,1987.33,757.487,null,119.536,86.047,0.0,0.0,null,1.32,1.444,1.266,null,0.0,0.0,0.0,null,-21.364,-9.826,0.0,0.0,null,null],[4440.0,null,null,1995.481,724.606,null,115.307,74.78,0.0,0.0,null,1.38,1.53,1.858,null,0.0,0.0,0.0,null,-26.31,-4.789,0.0,0.0,null,null],[4440.5,null,null,2004.163,687.842,null,111.163,67.818,0.0,0.0,null,1.454,1.61,2.46,null,0.0,0.0,0.0,null,-31.31,-4.322,0.0,0.0,null,null],[4441.0,null,null,2003.827,654.179,null,107.575,64.211,0.0,0.0,null,1.529,1.686,2.71,null,0.0,0.0,0.0,null,-35.235,-8.604,0.0,0.0,null,null],[4441.5,null,null,1997.55,631.042,null,105.042,62.588,0.0,0.0,null,1.585,1.749,2.769,null,0.0,0.0,0.0,null,-37.881,-10.647,0.0,0.0,null,null],[4442.0,null,null,1997.611,620.771,null,103.888,62.144,0.0,0.0,null,1.611,1.783,2.924,null,0.0,0.0,0.0,null,-39.88,-6.752,0.0,0.0,null,null],[4442.5,null,null,2003.269,620.775,null,103.919,61.708,0.0,0.0,null,1.611,1.788,3.175,null,0.0,0.0,0.0,null,-41.643,-3.162,0.0,0.0,null,null],[4443.0,null,null,2011.744,626.625,null,104.475,61.806,0.0,0.0,null,1.596,1.776,3.244,null,0.0,0.0,0.0,null,-43.187,-3.672,0.0,0.0,null,null],[4443.5,null,null,2020.321,633.653,null,104.886,64.043,0.0,0.0,null,1.578,1.765,3.13,null,0.0,0.0,0.0,null,-44.449,-4.043,0.0,0.0,null,null],[4444.0,null,null,2021.849,639.508,null,104.823,67.245,0.0,0.0,null,1.564,1.761,3.041,null,0.0,0.0,0.0,null,-45.326,-1.866,0.0,0.0,null,null],[4444.5,null,null,2013.391,643.481,null,104.401,69.216,0.0,0.0,null,1.554,1.759,3.029,null,0.0,0.0,0.0,null,-45.849,-0.017,0.0,0.0,null,null],[4445.0,null,null,2004.745,646.292,null,103.991,70.356,0.0,0.0,null,1.547,1.756,3.069,null,0.0,0.0,0.0,null,-46.198,1.681,0.0,0.0,null,null],[4445.5,null,null,2001.505,647.925,null,103.844,70.847,0.0,0.0,null,1.543,1.756,3.101,null,0.0,0.0,0.0,null,-46.48,5.901,0.0,0.0,null,null],[4446.0,null,null,2000.891,647.958,null,103.884,69.406,0.0,0.0,null,1.543,1.754,3.152,null,0.0,0.0,0.0,null,-46.644,8.015,0.0,0.0,null,null],[4446.5,null,null,1998.779,646.483,null,103.942,66.363,0.0,0.0,null,1.547,1.749,3.249,null,0.0,0.0,0.0,null,-46.61,3.418,0.0,0.0,null,null],[4447.0,null,null,1992.579,644.392,null,103.962,64.231,0.0,0.0,null,1.552,1.744,3.337,null,0.0,0.0,0.0,null,-46.338,-0.769,0.0,0.0,null,null],[4447.5,null,null,1983.169,641.441,null,104.02,64.036,0.0,0.0,null,1.559,1.747,3.33,null,0.0,0.0,0.0,null,-45.772,1.263,0.0,0.0,null,null],[4448.0,null,null,1976.596,637.547,null,104.268,65.251,0.0,0.0,null,1.569,1.754,3.237,null,0.0,0.0,0.0,null,-44.916,1.689,0.0,0.0,null,null],[4448.5,null,null,1975.473,633.667,null,104.772,67.429,0.0,0.0,null,1.578,1.76,3.142,null,0.0,0.0,0.0,null,-43.948,-5.911,0.0,0.0,null,null],[4449.0,null,null,1973.636,631.41,null,105.485,69.33,0.0,0.0,null,1.584,1.76,3.094,null,0.0,0.0,0.0,null,-43.098,-16.881,0.0,0.0,null,null],[4449.5,null,null,1969.208,632.516,null,106.32,69.803,0.0,0.0,null,1.581,1.755,2.967,null,0.0,0.0,0.0,null,-42.344,-24.052,0.0,0.0,null,null],[4450.0,null,null,1965.267,638.008,null,107.203,68.954,0.0,0.0,null,1.567,1.746,2.75,null,0.0,0.0,0.0,null,-41.454,-27.759,0.0,0.0,null,null],[4450.5,null,null,1963.922,645.356,null,108.085,67.412,0.0,0.0,null,1.55,1.727,2.519,null,0.0,0.0,0.0,null,-40.56,-32.133,0.0,0.0,null,null],[4451.0,null,null,1966.62,651.525,null,108.911,64.877,0.0,0.0,null,1.535,1.698,2.418,null,0.0,0.0,0.0,null,-40.297,-37.764,0.0,0.0,null,null],[4451.5,null,null,1971.657,657.148,null,109.627,61.983,0.0,0.0,null,1.522,1.666,2.524,null,0.0,0.0,0.0,null,-40.879,-40.785,0.0,0.0,null,null],[4452.0,null,null,1973.111,665.586,null,110.167,59.062,0.0,0.0,null,1.502,1.637,2.698,null,0.0,0.0,0.0,null,-41.765,-40.061,0.0,0.0,null,null],[4452.5,null,null,1970.158,675.847,null,110.508,56.261,0.0,0.0,null,1.48,1.607,2.823,null,0.0,0.0,0.0,null,-42.394,-37.736,0.0,0.0,null,null],[4453.0,null,null,1967.728,684.316,null,110.678,55.15,0.0,0.0,null,1.461,1.579,2.872,null,0.0,0.0,0.0,null,-42.454,-34.326,0.0,0.0,null,null],[4453.5,null,null,1966.791,689.138,null,110.736,58.986,0.0,0.0,null,1.451,1.561,2.776,null,0.0,0.0,0.0,null,-41.679,-28.946,0.0,0.0,null,null],[4454.0,null,null,1966.085,690.606,null,110.754,66.917,0.0,0.0,null,1.448,1.558,2.508,null,0.0,0.0,0.0,null,-40.237,-23.272,0.0,0.0,null,null],[4454.5,null,null,1967.39,689.567,null,110.753,73.691,0.0,0.0,null,1.45,1.563,2.232,null,0.0,0.0,0.0,null,-38.837,-19.991,0.0,0.0,null,null],[4455.0,null,null,1971.755,687.776,null,110.713,75.638,0.0,0.0,null,1.454,1.565,2.184,null,0.0,0.0,0.0,null,-38.092,-20.606,0.0,0.0,null,null],[4455.5,null,null,1978.009,686.349,null,110.572,73.707,0.0,0.0,null,1.457,1.559,2.335,null,0.0,0.0,0.0,null,-37.974,-23.838,0.0,0.0,null,null],[4456.0,null,null,1987.993,684.395,null,110.335,70.115,0.0,0.0,null,1.461,1.554,2.555,null,0.0,0.0,0.0,null,-37.955,-26.896,0.0,0.0,null,null],[4456.5,null,null,1998.701,681.396,null,110.172,67.041,0.0,0.0,null,1.468,1.553,2.793,null,0.0,0.0,0.0,null,-37.423,-26.656,0.0,0.0,null,null],[4457.0,null,null,2000.172,678.985,null,110.348,66.832,0.0,0.0,null,1.473,1.549,2.974,null,0.0,0.0,0.0,null,-35.95,-22.632,0.0,0.0,null,null],[4457.5,null,null,1990.079,678.595,null,111.028,71.242,0.0,0.0,null,1.474,1.531,3.036,null,0.0,0.0,0.0,null,-33.435,-17.139,0.0,0.0,null,null],[4458.0,null,null,1979.004,681.875,null,112.169,79.621,0.0,0.0,null,1.467,1.495,2.869,null,0.0,0.0,0.0,null,-29.955,-13.418,0.0,0.0,null,null],[4458.5,null,null,1974.836,692.375,null,113.646,90.878,0.0,0.0,null,1.444,1.44,2.373,null,0.0,0.0,0.0,null,-25.488,-11.491,0.0,0.0,null,null],[4459.0,null,null,1976.22,711.271,null,115.457,102.345,0.0,0.0,null,1.406,1.369,1.8,null,0.0,0.0,0.0,null,-20.294,-7.868,0.0,0.0,null,null],[4459.5,null,null,1977.847,735.742,null,117.798,110.456,0.0,0.0,null,1.359,1.293,1.439,null,0.0,0.0,0.0,null,-15.616,-2.218,0.0,0.0,null,null],[4460.0,null,null,1976.711,764.55,null,120.798,113.813,0.0,0.0,null,1.308,1.223,1.235,null,0.0,0.0,0.0,null,-12.626,-2.153,0.0,0.0,null,null],[4460.5,null,null,1974.036,794.24,null,124.091,113.306,0.0,0.0,null,1.259,1.169,1.09,null,0.0,0.0,0.0,null,-11.187,-9.698,0.0,0.0,null,null],[4461.0,null,null,1973.516,818.513,null,126.868,110.132,0.0,0.0,null,1.222,1.136,0.996,null,0.0,0.0,0.0,null,-10.585,-15.501,0.0,0.0,null,null],[4461.5,null,null,1979.721,834.056,null,128.395,106.913,0.0,0.0,null,1.199,1.124,0.985,null,0.0,0.0,0.0,null,-10.62,-12.106,0.0,0.0,null,null],[4462.0,null,null,1987.996,839.883,null,128.457,105.948,0.0,0.0,null,1.191,1.135,1.023,null,0.0,0.0,0.0,null,-11.117,-9.252,0.0,0.0,null,null],[4462.5,null,null,1990.493,834.122,null,127.362,106.691,0.0,0.0,null,1.199,1.169,1.028,null,0.0,0.0,0.0,null,-11.759,-15.238,0.0,0.0,null,null],[4463.0,null,null,1987.364,819.289,null,125.618,106.785,0.0,0.0,null,1.221,1.216,1.081,null,0.0,0.0,0.0,null,-12.693,-21.081,0.0,0.0,null,null],[4463.5,null,null,1982.909,801.034,null,123.571,105.367,0.0,0.0,null,1.248,1.26,1.304,null,0.0,0.0,0.0,null,-14.159,-17.77,0.0,0.0,null,null],[4464.0,null,null,1980.034,783.577,null,121.41,104.292,0.0,0.0,null,1.276,1.29,1.737,null,0.0,0.0,0.0,null,-15.771,-11.76,0.0,0.0,null,null],[4464.5,null,null,1978.229,770.879,null,119.475,103.687,0.0,0.0,null,1.297,1.299,2.102,null,0.0,0.0,0.0,null,-16.89,-12.906,0.0,0.0,null,null],[4465.0,null,null,1971.591,768.976,null,118.296,101.184,0.0,0.0,null,1.3,1.278,2.245,null,0.0,0.0,0.0,null,-17.234,-22.04,0.0,0.0,null,null],[4465.5,null,null,1964.351,779.775,null,118.25,97.032,0.0,0.0,null,1.282,1.225,2.335,null,0.0,0.0,0.0,null,-16.613,-31.798,0.0,0.0,null,null],[4466.0,null,null,1968.531,798.735,null,119.328,95.129,0.0,0.0,null,1.252,1.16,2.12,null,0.0,0.0,0.0,null,-14.91,-34.615,0.0,0.0,null,null],[4466.5,null,null,1978.872,820.982,null,121.186,98.399,0.0,0.0,null,1.218,1.103,1.574,null,0.0,0.0,0.0,null,-12.476,-29.254,0.0,0.0,null,null],[4467.0,null,null,1983.97,845.431,null,123.32,104.648,0.0,0.0,null,1.183,1.062,1.142,null,0.0,0.0,0.0,null,-10.174,-19.978,0.0,0.0,null,null],[4467.5,null,null,1988.919,872.804,null,125.334,109.179,0.0,0.0,null,1.146,1.037,0.962,null,0.0,0.0,0.0,null,-8.739,-13.803,0.0,0.0,null,null],[4468.0,null,null,1996.645,904.275,null,127.018,109.972,0.0,0.0,null,1.106,1.024,0.928,null,0.0,0.0,0.0,null,-8.232,-13.388,0.0,0.0,null,null],[4468.5,null,null,1998.099,934.75,null,128.271,107.582,0.0,0.0,null,1.07,1.016,0.96,null,0.0,0.0,0.0,null,-8.211,-15.647,0.0,0.0,null,null],[4469.0,null,null,1991.773,954.975,null,129.117,103.52,0.0,0.0,null,1.047,1.007,1.039,null,0.0,0.0,0.0,null,-8.394,-15.162,0.0,0.0,null,null],[4469.5,null,null,1986.354,959.061,null,129.648,101.569,0.0,0.0,null,1.043,1.001,1.122,null,0.0,0.0,0.0,null,-8.832,-11.406,0.0,0.0,null,null],[4470.0,null,null,1988.353,945.185,null,129.855,103.563,0.0,0.0,null,1.058,1.007,1.118,null,0.0,0.0,0.0,null,-9.615,-8.601,0.0,0.0,null,null],[4470.5,null,null,1994.742,915.362,null,129.543,107.276,0.0,0.0,null,1.092,1.036,1.039,null,0.0,0.0,0.0,null,-10.704,-8.401,0.0,0.0,null,null],[4471.0,null,null,1999.858,878.674,null,128.501,110.198,0.0,0.0,null,1.138,1.091,0.942,null,0.0,0.0,0.0,null,-12.051,-7.51,0.0,0.0,null,null],[4471.5,null,null,2000.163,843.934,null,126.666,112.146,0.0,0.0,null,1.185,1.168,0.905,null,0.0,0.0,0.0,null,-13.745,-6.886,0.0,0.0,null,null],[4472.0,null,null,1996.689,810.865,null,124.201,112.636,0.0,0.0,null,1.233,1.26,1.022,null,0.0,0.0,0.0,null,-15.884,-4.425,0.0,0.0,null,null],[4472.5,null,null,1991.193,776.946,null,121.397,110.828,0.0,0.0,null,1.287,1.357,1.273,null,0.0,0.0,0.0,null,-18.424,4.747,0.0,0.0,null,null],[4473.0,null,null,1984.475,743.318,null,118.45,105.817,0.0,0.0,null,1.345,1.449,1.54,null,0.0,0.0,0.0,null,-21.396,9.158,0.0,0.0,null,null],[4473.5,null,null,1979.066,711.635,null,115.494,97.884,0.0,0.0,null,1.405,1.529,1.874,null,0.0,0.0,0.0,null,-24.981,0.817,0.0,0.0,null,null],[4474.0,null,null,1978.298,683.898,null,112.782,88.317,0.0,0.0,null,1.462,1.589,2.457,null,0.0,0.0,0.0,null,-28.887,-7.496,0.0,0.0,null,null],[4474.5,null,null,1980.825,663.774,null,110.706,79.705,0.0,0.0,null,1.507,1.623,3.236,null,0.0,0.0,0.0,null,-32.176,-5.621,0.0,0.0,null,null],[4475.0,null,null,1981.657,651.875,null,109.624,74.978,0.0,0.0,null,1.534,1.634,3.546,null,0.0,0.0,0.0,null,-33.937,-2.992,0.0,0.0,null,null],[4475.5,null,null,1980.086,647.446,null,109.604,74.967,0.0,0.0,null,1.545,1.631,3.251,null,0.0,0.0,0.0,null,-33.889,-5.765,0.0,0.0,null,null],[4476.0,null,null,1978.304,649.899,null,110.364,77.237,0.0,0.0,null,1.539,1.62,2.777,null,0.0,0.0,0.0,null,-32.52,-8.371,0.0,0.0,null,null],[4476.5,null,null,1976.978,656.283,null,111.464,79.073,0.0,0.0,null,1.524,1.605,2.472,null,0.0,0.0,0.0,null,-30.806,-6.672,0.0,0.0,null,null],[4477.0,null,null,1977.708,661.964,null,112.527,80.93,0.0,0.0,null,1.511,1.59,2.318,null,0.0,0.0,0.0,null,-29.293,-5.308,0.0,0.0,null,null],[4477.5,null,null,1981.284,665.221,null,113.273,83.562,0.0,0.0,null,1.503,1.577,2.16,null,0.0,0.0,0.0,null,-27.817,-7.143,0.0,0.0,null,null],[4478.0,null,null,1983.64,667.89,null,113.543,86.229,0.0,0.0,null,1.497,1.569,1.958,null,0.0,0.0,0.0,null,-26.238,-10.574,0.0,0.0,null,null],[4478.5,null,null,1983.158,673.601,null,113.519,88.411,0.0,0.0,null,1.485,1.562,1.867,null,0.0,0.0,0.0,null,-24.816,-13.603,0.0,0.0,null,null],[4479.0,null,null,1986.539,683.557,null,113.767,89.051,0.0,0.0,null,1.463,1.545,1.913,null,0.0,0.0,0.0,null,-23.788,-15.167,0.0,0.0,null,null],[4479.5,null,null,1995.002,694.423,null,114.727,88.162,0.0,0.0,null,1.44,1.51,2.056,null,0.0,0.0,0.0,null,-23.189,-15.399,0.0,0.0,null,null],[4480.0,null,null,2000.501,702.27,null,116.198,87.506,0.0,0.0,null,1.424,1.463,2.186,null,0.0,0.0,0.0,null,-22.945,-16.272,0.0,0.0,null,null],[4480.5,null,null,2002.823,705.73,null,117.577,88.926,0.0,0.0,null,1.417,1.419,2.141,null,0.0,0.0,0.0,null,-22.585,-19.129,0.0,0.0,null,null],[4481.0,null,null,2005.301,706.202,null,118.467,91.511,0.0,0.0,null,1.416,1.395,1.792,null,0.0,0.0,0.0,null,-21.416,-22.433,0.0,0.0,null,null],[4481.5,null,null,2003.423,706.391,null,118.895,94.623,0.0,0.0,null,1.416,1.398,1.396,null,0.0,0.0,0.0,null,-19.631,-23.867,0.0,0.0,null,null],[4482.0,null,null,1996.467,706.617,null,119.091,96.929,0.0,0.0,null,1.415,1.417,1.21,null,0.0,0.0,0.0,null,-18.645,-23.754,0.0,0.0,null,null],[4482.5,null,null,1993.34,706.954,null,118.994,96.834,0.0,0.0,null,1.415,1.442,1.251,null,0.0,0.0,0.0,null,-19.473,-22.8,0.0,0.0,null,null],[4483.0,null,null,1996.578,707.907,null,118.231,94.935,0.0,0.0,null,1.413,1.471,1.388,null,0.0,0.0,0.0,null,-21.627,-19.931,0.0,0.0,null,null],[4483.5,null,null,1998.591,709.272,null,116.712,93.476,0.0,0.0,null,1.41,1.501,1.498,null,0.0,0.0,0.0,null,-24.392,-16.878,0.0,0.0,null,null],[4484.0,null,null,1992.792,709.578,null,114.855,92.131,0.0,0.0,null,1.409,1.518,1.666,null,0.0,0.0,0.0,null,-27.936,-16.588,0.0,0.0,null,null],[4484.5,null,null,1983.408,709.749,null,113.146,89.83,0.0,0.0,null,1.409,1.512,2.012,null,0.0,0.0,0.0,null,-32.279,-18.04,0.0,0.0,null,null],[4485.0,null,null,1979.971,711.332,null,111.841,87.423,0.0,0.0,null,1.406,1.489,2.506,null,0.0,0.0,0.0,null,-36.395,-19.171,0.0,0.0,null,null],[4485.5,null,null,1985.151,715.054,null,111.067,85.837,0.0,0.0,null,1.398,1.462,2.927,null,0.0,0.0,0.0,null,-39.207,-21.251,0.0,0.0,null,null],[4486.0,null,null,1990.633,719.394,null,110.867,85.191,0.0,0.0,null,1.39,1.439,3.12,null,0.0,0.0,0.0,null,-40.451,-23.14,0.0,0.0,null,null],[4486.5,null,null,1989.583,722.391,null,111.285,84.732,0.0,0.0,null,1.384,1.419,3.186,null,0.0,0.0,0.0,null,-40.316,-20.712,0.0,0.0,null,null],[4487.0,null,null,1988.148,721.811,null,112.199,84.109,0.0,0.0,null,1.385,1.408,3.228,null,0.0,0.0,0.0,null,-38.832,-18.103,0.0,0.0,null,null],[4487.5,null,null,1992.354,716.161,null,113.104,84.085,0.0,0.0,null,1.396,1.412,3.007,null,0.0,0.0,0.0,null,-35.944,-21.825,0.0,0.0,null,null],[4488.0,null,null,1995.613,706.309,null,113.583,85.25,0.0,0.0,null,1.416,1.436,2.358,null,0.0,0.0,0.0,null,-31.889,-25.559,0.0,0.0,null,null],[4488.5,null,null,1994.066,696.491,null,113.845,87.083,0.0,0.0,null,1.436,1.467,1.822,null,0.0,0.0,0.0,null,-27.147,-22.231,0.0,0.0,null,null],[4489.0,null,null,1992.956,691.705,null,114.523,89.084,0.0,0.0,null,1.446,1.478,1.578,null,0.0,0.0,0.0,null,-22.349,-17.685,0.0,0.0,null,null],[4489.5,null,null,1993.353,695.547,null,116.04,92.449,0.0,0.0,null,1.438,1.443,1.584,null,0.0,0.0,0.0,null,-18.594,-15.991,0.0,0.0,null,null],[4490.0,null,null,1993.053,706.39,null,118.231,97.191,0.0,0.0,null,1.416,1.372,1.775,null,0.0,0.0,0.0,null,-16.814,-12.837,0.0,0.0,null,null],[4490.5,null,null,1996.351,720.277,null,120.467,102.123,0.0,0.0,null,1.388,1.302,1.803,null,0.0,0.0,0.0,null,-15.987,-9.63,0.0,0.0,null,null],[4491.0,null,null,2003.75,734.757,null,122.265,105.715,0.0,0.0,null,1.361,1.256,1.457,null,0.0,0.0,0.0,null,-14.312,-12.554,0.0,0.0,null,null],[4491.5,null,null,2010.904,749.759,null,123.746,107.447,0.0,0.0,null,1.334,1.227,1.101,null,0.0,0.0,0.0,null,-11.941,-17.238,0.0,0.0,null,null],[4492.0,null,null,2015.447,766.852,null,125.282,107.458,0.0,0.0,null,1.304,1.205,1.029,null,0.0,0.0,0.0,null,-10.262,-17.297,0.0,0.0,null,null],[4492.5,null,null,2016.632,786.291,null,126.849,107.828,0.0,0.0,null,1.272,1.186,1.125,null,0.0,0.0,0.0,null,-9.488,-15.615,0.0,0.0,null,null],[4493.0,null,null,2016.837,802.945,null,127.964,108.973,0.0,0.0,null,1.245,1.172,1.175,null,0.0,0.0,0.0,null,-9.055,-15.893,0.0,0.0,null,null],[4493.5,null,null,2025.237,812.991,null,128.236,109.039,0.0,0.0,null,1.23,1.161,1.199,null,0.0,0.0,0.0,null,-8.743,-16.809,0.0,0.0,null,null],[4494.0,null,null,2038.476,818.141,null,127.867,108.453,0.0,0.0,null,1.222,1.15,1.257,null,0.0,0.0,0.0,null,-8.544,-19.559,0.0,0.0,null,null],[4494.5,null,null,2039.841,820.339,null,127.443,108.723,0.0,0.0,null,1.219,1.138,1.321,null,0.0,0.0,0.0,null,-8.421,-24.836,0.0,0.0,null,null],[4495.0,null,null,2026.329,820.229,null,127.302,110.401,0.0,0.0,null,1.219,1.127,1.333,null,0.0,0.0,0.0,null,-8.319,-28.011,0.0,0.0,null,null],[4495.5,null,null,2009.133,821.82,null,127.305,111.881,0.0,0.0,null,1.217,1.115,1.284,null,0.0,0.0,0.0,null,-8.21,-25.777,0.0,0.0,null,null],[4496.0,null,null,1997.128,828.618,null,127.214,112.114,0.0,0.0,null,1.207,1.106,1.241,null,0.0,0.0,0.0,null,-8.113,-21.748,0.0,0.0,null,null],[4496.5,null,null,1990.812,839.352,null,127.128,111.821,0.0,0.0,null,1.191,1.096,1.226,null,0.0,0.0,0.0,null,-8.092,-19.222,0.0,0.0,null,null],[4497.0,null,null,1985.192,851.914,null,127.358,111.809,0.0,0.0,null,1.174,1.083,1.229,null,0.0,0.0,0.0,null,-8.157,-16.324,0.0,0.0,null,null],[4497.5,null,null,1978.229,864.815,null,128.027,112.475,0.0,0.0,null,1.156,1.067,1.216,null,0.0,0.0,0.0,null,-8.224,-11.482,0.0,0.0,null,null],[4498.0,null,null,1975.585,873.131,null,128.898,113.433,0.0,0.0,null,1.145,1.055,1.19,null,0.0,0.0,0.0,null,-8.235,-7.109,0.0,0.0,null,null],[4498.5,null,null,1980.526,871.248,null,129.605,113.809,0.0,0.0,null,1.148,1.055,1.146,null,0.0,0.0,0.0,null,-8.293,-3.478,0.0,0.0,null,null],[4499.0,null,null,1995.625,858.145,null,129.913,113.39,0.0,0.0,null,1.165,1.076,1.087,null,0.0,0.0,0.0,null,-8.55,-0.39,0.0,0.0,null,null],[4499.5,null,null,2013.943,835.29,null,129.711,113.315,0.0,0.0,null,1.197,1.121,1.038,null,0.0,0.0,0.0,null,-8.987,0.58,0.0,0.0,null,null],[4500.0,null,null,2016.264,803.282,null,128.911,112.602,0.0,0.0,null,1.245,1.186,1.051,null,0.0,0.0,0.0,null,-9.406,4.589,0.0,0.0,null,null],[4500.5,null,null,1996.642,766.207,null,127.389,109.651,0.0,0.0,null,1.305,1.265,1.144,null,0.0,0.0,0.0,null,-9.712,15.195,0.0,0.0,null,null],[4501.0,null,null,1974.32,729.847,null,125.134,104.687,0.0,0.0,null,1.37,1.351,1.344,null,0.0,0.0,0.0,null,-10.075,21.627,0.0,0.0,null,null],[4501.5,null,null,1964.126,697.564,null,122.463,99.391,0.0,0.0,null,1.434,1.43,1.699,null,0.0,0.0,0.0,null,-10.619,13.857,0.0,0.0,null,null],[4502.0,null,null,1962.193,671.214,null,120.0,95.0,0.0,0.0,null,1.49,1.484,2.057,null,0.0,0.0,0.0,null,-11.099,-0.322,0.0,0.0,null,null],[4502.5,null,null,1963.078,654.649,null,118.376,91.823,0.0,0.0,null,1.528,1.504,2.225,null,0.0,0.0,0.0,null,-11.123,-12.467,0.0,0.0,null,null],[4503.0,null,null,1967.116,652.312,null,117.913,90.71,0.0,0.0,null,1.533,1.482,2.187,null,0.0,0.0,0.0,null,-10.646,-20.719,0.0,0.0,null,null],[4503.5,null,null,1973.215,666.882,null,118.575,92.511,0.0,0.0,null,1.5,1.414,2.086,null,0.0,0.0,0.0,null,-9.978,-24.029,0.0,0.0,null,null],[4504.0,null,null,1973.962,694.916,null,120.118,96.088,0.0,0.0,null,1.439,1.318,1.925,null,0.0,0.0,0.0,null,-9.346,-24.639,0.0,0.0,null,null],[4504.5,null,null,1969.788,730.285,null,122.274,100.543,0.0,0.0,null,1.369,1.227,1.642,null,0.0,0.0,0.0,null,-8.72,-26.001,0.0,0.0,null,null],[4505.0,null,null,1968.962,770.761,null,124.778,105.817,0.0,0.0,null,1.297,1.16,1.299,null,0.0,0.0,0.0,null,-8.043,-25.343,0.0,0.0,null,null],[4505.5,null,null,1975.048,816.619,null,127.264,110.432,0.0,0.0,null,1.225,1.11,1.08,null,0.0,0.0,0.0,null,-7.425,-18.331,0.0,0.0,null,null],[4506.0,null,null,1982.153,863.031,null,129.257,112.525,0.0,0.0,null,1.159,1.067,1.043,null,0.0,0.0,0.0,null,-7.002,-8.335,0.0,0.0,null,null],[4506.5,null,null,1987.329,902.104,null,130.349,111.762,0.0,0.0,null,1.109,1.028,1.092,null,0.0,0.0,0.0,null,-6.784,-2.492,0.0,0.0,null,null],[4507.0,null,null,1990.592,929.585,null,130.448,110.381,0.0,0.0,null,1.076,0.998,1.102,null,0.0,0.0,0.0,null,-6.713,-3.524,0.0,0.0,null,null],[4507.5,null,null,1990.961,945.552,null,129.898,110.841,0.0,0.0,null,1.058,0.98,1.085,null,0.0,0.0,0.0,null,-6.698,-6.94,0.0,0.0,null,null],[4508.0,null,null,1989.745,952.876,null,129.292,111.558,0.0,0.0,null,1.049,0.972,1.085,null,0.0,0.0,0.0,null,-6.605,-4.278,0.0,0.0,null,null],[4508.5,null,null,1989.628,957.059,null,129.062,110.246,0.0,0.0,null,1.045,0.968,1.098,null,0.0,0.0,0.0,null,-6.448,2.534,0.0,0.0,null,null],[4509.0,null,null,1987.609,962.717,null,129.204,108.967,0.0,0.0,null,1.039,0.965,1.112,null,0.0,0.0,0.0,null,-6.429,3.972,0.0,0.0,null,null],[4509.5,null,null,1982.078,969.63,null,129.44,109.528,0.0,0.0,null,1.031,0.962,1.138,null,0.0,0.0,0.0,null,-6.561,2.672,0.0,0.0,null,null],[4510.0,null,null,1978.025,973.945,null,129.568,111.424,0.0,0.0,null,1.027,0.957,1.166,null,0.0,0.0,0.0,null,-6.637,4.995,0.0,0.0,null,null],[4510.5,null,null,1976.345,974.254,null,129.641,113.914,0.0,0.0,null,1.026,0.954,1.151,null,0.0,0.0,0.0,null,-6.669,1.818,0.0,0.0,null,null],[4511.0,null,null,1974.556,969.781,null,129.776,115.107,0.0,0.0,null,1.031,0.955,1.103,null,0.0,0.0,0.0,null,-6.843,-8.805,0.0,0.0,null,null],[4511.5,null,null,1975.604,957.607,null,129.878,113.363,0.0,0.0,null,1.044,0.964,1.066,null,0.0,0.0,0.0,null,-7.129,-15.922,0.0,0.0,null,null],[4512.0,null,null,1981.035,933.719,null,129.584,110.756,0.0,0.0,null,1.071,0.988,1.053,null,0.0,0.0,0.0,null,-7.401,-16.568,0.0,0.0,null,null],[4512.5,null,null,1985.102,899.314,null,128.59,109.105,0.0,0.0,null,1.112,1.032,1.023,null,0.0,0.0,0.0,null,-7.66,-18.352,0.0,0.0,null,null],[4513.0,null,null,1985.304,858.792,null,126.999,107.494,0.0,0.0,null,1.164,1.1,1.012,null,0.0,0.0,0.0,null,-7.997,-23.898,0.0,0.0,null,null],[4513.5,null,null,1986.883,818.174,null,125.252,104.974,0.0,0.0,null,1.222,1.182,1.107,null,0.0,0.0,0.0,null,-8.479,-29.862,0.0,0.0,null,null],[4514.0,null,null,1992.033,780.577,null,123.781,103.002,0.0,0.0,null,1.281,1.258,1.379,null,0.0,0.0,0.0,null,-9.139,-29.891,0.0,0.0,null,null],[4514.5,null,null,1995.579,746.47,null,122.723,101.841,0.0,0.0,null,1.34,1.313,1.79,null,0.0,0.0,0.0,null,-9.889,-21.319,0.0,0.0,null,null],[4515.0,null,null,1994.645,716.346,null,121.978,101.378,0.0,0.0,null,1.396,1.347,1.978,null,0.0,0.0,0.0,null,-10.452,-12.316,0.0,0.0,null,null],[4515.5,null,null,1991.84,694.491,null,121.469,102.302,0.0,0.0,null,1.44,1.364,1.853,null,0.0,0.0,0.0,null,-10.63,-11.37,0.0,0.0,null,null],[4516.0,null,null,1988.507,686.481,null,121.303,105.264,0.0,0.0,null,1.457,1.363,1.689,null,0.0,0.0,0.0,null,-10.526,-15.393,0.0,0.0,null,null],[4516.5,null,null,1986.948,693.271,null,121.702,107.878,0.0,0.0,null,1.442,1.346,1.599,null,0.0,0.0,0.0,null,-10.404,-16.575,0.0,0.0,null,null],[4517.0,null,null,1986.897,707.496,null,122.754,108.877,0.0,0.0,null,1.413,1.319,1.49,null,0.0,0.0,0.0,null,-10.507,-15.716,0.0,0.0,null,null],[4517.5,null,null,1988.324,720.677,null,124.112,109.294,0.0,0.0,null,1.388,1.297,1.35,null,0.0,0.0,0.0,null,-10.942,-17.477,0.0,0.0,null,null],[4518.0,null,null,1993.697,727.498,null,124.986,111.074,0.0,0.0,null,1.375,1.295,1.247,null,0.0,0.0,0.0,null,-11.756,-20.708,0.0,0.0,null,null],[4518.5,null,null,2001.979,724.874,null,124.72,114.06,0.0,0.0,null,1.38,1.321,1.193,null,0.0,0.0,0.0,null,-12.961,-21.535,0.0,0.0,null,null],[4519.0,null,null,2006.176,715.344,null,123.395,114.99,0.0,0.0,null,1.398,1.382,1.125,null,0.0,0.0,0.0,null,-14.451,-18.603,0.0,0.0,null,null],[4519.5,null,null,2003.425,706.375,null,121.719,110.616,0.0,0.0,null,1.416,1.467,1.088,null,0.0,0.0,0.0,null,-15.943,-13.093,0.0,0.0,null,null],[4520.0,null,null,1998.314,702.114,null,120.289,101.5,0.0,0.0,null,1.424,1.535,1.329,null,0.0,0.0,0.0,null,-17.621,-8.69,0.0,0.0,null,null],[4520.5,null,null,1993.119,703.261,null,119.09,91.568,0.0,0.0,null,1.422,1.552,2.069,null,0.0,0.0,0.0,null,-20.194,-9.07,0.0,0.0,null,null],[4521.0,null,null,1984.997,708.724,null,117.813,85.268,0.0,0.0,null,1.411,1.516,2.977,null,0.0,0.0,0.0,null,-23.655,-14.37,0.0,0.0,null,null],[4521.5,null,null,1973.458,716.074,null,116.547,84.017,0.0,0.0,null,1.397,1.45,3.392,null,0.0,0.0,0.0,null,-26.774,-21.671,0.0,0.0,null,null],[4522.0,null,null,1964.739,726.355,null,115.905,85.679,0.0,0.0,null,1.377,1.379,3.047,null,0.0,0.0,0.0,null,-28.234,-29.464,0.0,0.0,null,null],[4522.5,null,null,1962.266,745.488,null,116.359,87.297,0.0,0.0,null,1.341,1.317,2.434,null,0.0,0.0,0.0,null,-27.654,-36.457,0.0,0.0,null,null],[4523.0,null,null,1960.357,772.928,null,117.659,88.857,0.0,0.0,null,1.294,1.268,1.887,null,0.0,0.0,0.0,null,-25.232,-40.724,0.0,0.0,null,null],[4523.5,null,null,1955.556,801.725,null,119.067,91.803,0.0,0.0,null,1.247,1.23,1.56,null,0.0,0.0,0.0,null,-21.69,-41.742,0.0,0.0,null,null],[4524.0,null,null,1951.908,828.575,null,120.113,94.806,0.0,0.0,null,1.207,1.198,1.366,null,0.0,0.0,0.0,null,-18.167,-42.3,0.0,0.0,null,null],[4524.5,null,null,1953.688,855.566,null,120.913,95.205,0.0,0.0,null,1.169,1.166,1.279,null,0.0,0.0,0.0,null,-15.429,-44.495,0.0,0.0,null,null],[4525.0,null,null,1958.155,882.273,null,121.806,94.251,0.0,0.0,null,1.133,1.135,1.246,null,0.0,0.0,0.0,null,-13.575,-45.875,0.0,0.0,null,null],[4525.5,null,null,1959.663,903.339,null,122.856,94.012,0.0,0.0,null,1.107,1.105,1.262,null,0.0,0.0,0.0,null,-12.676,-46.094,0.0,0.0,null,null],[4526.0,null,null,1957.472,914.45,null,123.783,94.014,0.0,0.0,null,1.094,1.081,1.278,null,0.0,0.0,0.0,null,-12.42,-47.145,0.0,0.0,null,null],[4526.5,null,null,1954.848,915.683,null,124.346,94.821,0.0,0.0,null,1.092,1.072,1.209,null,0.0,0.0,0.0,null,-11.944,-47.089,0.0,0.0,null,null],[4527.0,null,null,1952.987,907.73,null,124.549,97.482,0.0,0.0,null,1.102,1.084,1.084,null,0.0,0.0,0.0,null,-11.167,-42.01,0.0,0.0,null,null],[4527.5,null,null,1952.657,893.354,null,124.477,99.597,0.0,0.0,null,1.119,1.112,1.014,null,0.0,0.0,0.0,null,-10.667,-31.559,0.0,0.0,null,null],[4528.0,null,null,1956.373,878.519,null,124.118,98.175,0.0,0.0,null,1.138,1.148,1.067,null,0.0,0.0,0.0,null,-10.631,-22.869,0.0,0.0,null,null],[4528.5,null,null,1964.161,863.856,null,123.41,95.089,0.0,0.0,null,1.158,1.183,1.205,null,0.0,0.0,0.0,null,-10.951,-22.916,0.0,0.0,null,null],[4529.0,null,null,1969.961,844.468,null,122.313,93.39,0.0,0.0,null,1.184,1.215,1.376,null,0.0,0.0,0.0,null,-11.628,-28.496,0.0,0.0,null,null],[4529.5,null,null,1970.802,820.499,null,120.844,92.995,0.0,0.0,null,1.219,1.247,1.538,null,0.0,0.0,0.0,null,-12.624,-30.215,0.0,0.0,null,null],[4530.0,null,null,1972.553,796.626,null,119.079,92.295,0.0,0.0,null,1.255,1.275,1.679,null,0.0,0.0,0.0,null,-13.758,-26.027,0.0,0.0,null,null],[4530.5,null,null,1979.9,774.404,null,117.181,90.765,0.0,0.0,null,1.291,1.301,1.731,null,0.0,0.0,0.0,null,-14.531,-20.155,0.0,0.0,null,null],[4531.0,null,null,1985.174,756.628,null,115.552,88.636,0.0,0.0,null,1.322,1.325,1.698,null,0.0,0.0,0.0,null,-14.71,-14.622,0.0,0.0,null,null],[4531.5,null,null,1980.769,748.985,null,114.757,86.906,0.0,0.0,null,1.335,1.339,1.646,null,0.0,0.0,0.0,null,-14.585,-7.482,0.0,0.0,null,null],[4532.0,null,null,1972.4,752.967,null,115.088,85.387,0.0,0.0,null,1.328,1.333,1.747,null,0.0,0.0,0.0,null,-14.592,2.473,0.0,0.0,null,null],[4532.5,null,null,1972.329,762.239,null,116.218,84.521,0.0,0.0,null,1.312,1.304,1.982,null,0.0,0.0,0.0,null,-14.729,9.178,0.0,0.0,null,null],[4533.0,null,null,1979.933,770.334,null,117.421,86.127,0.0,0.0,null,1.298,1.267,2.014,null,0.0,0.0,0.0,null,-14.241,4.482,0.0,0.0,null,null],[4533.5,null,null,1984.663,775.619,null,118.243,90.64,0.0,0.0,null,1.289,1.236,1.685,null,0.0,0.0,0.0,null,-12.524,-6.61,0.0,0.0,null,null],[4534.0,null,null,1981.885,780.904,null,118.932,95.086,0.0,0.0,null,1.281,1.217,1.338,null,0.0,0.0,0.0,null,-10.269,-13.728,0.0,0.0,null,null],[4534.5,null,null,1972.4,790.322,null,120.055,97.044,0.0,0.0,null,1.265,1.201,1.217,null,0.0,0.0,0.0,null,-8.625,-12.525,0.0,0.0,null,null],[4535.0,null,null,1958.853,804.754,null,121.677,97.026,0.0,0.0,null,1.243,1.18,1.219,null,0.0,0.0,0.0,null,-7.83,-5.335,0.0,0.0,null,null],[4535.5,null,null,1949.763,818.234,null,123.198,97.265,0.0,0.0,null,1.222,1.153,1.236,null,0.0,0.0,0.0,null,-7.612,0.363,0.0,0.0,null,null],[4536.0,null,null,1950.875,824.482,null,123.993,97.656,0.0,0.0,null,1.213,1.131,1.236,null,0.0,0.0,0.0,null,-7.663,-2.097,0.0,0.0,null,null],[4536.5,null,null,1956.124,822.384,null,124.079,97.382,0.0,0.0,null,1.216,1.125,1.19,null,0.0,0.0,0.0,null,-7.725,-5.751,0.0,0.0,null,null],[4537.0,null,null,1960.523,815.287,null,124.026,98.135,0.0,0.0,null,1.227,1.136,1.134,null,0.0,0.0,0.0,null,-7.696,-4.174,0.0,0.0,null,null],[4537.5,null,null,1966.182,805.943,null,124.261,100.544,0.0,0.0,null,1.241,1.158,1.138,null,0.0,0.0,0.0,null,-7.618,0.578,0.0,0.0,null,null],[4538.0,null,null,1974.412,796.656,null,124.651,103.032,0.0,0.0,null,1.255,1.181,1.196,null,0.0,0.0,0.0,null,-7.551,8.506,0.0,0.0,null,null],[4538.5,null,null,1980.37,787.343,null,124.781,104.143,0.0,0.0,null,1.27,1.203,1.262,null,0.0,0.0,0.0,null,-7.583,16.117,0.0,0.0,null,null],[4539.0,null,null,1982.932,777.414,null,124.458,102.699,0.0,0.0,null,1.286,1.221,1.374,null,0.0,0.0,0.0,null,-7.742,15.91,0.0,0.0,null,null],[4539.5,null,null,1989.883,765.579,null,123.832,99.855,0.0,0.0,null,1.306,1.234,1.493,null,0.0,0.0,0.0,null,-7.924,9.93,0.0,0.0,null,null],[4540.0,null,null,1998.217,753.08,null,123.165,98.325,0.0,0.0,null,1.328,1.244,1.507,null,0.0,0.0,0.0,null,-8.052,5.258,0.0,0.0,null,null],[4540.5,null,null,2002.656,742.781,null,122.566,98.848,0.0,0.0,null,1.346,1.252,1.417,null,0.0,0.0,0.0,null,-8.142,1.888,0.0,0.0,null,null],[4541.0,null,null,2007.436,736.892,null,122.05,100.605,0.0,0.0,null,1.357,1.257,1.352,null,0.0,0.0,0.0,null,-8.203,-0.614,0.0,0.0,null,null],[4541.5,null,null,2013.334,735.544,null,121.685,102.437,0.0,0.0,null,1.36,1.259,1.352,null,0.0,0.0,0.0,null,-8.263,-3.457,0.0,0.0,null,null],[4542.0,null,null,2012.165,738.553,null,121.565,102.539,0.0,0.0,null,1.354,1.259,1.368,null,0.0,0.0,0.0,null,-8.34,-9.964,0.0,0.0,null,null],[4542.5,null,null,2007.684,743.245,null,121.715,101.839,0.0,0.0,null,1.345,1.259,1.365,null,0.0,0.0,0.0,null,-8.384,-16.88,0.0,0.0,null,null],[4543.0,null,null,2009.03,747.641,null,122.065,102.403,0.0,0.0,null,1.338,1.257,1.352,null,0.0,0.0,0.0,null,-8.414,-17.88,0.0,0.0,null,null],[4543.5,null,null,2012.541,751.149,null,122.528,102.975,0.0,0.0,null,1.331,1.254,1.354,null,0.0,0.0,0.0,null,-8.465,-17.105,0.0,0.0,null,null],[4544.0,null,null,2011.388,752.224,null,123.083,102.088,0.0,0.0,null,1.329,1.25,1.358,null,0.0,0.0,0.0,null,-8.511,-21.066,0.0,0.0,null,null],[4544.5,null,null,2009.955,748.069,null,123.743,100.924,0.0,0.0,null,1.337,1.251,1.34,null,0.0,0.0,0.0,null,-8.569,-26.127,0.0,0.0,null,null],[4545.0,null,null,2009.356,738.435,null,124.411,100.774,0.0,0.0,null,1.354,1.26,1.316,null,0.0,0.0,0.0,null,-8.695,-29.521,0.0,0.0,null,null],[4545.5,null,null,2000.992,726.455,null,124.891,101.281,0.0,0.0,null,1.377,1.28,1.291,null,0.0,0.0,0.0,null,-8.848,-30.432,0.0,0.0,null,null],[4546.0,null,null,1985.209,714.594,null,124.998,101.848,0.0,0.0,null,1.399,1.309,1.313,null,0.0,0.0,0.0,null,-8.999,-25.886,0.0,0.0,null,null],[4546.5,null,null,1974.474,703.974,null,124.616,101.485,0.0,0.0,null,1.421,1.342,1.382,null,0.0,0.0,0.0,null,-9.152,-19.443,0.0,0.0,null,null],[4547.0,null,null,1971.324,696.513,null,123.84,99.417,0.0,0.0,null,1.436,1.371,1.469,null,0.0,0.0,0.0,null,-9.262,-18.538,0.0,0.0,null,null],[4547.5,null,null,1971.794,693.516,null,122.997,95.891,0.0,0.0,null,1.442,1.388,1.531,null,0.0,0.0,0.0,null,-9.309,-21.603,0.0,0.0,null,null],[4548.0,null,null,1974.59,693.863,null,122.463,92.877,0.0,0.0,null,1.441,1.383,1.628,null,0.0,0.0,0.0,null,-9.36,-23.818,0.0,0.0,null,null],[4548.5,null,null,1976.516,696.509,null,122.377,92.351,0.0,0.0,null,1.436,1.358,1.75,null,0.0,0.0,0.0,null,-9.387,-25.6,0.0,0.0,null,null],[4549.0,null,null,1979.478,702.392,null,122.56,95.167,0.0,0.0,null,1.424,1.325,1.699,null,0.0,0.0,0.0,null,-9.178,-26.531,0.0,0.0,null,null],[4549.5,null,null,1986.984,711.517,null,122.729,99.896,0.0,0.0,null,1.405,1.295,1.478,null,0.0,0.0,0.0,null,-8.707,-24.419,0.0,0.0,null,null],[4550.0,null,null,1992.925,725.004,null,122.82,103.907,0.0,0.0,null,1.379,1.271,1.305,null,0.0,0.0,0.0,null,-8.24,-19.992,0.0,0.0,null,null],[4550.5,null,null,1990.167,743.619,null,123.054,105.954,0.0,0.0,null,1.345,1.25,1.246,null,0.0,0.0,0.0,null,-7.928,-13.456,0.0,0.0,null,null],[4551.0,null,null,1986.432,765.835,null,123.579,107.368,0.0,0.0,null,1.306,1.223,1.271,null,0.0,0.0,0.0,null,-7.699,-3.512,0.0,0.0,null,null],[4551.5,null,null,1990.228,788.181,null,124.27,109.275,0.0,0.0,null,1.269,1.19,1.33,null,0.0,0.0,0.0,null,-7.532,4.192,0.0,0.0,null,null],[4552.0,null,null,1994.349,809.063,null,124.884,110.242,0.0,0.0,null,1.236,1.154,1.365,null,0.0,0.0,0.0,null,-7.475,1.108,0.0,0.0,null,null],[4552.5,null,null,1993.915,828.093,null,125.402,109.632,0.0,0.0,null,1.208,1.123,1.316,null,0.0,0.0,0.0,null,-7.494,-11.169,0.0,0.0,null,null],[4553.0,null,null,1993.2,844.167,null,126.07,110.247,0.0,0.0,null,1.185,1.099,1.197,null,0.0,0.0,0.0,null,-7.545,-21.235,0.0,0.0,null,null],[4553.5,null,null,1990.726,854.431,null,127.061,113.26,0.0,0.0,null,1.17,1.083,1.123,null,0.0,0.0,0.0,null,-7.655,-22.293,0.0,0.0,null,null],[4554.0,null,null,1982.78,856.106,null,128.203,115.307,0.0,0.0,null,1.168,1.079,1.097,null,0.0,0.0,0.0,null,-7.891,-18.958,0.0,0.0,null,null],[4554.5,null,null,1977.998,847.575,null,129.055,114.624,0.0,0.0,null,1.18,1.099,1.04,null,0.0,0.0,0.0,null,-8.247,-19.578,0.0,0.0,null,null],[4555.0,null,null,1986.447,829.955,null,129.159,112.274,0.0,0.0,null,1.205,1.146,0.963,null,0.0,0.0,0.0,null,-8.717,-24.19,0.0,0.0,null,null],[4555.5,null,null,1998.255,806.726,null,128.186,108.6,0.0,0.0,null,1.24,1.209,0.972,null,0.0,0.0,0.0,null,-9.46,-28.498,0.0,0.0,null,null],[4556.0,null,null,1999.245,781.636,null,126.102,102.814,0.0,0.0,null,1.279,1.272,1.218,null,0.0,0.0,0.0,null,-10.497,-28.67,0.0,0.0,null,null],[4556.5,null,null,1992.714,759.125,null,123.386,98.082,0.0,0.0,null,1.317,1.324,1.613,null,0.0,0.0,0.0,null,-11.364,-22.057,0.0,0.0,null,null],[4557.0,null,null,1986.399,745.584,null,120.954,98.967,0.0,0.0,null,1.341,1.355,1.915,null,0.0,0.0,0.0,null,-11.642,-14.05,0.0,0.0,null,null],[4557.5,null,null,1980.755,745.842,null,119.584,102.922,0.0,0.0,null,1.341,1.344,2.16,null,0.0,0.0,0.0,null,-11.654,-11.712,0.0,0.0,null,null],[4558.0,null,null,1977.23,760.51,null,119.398,104.656,0.0,0.0,null,1.315,1.279,2.497,null,0.0,0.0,0.0,null,-11.883,-14.604,0.0,0.0,null,null],[4558.5,null,null,1975.863,786.047,null,120.033,104.281,0.0,0.0,null,1.272,1.187,2.513,null,0.0,0.0,0.0,null,-11.805,-17.875,0.0,0.0,null,null],[4559.0,null,null,1979.761,818.422,null,121.261,106.339,0.0,0.0,null,1.222,1.105,1.835,null,0.0,0.0,0.0,null,-10.522,-18.749,0.0,0.0,null,null],[4559.5,null,null,1992.221,858.15,null,123.278,111.868,0.0,0.0,null,1.165,1.046,1.158,null,0.0,0.0,0.0,null,-8.359,-20.606,0.0,0.0,null,null],[4560.0,null,null,2002.004,906.776,null,126.191,117.096,0.0,0.0,null,1.103,1.0,0.893,null,0.0,0.0,0.0,null,-6.597,-26.997,0.0,0.0,null,null],[4560.5,null,null,1994.441,958.838,null,129.415,118.074,0.0,0.0,null,1.043,0.96,0.893,null,0.0,0.0,0.0,null,-5.771,-29.769,0.0,0.0,null,null],[4561.0,null,null,1977.096,1004.395,null,131.905,116.678,0.0,0.0,null,0.996,0.928,0.944,null,0.0,0.0,0.0,null,-5.499,-24.338,0.0,0.0,null,null],[4561.5,null,null,1967.385,1037.875,null,133.039,116.124,0.0,0.0,null,0.964,0.907,0.958,null,0.0,0.0,0.0,null,-5.437,-19.128,0.0,0.0,null,null],[4562.0,null,null,1969.695,1059.296,null,133.055,115.758,0.0,0.0,null,0.944,0.895,0.971,null,0.0,0.0,0.0,null,-5.453,-16.786,0.0,0.0,null,null],[4562.5,null,null,1973.929,1068.762,null,132.634,114.65,0.0,0.0,null,0.936,0.888,0.989,null,0.0,0.0,0.0,null,-5.503,-11.379,0.0,0.0,null,null],[4563.0,null,null,1971.201,1065.743,null,132.3,113.429,0.0,0.0,null,0.938,0.885,1.001,null,0.0,0.0,0.0,null,-5.62,-6.675,0.0,0.0,null,null],[4563.5,null,null,1967.165,1051.018,null,132.172,112.421,0.0,0.0,null,0.951,0.889,0.992,null,0.0,0.0,0.0,null,-5.83,-8.367,0.0,0.0,null,null],[4564.0,null,null,1968.971,1025.453,null,132.076,112.35,0.0,0.0,null,0.975,0.905,0.978,null,0.0,0.0,0.0,null,-6.115,-13.604,0.0,0.0,null,null],[4564.5,null,null,1975.281,991.045,null,131.755,112.891,0.0,0.0,null,1.009,0.939,0.962,null,0.0,0.0,0.0,null,-6.458,-15.014,0.0,0.0,null,null],[4565.0,null,null,1982.578,952.84,null,131.04,111.921,0.0,0.0,null,1.049,0.989,0.945,null,0.0,0.0,0.0,null,-6.819,-10.599,0.0,0.0,null,null],[4565.5,null,null,1984.035,916.0,null,130.03,109.57,0.0,0.0,null,1.092,1.047,1.005,null,0.0,0.0,0.0,null,-7.21,-4.31,0.0,0.0,null,null],[4566.0,null,null,1973.643,880.863,null,129.086,108.057,0.0,0.0,null,1.135,1.103,1.209,null,0.0,0.0,0.0,null,-7.713,-1.303,0.0,0.0,null,null],[4566.5,null,null,1959.408,844.609,null,128.442,107.527,0.0,0.0,null,1.184,1.156,1.437,null,0.0,0.0,0.0,null,-8.285,-3.644,0.0,0.0,null,null],[4567.0,null,null,1955.619,806.815,null,127.824,106.599,0.0,0.0,null,1.239,1.206,1.534,null,0.0,0.0,0.0,null,-8.848,-9.423,0.0,0.0,null,null],[4567.5,null,null,1967.471,770.919,null,126.729,105.349,0.0,0.0,null,1.297,1.256,1.504,null,0.0,0.0,0.0,null,-9.392,-15.571,0.0,0.0,null,null],[4568.0,null,null,1984.482,742.452,null,125.139,104.099,0.0,0.0,null,1.347,1.295,1.488,null,0.0,0.0,0.0,null,-9.862,-19.62,0.0,0.0,null,null],[4568.5,null,null,1988.44,724.392,null,123.668,104.184,0.0,0.0,null,1.38,1.317,1.512,null,0.0,0.0,0.0,null,-10.185,-20.047,0.0,0.0,null,null],[4569.0,null,null,1985.167,714.942,null,122.914,105.586,0.0,0.0,null,1.399,1.323,1.538,null,0.0,0.0,0.0,null,-10.38,-17.381,0.0,0.0,null,null],[4569.5,null,null,1989.302,711.0,null,122.845,106.09,0.0,0.0,null,1.406,1.32,1.548,null,0.0,0.0,0.0,null,-10.478,-15.02,0.0,0.0,null,null],[4570.0,null,null,1990.49,712.378,null,122.921,105.795,0.0,0.0,null,1.404,1.312,1.529,null,0.0,0.0,0.0,null,-10.498,-14.794,0.0,0.0,null,null],[4570.5,null,null,1981.527,717.252,null,122.736,105.907,0.0,0.0,null,1.394,1.299,1.493,null,0.0,0.0,0.0,null,-10.474,-15.857,0.0,0.0,null,null],[4571.0,null,null,1974.668,722.853,null,122.434,105.428,0.0,0.0,null,1.383,1.288,1.44,null,0.0,0.0,0.0,null,-10.382,-17.728,0.0,0.0,null,null],[4571.5,null,null,1973.646,728.936,null,122.364,103.545,0.0,0.0,null,1.372,1.279,1.379,null,0.0,0.0,0.0,null,-10.23,-20.042,0.0,0.0,null,null],[4572.0,null,null,1976.23,735.416,null,122.555,102.279,0.0,0.0,null,1.36,1.268,1.356,null,0.0,0.0,0.0,null,-10.15,-19.583,0.0,0.0,null,null],[4572.5,null,null,1981.717,740.803,null,122.76,102.872,0.0,0.0,null,1.35,1.259,1.381,null,0.0,0.0,0.0,null,-10.248,-16.33,0.0,0.0,null,null],[4573.0,null,null,1985.338,744.521,null,122.779,103.965,0.0,0.0,null,1.343,1.254,1.381,null,0.0,0.0,0.0,null,-10.435,-14.824,0.0,0.0,null,null],[4573.5,null,null,1986.083,744.866,null,122.634,105.262,0.0,0.0,null,1.343,1.256,1.33,null,0.0,0.0,0.0,null,-10.629,-15.759,0.0,0.0,null,null],[4574.0,null,null,1988.173,741.151,null,122.535,106.982,0.0,0.0,null,1.349,1.27,1.266,null,0.0,0.0,0.0,null,-10.93,-12.99,0.0,0.0,null,null],[4574.5,null,null,1993.486,733.361,null,122.647,107.548,0.0,0.0,null,1.364,1.294,1.248,null,0.0,0.0,0.0,null,-11.536,-7.567,0.0,0.0,null,null],[4575.0,null,null,1998.684,721.155,null,122.766,106.513,0.0,0.0,null,1.387,1.329,1.296,null,0.0,0.0,0.0,null,-12.659,-4.091,0.0,0.0,null,null],[4575.5,null,null,1998.21,704.065,null,122.383,106.031,0.0,0.0,null,1.42,1.38,1.325,null,0.0,0.0,0.0,null,-14.34,-1.801,0.0,0.0,null,null],[4576.0,null,null,1998.087,684.005,null,121.12,104.689,0.0,0.0,null,1.462,1.452,1.274,null,0.0,0.0,0.0,null,-16.376,-1.442,0.0,0.0,null,null],[4576.5,null,null,1998.896,666.552,null,119.047,100.242,0.0,0.0,null,1.5,1.543,1.222,null,0.0,0.0,0.0,null,-18.588,-6.044,0.0,0.0,null,null],[4577.0,null,null,1994.04,656.513,null,116.64,93.268,0.0,0.0,null,1.523,1.626,1.297,null,0.0,0.0,0.0,null,-21.132,-10.333,0.0,0.0,null,null],[4577.5,null,null,1986.945,653.673,null,114.343,85.522,0.0,0.0,null,1.53,1.669,1.542,null,0.0,0.0,0.0,null,-24.626,-7.644,0.0,0.0,null,null],[4578.0,null,null,1988.909,654.92,null,112.229,77.98,0.0,0.0,null,1.527,1.666,1.94,null,0.0,0.0,0.0,null,-29.41,-2.311,0.0,0.0,null,null],[4578.5,null,null,1993.598,659.529,null,110.261,72.496,0.0,0.0,null,1.516,1.638,2.355,null,0.0,0.0,0.0,null,-34.684,-2.984,0.0,0.0,null,null],[4579.0,null,null,1993.379,668.187,null,108.724,69.058,0.0,0.0,null,1.497,1.599,2.546,null,0.0,0.0,0.0,null,-39.099,-10.54,0.0,0.0,null,null],[4579.5,null,null,1992.342,683.065,null,108.079,65.834,0.0,0.0,null,1.464,1.555,2.508,null,0.0,0.0,0.0,null,-42.241,-16.708,0.0,0.0,null,null],[4580.0,null,null,1990.281,702.848,null,108.41,62.39,0.0,0.0,null,1.423,1.508,2.536,null,0.0,0.0,0.0,null,-44.549,-15.621,0.0,0.0,null,null],[4580.5,null,null,1983.344,721.779,null,109.191,59.809,0.0,0.0,null,1.385,1.462,2.779,null,0.0,0.0,0.0,null,-46.303,-10.044,0.0,0.0,null,null],[4581.0,null,null,1976.637,735.594,null,109.652,58.272,0.0,0.0,null,1.359,1.426,3.233,null,0.0,0.0,0.0,null,-47.25,-7.253,0.0,0.0,null,null],[4581.5,null,null,1976.401,744.467,null,109.555,57.283,0.0,0.0,null,1.343,1.408,3.553,null,0.0,0.0,0.0,null,-46.906,-8.794,0.0,0.0,null,null],[4582.0,null,null,1979.096,747.705,null,109.466,58.291,0.0,0.0,null,1.337,1.409,3.382,null,0.0,0.0,0.0,null,-45.036,-7.959,0.0,0.0,null,null],[4582.5,null,null,1980.827,745.595,null,110.1,62.134,0.0,0.0,null,1.341,1.412,3.087,null,0.0,0.0,0.0,null,-41.828,-3.662,0.0,0.0,null,null],[4583.0,null,null,1981.787,741.53,null,111.471,67.954,0.0,0.0,null,1.349,1.399,2.898,null,0.0,0.0,0.0,null,-37.745,-2.694,0.0,0.0,null,null],[4583.5,null,null,1984.199,737.025,null,112.91,74.77,0.0,0.0,null,1.357,1.372,2.616,null,0.0,0.0,0.0,null,-33.006,-5.656,0.0,0.0,null,null],[4584.0,null,null,1992.41,730.627,null,113.943,83.887,0.0,0.0,null,1.369,1.348,2.083,null,0.0,0.0,0.0,null,-27.281,-7.206,0.0,0.0,null,null],[4584.5,null,null,1999.501,727.02,null,114.928,95.852,0.0,0.0,null,1.375,1.336,1.472,null,0.0,0.0,0.0,null,-20.675,-7.549,0.0,0.0,null,null],[4585.0,null,null,1997.502,737.138,null,116.684,107.336,0.0,0.0,null,1.357,1.322,1.058,null,0.0,0.0,0.0,null,-14.689,-7.82,0.0,0.0,null,null],[4585.5,null,null,1991.082,766.066,null,119.438,112.698,0.0,0.0,null,1.305,1.275,1.02,null,0.0,0.0,0.0,null,-11.217,-4.189,0.0,0.0,null,null],[4586.0,null,null,1988.211,808.632,null,122.387,111.598,0.0,0.0,null,1.237,1.191,1.323,null,0.0,0.0,0.0,null,-10.406,3.246,0.0,0.0,null,null],[4586.5,null,null,1988.168,857.208,null,124.465,109.962,0.0,0.0,null,1.167,1.101,1.54,null,0.0,0.0,0.0,null,-10.241,10.778,0.0,0.0,null,null],[4587.0,null,null,1988.681,906.582,null,125.523,111.892,0.0,0.0,null,1.103,1.028,1.302,null,0.0,0.0,0.0,null,-9.114,14.684,0.0,0.0,null,null],[4587.5,null,null,1990.146,954.372,null,126.543,115.786,0.0,0.0,null,1.048,0.973,1.003,null,0.0,0.0,0.0,null,-7.6,13.513,0.0,0.0,null,null],[4588.0,null,null,1990.14,1000.057,null,128.451,118.506,0.0,0.0,null,1.0,0.929,0.911,null,0.0,0.0,0.0,null,-6.929,13.506,0.0,0.0,null,null],[4588.5,null,null,1987.617,1038.27,null,130.951,118.438,0.0,0.0,null,0.963,0.898,0.898,null,0.0,0.0,0.0,null,-7.289,23.168,0.0,0.0,null,null],[4589.0,null,null,1984.658,1056.431,null,132.751,115.207,0.0,0.0,null,0.947,0.886,0.867,null,0.0,0.0,0.0,null,-8.399,33.595,0.0,0.0,null,null],[4589.5,null,null,1982.419,1046.139,null,132.816,111.666,0.0,0.0,null,0.956,0.904,0.838,null,0.0,0.0,0.0,null,-9.941,30.866,0.0,0.0,null,null],[4590.0,null,null,1981.996,1011.811,null,131.128,110.054,0.0,0.0,null,0.988,0.963,0.738,null,0.0,0.0,0.0,null,-11.453,17.209,0.0,0.0,null,null],[4590.5,null,null,1989.098,966.464,null,128.366,107.633,0.0,0.0,null,1.035,1.058,0.657,null,0.0,0.0,0.0,null,-13.063,4.319,0.0,0.0,null,null],[4591.0,null,null,2004.564,920.202,null,125.14,99.903,0.0,0.0,null,1.087,1.16,0.881,null,0.0,0.0,0.0,null,-15.88,-3.373,0.0,0.0,null,null],[4591.5,null,null,2020.087,875.615,null,121.643,87.73,0.0,0.0,null,1.142,1.244,1.863,null,0.0,0.0,0.0,null,-20.085,-7.803,0.0,0.0,null,null],[4592.0,null,null,2029.297,826.921,null,118.03,77.743,0.0,0.0,null,1.209,1.308,3.319,null,0.0,0.0,0.0,null,-23.771,-13.043,0.0,0.0,null,null],[4592.5,null,null,2026.998,773.162,null,114.798,74.384,0.0,0.0,null,1.293,1.364,3.853,null,0.0,0.0,0.0,null,-25.221,-18.21,0.0,0.0,null,null],[4593.0,null,null,2014.033,727.561,null,112.66,76.038,0.0,0.0,null,1.374,1.403,3.412,null,0.0,0.0,0.0,null,-24.558,-19.629,0.0,0.0,null,null],[4593.5,null,null,2000.327,704.588,null,112.126,79.832,0.0,0.0,null,1.419,1.41,3.009,null,0.0,0.0,0.0,null,-22.564,-13.402,0.0,0.0,null,null],[4594.0,null,null,1996.541,704.171,null,113.231,85.757,0.0,0.0,null,1.42,1.383,2.611,null,0.0,0.0,0.0,null,-19.768,2.785,0.0,0.0,null,null],[4594.5,null,null,1999.099,719.792,null,115.541,93.582,0.0,0.0,null,1.389,1.336,2.092,null,0.0,0.0,0.0,null,-16.545,15.483,0.0,0.0,null,null],[4595.0,null,null,1996.24,744.753,null,118.384,100.683,0.0,0.0,null,1.343,1.279,1.605,null,0.0,0.0,0.0,null,-13.383,8.886,0.0,0.0,null,null],[4595.5,null,null,1986.11,772.427,null,121.182,105.249,0.0,0.0,null,1.295,1.226,1.298,null,0.0,0.0,0.0,null,-10.956,-10.495,0.0,0.0,null,null],[4596.0,null,null,1978.883,797.843,null,123.584,109.001,0.0,0.0,null,1.253,1.185,1.147,null,0.0,0.0,0.0,null,-9.546,-23.139,0.0,0.0,null,null],[4596.5,null,null,1981.095,818.405,null,125.488,113.678,0.0,0.0,null,1.222,1.158,1.093,null,0.0,0.0,0.0,null,-8.839,-26.75,0.0,0.0,null,null],[4597.0,null,null,1981.997,833.044,null,126.952,116.669,0.0,0.0,null,1.2,1.14,1.101,null,0.0,0.0,0.0,null,-8.555,-27.58,0.0,0.0,null,null],[4597.5,null,null,1975.913,845.706,null,128.052,115.413,0.0,0.0,null,1.182,1.123,1.154,null,0.0,0.0,0.0,null,-8.603,-27.437,0.0,0.0,null,null],[4598.0,null,null,1966.921,858.793,null,128.797,113.073,0.0,0.0,null,1.164,1.109,1.217,null,0.0,0.0,0.0,null,-8.838,-23.361,0.0,0.0,null,null],[4598.5,null,null,1959.489,867.759,null,129.213,112.433,0.0,0.0,null,1.152,1.098,1.217,null,0.0,0.0,0.0,null,-9.098,-19.65,0.0,0.0,null,null],[4599.0,null,null,1956.915,868.301,null,129.432,111.907,0.0,0.0,null,1.152,1.094,1.182,null,0.0,0.0,0.0,null,-9.323,-21.372,0.0,0.0,null,null],[4599.5,null,null,1960.108,860.868,null,129.477,110.368,0.0,0.0,null,1.162,1.1,1.149,null,0.0,0.0,0.0,null,-9.693,-25.25,0.0,0.0,null,null],[4600.0,null,null,1962.358,845.617,null,129.012,109.426,0.0,0.0,null,1.183,1.122,1.133,null,0.0,0.0,0.0,null,-10.47,-24.26,0.0,0.0,null,null],[4600.5,null,null,1958.358,823.174,null,127.554,109.696,0.0,0.0,null,1.215,1.169,1.102,null,0.0,0.0,0.0,null,-11.697,-18.438,0.0,0.0,null,null],[4601.0,null,null,1956.972,796.135,null,124.978,109.421,0.0,0.0,null,1.256,1.241,1.028,null,0.0,0.0,0.0,null,-13.191,-11.865,0.0,0.0,null,null],[4601.5,null,null,1964.792,766.872,null,121.682,107.175,0.0,0.0,null,1.304,1.331,1.015,null,0.0,0.0,0.0,null,-14.698,-6.623,0.0,0.0,null,null],[4602.0,null,null,1976.213,738.488,null,118.263,102.896,0.0,0.0,null,1.354,1.425,1.204,null,0.0,0.0,0.0,null,-16.27,-4.179,0.0,0.0,null,null],[4602.5,null,null,1982.889,715.037,null,115.136,98.047,0.0,0.0,null,1.399,1.501,1.721,null,0.0,0.0,0.0,null,-18.392,-4.781,0.0,0.0,null,null],[4603.0,null,null,1984.284,697.777,null,112.566,95.031,0.0,0.0,null,1.433,1.54,2.689,null,0.0,0.0,0.0,null,-21.182,-4.827,0.0,0.0,null,null],[4603.5,null,null,1982.871,688.94,null,110.991,93.785,0.0,0.0,null,1.452,1.53,3.875,null,0.0,0.0,0.0,null,-23.725,-4.477,0.0,0.0,null,null],[4604.0,null,null,1987.895,694.908,null,111.008,92.044,0.0,0.0,null,1.439,1.468,4.565,null,0.0,0.0,0.0,null,-24.83,-8.335,0.0,0.0,null,null],[4604.5,null,null,2000.047,718.996,null,112.811,89.2,0.0,0.0,null,1.391,1.361,4.182,null,0.0,0.0,0.0,null,-23.856,-14.354,0.0,0.0,null,null],[4605.0,null,null,2004.527,756.691,null,115.881,88.883,0.0,0.0,null,1.322,1.243,3.053,null,0.0,0.0,0.0,null,-20.765,-16.792,0.0,0.0,null,null],[4605.5,null,null,2000.916,801.05,null,119.383,93.706,0.0,0.0,null,1.248,1.143,1.943,null,0.0,0.0,0.0,null,-16.131,-18.278,0.0,0.0,null,null],[4606.0,null,null,2003.156,848.12,null,122.799,102.613,0.0,0.0,null,1.179,1.069,1.197,null,0.0,0.0,0.0,null,-11.314,-19.888,0.0,0.0,null,null],[4606.5,null,null,2007.265,896.307,null,126.033,112.113,0.0,0.0,null,1.116,1.014,0.871,null,0.0,0.0,0.0,null,-7.749,-17.272,0.0,0.0,null,null],[4607.0,null,null,2001.691,944.585,null,128.956,117.837,0.0,0.0,null,1.059,0.97,0.822,null,0.0,0.0,0.0,null,-5.904,-12.377,0.0,0.0,null,null],[4607.5,null,null,1991.487,990.721,null,131.115,117.799,0.0,0.0,null,1.009,0.937,0.871,null,0.0,0.0,0.0,null,-5.255,-11.488,0.0,0.0,null,null],[4608.0,null,null,1981.038,1029.423,null,132.138,115.385,0.0,0.0,null,0.971,0.913,0.898,null,0.0,0.0,0.0,null,-5.115,-11.988,0.0,0.0,null,null],[4608.5,null,null,1970.915,1055.234,null,132.228,114.733,0.0,0.0,null,0.948,0.897,0.945,null,0.0,0.0,0.0,null,-5.138,-7.352,0.0,0.0,null,null],[4609.0,null,null,1964.269,1068.152,null,132.008,113.845,0.0,0.0,null,0.936,0.888,1.002,null,0.0,0.0,0.0,null,-5.244,0.018,0.0,0.0,null,null],[4609.5,null,null,1963.446,1071.315,null,131.95,111.007,0.0,0.0,null,0.933,0.884,1.018,null,0.0,0.0,0.0,null,-5.43,2.351,0.0,0.0,null,null],[4610.0,null,null,1969.683,1067.391,null,132.058,108.887,0.0,0.0,null,0.937,0.883,1.027,null,0.0,0.0,0.0,null,-5.635,-0.144,0.0,0.0,null,null],[4610.5,null,null,1982.292,1055.733,null,132.042,108.973,0.0,0.0,null,0.947,0.884,1.03,null,0.0,0.0,0.0,null,-5.813,-2.582,0.0,0.0,null,null],[4611.0,null,null,2002.194,1036.258,null,131.706,109.068,0.0,0.0,null,0.965,0.891,1.002,null,0.0,0.0,0.0,null,-6.002,-3.852,0.0,0.0,null,null],[4611.5,null,null,2020.916,1009.983,null,131.08,108.47,0.0,0.0,null,0.99,0.911,0.95,null,0.0,0.0,0.0,null,-6.285,-5.274,0.0,0.0,null,null],[4612.0,null,null,2026.79,978.63,null,130.236,108.411,0.0,0.0,null,1.022,0.947,0.949,null,0.0,0.0,0.0,null,-6.694,-6.933,0.0,0.0,null,null],[4612.5,null,null,2018.321,942.551,null,129.246,108.686,0.0,0.0,null,1.061,0.997,1.01,null,0.0,0.0,0.0,null,-7.173,-4.099,0.0,0.0,null,null],[4613.0,null,null,2008.909,904.512,null,128.251,107.696,0.0,0.0,null,1.106,1.057,1.108,null,0.0,0.0,0.0,null,-7.679,4.53,0.0,0.0,null,null],[4613.5,null,null,2009.136,867.496,null,127.348,104.616,0.0,0.0,null,1.153,1.12,1.211,null,0.0,0.0,0.0,null,-8.25,11.821,0.0,0.0,null,null],[4614.0,null,null,2011.866,831.429,null,126.477,100.513,0.0,0.0,null,1.203,1.174,1.352,null,0.0,0.0,0.0,null,-8.928,12.567,0.0,0.0,null,null],[4614.5,null,null,2012.98,795.429,null,125.477,98.047,0.0,0.0,null,1.257,1.215,1.548,null,0.0,0.0,0.0,null,-9.681,9.607,0.0,0.0,null,null],[4615.0,null,null,2020.65,761.568,null,124.227,98.785,0.0,0.0,null,1.313,1.249,1.607,null,0.0,0.0,0.0,null,-10.392,5.625,0.0,0.0,null,null],[4615.5,null,null,2033.855,729.624,null,122.85,101.58,0.0,0.0,null,1.371,1.293,1.448,null,0.0,0.0,0.0,null,-10.984,0.032,0.0,0.0,null,null],[4616.0,null,null,2035.723,699.093,null,121.64,104.221,0.0,0.0,null,1.43,1.353,1.27,null,0.0,0.0,0.0,null,-11.536,-8.168,0.0,0.0,null,null],[4616.5,null,null,2021.26,673.15,null,120.562,105.51,0.0,0.0,null,1.486,1.429,1.233,null,0.0,0.0,0.0,null,-12.242,-17.689,0.0,0.0,null,null],[4617.0,null,null,2002.252,652.869,null,119.082,104.516,0.0,0.0,null,1.532,1.515,1.331,null,0.0,0.0,0.0,null,-13.333,-23.099,0.0,0.0,null,null],[4617.5,null,null,1992.321,633.246,null,116.72,100.533,0.0,0.0,null,1.579,1.615,1.486,null,0.0,0.0,0.0,null,-15.0,-24.452,0.0,0.0,null,null],[4618.0,null,null,1996.925,610.998,null,113.627,95.204,0.0,0.0,null,1.637,1.731,1.595,null,0.0,0.0,0.0,null,-17.147,-26.607,0.0,0.0,null,null],[4618.5,null,null,2006.124,588.432,null,110.547,90.414,0.0,0.0,null,1.699,1.85,1.6,null,0.0,0.0,0.0,null,-19.558,-30.636,0.0,0.0,null,null],[4619.0,null,null,2012.2,570.541,null,108.219,85.605,0.0,0.0,null,1.753,1.948,1.694,null,0.0,0.0,0.0,null,-22.477,-33.196,0.0,0.0,null,null],[4619.5,null,null,2013.862,562.048,null,106.74,78.614,0.0,0.0,null,1.779,2.0,2.016,null,0.0,0.0,0.0,null,-26.367,-32.504,0.0,0.0,null,null],[4620.0,null,null,2011.123,564.563,null,105.688,69.638,0.0,0.0,null,1.771,1.999,2.432,null,0.0,0.0,0.0,null,-31.043,-30.43,0.0,0.0,null,null],[4620.5,null,null,2003.566,576.808,null,104.834,61.782,0.0,0.0,null,1.734,1.956,2.64,null,0.0,0.0,0.0,null,-35.701,-30.927,0.0,0.0,null,null],[4621.0,null,null,1996.951,595.652,null,104.39,56.989,0.0,0.0,null,1.679,1.885,2.676,null,0.0,0.0,0.0,null,-39.879,-34.095,0.0,0.0,null,null],[4621.5,null,null,1996.202,619.549,null,104.601,53.702,0.0,0.0,null,1.614,1.796,2.794,null,0.0,0.0,0.0,null,-43.55,-36.628,0.0,0.0,null,null],[4622.0,null,null,1996.548,648.545,null,105.328,50.502,0.0,0.0,null,1.542,1.708,2.978,null,0.0,0.0,0.0,null,-46.539,-37.343,0.0,0.0,null,null],[4622.5,null,null,1995.778,679.731,null,106.1,48.253,0.0,0.0,null,1.471,1.632,3.036,null,0.0,0.0,0.0,null,-48.663,-36.705,0.0,0.0,null,null],[4623.0,null,null,1999.554,708.789,null,106.522,47.931,0.0,0.0,null,1.411,1.568,2.986,null,0.0,0.0,0.0,null,-50.138,-33.472,0.0,0.0,null,null],[4623.5,null,null,2008.329,735.074,null,106.606,49.277,0.0,0.0,null,1.36,1.511,3.031,null,0.0,0.0,0.0,null,-51.313,-28.172,0.0,0.0,null,null],[4624.0,null,null,2011.541,760.45,null,106.646,51.541,0.0,0.0,null,1.315,1.458,3.204,null,0.0,0.0,0.0,null,-52.235,-25.104,0.0,0.0,null,null],[4624.5,null,null,2008.948,783.707,null,106.847,55.123,0.0,0.0,null,1.276,1.412,3.256,null,0.0,0.0,0.0,null,-52.753,-24.917,0.0,0.0,null,null],[4625.0,null,null,2005.71,803.849,null,107.253,60.081,0.0,0.0,null,1.244,1.373,3.125,null,0.0,0.0,0.0,null,-52.787,-21.357,0.0,0.0,null,null],[4625.5,null,null,2002.161,823.694,null,107.872,64.53,0.0,0.0,null,1.214,1.339,2.944,null,0.0,0.0,0.0,null,-52.418,-13.534,0.0,0.0,null,null],[4626.0,null,null,1999.725,844.033,null,108.733,66.337,0.0,0.0,null,1.185,1.307,2.854,null,0.0,0.0,0.0,null,-51.728,-11.41,0.0,0.0,null,null],[4626.5,null,null,2001.764,861.166,null,109.842,66.004,0.0,0.0,null,1.161,1.276,2.754,null,0.0,0.0,0.0,null,-50.799,-18.043,0.0,0.0,null,null],[4627.0,null,null,2006.712,875.589,null,111.131,67.043,0.0,0.0,null,1.142,1.248,2.551,null,0.0,0.0,0.0,null,-49.633,-23.726,0.0,0.0,null,null],[4627.5,null,null,2006.674,891.334,null,112.45,71.757,0.0,0.0,null,1.122,1.225,2.35,null,0.0,0.0,0.0,null,-48.149,-20.49,0.0,0.0,null,null],[4628.0,null,null,1998.873,906.568,null,113.609,77.341,0.0,0.0,null,1.103,1.205,2.244,null,0.0,0.0,0.0,null,-46.483,-14.128,0.0,0.0,null,null],[4628.5,null,null,1989.195,916.161,null,114.502,80.199,0.0,0.0,null,1.092,1.188,2.128,null,0.0,0.0,0.0,null,-44.908,-12.266,0.0,0.0,null,null],[4629.0,null,null,1983.84,920.439,null,115.155,81.029,0.0,0.0,null,1.086,1.175,1.951,null,0.0,0.0,0.0,null,-43.259,-17.14,0.0,0.0,null,null],[4629.5,null,null,1980.237,921.954,null,115.666,81.965,0.0,0.0,null,1.085,1.17,1.762,null,0.0,0.0,0.0,null,-41.274,-27.235,0.0,0.0,null,null],[4630.0,null,null,1973.57,920.549,null,116.114,82.912,0.0,0.0,null,1.086,1.173,1.644,null,0.0,0.0,0.0,null,-39.248,-38.481,0.0,0.0,null,null],[4630.5,null,null,1962.062,916.594,null,116.496,83.99,0.0,0.0,null,1.091,1.181,1.607,null,0.0,0.0,0.0,null,-37.71,-43.11,0.0,0.0,null,null],[4631.0,null,null,1951.988,911.904,null,116.738,84.859,0.0,0.0,null,1.097,1.186,1.669,null,0.0,0.0,0.0,null,-36.844,-41.192,0.0,0.0,null,null],[4631.5,null,null,1954.22,905.196,null,116.787,84.853,0.0,0.0,null,1.105,1.189,1.822,null,0.0,0.0,0.0,null,-36.424,-38.233,0.0,0.0,null,null],[4632.0,null,null,1964.994,894.812,null,116.676,84.445,0.0,0.0,null,1.118,1.195,1.931,null,0.0,0.0,0.0,null,-35.79,-35.534,0.0,0.0,null,null],[4632.5,null,null,1974.11,884.535,null,116.479,84.709,0.0,0.0,null,1.131,1.204,1.824,null,0.0,0.0,0.0,null,-34.378,-30.498,0.0,0.0,null,null],[4633.0,null,null,1986.307,878.159,null,116.319,85.84,0.0,0.0,null,1.139,1.213,1.686,null,0.0,0.0,0.0,null,-32.602,-23.44,0.0,0.0,null,null],[4633.5,null,null,2004.969,875.007,null,116.362,86.48,0.0,0.0,null,1.143,1.215,1.765,null,0.0,0.0,0.0,null,-31.215,-19.996,0.0,0.0,null,null],[4634.0,null,null,2014.842,874.804,null,116.684,86.304,0.0,0.0,null,1.143,1.207,2.018,null,0.0,0.0,0.0,null,-30.101,-19.172,0.0,0.0,null,null],[4634.5,null,null,2008.375,878.319,null,117.219,87.117,0.0,0.0,null,1.139,1.19,2.168,null,0.0,0.0,0.0,null,-28.577,-15.29,0.0,0.0,null,null],[4635.0,null,null,2002.544,882.203,null,117.881,89.771,0.0,0.0,null,1.134,1.169,2.041,null,0.0,0.0,0.0,null,-26.228,-10.95,0.0,0.0,null,null],[4635.5,null,null,2005.259,882.984,null,118.628,92.802,0.0,0.0,null,1.133,1.15,1.673,null,0.0,0.0,0.0,null,-22.86,-7.373,0.0,0.0,null,null],[4636.0,null,null,2012.018,882.231,null,119.387,95.014,0.0,0.0,null,1.133,1.137,1.293,null,0.0,0.0,0.0,null,-19.181,-2.003,0.0,0.0,null,null],[4636.5,null,null,2020.74,883.16,null,120.019,96.482,0.0,0.0,null,1.132,1.13,1.124,null,0.0,0.0,0.0,null,-16.663,2.806,0.0,0.0,null,null],[4637.0,null,null,2021.439,884.825,null,120.374,96.89,0.0,0.0,null,1.13,1.128,1.136,null,0.0,0.0,0.0,null,-15.649,3.406,0.0,0.0,null,null],[4637.5,null,null,2004.402,885.887,null,120.473,96.499,0.0,0.0,null,1.129,1.131,1.173,null,0.0,0.0,0.0,null,-15.19,4.263,0.0,0.0,null,null],[4638.0,null,null,1982.614,887.56,null,120.534,95.426,0.0,0.0,null,1.127,1.136,1.217,null,0.0,0.0,0.0,null,-14.824,6.875,0.0,0.0,null,null],[4638.5,null,null,1970.951,889.714,null,120.696,94.093,0.0,0.0,null,1.124,1.137,1.333,null,0.0,0.0,0.0,null,-14.581,6.821,0.0,0.0,null,null],[4639.0,null,null,1962.766,888.264,null,120.841,93.848,0.0,0.0,null,1.126,1.137,1.458,null,0.0,0.0,0.0,null,-14.037,3.267,0.0,0.0,null,null],[4639.5,null,null,1952.345,878.044,null,120.852,95.306,0.0,0.0,null,1.139,1.138,1.492,null,0.0,0.0,0.0,null,-12.909,2.727,0.0,0.0,null,null],[4640.0,null,null,1948.596,860.448,null,120.825,96.838,0.0,0.0,null,1.162,1.146,1.432,null,0.0,0.0,0.0,null,-11.559,1.073,0.0,0.0,null,null],[4640.5,null,null,1959.945,842.965,null,120.934,97.37,0.0,0.0,null,1.186,1.164,1.302,null,0.0,0.0,0.0,null,-10.447,-9.197,0.0,0.0,null,null],[4641.0,null,null,1980.691,831.298,null,121.177,97.622,0.0,0.0,null,1.203,1.187,1.18,null,0.0,0.0,0.0,null,-9.796,-21.089,0.0,0.0,null,null],[4641.5,null,null,1995.673,825.321,null,121.211,97.375,0.0,0.0,null,1.212,1.202,1.215,null,0.0,0.0,0.0,null,-9.526,-26.367,0.0,0.0,null,null],[4642.0,null,null,2001.165,821.983,null,120.668,95.791,0.0,0.0,null,1.217,1.207,1.398,null,0.0,0.0,0.0,null,-9.398,-28.247,0.0,0.0,null,null],[4642.5,null,null,2003.773,817.218,null,119.692,94.219,0.0,0.0,null,1.224,1.207,1.529,null,0.0,0.0,0.0,null,-9.317,-27.386,0.0,0.0,null,null],[4643.0,null,null,2008.531,811.411,null,118.896,94.192,0.0,0.0,null,1.232,1.208,1.521,null,0.0,0.0,0.0,null,-9.231,-23.796,0.0,0.0,null,null],[4643.5,null,null,2012.934,810.004,null,118.697,95.202,0.0,0.0,null,1.235,1.202,1.45,null,0.0,0.0,0.0,null,-9.026,-21.426,0.0,0.0,null,null],[4644.0,null,null,2014.304,816.268,null,119.035,95.898,0.0,0.0,null,1.225,1.18,1.466,null,0.0,0.0,0.0,null,-8.884,-18.91,0.0,0.0,null,null],[4644.5,null,null,2011.199,830.908,null,119.683,96.044,0.0,0.0,null,1.204,1.142,1.543,null,0.0,0.0,0.0,null,-8.809,-13.369,0.0,0.0,null,null],[4645.0,null,null,2004.183,851.512,null,120.585,96.083,0.0,0.0,null,1.174,1.1,1.523,null,0.0,0.0,0.0,null,-8.308,-10.468,0.0,0.0,null,null],[4645.5,null,null,1997.599,874.069,null,121.975,96.767,0.0,0.0,null,1.144,1.065,1.34,null,0.0,0.0,0.0,null,-7.326,-13.726,0.0,0.0,null,null],[4646.0,null,null,1996.828,897.218,null,124.001,97.699,0.0,0.0,null,1.115,1.036,1.148,null,0.0,0.0,0.0,null,-6.455,-14.713,0.0,0.0,null,null],[4646.5,null,null,2002.132,919.523,null,126.275,98.343,0.0,0.0,null,1.088,1.01,1.081,null,0.0,0.0,0.0,null,-6.011,-10.072,0.0,0.0,null,null],[4647.0,null,null,2007.036,937.309,null,128.097,98.167,0.0,0.0,null,1.067,0.99,1.069,null,0.0,0.0,0.0,null,-5.867,-7.427,0.0,0.0,null,null],[4647.5,null,null,2007.633,949.059,null,129.095,97.982,0.0,0.0,null,1.054,0.978,1.07,null,0.0,0.0,0.0,null,-5.863,-8.362,0.0,0.0,null,null],[4648.0,null,null,2005.777,957.057,null,129.454,98.687,0.0,0.0,null,1.045,0.976,1.053,null,0.0,0.0,0.0,null,-5.902,-9.376,0.0,0.0,null,null],[4648.5,null,null,2002.636,965.708,null,129.587,100.788,0.0,0.0,null,1.036,0.98,1.037,null,0.0,0.0,0.0,null,-5.964,-12.768,0.0,0.0,null,null],[4649.0,null,null,2002.002,977.136,null,129.697,103.816,0.0,0.0,null,1.023,0.976,1.07,null,0.0,0.0,0.0,null,-6.063,-19.682,0.0,0.0,null,null],[4649.5,null,null,2007.713,986.908,null,129.576,106.578,0.0,0.0,null,1.013,0.962,1.165,null,0.0,0.0,0.0,null,-6.244,-19.08,0.0,0.0,null,null],[4650.0,null,null,2014.911,990.831,null,129.029,107.327,0.0,0.0,null,1.009,0.946,1.222,null,0.0,0.0,0.0,null,-6.536,-7.703,0.0,0.0,null,null],[4650.5,null,null,2018.46,989.281,null,128.363,106.256,0.0,0.0,null,1.011,0.942,1.127,null,0.0,0.0,0.0,null,-6.899,0.555,0.0,0.0,null,null],[4651.0,null,null,2017.84,981.706,null,128.021,105.131,0.0,0.0,null,1.019,0.954,0.982,null,0.0,0.0,0.0,null,-7.354,-1.668,0.0,0.0,null,null],[4651.5,null,null,2016.147,966.83,null,127.899,104.469,0.0,0.0,null,1.034,0.98,0.923,null,0.0,0.0,0.0,null,-8.09,-7.091,0.0,0.0,null,null],[4652.0,null,null,2017.726,946.766,null,127.271,104.111,0.0,0.0,null,1.056,1.022,0.947,null,0.0,0.0,0.0,null,-9.393,-13.488,0.0,0.0,null,null],[4652.5,null,null,2023.632,922.418,null,125.493,103.773,0.0,0.0,null,1.084,1.078,0.986,null,0.0,0.0,0.0,null,-11.39,-20.096,0.0,0.0,null,null],[4653.0,null,null,2022.936,892.383,null,122.702,102.41,0.0,0.0,null,1.121,1.152,0.972,null,0.0,0.0,0.0,null,-13.884,-22.132,0.0,0.0,null,null],[4653.5,null,null,2008.668,859.507,null,119.732,98.554,0.0,0.0,null,1.163,1.234,0.989,null,0.0,0.0,0.0,null,-16.856,-17.955,0.0,0.0,null,null],[4654.0,null,null,1993.03,830.581,null,117.262,91.098,0.0,0.0,null,1.204,1.302,1.163,null,0.0,0.0,0.0,null,-20.585,-8.574,0.0,0.0,null,null],[4654.5,null,null,1985.966,810.148,null,115.179,81.245,0.0,0.0,null,1.234,1.337,1.642,null,0.0,0.0,0.0,null,-25.122,1.323,0.0,0.0,null,null],[4655.0,null,null,1980.57,797.319,null,113.089,72.819,0.0,0.0,null,1.254,1.344,2.452,null,0.0,0.0,0.0,null,-29.931,1.248,0.0,0.0,null,null],[4655.5,null,null,1974.692,791.268,null,111.215,67.997,0.0,0.0,null,1.264,1.328,3.248,null,0.0,0.0,0.0,null,-34.061,-10.929,0.0,0.0,null,null],[4656.0,null,null,1974.298,792.944,null,110.434,67.062,0.0,0.0,null,1.261,1.289,4.092,null,0.0,0.0,0.0,null,-36.589,-23.847,0.0,0.0,null,null],[4656.5,null,null,1976.862,801.46,null,111.343,70.244,0.0,0.0,null,1.248,1.243,4.48,null,0.0,0.0,0.0,null,-37.026,-27.743,0.0,0.0,null,null],[4657.0,null,null,1976.545,810.808,null,113.533,77.074,0.0,0.0,null,1.233,1.207,3.7,null,0.0,0.0,0.0,null,-34.877,-22.972,0.0,0.0,null,null],[4657.5,null,null,1973.985,817.48,null,115.933,86.105,0.0,0.0,null,1.223,1.187,2.236,null,0.0,0.0,0.0,null,-29.208,-19.983,0.0,0.0,null,null],[4658.0,null,null,1974.702,822.759,null,117.81,95.991,0.0,0.0,null,1.215,1.182,1.307,null,0.0,0.0,0.0,null,-21.087,-23.709,0.0,0.0,null,null],[4658.5,null,null,1980.898,828.548,null,119.333,104.419,0.0,0.0,null,1.207,1.182,1.005,null,0.0,0.0,0.0,null,-14.036,-27.1,0.0,0.0,null,null],[4659.0,null,null,1984.621,835.087,null,120.999,109.727,0.0,0.0,null,1.197,1.178,1.011,null,0.0,0.0,0.0,null,-10.264,-25.778,0.0,0.0,null,null],[4659.5,null,null,1982.212,842.535,null,122.844,112.828,0.0,0.0,null,1.187,1.164,1.077,null,0.0,0.0,0.0,null,-8.827,-21.291,0.0,0.0,null,null],[4660.0,null,null,1982.166,850.447,null,124.52,114.921,0.0,0.0,null,1.176,1.144,1.078,null,0.0,0.0,0.0,null,-8.294,-15.264,0.0,0.0,null,null],[4660.5,null,null,1992.659,856.681,null,125.845,115.217,0.0,0.0,null,1.167,1.125,1.069,null,0.0,0.0,0.0,null,-8.286,-16.659,0.0,0.0,null,null],[4661.0,null,null,2008.874,860.004,null,126.968,113.898,0.0,0.0,null,1.163,1.111,1.09,null,0.0,0.0,0.0,null,-8.736,-29.525,0.0,0.0,null,null],[4661.5,null,null,2013.1,860.561,null,127.938,112.568,0.0,0.0,null,1.162,1.109,1.078,null,0.0,0.0,0.0,null,-9.512,-42.335,0.0,0.0,null,null],[4662.0,null,null,2002.054,856.732,null,128.344,111.405,0.0,0.0,null,1.167,1.125,1.024,null,0.0,0.0,0.0,null,-10.501,-41.468,0.0,0.0,null,null],[4662.5,null,null,1996.679,845.393,null,127.614,110.107,0.0,0.0,null,1.183,1.159,0.982,null,0.0,0.0,0.0,null,-11.713,-27.431,0.0,0.0,null,null],[4663.0,null,null,2006.34,827.439,null,125.746,107.398,0.0,0.0,null,1.209,1.202,1.007,null,0.0,0.0,0.0,null,-13.156,-15.591,0.0,0.0,null,null],[4663.5,null,null,2014.299,807.882,null,123.415,101.07,0.0,0.0,null,1.238,1.247,1.155,null,0.0,0.0,0.0,null,-14.96,-15.843,0.0,0.0,null,null],[4664.0,null,null,2010.715,793.598,null,121.373,91.831,0.0,0.0,null,1.26,1.281,1.457,null,0.0,0.0,0.0,null,-17.343,-13.957,0.0,0.0,null,null],[4664.5,null,null,2001.895,791.377,null,119.971,83.302,0.0,0.0,null,1.264,1.288,2.008,null,0.0,0.0,0.0,null,-20.194,-6.618,0.0,0.0,null,null],[4665.0,null,null,1996.099,804.475,null,119.222,78.004,0.0,0.0,null,1.243,1.251,2.904,null,0.0,0.0,0.0,null,-22.616,-9.36,0.0,0.0,null,null],[4665.5,null,null,1993.414,828.571,null,119.182,78.511,0.0,0.0,null,1.207,1.176,3.599,null,0.0,0.0,0.0,null,-23.132,-20.249,0.0,0.0,null,null],[4666.0,null,null,1995.477,858.79,null,120.099,85.483,0.0,0.0,null,1.164,1.09,2.969,null,0.0,0.0,0.0,null,-20.568,-26.466,0.0,0.0,null,null],[4666.5,null,null,2005.402,895.002,null,122.077,95.534,0.0,0.0,null,1.117,1.017,1.617,null,0.0,0.0,0.0,null,-15.616,-27.455,0.0,0.0,null,null],[4667.0,null,null,2015.037,937.102,null,124.753,104.046,0.0,0.0,null,1.067,0.96,0.956,null,0.0,0.0,0.0,null,-10.67,-24.824,0.0,0.0,null,null],[4667.5,null,null,2019.19,982.693,null,127.437,109.415,0.0,0.0,null,1.018,0.917,0.796,null,0.0,0.0,0.0,null,-7.557,-20.568,0.0,0.0,null,null],[4668.0,null,null,2021.869,1030.097,null,129.531,112.118,0.0,0.0,null,0.971,0.887,0.784,null,0.0,0.0,0.0,null,-6.219,-17.639,0.0,0.0,null,null],[4668.5,null,null,2020.688,1073.6,null,130.857,113.364,0.0,0.0,null,0.931,0.869,0.793,null,0.0,0.0,0.0,null,-5.808,-15.807,0.0,0.0,null,null],[4669.0,null,null,2013.984,1102.784,null,131.557,113.078,0.0,0.0,null,0.907,0.86,0.835,null,0.0,0.0,0.0,null,-5.81,-13.42,0.0,0.0,null,null],[4669.5,null,null,2009.762,1109.211,null,131.758,112.09,0.0,0.0,null,0.902,0.861,0.895,null,0.0,0.0,0.0,null,-6.019,-9.235,0.0,0.0,null,null],[4670.0,null,null,2006.074,1087.928,null,131.439,110.419,0.0,0.0,null,0.919,0.875,0.935,null,0.0,0.0,0.0,null,-6.33,-6.395,0.0,0.0,null,null],[4670.5,null,null,1997.149,1041.009,null,130.599,107.866,0.0,0.0,null,0.961,0.908,0.933,null,0.0,0.0,0.0,null,-6.678,-6.721,0.0,0.0,null,null],[4671.0,null,null,1990.691,982.085,null,129.386,105.078,0.0,0.0,null,1.018,0.963,0.915,null,0.0,0.0,0.0,null,-7.03,-5.58,0.0,0.0,null,null],[4671.5,null,null,1994.806,927.324,null,128.088,102.605,0.0,0.0,null,1.078,1.028,0.971,null,0.0,0.0,0.0,null,-7.441,-5.889,0.0,0.0,null,null],[4672.0,null,null,2003.388,884.506,null,126.946,99.152,0.0,0.0,null,1.131,1.086,1.178,null,0.0,0.0,0.0,null,-8.022,-11.069,0.0,0.0,null,null],[4672.5,null,null,2006.811,853.778,null,125.961,95.501,0.0,0.0,null,1.171,1.128,1.478,null,0.0,0.0,0.0,null,-8.664,-14.676,0.0,0.0,null,null],[4673.0,null,null,2006.16,831.563,null,125.008,94.107,0.0,0.0,null,1.203,1.154,1.631,null,0.0,0.0,0.0,null,-9.081,-11.792,0.0,0.0,null,null],[4673.5,null,null,2011.054,815.88,null,124.091,94.438,0.0,0.0,null,1.226,1.163,1.595,null,0.0,0.0,0.0,null,-9.273,-9.521,0.0,0.0,null,null],[4674.0,null,null,2021.751,809.428,null,123.398,94.886,0.0,0.0,null,1.235,1.159,1.524,null,0.0,0.0,0.0,null,-9.481,-8.726,0.0,0.0,null,null],[4674.5,null,null,2028.912,812.711,null,123.129,96.276,0.0,0.0,null,1.23,1.145,1.431,null,0.0,0.0,0.0,null,-9.776,-8.118,0.0,0.0,null,null],[4675.0,null,null,2025.126,820.934,null,123.305,99.639,0.0,0.0,null,1.218,1.132,1.274,null,0.0,0.0,0.0,null,-10.125,-11.662,0.0,0.0,null,null],[4675.5,null,null,2014.227,829.817,null,123.688,102.908,0.0,0.0,null,1.205,1.13,1.128,null,0.0,0.0,0.0,null,-10.665,-19.539,0.0,0.0,null,null],[4676.0,null,null,2009.002,836.517,null,123.872,104.295,0.0,0.0,null,1.195,1.144,1.042,null,0.0,0.0,0.0,null,-11.595,-23.385,0.0,0.0,null,null],[4676.5,null,null,2012.07,836.562,null,123.457,104.029,0.0,0.0,null,1.195,1.176,1.022,null,0.0,0.0,0.0,null,-13.075,-20.685,0.0,0.0,null,null],[4677.0,null,null,2011.743,829.135,null,122.149,102.776,0.0,0.0,null,1.206,1.227,1.036,null,0.0,0.0,0.0,null,-15.185,-18.4,0.0,0.0,null,null],[4677.5,null,null,2005.102,818.964,null,119.878,100.74,0.0,0.0,null,1.221,1.29,1.061,null,0.0,0.0,0.0,null,-17.81,-19.494,0.0,0.0,null,null],[4678.0,null,null,2001.221,809.77,null,116.932,97.506,0.0,0.0,null,1.235,1.35,1.109,null,0.0,0.0,0.0,null,-20.908,-20.783,0.0,0.0,null,null],[4678.5,null,null,2001.993,801.327,null,113.844,91.7,0.0,0.0,null,1.248,1.39,1.264,null,0.0,0.0,0.0,null,-24.834,-16.988,0.0,0.0,null,null],[4679.0,null,null,2001.96,792.296,null,111.066,83.511,0.0,0.0,null,1.262,1.401,1.756,null,0.0,0.0,0.0,null,-30.017,-9.805,0.0,0.0,null,null],[4679.5,null,null,1998.388,780.483,null,108.757,75.834,0.0,0.0,null,1.281,1.394,2.674,null,0.0,0.0,0.0,null,-35.811,-7.428,0.0,0.0,null,null],[4680.0,null,null,1992.447,765.123,null,106.884,71.533,0.0,0.0,null,1.307,1.391,3.266,null,0.0,0.0,0.0,null,-40.494,-11.183,0.0,0.0,null,null],[4680.5,null,null,1989.577,747.912,null,105.4,70.833,0.0,0.0,null,1.337,1.406,3.243,null,0.0,0.0,0.0,null,-42.96,-14.86,0.0,0.0,null,null],[4681.0,null,null,1992.027,731.871,null,104.425,71.174,0.0,0.0,null,1.366,1.435,3.027,null,0.0,0.0,0.0,null,-43.459,-15.264,0.0,0.0,null,null],[4681.5,null,null,1996.004,720.626,null,104.29,70.392,0.0,0.0,null,1.388,1.463,2.962,null,0.0,0.0,0.0,null,-42.472,-15.044,0.0,0.0,null,null],[4682.0,null,null,1998.491,719.001,null,105.295,69.901,0.0,0.0,null,1.391,1.475,2.988,null,0.0,0.0,0.0,null,-40.109,-15.33,0.0,0.0,null,null],[4682.5,null,null,1998.652,729.569,null,107.494,71.483,0.0,0.0,null,1.371,1.455,2.983,null,0.0,0.0,0.0,null,-36.795,-15.711,0.0,0.0,null,null],[4683.0,null,null,2001.352,750.729,null,110.586,73.88,0.0,0.0,null,1.332,1.391,2.955,null,0.0,0.0,0.0,null,-32.976,-16.605,0.0,0.0,null,null],[4683.5,null,null,2008.139,776.707,null,113.963,76.989,0.0,0.0,null,1.287,1.293,2.759,null,0.0,0.0,0.0,null,-28.31,-17.896,0.0,0.0,null,null],[4684.0,null,null,2014.181,802.3,null,117.039,84.507,0.0,0.0,null,1.246,1.195,2.053,null,0.0,0.0,0.0,null,-22.285,-20.102,0.0,0.0,null,null],[4684.5,null,null,2010.95,826.206,null,119.643,96.695,0.0,0.0,null,1.21,1.125,1.237,null,0.0,0.0,0.0,null,-15.72,-22.672,0.0,0.0,null,null],[4685.0,null,null,2000.483,852.19,null,122.01,107.948,0.0,0.0,null,1.173,1.083,0.88,null,0.0,0.0,0.0,null,-10.463,-20.55,0.0,0.0,null,null],[4685.5,null,null,1992.807,882.658,null,124.327,113.759,0.0,0.0,null,1.133,1.059,0.829,null,0.0,0.0,0.0,null,-7.45,-8.483,0.0,0.0,null,null],[4686.0,null,null,1990.592,916.183,null,126.403,114.197,0.0,0.0,null,1.091,1.04,0.895,null,0.0,0.0,0.0,null,-6.15,6.657,0.0,0.0,null,null],[4686.5,null,null,1988.762,949.201,null,127.759,112.161,0.0,0.0,null,1.054,1.019,0.996,null,0.0,0.0,0.0,null,-5.717,16.034,0.0,0.0,null,null],[4687.0,null,null,1988.439,975.801,null,128.133,111.051,0.0,0.0,null,1.025,0.993,1.153,null,0.0,0.0,0.0,null,-5.686,18.087,0.0,0.0,null,null],[4687.5,null,null,1991.025,989.148,null,127.852,110.89,0.0,0.0,null,1.011,0.967,1.275,null,0.0,0.0,0.0,null,-5.722,12.723,0.0,0.0,null,null],[4688.0,null,null,1993.007,989.135,null,127.587,110.311,0.0,0.0,null,1.011,0.948,1.239,null,0.0,0.0,0.0,null,-5.615,2.426,0.0,0.0,null,null],[4688.5,null,null,1995.772,982.309,null,127.772,110.205,0.0,0.0,null,1.018,0.937,1.119,null,0.0,0.0,0.0,null,-5.386,-4.595,0.0,0.0,null,null],[4689.0,null,null,2003.661,976.901,null,128.305,111.435,0.0,0.0,null,1.024,0.935,1.049,null,0.0,0.0,0.0,null,-5.195,-4.611,0.0,0.0,null,null],[4689.5,null,null,2012.427,979.76,null,128.845,112.177,0.0,0.0,null,1.021,0.937,1.044,null,0.0,0.0,0.0,null,-5.155,-4.101,0.0,0.0,null,null],[4690.0,null,null,2012.915,990.45,null,129.204,111.749,0.0,0.0,null,1.01,0.939,1.051,null,0.0,0.0,0.0,null,-5.3,-10.027,0.0,0.0,null,null],[4690.5,null,null,2004.594,999.305,null,129.375,111.079,0.0,0.0,null,1.001,0.944,1.048,null,0.0,0.0,0.0,null,-5.58,-19.754,0.0,0.0,null,null],[4691.0,null,null,2001.672,995.563,null,129.291,110.473,0.0,0.0,null,1.004,0.955,1.045,null,0.0,0.0,0.0,null,-5.909,-22.984,0.0,0.0,null,null],[4691.5,null,null,2007.571,972.928,null,128.701,109.829,0.0,0.0,null,1.028,0.977,1.043,null,0.0,0.0,0.0,null,-6.302,-18.853,0.0,0.0,null,null],[4692.0,null,null,2007.9,931.565,null,127.274,109.474,0.0,0.0,null,1.073,1.016,1.032,null,0.0,0.0,0.0,null,-6.91,-19.967,0.0,0.0,null,null],[4692.5,null,null,1999.49,879.959,null,124.889,109.057,0.0,0.0,null,1.136,1.08,1.011,null,0.0,0.0,0.0,null,-7.765,-29.146,0.0,0.0,null,null],[4693.0,null,null,1996.545,828.86,null,121.863,106.638,0.0,0.0,null,1.206,1.171,0.999,null,0.0,0.0,0.0,null,-8.539,-38.974,0.0,0.0,null,null],[4693.5,null,null,2004.722,783.682,null,118.765,101.385,0.0,0.0,null,1.276,1.28,1.1,null,0.0,0.0,0.0,null,-9.127,-43.466,0.0,0.0,null,null],[4694.0,null,null,2015.024,743.755,null,116.131,94.481,0.0,0.0,null,1.345,1.384,1.534,null,0.0,0.0,0.0,null,-10.189,-42.601,0.0,0.0,null,null],[4694.5,null,null,2018.375,710.229,null,114.29,88.41,0.0,0.0,null,1.408,1.456,2.345,null,0.0,0.0,0.0,null,-12.035,-39.537,0.0,0.0,null,null],[4695.0,null,null,2011.61,684.748,null,113.357,84.999,0.0,0.0,null,1.46,1.483,3.011,null,0.0,0.0,0.0,null,-13.58,-40.825,0.0,0.0,null,null],[4695.5,null,null,1997.999,668.219,null,113.341,84.776,0.0,0.0,null,1.497,1.47,3.03,null,0.0,0.0,0.0,null,-13.774,-45.013,0.0,0.0,null,null],[4696.0,null,null,1987.973,662.96,null,114.259,87.229,0.0,0.0,null,1.508,1.428,2.49,null,0.0,0.0,0.0,null,-12.831,-43.734,0.0,0.0,null,null],[4696.5,null,null,1988.762,671.897,null,116.058,91.885,0.0,0.0,null,1.488,1.374,1.952,null,0.0,0.0,0.0,null,-11.426,-34.83,0.0,0.0,null,null],[4697.0,null,null,1993.145,692.153,null,118.429,98.38,0.0,0.0,null,1.445,1.323,1.514,null,0.0,0.0,0.0,null,-10.03,-25.48,0.0,0.0,null,null],[4697.5,null,null,1991.969,716.866,null,120.862,105.296,0.0,0.0,null,1.395,1.281,1.237,null,0.0,0.0,0.0,null,-9.014,-20.945,0.0,0.0,null,null],[4698.0,null,null,1990.64,741.503,null,122.814,110.819,0.0,0.0,null,1.349,1.25,1.162,null,0.0,0.0,0.0,null,-8.493,-20.289,0.0,0.0,null,null],[4698.5,null,null,1993.494,761.93,null,123.921,114.263,0.0,0.0,null,1.312,1.23,1.18,null,0.0,0.0,0.0,null,-8.282,-19.361,0.0,0.0,null,null],[4699.0,null,null,1994.421,773.983,null,124.173,114.713,0.0,0.0,null,1.292,1.226,1.206,null,0.0,0.0,0.0,null,-8.225,-12.145,0.0,0.0,null,null],[4699.5,null,null,1992.564,775.976,null,123.879,112.085,0.0,0.0,null,1.289,1.233,1.22,null,0.0,0.0,0.0,null,-8.286,-5.48,0.0,0.0,null,null],[4700.0,null,null,1990.838,770.459,null,123.448,108.481,0.0,0.0,null,1.298,1.241,1.287,null,0.0,0.0,0.0,null,-8.534,-7.958,0.0,0.0,null,null],[4700.5,null,null,1988.457,759.95,null,123.151,105.721,0.0,0.0,null,1.316,1.247,1.441,null,0.0,0.0,0.0,null,-8.944,-9.762,0.0,0.0,null,null],[4701.0,null,null,1987.584,747.79,null,123.081,103.744,0.0,0.0,null,1.337,1.256,1.531,null,0.0,0.0,0.0,null,-9.202,-3.827,0.0,0.0,null,null],[4701.5,null,null,1989.366,738.119,null,123.189,102.68,0.0,0.0,null,1.355,1.266,1.517,null,0.0,0.0,0.0,null,-9.099,-1.651,0.0,0.0,null,null],[4702.0,null,null,1985.833,734.714,null,123.385,102.781,0.0,0.0,null,1.361,1.269,1.493,null,0.0,0.0,0.0,null,-8.803,-8.823,0.0,0.0,null,null],[4702.5,null,null,1977.507,737.487,null,123.627,103.87,0.0,0.0,null,1.356,1.259,1.524,null,0.0,0.0,0.0,null,-8.523,-18.885,0.0,0.0,null,null],[4703.0,null,null,1977.137,744.386,null,123.9,105.364,0.0,0.0,null,1.343,1.237,1.496,null,0.0,0.0,0.0,null,-8.234,-29.581,0.0,0.0,null,null],[4703.5,null,null,1987.354,755.313,null,124.171,106.851,0.0,0.0,null,1.324,1.209,1.414,null,0.0,0.0,0.0,null,-7.93,-38.512,0.0,0.0,null,null],[4704.0,null,null,1998.335,770.257,null,124.42,108.216,0.0,0.0,null,1.298,1.181,1.353,null,0.0,0.0,0.0,null,-7.66,-42.723,0.0,0.0,null,null],[4704.5,null,null,2002.598,787.814,null,124.696,109.886,0.0,0.0,null,1.269,1.158,1.312,null,0.0,0.0,0.0,null,-7.42,-41.031,0.0,0.0,null,null],[4705.0,null,null,2001.746,806.947,null,125.073,111.234,0.0,0.0,null,1.239,1.14,1.258,null,0.0,0.0,0.0,null,-7.182,-35.775,0.0,0.0,null,null],[4705.5,null,null,2001.735,826.91,null,125.563,110.716,0.0,0.0,null,1.209,1.127,1.188,null,0.0,0.0,0.0,null,-7.011,-25.309,0.0,0.0,null,null],[4706.0,null,null,2007.331,843.748,null,126.073,109.118,0.0,0.0,null,1.185,1.115,1.175,null,0.0,0.0,0.0,null,-7.01,-8.434,0.0,0.0,null,null],[4706.5,null,null,2012.771,851.7,null,126.412,109.261,0.0,0.0,null,1.174,1.105,1.221,null,0.0,0.0,0.0,null,-7.148,5.317,0.0,0.0,null,null],[4707.0,null,null,2012.565,848.156,null,126.407,110.505,0.0,0.0,null,1.179,1.103,1.245,null,0.0,0.0,0.0,null,-7.301,7.693,0.0,0.0,null,null],[4707.5,null,null,2009.493,837.306,null,126.013,109.936,0.0,0.0,null,1.194,1.115,1.217,null,0.0,0.0,0.0,null,-7.414,0.435,0.0,0.0,null,null],[4708.0,null,null,2007.037,825.99,null,125.355,107.675,0.0,0.0,null,1.211,1.14,1.173,null,0.0,0.0,0.0,null,-7.459,-6.311,0.0,0.0,null,null],[4708.5,null,null,2002.844,818.612,null,124.664,106.029,0.0,0.0,null,1.222,1.17,1.11,null,0.0,0.0,0.0,null,-7.392,-8.404,0.0,0.0,null,null],[4709.0,null,null,2002.104,818.736,null,124.135,104.819,0.0,0.0,null,1.221,1.183,1.161,null,0.0,0.0,0.0,null,-7.4,-8.062,0.0,0.0,null,null],[4709.5,null,null,2007.466,825.507,null,123.81,102.265,0.0,0.0,null,1.211,1.163,1.522,null,0.0,0.0,0.0,null,-7.797,-4.025,0.0,0.0,null,null],[4710.0,null,null,2006.856,831.269,null,123.635,100.04,0.0,0.0,null,1.203,1.127,1.901,null,0.0,0.0,0.0,null,-8.282,6.615,0.0,0.0,null,null],[4710.5,null,null,1997.512,831.041,null,123.644,101.516,0.0,0.0,null,1.203,1.103,1.674,null,0.0,0.0,0.0,null,-8.083,14.859,0.0,0.0,null,null],[4711.0,null,null,1995.82,831.051,null,124.024,105.362,0.0,0.0,null,1.203,1.097,1.13,null,0.0,0.0,0.0,null,-7.177,9.226,0.0,0.0,null,null],[4711.5,null,null,2008.593,841.193,null,124.954,106.919,0.0,0.0,null,1.189,1.09,0.964,null,0.0,0.0,0.0,null,-6.349,-4.529,0.0,0.0,null,null],[4712.0,null,null,2023.842,864.065,null,126.353,105.829,0.0,0.0,null,1.157,1.069,1.124,null,0.0,0.0,0.0,null,-6.015,-15.351,0.0,0.0,null,null],[4712.5,null,null,2028.139,891.828,null,127.844,105.013,0.0,0.0,null,1.121,1.04,1.294,null,0.0,0.0,0.0,null,-5.929,-17.304,0.0,0.0,null,null],[4713.0,null,null,2019.531,913.311,null,128.994,106.856,0.0,0.0,null,1.095,1.016,1.27,null,0.0,0.0,0.0,null,-5.825,-12.199,0.0,0.0,null,null],[4713.5,null,null,2011.628,924.003,null,129.6,109.931,0.0,0.0,null,1.082,1.004,1.143,null,0.0,0.0,0.0,null,-5.69,-5.906,0.0,0.0,null,null],[4714.0,null,null,2011.26,926.886,null,129.779,111.502,0.0,0.0,null,1.079,1.0,1.041,null,0.0,0.0,0.0,null,-5.605,-1.125,0.0,0.0,null,null],[4714.5,null,null,2010.634,926.396,null,129.826,111.047,0.0,0.0,null,1.079,1.0,1.022,null,0.0,0.0,0.0,null,-5.607,-0.138,0.0,0.0,null,null],[4715.0,null,null,2005.17,925.421,null,129.941,110.948,0.0,0.0,null,1.081,1.002,1.052,null,0.0,0.0,0.0,null,-5.653,-4.017,0.0,0.0,null,null],[4715.5,null,null,1999.816,924.391,null,130.072,112.633,0.0,0.0,null,1.082,1.008,1.106,null,0.0,0.0,0.0,null,-5.706,-10.426,0.0,0.0,null,null],[4716.0,null,null,1996.737,919.872,null,130.047,114.736,0.0,0.0,null,1.087,1.014,1.168,null,0.0,0.0,0.0,null,-5.743,-12.187,0.0,0.0,null,null],[4716.5,null,null,1994.943,908.476,null,129.794,115.288,0.0,0.0,null,1.101,1.022,1.175,null,0.0,0.0,0.0,null,-5.715,-9.838,0.0,0.0,null,null],[4717.0,null,null,1995.486,893.848,null,129.402,114.57,0.0,0.0,null,1.119,1.034,1.156,null,0.0,0.0,0.0,null,-5.621,-9.606,0.0,0.0,null,null],[4717.5,null,null,2000.799,884.18,null,129.054,114.344,0.0,0.0,null,1.131,1.044,1.186,null,0.0,0.0,0.0,null,-5.551,-10.386,0.0,0.0,null,null],[4718.0,null,null,2007.962,883.437,null,128.882,113.786,0.0,0.0,null,1.132,1.05,1.239,null,0.0,0.0,0.0,null,-5.548,-5.406,0.0,0.0,null,null],[4718.5,null,null,2006.556,887.76,null,128.833,112.114,0.0,0.0,null,1.126,1.049,1.247,null,0.0,0.0,0.0,null,-5.55,1.354,0.0,0.0,null,null],[4719.0,null,null,1998.99,891.509,null,128.78,111.068,0.0,0.0,null,1.122,1.044,1.226,null,0.0,0.0,0.0,null,-5.491,2.046,0.0,0.0,null,null],[4719.5,null,null,1998.101,893.953,null,128.686,111.092,0.0,0.0,null,1.119,1.037,1.207,null,0.0,0.0,0.0,null,-5.387,-0.4,0.0,0.0,null,null],[4720.0,null,null,2002.823,898.295,null,128.623,110.985,0.0,0.0,null,1.113,1.03,1.177,null,0.0,0.0,0.0,null,-5.327,-3.527,0.0,0.0,null,null],[4720.5,null,null,2001.223,906.011,null,128.73,110.277,0.0,0.0,null,1.104,1.022,1.152,null,0.0,0.0,0.0,null,-5.305,-6.342,0.0,0.0,null,null],[4721.0,null,null,1992.911,914.637,null,129.102,109.126,0.0,0.0,null,1.093,1.013,1.133,null,0.0,0.0,0.0,null,-5.322,-6.6,0.0,0.0,null,null],[4721.5,null,null,1985.196,921.433,null,129.653,107.575,0.0,0.0,null,1.085,1.003,1.127,null,0.0,0.0,0.0,null,-5.471,-2.687,0.0,0.0,null,null],[4722.0,null,null,1976.25,925.035,null,130.149,107.392,0.0,0.0,null,1.081,0.997,1.12,null,0.0,0.0,0.0,null,-5.74,1.093,0.0,0.0,null,null],[4722.5,null,null,1964.546,923.668,null,130.429,108.836,0.0,0.0,null,1.083,1.0,1.103,null,0.0,0.0,0.0,null,-5.984,-0.707,0.0,0.0,null,null],[4723.0,null,null,1958.629,916.01,null,130.449,109.937,0.0,0.0,null,1.092,1.014,1.087,null,0.0,0.0,0.0,null,-6.168,-7.354,0.0,0.0,null,null],[4723.5,null,null,1964.102,904.155,null,130.251,109.81,0.0,0.0,null,1.106,1.034,1.094,null,0.0,0.0,0.0,null,-6.345,-14.133,0.0,0.0,null,null],[4724.0,null,null,1972.342,891.205,null,129.903,109.037,0.0,0.0,null,1.122,1.058,1.147,null,0.0,0.0,0.0,null,-6.54,-18.799,0.0,0.0,null,null],[4724.5,null,null,1976.043,877.044,null,129.452,106.715,0.0,0.0,null,1.14,1.083,1.193,null,0.0,0.0,0.0,null,-6.75,-24.261,0.0,0.0,null,null],[4725.0,null,null,1977.569,860.099,null,128.982,102.916,0.0,0.0,null,1.163,1.108,1.24,null,0.0,0.0,0.0,null,-6.938,-32.572,0.0,0.0,null,null],[4725.5,null,null,1977.52,843.112,null,128.63,100.161,0.0,0.0,null,1.186,1.129,1.309,null,0.0,0.0,0.0,null,-7.112,-43.277,0.0,0.0,null,null],[4726.0,null,null,1975.681,830.106,null,128.458,99.929,0.0,0.0,null,1.205,1.143,1.359,null,0.0,0.0,0.0,null,-7.281,-48.298,0.0,0.0,null,null],[4726.5,null,null,1976.837,821.785,null,128.369,99.797,0.0,0.0,null,1.217,1.148,1.371,null,0.0,0.0,0.0,null,-7.394,-43.623,0.0,0.0,null,null],[4727.0,null,null,1982.564,816.227,null,128.23,98.515,0.0,0.0,null,1.225,1.147,1.342,null,0.0,0.0,0.0,null,-7.422,-37.251,0.0,0.0,null,null],[4727.5,null,null,1986.61,811.762,null,128.026,99.171,0.0,0.0,null,1.232,1.146,1.311,null,0.0,0.0,0.0,null,-7.379,-31.909,0.0,0.0,null,null],[4728.0,null,null,1988.767,806.821,null,127.86,102.471,0.0,0.0,null,1.239,1.149,1.283,null,0.0,0.0,0.0,null,-7.329,-19.793,0.0,0.0,null,null],[4728.5,null,null,1993.989,801.251,null,127.836,105.521,0.0,0.0,null,1.248,1.161,1.253,null,0.0,0.0,0.0,null,-7.366,-4.392,0.0,0.0,null,null],[4729.0,null,null,2000.645,795.99,null,127.911,107.273,0.0,0.0,null,1.256,1.18,1.238,null,0.0,0.0,0.0,null,-7.506,0.255,0.0,0.0,null,null],[4729.5,null,null,2002.648,792.655,null,127.888,108.171,0.0,0.0,null,1.262,1.198,1.282,null,0.0,0.0,0.0,null,-7.668,-4.459,0.0,0.0,null,null],[4730.0,null,null,1999.49,790.987,null,127.601,106.952,0.0,0.0,null,1.264,1.205,1.394,null,0.0,0.0,0.0,null,-7.82,-5.889,0.0,0.0,null,null],[4730.5,null,null,1996.385,787.756,null,127.117,104.344,0.0,0.0,null,1.269,1.204,1.492,null,0.0,0.0,0.0,null,-7.951,-3.039,0.0,0.0,null,null],[4731.0,null,null,1995.926,781.868,null,126.71,102.668,0.0,0.0,null,1.279,1.201,1.482,null,0.0,0.0,0.0,null,-8.025,2.736,0.0,0.0,null,null],[4731.5,null,null,1997.803,777.474,null,126.612,102.194,0.0,0.0,null,1.286,1.202,1.388,null,0.0,0.0,0.0,null,-8.064,8.294,0.0,0.0,null,null],[4732.0,null,null,1999.665,778.246,null,126.845,101.884,0.0,0.0,null,1.285,1.203,1.336,null,0.0,0.0,0.0,null,-8.154,3.675,0.0,0.0,null,null],[4732.5,null,null,2001.57,784.068,null,127.236,102.029,0.0,0.0,null,1.275,1.195,1.355,null,0.0,0.0,0.0,null,-8.318,-12.533,0.0,0.0,null,null],[4733.0,null,null,2003.628,792.891,null,127.59,103.757,0.0,0.0,null,1.261,1.178,1.392,null,0.0,0.0,0.0,null,-8.486,-26.315,0.0,0.0,null,null],[4733.5,null,null,2003.03,801.42,null,127.881,105.934,0.0,0.0,null,1.248,1.159,1.397,null,0.0,0.0,0.0,null,-8.548,-31.477,0.0,0.0,null,null],[4734.0,null,null,1998.931,806.297,null,128.19,106.34,0.0,0.0,null,1.24,1.145,1.34,null,0.0,0.0,0.0,null,-8.501,-34.823,0.0,0.0,null,null],[4734.5,null,null,1992.291,808.166,null,128.524,104.784,0.0,0.0,null,1.237,1.138,1.292,null,0.0,0.0,0.0,null,-8.526,-41.312,0.0,0.0,null,null],[4735.0,null,null,1984.41,812.21,null,128.812,104.219,0.0,0.0,null,1.231,1.134,1.289,null,0.0,0.0,0.0,null,-8.725,-45.548,0.0,0.0,null,null],[4735.5,null,null,1978.892,819.681,null,129.017,105.243,0.0,0.0,null,1.22,1.131,1.326,null,0.0,0.0,0.0,null,-8.864,-36.986,0.0,0.0,null,null],[4736.0,null,null,1984.117,826.003,null,129.102,105.526,0.0,0.0,null,1.211,1.129,1.346,null,0.0,0.0,0.0,null,-8.705,-19.239,0.0,0.0,null,null],[4736.5,null,null,1994.309,827.274,null,128.98,103.906,0.0,0.0,null,1.209,1.13,1.318,null,0.0,0.0,0.0,null,-8.358,-7.693,0.0,0.0,null,null],[4737.0,null,null,1993.25,822.655,null,128.63,104.185,0.0,0.0,null,1.216,1.134,1.3,null,0.0,0.0,0.0,null,-8.071,-6.813,0.0,0.0,null,null],[4737.5,null,null,1981.045,812.902,null,128.21,107.688,0.0,0.0,null,1.23,1.145,1.282,null,0.0,0.0,0.0,null,-7.99,-8.166,0.0,0.0,null,null],[4738.0,null,null,1971.596,802.496,null,128.028,110.661,0.0,0.0,null,1.246,1.165,1.289,null,0.0,0.0,0.0,null,-8.067,-5.483,0.0,0.0,null,null],[4738.5,null,null,1970.338,796.856,null,128.263,110.41,0.0,0.0,null,1.255,1.183,1.337,null,0.0,0.0,0.0,null,-8.125,-2.194,0.0,0.0,null,null],[4739.0,null,null,1973.803,796.075,null,128.741,107.42,0.0,0.0,null,1.256,1.187,1.412,null,0.0,0.0,0.0,null,-8.082,-2.655,0.0,0.0,null,null],[4739.5,null,null,1978.138,796.795,null,129.104,102.863,0.0,0.0,null,1.255,1.177,1.49,null,0.0,0.0,0.0,null,-8.006,-5.901,0.0,0.0,null,null],[4740.0,null,null,1981.948,796.229,null,129.21,99.858,0.0,0.0,null,1.256,1.166,1.476,null,0.0,0.0,0.0,null,-7.853,-10.248,0.0,0.0,null,null],[4740.5,null,null,1983.7,793.809,null,129.226,100.369,0.0,0.0,null,1.26,1.161,1.382,null,0.0,0.0,0.0,null,-7.586,-14.307,0.0,0.0,null,null],[4741.0,null,null,1980.24,792.974,null,129.35,102.251,0.0,0.0,null,1.261,1.156,1.311,null,0.0,0.0,0.0,null,-7.272,-13.742,0.0,0.0,null,null],[4741.5,null,null,1974.734,799.357,null,129.596,103.94,0.0,0.0,null,1.251,1.149,1.269,null,0.0,0.0,0.0,null,-6.963,-8.499,0.0,0.0,null,null],[4742.0,null,null,1976.452,813.217,null,129.84,105.463,0.0,0.0,null,1.23,1.137,1.237,null,0.0,0.0,0.0,null,-6.739,-5.429,0.0,0.0,null,null],[4742.5,null,null,1985.736,828.256,null,129.988,106.299,0.0,0.0,null,1.207,1.119,1.284,null,0.0,0.0,0.0,null,-6.692,-7.355,0.0,0.0,null,null],[4743.0,null,null,1993.501,840.125,null,130.053,107.011,0.0,0.0,null,1.19,1.093,1.367,null,0.0,0.0,0.0,null,-6.689,-11.195,0.0,0.0,null,null],[4743.5,null,null,1994.309,850.146,null,130.086,107.718,0.0,0.0,null,1.176,1.071,1.33,null,0.0,0.0,0.0,null,-6.499,-13.77,0.0,0.0,null,null],[4744.0,null,null,1991.65,860.821,null,130.074,107.558,0.0,0.0,null,1.162,1.057,1.202,null,0.0,0.0,0.0,null,-6.178,-11.07,0.0,0.0,null,null],[4744.5,null,null,1992.285,872.81,null,129.989,107.746,0.0,0.0,null,1.146,1.045,1.13,null,0.0,0.0,0.0,null,-5.921,-3.73,0.0,0.0,null,null],[4745.0,null,null,1997.823,885.094,null,129.853,108.805,0.0,0.0,null,1.13,1.033,1.126,null,0.0,0.0,0.0,null,-5.729,3.108,0.0,0.0,null,null],[4745.5,null,null,2000.333,895.551,null,129.699,110.379,0.0,0.0,null,1.117,1.024,1.136,null,0.0,0.0,0.0,null,-5.547,6.876,0.0,0.0,null,null],[4746.0,null,null,1996.483,902.623,null,129.515,111.95,0.0,0.0,null,1.108,1.019,1.128,null,0.0,0.0,0.0,null,-5.467,7.906,0.0,0.0,null,null],[4746.5,null,null,1993.403,904.632,null,129.27,112.89,0.0,0.0,null,1.105,1.023,1.11,null,0.0,0.0,0.0,null,-5.537,7.711,0.0,0.0,null,null],[4747.0,null,null,1994.45,898.596,null,128.963,113.076,0.0,0.0,null,1.113,1.033,1.102,null,0.0,0.0,0.0,null,-5.635,8.966,0.0,0.0,null,null],[4747.5,null,null,1989.284,885.373,null,128.609,113.403,0.0,0.0,null,1.129,1.049,1.108,null,0.0,0.0,0.0,null,-5.661,10.201,0.0,0.0,null,null],[4748.0,null,null,1980.681,870.541,null,128.233,113.121,0.0,0.0,null,1.149,1.07,1.139,null,0.0,0.0,0.0,null,-5.645,2.292,0.0,0.0,null,null],[4748.5,null,null,1983.732,858.232,null,127.821,111.466,0.0,0.0,null,1.165,1.093,1.183,null,0.0,0.0,0.0,null,-5.634,-13.12,0.0,0.0,null,null],[4749.0,null,null,1996.85,849.387,null,127.343,108.956,0.0,0.0,null,1.177,1.11,1.25,null,0.0,0.0,0.0,null,-5.66,-22.882,0.0,0.0,null,null],[4749.5,null,null,2001.834,842.928,null,126.783,106.515,0.0,0.0,null,1.186,1.117,1.335,null,0.0,0.0,0.0,null,-5.72,-25.146,0.0,0.0,null,null],[4750.0,null,null,1994.36,835.878,null,126.138,105.887,0.0,0.0,null,1.196,1.119,1.366,null,0.0,0.0,0.0,null,-5.722,-28.519,0.0,0.0,null,null],[4750.5,null,null,1986.694,826.873,null,125.434,107.536,0.0,0.0,null,1.209,1.124,1.302,null,0.0,0.0,0.0,null,-5.604,-31.795,0.0,0.0,null,null],[4751.0,null,null,1987.867,819.468,null,124.743,109.613,0.0,0.0,null,1.22,1.135,1.219,null,0.0,0.0,0.0,null,-5.441,-31.232,0.0,0.0,null,null],[4751.5,null,null,1992.498,817.422,null,124.195,109.367,0.0,0.0,null,1.223,1.146,1.206,null,0.0,0.0,0.0,null,-5.364,-30.255,0.0,0.0,null,null],[4752.0,null,null,1991.992,822.685,null,123.922,105.807,0.0,0.0,null,1.216,1.15,1.261,null,0.0,0.0,0.0,null,-5.375,-30.706,0.0,0.0,null,null],[4752.5,null,null,1989.994,834.88,null,123.987,100.889,0.0,0.0,null,1.198,1.142,1.294,null,0.0,0.0,0.0,null,-5.369,-30.256,0.0,0.0,null,null],[4753.0,null,null,1990.695,850.833,null,124.337,98.218,0.0,0.0,null,1.175,1.12,1.344,null,0.0,0.0,0.0,null,-5.305,-26.167,0.0,0.0,null,null],[4753.5,null,null,1986.12,867.716,null,124.876,98.96,0.0,0.0,null,1.152,1.084,1.409,null,0.0,0.0,0.0,null,-5.192,-22.323,0.0,0.0,null,null],[4754.0,null,null,1975.07,886.09,null,125.596,101.514,0.0,0.0,null,1.129,1.044,1.38,null,0.0,0.0,0.0,null,-4.978,-24.391,0.0,0.0,null,null],[4754.5,null,null,1967.548,906.868,null,126.575,104.474,0.0,0.0,null,1.103,1.011,1.204,null,0.0,0.0,0.0,null,-4.685,-27.818,0.0,0.0,null,null],[4755.0,null,null,1965.508,930.068,null,127.86,106.627,0.0,0.0,null,1.075,0.984,1.064,null,0.0,0.0,0.0,null,-4.439,-25.557,0.0,0.0,null,null],[4755.5,null,null,1961.67,955.303,null,129.257,108.031,0.0,0.0,null,1.047,0.96,1.02,null,0.0,0.0,0.0,null,-4.295,-15.415,0.0,0.0,null,null],[4756.0,null,null,1958.135,977.534,null,130.371,110.25,0.0,0.0,null,1.023,0.941,1.005,null,0.0,0.0,0.0,null,-4.241,-5.488,0.0,0.0,null,null],[4756.5,null,null,1961.434,991.514,null,130.917,113.33,0.0,0.0,null,1.009,0.932,1.023,null,0.0,0.0,0.0,null,-4.273,-7.722,0.0,0.0,null,null],[4757.0,null,null,1973.259,1000.917,null,130.964,114.676,0.0,0.0,null,0.999,0.931,1.055,null,0.0,0.0,0.0,null,-4.385,-16.151,0.0,0.0,null,null],[4757.5,null,null,1989.051,1009.421,null,130.83,113.684,0.0,0.0,null,0.991,0.935,1.059,null,0.0,0.0,0.0,null,-4.554,-14.1,0.0,0.0,null,null],[4758.0,null,null,2002.557,1011.591,null,130.715,112.12,0.0,0.0,null,0.989,0.938,1.051,null,0.0,0.0,0.0,null,-4.716,-4.941,0.0,0.0,null,null],[4758.5,null,null,2007.809,1003.465,null,130.479,110.987,0.0,0.0,null,0.997,0.941,1.077,null,0.0,0.0,0.0,null,-4.815,-2.286,0.0,0.0,null,null],[4759.0,null,null,2003.777,989.0,null,129.807,110.973,0.0,0.0,null,1.011,0.946,1.101,null,0.0,0.0,0.0,null,-4.911,-8.125,0.0,0.0,null,null],[4759.5,null,null,1995.759,970.392,null,128.699,111.609,0.0,0.0,null,1.031,0.959,1.076,null,0.0,0.0,0.0,null,-5.068,-16.322,0.0,0.0,null,null],[4760.0,null,null,1991.566,946.388,null,127.586,111.357,0.0,0.0,null,1.057,0.985,1.053,null,0.0,0.0,0.0,null,-5.255,-21.142,0.0,0.0,null,null],[4760.5,null,null,1988.656,920.458,null,126.917,110.392,0.0,0.0,null,1.086,1.021,1.06,null,0.0,0.0,0.0,null,-5.454,-20.872,0.0,0.0,null,null],[4761.0,null,null,1979.505,899.28,null,126.744,109.037,0.0,0.0,null,1.112,1.057,1.119,null,0.0,0.0,0.0,null,-5.66,-12.846,0.0,0.0,null,null],[4761.5,null,null,1970.982,886.979,null,126.713,105.697,0.0,0.0,null,1.127,1.085,1.234,null,0.0,0.0,0.0,null,-5.832,-0.957,0.0,0.0,null,null],[4762.0,null,null,1972.183,879.367,null,126.526,101.312,0.0,0.0,null,1.137,1.095,1.38,null,0.0,0.0,0.0,null,-5.983,5.012,0.0,0.0,null,null],[4762.5,null,null,1978.391,870.563,null,126.299,100.019,0.0,0.0,null,1.149,1.09,1.515,null,0.0,0.0,0.0,null,-6.145,5.705,0.0,0.0,null,null],[4763.0,null,null,1982.812,861.578,null,126.404,102.929,0.0,0.0,null,1.161,1.079,1.49,null,0.0,0.0,0.0,null,-6.184,5.831,0.0,0.0,null,null],[4763.5,null,null,1985.322,856.401,null,126.969,106.689,0.0,0.0,null,1.168,1.076,1.286,null,0.0,0.0,0.0,null,-5.969,2.863,0.0,0.0,null,null],[4764.0,null,null,1985.726,856.192,null,127.713,108.547,0.0,0.0,null,1.168,1.08,1.11,null,0.0,0.0,0.0,null,-5.593,-1.406,0.0,0.0,null,null],[4764.5,null,null,1987.638,861.796,null,128.276,107.977,0.0,0.0,null,1.16,1.081,1.092,null,0.0,0.0,0.0,null,-5.285,-3.319,0.0,0.0,null,null],[4765.0,null,null,1993.404,872.593,null,128.568,106.029,0.0,0.0,null,1.146,1.075,1.172,null,0.0,0.0,0.0,null,-5.169,-8.018,0.0,0.0,null,null],[4765.5,null,null,1996.834,881.853,null,128.794,105.262,0.0,0.0,null,1.134,1.067,1.241,null,0.0,0.0,0.0,null,-5.203,-14.844,0.0,0.0,null,null],[4766.0,null,null,1993.968,885.154,null,129.122,107.04,0.0,0.0,null,1.13,1.06,1.269,null,0.0,0.0,0.0,null,-5.253,-14.279,0.0,0.0,null,null],[4766.5,null,null,1993.816,885.289,null,129.469,108.813,0.0,0.0,null,1.13,1.055,1.234,null,0.0,0.0,0.0,null,-5.225,-6.219,0.0,0.0,null,null],[4767.0,null,null,1999.637,884.863,null,129.677,108.607,0.0,0.0,null,1.13,1.051,1.163,null,0.0,0.0,0.0,null,-5.14,-0.749,0.0,0.0,null,null],[4767.5,null,null,2000.894,882.778,null,129.745,107.148,0.0,0.0,null,1.133,1.051,1.13,null,0.0,0.0,0.0,null,-5.073,-1.663,0.0,0.0,null,null],[4768.0,null,null,1992.719,877.119,null,129.786,106.312,0.0,0.0,null,1.14,1.056,1.15,null,0.0,0.0,0.0,null,-5.035,-3.116,0.0,0.0,null,null],[4768.5,null,null,1984.349,867.203,null,129.764,106.833,0.0,0.0,null,1.153,1.068,1.167,null,0.0,0.0,0.0,null,-5.023,1.018,0.0,0.0,null,null],[4769.0,null,null,1984.679,853.648,null,129.443,107.302,0.0,0.0,null,1.171,1.095,1.138,null,0.0,0.0,0.0,null,-5.047,7.771,0.0,0.0,null,null],[4769.5,null,null,1992.047,837.84,null,128.572,105.557,0.0,0.0,null,1.194,1.132,1.132,null,0.0,0.0,0.0,null,-5.13,10.7,0.0,0.0,null,null],[4770.0,null,null,1997.424,821.157,null,127.185,102.341,0.0,0.0,null,1.218,1.17,1.213,null,0.0,0.0,0.0,null,-5.277,5.654,0.0,0.0,null,null],[4770.5,null,null,1995.085,806.856,null,125.7,100.158,0.0,0.0,null,1.239,1.197,1.369,null,0.0,0.0,0.0,null,-5.489,-1.46,0.0,0.0,null,null],[4771.0,null,null,1987.31,799.231,null,124.587,99.415,0.0,0.0,null,1.251,1.203,1.567,null,0.0,0.0,0.0,null,-5.706,-1.556,0.0,0.0,null,null],[4771.5,null,null,1981.875,802.185,null,124.043,99.133,0.0,0.0,null,1.247,1.185,1.69,null,0.0,0.0,0.0,null,-5.772,1.844,0.0,0.0,null,null],[4772.0,null,null,1982.056,817.965,null,123.953,98.949,0.0,0.0,null,1.223,1.147,1.653,null,0.0,0.0,0.0,null,-5.611,-0.796,0.0,0.0,null,null],[4772.5,null,null,1983.228,843.449,null,124.119,99.704,0.0,0.0,null,1.186,1.096,1.479,null,0.0,0.0,0.0,null,-5.287,-5.683,0.0,0.0,null,null],[4773.0,null,null,1982.162,872.495,null,124.5,102.685,0.0,0.0,null,1.146,1.045,1.312,null,0.0,0.0,0.0,null,-4.909,-6.553,0.0,0.0,null,null],[4773.5,null,null,1983.385,904.045,null,125.214,107.314,0.0,0.0,null,1.106,1.003,1.156,null,0.0,0.0,0.0,null,-4.626,-5.347,0.0,0.0,null,null],[4774.0,null,null,1990.833,941.05,null,126.314,111.766,0.0,0.0,null,1.063,0.971,1.036,null,0.0,0.0,0.0,null,-4.567,-6.512,0.0,0.0,null,null],[4774.5,null,null,1997.058,981.371,null,127.639,113.831,0.0,0.0,null,1.019,0.945,0.977,null,0.0,0.0,0.0,null,-4.655,-10.661,0.0,0.0,null,null],[4775.0,null,null,1998.7,1016.848,null,129.009,113.934,0.0,0.0,null,0.983,0.924,0.958,null,0.0,0.0,0.0,null,-4.836,-15.016,0.0,0.0,null,null],[4775.5,null,null,1999.173,1039.585,null,130.28,113.108,0.0,0.0,null,0.962,0.912,0.958,null,0.0,0.0,0.0,null,-5.243,-13.998,0.0,0.0,null,null],[4776.0,null,null,2001.555,1044.626,null,131.205,112.92,0.0,0.0,null,0.957,0.913,0.951,null,0.0,0.0,0.0,null,-5.994,-9.646,0.0,0.0,null,null],[4776.5,null,null,2003.129,1027.448,null,131.398,113.406,0.0,0.0,null,0.973,0.932,0.94,null,0.0,0.0,0.0,null,-7.143,-5.811,0.0,0.0,null,null],[4777.0,null,null,2001.641,986.242,null,130.55,113.985,0.0,0.0,null,1.014,0.978,0.928,null,0.0,0.0,0.0,null,-8.627,-5.043,0.0,0.0,null,null],[4777.5,null,null,1999.924,928.513,null,128.717,113.72,0.0,0.0,null,1.077,1.06,0.881,null,0.0,0.0,0.0,null,-10.168,-4.78,0.0,0.0,null,null],[4778.0,null,null,2000.064,869.309,null,126.249,109.831,0.0,0.0,null,1.15,1.177,0.836,null,0.0,0.0,0.0,null,-11.823,0.023,0.0,0.0,null,null],[4778.5,null,null,1996.721,819.604,null,123.446,99.652,0.0,0.0,null,1.22,1.3,1.0,null,0.0,0.0,0.0,null,-14.36,7.079,0.0,0.0,null,null],[4779.0,null,null,1987.861,779.558,null,120.308,84.821,0.0,0.0,null,1.283,1.391,1.727,null,0.0,0.0,0.0,null,-18.05,8.221,0.0,0.0,null,null],[4779.5,null,null,1983.837,745.305,null,116.905,71.29,0.0,0.0,null,1.342,1.437,3.191,null,0.0,0.0,0.0,null,-21.715,0.578,0.0,0.0,null,null],[4780.0,null,null,1989.038,717.268,null,113.866,64.368,0.0,0.0,null,1.394,1.447,4.038,null,0.0,0.0,0.0,null,-23.756,-7.673,0.0,0.0,null,null],[4780.5,null,null,1995.038,702.237,null,112.192,66.021,0.0,0.0,null,1.424,1.425,3.572,null,0.0,0.0,0.0,null,-23.472,-11.59,0.0,0.0,null,null],[4781.0,null,null,1999.692,705.121,null,112.496,73.397,0.0,0.0,null,1.418,1.372,2.715,null,0.0,0.0,0.0,null,-21.15,-14.339,0.0,0.0,null,null],[4781.5,null,null,2006.566,725.597,null,114.512,82.213,0.0,0.0,null,1.378,1.303,2.117,null,0.0,0.0,0.0,null,-17.616,-17.69,0.0,0.0,null,null],[4782.0,null,null,2007.526,758.932,null,117.478,91.296,0.0,0.0,null,1.318,1.232,1.652,null,0.0,0.0,0.0,null,-13.717,-18.074,0.0,0.0,null,null],[4782.5,null,null,1999.234,797.75,null,120.751,100.008,0.0,0.0,null,1.254,1.17,1.294,null,0.0,0.0,0.0,null,-10.196,-15.753,0.0,0.0,null,null],[4783.0,null,null,1994.674,833.809,null,124.044,106.089,0.0,0.0,null,1.199,1.12,1.117,null,0.0,0.0,0.0,null,-7.723,-12.545,0.0,0.0,null,null],[4783.5,null,null,1996.397,863.084,null,127.131,109.236,0.0,0.0,null,1.159,1.084,1.069,null,0.0,0.0,0.0,null,-6.45,-7.538,0.0,0.0,null,null],[4784.0,null,null,1995.23,886.072,null,129.621,112.188,0.0,0.0,null,1.129,1.059,1.06,null,0.0,0.0,0.0,null,-5.907,-0.753,0.0,0.0,null,null],[4784.5,null,null,1992.26,903.724,null,131.212,115.487,0.0,0.0,null,1.107,1.043,1.056,null,0.0,0.0,0.0,null,-5.662,4.128,0.0,0.0,null,null],[4785.0,null,null,1992.909,915.893,null,131.949,117.193,0.0,0.0,null,1.092,1.032,1.097,null,0.0,0.0,0.0,null,-5.529,1.784,0.0,0.0,null,null],[4785.5,null,null,1995.788,921.877,null,132.134,116.972,0.0,0.0,null,1.085,1.024,1.144,null,0.0,0.0,0.0,null,-5.373,-4.384,0.0,0.0,null,null],[4786.0,null,null,1997.521,923.967,null,131.961,116.78,0.0,0.0,null,1.082,1.016,1.162,null,0.0,0.0,0.0,null,-5.176,-4.134,0.0,0.0,null,null],[4786.5,null,null,1995.285,924.604,null,131.448,117.705,0.0,0.0,null,1.082,1.012,1.159,null,0.0,0.0,0.0,null,-5.068,-0.839,0.0,0.0,null,null],[4787.0,null,null,1991.842,921.804,null,130.608,119.152,0.0,0.0,null,1.085,1.015,1.148,null,0.0,0.0,0.0,null,-5.095,-3.887,0.0,0.0,null,null],[4787.5,null,null,1990.328,912.618,null,129.595,119.758,0.0,0.0,null,1.096,1.023,1.147,null,0.0,0.0,0.0,null,-5.152,-9.029,0.0,0.0,null,null],[4788.0,null,null,1989.249,900.546,null,128.651,118.803,0.0,0.0,null,1.11,1.034,1.169,null,0.0,0.0,0.0,null,-5.186,-7.147,0.0,0.0,null,null],[4788.5,null,null,1989.233,890.978,null,127.877,117.186,0.0,0.0,null,1.122,1.048,1.183,null,0.0,0.0,0.0,null,-5.22,0.504,0.0,0.0,null,null],[4789.0,null,null,1991.972,885.172,null,127.258,116.681,0.0,0.0,null,1.13,1.063,1.186,null,0.0,0.0,0.0,null,-5.284,6.762,0.0,0.0,null,null],[4789.5,null,null,1993.724,881.568,null,126.798,117.351,0.0,0.0,null,1.134,1.072,1.206,null,0.0,0.0,0.0,null,-5.42,7.188,0.0,0.0,null,null],[4790.0,null,null,1992.754,879.813,null,126.618,117.119,0.0,0.0,null,1.137,1.075,1.24,null,0.0,0.0,0.0,null,-5.619,-0.356,0.0,0.0,null,null],[4790.5,null,null,1992.72,881.357,null,126.809,115.916,0.0,0.0,null,1.135,1.071,1.277,null,0.0,0.0,0.0,null,-5.828,-11.801,0.0,0.0,null,null],[4791.0,null,null,1997.167,884.234,null,127.266,115.354,0.0,0.0,null,1.131,1.064,1.291,null,0.0,0.0,0.0,null,-5.983,-17.166,0.0,0.0,null,null],[4791.5,null,null,1998.512,883.315,null,127.761,115.085,0.0,0.0,null,1.132,1.061,1.246,null,0.0,0.0,0.0,null,-6.031,-13.991,0.0,0.0,null,null],[4792.0,null,null,1990.401,875.618,null,128.061,114.414,0.0,0.0,null,1.142,1.069,1.13,null,0.0,0.0,0.0,null,-6.048,-12.143,0.0,0.0,null,null],[4792.5,null,null,1982.419,860.598,null,127.956,113.707,0.0,0.0,null,1.162,1.091,1.05,null,0.0,0.0,0.0,null,-6.199,-15.103,0.0,0.0,null,null],[4793.0,null,null,1982.613,838.698,null,127.197,111.785,0.0,0.0,null,1.192,1.134,1.033,null,0.0,0.0,0.0,null,-6.515,-14.341,0.0,0.0,null,null],[4793.5,null,null,1982.922,812.158,null,125.537,108.113,0.0,0.0,null,1.231,1.199,1.055,null,0.0,0.0,0.0,null,-7.032,-9.909,0.0,0.0,null,null],[4794.0,null,null,1978.672,782.608,null,122.91,104.563,0.0,0.0,null,1.278,1.277,1.207,null,0.0,0.0,0.0,null,-7.768,-9.481,0.0,0.0,null,null],[4794.5,null,null,1977.081,751.948,null,119.683,101.721,0.0,0.0,null,1.33,1.357,1.483,null,0.0,0.0,0.0,null,-8.458,-12.078,0.0,0.0,null,null],[4795.0,null,null,1982.382,727.072,null,116.651,99.417,0.0,0.0,null,1.375,1.422,1.79,null,0.0,0.0,0.0,null,-9.003,-15.058,0.0,0.0,null,null],[4795.5,null,null,1990.007,716.205,null,114.625,98.457,0.0,0.0,null,1.396,1.443,2.136,null,0.0,0.0,0.0,null,-9.712,-18.088,0.0,0.0,null,null],[4796.0,null,null,1994.687,723.313,null,113.997,98.635,0.0,0.0,null,1.383,1.387,2.627,null,0.0,0.0,0.0,null,-10.579,-18.233,0.0,0.0,null,null],[4796.5,null,null,1994.865,747.778,null,114.631,99.144,0.0,0.0,null,1.337,1.274,3.072,null,0.0,0.0,0.0,null,-10.966,-16.468,0.0,0.0,null,null],[4797.0,null,null,1992.78,785.503,null,116.199,101.768,0.0,0.0,null,1.273,1.158,2.654,null,0.0,0.0,0.0,null,-10.191,-18.091,0.0,0.0,null,null],[4797.5,null,null,1992.738,831.009,null,118.483,108.311,0.0,0.0,null,1.203,1.07,1.613,null,0.0,0.0,0.0,null,-8.223,-19.315,0.0,0.0,null,null],[4798.0,null,null,1994.724,883.172,null,121.372,117.175,0.0,0.0,null,1.132,1.005,0.978,null,0.0,0.0,0.0,null,-5.966,-15.744,0.0,0.0,null,null],[4798.5,null,null,1992.82,942.15,null,124.61,123.604,0.0,0.0,null,1.061,0.951,0.824,null,0.0,0.0,0.0,null,-4.422,-10.355,0.0,0.0,null,null],[4799.0,null,null,1989.971,1002.634,null,127.631,125.436,0.0,0.0,null,0.997,0.909,0.891,null,0.0,0.0,0.0,null,-3.719,-7.234,0.0,0.0,null,null],[4799.5,null,null,1991.257,1054.04,null,129.796,124.403,0.0,0.0,null,0.949,0.88,0.926,null,0.0,0.0,0.0,null,-3.404,-6.737,0.0,0.0,null,null],[4800.0,null,null,1995.532,1087.495,null,130.802,123.339,0.0,0.0,null,0.92,0.864,0.921,null,0.0,0.0,0.0,null,-3.223,-7.439,0.0,0.0,null,null],[4800.5,null,null,2000.88,1102.382,null,130.917,123.232,0.0,0.0,null,0.907,0.858,0.939,null,0.0,0.0,0.0,null,-3.131,-5.431,0.0,0.0,null,null],[4801.0,null,null,2008.488,1104.614,null,130.69,123.909,0.0,0.0,null,0.905,0.858,0.968,null,0.0,0.0,0.0,null,-3.143,0.793,0.0,0.0,null,null],[4801.5,null,null,2013.065,1101.452,null,130.569,124.777,0.0,0.0,null,0.908,0.86,1.016,null,0.0,0.0,0.0,null,-3.248,5.641,0.0,0.0,null,null],[4802.0,null,null,2010.55,1096.945,null,130.729,124.68,0.0,0.0,null,0.912,0.86,1.063,null,0.0,0.0,0.0,null,-3.398,5.31,0.0,0.0,null,null],[4802.5,null,null,2004.155,1092.704,null,131.077,123.041,0.0,0.0,null,0.915,0.857,1.077,null,0.0,0.0,0.0,null,-3.559,2.736,0.0,0.0,null,null],[4803.0,null,null,2002.049,1088.02,null,131.473,120.414,0.0,0.0,null,0.919,0.855,1.06,null,0.0,0.0,0.0,null,-3.658,-0.581,0.0,0.0,null,null],[4803.5,null,null,2006.995,1084.32,null,131.844,118.979,0.0,0.0,null,0.922,0.854,1.031,null,0.0,0.0,0.0,null,-3.614,-4.234,0.0,0.0,null,null],[4804.0,null,null,2013.38,1085.638,null,132.153,118.32,0.0,0.0,null,0.921,0.856,1.016,null,0.0,0.0,0.0,null,-3.49,-2.729,0.0,0.0,null,null],[4804.5,null,null,2013.014,1092.935,null,132.359,116.688,0.0,0.0,null,0.915,0.859,1.012,null,0.0,0.0,0.0,null,-3.453,0.927,0.0,0.0,null,null],[4805.0,null,null,2006.282,1101.527,null,132.435,114.811,0.0,0.0,null,0.908,0.86,1.023,null,0.0,0.0,0.0,null,-3.516,-3.42,0.0,0.0,null,null],[4805.5,null,null,1998.398,1106.942,null,132.332,114.448,0.0,0.0,null,0.903,0.859,1.032,null,0.0,0.0,0.0,null,-3.546,-11.274,0.0,0.0,null,null],[4806.0,null,null,1996.759,1104.759,null,131.982,114.146,0.0,0.0,null,0.905,0.858,1.027,null,0.0,0.0,0.0,null,-3.565,-9.096,0.0,0.0,null,null],[4806.5,null,null,2005.345,1088.626,null,131.365,112.36,0.0,0.0,null,0.919,0.863,0.997,null,0.0,0.0,0.0,null,-3.694,3.964,0.0,0.0,null,null],[4807.0,null,null,2017.965,1057.13,null,130.457,111.75,0.0,0.0,null,0.946,0.883,0.961,null,0.0,0.0,0.0,null,-3.943,17.33,0.0,0.0,null,null],[4807.5,null,null,2024.469,1018.47,null,129.309,112.686,0.0,0.0,null,0.982,0.923,0.92,null,0.0,0.0,0.0,null,-4.208,19.872,0.0,0.0,null,null],[4808.0,null,null,2026.604,982.89,null,128.143,111.681,0.0,0.0,null,1.017,0.981,0.9,null,0.0,0.0,0.0,null,-4.45,13.226,0.0,0.0,null,null],[4808.5,null,null,2027.352,950.39,null,127.242,108.157,0.0,0.0,null,1.052,1.042,1.021,null,0.0,0.0,0.0,null,-4.781,9.007,0.0,0.0,null,null],[4809.0,null,null,2022.435,912.513,null,126.615,103.881,0.0,0.0,null,1.096,1.092,1.309,null,0.0,0.0,0.0,null,-5.291,8.95,0.0,0.0,null,null],[4809.5,null,null,2015.941,864.355,null,125.79,100.118,0.0,0.0,null,1.157,1.134,1.559,null,0.0,0.0,0.0,null,-5.842,7.199,0.0,0.0,null,null],[4810.0,null,null,2011.987,812.097,null,124.236,98.588,0.0,0.0,null,1.231,1.185,1.545,null,0.0,0.0,0.0,null,-6.308,2.335,0.0,0.0,null,null],[4810.5,null,null,2008.156,770.061,null,122.116,99.858,0.0,0.0,null,1.299,1.252,1.399,null,0.0,0.0,0.0,null,-6.596,-2.966,0.0,0.0,null,null],[4811.0,null,null,2006.511,753.023,null,120.407,100.861,0.0,0.0,null,1.328,1.31,1.265,null,0.0,0.0,0.0,null,-6.565,-7.516,0.0,0.0,null,null],[4811.5,null,null,2011.23,763.82,null,120.042,98.421,0.0,0.0,null,1.309,1.314,1.352,null,0.0,0.0,0.0,null,-6.528,-9.957,0.0,0.0,null,null],[4812.0,null,null,2018.941,791.098,null,120.968,94.13,0.0,0.0,null,1.264,1.252,1.847,null,0.0,0.0,0.0,null,-7.158,-8.745,0.0,0.0,null,null],[4812.5,null,null,2026.139,821.295,null,122.11,93.12,0.0,0.0,null,1.218,1.164,2.314,null,0.0,0.0,0.0,null,-8.014,-1.626,0.0,0.0,null,null],[4813.0,null,null,2032.421,846.943,null,122.582,98.658,0.0,0.0,null,1.181,1.098,1.865,null,0.0,0.0,0.0,null,-7.901,4.693,0.0,0.0,null,null],[4813.5,null,null,2032.767,866.061,null,122.8,107.478,0.0,0.0,null,1.155,1.067,1.135,null,0.0,0.0,0.0,null,-6.868,1.589,0.0,0.0,null,null],[4814.0,null,null,2027.563,883.852,null,123.812,114.151,0.0,0.0,null,1.131,1.057,0.866,null,0.0,0.0,0.0,null,-5.957,-3.516,0.0,0.0,null,null],[4814.5,null,null,2023.544,905.028,null,125.637,114.81,0.0,0.0,null,1.105,1.055,0.881,null,0.0,0.0,0.0,null,-5.801,-1.318,0.0,0.0,null,null],[4815.0,null,null,2022.039,921.382,null,126.885,110.782,0.0,0.0,null,1.085,1.058,1.038,null,0.0,0.0,0.0,null,-6.667,2.388,0.0,0.0,null,null],[4815.5,null,null,2021.512,919.502,null,126.147,106.855,0.0,0.0,null,1.088,1.078,1.21,null,0.0,0.0,0.0,null,-8.066,-0.677,0.0,0.0,null,null],[4816.0,null,null,2021.245,898.864,null,123.507,105.078,0.0,0.0,null,1.113,1.126,1.136,null,0.0,0.0,0.0,null,-8.724,-6.915,0.0,0.0,null,null],[4816.5,null,null,2021.004,869.301,null,120.486,103.033,0.0,0.0,null,1.15,1.193,1.022,null,0.0,0.0,0.0,null,-8.795,-12.12,0.0,0.0,null,null],[4817.0,null,null,2022.792,839.596,null,118.374,98.076,0.0,0.0,null,1.191,1.249,1.257,null,0.0,0.0,0.0,null,-10.046,-19.668,0.0,0.0,null,null],[4817.5,null,null,2023.617,814.552,null,117.01,92.09,0.0,0.0,null,1.228,1.279,1.935,null,0.0,0.0,0.0,null,-12.665,-29.92,0.0,0.0,null,null],[4818.0,null,null,2018.199,794.095,null,115.541,90.141,0.0,0.0,null,1.259,1.294,2.372,null,0.0,0.0,0.0,null,-14.531,-34.405,0.0,0.0,null,null],[4818.5,null,null,2010.277,777.226,null,113.851,93.866,0.0,0.0,null,1.287,1.307,2.0,null,0.0,0.0,0.0,null,-14.288,-30.074,0.0,0.0,null,null],[4819.0,null,null,2005.407,768.026,null,112.745,98.45,0.0,0.0,null,1.302,1.308,1.676,null,0.0,0.0,0.0,null,-12.982,-24.451,0.0,0.0,null,null],[4819.5,null,null,2003.805,770.225,null,112.986,99.589,0.0,0.0,null,1.298,1.288,1.695,null,0.0,0.0,0.0,null,-12.003,-21.292,0.0,0.0,null,null],[4820.0,null,null,2009.092,781.422,null,114.528,98.99,0.0,0.0,null,1.28,1.251,1.848,null,0.0,0.0,0.0,null,-11.31,-20.644,0.0,0.0,null,null],[4820.5,null,null,2019.605,798.558,null,116.607,100.103,0.0,0.0,null,1.252,1.211,1.73,null,0.0,0.0,0.0,null,-9.983,-23.074,0.0,0.0,null,null],[4821.0,null,null,2023.133,820.432,null,118.521,102.831,0.0,0.0,null,1.219,1.175,1.447,null,0.0,0.0,0.0,null,-8.011,-24.789,0.0,0.0,null,null],[4821.5,null,null,2016.001,844.39,null,120.174,105.173,0.0,0.0,null,1.184,1.137,1.276,null,0.0,0.0,0.0,null,-6.333,-21.118,0.0,0.0,null,null],[4822.0,null,null,2006.646,867.063,null,121.802,107.067,0.0,0.0,null,1.153,1.097,1.232,null,0.0,0.0,0.0,null,-5.438,-16.334,0.0,0.0,null,null],[4822.5,null,null,2000.092,888.238,null,123.508,108.914,0.0,0.0,null,1.126,1.061,1.189,null,0.0,0.0,0.0,null,-4.918,-15.068,0.0,0.0,null,null],[4823.0,null,null,1992.755,908.319,null,125.188,110.681,0.0,0.0,null,1.101,1.032,1.126,null,0.0,0.0,0.0,null,-4.459,-11.735,0.0,0.0,null,null],[4823.5,null,null,1984.481,925.218,null,126.586,111.383,0.0,0.0,null,1.081,1.013,1.077,null,0.0,0.0,0.0,null,-4.118,-3.981,0.0,0.0,null,null],[4824.0,null,null,1979.689,937.026,null,127.361,110.371,0.0,0.0,null,1.067,1.004,1.02,null,0.0,0.0,0.0,null,-3.913,1.614,0.0,0.0,null,null],[4824.5,null,null,1979.352,941.255,null,127.288,108.373,0.0,0.0,null,1.062,1.005,0.982,null,0.0,0.0,0.0,null,-3.901,0.754,0.0,0.0,null,null],[4825.0,null,null,1981.429,936.708,null,126.437,107.573,0.0,0.0,null,1.068,1.014,1.016,null,0.0,0.0,0.0,null,-4.129,-2.188,0.0,0.0,null,null],[4825.5,null,null,1988.695,924.021,null,125.124,108.231,0.0,0.0,null,1.082,1.032,1.086,null,0.0,0.0,0.0,null,-4.472,-3.566,0.0,0.0,null,null],[4826.0,null,null,1999.568,904.106,null,123.755,108.363,0.0,0.0,null,1.106,1.064,1.104,null,0.0,0.0,0.0,null,-4.818,-4.996,0.0,0.0,null,null],[4826.5,null,null,2003.538,878.783,null,122.534,107.151,0.0,0.0,null,1.138,1.107,1.096,null,0.0,0.0,0.0,null,-5.201,-7.369,0.0,0.0,null,null],[4827.0,null,null,1996.274,851.451,null,121.236,106.137,0.0,0.0,null,1.174,1.154,1.129,null,0.0,0.0,0.0,null,-5.717,-7.847,0.0,0.0,null,null],[4827.5,null,null,1988.654,823.074,null,119.412,105.98,0.0,0.0,null,1.215,1.199,1.259,null,0.0,0.0,0.0,null,-6.478,-4.184,0.0,0.0,null,null],[4828.0,null,null,1987.626,793.813,null,116.958,104.766,0.0,0.0,null,1.26,1.244,1.444,null,0.0,0.0,0.0,null,-7.329,-0.836,0.0,0.0,null,null],[4828.5,null,null,1989.323,767.197,null,114.454,101.842,0.0,0.0,null,1.303,1.293,1.531,null,0.0,0.0,0.0,null,-7.875,-1.544,0.0,0.0,null,null],[4829.0,null,null,1987.817,747.085,null,112.848,97.65,0.0,0.0,null,1.339,1.341,1.535,null,0.0,0.0,0.0,null,-8.226,-4.519,0.0,0.0,null,null],[4829.5,null,null,1982.574,738.549,null,112.709,92.929,0.0,0.0,null,1.354,1.364,1.682,null,0.0,0.0,0.0,null,-9.066,-8.979,0.0,0.0,null,null],[4830.0,null,null,1977.917,746.541,null,113.714,89.039,0.0,0.0,null,1.34,1.343,2.051,null,0.0,0.0,0.0,null,-10.239,-15.619,0.0,0.0,null,null],[4830.5,null,null,1975.655,768.347,null,115.081,88.092,0.0,0.0,null,1.301,1.279,2.397,null,0.0,0.0,0.0,null,-10.523,-18.61,0.0,0.0,null,null],[4831.0,null,null,1977.105,795.95,null,116.455,91.433,0.0,0.0,null,1.256,1.196,2.215,null,0.0,0.0,0.0,null,-9.306,-11.967,0.0,0.0,null,null],[4831.5,null,null,1982.336,826.297,null,118.18,97.829,0.0,0.0,null,1.21,1.12,1.622,null,0.0,0.0,0.0,null,-7.298,-6.708,0.0,0.0,null,null],[4832.0,null,null,1987.095,860.04,null,120.615,104.189,0.0,0.0,null,1.163,1.06,1.174,null,0.0,0.0,0.0,null,-5.467,-13.485,0.0,0.0,null,null],[4832.5,null,null,1986.869,896.18,null,123.434,108.874,0.0,0.0,null,1.116,1.014,0.973,null,0.0,0.0,0.0,null,-4.246,-24.757,0.0,0.0,null,null],[4833.0,null,null,1986.371,932.108,null,125.822,112.851,0.0,0.0,null,1.073,0.982,0.932,null,0.0,0.0,0.0,null,-3.616,-28.887,0.0,0.0,null,null],[4833.5,null,null,1991.626,966.65,null,127.2,115.595,0.0,0.0,null,1.035,0.96,0.952,null,0.0,0.0,0.0,null,-3.342,-25.906,0.0,0.0,null,null],[4834.0,null,null,1998.809,997.438,null,127.695,117.129,0.0,0.0,null,1.003,0.946,0.989,null,0.0,0.0,0.0,null,-3.24,-20.762,0.0,0.0,null,null],[4834.5,null,null,1999.906,1022.167,null,127.843,119.427,0.0,0.0,null,0.978,0.933,1.027,null,0.0,0.0,0.0,null,-3.192,-17.469,0.0,0.0,null,null],[4835.0,null,null,1995.819,1039.029,null,128.005,122.836,0.0,0.0,null,0.962,0.917,1.086,null,0.0,0.0,0.0,null,-3.091,-17.017,0.0,0.0,null,null],[4835.5,null,null,1996.14,1048.118,null,128.201,125.6,0.0,0.0,null,0.954,0.898,1.141,null,0.0,0.0,0.0,null,-2.911,-19.729,0.0,0.0,null,null],[4836.0,null,null,2004.327,1052.129,null,128.368,126.381,0.0,0.0,null,0.95,0.883,1.12,null,0.0,0.0,0.0,null,-2.741,-25.622,0.0,0.0,null,null],[4836.5,null,null,2013.203,1054.399,null,128.573,125.953,0.0,0.0,null,0.948,0.873,1.05,null,0.0,0.0,0.0,null,-2.63,-27.741,0.0,0.0,null,null],[4837.0,null,null,2013.437,1059.073,null,128.918,125.939,0.0,0.0,null,0.944,0.868,0.985,null,0.0,0.0,0.0,null,-2.547,-20.551,0.0,0.0,null,null],[4837.5,null,null,2008.567,1069.937,null,129.428,127.216,0.0,0.0,null,0.935,0.865,0.96,null,0.0,0.0,0.0,null,-2.482,-10.365,0.0,0.0,null,null],[4838.0,null,null,2006.723,1084.002,null,129.988,128.184,0.0,0.0,null,0.923,0.864,0.975,null,0.0,0.0,0.0,null,-2.419,-6.184,0.0,0.0,null,null],[4838.5,null,null,2006.427,1093.929,null,130.437,127.997,0.0,0.0,null,0.914,0.86,1.01,null,0.0,0.0,0.0,null,-2.342,-8.631,0.0,0.0,null,null],[4839.0,null,null,1999.902,1097.736,null,130.766,127.077,0.0,0.0,null,0.911,0.858,1.033,null,0.0,0.0,0.0,null,-2.282,-12.509,0.0,0.0,null,null],[4839.5,null,null,1990.267,1097.364,null,131.099,126.194,0.0,0.0,null,0.911,0.858,1.031,null,0.0,0.0,0.0,null,-2.291,-11.465,0.0,0.0,null,null],[4840.0,null,null,1986.613,1093.003,null,131.497,124.644,0.0,0.0,null,0.915,0.861,1.042,null,0.0,0.0,0.0,null,-2.345,-5.589,0.0,0.0,null,null],[4840.5,null,null,1990.794,1086.429,null,131.855,121.768,0.0,0.0,null,0.92,0.863,1.06,null,0.0,0.0,0.0,null,-2.392,-3.363,0.0,0.0,null,null],[4841.0,null,null,1993.427,1081.978,null,132.043,117.969,0.0,0.0,null,0.924,0.865,1.063,null,0.0,0.0,0.0,null,-2.409,-7.446,0.0,0.0,null,null],[4841.5,null,null,1989.75,1082.815,null,132.009,115.085,0.0,0.0,null,0.924,0.867,1.048,null,0.0,0.0,0.0,null,-2.447,-9.515,0.0,0.0,null,null],[4842.0,null,null,1989.061,1088.062,null,131.83,113.419,0.0,0.0,null,0.919,0.867,1.047,null,0.0,0.0,0.0,null,-2.534,-9.557,0.0,0.0,null,null],[4842.5,null,null,1997.675,1094.541,null,131.65,112.37,0.0,0.0,null,0.914,0.862,1.048,null,0.0,0.0,0.0,null,-2.618,-15.139,0.0,0.0,null,null],[4843.0,null,null,2003.137,1100.744,null,131.481,112.885,0.0,0.0,null,0.908,0.857,1.033,null,0.0,0.0,0.0,null,-2.648,-18.032,0.0,0.0,null,null],[4843.5,null,null,1998.99,1105.726,null,131.244,115.22,0.0,0.0,null,0.904,0.851,1.028,null,0.0,0.0,0.0,null,-2.665,-9.26,0.0,0.0,null,null],[4844.0,null,null,1993.919,1109.44,null,130.877,116.418,0.0,0.0,null,0.901,0.846,1.034,null,0.0,0.0,0.0,null,-2.667,1.268,0.0,0.0,null,null],[4844.5,null,null,1992.379,1111.219,null,130.404,114.749,0.0,0.0,null,0.9,0.843,1.025,null,0.0,0.0,0.0,null,-2.632,0.544,0.0,0.0,null,null],[4845.0,null,null,1991.956,1108.507,null,129.859,112.511,0.0,0.0,null,0.902,0.845,0.987,null,0.0,0.0,0.0,null,-2.639,-7.857,0.0,0.0,null,null],[4845.5,null,null,1993.679,1098.799,null,129.285,111.714,0.0,0.0,null,0.91,0.854,0.95,null,0.0,0.0,0.0,null,-2.775,-12.85,0.0,0.0,null,null],[4846.0,null,null,1994.867,1079.903,null,128.725,111.78,0.0,0.0,null,0.926,0.872,0.949,null,0.0,0.0,0.0,null,-2.949,-11.169,0.0,0.0,null,null],[4846.5,null,null,1992.374,1050.66,null,128.163,111.598,0.0,0.0,null,0.952,0.901,0.969,null,0.0,0.0,0.0,null,-3.044,-9.488,0.0,0.0,null,null],[4847.0,null,null,1991.36,1015.112,null,127.599,111.069,0.0,0.0,null,0.985,0.938,0.995,null,0.0,0.0,0.0,null,-3.124,-10.76,0.0,0.0,null,null],[4847.5,null,null,1995.25,978.988,null,127.022,111.135,0.0,0.0,null,1.021,0.98,1.063,null,0.0,0.0,0.0,null,-3.358,-10.566,0.0,0.0,null,null],[4848.0,null,null,2000.35,942.06,null,126.465,110.779,0.0,0.0,null,1.062,1.022,1.191,null,0.0,0.0,0.0,null,-3.795,-10.947,0.0,0.0,null,null],[4848.5,null,null,2004.701,903.895,null,126.042,108.754,0.0,0.0,null,1.106,1.058,1.326,null,0.0,0.0,0.0,null,-4.294,-18.862,0.0,0.0,null,null],[4849.0,null,null,2007.471,868.009,null,125.851,106.704,0.0,0.0,null,1.152,1.092,1.388,null,0.0,0.0,0.0,null,-4.68,-28.935,0.0,0.0,null,null],[4849.5,null,null,2006.354,837.899,null,125.824,107.485,0.0,0.0,null,1.193,1.125,1.343,null,0.0,0.0,0.0,null,-4.913,-32.567,0.0,0.0,null,null],[4850.0,null,null,2001.133,815.784,null,125.738,110.516,0.0,0.0,null,1.226,1.156,1.292,null,0.0,0.0,0.0,null,-5.093,-33.644,0.0,0.0,null,null],[4850.5,null,null,1997.403,802.771,null,125.464,112.357,0.0,0.0,null,1.246,1.178,1.306,null,0.0,0.0,0.0,null,-5.306,-37.638,0.0,0.0,null,null],[4851.0,null,null,1994.892,796.172,null,125.123,111.712,0.0,0.0,null,1.256,1.193,1.342,null,0.0,0.0,0.0,null,-5.528,-39.887,0.0,0.0,null,null],[4851.5,null,null,1989.317,790.228,null,125.015,110.886,0.0,0.0,null,1.265,1.207,1.322,null,0.0,0.0,0.0,null,-5.711,-35.604,0.0,0.0,null,null],[4852.0,null,null,1981.227,780.568,null,125.19,111.57,0.0,0.0,null,1.281,1.225,1.28,null,0.0,0.0,0.0,null,-5.932,-24.599,0.0,0.0,null,null],[4852.5,null,null,1976.567,764.965,null,125.031,112.355,0.0,0.0,null,1.307,1.249,1.29,null,0.0,0.0,0.0,null,-6.23,-11.884,0.0,0.0,null,null],[4853.0,null,null,1974.459,744.657,null,123.667,111.455,0.0,0.0,null,1.343,1.284,1.318,null,0.0,0.0,0.0,null,-6.543,-3.335,0.0,0.0,null,null],[4853.5,null,null,1972.569,723.776,null,120.979,109.736,0.0,0.0,null,1.382,1.339,1.296,null,0.0,0.0,0.0,null,-6.753,-2.374,0.0,0.0,null,null],[4854.0,null,null,1972.723,710.175,null,117.958,107.107,0.0,0.0,null,1.408,1.404,1.243,null,0.0,0.0,0.0,null,-6.713,-7.067,0.0,0.0,null,null],[4854.5,null,null,1975.606,710.778,null,115.84,101.385,0.0,0.0,null,1.407,1.44,1.342,null,0.0,0.0,0.0,null,-6.704,-10.818,0.0,0.0,null,null],[4855.0,null,null,1978.074,725.069,null,115.015,93.556,0.0,0.0,null,1.379,1.411,1.85,null,0.0,0.0,0.0,null,-7.556,-10.189,0.0,0.0,null,null],[4855.5,null,null,1981.278,744.279,null,114.853,89.053,0.0,0.0,null,1.344,1.336,2.643,null,0.0,0.0,0.0,null,-8.984,-13.66,0.0,0.0,null,null],[4856.0,null,null,1986.192,762.049,null,114.69,91.124,0.0,0.0,null,1.312,1.258,2.568,null,0.0,0.0,0.0,null,-9.156,-22.445,0.0,0.0,null,null],[4856.5,null,null,1989.454,779.282,null,114.784,98.504,0.0,0.0,null,1.283,1.2,1.644,null,0.0,0.0,0.0,null,-7.478,-23.185,0.0,0.0,null,null],[4857.0,null,null,1990.726,799.554,null,115.861,106.846,0.0,0.0,null,1.251,1.159,1.146,null,0.0,0.0,0.0,null,-5.433,-14.249,0.0,0.0,null,null],[4857.5,null,null,1990.989,823.743,null,117.983,111.213,0.0,0.0,null,1.214,1.13,1.076,null,0.0,0.0,0.0,null,-4.176,-8.948,0.0,0.0,null,null],[4858.0,null,null,1985.79,849.359,null,120.294,111.347,0.0,0.0,null,1.177,1.106,1.113,null,0.0,0.0,0.0,null,-3.597,-9.851,0.0,0.0,null,null],[4858.5,null,null,1974.744,872.88,null,121.818,111.121,0.0,0.0,null,1.146,1.088,1.164,null,0.0,0.0,0.0,null,-3.377,-12.368,0.0,0.0,null,null],[4859.0,null,null,1965.982,890.572,null,122.26,113.319,0.0,0.0,null,1.123,1.076,1.204,null,0.0,0.0,0.0,null,-3.254,-14.268,0.0,0.0,null,null],[4859.5,null,null,1965.226,897.614,null,122.082,116.167,0.0,0.0,null,1.114,1.071,1.199,null,0.0,0.0,0.0,null,-3.117,-13.243,0.0,0.0,null,null],[4860.0,null,null,1972.149,896.871,null,121.947,117.404,0.0,0.0,null,1.115,1.067,1.228,null,0.0,0.0,0.0,null,-3.074,-9.202,0.0,0.0,null,null],[4860.5,null,null,1984.792,898.857,null,122.236,115.672,0.0,0.0,null,1.113,1.058,1.271,null,0.0,0.0,0.0,null,-3.051,-8.07,0.0,0.0,null,null],[4861.0,null,null,1992.627,910.215,null,123.098,113.237,0.0,0.0,null,1.099,1.043,1.284,null,0.0,0.0,0.0,null,-2.85,-9.675,0.0,0.0,null,null],[4861.5,null,null,1989.067,929.814,null,124.533,112.537,0.0,0.0,null,1.075,1.017,1.265,null,0.0,0.0,0.0,null,-2.519,-7.665,0.0,0.0,null,null],[4862.0,null,null,1981.238,957.607,null,126.344,113.295,0.0,0.0,null,1.044,0.978,1.239,null,0.0,0.0,0.0,null,-2.256,-4.561,0.0,0.0,null,null],[4862.5,null,null,1976.697,992.179,null,128.25,113.448,0.0,0.0,null,1.008,0.933,1.202,null,0.0,0.0,0.0,null,-2.164,-9.118,0.0,0.0,null,null],[4863.0,null,null,1976.822,1028.224,null,130.047,112.129,0.0,0.0,null,0.973,0.891,1.131,null,0.0,0.0,0.0,null,-2.179,-19.24,0.0,0.0,null,null],[4863.5,null,null,1984.507,1062.733,null,131.616,110.926,0.0,0.0,null,0.941,0.859,1.007,null,0.0,0.0,0.0,null,-2.139,-24.988,0.0,0.0,null,null],[4864.0,null,null,1994.165,1094.634,null,132.781,110.804,0.0,0.0,null,0.914,0.838,0.886,null,0.0,0.0,0.0,null,-2.0,-22.053,0.0,0.0,null,null],[4864.5,null,null,1996.472,1120.135,null,133.333,110.99,0.0,0.0,null,0.893,0.829,0.845,null,0.0,0.0,0.0,null,-1.902,-15.793,0.0,0.0,null,null],[4865.0,null,null,1998.607,1134.425,null,133.256,110.02,0.0,0.0,null,0.882,0.832,0.861,null,0.0,0.0,0.0,null,-1.919,-13.525,0.0,0.0,null,null],[4865.5,null,null,2003.354,1134.009,null,132.764,109.291,0.0,0.0,null,0.882,0.844,0.877,null,0.0,0.0,0.0,null,-2.01,-16.171,0.0,0.0,null,null],[4866.0,null,null,2004.055,1115.268,null,131.991,109.083,0.0,0.0,null,0.897,0.865,0.909,null,0.0,0.0,0.0,null,-2.097,-20.544,0.0,0.0,null,null],[4866.5,null,null,2006.85,1077.302,null,130.872,108.722,0.0,0.0,null,0.928,0.894,0.978,null,0.0,0.0,0.0,null,-2.164,-23.13,0.0,0.0,null,null],[4867.0,null,null,2017.358,1027.137,null,129.444,108.065,0.0,0.0,null,0.974,0.933,1.048,null,0.0,0.0,0.0,null,-2.283,-23.317,0.0,0.0,null,null],[4867.5,null,null,2023.032,977.508,null,128.032,106.902,0.0,0.0,null,1.023,0.983,1.096,null,0.0,0.0,0.0,null,-2.469,-23.134,0.0,0.0,null,null],[4868.0,null,null,2015.678,942.345,null,127.118,105.161,0.0,0.0,null,1.061,1.033,1.12,null,0.0,0.0,0.0,null,-2.599,-23.549,0.0,0.0,null,null],[4868.5,null,null,2004.771,929.95,null,126.921,102.57,0.0,0.0,null,1.075,1.055,1.239,null,0.0,0.0,0.0,null,-2.655,-20.511,0.0,0.0,null,null],[4869.0,null,null,2001.246,938.024,null,127.212,99.013,0.0,0.0,null,1.066,1.035,1.472,null,0.0,0.0,0.0,null,-2.837,-14.949,0.0,0.0,null,null],[4869.5,null,null,2006.079,957.569,null,127.67,95.836,0.0,0.0,null,1.044,0.983,1.666,null,0.0,0.0,0.0,null,-3.141,-13.071,0.0,0.0,null,null],[4870.0,null,null,2008.745,981.878,null,128.291,96.291,0.0,0.0,null,1.018,0.927,1.519,null,0.0,0.0,0.0,null,-3.212,-13.455,0.0,0.0,null,null],[4870.5,null,null,2005.017,1010.414,null,129.282,99.983,0.0,0.0,null,0.99,0.885,1.126,null,0.0,0.0,0.0,null,-2.85,-11.159,0.0,0.0,null,null],[4871.0,null,null,2001.224,1047.209,null,130.643,102.951,0.0,0.0,null,0.955,0.854,0.85,null,0.0,0.0,0.0,null,-2.299,-6.834,0.0,0.0,null,null],[4871.5,null,null,2002.069,1093.503,null,132.045,103.514,0.0,0.0,null,0.914,0.83,0.795,null,0.0,0.0,0.0,null,-1.888,-5.743,0.0,0.0,null,null],[4872.0,null,null,2003.928,1139.727,null,133.035,103.753,0.0,0.0,null,0.877,0.811,0.828,null,0.0,0.0,0.0,null,-1.706,-8.0,0.0,0.0,null,null],[4872.5,null,null,2001.786,1171.449,null,133.398,105.045,0.0,0.0,null,0.854,0.802,0.88,null,0.0,0.0,0.0,null,-1.684,-9.354,0.0,0.0,null,null],[4873.0,null,null,1993.286,1180.988,null,133.243,105.809,0.0,0.0,null,0.847,0.805,0.921,null,0.0,0.0,0.0,null,-1.72,-9.357,0.0,0.0,null,null],[4873.5,null,null,1984.136,1169.772,null,132.774,104.439,0.0,0.0,null,0.855,0.817,0.931,null,0.0,0.0,0.0,null,-1.758,-13.393,0.0,0.0,null,null],[4874.0,null,null,1982.959,1143.583,null,132.137,102.269,0.0,0.0,null,0.874,0.835,0.927,null,0.0,0.0,0.0,null,-1.821,-19.98,0.0,0.0,null,null],[4874.5,null,null,1987.067,1109.643,null,131.371,102.178,0.0,0.0,null,0.901,0.859,0.954,null,0.0,0.0,0.0,null,-1.997,-20.321,0.0,0.0,null,null],[4875.0,null,null,1989.74,1072.942,null,130.455,104.224,0.0,0.0,null,0.932,0.892,1.0,null,0.0,0.0,0.0,null,-2.24,-14.914,0.0,0.0,null,null],[4875.5,null,null,1991.111,1034.221,null,129.348,106.594,0.0,0.0,null,0.967,0.93,1.017,null,0.0,0.0,0.0,null,-2.393,-10.531,0.0,0.0,null,null],[4876.0,null,null,1995.041,993.412,null,128.1,108.12,0.0,0.0,null,1.007,0.967,1.046,null,0.0,0.0,0.0,null,-2.489,-5.641,0.0,0.0,null,null],[4876.5,null,null,1997.941,951.387,null,126.962,107.464,0.0,0.0,null,1.051,1.004,1.115,null,0.0,0.0,0.0,null,-2.734,-0.449,0.0,0.0,null,null],[4877.0,null,null,1995.955,910.666,null,126.233,105.026,0.0,0.0,null,1.098,1.045,1.18,null,0.0,0.0,0.0,null,-3.115,-3.378,0.0,0.0,null,null],[4877.5,null,null,1990.493,876.543,null,125.982,103.642,0.0,0.0,null,1.141,1.089,1.193,null,0.0,0.0,0.0,null,-3.463,-11.383,0.0,0.0,null,null],[4878.0,null,null,1988.205,851.563,null,125.962,104.277,0.0,0.0,null,1.174,1.128,1.219,null,0.0,0.0,0.0,null,-3.795,-12.717,0.0,0.0,null,null],[4878.5,null,null,1993.744,832.289,null,125.881,105.411,0.0,0.0,null,1.202,1.153,1.338,null,0.0,0.0,0.0,null,-4.263,-10.617,0.0,0.0,null,null],[4879.0,null,null,2002.319,815.765,null,125.656,106.941,0.0,0.0,null,1.226,1.169,1.435,null,0.0,0.0,0.0,null,-4.818,-11.515,0.0,0.0,null,null],[4879.5,null,null,2005.843,799.358,null,125.446,109.184,0.0,0.0,null,1.251,1.188,1.392,null,0.0,0.0,0.0,null,-5.235,-11.671,0.0,0.0,null,null],[4880.0,null,null,2003.645,780.528,null,125.313,110.724,0.0,0.0,null,1.281,1.214,1.274,null,0.0,0.0,0.0,null,-5.496,-8.874,0.0,0.0,null,null],[4880.5,null,null,2002.928,761.178,null,124.826,110.1,0.0,0.0,null,1.314,1.246,1.224,null,0.0,0.0,0.0,null,-5.825,-6.897,0.0,0.0,null,null],[4881.0,null,null,2000.984,742.754,null,123.387,107.352,0.0,0.0,null,1.346,1.289,1.246,null,0.0,0.0,0.0,null,-6.314,-8.519,0.0,0.0,null,null],[4881.5,null,null,1993.474,724.935,null,120.947,104.198,0.0,0.0,null,1.379,1.343,1.307,null,0.0,0.0,0.0,null,-6.772,-13.24,0.0,0.0,null,null],[4882.0,null,null,1987.221,711.488,null,118.244,102.402,0.0,0.0,null,1.406,1.402,1.35,null,0.0,0.0,0.0,null,-6.931,-16.34,0.0,0.0,null,null],[4882.5,null,null,1985.269,707.5,null,116.177,100.81,0.0,0.0,null,1.413,1.443,1.45,null,0.0,0.0,0.0,null,-6.954,-16.699,0.0,0.0,null,null],[4883.0,null,null,1982.45,713.955,null,115.082,98.299,0.0,0.0,null,1.401,1.435,1.834,null,0.0,0.0,0.0,null,-7.427,-16.806,0.0,0.0,null,null],[4883.5,null,null,1978.972,731.396,null,114.711,96.571,0.0,0.0,null,1.367,1.373,2.449,null,0.0,0.0,0.0,null,-8.224,-16.221,0.0,0.0,null,null],[4884.0,null,null,1976.639,758.6,null,114.864,98.298,0.0,0.0,null,1.318,1.284,2.566,null,0.0,0.0,0.0,null,-8.201,-15.06,0.0,0.0,null,null],[4884.5,null,null,1972.445,789.178,null,115.77,103.591,0.0,0.0,null,1.267,1.198,1.926,null,0.0,0.0,0.0,null,-6.817,-14.796,0.0,0.0,null,null],[4885.0,null,null,1968.215,817.89,null,117.586,111.012,0.0,0.0,null,1.223,1.128,1.361,null,0.0,0.0,0.0,null,-4.901,-13.51,0.0,0.0,null,null],[4885.5,null,null,1965.901,845.037,null,119.855,117.228,0.0,0.0,null,1.183,1.079,1.089,null,0.0,0.0,0.0,null,-3.467,-9.583,0.0,0.0,null,null],[4886.0,null,null,1967.139,873.042,null,121.76,118.872,0.0,0.0,null,1.145,1.056,0.972,null,0.0,0.0,0.0,null,-2.753,-5.876,0.0,0.0,null,null],[4886.5,null,null,1975.316,903.229,null,122.864,116.39,0.0,0.0,null,1.107,1.051,0.926,null,0.0,0.0,0.0,null,-2.507,-8.938,0.0,0.0,null,null],[4887.0,null,null,1988.886,936.148,null,123.408,112.446,0.0,0.0,null,1.068,1.045,0.981,null,0.0,0.0,0.0,null,-2.541,-17.016,0.0,0.0,null,null],[4887.5,null,null,1995.993,966.095,null,123.853,108.736,0.0,0.0,null,1.035,1.023,1.204,null,0.0,0.0,0.0,null,-2.959,-21.626,0.0,0.0,null,null],[4888.0,null,null,1992.109,984.154,null,124.364,106.166,0.0,0.0,null,1.016,0.993,1.447,null,0.0,0.0,0.0,null,-3.491,-22.618,0.0,0.0,null,null],[4888.5,null,null,1988.606,989.778,null,124.85,106.728,0.0,0.0,null,1.01,0.963,1.426,null,0.0,0.0,0.0,null,-3.416,-25.92,0.0,0.0,null,null],[4889.0,null,null,1989.344,992.83,null,125.473,108.813,0.0,0.0,null,1.007,0.937,1.222,null,0.0,0.0,0.0,null,-2.736,-29.812,0.0,0.0,null,null],[4889.5,null,null,1985.156,1001.95,null,126.682,110.249,0.0,0.0,null,0.998,0.917,1.083,null,0.0,0.0,0.0,null,-2.208,-28.658,0.0,0.0,null,null],[4890.0,null,null,1976.593,1020.901,null,128.649,110.836,0.0,0.0,null,0.98,0.899,1.017,null,0.0,0.0,0.0,null,-2.112,-23.362,0.0,0.0,null,null],[4890.5,null,null,1971.129,1048.736,null,130.909,111.881,0.0,0.0,null,0.954,0.882,0.961,null,0.0,0.0,0.0,null,-2.095,-18.371,0.0,0.0,null,null],[4891.0,null,null,1973.238,1079.127,null,132.724,113.494,0.0,0.0,null,0.927,0.866,0.959,null,0.0,0.0,0.0,null,-2.052,-16.377,0.0,0.0,null,null],[4891.5,null,null,1983.091,1104.2,null,133.75,114.748,0.0,0.0,null,0.906,0.853,1.002,null,0.0,0.0,0.0,null,-2.103,-17.466,0.0,0.0,null,null],[4892.0,null,null,1992.15,1121.927,null,134.176,114.915,0.0,0.0,null,0.891,0.845,1.012,null,0.0,0.0,0.0,null,-2.214,-19.189,0.0,0.0,null,null],[4892.5,null,null,1989.435,1132.58,null,134.314,114.099,0.0,0.0,null,0.883,0.843,0.984,null,0.0,0.0,0.0,null,-2.226,-18.187,0.0,0.0,null,null],[4893.0,null,null,1978.894,1133.639,null,134.237,112.838,0.0,0.0,null,0.882,0.844,0.955,null,0.0,0.0,0.0,null,-2.12,-17.814,0.0,0.0,null,null],[4893.5,null,null,1971.29,1122.056,null,133.804,111.725,0.0,0.0,null,0.891,0.846,0.945,null,0.0,0.0,0.0,null,-2.039,-20.104,0.0,0.0,null,null],[4894.0,null,null,1972.755,1096.624,null,132.854,111.783,0.0,0.0,null,0.912,0.858,0.928,null,0.0,0.0,0.0,null,-2.113,-20.823,0.0,0.0,null,null],[4894.5,null,null,1983.307,1057.018,null,131.39,112.106,0.0,0.0,null,0.946,0.888,0.908,null,0.0,0.0,0.0,null,-2.333,-19.369,0.0,0.0,null,null],[4895.0,null,null,1994.127,1007.366,null,129.676,112.198,0.0,0.0,null,0.993,0.938,0.937,null,0.0,0.0,0.0,null,-2.587,-18.088,0.0,0.0,null,null],[4895.5,null,null,1995.227,961.611,null,128.09,112.701,0.0,0.0,null,1.04,1.004,0.978,null,0.0,0.0,0.0,null,-2.76,-13.962,0.0,0.0,null,null],[4896.0,null,null,1990.853,934.794,null,126.928,112.123,0.0,0.0,null,1.07,1.061,1.096,null,0.0,0.0,0.0,null,-2.954,-6.755,0.0,0.0,null,null],[4896.5,null,null,1991.524,928.323,null,126.253,108.233,0.0,0.0,null,1.077,1.074,1.468,null,0.0,0.0,0.0,null,-3.411,-3.377,0.0,0.0,null,null],[4897.0,null,null,2003.454,932.032,null,125.924,104.022,0.0,0.0,null,1.073,1.037,2.126,null,0.0,0.0,0.0,null,-3.977,-7.96,0.0,0.0,null,null],[4897.5,null,null,2021.906,941.116,null,125.804,104.92,0.0,0.0,null,1.063,0.981,2.221,null,0.0,0.0,0.0,null,-4.013,-15.072,0.0,0.0,null,null],[4898.0,null,null,2030.676,957.447,null,126.031,110.593,0.0,0.0,null,1.044,0.936,1.42,null,0.0,0.0,0.0,null,-3.28,-15.515,0.0,0.0,null,null],[4898.5,null,null,2023.739,985.111,null,126.907,115.93,0.0,0.0,null,1.015,0.903,0.902,null,0.0,0.0,0.0,null,-2.28,-8.944,0.0,0.0,null,null],[4899.0,null,null,2012.318,1029.278,null,128.46,118.17,0.0,0.0,null,0.972,0.872,0.822,null,0.0,0.0,0.0,null,-1.551,-4.584,0.0,0.0,null,null],[4899.5,null,null,2005.136,1085.675,null,130.263,118.684,0.0,0.0,null,0.921,0.844,0.875,null,0.0,0.0,0.0,null,-1.185,-7.335,0.0,0.0,null,null],[4900.0,null,null,2000.586,1135.301,null,131.679,119.055,0.0,0.0,null,0.881,0.823,0.881,null,0.0,0.0,0.0,null,-1.026,-13.17,0.0,0.0,null,null],[4900.5,null,null,2001.055,1162.623,null,132.417,120.299,0.0,0.0,null,0.86,0.811,0.887,null,0.0,0.0,0.0,null,-0.935,-16.714,0.0,0.0,null,null],[4901.0,null,null,2003.141,1165.741,null,132.718,122.089,0.0,0.0,null,0.858,0.811,0.906,null,0.0,0.0,0.0,null,-0.862,-12.943,0.0,0.0,null,null],[4901.5,null,null,2000.405,1150.931,null,132.98,122.675,0.0,0.0,null,0.869,0.821,0.917,null,0.0,0.0,0.0,null,-0.822,-2.904,0.0,0.0,null,null],[4902.0,null,null,1996.997,1131.1,null,133.352,121.607,0.0,0.0,null,0.884,0.838,0.928,null,0.0,0.0,0.0,null,-0.848,5.226,0.0,0.0,null,null],[4902.5,null,null,2000.054,1120.175,null,133.664,120.334,0.0,0.0,null,0.893,0.85,0.975,null,0.0,0.0,0.0,null,-0.958,7.363,0.0,0.0,null,null],[4903.0,null,null,2004.751,1121.686,null,133.739,118.793,0.0,0.0,null,0.892,0.852,1.088,null,0.0,0.0,0.0,null,-1.124,8.922,0.0,0.0,null,null],[4903.5,null,null,2005.125,1129.754,null,133.643,116.226,0.0,0.0,null,0.885,0.841,1.215,null,0.0,0.0,0.0,null,-1.262,9.634,0.0,0.0,null,null],[4904.0,null,null,2003.16,1139.79,null,133.625,113.682,0.0,0.0,null,0.877,0.825,1.206,null,0.0,0.0,0.0,null,-1.27,6.36,0.0,0.0,null,null],[4904.5,null,null,1998.362,1151.911,null,133.797,112.233,0.0,0.0,null,0.868,0.809,1.084,null,0.0,0.0,0.0,null,-1.136,2.095,0.0,0.0,null,null],[4905.0,null,null,1993.902,1164.779,null,133.968,111.822,0.0,0.0,null,0.859,0.796,0.948,null,0.0,0.0,0.0,null,-0.971,-2.21,0.0,0.0,null,null],[4905.5,null,null,1995.184,1173.184,null,133.777,111.884,0.0,0.0,null,0.852,0.79,0.876,null,0.0,0.0,0.0,null,-0.966,-6.591,0.0,0.0,null,null],[4906.0,null,null,1994.802,1168.892,null,132.919,112.073,0.0,0.0,null,0.856,0.797,0.843,null,0.0,0.0,0.0,null,-1.228,-6.634,0.0,0.0,null,null],[4906.5,null,null,1986.654,1142.961,null,131.438,111.543,0.0,0.0,null,0.875,0.823,0.816,null,0.0,0.0,0.0,null,-1.617,-1.064,0.0,0.0,null,null],[4907.0,null,null,1982.674,1097.612,null,129.691,110.454,0.0,0.0,null,0.911,0.87,0.819,null,0.0,0.0,0.0,null,-1.899,5.47,0.0,0.0,null,null],[4907.5,null,null,1999.005,1043.754,null,128.044,108.578,0.0,0.0,null,0.958,0.933,0.854,null,0.0,0.0,0.0,null,-2.093,11.85,0.0,0.0,null,null],[4908.0,null,null,2021.015,986.661,null,126.681,105.925,0.0,0.0,null,1.014,1.002,0.977,null,0.0,0.0,0.0,null,-2.424,17.122,0.0,0.0,null,null],[4908.5,null,null,2020.621,927.281,null,125.624,103.503,0.0,0.0,null,1.078,1.069,1.297,null,0.0,0.0,0.0,null,-2.968,17.165,0.0,0.0,null,null],[4909.0,null,null,2000.783,871.877,null,124.867,102.192,0.0,0.0,null,1.147,1.127,1.657,null,0.0,0.0,0.0,null,-3.591,14.311,0.0,0.0,null,null],[4909.5,null,null,1993.064,826.893,null,124.475,102.411,0.0,0.0,null,1.209,1.172,1.754,null,0.0,0.0,0.0,null,-4.125,12.196,0.0,0.0,null,null],[4910.0,null,null,2001.28,796.387,null,124.586,103.131,0.0,0.0,null,1.256,1.197,1.664,null,0.0,0.0,0.0,null,-4.532,10.123,0.0,0.0,null,null],[4910.5,null,null,2004.169,782.111,null,125.249,102.819,0.0,0.0,null,1.279,1.202,1.58,null,0.0,0.0,0.0,null,-4.814,8.04,0.0,0.0,null,null],[4911.0,null,null,1998.679,781.699,null,126.305,102.509,0.0,0.0,null,1.279,1.191,1.488,null,0.0,0.0,0.0,null,-4.928,8.076,0.0,0.0,null,null],[4911.5,null,null,1997.297,790.042,null,127.315,105.014,0.0,0.0,null,1.266,1.176,1.334,null,0.0,0.0,0.0,null,-4.849,11.087,0.0,0.0,null,null],[4912.0,null,null,2000.254,801.754,null,127.732,108.809,0.0,0.0,null,1.247,1.163,1.222,null,0.0,0.0,0.0,null,-4.726,16.121,0.0,0.0,null,null],[4912.5,null,null,2000.431,813.095,null,127.281,111.119,0.0,0.0,null,1.23,1.157,1.182,null,0.0,0.0,0.0,null,-4.647,17.157,0.0,0.0,null,null],[4913.0,null,null,1995.344,825.432,null,126.234,111.631,0.0,0.0,null,1.211,1.16,1.168,null,0.0,0.0,0.0,null,-4.482,12.619,0.0,0.0,null,null],[4913.5,null,null,1987.272,842.516,null,125.23,110.102,0.0,0.0,null,1.187,1.161,1.121,null,0.0,0.0,0.0,null,-4.209,7.596,0.0,0.0,null,null],[4914.0,null,null,1984.109,864.118,null,124.715,106.851,0.0,0.0,null,1.157,1.14,1.238,null,0.0,0.0,0.0,null,-4.377,3.438,0.0,0.0,null,null],[4914.5,null,null,1989.415,886.208,null,124.616,105.169,0.0,0.0,null,1.128,1.092,1.647,null,0.0,0.0,0.0,null,-5.243,1.114,0.0,0.0,null,null],[4915.0,null,null,1993.566,904.969,null,124.595,107.941,0.0,0.0,null,1.105,1.037,1.848,null,0.0,0.0,0.0,null,-5.585,6.209,0.0,0.0,null,null],[4915.5,null,null,1989.824,920.431,null,124.592,114.573,0.0,0.0,null,1.086,0.997,1.445,null,0.0,0.0,0.0,null,-4.463,13.509,0.0,0.0,null,null],[4916.0,null,null,1988.232,938.537,null,124.93,121.498,0.0,0.0,null,1.065,0.972,1.03,null,0.0,0.0,0.0,null,-2.821,12.611,0.0,0.0,null,null],[4916.5,null,null,1993.576,965.246,null,125.864,125.859,0.0,0.0,null,1.036,0.952,0.957,null,0.0,0.0,0.0,null,-1.88,4.868,0.0,0.0,null,null],[4917.0,null,null,1995.217,998.395,null,127.234,126.982,0.0,0.0,null,1.002,0.932,1.006,null,0.0,0.0,0.0,null,-1.565,-0.966,0.0,0.0,null,null],[4917.5,null,null,1992.172,1032.17,null,128.574,125.394,0.0,0.0,null,0.969,0.913,1.02,null,0.0,0.0,0.0,null,-1.459,-4.699,0.0,0.0,null,null],[4918.0,null,null,1996.516,1060.434,null,129.517,123.077,0.0,0.0,null,0.943,0.896,1.022,null,0.0,0.0,0.0,null,-1.347,-11.079,0.0,0.0,null,null],[4918.5,null,null,2007.694,1076.92,null,130.035,122.729,0.0,0.0,null,0.929,0.88,1.005,null,0.0,0.0,0.0,null,-1.125,-18.017,0.0,0.0,null,null],[4919.0,null,null,2013.907,1082.319,null,130.344,124.371,0.0,0.0,null,0.924,0.867,0.997,null,0.0,0.0,0.0,null,-0.906,-19.537,0.0,0.0,null,null],[4919.5,null,null,2011.575,1085.117,null,130.652,125.63,0.0,0.0,null,0.922,0.858,0.987,null,0.0,0.0,0.0,null,-0.801,-17.229,0.0,0.0,null,null],[4920.0,null,null,2002.693,1088.796,null,131.044,124.276,0.0,0.0,null,0.918,0.854,0.963,null,0.0,0.0,0.0,null,-0.767,-13.602,0.0,0.0,null,null],[4920.5,null,null,1990.244,1093.161,null,131.514,121.685,0.0,0.0,null,0.915,0.856,0.965,null,0.0,0.0,0.0,null,-0.754,-6.431,0.0,0.0,null,null],[4921.0,null,null,1980.685,1099.473,null,131.94,119.469,0.0,0.0,null,0.91,0.858,0.98,null,0.0,0.0,0.0,null,-0.766,0.96,0.0,0.0,null,null],[4921.5,null,null,1974.003,1107.425,null,132.142,117.183,0.0,0.0,null,0.903,0.857,0.993,null,0.0,0.0,0.0,null,-0.806,-0.162,0.0,0.0,null,null],[4922.0,null,null,1966.069,1114.91,null,132.06,113.933,0.0,0.0,null,0.897,0.853,1.038,null,0.0,0.0,0.0,null,-0.839,-8.025,0.0,0.0,null,null],[4922.5,null,null,1958.965,1120.861,null,131.821,112.366,0.0,0.0,null,0.892,0.847,1.086,null,0.0,0.0,0.0,null,-0.822,-12.389,0.0,0.0,null,null],[4923.0,null,null,1956.922,1123.914,null,131.596,114.016,0.0,0.0,null,0.89,0.838,1.067,null,0.0,0.0,0.0,null,-0.774,-11.406,0.0,0.0,null,null],[4923.5,null,null,1957.442,1125.846,null,131.46,115.721,0.0,0.0,null,0.888,0.83,0.994,null,0.0,0.0,0.0,null,-0.78,-10.111,0.0,0.0,null,null],[4924.0,null,null,1960.143,1131.734,null,131.353,114.065,0.0,0.0,null,0.884,0.823,0.954,null,0.0,0.0,0.0,null,-0.901,-12.422,0.0,0.0,null,null],[4924.5,null,null,1969.58,1143.029,null,131.186,111.06,0.0,0.0,null,0.875,0.817,0.946,null,0.0,0.0,0.0,null,-1.058,-17.082,0.0,0.0,null,null],[4925.0,null,null,1980.212,1155.927,null,130.993,110.469,0.0,0.0,null,0.865,0.815,0.94,null,0.0,0.0,0.0,null,-1.149,-21.77,0.0,0.0,null,null],[4925.5,null,null,1985.224,1166.917,null,130.883,111.222,0.0,0.0,null,0.857,0.813,0.935,null,0.0,0.0,0.0,null,-1.182,-24.617,0.0,0.0,null,null],[4926.0,null,null,1987.939,1172.674,null,130.886,111.243,0.0,0.0,null,0.853,0.813,0.942,null,0.0,0.0,0.0,null,-1.253,-22.991,0.0,0.0,null,null],[4926.5,null,null,1992.839,1167.008,null,130.866,110.288,0.0,0.0,null,0.857,0.816,0.962,null,0.0,0.0,0.0,null,-1.404,-17.997,0.0,0.0,null,null],[4927.0,null,null,1996.333,1145.202,null,130.662,109.155,0.0,0.0,null,0.873,0.827,0.973,null,0.0,0.0,0.0,null,-1.627,-15.215,0.0,0.0,null,null],[4927.5,null,null,1993.464,1110.806,null,130.263,108.049,0.0,0.0,null,0.9,0.851,0.943,null,0.0,0.0,0.0,null,-1.825,-16.122,0.0,0.0,null,null],[4928.0,null,null,1985.431,1071.724,null,129.83,107.818,0.0,0.0,null,0.933,0.889,0.893,null,0.0,0.0,0.0,null,-1.93,-14.62,0.0,0.0,null,null],[4928.5,null,null,1981.167,1034.715,null,129.504,108.044,0.0,0.0,null,0.966,0.935,0.904,null,0.0,0.0,0.0,null,-2.043,-8.275,0.0,0.0,null,null],[4929.0,null,null,1984.542,1003.376,null,129.236,107.114,0.0,0.0,null,0.997,0.973,1.066,null,0.0,0.0,0.0,null,-2.31,-2.301,0.0,0.0,null,null],[4929.5,null,null,1989.135,974.787,null,128.9,105.505,0.0,0.0,null,1.026,0.998,1.319,null,0.0,0.0,0.0,null,-2.727,-3.431,0.0,0.0,null,null],[4930.0,null,null,1987.889,944.513,null,128.485,106.317,0.0,0.0,null,1.059,1.019,1.422,null,0.0,0.0,0.0,null,-3.145,-7.683,0.0,0.0,null,null],[4930.5,null,null,1980.512,914.787,null,128.174,108.771,0.0,0.0,null,1.093,1.038,1.325,null,0.0,0.0,0.0,null,-3.398,-6.381,0.0,0.0,null,null],[4931.0,null,null,1974.074,892.139,null,128.189,109.968,0.0,0.0,null,1.121,1.052,1.231,null,0.0,0.0,0.0,null,-3.478,0.754,0.0,0.0,null,null],[4931.5,null,null,1976.931,879.733,null,128.54,110.33,0.0,0.0,null,1.137,1.057,1.215,null,0.0,0.0,0.0,null,-3.505,4.914,0.0,0.0,null,null],[4932.0,null,null,1987.617,876.483,null,129.025,111.767,0.0,0.0,null,1.141,1.056,1.221,null,0.0,0.0,0.0,null,-3.568,2.518,0.0,0.0,null,null],[4932.5,null,null,1995.277,878.537,null,129.461,114.051,0.0,0.0,null,1.138,1.054,1.219,null,0.0,0.0,0.0,null,-3.634,-0.645,0.0,0.0,null,null],[4933.0,null,null,1993.733,882.959,null,129.856,115.702,0.0,0.0,null,1.133,1.053,1.204,null,0.0,0.0,0.0,null,-3.63,0.121,0.0,0.0,null,null],[4933.5,null,null,1991.472,887.695,null,130.277,114.974,0.0,0.0,null,1.127,1.054,1.203,null,0.0,0.0,0.0,null,-3.576,-0.091,0.0,0.0,null,null],[4934.0,null,null,1996.791,891.085,null,130.683,112.681,0.0,0.0,null,1.122,1.053,1.195,null,0.0,0.0,0.0,null,-3.542,-6.376,0.0,0.0,null,null],[4934.5,null,null,2003.125,892.979,null,130.91,111.919,0.0,0.0,null,1.12,1.053,1.157,null,0.0,0.0,0.0,null,-3.562,-13.302,0.0,0.0,null,null],[4935.0,null,null,2002.313,894.473,null,130.777,112.659,0.0,0.0,null,1.118,1.053,1.12,null,0.0,0.0,0.0,null,-3.633,-13.462,0.0,0.0,null,null],[4935.5,null,null,1997.89,894.206,null,130.231,112.518,0.0,0.0,null,1.118,1.054,1.141,null,0.0,0.0,0.0,null,-3.711,-9.397,0.0,0.0,null,null],[4936.0,null,null,1995.187,890.799,null,129.345,111.907,0.0,0.0,null,1.123,1.055,1.189,null,0.0,0.0,0.0,null,-3.724,-9.902,0.0,0.0,null,null],[4936.5,null,null,1991.968,884.813,null,128.241,112.727,0.0,0.0,null,1.13,1.058,1.202,null,0.0,0.0,0.0,null,-3.656,-13.419,0.0,0.0,null,null],[4937.0,null,null,1989.255,876.09,null,127.045,113.794,0.0,0.0,null,1.141,1.068,1.173,null,0.0,0.0,0.0,null,-3.526,-10.095,0.0,0.0,null,null],[4937.5,null,null,1990.687,865.568,null,125.898,112.956,0.0,0.0,null,1.155,1.086,1.131,null,0.0,0.0,0.0,null,-3.363,-2.479,0.0,0.0,null,null],[4938.0,null,null,1993.088,857.198,null,124.939,110.939,0.0,0.0,null,1.167,1.103,1.14,null,0.0,0.0,0.0,null,-3.273,-2.175,0.0,0.0,null,null],[4938.5,null,null,1993.215,852.872,null,124.283,108.54,0.0,0.0,null,1.173,1.11,1.26,null,0.0,0.0,0.0,null,-3.371,-7.296,0.0,0.0,null,null],[4939.0,null,null,1999.057,852.385,null,124.0,105.792,0.0,0.0,null,1.173,1.107,1.387,null,0.0,0.0,0.0,null,-3.552,-9.753,0.0,0.0,null,null],[4939.5,null,null,2009.22,857.928,null,124.077,103.878,0.0,0.0,null,1.166,1.095,1.38,null,0.0,0.0,0.0,null,-3.551,-10.087,0.0,0.0,null,null],[4940.0,null,null,2010.975,871.824,null,124.467,104.22,0.0,0.0,null,1.147,1.071,1.306,null,0.0,0.0,0.0,null,-3.352,-10.681,0.0,0.0,null,null],[4940.5,null,null,1999.96,891.784,null,125.127,105.461,0.0,0.0,null,1.121,1.039,1.274,null,0.0,0.0,0.0,null,-3.2,-10.156,0.0,0.0,null,null],[4941.0,null,null,1986.742,912.081,null,125.971,106.48,0.0,0.0,null,1.096,1.007,1.248,null,0.0,0.0,0.0,null,-3.133,-9.977,0.0,0.0,null,null],[4941.5,null,null,1981.948,927.257,null,126.803,106.59,0.0,0.0,null,1.078,0.985,1.108,null,0.0,0.0,0.0,null,-2.964,-10.892,0.0,0.0,null,null],[4942.0,null,null,1983.681,933.797,null,127.359,105.748,0.0,0.0,null,1.071,0.978,0.974,null,0.0,0.0,0.0,null,-2.729,-10.085,0.0,0.0,null,null],[4942.5,null,null,1982.48,931.974,null,127.443,105.076,0.0,0.0,null,1.073,0.988,0.932,null,0.0,0.0,0.0,null,-2.6,-6.411,0.0,0.0,null,null],[4943.0,null,null,1978.499,923.681,null,126.989,105.422,0.0,0.0,null,1.083,1.013,0.944,null,0.0,0.0,0.0,null,-2.628,-1.412,0.0,0.0,null,null],[4943.5,null,null,1976.46,906.961,null,125.964,104.663,0.0,0.0,null,1.103,1.05,1.0,null,0.0,0.0,0.0,null,-2.814,3.067,0.0,0.0,null,null],[4944.0,null,null,1977.573,879.261,null,124.318,101.847,0.0,0.0,null,1.137,1.098,1.088,null,0.0,0.0,0.0,null,-3.167,4.884,0.0,0.0,null,null],[4944.5,null,null,1982.485,843.67,null,122.105,99.304,0.0,0.0,null,1.185,1.156,1.201,null,0.0,0.0,0.0,null,-3.614,2.175,0.0,0.0,null,null],[4945.0,null,null,1988.233,806.863,null,119.621,97.453,0.0,0.0,null,1.239,1.219,1.31,null,0.0,0.0,0.0,null,-4.035,-1.702,0.0,0.0,null,null],[4945.5,null,null,1988.372,773.825,null,117.327,95.682,0.0,0.0,null,1.292,1.274,1.405,null,0.0,0.0,0.0,null,-4.37,-3.469,0.0,0.0,null,null],[4946.0,null,null,1984.328,747.921,null,115.577,94.511,0.0,0.0,null,1.337,1.311,1.514,null,0.0,0.0,0.0,null,-4.644,-6.668,0.0,0.0,null,null],[4946.5,null,null,1982.561,729.436,null,114.384,93.851,0.0,0.0,null,1.371,1.328,1.622,null,0.0,0.0,0.0,null,-4.879,-13.749,0.0,0.0,null,null],[4947.0,null,null,1986.9,716.874,null,113.538,93.028,0.0,0.0,null,1.395,1.332,1.713,null,0.0,0.0,0.0,null,-5.011,-21.497,0.0,0.0,null,null],[4947.5,null,null,1995.065,710.91,null,113.056,93.845,0.0,0.0,null,1.407,1.331,1.695,null,0.0,0.0,0.0,null,-4.993,-25.543,0.0,0.0,null,null],[4948.0,null,null,1999.872,714.049,null,113.275,96.726,0.0,0.0,null,1.4,1.328,1.523,null,0.0,0.0,0.0,null,-4.842,-27.201,0.0,0.0,null,null],[4948.5,null,null,1996.579,729.63,null,114.429,98.973,0.0,0.0,null,1.371,1.314,1.39,null,0.0,0.0,0.0,null,-4.585,-27.044,0.0,0.0,null,null],[4949.0,null,null,1990.94,758.701,null,116.339,99.656,0.0,0.0,null,1.318,1.269,1.406,null,0.0,0.0,0.0,null,-4.293,-24.628,0.0,0.0,null,null],[4949.5,null,null,1985.853,797.527,null,118.5,100.954,0.0,0.0,null,1.254,1.193,1.559,null,0.0,0.0,0.0,null,-4.029,-21.145,0.0,0.0,null,null],[4950.0,null,null,1979.303,839.203,null,120.499,103.342,0.0,0.0,null,1.192,1.11,1.608,null,0.0,0.0,0.0,null,-3.702,-19.261,0.0,0.0,null,null],[4950.5,null,null,1976.462,878.363,null,122.317,106.312,0.0,0.0,null,1.138,1.042,1.404,null,0.0,0.0,0.0,null,-3.109,-16.565,0.0,0.0,null,null],[4951.0,null,null,1982.847,913.987,null,124.184,110.956,0.0,0.0,null,1.094,0.992,1.111,null,0.0,0.0,0.0,null,-2.282,-13.719,0.0,0.0,null,null],[4951.5,null,null,1990.408,949.985,null,126.115,116.511,0.0,0.0,null,1.053,0.956,0.937,null,0.0,0.0,0.0,null,-1.584,-12.614,0.0,0.0,null,null],[4952.0,null,null,1988.736,989.683,null,127.796,120.663,0.0,0.0,null,1.01,0.929,0.926,null,0.0,0.0,0.0,null,-1.235,-11.988,0.0,0.0,null,null],[4952.5,null,null,1984.14,1028.426,null,128.914,123.606,0.0,0.0,null,0.972,0.907,0.968,null,0.0,0.0,0.0,null,-1.103,-12.654,0.0,0.0,null,null],[4953.0,null,null,1987.298,1057.837,null,129.448,125.34,0.0,0.0,null,0.945,0.889,1.01,null,0.0,0.0,0.0,null,-1.042,-18.66,0.0,0.0,null,null],[4953.5,null,null,1990.003,1075.409,null,129.684,124.307,0.0,0.0,null,0.93,0.874,1.028,null,0.0,0.0,0.0,null,-1.04,-26.733,0.0,0.0,null,null],[4954.0,null,null,1979.541,1084.411,null,129.919,122.714,0.0,0.0,null,0.922,0.863,1.028,null,0.0,0.0,0.0,null,-1.027,-30.752,0.0,0.0,null,null],[4954.5,null,null,1963.28,1090.75,null,130.218,122.477,0.0,0.0,null,0.917,0.853,1.014,null,0.0,0.0,0.0,null,-0.934,-29.367,0.0,0.0,null,null],[4955.0,null,null,1958.966,1099.792,null,130.467,122.166,0.0,0.0,null,0.909,0.844,1.002,null,0.0,0.0,0.0,null,-0.87,-21.653,0.0,0.0,null,null],[4955.5,null,null,1972.475,1112.309,null,130.605,120.179,0.0,0.0,null,0.899,0.836,0.986,null,0.0,0.0,0.0,null,-0.943,-11.512,0.0,0.0,null,null],[4956.0,null,null,1992.327,1124.612,null,130.723,117.645,0.0,0.0,null,0.889,0.829,0.971,null,0.0,0.0,0.0,null,-0.997,-9.431,0.0,0.0,null,null],[4956.5,null,null,1996.332,1133.706,null,130.902,115.659,0.0,0.0,null,0.882,0.824,0.973,null,0.0,0.0,0.0,null,-0.87,-15.593,0.0,0.0,null,null],[4957.0,null,null,1984.762,1136.684,null,131.073,115.174,0.0,0.0,null,0.88,0.823,0.979,null,0.0,0.0,0.0,null,-0.742,-21.479,0.0,0.0,null,null],[4957.5,null,null,1977.449,1127.539,null,131.053,115.799,0.0,0.0,null,0.887,0.83,0.951,null,0.0,0.0,0.0,null,-0.866,-24.438,0.0,0.0,null,null],[4958.0,null,null,1980.143,1104.617,null,130.712,116.094,0.0,0.0,null,0.905,0.852,0.897,null,0.0,0.0,0.0,null,-1.245,-22.99,0.0,0.0,null,null],[4958.5,null,null,1980.598,1070.368,null,130.04,115.671,0.0,0.0,null,0.934,0.89,0.878,null,0.0,0.0,0.0,null,-1.729,-14.828,0.0,0.0,null,null],[4959.0,null,null,1975.764,1026.681,null,129.056,114.537,0.0,0.0,null,0.974,0.938,0.948,null,0.0,0.0,0.0,null,-2.216,-8.665,0.0,0.0,null,null],[4959.5,null,null,1974.781,974.38,null,127.69,111.683,0.0,0.0,null,1.026,0.992,1.086,null,0.0,0.0,0.0,null,-2.66,-7.703,0.0,0.0,null,null],[4960.0,null,null,1983.074,919.409,null,125.877,108.134,0.0,0.0,null,1.088,1.054,1.21,null,0.0,0.0,0.0,null,-3.04,-5.355,0.0,0.0,null,null],[4960.5,null,null,1991.006,867.827,null,123.786,107.523,0.0,0.0,null,1.152,1.125,1.265,null,0.0,0.0,0.0,null,-3.365,-4.605,0.0,0.0,null,null],[4961.0,null,null,1985.36,824.225,null,121.8,109.801,0.0,0.0,null,1.213,1.195,1.319,null,0.0,0.0,0.0,null,-3.676,-5.486,0.0,0.0,null,null],[4961.5,null,null,1970.716,793.116,null,120.299,109.61,0.0,0.0,null,1.261,1.243,1.485,null,0.0,0.0,0.0,null,-4.05,1.91,0.0,0.0,null,null],[4962.0,null,null,1961.26,775.329,null,119.447,105.133,0.0,0.0,null,1.29,1.26,1.729,null,0.0,0.0,0.0,null,-4.481,13.961,0.0,0.0,null,null],[4962.5,null,null,1961.333,767.142,null,119.184,100.841,0.0,0.0,null,1.304,1.256,1.812,null,0.0,0.0,0.0,null,-4.793,14.61,0.0,0.0,null,null],[4963.0,null,null,1964.563,760.724,null,119.386,100.699,0.0,0.0,null,1.315,1.243,1.631,null,0.0,0.0,0.0,null,-4.85,3.482,0.0,0.0,null,null],[4963.5,null,null,1967.979,750.028,null,119.959,104.302,0.0,0.0,null,1.333,1.235,1.39,null,0.0,0.0,0.0,null,-4.754,-4.865,0.0,0.0,null,null],[4964.0,null,null,1974.93,737.606,null,120.718,109.155,0.0,0.0,null,1.356,1.247,1.24,null,0.0,0.0,0.0,null,-4.658,-2.395,0.0,0.0,null,null],[4964.5,null,null,1985.081,731.201,null,121.354,112.009,0.0,0.0,null,1.368,1.278,1.187,null,0.0,0.0,0.0,null,-4.605,3.389,0.0,0.0,null,null],[4965.0,null,null,1989.949,733.119,null,121.626,112.396,0.0,0.0,null,1.364,1.311,1.235,null,0.0,0.0,0.0,null,-4.609,2.903,0.0,0.0,null,null],[4965.5,null,null,1988.183,745.376,null,121.549,110.376,0.0,0.0,null,1.342,1.32,1.391,null,0.0,0.0,0.0,null,-4.609,-1.854,0.0,0.0,null,null],[4966.0,null,null,1986.174,771.811,null,121.446,105.66,0.0,0.0,null,1.296,1.283,1.596,null,0.0,0.0,0.0,null,-4.547,-3.785,0.0,0.0,null,null],[4966.5,null,null,1984.771,808.851,null,121.659,100.398,0.0,0.0,null,1.236,1.2,1.873,null,0.0,0.0,0.0,null,-4.529,-1.776,0.0,0.0,null,null],[4967.0,null,null,1983.597,849.861,null,122.23,98.015,0.0,0.0,null,1.177,1.097,2.052,null,0.0,0.0,0.0,null,-4.458,-0.226,0.0,0.0,null,null],[4967.5,null,null,1984.318,893.371,null,123.033,99.377,0.0,0.0,null,1.119,1.004,1.732,null,0.0,0.0,0.0,null,-3.866,-2.542,0.0,0.0,null,null],[4968.0,null,null,1983.254,940.233,null,124.083,103.716,0.0,0.0,null,1.064,0.936,1.161,null,0.0,0.0,0.0,null,-2.714,-9.485,0.0,0.0,null,null],[4968.5,null,null,1980.854,993.204,null,125.514,110.215,0.0,0.0,null,1.007,0.889,0.844,null,0.0,0.0,0.0,null,-1.617,-14.18,0.0,0.0,null,null],[4969.0,null,null,1983.1,1055.132,null,127.316,116.393,0.0,0.0,null,0.948,0.855,0.808,null,0.0,0.0,0.0,null,-0.975,-12.619,0.0,0.0,null,null],[4969.5,null,null,1985.514,1118.442,null,129.192,119.167,0.0,0.0,null,0.894,0.828,0.862,null,0.0,0.0,0.0,null,-0.647,-10.026,0.0,0.0,null,null],[4970.0,null,null,1981.804,1167.222,null,130.7,118.535,0.0,0.0,null,0.857,0.809,0.893,null,0.0,0.0,0.0,null,-0.421,-9.7,0.0,0.0,null,null],[4970.5,null,null,1980.39,1194.153,null,131.619,118.004,0.0,0.0,null,0.837,0.798,0.929,null,0.0,0.0,0.0,null,-0.231,-8.524,0.0,0.0,null,null],[4971.0,null,null,1985.829,1203.528,null,132.137,118.323,0.0,0.0,null,0.831,0.791,0.95,null,0.0,0.0,0.0,null,-0.074,-5.398,0.0,0.0,null,null],[4971.5,null,null,1994.414,1203.331,null,132.572,117.775,0.0,0.0,null,0.831,0.785,0.96,null,0.0,0.0,0.0,null,0.027,-4.576,0.0,0.0,null,null],[4972.0,null,null,2002.469,1201.031,null,133.041,116.75,0.0,0.0,null,0.833,0.78,0.96,null,0.0,0.0,0.0,null,0.058,-7.673,0.0,0.0,null,null],[4972.5,null,null,2008.365,1201.991,null,133.434,115.976,0.0,0.0,null,0.832,0.776,0.938,null,0.0,0.0,0.0,null,0.054,-12.407,0.0,0.0,null,null],[4973.0,null,null,2008.775,1207.244,null,133.616,114.039,0.0,0.0,null,0.828,0.774,0.92,null,0.0,0.0,0.0,null,0.058,-12.625,0.0,0.0,null,null],[4973.5,null,null,2001.544,1215.461,null,133.638,111.701,0.0,0.0,null,0.823,0.774,0.917,null,0.0,0.0,0.0,null,0.068,-7.033,0.0,0.0,null,null],[4974.0,null,null,1992.146,1223.954,null,133.656,112.046,0.0,0.0,null,0.817,0.775,0.92,null,0.0,0.0,0.0,null,0.069,-1.443,0.0,0.0,null,null],[4974.5,null,null,1991.24,1224.889,null,133.737,114.274,0.0,0.0,null,0.816,0.778,0.935,null,0.0,0.0,0.0,null,0.004,-1.284,0.0,0.0,null,null],[4975.0,null,null,1999.177,1209.25,null,133.784,113.918,0.0,0.0,null,0.827,0.788,0.928,null,0.0,0.0,0.0,null,-0.239,-2.194,0.0,0.0,null,null],[4975.5,null,null,2007.192,1173.885,null,133.557,111.063,0.0,0.0,null,0.852,0.811,0.883,null,0.0,0.0,0.0,null,-0.7,-0.894,0.0,0.0,null,null],[4976.0,null,null,2009.29,1123.1,null,132.665,109.32,0.0,0.0,null,0.89,0.852,0.826,null,0.0,0.0,0.0,null,-1.334,-4.335,0.0,0.0,null,null],[4976.5,null,null,2006.837,1062.5,null,130.708,109.017,0.0,0.0,null,0.941,0.911,0.849,null,0.0,0.0,0.0,null,-2.14,-10.824,0.0,0.0,null,null],[4977.0,null,null,2003.646,997.14,null,127.694,108.745,0.0,0.0,null,1.003,0.987,0.947,null,0.0,0.0,0.0,null,-2.931,-13.496,0.0,0.0,null,null],[4977.5,null,null,1997.129,935.489,null,124.293,107.335,0.0,0.0,null,1.069,1.079,1.001,null,0.0,0.0,0.0,null,-3.317,-14.501,0.0,0.0,null,null],[4978.0,null,null,1983.805,888.614,null,121.506,103.065,0.0,0.0,null,1.125,1.169,1.115,null,0.0,0.0,0.0,null,-3.554,-17.175,0.0,0.0,null,null],[4978.5,null,null,1973.523,860.676,null,119.966,95.657,0.0,0.0,null,1.162,1.212,1.689,null,0.0,0.0,0.0,null,-4.772,-18.614,0.0,0.0,null,null],[4979.0,null,null,1972.714,849.083,null,119.536,89.557,0.0,0.0,null,1.178,1.185,3.161,null,0.0,0.0,0.0,null,-6.887,-20.359,0.0,0.0,null,null],[4979.5,null,null,1973.971,851.495,null,119.759,92.083,0.0,0.0,null,1.174,1.122,3.524,null,0.0,0.0,0.0,null,-7.757,-23.895,0.0,0.0,null,null],[4980.0,null,null,1972.264,868.162,null,120.608,103.006,0.0,0.0,null,1.152,1.058,1.741,null,0.0,0.0,0.0,null,-6.168,-28.081,0.0,0.0,null,null],[4980.5,null,null,1970.184,902.754,null,122.515,112.703,0.0,0.0,null,1.108,0.997,1.002,null,0.0,0.0,0.0,null,-3.609,-31.685,0.0,0.0,null,null],[4981.0,null,null,1968.444,959.838,null,125.616,115.842,0.0,0.0,null,1.042,0.932,0.899,null,0.0,0.0,0.0,null,-1.785,-30.782,0.0,0.0,null,null],[4981.5,null,null,1972.019,1032.091,null,129.216,116.454,0.0,0.0,null,0.969,0.873,0.927,null,0.0,0.0,0.0,null,-0.752,-21.895,0.0,0.0,null,null],[4982.0,null,null,1977.929,1103.585,null,132.143,118.568,0.0,0.0,null,0.906,0.829,0.881,null,0.0,0.0,0.0,null,-0.086,-11.068,0.0,0.0,null,null],[4982.5,null,null,1974.68,1165.948,null,133.659,121.35,0.0,0.0,null,0.858,0.802,0.838,null,0.0,0.0,0.0,null,0.307,-8.552,0.0,0.0,null,null],[4983.0,null,null,1966.981,1218.103,null,134.003,121.369,0.0,0.0,null,0.821,0.782,0.835,null,0.0,0.0,0.0,null,0.502,-12.415,0.0,0.0,null,null],[4983.5,null,null,1968.08,1256.838,null,133.912,118.013,0.0,0.0,null,0.796,0.763,0.869,null,0.0,0.0,0.0,null,0.613,-12.876,0.0,0.0,null,null],[4984.0,null,null,1973.386,1276.902,null,133.84,115.774,0.0,0.0,null,0.783,0.75,0.896,null,0.0,0.0,0.0,null,0.678,-12.869,0.0,0.0,null,null],[4984.5,null,null,1972.58,1275.71,null,133.809,116.969,0.0,0.0,null,0.784,0.745,0.867,null,0.0,0.0,0.0,null,0.742,-18.67,0.0,0.0,null,null],[4985.0,null,null,1967.567,1253.976,null,133.707,118.574,0.0,0.0,null,0.797,0.752,0.82,null,0.0,0.0,0.0,null,0.837,-25.805,0.0,0.0,null,null],[4985.5,null,null,1964.56,1217.956,null,133.519,117.255,0.0,0.0,null,0.821,0.771,0.791,null,0.0,0.0,0.0,null,0.916,-28.956,0.0,0.0,null,null],[4986.0,null,null,1965.324,1175.282,null,133.357,113.877,0.0,0.0,null,0.851,0.804,0.795,null,0.0,0.0,0.0,null,0.919,-26.198,0.0,0.0,null,null],[4986.5,null,null,1970.49,1130.782,null,133.299,111.24,0.0,0.0,null,0.884,0.845,0.871,null,0.0,0.0,0.0,null,0.835,-18.552,0.0,0.0,null,null],[4987.0,null,null,1977.441,1087.594,null,133.28,110.747,0.0,0.0,null,0.919,0.889,1.046,null,0.0,0.0,0.0,null,0.696,-14.394,0.0,0.0,null,null],[4987.5,null,null,1979.374,1047.257,null,133.226,110.492,0.0,0.0,null,0.955,0.925,1.259,null,0.0,0.0,0.0,null,0.563,-19.231,0.0,0.0,null,null],[4988.0,null,null,1975.799,1014.154,null,133.165,109.213,0.0,0.0,null,0.986,0.95,1.386,null,0.0,0.0,0.0,null,0.429,-27.009,0.0,0.0,null,null],[4988.5,null,null,1974.593,999.416,null,133.142,107.442,0.0,0.0,null,1.001,0.958,1.373,null,0.0,0.0,0.0,null,0.284,-30.42,0.0,0.0,null,null],[4989.0,null,null,1973.406,1008.531,null,133.186,106.447,0.0,0.0,null,0.992,0.941,1.339,null,0.0,0.0,0.0,null,0.198,-25.376,0.0,0.0,null,null],[4989.5,null,null,1970.885,1033.696,null,133.243,106.02,0.0,0.0,null,0.967,0.9,1.33,null,0.0,0.0,0.0,null,0.182,-14.547,0.0,0.0,null,null],[4990.0,null,null,1974.554,1064.946,null,133.132,105.721,0.0,0.0,null,0.939,0.856,1.225,null,0.0,0.0,0.0,null,0.143,-8.319,0.0,0.0,null,null],[4990.5,null,null,1980.699,1093.648,null,132.725,105.168,0.0,0.0,null,0.914,0.826,0.988,null,0.0,0.0,0.0,null,-0.038,-11.387,0.0,0.0,null,null],[4991.0,null,null,1985.33,1108.338,null,132.028,104.082,0.0,0.0,null,0.902,0.819,0.831,null,0.0,0.0,0.0,null,-0.402,-15.434,0.0,0.0,null,null],[4991.5,null,null,1990.426,1105.181,null,131.096,103.321,0.0,0.0,null,0.905,0.837,0.768,null,0.0,0.0,0.0,null,-0.818,-12.199,0.0,0.0,null,null],[4992.0,null,null,1989.521,1088.397,null,129.995,102.59,0.0,0.0,null,0.919,0.882,0.748,null,0.0,0.0,0.0,null,-1.107,1.971,0.0,0.0,null,null],[4992.5,null,null,1983.31,1059.529,null,128.779,101.604,0.0,0.0,null,0.944,0.936,0.821,null,0.0,0.0,0.0,null,-1.373,17.954,0.0,0.0,null,null],[4993.0,null,null,1984.684,1018.796,null,127.493,101.685,0.0,0.0,null,0.982,0.984,1.067,null,0.0,0.0,0.0,null,-1.879,21.741,0.0,0.0,null,null],[4993.5,null,null,1996.196,970.792,null,126.225,104.318,0.0,0.0,null,1.03,1.025,1.341,null,0.0,0.0,0.0,null,-2.509,14.339,0.0,0.0,null,null],[4994.0,null,null,2007.177,920.241,null,125.114,108.119,0.0,0.0,null,1.087,1.068,1.462,null,0.0,0.0,0.0,null,-2.89,6.748,0.0,0.0,null,null],[4994.5,null,null,2008.842,870.76,null,124.244,110.481,0.0,0.0,null,1.148,1.108,1.415,null,0.0,0.0,0.0,null,-2.954,0.955,0.0,0.0,null,null],[4995.0,null,null,2003.867,829.877,null,123.618,111.13,0.0,0.0,null,1.205,1.138,1.389,null,0.0,0.0,0.0,null,-2.954,-2.869,0.0,0.0,null,null],[4995.5,null,null,2005.266,805.41,null,123.206,111.158,0.0,0.0,null,1.242,1.16,1.368,null,0.0,0.0,0.0,null,-3.007,-2.566,0.0,0.0,null,null],[4996.0,null,null,2012.748,797.434,null,122.962,110.86,0.0,0.0,null,1.254,1.177,1.326,null,0.0,0.0,0.0,null,-3.052,-3.397,0.0,0.0,null,null],[4996.5,null,null,2009.897,797.587,null,122.82,108.991,0.0,0.0,null,1.254,1.188,1.304,null,0.0,0.0,0.0,null,-3.049,-8.036,0.0,0.0,null,null],[4997.0,null,null,1995.458,797.071,null,122.695,106.775,0.0,0.0,null,1.255,1.194,1.307,null,0.0,0.0,0.0,null,-3.025,-9.29,0.0,0.0,null,null],[4997.5,null,null,1985.804,792.388,null,122.475,106.433,0.0,0.0,null,1.262,1.199,1.357,null,0.0,0.0,0.0,null,-2.977,-4.497,0.0,0.0,null,null],[4998.0,null,null,1983.519,784.006,null,122.151,106.841,0.0,0.0,null,1.276,1.207,1.392,null,0.0,0.0,0.0,null,-2.905,0.312,0.0,0.0,null,null],[4998.5,null,null,1979.323,775.853,null,121.893,105.912,0.0,0.0,null,1.289,1.218,1.379,null,0.0,0.0,0.0,null,-2.797,1.733,0.0,0.0,null,null],[4999.0,null,null,1974.907,778.43,null,121.945,105.46,0.0,0.0,null,1.285,1.219,1.374,null,0.0,0.0,0.0,null,-2.601,0.267,0.0,0.0,null,null],[4999.5,null,null,1973.964,801.263,null,122.465,107.203,0.0,0.0,null,1.248,1.191,1.41,null,0.0,0.0,0.0,null,-2.41,-3.037,0.0,0.0,null,null],[5000.0,null,null,1972.64,841.073,null,123.443,108.501,0.0,0.0,null,1.189,1.125,1.499,null,0.0,0.0,0.0,null,-2.357,-2.037,0.0,0.0,null,null],[5000.5,null,null,1968.29,885.612,null,124.759,107.977,0.0,0.0,null,1.129,1.039,1.492,null,0.0,0.0,0.0,null,-2.207,0.576,0.0,0.0,null,null],[5001.0,null,null,1964.741,928.114,null,126.279,107.735,0.0,0.0,null,1.077,0.962,1.266,null,0.0,0.0,0.0,null,-1.569,-4.231,0.0,0.0,null,null],[5001.5,null,null,1967.937,970.525,null,127.939,108.006,0.0,0.0,null,1.03,0.905,0.983,null,0.0,0.0,0.0,null,-0.621,-14.093,0.0,0.0,null,null],[5002.0,null,null,1977.424,1019.624,null,129.637,108.16,0.0,0.0,null,0.981,0.862,0.841,null,0.0,0.0,0.0,null,0.131,-11.593,0.0,0.0,null,null],[5002.5,null,null,1984.509,1076.809,null,131.17,108.739,0.0,0.0,null,0.929,0.827,0.839,null,0.0,0.0,0.0,null,0.513,7.879,0.0,0.0,null,null],[5003.0,null,null,1986.602,1132.596,null,132.374,109.008,0.0,0.0,null,0.883,0.802,0.854,null,0.0,0.0,0.0,null,0.679,20.307,0.0,0.0,null,null],[5003.5,null,null,1993.02,1176.757,null,133.17,108.294,0.0,0.0,null,0.85,0.786,0.873,null,0.0,0.0,0.0,null,0.723,8.808,0.0,0.0,null,null],[5004.0,null,null,2009.213,1205.77,null,133.519,107.94,0.0,0.0,null,0.829,0.777,0.912,null,0.0,0.0,0.0,null,0.626,-7.581,0.0,0.0,null,null],[5004.5,null,null,2022.177,1216.027,null,133.412,108.062,0.0,0.0,null,0.822,0.774,0.953,null,0.0,0.0,0.0,null,0.463,-6.644,0.0,0.0,null,null],[5005.0,null,null,2019.498,1202.736,null,132.87,107.5,0.0,0.0,null,0.831,0.779,0.953,null,0.0,0.0,0.0,null,0.374,0.627,0.0,0.0,null,null],[5005.5,null,null,2004.281,1169.105,null,131.998,106.359,0.0,0.0,null,0.855,0.794,0.903,null,0.0,0.0,0.0,null,0.362,-1.085,0.0,0.0,null,null],[5006.0,null,null,1992.792,1126.436,null,131.044,104.857,0.0,0.0,null,0.888,0.822,0.853,null,0.0,0.0,0.0,null,0.283,-6.689,0.0,0.0,null,null],[5006.5,null,null,1990.708,1082.863,null,130.286,102.966,0.0,0.0,null,0.923,0.863,0.855,null,0.0,0.0,0.0,null,0.108,-6.844,0.0,0.0,null,null],[5007.0,null,null,1987.911,1039.94,null,129.742,101.423,0.0,0.0,null,0.962,0.911,0.945,null,0.0,0.0,0.0,null,-0.153,-4.276,0.0,0.0,null,null],[5007.5,null,null,1977.352,996.583,null,129.169,100.629,0.0,0.0,null,1.003,0.958,1.13,null,0.0,0.0,0.0,null,-0.534,-4.473,0.0,0.0,null,null],[5008.0,null,null,1967.502,954.424,null,128.385,101.817,0.0,0.0,null,1.048,0.999,1.306,null,0.0,0.0,0.0,null,-0.923,-8.297,0.0,0.0,null,null],[5008.5,null,null,1965.754,916.394,null,127.496,105.356,0.0,0.0,null,1.091,1.035,1.384,null,0.0,0.0,0.0,null,-1.177,-11.704,0.0,0.0,null,null],[5009.0,null,null,1967.301,886.531,null,126.839,109.582,0.0,0.0,null,1.128,1.06,1.391,null,0.0,0.0,0.0,null,-1.344,-9.025,0.0,0.0,null,null],[5009.5,null,null,1968.551,869.293,null,126.621,112.335,0.0,0.0,null,1.15,1.072,1.393,null,0.0,0.0,0.0,null,-1.498,-2.183,0.0,0.0,null,null],[5010.0,null,null,1970.13,865.827,null,126.624,112.752,0.0,0.0,null,1.155,1.074,1.349,null,0.0,0.0,0.0,null,-1.601,-1.18,0.0,0.0,null,null],[5010.5,null,null,1971.206,871.906,null,126.428,111.769,0.0,0.0,null,1.147,1.067,1.237,null,0.0,0.0,0.0,null,-1.647,-5.868,0.0,0.0,null,null],[5011.0,null,null,1972.526,880.06,null,125.897,110.116,0.0,0.0,null,1.136,1.058,1.147,null,0.0,0.0,0.0,null,-1.672,-6.345,0.0,0.0,null,null],[5011.5,null,null,1979.039,885.444,null,125.308,108.57,0.0,0.0,null,1.129,1.052,1.115,null,0.0,0.0,0.0,null,-1.713,-4.042,0.0,0.0,null,null],[5012.0,null,null,1989.204,888.468,null,124.98,108.423,0.0,0.0,null,1.126,1.05,1.139,null,0.0,0.0,0.0,null,-1.761,-6.244,0.0,0.0,null,null],[5012.5,null,null,1992.37,890.247,null,124.925,109.701,0.0,0.0,null,1.123,1.051,1.159,null,0.0,0.0,0.0,null,-1.782,-6.925,0.0,0.0,null,null],[5013.0,null,null,1985.251,889.505,null,124.887,110.936,0.0,0.0,null,1.124,1.056,1.147,null,0.0,0.0,0.0,null,-1.783,1.534,0.0,0.0,null,null],[5013.5,null,null,1974.252,885.468,null,124.671,111.507,0.0,0.0,null,1.129,1.063,1.116,null,0.0,0.0,0.0,null,-1.786,12.091,0.0,0.0,null,null],[5014.0,null,null,1975.124,877.591,null,124.311,110.425,0.0,0.0,null,1.139,1.074,1.115,null,0.0,0.0,0.0,null,-1.799,15.649,0.0,0.0,null,null],[5014.5,null,null,1994.882,864.073,null,123.828,109.004,0.0,0.0,null,1.157,1.091,1.164,null,0.0,0.0,0.0,null,-1.838,11.664,0.0,0.0,null,null],[5015.0,null,null,2014.555,844.281,null,123.034,109.535,0.0,0.0,null,1.184,1.117,1.214,null,0.0,0.0,0.0,null,-1.895,2.897,0.0,0.0,null,null],[5015.5,null,null,2016.194,821.458,null,121.768,111.178,0.0,0.0,null,1.217,1.154,1.229,null,0.0,0.0,0.0,null,-1.936,-6.412,0.0,0.0,null,null],[5016.0,null,null,2005.286,799.083,null,120.193,110.664,0.0,0.0,null,1.251,1.205,1.225,null,0.0,0.0,0.0,null,-1.954,-13.008,0.0,0.0,null,null],[5016.5,null,null,1997.091,779.721,null,118.689,107.555,0.0,0.0,null,1.283,1.256,1.281,null,0.0,0.0,0.0,null,-1.977,-20.105,0.0,0.0,null,null],[5017.0,null,null,1997.913,765.641,null,117.512,102.74,0.0,0.0,null,1.306,1.29,1.472,null,0.0,0.0,0.0,null,-2.051,-28.61,0.0,0.0,null,null],[5017.5,null,null,1998.915,757.989,null,116.698,98.582,0.0,0.0,null,1.319,1.295,1.697,null,0.0,0.0,0.0,null,-2.194,-34.279,0.0,0.0,null,null],[5018.0,null,null,1990.01,755.727,null,116.26,97.015,0.0,0.0,null,1.323,1.278,1.779,null,0.0,0.0,0.0,null,-2.314,-35.505,0.0,0.0,null,null],[5018.5,null,null,1978.021,754.72,null,116.334,97.185,0.0,0.0,null,1.325,1.25,1.658,null,0.0,0.0,0.0,null,-2.288,-35.251,0.0,0.0,null,null],[5019.0,null,null,1979.194,752.921,null,116.998,97.644,0.0,0.0,null,1.328,1.223,1.506,null,0.0,0.0,0.0,null,-2.073,-31.58,0.0,0.0,null,null],[5019.5,null,null,1988.973,754.259,null,118.013,99.212,0.0,0.0,null,1.326,1.205,1.35,null,0.0,0.0,0.0,null,-1.723,-23.668,0.0,0.0,null,null],[5020.0,null,null,1991.701,765.023,null,118.967,102.458,0.0,0.0,null,1.307,1.198,1.18,null,0.0,0.0,0.0,null,-1.327,-17.643,0.0,0.0,null,null],[5020.5,null,null,1982.535,784.016,null,119.631,105.713,0.0,0.0,null,1.275,1.194,1.089,null,0.0,0.0,0.0,null,-0.967,-15.277,0.0,0.0,null,null],[5021.0,null,null,1975.091,803.425,null,120.04,106.792,0.0,0.0,null,1.245,1.178,1.136,null,0.0,0.0,0.0,null,-0.683,-12.292,0.0,0.0,null,null],[5021.5,null,null,1982.368,816.343,null,120.288,105.469,0.0,0.0,null,1.225,1.146,1.372,null,0.0,0.0,0.0,null,-0.471,-6.748,0.0,0.0,null,null],[5022.0,null,null,1997.32,821.652,null,120.406,103.781,0.0,0.0,null,1.217,1.112,1.572,null,0.0,0.0,0.0,null,-0.196,-2.922,0.0,0.0,null,null],[5022.5,null,null,2002.813,822.421,null,120.445,104.864,0.0,0.0,null,1.216,1.094,1.413,null,0.0,0.0,0.0,null,0.356,-5.374,0.0,0.0,null,null],[5023.0,null,null,1995.554,824.338,null,120.599,108.335,0.0,0.0,null,1.213,1.086,1.176,null,0.0,0.0,0.0,null,1.09,-11.16,0.0,0.0,null,null],[5023.5,null,null,1988.0,834.684,null,121.133,110.975,0.0,0.0,null,1.198,1.071,1.116,null,0.0,0.0,0.0,null,1.653,-11.664,0.0,0.0,null,null],[5024.0,null,null,1991.335,855.101,null,122.077,111.573,0.0,0.0,null,1.169,1.045,1.216,null,0.0,0.0,0.0,null,1.922,-7.096,0.0,0.0,null,null],[5024.5,null,null,2002.241,879.233,null,123.189,111.968,0.0,0.0,null,1.137,1.018,1.286,null,0.0,0.0,0.0,null,2.031,-5.104,0.0,0.0,null,null],[5025.0,null,null,2007.656,899.263,null,124.226,113.362,0.0,0.0,null,1.112,0.995,1.238,null,0.0,0.0,0.0,null,2.05,-10.674,0.0,0.0,null,null],[5025.5,null,null,2004.557,913.404,null,125.053,114.264,0.0,0.0,null,1.095,0.977,1.146,null,0.0,0.0,0.0,null,2.009,-20.345,0.0,0.0,null,null],[5026.0,null,null,2001.937,921.658,null,125.597,113.05,0.0,0.0,null,1.085,0.967,1.072,null,0.0,0.0,0.0,null,1.954,-24.737,0.0,0.0,null,null],[5026.5,null,null,2005.314,922.664,null,125.86,110.065,0.0,0.0,null,1.084,0.969,1.036,null,0.0,0.0,0.0,null,1.892,-19.362,0.0,0.0,null,null],[5027.0,null,null,2013.8,918.176,null,125.961,107.708,0.0,0.0,null,1.089,0.985,0.992,null,0.0,0.0,0.0,null,1.819,-10.291,0.0,0.0,null,null],[5027.5,null,null,2017.025,914.807,null,126.102,106.049,0.0,0.0,null,1.093,1.009,0.952,null,0.0,0.0,0.0,null,1.737,-6.779,0.0,0.0,null,null],[5028.0,null,null,2004.97,914.927,null,126.47,105.237,0.0,0.0,null,1.093,1.028,1.002,null,0.0,0.0,0.0,null,1.595,-9.139,0.0,0.0,null,null],[5028.5,null,null,1986.767,914.273,null,127.028,105.462,0.0,0.0,null,1.094,1.03,1.156,null,0.0,0.0,0.0,null,1.394,-9.17,0.0,0.0,null,null],[5029.0,null,null,1977.736,909.118,null,127.503,107.286,0.0,0.0,null,1.1,1.023,1.286,null,0.0,0.0,0.0,null,1.267,-4.656,0.0,0.0,null,null],[5029.5,null,null,1974.785,896.933,null,127.683,110.401,0.0,0.0,null,1.115,1.023,1.22,null,0.0,0.0,0.0,null,1.31,-2.089,0.0,0.0,null,null],[5030.0,null,null,1972.715,878.344,null,127.562,113.169,0.0,0.0,null,1.139,1.042,1.023,null,0.0,0.0,0.0,null,1.46,-4.499,0.0,0.0,null,null],[5030.5,null,null,1975.843,858.598,null,127.224,113.03,0.0,0.0,null,1.165,1.077,0.918,null,0.0,0.0,0.0,null,1.535,-6.703,0.0,0.0,null,null],[5031.0,null,null,1984.114,842.866,null,126.608,110.556,0.0,0.0,null,1.186,1.118,0.978,null,0.0,0.0,0.0,null,1.326,-7.834,0.0,0.0,null,null],[5031.5,null,null,1990.411,828.912,null,125.392,107.686,0.0,0.0,null,1.206,1.158,1.148,null,0.0,0.0,0.0,null,0.867,-10.212,0.0,0.0,null,null],[5032.0,null,null,1992.797,812.047,null,123.342,106.11,0.0,0.0,null,1.231,1.202,1.291,null,0.0,0.0,0.0,null,0.429,-11.211,0.0,0.0,null,null],[5032.5,null,null,1990.79,793.599,null,120.846,106.414,0.0,0.0,null,1.26,1.244,1.296,null,0.0,0.0,0.0,null,0.189,-8.312,0.0,0.0,null,null],[5033.0,null,null,1985.744,786.366,null,118.819,105.919,0.0,0.0,null,1.272,1.265,1.3,null,0.0,0.0,0.0,null,0.165,-5.866,0.0,0.0,null,null],[5033.5,null,null,1987.903,804.3,null,117.982,100.991,0.0,0.0,null,1.243,1.232,1.481,null,0.0,0.0,0.0,null,0.261,-4.162,0.0,0.0,null,null],[5034.0,null,null,1995.856,845.903,null,118.382,93.784,0.0,0.0,null,1.182,1.14,1.884,null,0.0,0.0,0.0,null,0.188,1.572,0.0,0.0,null,null],[5034.5,null,null,1996.269,897.715,null,119.55,90.419,0.0,0.0,null,1.114,1.027,1.991,null,0.0,0.0,0.0,null,-0.027,7.14,0.0,0.0,null,null],[5035.0,null,null,1992.187,951.29,null,121.117,93.636,0.0,0.0,null,1.051,0.933,1.428,null,0.0,0.0,0.0,null,0.124,4.31,0.0,0.0,null,null],[5035.5,null,null,1991.667,1007.403,null,123.141,101.002,0.0,0.0,null,0.993,0.871,0.82,null,0.0,0.0,0.0,null,0.791,-5.846,0.0,0.0,null,null],[5036.0,null,null,1990.156,1069.33,null,125.665,107.283,0.0,0.0,null,0.935,0.833,0.649,null,0.0,0.0,0.0,null,1.515,-14.362,0.0,0.0,null,null],[5036.5,null,null,1985.971,1134.184,null,128.225,110.541,0.0,0.0,null,0.882,0.809,0.71,null,0.0,0.0,0.0,null,1.935,-15.756,0.0,0.0,null,null],[5037.0,null,null,1983.8,1187.865,null,130.095,112.065,0.0,0.0,null,0.842,0.795,0.782,null,0.0,0.0,0.0,null,2.089,-14.347,0.0,0.0,null,null],[5037.5,null,null,1979.825,1216.237,null,130.934,113.114,0.0,0.0,null,0.822,0.792,0.8,null,0.0,0.0,0.0,null,2.107,-15.203,0.0,0.0,null,null],[5038.0,null,null,1970.321,1218.292,null,130.986,113.366,0.0,0.0,null,0.821,0.796,0.798,null,0.0,0.0,0.0,null,2.142,-17.112,0.0,0.0,null,null],[5038.5,null,null,1962.371,1201.183,null,130.683,113.059,0.0,0.0,null,0.833,0.804,0.855,null,0.0,0.0,0.0,null,2.244,-18.228,0.0,0.0,null,null],[5039.0,null,null,1965.474,1172.344,null,130.252,113.152,0.0,0.0,null,0.853,0.815,0.95,null,0.0,0.0,0.0,null,2.353,-17.209,0.0,0.0,null,null],[5039.5,null,null,1975.476,1141.766,null,129.733,114.531,0.0,0.0,null,0.876,0.832,0.978,null,0.0,0.0,0.0,null,2.396,-12.508,0.0,0.0,null,null],[5040.0,null,null,1983.614,1117.663,null,129.234,116.689,0.0,0.0,null,0.895,0.853,0.948,null,0.0,0.0,0.0,null,2.366,-8.861,0.0,0.0,null,null],[5040.5,null,null,1989.35,1103.044,null,128.974,117.396,0.0,0.0,null,0.907,0.868,0.908,null,0.0,0.0,0.0,null,2.279,-10.299,0.0,0.0,null,null],[5041.0,null,null,1992.468,1096.696,null,128.971,116.602,0.0,0.0,null,0.912,0.872,0.886,null,0.0,0.0,0.0,null,2.165,-13.247,0.0,0.0,null,null],[5041.5,null,null,1989.556,1095.372,null,129.001,116.51,0.0,0.0,null,0.913,0.868,0.898,null,0.0,0.0,0.0,null,2.048,-11.999,0.0,0.0,null,null],[5042.0,null,null,1984.129,1094.526,null,128.943,117.155,0.0,0.0,null,0.914,0.861,0.94,null,0.0,0.0,0.0,null,1.969,-4.825,0.0,0.0,null,null],[5042.5,null,null,1984.316,1092.848,null,128.911,117.094,0.0,0.0,null,0.915,0.858,0.947,null,0.0,0.0,0.0,null,1.964,3.178,0.0,0.0,null,null],[5043.0,null,null,1990.901,1090.228,null,128.98,116.756,0.0,0.0,null,0.917,0.86,0.891,null,0.0,0.0,0.0,null,2.02,4.837,0.0,0.0,null,null],[5043.5,null,null,1993.649,1086.388,null,129.012,116.637,0.0,0.0,null,0.92,0.869,0.869,null,0.0,0.0,0.0,null,2.007,1.381,0.0,0.0,null,null],[5044.0,null,null,1986.773,1083.109,null,128.84,115.355,0.0,0.0,null,0.923,0.877,0.933,null,0.0,0.0,0.0,null,1.836,-1.691,0.0,0.0,null,null],[5044.5,null,null,1977.546,1081.647,null,128.487,113.426,0.0,0.0,null,0.925,0.883,1.042,null,0.0,0.0,0.0,null,1.623,-6.111,0.0,0.0,null,null],[5045.0,null,null,1971.358,1080.702,null,128.226,112.883,0.0,0.0,null,0.925,0.883,1.106,null,0.0,0.0,0.0,null,1.545,-12.313,0.0,0.0,null,null],[5045.5,null,null,1966.51,1081.667,null,128.283,113.878,0.0,0.0,null,0.924,0.88,1.09,null,0.0,0.0,0.0,null,1.594,-12.436,0.0,0.0,null,null],[5046.0,null,null,1964.242,1088.025,null,128.558,114.629,0.0,0.0,null,0.919,0.869,1.058,null,0.0,0.0,0.0,null,1.637,-4.644,0.0,0.0,null,null],[5046.5,null,null,1969.777,1101.04,null,128.826,114.372,0.0,0.0,null,0.908,0.851,1.054,null,0.0,0.0,0.0,null,1.593,0.865,0.0,0.0,null,null],[5047.0,null,null,1980.533,1121.235,null,129.049,114.028,0.0,0.0,null,0.892,0.831,1.046,null,0.0,0.0,0.0,null,1.441,-2.054,0.0,0.0,null,null],[5047.5,null,null,1988.971,1146.622,null,129.392,114.473,0.0,0.0,null,0.872,0.812,1.003,null,0.0,0.0,0.0,null,1.257,-4.524,0.0,0.0,null,null],[5048.0,null,null,1990.178,1172.547,null,130.005,114.112,0.0,0.0,null,0.853,0.796,0.945,null,0.0,0.0,0.0,null,1.116,-1.118,0.0,0.0,null,null],[5048.5,null,null,1986.611,1197.217,null,130.828,111.34,0.0,0.0,null,0.835,0.783,0.905,null,0.0,0.0,0.0,null,1.026,-2.98,0.0,0.0,null,null],[5049.0,null,null,1983.161,1221.763,null,131.597,108.17,0.0,0.0,null,0.818,0.773,0.891,null,0.0,0.0,0.0,null,0.991,-16.62,0.0,0.0,null,null],[5049.5,null,null,1982.697,1241.786,null,132.122,106.378,0.0,0.0,null,0.805,0.765,0.908,null,0.0,0.0,0.0,null,0.986,-29.483,0.0,0.0,null,null],[5050.0,null,null,1984.658,1250.539,null,132.377,104.565,0.0,0.0,null,0.8,0.761,0.915,null,0.0,0.0,0.0,null,0.956,-29.194,0.0,0.0,null,null],[5050.5,null,null,1988.093,1243.969,null,132.328,102.8,0.0,0.0,null,0.804,0.765,0.891,null,0.0,0.0,0.0,null,0.87,-15.341,0.0,0.0,null,null],[5051.0,null,null,1992.531,1217.709,null,131.871,103.023,0.0,0.0,null,0.821,0.781,0.838,null,0.0,0.0,0.0,null,0.719,-1.716,0.0,0.0,null,null],[5051.5,null,null,1997.998,1170.523,null,130.953,104.341,0.0,0.0,null,0.854,0.815,0.791,null,0.0,0.0,0.0,null,0.498,-3.527,0.0,0.0,null,null],[5052.0,null,null,2004.631,1108.703,null,129.67,104.082,0.0,0.0,null,0.902,0.87,0.796,null,0.0,0.0,0.0,null,0.23,-13.772,0.0,0.0,null,null],[5052.5,null,null,2010.4,1041.816,null,128.209,103.042,0.0,0.0,null,0.96,0.941,0.869,null,0.0,0.0,0.0,null,-0.061,-17.023,0.0,0.0,null,null],[5053.0,null,null,2013.809,975.982,null,126.704,102.945,0.0,0.0,null,1.025,1.02,1.095,null,0.0,0.0,0.0,null,-0.403,-14.035,0.0,0.0,null,null],[5053.5,null,null,2017.041,912.157,null,125.14,102.945,0.0,0.0,null,1.096,1.095,1.434,null,0.0,0.0,0.0,null,-0.811,-14.759,0.0,0.0,null,null],[5054.0,null,null,2016.38,850.5,null,123.513,102.193,0.0,0.0,null,1.176,1.163,1.657,null,0.0,0.0,0.0,null,-1.214,-17.949,0.0,0.0,null,null],[5054.5,null,null,2008.407,795.513,null,121.971,101.33,0.0,0.0,null,1.257,1.229,1.668,null,0.0,0.0,0.0,null,-1.502,-16.414,0.0,0.0,null,null],[5055.0,null,null,2000.072,755.587,null,120.783,101.092,0.0,0.0,null,1.323,1.282,1.596,null,0.0,0.0,0.0,null,-1.683,-9.678,0.0,0.0,null,null],[5055.5,null,null,1999.704,735.421,null,120.195,101.4,0.0,0.0,null,1.36,1.303,1.616,null,0.0,0.0,0.0,null,-1.821,-8.421,0.0,0.0,null,null],[5056.0,null,null,2003.899,738.951,null,120.302,102.796,0.0,0.0,null,1.353,1.285,1.663,null,0.0,0.0,0.0,null,-1.855,-17.109,0.0,0.0,null,null],[5056.5,null,null,2004.401,767.906,null,121.004,105.252,0.0,0.0,null,1.302,1.234,1.614,null,0.0,0.0,0.0,null,-1.699,-27.107,0.0,0.0,null,null],[5057.0,null,null,2000.424,816.698,null,122.134,107.109,0.0,0.0,null,1.224,1.154,1.487,null,0.0,0.0,0.0,null,-1.414,-30.631,0.0,0.0,null,null],[5057.5,null,null,1996.921,876.193,null,123.622,107.095,0.0,0.0,null,1.141,1.059,1.433,null,0.0,0.0,0.0,null,-1.134,-28.454,0.0,0.0,null,null],[5058.0,null,null,1994.445,938.422,null,125.423,106.16,0.0,0.0,null,1.066,0.97,1.366,null,0.0,0.0,0.0,null,-0.799,-25.138,0.0,0.0,null,null],[5058.5,null,null,1992.906,997.0,null,127.42,105.442,0.0,0.0,null,1.003,0.901,1.141,null,0.0,0.0,0.0,null,-0.31,-21.721,0.0,0.0,null,null],[5059.0,null,null,1995.142,1052.08,null,129.39,104.515,0.0,0.0,null,0.95,0.853,0.902,null,0.0,0.0,0.0,null,0.209,-15.378,0.0,0.0,null,null],[5059.5,null,null,1994.721,1111.111,null,131.046,103.31,0.0,0.0,null,0.9,0.819,0.802,null,0.0,0.0,0.0,null,0.586,-9.166,0.0,0.0,null,null],[5060.0,null,null,1985.685,1170.472,null,132.153,102.291,0.0,0.0,null,0.854,0.794,0.83,null,0.0,0.0,0.0,null,0.833,-8.629,0.0,0.0,null,null],[5060.5,null,null,1978.301,1211.713,null,132.633,102.181,0.0,0.0,null,0.825,0.781,0.845,null,0.0,0.0,0.0,null,1.043,-11.178,0.0,0.0,null,null],[5061.0,null,null,1980.294,1220.36,null,132.54,103.206,0.0,0.0,null,0.819,0.78,0.836,null,0.0,0.0,0.0,null,1.194,-12.214,0.0,0.0,null,null],[5061.5,null,null,1986.844,1198.894,null,131.909,104.576,0.0,0.0,null,0.834,0.793,0.846,null,0.0,0.0,0.0,null,1.23,-13.156,0.0,0.0,null,null],[5062.0,null,null,1995.406,1156.803,null,130.748,104.732,0.0,0.0,null,0.864,0.82,0.868,null,0.0,0.0,0.0,null,1.188,-13.312,0.0,0.0,null,null],[5062.5,null,null,2000.346,1104.505,null,129.158,103.806,0.0,0.0,null,0.905,0.864,0.884,null,0.0,0.0,0.0,null,1.101,-13.63,0.0,0.0,null,null],[5063.0,null,null,1997.012,1060.293,null,127.466,103.541,0.0,0.0,null,0.943,0.916,0.919,null,0.0,0.0,0.0,null,0.955,-16.901,0.0,0.0,null,null],[5063.5,null,null,1992.625,1042.412,null,126.156,103.824,0.0,0.0,null,0.959,0.952,1.054,null,0.0,0.0,0.0,null,0.744,-19.05,0.0,0.0,null,null],[5064.0,null,null,1993.226,1049.082,null,125.545,104.775,0.0,0.0,null,0.953,0.947,1.364,null,0.0,0.0,0.0,null,0.477,-16.677,0.0,0.0,null,null],[5064.5,null,null,1992.124,1064.368,null,125.566,107.403,0.0,0.0,null,0.94,0.906,1.735,null,0.0,0.0,0.0,null,0.255,-16.57,0.0,0.0,null,null],[5065.0,null,null,1985.821,1080.822,null,125.943,110.208,0.0,0.0,null,0.925,0.857,1.661,null,0.0,0.0,0.0,null,0.36,-18.715,0.0,0.0,null,null],[5065.5,null,null,1982.581,1103.157,null,126.532,110.914,0.0,0.0,null,0.906,0.82,1.156,null,0.0,0.0,0.0,null,0.858,-16.431,0.0,0.0,null,null],[5066.0,null,null,1982.82,1136.581,null,127.379,111.148,0.0,0.0,null,0.88,0.793,0.801,null,0.0,0.0,0.0,null,1.418,-10.034,0.0,0.0,null,null],[5066.5,null,null,1980.672,1183.237,null,128.532,111.725,0.0,0.0,null,0.845,0.771,0.725,null,0.0,0.0,0.0,null,1.76,-3.744,0.0,0.0,null,null],[5067.0,null,null,1975.632,1233.972,null,129.863,111.836,0.0,0.0,null,0.81,0.755,0.767,null,0.0,0.0,0.0,null,1.96,1.058,0.0,0.0,null,null],[5067.5,null,null,1971.479,1269.859,null,131.09,112.095,0.0,0.0,null,0.787,0.749,0.793,null,0.0,0.0,0.0,null,2.169,4.853,0.0,0.0,null,null],[5068.0,null,null,1967.005,1281.173,null,132.033,111.753,0.0,0.0,null,0.781,0.754,0.804,null,0.0,0.0,0.0,null,2.327,4.141,0.0,0.0,null,null],[5068.5,null,null,1963.068,1271.865,null,132.741,109.816,0.0,0.0,null,0.786,0.767,0.813,null,0.0,0.0,0.0,null,2.308,-2.291,0.0,0.0,null,null],[5069.0,null,null,1962.254,1247.677,null,133.32,107.795,0.0,0.0,null,0.801,0.781,0.87,null,0.0,0.0,0.0,null,2.159,-6.203,0.0,0.0,null,null],[5069.5,null,null,1965.175,1216.253,null,133.741,106.963,0.0,0.0,null,0.822,0.794,1.006,null,0.0,0.0,0.0,null,2.019,-0.719,0.0,0.0,null,null],[5070.0,null,null,1969.433,1189.408,null,133.889,107.313,0.0,0.0,null,0.841,0.804,1.084,null,0.0,0.0,0.0,null,1.88,7.664,0.0,0.0,null,null],[5070.5,null,null,1970.046,1170.224,null,133.72,109.108,0.0,0.0,null,0.855,0.813,1.055,null,0.0,0.0,0.0,null,1.69,10.786,0.0,0.0,null,null],[5071.0,null,null,1966.223,1154.66,null,133.307,111.996,0.0,0.0,null,0.866,0.82,0.966,null,0.0,0.0,0.0,null,1.516,8.034,0.0,0.0,null,null],[5071.5,null,null,1965.933,1141.894,null,132.72,113.927,0.0,0.0,null,0.876,0.826,0.915,null,0.0,0.0,0.0,null,1.37,3.858,0.0,0.0,null,null],[5072.0,null,null,1971.574,1132.299,null,131.933,113.332,0.0,0.0,null,0.883,0.832,0.909,null,0.0,0.0,0.0,null,1.184,1.326,0.0,0.0,null,null],[5072.5,null,null,1976.503,1121.258,null,130.935,111.565,0.0,0.0,null,0.892,0.841,0.912,null,0.0,0.0,0.0,null,0.969,-4.07,0.0,0.0,null,null],[5073.0,null,null,1976.131,1104.195,null,129.838,110.62,0.0,0.0,null,0.906,0.856,0.921,null,0.0,0.0,0.0,null,0.778,-13.264,0.0,0.0,null,null],[5073.5,null,null,1972.952,1080.47,null,128.775,110.24,0.0,0.0,null,0.926,0.876,0.921,null,0.0,0.0,0.0,null,0.574,-15.143,0.0,0.0,null,null],[5074.0,null,null,1972.022,1050.233,null,127.761,109.256,0.0,0.0,null,0.952,0.904,0.937,null,0.0,0.0,0.0,null,0.364,-10.331,0.0,0.0,null,null],[5074.5,null,null,1974.636,1014.521,null,126.746,107.598,0.0,0.0,null,0.986,0.942,0.968,null,0.0,0.0,0.0,null,0.196,-11.792,0.0,0.0,null,null],[5075.0,null,null,1977.321,976.458,null,125.78,106.022,0.0,0.0,null,1.024,0.986,1.019,null,0.0,0.0,0.0,null,0.027,-19.31,0.0,0.0,null,null],[5075.5,null,null,1977.41,939.601,null,125.03,105.401,0.0,0.0,null,1.064,1.032,1.099,null,0.0,0.0,0.0,null,-0.213,-24.421,0.0,0.0,null,null],[5076.0,null,null,1975.51,906.349,null,124.574,106.057,0.0,0.0,null,1.103,1.073,1.191,null,0.0,0.0,0.0,null,-0.5,-24.351,0.0,0.0,null,null],[5076.5,null,null,1973.22,876.557,null,124.249,106.1,0.0,0.0,null,1.141,1.105,1.304,null,0.0,0.0,0.0,null,-0.79,-18.587,0.0,0.0,null,null],[5077.0,null,null,1971.741,850.062,null,123.807,104.882,0.0,0.0,null,1.176,1.127,1.405,null,0.0,0.0,0.0,null,-1.042,-15.841,0.0,0.0,null,null],[5077.5,null,null,1972.978,828.08,null,123.307,103.757,0.0,0.0,null,1.208,1.146,1.419,null,0.0,0.0,0.0,null,-1.239,-22.466,0.0,0.0,null,null],[5078.0,null,null,1978.513,811.818,null,123.132,104.184,0.0,0.0,null,1.232,1.164,1.333,null,0.0,0.0,0.0,null,-1.355,-27.482,0.0,0.0,null,null],[5078.5,null,null,1985.512,802.142,null,123.495,106.354,0.0,0.0,null,1.247,1.179,1.254,null,0.0,0.0,0.0,null,-1.404,-24.279,0.0,0.0,null,null],[5079.0,null,null,1988.751,798.949,null,124.082,108.906,0.0,0.0,null,1.252,1.183,1.29,null,0.0,0.0,0.0,null,-1.42,-21.416,0.0,0.0,null,null],[5079.5,null,null,1985.428,800.125,null,124.289,109.293,0.0,0.0,null,1.25,1.179,1.359,null,0.0,0.0,0.0,null,-1.434,-23.62,0.0,0.0,null,null],[5080.0,null,null,1978.954,802.331,null,123.844,107.118,0.0,0.0,null,1.246,1.176,1.366,null,0.0,0.0,0.0,null,-1.454,-26.998,0.0,0.0,null,null],[5080.5,null,null,1979.018,803.732,null,123.094,105.553,0.0,0.0,null,1.244,1.176,1.308,null,0.0,0.0,0.0,null,-1.469,-28.99,0.0,0.0,null,null],[5081.0,null,null,1986.581,804.056,null,122.608,106.707,0.0,0.0,null,1.244,1.176,1.247,null,0.0,0.0,0.0,null,-1.358,-29.436,0.0,0.0,null,null],[5081.5,null,null,1989.818,804.633,null,122.533,109.318,0.0,0.0,null,1.243,1.171,1.237,null,0.0,0.0,0.0,null,-1.046,-26.222,0.0,0.0,null,null],[5082.0,null,null,1984.478,806.619,null,122.547,111.177,0.0,0.0,null,1.24,1.165,1.267,null,0.0,0.0,0.0,null,-0.658,-21.666,0.0,0.0,null,null],[5082.5,null,null,1982.403,809.332,null,122.415,111.436,0.0,0.0,null,1.236,1.16,1.288,null,0.0,0.0,0.0,null,-0.411,-21.841,0.0,0.0,null,null],[5083.0,null,null,1983.254,811.101,null,122.298,110.344,0.0,0.0,null,1.233,1.158,1.266,null,0.0,0.0,0.0,null,-0.443,-23.511,0.0,0.0,null,null],[5083.5,null,null,1984.034,811.624,null,122.565,109.15,0.0,0.0,null,1.232,1.157,1.244,null,0.0,0.0,0.0,null,-0.706,-22.677,0.0,0.0,null,null],[5084.0,null,null,1988.135,812.059,null,123.422,109.114,0.0,0.0,null,1.231,1.154,1.26,null,0.0,0.0,0.0,null,-0.992,-24.643,0.0,0.0,null,null],[5084.5,null,null,1992.107,812.87,null,124.71,110.186,0.0,0.0,null,1.23,1.15,1.274,null,0.0,0.0,0.0,null,-1.125,-26.32,0.0,0.0,null,null],[5085.0,null,null,1991.089,813.521,null,126.049,110.904,0.0,0.0,null,1.229,1.145,1.258,null,0.0,0.0,0.0,null,-1.135,-23.945,0.0,0.0,null,null],[5085.5,null,null,1987.978,813.521,null,127.107,111.123,0.0,0.0,null,1.229,1.142,1.244,null,0.0,0.0,0.0,null,-1.124,-20.802,0.0,0.0,null,null],[5086.0,null,null,1981.712,812.572,null,127.681,112.117,0.0,0.0,null,1.231,1.141,1.253,null,0.0,0.0,0.0,null,-1.155,-15.781,0.0,0.0,null,null],[5086.5,null,null,1971.878,811.332,null,127.697,113.613,0.0,0.0,null,1.233,1.144,1.269,null,0.0,0.0,0.0,null,-1.215,-11.92,0.0,0.0,null,null],[5087.0,null,null,1967.364,811.816,null,127.312,113.815,0.0,0.0,null,1.232,1.15,1.273,null,0.0,0.0,0.0,null,-1.24,-16.328,0.0,0.0,null,null],[5087.5,null,null,1968.279,814.876,null,126.829,112.712,0.0,0.0,null,1.227,1.155,1.276,null,0.0,0.0,0.0,null,-1.168,-21.29,0.0,0.0,null,null],[5088.0,null,null,1968.588,816.955,null,126.401,111.638,0.0,0.0,null,1.224,1.155,1.27,null,0.0,0.0,0.0,null,-0.976,-16.486,0.0,0.0,null,null],[5088.5,null,null,1970.391,815.859,null,125.939,111.404,0.0,0.0,null,1.226,1.15,1.276,null,0.0,0.0,0.0,null,-0.693,-5.831,0.0,0.0,null,null],[5089.0,null,null,1975.536,812.989,null,125.32,110.956,0.0,0.0,null,1.23,1.147,1.255,null,0.0,0.0,0.0,null,-0.389,0.891,0.0,0.0,null,null],[5089.5,null,null,1980.181,809.294,null,124.606,109.273,0.0,0.0,null,1.236,1.15,1.221,null,0.0,0.0,0.0,null,-0.188,0.043,0.0,0.0,null,null],[5090.0,null,null,1984.731,804.576,null,124.03,107.69,0.0,0.0,null,1.243,1.159,1.221,null,0.0,0.0,0.0,null,-0.155,-1.057,0.0,0.0,null,null],[5090.5,null,null,1991.013,801.44,null,123.746,107.749,0.0,0.0,null,1.248,1.17,1.247,null,0.0,0.0,0.0,null,-0.233,7.419,0.0,0.0,null,null],[5091.0,null,null,1993.106,802.393,null,123.598,107.737,0.0,0.0,null,1.246,1.178,1.294,null,0.0,0.0,0.0,null,-0.352,14.093,0.0,0.0,null,null],[5091.5,null,null,1987.927,809.816,null,123.297,106.967,0.0,0.0,null,1.235,1.176,1.351,null,0.0,0.0,0.0,null,-0.491,5.8,0.0,0.0,null,null],[5092.0,null,null,1981.583,829.383,null,122.866,106.163,0.0,0.0,null,1.206,1.157,1.38,null,0.0,0.0,0.0,null,-0.534,-8.384,0.0,0.0,null,null],[5092.5,null,null,1979.701,862.449,null,122.712,105.825,0.0,0.0,null,1.159,1.109,1.384,null,0.0,0.0,0.0,null,-0.429,-14.063,0.0,0.0,null,null],[5093.0,null,null,1984.74,903.977,null,123.239,106.467,0.0,0.0,null,1.106,1.034,1.453,null,0.0,0.0,0.0,null,-0.28,-16.908,0.0,0.0,null,null],[5093.5,null,null,1996.385,950.254,null,124.485,108.484,0.0,0.0,null,1.052,0.959,1.416,null,0.0,0.0,0.0,null,-0.019,-21.121,0.0,0.0,null,null],[5094.0,null,null,2006.092,1000.398,null,126.152,111.487,0.0,0.0,null,1.0,0.902,1.122,null,0.0,0.0,0.0,null,0.516,-23.413,0.0,0.0,null,null],[5094.5,null,null,2006.842,1053.739,null,127.918,114.208,0.0,0.0,null,0.949,0.859,0.85,null,0.0,0.0,0.0,null,1.147,-24.549,0.0,0.0,null,null],[5095.0,null,null,2002.167,1116.585,null,129.693,114.985,0.0,0.0,null,0.896,0.818,0.834,null,0.0,0.0,0.0,null,1.629,-27.933,0.0,0.0,null,null],[5095.5,null,null,1998.649,1189.537,null,131.491,114.484,0.0,0.0,null,0.841,0.778,0.932,null,0.0,0.0,0.0,null,1.982,-33.196,0.0,0.0,null,null],[5096.0,null,null,1994.54,1255.366,null,133.108,115.091,0.0,0.0,null,0.797,0.746,0.92,null,0.0,0.0,0.0,null,2.276,-36.81,0.0,0.0,null,null],[5096.5,null,null,1986.559,1301.059,null,134.209,116.603,0.0,0.0,null,0.769,0.724,0.835,null,0.0,0.0,0.0,null,2.511,-32.821,0.0,0.0,null,null],[5097.0,null,null,1979.034,1333.815,null,134.665,116.933,0.0,0.0,null,0.75,0.71,0.766,null,0.0,0.0,0.0,null,2.676,-24.226,0.0,0.0,null,null],[5097.5,null,null,1975.009,1361.635,null,134.652,115.396,0.0,0.0,null,0.734,0.699,0.785,null,0.0,0.0,0.0,null,2.712,-20.281,0.0,0.0,null,null],[5098.0,null,null,1972.767,1379.848,null,134.454,112.685,0.0,0.0,null,0.725,0.692,0.809,null,0.0,0.0,0.0,null,2.629,-18.954,0.0,0.0,null,null],[5098.5,null,null,1970.335,1381.0,null,134.205,110.609,0.0,0.0,null,0.724,0.691,0.804,null,0.0,0.0,0.0,null,2.496,-8.496,0.0,0.0,null,null],[5099.0,null,null,1970.61,1359.964,null,133.767,110.657,0.0,0.0,null,0.735,0.7,0.781,null,0.0,0.0,0.0,null,2.338,7.631,0.0,0.0,null,null],[5099.5,null,null,1972.802,1315.642,null,132.929,112.179,0.0,0.0,null,0.76,0.723,0.738,null,0.0,0.0,0.0,null,2.18,14.507,0.0,0.0,null,null],[5100.0,null,null,1972.663,1255.868,null,131.721,111.26,0.0,0.0,null,0.796,0.765,0.703,null,0.0,0.0,0.0,null,2.035,7.485,0.0,0.0,null,null],[5100.5,null,null,1969.814,1193.654,null,130.325,107.305,0.0,0.0,null,0.838,0.818,0.735,null,0.0,0.0,0.0,null,1.784,-4.79,0.0,0.0,null,null],[5101.0,null,null,1970.09,1136.192,null,128.855,104.123,0.0,0.0,null,0.88,0.875,0.904,null,0.0,0.0,0.0,null,1.305,-8.059,0.0,0.0,null,null],[5101.5,null,null,1977.406,1080.377,null,127.355,103.758,0.0,0.0,null,0.926,0.927,1.144,null,0.0,0.0,0.0,null,0.751,0.619,0.0,0.0,null,null],[5102.0,null,null,1985.339,1020.27,null,125.873,104.045,0.0,0.0,null,0.98,0.975,1.27,null,0.0,0.0,0.0,null,0.371,3.74,0.0,0.0,null,null],[5102.5,null,null,1987.825,957.35,null,124.529,104.151,0.0,0.0,null,1.045,1.017,1.262,null,0.0,0.0,0.0,null,0.219,-10.468,0.0,0.0,null,null],[5103.0,null,null,1988.436,901.973,null,123.516,105.02,0.0,0.0,null,1.109,1.056,1.244,null,0.0,0.0,0.0,null,0.147,-25.656,0.0,0.0,null,null],[5103.5,null,null,1989.625,862.71,null,122.965,107.047,0.0,0.0,null,1.159,1.09,1.211,null,0.0,0.0,0.0,null,0.035,-26.232,0.0,0.0,null,null],[5104.0,null,null,1987.66,839.534,null,122.825,108.241,0.0,0.0,null,1.191,1.12,1.182,null,0.0,0.0,0.0,null,-0.095,-20.659,0.0,0.0,null,null],[5104.5,null,null,1980.845,827.096,null,123.003,106.808,0.0,0.0,null,1.209,1.143,1.214,null,0.0,0.0,0.0,null,-0.151,-18.948,0.0,0.0,null,null],[5105.0,null,null,1975.267,818.131,null,123.464,103.597,0.0,0.0,null,1.222,1.158,1.274,null,0.0,0.0,0.0,null,-0.118,-21.928,0.0,0.0,null,null],[5105.5,null,null,1977.411,809.659,null,124.226,101.092,0.0,0.0,null,1.235,1.165,1.309,null,0.0,0.0,0.0,null,-0.128,-26.094,0.0,0.0,null,null],[5106.0,null,null,1984.893,802.823,null,125.325,100.4,0.0,0.0,null,1.246,1.167,1.304,null,0.0,0.0,0.0,null,-0.242,-25.634,0.0,0.0,null,null],[5106.5,null,null,1991.128,798.818,null,126.697,101.886,0.0,0.0,null,1.252,1.164,1.295,null,0.0,0.0,0.0,null,-0.339,-16.661,0.0,0.0,null,null],[5107.0,null,null,1999.257,798.454,null,128.168,105.006,0.0,0.0,null,1.252,1.155,1.27,null,0.0,0.0,0.0,null,-0.326,0.379,0.0,0.0,null,null],[5107.5,null,null,2009.541,802.093,null,129.569,108.555,0.0,0.0,null,1.247,1.143,1.222,null,0.0,0.0,0.0,null,-0.324,15.063,0.0,0.0,null,null],[5108.0,null,null,2010.289,807.39,null,130.774,110.607,0.0,0.0,null,1.239,1.13,1.204,null,0.0,0.0,0.0,null,-0.403,10.864,0.0,0.0,null,null],[5108.5,null,null,2002.25,811.57,null,131.614,110.457,0.0,0.0,null,1.232,1.122,1.198,null,0.0,0.0,0.0,null,-0.417,-9.818,0.0,0.0,null,null],[5109.0,null,null,1998.736,815.091,null,131.84,109.16,0.0,0.0,null,1.227,1.119,1.189,null,0.0,0.0,0.0,null,-0.258,-17.153,0.0,0.0,null,null],[5109.5,null,null,1999.028,818.95,null,131.267,108.063,0.0,0.0,null,1.221,1.123,1.163,null,0.0,0.0,0.0,null,-0.019,-2.453,0.0,0.0,null,null],[5110.0,null,null,1995.978,823.455,null,129.942,108.044,0.0,0.0,null,1.214,1.127,1.164,null,0.0,0.0,0.0,null,0.165,10.05,0.0,0.0,null,null],[5110.5,null,null,1993.338,827.821,null,128.188,108.902,0.0,0.0,null,1.208,1.13,1.204,null,0.0,0.0,0.0,null,0.275,7.798,0.0,0.0,null,null],[5111.0,null,null,1994.021,831.263,null,126.368,109.542,0.0,0.0,null,1.203,1.129,1.261,null,0.0,0.0,0.0,null,0.379,-0.548,0.0,0.0,null,null],[5111.5,null,null,1995.606,834.277,null,124.712,109.048,0.0,0.0,null,1.199,1.124,1.302,null,0.0,0.0,0.0,null,0.549,-3.99,0.0,0.0,null,null],[5112.0,null,null,2000.531,838.167,null,123.343,107.384,0.0,0.0,null,1.193,1.114,1.306,null,0.0,0.0,0.0,null,0.768,-2.885,0.0,0.0,null,null],[5112.5,null,null,2007.371,843.927,null,122.376,105.208,0.0,0.0,null,1.185,1.1,1.295,null,0.0,0.0,0.0,null,0.918,-6.674,0.0,0.0,null,null],[5113.0,null,null,2009.821,852.652,null,121.928,104.317,0.0,0.0,null,1.173,1.086,1.272,null,0.0,0.0,0.0,null,0.96,-17.69,0.0,0.0,null,null],[5113.5,null,null,2005.26,865.577,null,122.017,105.104,0.0,0.0,null,1.155,1.072,1.246,null,0.0,0.0,0.0,null,0.967,-23.048,0.0,0.0,null,null],[5114.0,null,null,2003.151,881.958,null,122.538,105.316,0.0,0.0,null,1.134,1.06,1.217,null,0.0,0.0,0.0,null,0.958,-16.265,0.0,0.0,null,null],[5114.5,null,null,2007.924,898.661,null,123.36,103.502,0.0,0.0,null,1.113,1.048,1.188,null,0.0,0.0,0.0,null,0.873,-7.982,0.0,0.0,null,null],[5115.0,null,null,2010.348,913.714,null,124.382,102.17,0.0,0.0,null,1.094,1.034,1.169,null,0.0,0.0,0.0,null,0.744,-9.899,0.0,0.0,null,null],[5115.5,null,null,2006.767,928.448,null,125.481,104.033,0.0,0.0,null,1.077,1.017,1.167,null,0.0,0.0,0.0,null,0.702,-20.351,0.0,0.0,null,null],[5116.0,null,null,2005.322,946.009,null,126.425,107.206,0.0,0.0,null,1.057,0.998,1.165,null,0.0,0.0,0.0,null,0.808,-23.506,0.0,0.0,null,null],[5116.5,null,null,2005.809,964.935,null,127.02,109.194,0.0,0.0,null,1.036,0.979,1.158,null,0.0,0.0,0.0,null,0.948,-12.478,0.0,0.0,null,null],[5117.0,null,null,2000.65,980.616,null,127.261,109.702,0.0,0.0,null,1.02,0.962,1.141,null,0.0,0.0,0.0,null,1.004,-0.51,0.0,0.0,null,null],[5117.5,null,null,1989.555,992.3,null,127.36,109.347,0.0,0.0,null,1.008,0.946,1.107,null,0.0,0.0,0.0,null,1.01,-1.491,0.0,0.0,null,null],[5118.0,null,null,1981.859,1002.17,null,127.492,108.163,0.0,0.0,null,0.998,0.933,1.064,null,0.0,0.0,0.0,null,1.065,-11.121,0.0,0.0,null,null],[5118.5,null,null,1984.688,1010.294,null,127.574,106.29,0.0,0.0,null,0.99,0.927,1.041,null,0.0,0.0,0.0,null,1.233,-16.697,0.0,0.0,null,null],[5119.0,null,null,1988.312,1018.378,null,127.395,104.025,0.0,0.0,null,0.982,0.926,1.022,null,0.0,0.0,0.0,null,1.612,-11.603,0.0,0.0,null,null],[5119.5,null,null,1985.849,1031.565,null,126.864,101.995,0.0,0.0,null,0.969,0.928,0.962,null,0.0,0.0,0.0,null,2.118,-1.581,0.0,0.0,null,null],[5120.0,null,null,1984.489,1050.029,null,126.129,100.139,0.0,0.0,null,0.952,0.923,0.977,null,0.0,0.0,0.0,null,2.366,2.464,0.0,0.0,null,null],[5120.5,null,null,1987.795,1069.296,null,125.451,97.844,0.0,0.0,null,0.935,0.907,1.181,null,0.0,0.0,0.0,null,2.201,0.826,0.0,0.0,null,null],[5121.0,null,null,1990.554,1088.6,null,125.011,96.522,0.0,0.0,null,0.919,0.885,1.329,null,0.0,0.0,0.0,null,2.096,-0.164,0.0,0.0,null,null],[5121.5,null,null,1991.802,1106.797,null,124.735,97.535,0.0,0.0,null,0.904,0.868,1.141,null,0.0,0.0,0.0,null,2.436,3.173,0.0,0.0,null,null],[5122.0,null,null,1990.26,1122.304,null,124.403,100.091,0.0,0.0,null,0.891,0.855,0.907,null,0.0,0.0,0.0,null,2.911,8.594,0.0,0.0,null,null],[5122.5,null,null,1988.726,1137.895,null,123.995,101.736,0.0,0.0,null,0.879,0.843,0.902,null,0.0,0.0,0.0,null,3.126,6.24,0.0,0.0,null,null],[5123.0,null,null,1994.544,1157.262,null,123.734,102.846,0.0,0.0,null,0.864,0.83,0.953,null,0.0,0.0,0.0,null,3.274,-7.166,0.0,0.0,null,null],[5123.5,null,null,2002.255,1173.244,null,123.858,104.5,0.0,0.0,null,0.852,0.823,0.915,null,0.0,0.0,0.0,null,3.511,-18.983,0.0,0.0,null,null],[5124.0,null,null,2002.659,1178.26,null,124.263,106.112,0.0,0.0,null,0.849,0.821,0.911,null,0.0,0.0,0.0,null,3.586,-22.328,0.0,0.0,null,null],[5124.5,null,null,2000.845,1172.973,null,124.5,106.833,0.0,0.0,null,0.853,0.825,0.936,null,0.0,0.0,0.0,null,3.488,-23.783,0.0,0.0,null,null],[5125.0,null,null,2001.451,1162.525,null,124.313,107.519,0.0,0.0,null,0.86,0.836,0.944,null,0.0,0.0,0.0,null,3.523,-23.749,0.0,0.0,null,null],[5125.5,null,null,2002.356,1153.234,null,124.046,107.609,0.0,0.0,null,0.867,0.848,0.94,null,0.0,0.0,0.0,null,3.751,-19.254,0.0,0.0,null,null],[5126.0,null,null,2004.301,1151.913,null,124.268,105.736,0.0,0.0,null,0.868,0.855,0.968,null,0.0,0.0,0.0,null,3.958,-13.613,0.0,0.0,null,null],[5126.5,null,null,2004.405,1157.592,null,125.021,103.37,0.0,0.0,null,0.864,0.849,1.04,null,0.0,0.0,0.0,null,4.044,-9.588,0.0,0.0,null,null],[5127.0,null,null,1996.301,1161.394,null,125.732,102.635,0.0,0.0,null,0.861,0.835,1.099,null,0.0,0.0,0.0,null,4.062,-8.15,0.0,0.0,null,null],[5127.5,null,null,1988.407,1159.558,null,125.887,103.143,0.0,0.0,null,0.862,0.824,1.082,null,0.0,0.0,0.0,null,4.087,-11.893,0.0,0.0,null,null],[5128.0,null,null,1987.337,1156.19,null,125.669,103.812,0.0,0.0,null,0.865,0.824,0.971,null,0.0,0.0,0.0,null,4.147,-15.62,0.0,0.0,null,null],[5128.5,null,null,1985.783,1156.25,null,125.682,104.371,0.0,0.0,null,0.865,0.829,0.879,null,0.0,0.0,0.0,null,4.22,-11.805,0.0,0.0,null,null],[5129.0,null,null,1979.731,1160.604,null,126.112,104.141,0.0,0.0,null,0.862,0.831,0.882,null,0.0,0.0,0.0,null,4.32,-8.902,0.0,0.0,null,null],[5129.5,null,null,1976.617,1167.735,null,126.51,103.054,0.0,0.0,null,0.856,0.83,0.941,null,0.0,0.0,0.0,null,4.443,-12.596,0.0,0.0,null,null],[5130.0,null,null,1980.609,1172.856,null,126.439,101.829,0.0,0.0,null,0.853,0.83,0.978,null,0.0,0.0,0.0,null,4.556,-13.921,0.0,0.0,null,null],[5130.5,null,null,1985.407,1172.866,null,126.012,101.255,0.0,0.0,null,0.853,0.83,0.984,null,0.0,0.0,0.0,null,4.579,-12.486,0.0,0.0,null,null],[5131.0,null,null,1987.154,1168.096,null,125.68,101.861,0.0,0.0,null,0.856,0.83,0.992,null,0.0,0.0,0.0,null,4.518,-14.521,0.0,0.0,null,null],[5131.5,null,null,1987.897,1164.7,null,125.653,103.721,0.0,0.0,null,0.859,0.828,0.986,null,0.0,0.0,0.0,null,4.438,-17.138,0.0,0.0,null,null],[5132.0,null,null,1986.795,1168.645,null,125.764,105.166,0.0,0.0,null,0.856,0.822,0.981,null,0.0,0.0,0.0,null,4.386,-18.517,0.0,0.0,null,null],[5132.5,null,null,1981.025,1181.514,null,125.852,105.775,0.0,0.0,null,0.846,0.812,0.976,null,0.0,0.0,0.0,null,4.321,-23.17,0.0,0.0,null,null],[5133.0,null,null,1979.546,1199.483,null,125.986,106.604,0.0,0.0,null,0.834,0.797,0.996,null,0.0,0.0,0.0,null,4.235,-30.047,0.0,0.0,null,null],[5133.5,null,null,1988.636,1217.389,null,126.265,108.224,0.0,0.0,null,0.821,0.78,0.985,null,0.0,0.0,0.0,null,4.165,-31.555,0.0,0.0,null,null],[5134.0,null,null,1995.652,1233.451,null,126.596,109.106,0.0,0.0,null,0.811,0.768,0.929,null,0.0,0.0,0.0,null,4.164,-25.522,0.0,0.0,null,null],[5134.5,null,null,1992.839,1250.019,null,126.847,108.254,0.0,0.0,null,0.8,0.76,0.878,null,0.0,0.0,0.0,null,4.173,-17.825,0.0,0.0,null,null],[5135.0,null,null,1991.196,1268.478,null,127.032,108.009,0.0,0.0,null,0.788,0.753,0.848,null,0.0,0.0,0.0,null,4.121,-11.809,0.0,0.0,null,null],[5135.5,null,null,1991.777,1288.681,null,127.32,109.642,0.0,0.0,null,0.776,0.745,0.857,null,0.0,0.0,0.0,null,4.031,-6.616,0.0,0.0,null,null],[5136.0,null,null,1986.588,1309.531,null,127.851,111.178,0.0,0.0,null,0.764,0.735,0.864,null,0.0,0.0,0.0,null,3.967,-4.459,0.0,0.0,null,null],[5136.5,null,null,1982.351,1326.742,null,128.552,111.974,0.0,0.0,null,0.754,0.727,0.859,null,0.0,0.0,0.0,null,3.862,-7.517,0.0,0.0,null,null],[5137.0,null,null,1984.92,1333.529,null,129.188,113.122,0.0,0.0,null,0.75,0.722,0.852,null,0.0,0.0,0.0,null,3.592,-10.098,0.0,0.0,null,null],[5137.5,null,null,1986.238,1323.347,null,129.453,112.3,0.0,0.0,null,0.756,0.724,0.834,null,0.0,0.0,0.0,null,3.222,-5.711,0.0,0.0,null,null],[5138.0,null,null,1985.817,1287.573,null,129.073,108.147,0.0,0.0,null,0.777,0.739,0.803,null,0.0,0.0,0.0,null,2.801,3.646,0.0,0.0,null,null],[5138.5,null,null,1988.014,1225.496,null,127.901,103.775,0.0,0.0,null,0.816,0.777,0.747,null,0.0,0.0,0.0,null,2.32,6.007,0.0,0.0,null,null],[5139.0,null,null,1991.98,1149.548,null,125.951,103.175,0.0,0.0,null,0.87,0.845,0.709,null,0.0,0.0,0.0,null,1.913,-2.145,0.0,0.0,null,null],[5139.5,null,null,1994.924,1071.629,null,123.321,105.867,0.0,0.0,null,0.933,0.938,0.784,null,0.0,0.0,0.0,null,1.579,-12.944,0.0,0.0,null,null],[5140.0,null,null,1995.069,994.413,null,120.167,106.345,0.0,0.0,null,1.006,1.038,1.073,null,0.0,0.0,0.0,null,1.073,-21.396,0.0,0.0,null,null],[5140.5,null,null,1995.145,917.975,null,116.763,102.009,0.0,0.0,null,1.089,1.131,1.536,null,0.0,0.0,0.0,null,0.425,-27.804,0.0,0.0,null,null],[5141.0,null,null,1997.208,843.955,null,113.488,96.769,0.0,0.0,null,1.185,1.218,1.772,null,0.0,0.0,0.0,null,0.001,-29.185,0.0,0.0,null,null],[5141.5,null,null,1999.972,776.336,null,110.749,94.802,0.0,0.0,null,1.288,1.301,1.758,null,0.0,0.0,0.0,null,-0.102,-24.517,0.0,0.0,null,null],[5142.0,null,null,1998.917,724.148,null,108.778,96.001,0.0,0.0,null,1.381,1.369,1.715,null,0.0,0.0,0.0,null,-0.085,-18.124,0.0,0.0,null,null],[5142.5,null,null,1996.55,694.256,null,107.581,98.496,0.0,0.0,null,1.44,1.407,1.715,null,0.0,0.0,0.0,null,-0.112,-14.404,0.0,0.0,null,null],[5143.0,null,null,1999.32,685.22,null,107.107,99.674,0.0,0.0,null,1.459,1.413,1.69,null,0.0,0.0,0.0,null,-0.211,-15.004,0.0,0.0,null,null],[5143.5,null,null,2005.219,689.764,null,107.354,98.842,0.0,0.0,null,1.45,1.399,1.622,null,0.0,0.0,0.0,null,-0.326,-17.866,0.0,0.0,null,null],[5144.0,null,null,2002.709,700.905,null,108.201,98.276,0.0,0.0,null,1.427,1.375,1.533,null,0.0,0.0,0.0,null,-0.36,-18.124,0.0,0.0,null,null],[5144.5,null,null,1993.504,715.143,null,109.379,99.459,0.0,0.0,null,1.398,1.346,1.476,null,0.0,0.0,0.0,null,-0.232,-15.271,0.0,0.0,null,null],[5145.0,null,null,1987.534,731.844,null,110.664,101.056,0.0,0.0,null,1.366,1.312,1.444,null,0.0,0.0,0.0,null,-0.05,-11.851,0.0,0.0,null,null],[5145.5,null,null,1985.736,750.334,null,112.049,101.61,0.0,0.0,null,1.333,1.275,1.421,null,0.0,0.0,0.0,null,0.026,-8.584,0.0,0.0,null,null],[5146.0,null,null,1985.104,770.014,null,113.66,101.763,0.0,0.0,null,1.299,1.236,1.402,null,0.0,0.0,0.0,null,0.019,-5.411,0.0,0.0,null,null],[5146.5,null,null,1988.836,790.333,null,115.478,101.993,0.0,0.0,null,1.265,1.197,1.373,null,0.0,0.0,0.0,null,0.06,-2.162,0.0,0.0,null,null],[5147.0,null,null,1997.369,808.973,null,117.282,101.921,0.0,0.0,null,1.236,1.162,1.34,null,0.0,0.0,0.0,null,0.135,1.795,0.0,0.0,null,null],[5147.5,null,null,2005.108,824.866,null,118.86,102.178,0.0,0.0,null,1.212,1.133,1.3,null,0.0,0.0,0.0,null,0.179,4.112,0.0,0.0,null,null],[5148.0,null,null,2008.06,838.998,null,120.196,103.358,0.0,0.0,null,1.192,1.111,1.258,null,0.0,0.0,0.0,null,0.202,4.892,0.0,0.0,null,null],[5148.5,null,null,2006.353,852.369,null,121.368,104.933,0.0,0.0,null,1.173,1.097,1.227,null,0.0,0.0,0.0,null,0.308,5.81,0.0,0.0,null,null],[5149.0,null,null,2005.478,864.15,null,122.338,106.458,0.0,0.0,null,1.157,1.088,1.186,null,0.0,0.0,0.0,null,0.565,3.423,0.0,0.0,null,null],[5149.5,null,null,2010.704,872.612,null,122.956,109.071,0.0,0.0,null,1.146,1.085,1.144,null,0.0,0.0,0.0,null,0.859,-2.062,0.0,0.0,null,null],[5150.0,null,null,2016.806,873.938,null,123.054,112.297,0.0,0.0,null,1.144,1.089,1.133,null,0.0,0.0,0.0,null,0.943,-4.062,0.0,0.0,null,null],[5150.5,null,null,2015.906,861.886,null,122.45,114.379,0.0,0.0,null,1.16,1.106,1.153,null,0.0,0.0,0.0,null,0.775,-0.62,0.0,0.0,null,null],[5151.0,null,null,2009.615,837.862,null,120.969,114.202,0.0,0.0,null,1.194,1.145,1.166,null,0.0,0.0,0.0,null,0.534,3.169,0.0,0.0,null,null],[5151.5,null,null,2006.777,811.012,null,118.682,113.22,0.0,0.0,null,1.233,1.205,1.164,null,0.0,0.0,0.0,null,0.379,2.809,0.0,0.0,null,null],[5152.0,null,null,2008.799,789.9,null,116.1,111.866,0.0,0.0,null,1.266,1.27,1.201,null,0.0,0.0,0.0,null,0.297,-0.231,0.0,0.0,null,null],[5152.5,null,null,2012.78,777.303,null,113.968,108.162,0.0,0.0,null,1.286,1.314,1.388,null,0.0,0.0,0.0,null,0.189,-2.439,0.0,0.0,null,null],[5153.0,null,null,2014.448,772.232,null,112.714,101.604,0.0,0.0,null,1.295,1.317,1.741,null,0.0,0.0,0.0,null,0.017,-4.785,0.0,0.0,null,null],[5153.5,null,null,2011.828,770.657,null,112.065,95.373,0.0,0.0,null,1.298,1.287,2.026,null,0.0,0.0,0.0,null,-0.104,-5.373,0.0,0.0,null,null],[5154.0,null,null,2004.114,771.927,null,111.518,91.715,0.0,0.0,null,1.295,1.247,1.925,null,0.0,0.0,0.0,null,0.0,-3.788,0.0,0.0,null,null],[5154.5,null,null,1996.517,778.805,null,111.159,89.852,0.0,0.0,null,1.284,1.212,1.597,null,0.0,0.0,0.0,null,0.422,-7.025,0.0,0.0,null,null],[5155.0,null,null,1992.632,793.334,null,111.648,88.819,0.0,0.0,null,1.261,1.184,1.33,null,0.0,0.0,0.0,null,1.077,-14.999,0.0,0.0,null,null],[5155.5,null,null,1989.134,818.456,null,113.396,88.695,0.0,0.0,null,1.222,1.154,1.205,null,0.0,0.0,0.0,null,1.663,-18.188,0.0,0.0,null,null],[5156.0,null,null,1985.234,854.931,null,115.942,89.33,0.0,0.0,null,1.17,1.115,1.24,null,0.0,0.0,0.0,null,1.909,-14.548,0.0,0.0,null,null],[5156.5,null,null,1983.977,896.358,null,118.257,89.835,0.0,0.0,null,1.116,1.069,1.334,null,0.0,0.0,0.0,null,2.033,-7.452,0.0,0.0,null,null],[5157.0,null,null,1987.347,934.964,null,119.701,90.514,0.0,0.0,null,1.07,1.025,1.332,null,0.0,0.0,0.0,null,2.408,0.269,0.0,0.0,null,null],[5157.5,null,null,1993.078,969.644,null,120.532,92.044,0.0,0.0,null,1.031,0.985,1.203,null,0.0,0.0,0.0,null,2.946,1.764,0.0,0.0,null,null],[5158.0,null,null,1994.659,1000.562,null,121.332,93.454,0.0,0.0,null,0.999,0.949,1.105,null,0.0,0.0,0.0,null,3.306,-2.039,0.0,0.0,null,null],[5158.5,null,null,1992.217,1026.077,null,122.279,94.541,0.0,0.0,null,0.975,0.918,1.062,null,0.0,0.0,0.0,null,3.449,-2.929,0.0,0.0,null,null],[5159.0,null,null,1992.848,1045.881,null,123.136,96.266,0.0,0.0,null,0.956,0.898,0.995,null,0.0,0.0,0.0,null,3.537,-0.414,0.0,0.0,null,null],[5159.5,null,null,1996.249,1062.219,null,123.665,98.022,0.0,0.0,null,0.941,0.887,0.942,null,0.0,0.0,0.0,null,3.604,-2.468,0.0,0.0,null,null],[5160.0,null,null,1997.074,1074.783,null,123.881,98.259,0.0,0.0,null,0.93,0.882,0.952,null,0.0,0.0,0.0,null,3.633,-7.393,0.0,0.0,null,null],[5160.5,null,null,1998.043,1081.582,null,123.976,96.93,0.0,0.0,null,0.925,0.881,0.987,null,0.0,0.0,0.0,null,3.615,-9.68,0.0,0.0,null,null],[5161.0,null,null,1999.365,1081.802,null,124.07,95.855,0.0,0.0,null,0.924,0.884,1.003,null,0.0,0.0,0.0,null,3.544,-12.423,0.0,0.0,null,null],[5161.5,null,null,1994.991,1075.48,null,124.127,95.853,0.0,0.0,null,0.93,0.889,1.029,null,0.0,0.0,0.0,null,3.42,-18.472,0.0,0.0,null,null],[5162.0,null,null,1989.476,1064.433,null,124.141,96.176,0.0,0.0,null,0.939,0.896,1.085,null,0.0,0.0,0.0,null,3.329,-23.443,0.0,0.0,null,null],[5162.5,null,null,1992.354,1055.695,null,124.24,96.57,0.0,0.0,null,0.947,0.904,1.087,null,0.0,0.0,0.0,null,3.386,-26.428,0.0,0.0,null,null],[5163.0,null,null,1998.957,1054.106,null,124.577,97.661,0.0,0.0,null,0.949,0.908,1.062,null,0.0,0.0,0.0,null,3.562,-29.665,0.0,0.0,null,null],[5163.5,null,null,1999.821,1058.435,null,125.136,97.908,0.0,0.0,null,0.945,0.903,1.083,null,0.0,0.0,0.0,null,3.753,-30.904,0.0,0.0,null,null],[5164.0,null,null,1997.615,1067.05,null,125.744,96.584,0.0,0.0,null,0.937,0.893,1.108,null,0.0,0.0,0.0,null,3.889,-29.338,0.0,0.0,null,null],[5164.5,null,null,1995.247,1078.461,null,126.193,96.332,0.0,0.0,null,0.927,0.881,1.075,null,0.0,0.0,0.0,null,3.949,-26.099,0.0,0.0,null,null],[5165.0,null,null,1990.581,1086.415,null,126.427,98.337,0.0,0.0,null,0.92,0.871,1.02,null,0.0,0.0,0.0,null,3.88,-24.223,0.0,0.0,null,null],[5165.5,null,null,1985.882,1087.931,null,126.567,100.59,0.0,0.0,null,0.919,0.866,0.988,null,0.0,0.0,0.0,null,3.703,-26.727,0.0,0.0,null,null],[5166.0,null,null,1983.147,1086.49,null,126.716,101.074,0.0,0.0,null,0.92,0.868,0.97,null,0.0,0.0,0.0,null,3.569,-29.227,0.0,0.0,null,null],[5166.5,null,null,1978.937,1086.62,null,126.782,99.798,0.0,0.0,null,0.92,0.872,0.996,null,0.0,0.0,0.0,null,3.547,-24.224,0.0,0.0,null,null],[5167.0,null,null,1973.854,1088.742,null,126.547,97.671,0.0,0.0,null,0.918,0.876,1.047,null,0.0,0.0,0.0,null,3.565,-15.294,0.0,0.0,null,null],[5167.5,null,null,1974.913,1090.334,null,125.955,96.255,0.0,0.0,null,0.917,0.876,1.076,null,0.0,0.0,0.0,null,3.554,-10.097,0.0,0.0,null,null],[5168.0,null,null,1980.095,1089.433,null,125.203,96.437,0.0,0.0,null,0.918,0.875,1.065,null,0.0,0.0,0.0,null,3.524,-10.743,0.0,0.0,null,null],[5168.5,null,null,1979.67,1088.208,null,124.627,98.127,0.0,0.0,null,0.919,0.873,1.067,null,0.0,0.0,0.0,null,3.457,-14.712,0.0,0.0,null,null],[5169.0,null,null,1976.444,1090.123,null,124.516,99.919,0.0,0.0,null,0.917,0.869,1.07,null,0.0,0.0,0.0,null,3.363,-18.162,0.0,0.0,null,null],[5169.5,null,null,1978.5,1098.136,null,124.948,100.031,0.0,0.0,null,0.911,0.862,1.036,null,0.0,0.0,0.0,null,3.347,-19.214,0.0,0.0,null,null],[5170.0,null,null,1979.216,1111.231,null,125.801,99.281,0.0,0.0,null,0.9,0.851,1.012,null,0.0,0.0,0.0,null,3.461,-20.84,0.0,0.0,null,null],[5170.5,null,null,1974.335,1126.219,null,126.851,99.579,0.0,0.0,null,0.888,0.838,1.008,null,0.0,0.0,0.0,null,3.611,-23.856,0.0,0.0,null,null],[5171.0,null,null,1971.057,1139.654,null,127.908,100.711,0.0,0.0,null,0.877,0.826,0.998,null,0.0,0.0,0.0,null,3.694,-26.273,0.0,0.0,null,null],[5171.5,null,null,1972.385,1151.268,null,128.888,101.451,0.0,0.0,null,0.869,0.818,0.956,null,0.0,0.0,0.0,null,3.68,-27.693,0.0,0.0,null,null],[5172.0,null,null,1974.366,1162.752,null,129.839,102.131,0.0,0.0,null,0.86,0.815,0.923,null,0.0,0.0,0.0,null,3.617,-28.044,0.0,0.0,null,null],[5172.5,null,null,1973.842,1173.251,null,130.764,102.801,0.0,0.0,null,0.852,0.812,0.933,null,0.0,0.0,0.0,null,3.586,-25.909,0.0,0.0,null,null],[5173.0,null,null,1970.001,1179.099,null,131.436,101.913,0.0,0.0,null,0.848,0.807,0.967,null,0.0,0.0,0.0,null,3.62,-20.687,0.0,0.0,null,null],[5173.5,null,null,1967.78,1179.865,null,131.578,99.785,0.0,0.0,null,0.848,0.804,0.981,null,0.0,0.0,0.0,null,3.699,-15.729,0.0,0.0,null,null],[5174.0,null,null,1971.295,1178.592,null,131.209,98.433,0.0,0.0,null,0.848,0.805,0.962,null,0.0,0.0,0.0,null,3.821,-14.587,0.0,0.0,null,null],[5174.5,null,null,1977.306,1176.323,null,130.689,98.417,0.0,0.0,null,0.85,0.807,0.937,null,0.0,0.0,0.0,null,3.914,-18.358,0.0,0.0,null,null],[5175.0,null,null,1982.877,1173.747,null,130.386,98.388,0.0,0.0,null,0.852,0.808,0.95,null,0.0,0.0,0.0,null,3.878,-21.666,0.0,0.0,null,null],[5175.5,null,null,1988.898,1173.518,null,130.295,98.218,0.0,0.0,null,0.852,0.807,0.978,null,0.0,0.0,0.0,null,3.738,-18.077,0.0,0.0,null,null],[5176.0,null,null,1992.152,1175.762,null,130.088,98.359,0.0,0.0,null,0.851,0.806,0.974,null,0.0,0.0,0.0,null,3.605,-11.372,0.0,0.0,null,null],[5176.5,null,null,1990.268,1177.778,null,129.565,98.731,0.0,0.0,null,0.849,0.807,0.965,null,0.0,0.0,0.0,null,3.521,-12.381,0.0,0.0,null,null],[5177.0,null,null,1990.108,1177.419,null,128.88,99.157,0.0,0.0,null,0.849,0.811,0.962,null,0.0,0.0,0.0,null,3.489,-17.43,0.0,0.0,null,null],[5177.5,null,null,1996.448,1174.431,null,128.359,100.386,0.0,0.0,null,0.851,0.813,0.959,null,0.0,0.0,0.0,null,3.538,-18.923,0.0,0.0,null,null],[5178.0,null,null,2001.865,1171.349,null,128.185,102.107,0.0,0.0,null,0.854,0.813,0.971,null,0.0,0.0,0.0,null,3.641,-18.927,0.0,0.0,null,null],[5178.5,null,null,2001.007,1171.528,null,128.246,103.051,0.0,0.0,null,0.854,0.812,0.996,null,0.0,0.0,0.0,null,3.734,-16.11,0.0,0.0,null,null],[5179.0,null,null,1997.286,1173.526,null,128.306,102.511,0.0,0.0,null,0.852,0.811,1.005,null,0.0,0.0,0.0,null,3.813,-10.914,0.0,0.0,null,null],[5179.5,null,null,1993.643,1172.691,null,128.299,102.18,0.0,0.0,null,0.853,0.81,0.993,null,0.0,0.0,0.0,null,3.911,-8.817,0.0,0.0,null,null],[5180.0,null,null,1992.124,1169.562,null,128.389,102.316,0.0,0.0,null,0.855,0.808,0.984,null,0.0,0.0,0.0,null,4.002,-10.106,0.0,0.0,null,null],[5180.5,null,null,1996.904,1168.24,null,128.745,102.459,0.0,0.0,null,0.856,0.807,0.988,null,0.0,0.0,0.0,null,4.034,-12.392,0.0,0.0,null,null],[5181.0,null,null,2007.703,1170.432,null,129.348,102.317,0.0,0.0,null,0.854,0.809,0.978,null,0.0,0.0,0.0,null,4.007,-18.619,0.0,0.0,null,null],[5181.5,null,null,2018.331,1173.801,null,129.996,102.108,0.0,0.0,null,0.852,0.81,0.977,null,0.0,0.0,0.0,null,3.982,-25.228,0.0,0.0,null,null],[5182.0,null,null,2022.446,1174.281,null,130.462,102.052,0.0,0.0,null,0.852,0.811,0.993,null,0.0,0.0,0.0,null,3.99,-27.111,0.0,0.0,null,null],[5182.5,null,null,2018.527,1170.683,null,130.677,102.149,0.0,0.0,null,0.854,0.81,1.008,null,0.0,0.0,0.0,null,3.96,-28.459,0.0,0.0,null,null],[5183.0,null,null,2010.029,1165.926,null,130.722,101.321,0.0,0.0,null,0.858,0.808,1.009,null,0.0,0.0,0.0,null,3.863,-30.769,0.0,0.0,null,null],[5183.5,null,null,2004.971,1164.594,null,130.679,99.951,0.0,0.0,null,0.859,0.808,0.989,null,0.0,0.0,0.0,null,3.827,-29.501,0.0,0.0,null,null],[5184.0,null,null,2007.861,1169.285,null,130.564,100.354,0.0,0.0,null,0.855,0.808,0.966,null,0.0,0.0,0.0,null,3.97,-25.617,0.0,0.0,null,null],[5184.5,null,null,2014.567,1177.673,null,130.355,101.738,0.0,0.0,null,0.849,0.805,0.967,null,0.0,0.0,0.0,null,4.21,-21.861,0.0,0.0,null,null],[5185.0,null,null,2018.064,1185.214,null,130.048,100.353,0.0,0.0,null,0.844,0.799,0.991,null,0.0,0.0,0.0,null,4.376,-16.636,0.0,0.0,null,null],[5185.5,null,null,2017.576,1189.893,null,129.686,96.965,0.0,0.0,null,0.84,0.794,0.992,null,0.0,0.0,0.0,null,4.445,-11.697,0.0,0.0,null,null],[5186.0,null,null,2015.834,1192.469,null,129.338,96.542,0.0,0.0,null,0.839,0.79,0.971,null,0.0,0.0,0.0,null,4.484,-12.705,0.0,0.0,null,null],[5186.5,null,null,2014.237,1193.398,null,129.061,98.863,0.0,0.0,null,0.838,0.787,0.97,null,0.0,0.0,0.0,null,4.534,-18.142,0.0,0.0,null,null],[5187.0,null,null,2013.807,1196.207,null,128.917,99.575,0.0,0.0,null,0.836,0.784,0.964,null,0.0,0.0,0.0,null,4.612,-21.53,0.0,0.0,null,null],[5187.5,null,null,2012.85,1202.821,null,128.989,98.287,0.0,0.0,null,0.831,0.781,0.949,null,0.0,0.0,0.0,null,4.693,-20.172,0.0,0.0,null,null],[5188.0,null,null,2010.627,1211.643,null,129.284,97.415,0.0,0.0,null,0.825,0.779,0.942,null,0.0,0.0,0.0,null,4.72,-13.258,0.0,0.0,null,null],[5188.5,null,null,2014.43,1220.967,null,129.666,97.749,0.0,0.0,null,0.819,0.776,0.952,null,0.0,0.0,0.0,null,4.684,-3.799,0.0,0.0,null,null],[5189.0,null,null,2025.968,1229.487,null,129.961,98.415,0.0,0.0,null,0.813,0.772,0.956,null,0.0,0.0,0.0,null,4.586,-0.005,0.0,0.0,null,null],[5189.5,null,null,2028.503,1233.611,null,130.131,99.947,0.0,0.0,null,0.811,0.769,0.941,null,0.0,0.0,0.0,null,4.408,-2.752,0.0,0.0,null,null],[5190.0,null,null,2013.273,1233.961,null,130.284,102.381,0.0,0.0,null,0.81,0.768,0.934,null,0.0,0.0,0.0,null,4.228,-5.827,0.0,0.0,null,null],[5190.5,null,null,1996.339,1234.237,null,130.528,104.082,0.0,0.0,null,0.81,0.767,0.947,null,0.0,0.0,0.0,null,4.146,-7.455,0.0,0.0,null,null],[5191.0,null,null,1991.316,1234.89,null,130.799,103.347,0.0,0.0,null,0.81,0.765,0.969,null,0.0,0.0,0.0,null,4.187,-11.217,0.0,0.0,null,null],[5191.5,null,null,1994.695,1236.462,null,130.9,101.553,0.0,0.0,null,0.809,0.764,0.968,null,0.0,0.0,0.0,null,4.289,-17.014,0.0,0.0,null,null],[5192.0,null,null,1999.027,1240.397,null,130.72,100.394,0.0,0.0,null,0.806,0.763,0.947,null,0.0,0.0,0.0,null,4.35,-21.813,0.0,0.0,null,null],[5192.5,null,null,2001.737,1244.008,null,130.371,99.829,0.0,0.0,null,0.804,0.763,0.933,null,0.0,0.0,0.0,null,4.328,-24.444,0.0,0.0,null,null],[5193.0,null,null,2002.007,1243.291,null,130.116,99.898,0.0,0.0,null,0.804,0.765,0.94,null,0.0,0.0,0.0,null,4.269,-26.388,0.0,0.0,null,null],[5193.5,null,null,2000.673,1239.246,null,130.105,100.239,0.0,0.0,null,0.807,0.767,0.949,null,0.0,0.0,0.0,null,4.263,-27.818,0.0,0.0,null,null],[5194.0,null,null,1999.515,1235.617,null,130.246,100.485,0.0,0.0,null,0.809,0.768,0.941,null,0.0,0.0,0.0,null,4.371,-27.515,0.0,0.0,null,null],[5194.5,null,null,1998.402,1235.575,null,130.334,100.098,0.0,0.0,null,0.809,0.768,0.943,null,0.0,0.0,0.0,null,4.521,-26.631,0.0,0.0,null,null],[5195.0,null,null,1997.029,1238.311,null,130.274,98.857,0.0,0.0,null,0.808,0.764,0.959,null,0.0,0.0,0.0,null,4.578,-28.287,0.0,0.0,null,null],[5195.5,null,null,1995.179,1241.073,null,130.142,97.543,0.0,0.0,null,0.806,0.76,0.966,null,0.0,0.0,0.0,null,4.57,-31.614,0.0,0.0,null,null],[5196.0,null,null,1993.508,1244.033,null,130.081,97.405,0.0,0.0,null,0.804,0.758,0.948,null,0.0,0.0,0.0,null,4.59,-30.812,0.0,0.0,null,null],[5196.5,null,null,1990.032,1248.268,null,130.151,96.955,0.0,0.0,null,0.801,0.758,0.916,null,0.0,0.0,0.0,null,4.655,-25.602,0.0,0.0,null,null],[5197.0,null,null,1985.221,1254.78,null,130.239,96.133,0.0,0.0,null,0.797,0.757,0.92,null,0.0,0.0,0.0,null,4.714,-21.995,0.0,0.0,null,null],[5197.5,null,null,1985.189,1263.566,null,130.203,97.123,0.0,0.0,null,0.791,0.753,0.95,null,0.0,0.0,0.0,null,4.735,-21.052,0.0,0.0,null,null],[5198.0,null,null,1993.579,1271.986,null,130.001,99.452,0.0,0.0,null,0.786,0.748,0.967,null,0.0,0.0,0.0,null,4.721,-19.572,0.0,0.0,null,null],[5198.5,null,null,2002.243,1277.36,null,129.7,100.702,0.0,0.0,null,0.783,0.744,0.958,null,0.0,0.0,0.0,null,4.762,-18.396,0.0,0.0,null,null],[5199.0,null,null,2001.323,1280.198,null,129.441,100.452,0.0,0.0,null,0.781,0.742,0.925,null,0.0,0.0,0.0,null,4.918,-20.783,0.0,0.0,null,null],[5199.5,null,null,1995.018,1282.248,null,129.333,100.578,0.0,0.0,null,0.78,0.741,0.904,null,0.0,0.0,0.0,null,5.101,-25.937,0.0,0.0,null,null],[5200.0,null,null,1991.65,1283.594,null,129.368,102.262,0.0,0.0,null,0.779,0.739,0.899,null,0.0,0.0,0.0,null,5.148,-27.791,0.0,0.0,null,null],[5200.5,null,null,1991.928,1285.273,null,129.468,103.509,0.0,0.0,null,0.778,0.74,0.907,null,0.0,0.0,0.0,null,5.002,-23.442,0.0,0.0,null,null],[5201.0,null,null,1993.057,1286.818,null,129.599,102.18,0.0,0.0,null,0.777,0.741,0.915,null,0.0,0.0,0.0,null,4.807,-17.815,0.0,0.0,null,null],[5201.5,null,null,1994.148,1286.29,null,129.786,99.749,0.0,0.0,null,0.777,0.744,0.916,null,0.0,0.0,0.0,null,4.741,-13.448,0.0,0.0,null,null],[5202.0,null,null,1993.9,1281.795,null,130.085,98.264,0.0,0.0,null,0.78,0.746,0.929,null,0.0,0.0,0.0,null,4.741,-5.487,0.0,0.0,null,null],[5202.5,null,null,1992.586,1275.889,null,130.53,97.825,0.0,0.0,null,0.784,0.75,0.943,null,0.0,0.0,0.0,null,4.663,0.652,0.0,0.0,null,null],[5203.0,null,null,1994.808,1270.269,null,131.014,97.606,0.0,0.0,null,0.787,0.754,0.944,null,0.0,0.0,0.0,null,4.546,-3.709,0.0,0.0,null,null],[5203.5,null,null,1999.455,1263.906,null,131.317,96.447,0.0,0.0,null,0.791,0.759,0.95,null,0.0,0.0,0.0,null,4.456,-10.377,0.0,0.0,null,null],[5204.0,null,null,2001.471,1255.305,null,131.303,96.303,0.0,0.0,null,0.797,0.764,0.952,null,0.0,0.0,0.0,null,4.336,-11.35,0.0,0.0,null,null],[5204.5,null,null,2005.412,1246.556,null,131.044,98.704,0.0,0.0,null,0.802,0.767,0.959,null,0.0,0.0,0.0,null,4.215,-12.891,0.0,0.0,null,null],[5205.0,null,null,2016.009,1240.874,null,130.73,100.477,0.0,0.0,null,0.806,0.77,0.962,null,0.0,0.0,0.0,null,4.191,-16.062,0.0,0.0,null,null],[5205.5,null,null,2025.323,1238.535,null,130.501,98.681,0.0,0.0,null,0.807,0.772,0.967,null,0.0,0.0,0.0,null,4.274,-15.823,0.0,0.0,null,null],[5206.0,null,null,2023.878,1237.492,null,130.327,95.83,0.0,0.0,null,0.808,0.772,0.976,null,0.0,0.0,0.0,null,4.408,-13.964,0.0,0.0,null,null],[5206.5,null,null,2015.369,1235.814,null,130.097,95.076,0.0,0.0,null,0.809,0.77,0.961,null,0.0,0.0,0.0,null,4.545,-11.571,0.0,0.0,null,null],[5207.0,null,null,2004.997,1234.114,null,129.874,96.35,0.0,0.0,null,0.81,0.768,0.95,null,0.0,0.0,0.0,null,4.656,-5.748,0.0,0.0,null,null],[5207.5,null,null,1993.637,1235.033,null,129.893,97.854,0.0,0.0,null,0.81,0.766,0.953,null,0.0,0.0,0.0,null,4.754,0.364,0.0,0.0,null,null],[5208.0,null,null,1987.866,1239.687,null,130.267,98.533,0.0,0.0,null,0.807,0.767,0.951,null,0.0,0.0,0.0,null,4.866,0.337,0.0,0.0,null,null],[5208.5,null,null,1994.673,1244.416,null,130.806,98.205,0.0,0.0,null,0.804,0.768,0.95,null,0.0,0.0,0.0,null,4.958,-3.024,0.0,0.0,null,null],[5209.0,null,null,2006.969,1246.64,null,131.15,97.834,0.0,0.0,null,0.802,0.767,0.966,null,0.0,0.0,0.0,null,4.99,-1.021,0.0,0.0,null,null],[5209.5,null,null,2010.088,1247.185,null,131.096,97.586,0.0,0.0,null,0.802,0.765,0.99,null,0.0,0.0,0.0,null,4.956,2.133,0.0,0.0,null,null],[5210.0,null,null,2006.811,1247.827,null,130.806,96.844,0.0,0.0,null,0.801,0.763,0.99,null,0.0,0.0,0.0,null,4.906,-0.896,0.0,0.0,null,null],[5210.5,null,null,2007.902,1248.156,null,130.607,95.917,0.0,0.0,null,0.801,0.762,0.957,null,0.0,0.0,0.0,null,4.897,-5.132,0.0,0.0,null,null],[5211.0,null,null,2011.972,1248.187,null,130.6,95.889,0.0,0.0,null,0.801,0.762,0.947,null,0.0,0.0,0.0,null,4.911,-4.608,0.0,0.0,null,null],[5211.5,null,null,2013.353,1246.947,null,130.609,96.531,0.0,0.0,null,0.802,0.763,0.954,null,0.0,0.0,0.0,null,4.882,-4.326,0.0,0.0,null,null],[5212.0,null,null,2013.28,1242.015,null,130.462,97.179,0.0,0.0,null,0.805,0.765,0.952,null,0.0,0.0,0.0,null,4.789,-9.469,0.0,0.0,null,null],[5212.5,null,null,2015.196,1233.727,null,130.214,97.758,0.0,0.0,null,0.811,0.769,0.924,null,0.0,0.0,0.0,null,4.648,-17.774,0.0,0.0,null,null],[5213.0,null,null,2018.012,1225.959,null,130.111,98.0,0.0,0.0,null,0.816,0.777,0.906,null,0.0,0.0,0.0,null,4.499,-20.426,0.0,0.0,null,null],[5213.5,null,null,2020.616,1218.281,null,130.338,98.176,0.0,0.0,null,0.821,0.786,0.922,null,0.0,0.0,0.0,null,4.402,-17.277,0.0,0.0,null,null],[5214.0,null,null,2022.062,1207.481,null,130.791,97.956,0.0,0.0,null,0.828,0.793,0.943,null,0.0,0.0,0.0,null,4.367,-18.335,0.0,0.0,null,null],[5214.5,null,null,2021.983,1191.706,null,131.179,97.056,0.0,0.0,null,0.839,0.799,0.956,null,0.0,0.0,0.0,null,4.321,-22.545,0.0,0.0,null,null],[5215.0,null,null,2022.015,1169.261,null,131.252,96.446,0.0,0.0,null,0.855,0.809,0.957,null,0.0,0.0,0.0,null,4.16,-21.919,0.0,0.0,null,null],[5215.5,null,null,2017.818,1133.497,null,130.906,97.603,0.0,0.0,null,0.882,0.827,0.954,null,0.0,0.0,0.0,null,3.903,-17.839,0.0,0.0,null,null],[5216.0,null,null,2008.166,1082.634,null,130.198,99.254,0.0,0.0,null,0.924,0.857,0.945,null,0.0,0.0,0.0,null,3.628,-14.911,0.0,0.0,null,null],[5216.5,null,null,2001.247,1027.717,null,129.276,98.861,0.0,0.0,null,0.973,0.905,0.909,null,0.0,0.0,0.0,null,3.392,-14.347,0.0,0.0,null,null],[5217.0,null,null,2002.309,981.392,null,128.282,95.641,0.0,0.0,null,1.019,0.968,0.878,null,0.0,0.0,0.0,null,3.216,-17.463,0.0,0.0,null,null],[5217.5,null,null,2007.31,944.043,null,127.247,91.462,0.0,0.0,null,1.059,1.031,1.014,null,0.0,0.0,0.0,null,3.011,-17.628,0.0,0.0,null,null],[5218.0,null,null,2013.093,912.162,null,126.139,88.846,0.0,0.0,null,1.096,1.076,1.345,null,0.0,0.0,0.0,null,2.637,-11.387,0.0,0.0,null,null],[5218.5,null,null,2018.621,881.814,null,124.951,89.817,0.0,0.0,null,1.134,1.105,1.558,null,0.0,0.0,0.0,null,2.195,-7.322,0.0,0.0,null,null],[5219.0,null,null,2022.287,848.991,null,123.772,93.271,0.0,0.0,null,1.178,1.13,1.478,null,0.0,0.0,0.0,null,1.938,-7.888,0.0,0.0,null,null],[5219.5,null,null,2023.721,817.684,null,122.798,95.933,0.0,0.0,null,1.223,1.155,1.315,null,0.0,0.0,0.0,null,1.948,-6.147,0.0,0.0,null,null],[5220.0,null,null,2025.902,798.08,null,122.156,97.364,0.0,0.0,null,1.253,1.173,1.243,null,0.0,0.0,0.0,null,2.066,-4.021,0.0,0.0,null,null],[5220.5,null,null,2024.153,792.831,null,121.807,98.568,0.0,0.0,null,1.261,1.181,1.235,null,0.0,0.0,0.0,null,2.093,-4.029,0.0,0.0,null,null],[5221.0,null,null,2018.469,795.839,null,121.588,99.304,0.0,0.0,null,1.257,1.182,1.289,null,0.0,0.0,0.0,null,2.036,-4.476,0.0,0.0,null,null],[5221.5,null,null,2014.682,801.515,null,121.329,100.227,0.0,0.0,null,1.248,1.184,1.312,null,0.0,0.0,0.0,null,2.016,-9.261,0.0,0.0,null,null],[5222.0,null,null,2014.236,806.358,null,120.95,101.41,0.0,0.0,null,1.24,1.19,1.302,null,0.0,0.0,0.0,null,1.998,-16.96,0.0,0.0,null,null],[5222.5,null,null,2010.231,808.276,null,120.52,100.596,0.0,0.0,null,1.237,1.193,1.356,null,0.0,0.0,0.0,null,1.861,-18.025,0.0,0.0,null,null],[5223.0,null,null,2004.034,806.624,null,120.245,97.536,0.0,0.0,null,1.24,1.19,1.43,null,0.0,0.0,0.0,null,1.7,-16.38,0.0,0.0,null,null],[5223.5,null,null,2001.716,803.069,null,120.285,96.183,0.0,0.0,null,1.245,1.185,1.407,null,0.0,0.0,0.0,null,1.676,-20.247,0.0,0.0,null,null],[5224.0,null,null,2004.531,799.949,null,120.621,97.412,0.0,0.0,null,1.25,1.181,1.302,null,0.0,0.0,0.0,null,1.733,-25.843,0.0,0.0,null,null],[5224.5,null,null,2009.825,799.593,null,121.045,98.004,0.0,0.0,null,1.251,1.176,1.261,null,0.0,0.0,0.0,null,1.722,-24.535,0.0,0.0,null,null],[5225.0,null,null,2013.19,800.258,null,121.311,96.955,0.0,0.0,null,1.25,1.171,1.262,null,0.0,0.0,0.0,null,1.622,-19.067,0.0,0.0,null,null],[5225.5,null,null,2013.852,797.727,null,121.342,96.005,0.0,0.0,null,1.254,1.171,1.266,null,0.0,0.0,0.0,null,1.55,-15.347,0.0,0.0,null,null],[5226.0,null,null,2013.499,793.008,null,121.28,95.688,0.0,0.0,null,1.261,1.178,1.264,null,0.0,0.0,0.0,null,1.6,-13.749,0.0,0.0,null,null],[5226.5,null,null,2012.76,791.089,null,121.297,96.125,0.0,0.0,null,1.264,1.19,1.249,null,0.0,0.0,0.0,null,1.702,-12.533,0.0,0.0,null,null],[5227.0,null,null,2012.78,792.146,null,121.42,98.642,0.0,0.0,null,1.262,1.198,1.273,null,0.0,0.0,0.0,null,1.691,-14.722,0.0,0.0,null,null],[5227.5,null,null,2010.811,791.978,null,121.594,102.656,0.0,0.0,null,1.263,1.2,1.345,null,0.0,0.0,0.0,null,1.635,-19.605,0.0,0.0,null,null],[5228.0,null,null,2005.052,791.312,null,121.77,104.936,0.0,0.0,null,1.264,1.195,1.404,null,0.0,0.0,0.0,null,1.785,-18.415,0.0,0.0,null,null],[5228.5,null,null,2001.297,793.528,null,121.982,103.357,0.0,0.0,null,1.26,1.185,1.399,null,0.0,0.0,0.0,null,2.142,-12.3,0.0,0.0,null,null],[5229.0,null,null,2004.229,799.987,null,122.348,100.79,0.0,0.0,null,1.25,1.172,1.354,null,0.0,0.0,0.0,null,2.441,-12.476,0.0,0.0,null,null],[5229.5,null,null,2008.841,810.537,null,122.977,100.778,0.0,0.0,null,1.234,1.152,1.312,null,0.0,0.0,0.0,null,2.602,-16.506,0.0,0.0,null,null],[5230.0,null,null,2010.502,825.068,null,123.858,101.812,0.0,0.0,null,1.212,1.129,1.284,null,0.0,0.0,0.0,null,2.755,-15.906,0.0,0.0,null,null],[5230.5,null,null,2006.727,841.984,null,124.845,101.134,0.0,0.0,null,1.188,1.105,1.253,null,0.0,0.0,0.0,null,2.941,-16.217,0.0,0.0,null,null],[5231.0,null,null,2000.247,858.939,null,125.711,99.866,0.0,0.0,null,1.164,1.083,1.234,null,0.0,0.0,0.0,null,3.117,-20.603,0.0,0.0,null,null],[5231.5,null,null,1996.018,874.782,null,126.267,99.277,0.0,0.0,null,1.143,1.064,1.213,null,0.0,0.0,0.0,null,3.267,-22.15,0.0,0.0,null,null],[5232.0,null,null,1993.927,889.053,null,126.567,98.795,0.0,0.0,null,1.125,1.049,1.183,null,0.0,0.0,0.0,null,3.352,-21.276,0.0,0.0,null,null],[5232.5,null,null,1990.653,901.513,null,126.834,98.546,0.0,0.0,null,1.109,1.038,1.146,null,0.0,0.0,0.0,null,3.346,-22.766,0.0,0.0,null,null],[5233.0,null,null,1990.645,912.129,null,127.192,98.298,0.0,0.0,null,1.096,1.03,1.133,null,0.0,0.0,0.0,null,3.33,-23.617,0.0,0.0,null,null],[5233.5,null,null,1999.094,920.612,null,127.575,97.32,0.0,0.0,null,1.086,1.023,1.14,null,0.0,0.0,0.0,null,3.342,-21.63,0.0,0.0,null,null],[5234.0,null,null,2008.727,926.428,null,127.825,95.793,0.0,0.0,null,1.079,1.017,1.152,null,0.0,0.0,0.0,null,3.291,-17.695,0.0,0.0,null,null],[5234.5,null,null,2013.056,930.035,null,127.864,94.775,0.0,0.0,null,1.075,1.013,1.151,null,0.0,0.0,0.0,null,3.158,-15.429,0.0,0.0,null,null],[5235.0,null,null,2015.81,931.784,null,127.742,94.544,0.0,0.0,null,1.073,1.011,1.123,null,0.0,0.0,0.0,null,3.058,-14.296,0.0,0.0,null,null],[5235.5,null,null,2019.502,932.029,null,127.605,95.08,0.0,0.0,null,1.073,1.009,1.124,null,0.0,0.0,0.0,null,2.976,-11.375,0.0,0.0,null,null],[5236.0,null,null,2020.211,932.534,null,127.566,96.047,0.0,0.0,null,1.072,1.006,1.181,null,0.0,0.0,0.0,null,2.771,-7.463,0.0,0.0,null,null],[5236.5,null,null,2020.943,933.716,null,127.647,96.271,0.0,0.0,null,1.071,1.005,1.231,null,0.0,0.0,0.0,null,2.512,-6.529,0.0,0.0,null,null],[5237.0,null,null,2027.912,934.005,null,127.802,95.52,0.0,0.0,null,1.071,1.005,1.226,null,0.0,0.0,0.0,null,2.427,-9.081,0.0,0.0,null,null],[5237.5,null,null,2031.972,934.064,null,127.946,96.071,0.0,0.0,null,1.071,1.006,1.175,null,0.0,0.0,0.0,null,2.556,-9.669,0.0,0.0,null,null],[5238.0,null,null,2026.486,935.098,null,128.049,98.889,1.43,64.891,null,1.069,1.005,1.148,null,1.851,1.081,1.713,null,2.749,-5.324,13.518,4.642,null,null],[5238.5,null,null,2017.399,936.843,null,128.149,101.222,1.471,66.598,null,1.067,1.005,1.142,null,1.815,1.105,1.646,null,2.871,-3.072,13.628,4.581,null,null],[5239.0,null,null,2013.421,939.399,null,128.302,101.209,1.519,68.008,null,1.065,1.006,1.135,null,1.782,1.145,1.556,null,2.88,-9.385,13.819,4.479,null,null],[5239.5,null,null,2012.771,940.341,null,128.48,99.849,1.546,68.159,null,1.063,1.008,1.133,null,1.772,1.171,1.513,null,2.806,-17.052,13.987,4.432,null,null],[5240.0,null,null,2010.636,936.465,null,128.563,99.236,1.573,68.281,null,1.068,1.013,1.138,null,1.757,1.189,1.478,null,2.676,-20.112,14.115,4.407,null,null],[5240.5,null,null,2006.06,927.756,null,128.473,99.95,1.572,69.475,null,1.078,1.022,1.147,null,1.795,1.213,1.479,null,2.52,-21.672,14.403,4.405,null,null],[5241.0,null,null,2000.253,917.735,null,128.257,101.639,1.54,71.371,null,1.09,1.033,1.154,null,1.901,1.257,1.512,null,2.405,-22.052,14.942,4.412,null,null],[5241.5,null,null,1994.965,908.938,null,128.024,102.692,1.536,72.859,null,1.1,1.044,1.168,null,1.968,1.317,1.495,null,2.338,-17.036,15.439,4.351,null,null],[5242.0,null,null,1993.859,901.06,null,127.817,101.941,1.554,73.435,null,1.11,1.057,1.197,null,1.98,1.365,1.451,null,2.246,-9.111,15.713,4.271,null,null],[5242.5,null,null,1996.642,891.985,null,127.582,100.147,1.543,73.505,null,1.121,1.07,1.239,null,2.004,1.367,1.466,null,2.115,-4.249,15.785,4.285,null,null],[5243.0,null,null,1998.129,881.571,null,127.273,99.365,1.481,72.867,null,1.134,1.083,1.278,null,2.089,1.333,1.568,null,1.981,-4.859,15.769,4.39,null,null],[5243.5,null,null,1996.19,872.006,null,126.991,100.215,1.345,70.27,null,1.147,1.095,1.301,null,2.308,1.272,1.824,null,1.876,-7.932,15.754,4.604,null,null],[5244.0,null,null,1995.259,865.542,null,126.905,101.818,1.177,66.906,null,1.155,1.1,1.31,null,2.638,1.2,2.207,null,1.797,-6.436,15.743,4.876,null,null],[5244.5,null,null,1998.013,862.31,null,127.027,102.902,1.037,65.195,null,1.16,1.099,1.306,null,3.011,1.159,2.599,null,1.731,1.306,15.851,5.075,null,null],[5245.0,null,null,2002.036,861.097,null,127.176,103.091,0.928,64.685,null,1.161,1.094,1.299,null,3.398,1.145,2.97,null,1.688,6.892,16.034,5.197,null,null],[5245.5,null,null,2007.055,860.362,null,127.188,102.374,0.868,64.585,null,1.162,1.09,1.287,null,3.66,1.15,3.184,null,1.64,4.505,16.208,5.232,null,null],[5246.0,null,null,2013.789,860.363,null,127.088,101.464,0.877,65.458,null,1.162,1.09,1.282,null,3.66,1.187,3.087,null,1.517,0.054,16.384,5.124,null,null],[5246.5,null,null,2016.885,861.514,null,127.034,101.843,0.908,66.44,null,1.161,1.093,1.281,null,3.543,1.22,2.904,null,1.329,-0.628,16.424,4.995,null,null],[5247.0,null,null,2019.993,863.703,null,127.135,103.04,0.932,66.669,null,1.158,1.097,1.278,null,3.426,1.229,2.789,null,1.199,2.166,16.294,4.921,null,null],[5247.5,null,null,2031.918,865.375,null,127.316,102.477,0.963,66.509,null,1.156,1.096,1.286,null,3.278,1.235,2.655,null,1.225,6.062,16.107,4.84,null,null],[5248.0,null,null,2047.986,866.999,null,127.389,100.94,1.017,66.275,null,1.153,1.089,1.294,null,3.064,1.248,2.457,null,1.416,7.09,15.879,4.724,null,null],[5248.5,null,null,2052.763,869.545,null,127.314,100.429,1.09,66.282,null,1.15,1.078,1.296,null,2.829,1.278,2.217,null,1.643,3.332,15.716,4.566,null,null],[5249.0,null,null,2047.597,872.9,null,127.284,99.934,1.163,66.707,null,1.146,1.067,1.279,null,2.645,1.317,2.01,null,1.738,-4.194,15.677,4.417,null,null],[5249.5,null,null,2041.709,877.13,null,127.511,99.682,1.254,68.248,null,1.14,1.057,1.26,null,2.464,1.373,1.8,null,1.75,-10.407,15.736,4.258,null,null],[5250.0,null,null,2040.863,882.9,null,127.995,100.754,1.389,71.38,null,1.133,1.046,1.234,null,2.261,1.475,1.542,null,1.83,-15.192,15.973,4.027,null,null],[5250.5,null,null,2048.061,889.298,null,128.578,101.671,1.514,74.565,null,1.124,1.037,1.197,null,2.112,1.589,1.333,null,1.98,-19.957,16.301,3.811,null,null],[5251.0,null,null,2057.593,894.863,null,129.107,100.991,1.59,76.873,null,1.117,1.03,1.17,null,2.042,1.667,1.226,null,2.11,-23.602,16.571,3.69,null,null],[5251.5,null,null,2060.602,898.691,null,129.52,100.502,1.657,78.948,null,1.113,1.026,1.163,null,1.981,1.709,1.159,null,2.187,-23.713,16.758,3.638,null,null],[5252.0,null,null,2059.248,899.953,null,129.847,101.017,1.703,80.356,null,1.111,1.024,1.178,null,1.932,1.712,1.129,null,2.202,-20.26,16.803,3.643,null,null],[5252.5,null,null,2060.675,899.766,null,130.065,101.193,1.712,80.308,null,1.111,1.024,1.191,null,1.91,1.676,1.14,null,2.163,-16.677,16.695,3.698,null,null],[5253.0,null,null,2058.154,901.219,null,130.078,100.861,1.68,78.218,null,1.11,1.024,1.185,null,1.907,1.572,1.219,null,2.107,-16.933,16.356,3.873,null,null],[5253.5,null,null,2050.819,903.486,null,129.931,100.548,1.599,74.231,null,1.107,1.022,1.194,null,1.923,1.39,1.397,null,2.021,-17.179,15.701,4.217,null,null],[5254.0,null,null,2047.467,903.903,null,129.825,100.883,1.499,70.19,null,1.106,1.018,1.198,null,1.962,1.216,1.623,null,1.902,-13.619,15.008,4.593,null,null],[5254.5,null,null,2047.321,903.442,null,129.898,101.545,1.429,67.567,null,1.107,1.018,1.186,null,1.991,1.113,1.791,null,1.837,-10.377,14.525,4.847,null,null],[5255.0,null,null,2050.071,903.475,null,130.133,102.331,1.396,65.942,null,1.107,1.022,1.168,null,1.992,1.057,1.886,null,1.866,-9.262,14.199,4.987,null,null],[5255.5,null,null,2056.369,902.46,null,130.375,103.117,1.387,64.84,null,1.108,1.025,1.159,null,1.968,1.02,1.929,null,1.888,-6.6,13.941,5.071,null,null],[5256.0,null,null,2056.118,898.052,null,130.423,102.771,1.403,64.612,null,1.114,1.029,1.16,null,1.923,1.004,1.915,null,1.836,-7.158,13.774,5.091,null,null],[5256.5,null,null,2042.406,888.566,null,130.242,101.227,1.423,64.655,null,1.125,1.04,1.148,null,1.866,0.981,1.902,null,1.729,-13.801,13.554,5.129,null,null],[5257.0,null,null,2030.156,874.078,null,129.953,99.82,1.426,64.156,null,1.144,1.059,1.147,null,1.816,0.938,1.937,null,1.621,-16.62,13.227,5.234,null,null],[5257.5,null,null,2030.084,859.263,null,129.579,99.565,1.395,62.352,null,1.164,1.081,1.193,null,1.795,0.88,2.045,null,1.569,-10.657,12.788,5.4,null,null],[5258.0,null,null,2032.991,846.674,null,129.024,99.6,1.347,59.715,null,1.181,1.103,1.252,null,1.789,0.821,2.182,null,1.563,-5.405,12.305,5.565,null,null],[5258.5,null,null,2030.107,833.29,null,128.302,99.024,1.324,57.683,null,1.2,1.124,1.296,null,1.764,0.786,2.244,null,1.546,-5.917,11.924,5.627,null,null],[5259.0,null,null,2024.434,820.411,null,127.594,97.867,1.34,56.669,null,1.219,1.14,1.311,null,1.696,0.778,2.18,null,1.515,-8.644,11.597,5.53,null,null],[5259.5,null,null,2019.529,813.384,null,127.111,97.113,1.372,56.286,null,1.229,1.147,1.32,null,1.628,0.786,2.072,null,1.496,-13.705,11.397,5.383,null,null],[5260.0,null,null,2016.016,812.781,null,126.911,97.234,1.407,56.662,null,1.23,1.141,1.37,null,1.582,0.797,1.984,null,1.467,-20.563,11.363,5.288,null,null],[5260.5,null,null,2013.736,815.33,null,126.832,96.91,1.464,58.451,null,1.226,1.127,1.411,null,1.543,0.822,1.88,null,1.442,-26.084,11.531,5.208,0.026,null],[5261.0,null,null,2012.964,820.94,null,126.718,96.951,1.533,61.369,null,1.218,1.11,1.37,null,1.528,0.867,1.766,null,1.483,-29.889,11.962,5.123,0.037,null],[5261.5,null,null,2016.018,830.675,null,126.64,98.622,1.592,64.873,null,1.204,1.096,1.267,null,1.538,0.924,1.669,null,1.631,-34.22,12.506,5.028,0.05,null],[5262.0,null,null,2020.088,843.359,null,126.799,101.317,1.693,70.837,null,1.186,1.085,1.194,null,1.577,1.066,1.499,null,1.829,-36.017,13.654,4.783,0.056,null],[5262.5,null,null,2015.64,855.364,null,127.243,103.556,1.8,77.381,null,1.169,1.075,1.177,null,1.635,1.246,1.317,null,1.989,-31.608,15.039,4.488,0.051,null],[5263.0,null,null,2004.932,861.315,null,127.806,105.064,1.812,79.593,null,1.161,1.068,1.188,null,1.689,1.319,1.281,null,2.097,-27.178,15.63,4.399,0.038,null],[5263.5,null,null,1997.692,859.202,null,128.291,105.912,1.736,77.589,null,1.164,1.067,1.188,null,1.771,1.294,1.371,null,2.171,-28.504,15.675,4.497,0.025,null],[5264.0,null,null,1994.742,851.89,null,128.654,106.452,1.642,74.445,null,1.174,1.077,1.181,null,1.857,1.239,1.502,null,2.138,-28.589,15.558,4.664,0.019,null],[5264.5,null,null,1995.427,843.348,null,129.012,106.505,1.572,72.086,null,1.186,1.097,1.194,null,1.916,1.185,1.619,null,1.958,-26.823,15.369,4.816,0.026,null],[5265.0,null,null,1997.354,835.55,null,129.463,105.831,1.526,70.685,2.947,1.197,1.118,1.26,null,1.944,1.138,1.71,null,1.679,-30.794,15.154,4.944,0.037,2.28],[5265.5,null,null,1995.115,828.781,null,129.873,105.552,1.516,70.477,2.996,1.207,1.13,1.351,null,1.937,1.114,1.74,null,1.375,-37.4,14.996,4.998,0.041,2.274],[5266.0,null,null,1990.569,821.44,null,129.962,106.093,1.546,71.262,3.02,1.217,1.135,1.39,null,1.897,1.126,1.686,null,1.157,-39.064,14.968,4.936,0.038,2.268],[5266.5,null,null,1991.056,813.368,null,129.623,105.83,1.614,72.141,3.004,1.229,1.14,1.362,null,1.819,1.157,1.574,null,1.096,-36.502,14.978,4.807,0.032,2.269],[5267.0,null,null,1998.578,807.97,null,129.007,105.291,1.698,73.066,3.027,1.238,1.147,1.304,null,1.723,1.188,1.451,null,1.121,-33.204,14.929,4.664,0.028,2.274],[5267.5,null,null,2007.47,807.077,null,128.355,105.934,1.775,74.663,3.034,1.239,1.151,1.282,null,1.647,1.237,1.334,null,1.164,-33.48,14.927,4.483,0.026,2.277],[5268.0,null,null,2005.335,808.277,null,127.805,106.335,1.845,76.801,2.947,1.237,1.152,1.28,null,1.606,1.337,1.207,null,1.265,-37.544,15.129,4.211,0.03,2.279],[5268.5,null,null,1993.04,808.016,null,127.321,105.312,1.887,78.306,2.843,1.238,1.155,1.28,null,1.603,1.463,1.098,null,1.39,-39.655,15.447,3.926,0.034,2.283],[5269.0,null,null,1984.275,805.956,null,126.723,104.894,1.879,78.749,2.82,1.241,1.161,1.287,null,1.643,1.569,1.048,null,1.408,-37.747,15.762,3.731,0.036,2.287],[5269.5,null,null,1980.268,800.661,null,125.842,105.62,1.831,78.736,2.911,1.249,1.172,1.3,null,1.726,1.657,1.041,null,1.295,-34.177,16.119,3.611,0.034,2.285],[5270.0,null,null,1974.609,789.45,null,124.624,106.084,1.746,78.064,3.044,1.267,1.189,1.324,null,1.841,1.679,1.097,null,1.188,-27.932,16.393,3.624,0.033,2.278],[5270.5,null,null,1972.789,773.947,null,123.155,105.642,1.616,76.128,3.095,1.292,1.21,1.345,null,1.989,1.568,1.281,null,1.162,-21.58,16.365,3.893,0.033,2.275],[5271.0,null,null,1979.913,759.961,null,121.807,104.203,1.48,73.718,3.038,1.316,1.239,1.365,null,2.136,1.392,1.546,null,1.132,-21.564,16.106,4.312,0.034,2.278],[5271.5,null,null,1989.659,750.649,null,121.123,101.69,1.382,71.85,2.918,1.332,1.267,1.381,null,2.234,1.249,1.798,null,1.071,-24.519,15.756,4.696,0.037,2.284],[5272.0,null,null,1994.407,746.095,null,121.466,99.429,1.319,70.195,2.837,1.34,1.279,1.437,null,2.282,1.138,2.011,null,1.039,-22.133,15.364,5.018,0.041,2.292],[5272.5,null,null,1993.861,745.668,null,122.804,98.87,1.291,68.262,2.924,1.341,1.264,1.573,null,2.279,1.068,2.135,null,1.033,-16.673,15.03,5.224,0.044,2.304],[5273.0,null,null,1993.213,747.583,null,124.772,99.424,1.32,65.522,3.205,1.338,1.233,1.696,null,2.16,1.015,2.126,null,0.934,-15.73,14.537,5.314,0.044,2.321],[5273.5,null,null,1995.01,749.554,null,126.93,100.094,1.367,63.24,3.477,1.334,1.208,1.613,null,1.999,0.964,2.073,null,0.694,-19.02,13.942,5.368,0.042,2.334],[5274.0,null,null,1999.422,751.303,null,128.987,102.257,1.46,66.268,3.476,1.331,1.198,1.371,null,1.927,1.031,1.9,null,0.452,-23.396,14.359,5.205,0.037,2.326],[5274.5,null,null,2005.692,754.524,null,130.764,106.459,1.662,75.68,3.259,1.325,1.196,1.234,null,1.898,1.291,1.514,null,0.334,-27.838,16.091,4.687,0.036,2.298],[5275.0,null,null,2007.46,761.373,null,131.92,110.226,1.799,83.488,3.127,1.313,1.196,1.25,null,1.897,1.521,1.25,null,0.332,-27.948,17.424,4.258,0.043,2.272],[5275.5,null,null,2006.085,770.356,null,132.034,111.291,1.823,85.989,3.177,1.298,1.2,1.298,null,1.905,1.599,1.191,null,0.4,-20.208,17.725,4.113,0.047,2.26],[5276.0,null,null,2008.061,777.157,null,130.956,110.064,1.847,86.074,3.299,1.287,1.209,1.336,null,1.88,1.634,1.151,null,0.525,-8.752,17.732,4.029,0.044,2.262],[5276.5,null,null,2010.002,778.604,null,128.944,107.62,1.829,82.991,3.406,1.284,1.218,1.36,null,1.816,1.526,1.197,null,0.648,-1.093,16.972,4.144,0.043,2.276],[5277.0,null,null,2006.179,775.659,null,126.59,105.483,1.765,78.235,3.493,1.289,1.224,1.383,null,1.756,1.339,1.318,null,0.713,-0.649,15.84,4.402,0.043,2.294],[5277.5,null,null,2000.307,770.134,null,124.561,104.134,1.706,74.389,3.473,1.298,1.229,1.41,null,1.733,1.222,1.422,null,0.69,-6.573,15.096,4.59,0.027,2.306],[5278.0,null,null,1995.099,764.552,null,123.295,102.576,1.59,69.875,3.319,1.308,1.233,1.426,null,1.779,1.112,1.61,null,0.611,-13.349,14.401,4.817,0.008,2.31],[5278.5,null,null,1991.161,763.046,null,122.838,100.813,1.432,65.878,3.15,1.311,1.233,1.432,null,1.922,1.038,1.856,null,0.51,-14.007,14.011,5.014,0.013,2.309],[5279.0,null,null,1993.271,765.69,null,122.973,100.423,1.325,64.116,3.136,1.306,1.225,1.437,null,2.075,1.015,2.045,null,0.428,-8.339,14.018,5.124,0.028,2.304],[5279.5,null,null,2002.662,767.64,null,123.347,101.768,1.274,64.005,3.203,1.303,1.216,1.438,null,2.174,1.016,2.139,null,0.454,-1.518,14.139,5.164,0.031,2.297],[5280.0,0.164,null,2013.459,767.286,null,123.529,102.797,1.263,64.85,3.223,1.303,1.212,1.405,1.123,2.218,1.028,2.157,null,0.613,0.796,14.3,5.161,0.029,2.291],[5280.5,0.17,null,2017.056,765.915,null,123.226,102.945,1.279,66.441,3.18,1.306,1.213,1.36,1.094,2.232,1.054,2.118,null,0.808,-1.478,14.573,5.131,0.031,2.289],[5281.0,0.185,null,2009.205,763.983,null,122.448,103.4,1.328,68.777,3.102,1.309,1.216,1.345,1.057,2.22,1.126,1.978,null,0.913,-3.81,15.049,4.967,0.037,2.291],[5281.5,0.207,null,1996.806,763.271,null,121.532,103.861,1.397,71.082,2.972,1.31,1.221,1.354,1.068,2.185,1.227,1.787,null,0.916,-3.153,15.582,4.721,0.042,2.296],[5282.0,0.187,null,1991.139,765.026,null,120.952,102.631,1.456,73.592,2.895,1.307,1.225,1.321,1.077,2.164,1.335,1.625,null,0.906,-2.357,16.089,4.479,0.043,2.301],[5282.5,0.173,null,1994.895,766.877,null,121.002,99.767,1.498,76.658,2.951,1.304,1.219,1.361,1.073,2.176,1.464,1.491,null,0.859,-5.189,16.639,4.227,0.043,2.303],[5283.0,0.176,null,2000.897,767.676,null,121.607,96.893,1.527,78.0,3.065,1.303,1.205,1.543,1.018,2.174,1.554,1.4,null,0.663,-7.53,16.95,4.051,0.047,2.306],[5283.5,0.174,null,2003.128,769.213,null,122.543,96.345,1.561,75.702,3.158,1.3,1.192,1.618,1.112,2.067,1.512,1.367,null,0.454,-1.622,16.464,4.044,0.046,2.311],[5284.0,0.164,null,2004.449,769.122,null,123.745,99.482,1.605,72.481,3.185,1.3,1.189,1.437,1.109,1.914,1.414,1.353,null,0.377,6.143,15.674,4.115,0.037,2.315],[5284.5,0.154,9.383,2009.633,764.566,46.373,125.084,103.564,1.663,71.289,3.097,1.308,1.195,1.245,0.955,1.774,1.321,1.343,null,0.28,1.654,15.033,4.231,0.034,2.312],[5285.0,0.122,9.08,2016.381,758.248,47.508,126.153,105.078,1.762,71.553,2.941,1.319,1.207,1.256,0.983,1.577,1.182,1.333,null,0.036,-12.59,14.133,4.448,0.041,2.303],[5285.5,0.13,8.911,2016.799,751.127,48.26,126.426,104.08,1.834,72.053,2.832,1.331,1.226,1.315,1.006,1.433,1.072,1.337,null,-0.172,-22.583,13.419,4.648,0.049,2.292],[5286.0,0.107,8.91,2007.723,738.32,48.54,125.482,102.504,1.83,71.904,2.839,1.354,1.257,1.314,1.046,1.417,1.039,1.364,null,-0.218,-23.736,13.241,4.73,0.048,2.285],[5286.5,0.128,8.872,1999.627,718.695,48.698,123.24,101.274,1.791,70.981,2.937,1.391,1.298,1.321,1.107,1.457,1.05,1.388,null,-0.141,-20.623,13.328,4.71,0.044,2.28],[5287.0,0.145,8.854,2002.374,696.773,48.65,120.141,99.71,1.765,70.105,2.995,1.435,1.343,1.38,1.112,1.489,1.084,1.374,null,-0.017,-17.228,13.423,4.595,0.042,2.282],[5287.5,0.169,9.028,2014.456,676.36,48.115,117.003,98.321,1.766,69.598,2.944,1.479,1.394,1.474,1.147,1.483,1.119,1.326,null,0.058,-14.617,13.367,4.435,0.044,2.292],[5288.0,0.174,9.416,2021.134,662.025,47.275,114.641,98.075,1.776,69.306,2.853,1.511,1.44,1.564,1.209,1.464,1.156,1.267,null,-0.019,-13.197,13.276,4.264,0.046,2.307],[5288.5,0.176,9.598,2013.008,656.968,46.898,113.622,97.586,1.797,69.206,2.79,1.522,1.454,1.707,1.245,1.442,1.217,1.187,null,-0.255,-12.734,13.231,4.039,0.046,2.318],[5289.0,0.186,9.582,1997.861,657.958,46.683,114.066,96.449,1.802,68.898,2.798,1.52,1.425,1.954,1.33,1.445,1.286,1.124,null,-0.52,-13.461,13.3,3.84,0.051,2.327],[5289.5,0.181,9.564,1992.775,660.44,46.044,115.638,96.374,1.785,68.753,2.857,1.514,1.381,1.995,1.558,1.482,1.338,1.107,null,-0.673,-14.961,13.507,3.746,0.058,2.337],[5290.0,0.18,9.493,2000.983,664.385,45.616,117.976,97.308,1.737,69.578,2.883,1.505,1.344,1.687,1.53,1.572,1.376,1.142,0.065,-0.67,-17.005,13.924,3.754,0.059,2.344],[5290.5,0.158,9.369,2006.929,671.905,46.259,120.851,98.806,1.628,71.12,2.887,1.488,1.317,1.372,1.213,1.787,1.403,1.274,0.067,-0.63,-19.228,14.766,3.906,0.057,2.338],[5291.0,0.15,9.093,2000.156,684.566,47.722,123.92,101.971,1.492,72.314,2.91,1.461,1.297,1.266,1.081,2.076,1.397,1.488,0.076,-0.697,-21.753,15.747,4.185,0.054,2.317],[5291.5,0.133,8.845,1988.701,700.271,48.798,126.628,105.231,1.427,73.127,2.935,1.428,1.282,1.321,1.054,2.242,1.385,1.619,0.086,-0.885,-25.735,16.328,4.375,0.047,2.292],[5292.0,0.092,8.823,1986.253,712.549,49.077,128.357,106.565,1.433,73.891,2.972,1.403,1.274,1.365,1.094,2.262,1.387,1.631,0.096,-1.088,-28.6,16.555,4.43,0.035,2.271],[5292.5,0.081,8.816,1995.21,716.513,49.232,128.78,106.9,1.485,74.779,3.005,1.396,1.276,1.357,1.084,2.188,1.374,1.592,0.102,-1.2,-25.008,16.562,4.473,0.027,2.259],[5293.0,0.067,8.81,2008.039,712.682,49.52,128.089,107.603,1.589,75.646,2.978,1.403,1.287,1.365,1.052,2.005,1.333,1.503,0.104,-1.183,-15.629,16.216,4.517,0.025,2.255],[5293.5,0.072,8.841,2014.593,703.754,49.793,126.881,108.14,1.706,75.783,2.945,1.421,1.305,1.379,1.068,1.799,1.273,1.411,0.104,-0.996,-10.275,15.626,4.555,0.029,2.259],[5294.0,0.079,8.942,2011.13,692.622,49.865,125.679,107.837,1.808,75.322,2.94,1.444,1.328,1.403,1.09,1.614,1.197,1.347,0.102,-0.671,-14.057,14.875,4.612,0.035,2.265],[5294.5,0.075,9.032,2003.807,682.19,49.844,124.652,106.402,1.888,74.635,2.974,1.466,1.351,1.441,1.112,1.466,1.122,1.306,0.1,-0.355,-19.9,14.122,4.672,0.038,2.27],[5295.0,0.084,9.154,2001.271,674.49,49.647,123.691,104.19,1.941,73.028,3.069,1.483,1.37,1.5,1.181,1.348,1.061,1.27,0.1,-0.129,-21.897,13.349,4.67,0.039,2.273],[5295.5,0.084,9.3,2005.379,670.739,49.226,122.567,102.033,1.986,71.874,3.147,1.491,1.384,1.568,1.214,1.271,1.081,1.18,0.101,0.1,-20.634,12.883,4.435,0.04,2.273],[5296.0,0.089,9.416,2007.916,670.898,49.201,121.242,99.695,2.016,72.393,3.111,1.491,1.392,1.591,1.22,1.275,1.194,1.071,0.1,0.375,-19.975,13.124,4.091,0.041,2.274],[5296.5,0.084,9.454,1998.421,673.874,49.33,119.954,97.373,2.021,73.546,3.04,1.484,1.39,1.565,1.26,1.318,1.31,1.008,0.097,0.611,-22.048,13.598,3.86,0.041,2.279],[5297.0,0.101,9.478,1983.47,678.535,49.142,118.982,96.498,2.025,75.308,2.965,1.474,1.375,1.537,1.266,1.36,1.432,0.952,0.088,0.772,-27.002,14.068,3.652,0.045,2.293],[5297.5,0.105,9.503,1976.333,684.985,49.085,118.285,97.193,2.022,76.752,2.78,1.46,1.356,1.511,1.168,1.398,1.514,0.924,0.077,0.906,-29.461,14.436,3.539,0.05,2.313],[5298.0,0.119,9.517,1977.419,693.504,49.047,117.55,98.634,1.968,76.206,2.567,1.442,1.339,1.455,1.137,1.429,1.45,0.993,0.068,1.068,-26.123,14.357,3.691,0.043,2.332],[5298.5,0.118,9.536,1978.851,704.209,48.328,116.636,100.321,1.864,73.237,2.572,1.42,1.326,1.399,1.148,1.439,1.245,1.173,0.062,1.22,-22.365,13.7,4.125,0.03,2.341],[5299.0,0.119,9.514,1979.749,715.738,47.806,115.802,100.681,1.757,69.126,2.77,1.397,1.312,1.394,1.131,1.429,1.039,1.388,0.061,1.306,-21.207,12.826,4.607,0.028,2.342],[5299.5,0.119,9.511,1980.208,728.158,47.975,115.481,99.605,1.64,64.176,2.957,1.373,1.293,1.435,1.134,1.432,0.891,1.622,0.062,1.307,-16.104,11.989,5.015,0.037,2.338],[5300.0,0.119,9.733,1979.695,742.065,47.696,115.789,99.465,1.523,59.628,3.07,1.348,1.273,1.444,1.154,1.468,0.802,1.835,0.063,1.283,-5.962,11.412,5.283,0.044,2.333],[5300.5,0.124,9.989,1981.312,756.602,46.832,116.425,100.149,1.474,58.119,3.108,1.322,1.254,1.407,1.119,1.513,0.798,1.897,0.062,1.327,0.598,11.39,5.3,0.047,2.332],[5301.0,0.123,10.109,1985.933,769.754,46.062,117.074,99.444,1.484,60.133,3.045,1.299,1.236,1.377,1.085,1.579,0.879,1.805,0.06,1.411,-1.05,11.97,5.072,0.045,2.334],[5301.5,0.124,10.196,1987.107,780.278,45.694,117.72,97.082,1.524,64.307,2.952,1.282,1.217,1.36,1.072,1.665,1.035,1.622,0.059,1.436,-5.528,12.966,4.675,0.042,2.334],[5302.0,0.13,10.215,1983.481,788.135,45.669,118.488,95.471,1.582,67.94,2.902,1.269,1.201,1.337,1.067,1.701,1.182,1.445,0.06,1.41,-11.145,13.748,4.327,0.039,2.331],[5302.5,0.147,10.097,1982.595,793.232,45.933,119.294,96.145,1.654,70.419,2.986,1.261,1.189,1.317,1.029,1.674,1.286,1.305,0.061,1.374,-19.581,14.129,4.084,0.038,2.326],[5303.0,0.145,10.058,1984.501,794.834,46.214,119.808,97.645,1.751,72.539,3.154,1.258,1.179,1.301,1.007,1.607,1.388,1.162,0.062,1.371,-24.566,14.35,3.843,0.038,2.324],[5303.5,0.153,9.996,1984.144,792.049,46.274,119.785,98.629,1.896,74.235,3.234,1.263,1.178,1.287,1.039,1.49,1.493,1.002,0.062,1.456,-19.047,14.374,3.58,0.037,2.326],[5304.0,0.16,9.93,1986.279,786.089,46.276,119.385,99.669,2.056,75.278,3.186,1.272,1.19,1.249,1.04,1.346,1.532,0.878,0.061,1.585,-9.645,14.088,3.413,0.038,2.328],[5304.5,0.16,9.849,1996.959,780.636,46.298,119.029,100.026,2.171,75.878,3.165,1.281,1.21,1.236,0.991,1.226,1.466,0.836,0.063,1.612,-7.869,13.57,3.438,0.039,2.325],[5305.0,0.165,9.869,2010.193,779.166,45.911,118.955,98.456,2.204,75.754,3.174,1.283,1.224,1.328,0.951,1.157,1.338,0.868,0.065,1.511,-10.477,13.024,3.623,0.036,2.32],[5305.5,0.168,9.972,2016.502,780.197,45.269,119.072,96.669,2.156,74.326,3.14,1.282,1.227,1.482,1.009,1.145,1.208,0.951,0.065,1.405,-11.616,12.606,3.88,0.034,2.319],[5306.0,0.17,10.021,2013.763,780.454,44.764,119.18,96.709,2.016,70.821,3.087,1.281,1.223,1.562,1.125,1.198,1.084,1.118,0.063,1.406,-12.862,12.292,4.23,0.035,2.324],[5306.5,0.174,10.016,2006.145,780.391,44.438,119.271,98.071,1.794,65.955,2.968,1.281,1.216,1.504,1.136,1.327,0.966,1.386,0.061,1.514,-14.81,12.082,4.661,0.041,2.33],[5307.0,0.172,9.969,1999.385,782.284,44.469,119.557,99.039,1.607,63.083,2.773,1.278,1.207,1.408,1.089,1.488,0.916,1.624,0.058,1.661,-14.72,12.17,4.929,0.047,2.336],[5307.5,0.181,9.916,1996.143,787.413,44.952,120.163,99.01,1.503,63.164,2.616,1.27,1.195,1.357,1.039,1.644,0.946,1.737,0.058,1.753,-12.165,12.603,4.946,0.051,2.336],[5308.0,0.175,9.908,1993.739,796.548,45.535,120.911,98.907,1.469,64.387,2.576,1.255,1.181,1.334,1.019,1.75,1.009,1.735,0.059,1.786,-8.969,13.12,4.83,0.051,2.331],[5308.5,0.17,9.938,1989.632,808.117,45.904,121.473,99.437,1.501,66.471,2.608,1.237,1.167,1.316,1.018,1.776,1.101,1.622,0.06,1.843,-6.317,13.616,4.604,0.05,2.327],[5309.0,0.164,10.095,1983.793,818.597,45.954,121.665,99.452,1.577,70.065,2.766,1.222,1.155,1.287,1.007,1.764,1.246,1.427,0.059,1.945,-6.744,14.201,4.25,0.046,2.326],[5309.5,0.164,10.131,1981.011,825.746,46.087,121.548,98.622,1.654,73.418,3.031,1.211,1.146,1.276,0.996,1.74,1.398,1.25,0.059,2.052,-7.274,14.692,3.909,0.04,2.327],[5310.0,0.188,9.94,1983.172,831.226,46.518,121.306,97.952,1.719,74.756,3.189,1.203,1.137,1.281,0.999,1.698,1.501,1.133,0.058,2.166,-6.522,14.899,3.687,0.033,2.327],[5310.5,0.209,9.818,1985.951,836.962,46.868,121.042,98.177,1.765,74.512,3.105,1.195,1.13,1.291,0.999,1.649,1.534,1.075,0.057,2.257,-9.737,14.859,3.594,0.034,2.328],[5311.0,0.22,9.819,1985.094,841.434,46.953,120.748,98.662,1.767,73.691,2.937,1.188,1.127,1.311,0.998,1.622,1.482,1.097,0.056,2.266,-17.053,14.639,3.67,0.042,2.332],[5311.5,0.223,9.836,1984.26,841.08,46.834,120.434,98.995,1.728,72.311,2.844,1.189,1.127,1.316,1.0,1.61,1.348,1.2,0.056,2.232,-19.904,14.202,3.924,0.052,2.332],[5312.0,0.207,9.801,1986.81,837.136,46.737,120.2,99.056,1.679,69.849,2.859,1.195,1.129,1.302,1.001,1.588,1.189,1.344,0.059,2.224,-17.152,13.613,4.269,0.058,2.324],[5312.5,0.212,9.646,1987.534,833.179,46.956,120.154,98.951,1.633,66.036,2.881,1.2,1.132,1.286,0.992,1.55,1.034,1.509,0.063,2.248,-12.1,12.926,4.664,0.059,2.314],[5313.0,0.209,9.663,1987.389,830.326,47.262,120.279,98.721,1.592,62.5,2.815,1.204,1.137,1.294,0.98,1.517,0.913,1.665,0.064,2.281,-5.785,12.334,5.022,0.059,2.312],[5313.5,0.209,9.845,1990.594,826.331,47.23,120.453,98.719,1.565,60.923,2.686,1.21,1.144,1.308,0.991,1.494,0.84,1.783,0.062,2.288,0.107,11.943,5.288,0.057,2.318],[5314.0,0.191,10.003,1996.287,820.164,46.84,120.63,99.23,1.562,60.116,2.6,1.219,1.151,1.309,0.994,1.431,0.757,1.898,0.061,2.221,0.62,11.416,5.617,0.054,2.322],[5314.5,0.184,9.798,2002.575,814.111,46.682,120.946,99.394,1.587,60.209,2.641,1.228,1.155,1.307,0.991,1.362,0.704,1.933,0.064,2.089,0.358,11.033,5.813,0.054,2.316],[5315.0,0.154,9.578,2008.96,810.694,46.891,121.55,98.829,1.611,60.915,2.698,1.234,1.158,1.313,1.0,1.348,0.713,1.891,0.07,1.964,1.981,11.088,5.771,0.058,2.303],[5315.5,0.156,9.994,2013.736,810.038,46.794,122.396,98.964,1.617,61.086,2.676,1.235,1.16,1.315,1.004,1.372,0.74,1.854,0.075,1.876,-0.668,11.327,5.679,0.059,2.291],[5316.0,0.124,10.22,2016.236,811.266,46.807,123.287,100.134,1.61,61.205,2.634,1.233,1.156,1.331,0.998,1.41,0.767,1.837,0.078,1.814,-6.608,11.585,5.602,0.052,2.284],[5316.5,0.147,9.87,2017.559,813.409,47.057,124.067,100.66,1.586,61.999,2.614,1.229,1.148,1.354,0.997,1.472,0.797,1.847,0.077,1.782,-9.135,11.911,5.549,0.043,2.286],[5317.0,0.157,9.654,2017.19,814.941,47.026,124.733,100.924,1.539,63.667,2.592,1.227,1.139,1.346,1.023,1.588,0.845,1.878,0.073,1.79,-10.213,12.464,5.475,0.042,2.295],[5317.5,0.143,9.716,2019.416,814.638,46.759,125.356,102.207,1.488,65.22,2.601,1.228,1.135,1.307,1.027,1.714,0.894,1.916,0.069,1.825,-14.806,13.013,5.4,0.044,2.304],[5318.0,0.131,9.487,2022.393,813.021,47.199,125.875,104.19,1.468,66.24,2.607,1.23,1.136,1.264,1.01,1.79,0.941,1.905,0.066,1.848,-18.334,13.418,5.297,0.05,2.31],[5318.5,0.129,8.959,2020.192,811.796,48.143,126.086,104.557,1.498,67.427,2.558,1.232,1.14,1.237,0.996,1.816,1.023,1.784,0.064,1.824,-16.97,13.89,5.054,0.057,2.316],[5319.0,0.102,8.952,2015.989,811.805,48.433,125.853,103.017,1.562,68.744,2.522,1.232,1.148,1.24,1.018,1.794,1.116,1.611,0.065,1.786,-12.022,14.306,4.763,0.058,2.315],[5319.5,0.077,9.232,2017.544,811.657,48.689,125.257,101.678,1.622,69.665,2.589,1.232,1.157,1.267,1.049,1.745,1.157,1.509,0.07,1.793,-8.175,14.449,4.634,0.05,2.302],[5320.0,0.101,9.274,2019.628,810.252,49.468,124.49,102.542,1.679,71.316,2.708,1.234,1.165,1.295,1.086,1.7,1.185,1.436,0.077,1.831,-9.08,14.57,4.565,0.039,2.286],[5320.5,0.117,9.28,2015.23,808.858,50.16,123.72,104.495,1.762,74.423,2.769,1.236,1.172,1.304,1.086,1.659,1.259,1.322,0.08,1.858,-11.988,14.916,4.402,0.034,2.281],[5321.0,0.118,9.403,2008.892,809.534,50.54,123.012,104.722,1.85,77.795,2.785,1.235,1.175,1.309,1.068,1.633,1.368,1.197,0.074,1.865,-12.417,15.42,4.189,0.036,2.292],[5321.5,0.141,9.717,2003.573,812.396,50.639,122.329,103.343,1.92,80.562,2.817,1.231,1.171,1.334,1.057,1.62,1.494,1.089,0.065,1.827,-10.308,15.882,3.955,0.041,2.313],[5322.0,0.13,9.944,1998.174,814.399,50.734,121.672,102.654,1.985,82.959,2.888,1.228,1.165,1.346,1.068,1.604,1.634,0.985,0.058,1.712,-9.673,16.263,3.702,0.044,2.33],[5322.5,0.134,10.15,1996.246,814.273,50.772,121.137,104.153,2.062,84.416,2.925,1.228,1.162,1.331,1.073,1.566,1.769,0.888,0.055,1.591,-13.364,16.481,3.463,0.045,2.337],[5323.0,0.122,10.246,2000.201,813.318,50.84,120.804,105.647,2.129,84.345,2.844,1.23,1.161,1.308,1.053,1.513,1.862,0.814,0.055,1.574,-18.248,16.441,3.281,0.045,2.339],[5323.5,0.13,10.106,2007.656,811.171,51.107,120.696,105.26,2.122,82.376,2.719,1.233,1.159,1.298,1.036,1.476,1.751,0.851,0.055,1.628,-17.542,16.002,3.416,0.047,2.34],[5324.0,0.126,10.103,2019.759,806.498,51.17,120.819,103.998,2.048,79.419,2.719,1.24,1.163,1.302,1.044,1.455,1.477,0.996,0.054,1.665,-11.111,15.217,3.856,0.049,2.343],[5324.5,0.13,10.221,2029.03,800.334,50.926,121.092,104.202,1.97,76.574,2.824,1.249,1.174,1.301,1.053,1.435,1.254,1.153,0.054,1.673,-5.662,14.434,4.296,0.047,2.343],[5325.0,0.116,10.215,2024.058,793.471,50.81,121.413,105.231,1.898,73.589,2.84,1.26,1.194,1.292,1.042,1.428,1.105,1.298,0.055,1.576,-6.851,13.838,4.66,0.046,2.341],[5325.5,0.118,10.157,2013.225,785.431,50.854,121.737,105.016,1.815,69.903,2.781,1.273,1.215,1.338,1.047,1.435,0.988,1.461,0.058,1.361,-13.179,13.298,5.013,0.049,2.337],[5326.0,0.126,10.083,2011.259,776.745,50.814,121.98,103.623,1.726,65.887,2.759,1.287,1.231,1.448,1.038,1.451,0.886,1.644,0.061,1.13,-19.109,12.789,5.367,0.053,2.33],[5326.5,0.121,10.143,2011.809,768.672,50.26,122.028,102.753,1.667,63.705,2.806,1.301,1.24,1.541,1.062,1.47,0.832,1.768,0.065,0.959,-22.427,12.516,5.584,0.052,2.322],[5327.0,0.092,10.189,2003.35,762.032,49.189,121.871,102.752,1.627,63.757,2.843,1.312,1.247,1.542,1.107,1.51,0.821,1.838,0.068,0.875,-25.177,12.539,5.664,0.05,2.317],[5327.5,0.106,9.847,1994.075,757.24,48.642,121.606,103.347,1.621,64.778,2.884,1.321,1.249,1.473,1.098,1.532,0.832,1.841,0.069,0.917,-28.223,12.672,5.651,0.048,2.315],[5328.0,0.137,9.478,1997.24,755.513,49.081,121.307,103.978,1.671,65.739,2.933,1.324,1.244,1.416,1.1,1.49,0.846,1.762,0.069,1.022,-27.185,12.709,5.574,0.045,2.315],[5328.5,0.135,9.479,2008.563,757.77,49.506,120.997,104.001,1.736,66.552,2.957,1.32,1.237,1.378,1.096,1.435,0.861,1.667,0.067,1.043,-21.387,12.706,5.477,0.041,2.319],[5329.0,0.129,9.638,2013.291,762.175,49.272,120.657,104.024,1.801,67.981,2.959,1.312,1.233,1.354,1.071,1.385,0.885,1.566,0.065,0.917,-16.201,12.731,5.339,0.041,2.324],[5329.5,0.136,9.737,2010.598,764.964,48.84,120.373,105.236,1.86,70.077,2.963,1.307,1.233,1.361,1.073,1.354,0.928,1.461,0.063,0.779,-12.583,12.864,5.148,0.045,2.327],[5330.0,0.141,9.739,2014.695,765.095,48.847,120.445,106.654,1.888,72.115,3.01,1.307,1.233,1.389,1.106,1.374,1.006,1.369,0.064,0.81,-8.067,13.242,4.898,0.05,2.324],[5330.5,0.133,9.734,2021.148,764.184,49.323,121.07,106.617,1.863,74.081,3.087,1.309,1.231,1.405,1.109,1.48,1.16,1.282,0.068,0.878,-4.623,14.067,4.528,0.052,2.316],[5331.0,0.12,9.806,2014.22,760.804,50.017,122.05,105.216,1.792,75.778,3.161,1.314,1.231,1.393,1.094,1.662,1.374,1.212,0.072,0.745,-4.587,15.171,4.12,0.047,2.308],[5331.5,0.126,9.987,2002.981,750.927,50.308,122.924,104.152,1.711,76.916,3.133,1.332,1.242,1.38,1.09,1.851,1.582,1.172,0.075,0.477,-1.729,16.152,3.802,0.043,2.303],[5332.0,0.113,10.162,2004.031,735.924,50.1,123.36,105.146,1.641,77.546,3.027,1.359,1.267,1.37,1.08,2.011,1.729,1.163,0.079,0.275,4.165,16.836,3.616,0.047,2.298],[5332.5,0.089,10.156,2010.431,721.418,49.653,123.378,106.898,1.558,77.51,2.956,1.386,1.301,1.374,1.09,2.154,1.754,1.229,0.086,0.152,4.948,17.12,3.623,0.051,2.287],[5333.0,0.092,10.005,2012.333,710.36,49.156,123.215,105.248,1.471,76.64,3.047,1.408,1.337,1.426,1.153,2.253,1.649,1.372,0.094,0.031,-1.185,16.911,3.814,0.049,2.275],[5333.5,0.13,9.915,2015.35,701.263,48.795,122.981,99.98,1.42,74.505,3.252,1.426,1.366,1.54,1.182,2.244,1.485,1.517,0.099,-0.07,-6.945,16.274,4.078,0.046,2.267],[5334.0,0.161,9.927,2019.74,692.88,48.665,122.458,96.099,1.385,70.962,3.397,1.443,1.385,1.662,1.238,2.192,1.336,1.646,0.099,-0.073,-7.542,15.507,4.316,0.047,2.27],[5334.5,0.175,10.041,2016.557,685.614,48.469,121.389,96.525,1.343,67.472,3.386,1.459,1.395,1.717,1.312,2.173,1.234,1.763,0.093,0.046,-7.536,14.905,4.485,0.05,2.282],[5335.0,0.18,10.257,2007.33,680.06,47.756,119.815,98.104,1.318,65.993,3.276,1.47,1.403,1.723,1.315,2.175,1.192,1.824,0.085,0.189,-10.722,14.637,4.556,0.049,2.299],[5335.5,0.164,10.25,2004.508,675.535,46.671,118.125,98.37,1.317,66.496,3.178,1.48,1.406,1.682,1.262,2.183,1.2,1.82,0.078,0.261,-16.337,14.686,4.543,0.047,2.313],[5336.0,0.169,9.957,2012.972,673.526,45.742,116.814,97.99,1.334,67.927,3.121,1.485,1.401,1.618,1.275,2.192,1.24,1.77,0.075,0.283,-21.993,14.93,4.471,0.042,2.321],[5336.5,0.179,9.859,2022.761,677.36,45.179,116.159,99.214,1.393,69.711,3.058,1.476,1.39,1.557,1.203,2.13,1.282,1.663,0.071,0.392,-26.342,15.128,4.38,0.04,2.327],[5337.0,0.174,9.866,2025.396,685.668,45.178,116.108,101.85,1.496,71.759,2.959,1.458,1.375,1.525,1.139,1.991,1.317,1.513,0.066,0.644,-27.235,15.175,4.276,0.042,2.337],[5337.5,0.169,9.856,2019.751,692.864,45.508,116.467,103.612,1.616,73.83,2.857,1.443,1.359,1.502,1.17,1.856,1.38,1.35,0.06,0.889,-24.572,15.284,4.115,0.048,2.35],[5338.0,0.171,9.854,2013.246,697.087,45.746,117.2,103.667,1.74,75.345,2.744,1.435,1.345,1.476,1.22,1.739,1.46,1.194,0.056,0.915,-22.881,15.427,3.925,0.053,2.357],[5338.5,0.175,9.86,2016.451,698.875,45.942,118.447,103.704,1.843,77.215,2.647,1.431,1.334,1.467,1.177,1.665,1.551,1.077,0.058,0.683,-24.005,15.666,3.753,0.054,2.353],[5339.0,0.147,9.825,2024.125,699.277,46.27,120.244,103.875,1.937,80.995,2.621,1.43,1.326,1.495,1.113,1.642,1.693,0.974,0.063,0.393,-25.026,16.234,3.566,0.053,2.341],[5339.5,0.137,9.602,2020.875,700.073,46.66,122.262,103.732,1.997,83.827,2.728,1.428,1.316,1.544,1.118,1.629,1.762,0.925,0.068,0.227,-24.14,16.609,3.499,0.054,2.33],[5340.0,0.145,9.349,2008.243,701.972,46.745,123.915,104.063,1.996,82.867,2.927,1.425,1.307,1.542,1.143,1.604,1.655,0.974,0.069,0.186,-22.844,16.346,3.682,0.056,2.328],[5340.5,0.129,9.297,2000.512,704.076,46.484,124.779,105.073,1.938,78.679,3.024,1.42,1.304,1.493,1.136,1.55,1.376,1.152,0.066,0.214,-19.414,15.358,4.216,0.056,2.332],[5341.0,0.116,9.304,1999.793,707.284,46.571,124.859,105.903,1.853,72.983,2.939,1.414,1.309,1.459,1.099,1.461,1.045,1.429,0.066,0.299,-15.649,13.838,4.997,0.056,2.332],[5341.5,0.088,9.316,2001.885,711.822,47.258,124.435,105.734,1.74,66.651,2.897,1.405,1.313,1.472,1.097,1.393,0.805,1.766,0.069,0.402,-14.066,12.387,5.784,0.052,2.325],[5342.0,0.088,9.307,2006.93,716.361,48.004,123.715,105.066,1.616,60.921,2.929,1.396,1.311,1.519,1.146,1.378,0.66,2.102,0.071,0.465,-7.936,11.376,6.417,0.045,2.318],[5342.5,0.09,9.327,2016.366,720.08,48.365,122.732,104.046,1.544,59.073,2.835,1.389,1.305,1.538,1.196,1.426,0.628,2.271,0.068,0.503,6.015,11.235,6.639,0.044,2.324],[5343.0,0.097,9.6,2025.461,722.404,48.264,121.565,103.307,1.515,61.518,2.702,1.384,1.299,1.501,1.21,1.558,0.698,2.24,0.061,0.503,17.102,12.048,6.429,0.045,2.34],[5343.5,0.104,9.972,2024.779,724.025,48.069,120.527,104.481,1.517,66.048,2.748,1.381,1.298,1.471,1.175,1.73,0.837,2.081,0.056,0.382,17.535,13.415,5.982,0.042,2.353],[5344.0,0.106,10.055,2019.054,726.02,48.101,119.973,107.646,1.54,70.456,2.926,1.377,1.299,1.451,1.137,1.882,1.002,1.89,0.055,0.156,14.248,14.807,5.511,0.041,2.355],[5344.5,0.115,10.05,2017.83,728.29,48.353,119.964,110.131,1.564,74.242,2.989,1.373,1.3,1.439,1.129,2.001,1.158,1.735,0.056,-0.035,12.391,15.981,5.138,0.047,2.351],[5345.0,0.111,10.081,2017.312,728.996,48.791,120.24,111.055,1.552,76.606,2.932,1.372,1.301,1.468,1.11,2.117,1.234,1.715,0.059,-0.201,15.882,16.769,5.043,0.052,2.345],[5345.5,0.121,10.101,2009.999,725.765,49.169,120.553,111.776,1.521,77.075,2.987,1.378,1.303,1.495,1.125,2.196,1.223,1.797,0.062,-0.408,25.339,17.057,5.178,0.051,2.338],[5346.0,0.124,10.105,2003.839,720.463,49.418,120.944,112.505,1.513,76.727,3.138,1.388,1.308,1.49,1.182,2.202,1.184,1.861,0.064,-0.557,33.808,17.011,5.334,0.046,2.333],[5346.5,0.104,10.079,2006.676,718.787,49.845,121.61,112.94,1.537,76.132,3.127,1.391,1.314,1.464,1.146,2.116,1.111,1.906,0.067,-0.577,34.172,16.597,5.552,0.044,2.329],[5347.0,0.105,9.9,2014.92,720.511,50.582,122.506,112.477,1.575,75.563,2.959,1.388,1.315,1.466,1.103,1.989,1.027,1.938,0.068,-0.598,32.146,15.99,5.785,0.043,2.325],[5347.5,0.117,9.913,2018.426,721.311,50.96,123.247,110.935,1.595,74.757,2.847,1.386,1.309,1.521,1.138,1.913,0.983,1.945,0.069,-0.736,30.856,15.579,5.879,0.043,2.322],[5348.0,0.106,10.044,2021.091,721.055,50.782,123.492,108.325,1.614,73.683,2.829,1.387,1.3,1.553,1.192,1.861,0.986,1.889,0.071,-0.906,26.192,15.336,5.774,0.043,2.317],[5348.5,0.098,10.048,2031.008,721.132,50.414,123.307,105.375,1.663,73.635,2.933,1.387,1.295,1.519,1.181,1.803,1.045,1.732,0.075,-0.943,22.811,15.309,5.45,0.041,2.311],[5349.0,0.096,10.051,2036.272,722.102,49.93,123.041,103.89,1.743,75.225,3.099,1.385,1.298,1.458,1.128,1.748,1.168,1.507,0.076,-0.817,23.441,15.552,4.963,0.041,2.307],[5349.5,0.1,10.069,2026.055,724.329,49.556,122.926,104.396,1.824,76.685,3.11,1.381,1.302,1.425,1.102,1.687,1.285,1.318,0.075,-0.632,25.824,15.706,4.546,0.047,2.31],[5350.0,0.096,10.11,2015.786,727.634,49.604,122.792,105.173,1.91,77.1,2.954,1.374,1.302,1.44,1.101,1.591,1.341,1.187,0.073,-0.405,30.739,15.502,4.29,0.055,2.314],[5350.5,0.089,10.027,2016.672,728.796,49.996,122.28,105.217,1.964,75.96,2.842,1.372,1.301,1.465,1.129,1.477,1.28,1.156,0.071,-0.169,34.634,14.807,4.302,0.056,2.316],[5351.0,0.09,9.942,2021.764,727.182,50.005,121.3,104.989,1.942,73.067,2.788,1.375,1.302,1.483,1.143,1.387,1.109,1.262,0.069,-0.021,28.476,13.764,4.636,0.051,2.322],[5351.5,0.113,9.965,2025.8,724.374,49.208,120.111,103.606,1.873,69.179,2.769,1.381,1.306,1.489,1.176,1.328,0.929,1.443,0.065,0.054,16.375,12.707,5.11,0.044,2.331],[5352.0,0.113,9.974,2025.678,721.208,48.323,119.051,100.932,1.788,65.518,2.822,1.387,1.312,1.483,1.161,1.306,0.8,1.639,0.061,0.122,11.333,11.92,5.543,0.04,2.341],[5352.5,0.107,9.966,2021.964,717.202,47.935,118.345,98.525,1.71,63.857,2.949,1.394,1.319,1.477,1.157,1.333,0.745,1.792,0.059,0.131,14.334,11.633,5.799,0.039,2.346],[5353.0,0.101,9.964,2020.967,714.668,47.984,118.122,98.237,1.662,64.336,3.051,1.399,1.323,1.486,1.182,1.391,0.751,1.852,0.059,0.029,16.395,11.795,5.829,0.039,2.348],[5353.5,0.101,9.962,2024.18,716.943,48.093,118.41,100.04,1.662,65.024,3.08,1.395,1.317,1.522,1.171,1.417,0.781,1.815,0.057,-0.105,13.361,12.026,5.715,0.038,2.351],[5354.0,0.101,9.962,2027.518,723.435,47.686,119.171,102.018,1.687,64.681,3.067,1.382,1.302,1.564,1.175,1.402,0.803,1.747,0.056,-0.174,8.66,12.072,5.582,0.037,2.354],[5354.5,0.116,9.965,2032.221,731.205,47.015,120.271,102.262,1.717,63.943,3.081,1.368,1.281,1.567,1.238,1.359,0.802,1.695,0.056,-0.215,5.563,11.913,5.515,0.037,2.352],[5355.0,0.126,9.968,2041.177,738.175,46.766,121.453,101.653,1.761,64.012,3.118,1.355,1.261,1.521,1.204,1.302,0.795,1.637,0.058,-0.378,3.857,11.705,5.462,0.037,2.346],[5355.5,0.128,9.955,2048.371,743.062,47.034,122.478,102.546,1.82,64.9,3.054,1.346,1.249,1.445,1.114,1.255,0.807,1.555,0.061,-0.616,2.06,11.652,5.356,0.044,2.337],[5356.0,0.118,9.937,2046.502,745.329,47.499,123.223,104.729,1.875,65.748,2.887,1.342,1.247,1.379,1.118,1.219,0.823,1.48,0.064,-0.701,2.116,11.665,5.257,0.054,2.33],[5356.5,0.123,9.747,2040.227,747.513,47.919,123.636,105.723,1.928,67.185,2.766,1.338,1.253,1.341,1.085,1.198,0.853,1.406,0.065,-0.588,6.706,11.789,5.131,0.058,2.326],[5357.0,0.149,9.491,2035.501,750.793,48.131,123.648,105.451,1.987,69.192,2.744,1.332,1.261,1.349,1.048,1.186,0.904,1.315,0.063,-0.432,8.111,12.038,4.948,0.057,2.33],[5357.5,0.156,9.442,2027.199,752.222,48.161,123.23,105.098,2.039,70.265,2.777,1.329,1.266,1.396,1.073,1.173,0.945,1.243,0.06,-0.332,0.233,12.216,4.8,0.058,2.338],[5358.0,0.136,9.532,2014.16,749.692,48.136,122.422,104.499,2.055,69.816,2.88,1.334,1.269,1.445,1.102,1.159,0.939,1.235,0.058,-0.265,-9.16,12.166,4.811,0.063,2.342],[5358.5,0.164,9.811,2007.765,743.829,47.8,121.355,103.14,2.045,68.812,3.106,1.344,1.271,1.475,1.115,1.14,0.893,1.278,0.06,-0.158,-12.917,11.909,4.953,0.064,2.339],[5359.0,0.125,9.999,2014.009,737.596,47.404,120.315,101.492,2.047,68.429,3.307,1.356,1.274,1.477,1.121,1.112,0.846,1.315,0.064,0.05,-12.23,11.624,5.099,0.063,2.33],[5359.5,0.123,10.044,2021.08,734.943,47.302,119.618,101.07,2.075,68.73,3.359,1.361,1.279,1.453,1.098,1.071,0.808,1.326,0.068,0.323,-9.241,11.351,5.214,0.06,2.322],[5360.0,0.112,9.985,2017.967,736.948,47.435,119.419,102.394,2.134,69.833,3.336,1.357,1.278,1.449,1.08,1.032,0.803,1.287,0.071,0.586,-7.917,11.238,5.2,0.054,2.316],[5360.5,0.14,9.866,2010.141,739.872,47.433,119.603,102.98,2.238,72.37,3.3,1.352,1.273,1.453,1.071,1.01,0.88,1.158,0.07,0.798,-10.314,11.541,4.895,0.049,2.316],[5361.0,0.16,9.855,2008.115,742.49,47.11,119.88,101.594,2.356,75.781,3.171,1.347,1.268,1.451,1.096,1.009,1.025,0.993,0.067,0.928,-12.726,12.135,4.421,0.052,2.322],[5361.5,0.138,9.912,2006.297,745.225,46.848,119.962,100.76,2.43,78.563,3.033,1.342,1.266,1.433,1.098,1.023,1.169,0.878,0.063,0.971,-11.249,12.691,4.041,0.057,2.331],[5362.0,0.11,10.036,1998.424,747.162,46.666,119.801,101.917,2.437,79.706,3.006,1.338,1.268,1.413,1.099,1.055,1.269,0.832,0.06,0.964,-4.928,13.126,3.841,0.056,2.339],[5362.5,0.101,10.176,1996.984,747.68,46.313,119.635,103.741,2.376,79.209,3.013,1.337,1.272,1.411,1.104,1.106,1.297,0.853,0.057,0.961,-0.594,13.412,3.839,0.05,2.346],[5363.0,0.103,10.207,2012.039,747.689,45.894,119.657,105.782,2.267,77.658,3.032,1.337,1.273,1.414,1.103,1.158,1.23,0.946,0.056,0.945,-6.563,13.392,4.053,0.047,2.348],[5363.5,0.112,10.203,2030.994,746.3,45.711,119.752,107.196,2.149,75.46,3.14,1.34,1.274,1.424,1.11,1.197,1.119,1.074,0.057,0.849,-16.442,13.128,4.365,0.047,2.346],[5364.0,0.112,10.21,2036.07,741.218,45.668,119.65,106.435,2.072,73.476,3.213,1.349,1.279,1.435,1.121,1.227,1.058,1.16,0.058,0.638,-22.356,12.976,4.553,0.045,2.343],[5364.5,0.104,10.227,2030.151,731.787,45.731,119.273,104.109,2.029,72.042,3.132,1.367,1.29,1.451,1.11,1.251,1.036,1.208,0.06,0.353,-27.886,12.961,4.645,0.044,2.342],[5365.0,0.113,10.247,2022.007,719.268,45.954,118.818,102.385,2.006,71.858,3.003,1.39,1.312,1.447,1.132,1.27,1.03,1.233,0.062,0.099,-33.659,13.005,4.688,0.047,2.339],[5365.5,0.126,10.357,2012.917,705.915,45.956,118.527,102.131,1.997,72.874,2.994,1.417,1.345,1.422,1.136,1.285,1.038,1.238,0.065,-0.031,-32.526,13.109,4.687,0.05,2.336],[5366.0,0.124,10.615,2004.083,693.822,45.28,118.411,102.074,2.003,73.805,3.075,1.441,1.381,1.434,1.112,1.291,1.054,1.225,0.068,-0.058,-22.809,13.207,4.651,0.049,2.332],[5366.5,0.128,10.616,2002.27,685.067,44.301,118.229,101.32,2.031,74.041,3.2,1.46,1.402,1.567,1.126,1.282,1.097,1.171,0.07,-0.112,-13.158,13.293,4.503,0.045,2.328],[5367.0,0.13,10.366,2010.473,680.538,43.629,117.831,100.29,2.059,73.869,3.284,1.469,1.405,1.752,1.256,1.278,1.179,1.087,0.071,-0.281,-10.412,13.437,4.241,0.044,2.328],[5367.5,0.139,10.331,2024.041,678.095,43.505,117.404,100.089,2.057,74.014,3.23,1.475,1.401,1.799,1.326,1.313,1.281,1.026,0.071,-0.44,-11.961,13.793,3.999,0.046,2.327],[5368.0,0.129,10.291,2030.13,676.382,43.983,117.27,100.891,2.021,75.404,3.138,1.478,1.396,1.69,1.242,1.399,1.405,0.997,0.074,-0.384,-12.27,14.421,3.818,0.047,2.322],[5368.5,0.129,10.042,2025.003,677.872,44.615,117.49,102.919,1.931,77.704,3.151,1.475,1.385,1.561,1.201,1.559,1.536,1.013,0.078,-0.079,-12.024,15.316,3.703,0.045,2.313],[5369.0,0.15,9.911,2018.271,683.942,44.765,117.817,105.529,1.832,78.863,3.137,1.462,1.368,1.493,1.129,1.701,1.563,1.089,0.079,0.266,-9.897,15.901,3.776,0.042,2.309],[5369.5,0.143,9.918,2012.458,692.338,44.543,117.986,107.674,1.787,78.801,2.984,1.444,1.35,1.483,1.122,1.746,1.5,1.165,0.076,0.484,-6.816,15.933,3.943,0.04,2.315],[5370.0,0.142,9.91,2006.133,700.887,44.285,117.982,108.691,1.763,77.921,2.82,1.427,1.335,1.488,1.195,1.751,1.416,1.238,0.069,0.606,-5.526,15.758,4.135,0.041,2.328],[5370.5,0.137,9.848,2006.599,708.798,44.485,118.11,108.916,1.743,75.843,2.751,1.411,1.325,1.444,1.158,1.727,1.3,1.332,0.063,0.705,-2.447,15.367,4.397,0.047,2.339],[5371.0,0.127,9.782,2018.81,715.312,45.569,118.746,109.535,1.751,74.122,2.803,1.398,1.318,1.401,1.11,1.673,1.204,1.39,0.061,0.735,1.465,14.954,4.613,0.055,2.342],[5371.5,0.129,9.78,2034.978,720.094,46.765,119.869,111.001,1.802,75.311,2.957,1.389,1.31,1.433,1.093,1.635,1.22,1.344,0.062,0.641,-0.859,15.04,4.582,0.056,2.339],[5372.0,0.124,9.794,2044.632,723.124,47.677,121.014,111.949,1.875,78.778,3.111,1.383,1.295,1.501,1.118,1.638,1.355,1.216,0.063,0.475,-6.867,15.691,4.314,0.051,2.336],[5372.5,0.125,9.838,2044.775,722.183,48.626,121.769,110.302,1.952,82.119,3.154,1.385,1.281,1.523,1.184,1.65,1.541,1.078,0.063,0.289,-10.522,16.446,3.978,0.046,2.335],[5373.0,0.123,9.864,2036.901,716.639,49.688,122.208,107.066,2.038,85.332,3.108,1.395,1.278,1.468,1.2,1.642,1.741,0.949,0.065,0.057,-12.482,17.085,3.655,0.043,2.332],[5373.5,0.112,9.834,2028.972,711.112,50.676,122.744,106.068,2.103,88.214,3.065,1.406,1.287,1.4,1.132,1.628,1.887,0.864,0.071,-0.256,-12.501,17.486,3.442,0.044,2.32],[5374.0,0.112,9.782,2029.691,709.502,51.023,123.624,106.628,2.123,88.97,3.019,1.409,1.305,1.378,1.066,1.613,1.89,0.854,0.081,-0.619,-8.507,17.493,3.436,0.049,2.301],[5374.5,0.098,9.745,2031.398,708.166,50.331,124.485,105.332,2.11,87.356,2.914,1.412,1.323,1.401,1.061,1.588,1.762,0.904,0.086,-0.938,-2.572,17.109,3.613,0.053,2.29],[5375.0,0.113,9.776,2023.62,699.655,49.787,124.553,102.483,2.095,84.778,2.821,1.429,1.342,1.449,1.12,1.529,1.57,0.978,0.086,-1.177,2.18,16.356,3.88,0.051,2.292],[5375.5,0.114,9.827,2014.457,681.358,49.334,123.323,100.806,2.084,82.132,2.899,1.468,1.372,1.483,1.166,1.452,1.388,1.049,0.085,-1.368,0.341,15.457,4.142,0.046,2.299],[5376.0,0.11,9.796,2017.407,655.613,49.03,120.871,101.234,2.09,79.385,3.13,1.525,1.427,1.471,1.151,1.364,1.252,1.091,0.087,-1.535,-8.427,14.556,4.319,0.041,2.302],[5376.5,0.113,9.652,2026.386,626.015,49.118,117.755,102.419,2.112,76.803,3.302,1.597,1.515,1.46,1.151,1.278,1.18,1.084,0.092,-1.702,-16.739,13.781,4.336,0.037,2.299],[5377.0,0.106,9.54,2025.218,598.199,49.316,114.773,102.181,2.12,75.344,3.28,1.672,1.624,1.516,1.201,1.233,1.2,1.029,0.1,-1.868,-17.746,13.339,4.133,0.035,2.293],[5377.5,0.106,9.597,2014.575,579.984,49.026,112.644,99.563,2.096,74.236,3.2,1.724,1.716,1.703,1.33,1.231,1.242,0.992,0.104,-1.993,-13.911,13.175,3.938,0.037,2.292],[5378.0,0.125,9.845,2015.869,573.799,47.816,111.713,95.493,1.996,71.977,3.208,1.743,1.74,2.112,1.47,1.293,1.242,1.042,0.1,-2.038,-10.947,13.141,3.928,0.043,2.3],[5378.5,0.14,9.993,2029.939,574.68,45.973,111.921,92.319,1.801,68.138,3.293,1.74,1.692,2.599,1.666,1.435,1.2,1.2,0.089,-2.026,-10.359,13.115,4.059,0.05,2.318],[5379.0,0.13,10.011,2034.091,578.485,44.326,113.023,93.103,1.626,64.243,3.34,1.729,1.609,2.692,2.052,1.571,1.136,1.387,0.078,-1.961,-8.219,13.022,4.259,0.047,2.337],[5379.5,0.118,10.028,2027.016,585.212,43.749,114.87,98.604,1.544,62.831,3.27,1.709,1.528,2.179,2.042,1.636,1.08,1.517,0.074,-1.795,-2.907,12.898,4.435,0.038,2.342],[5380.0,0.096,9.85,2022.267,595.921,44.893,117.517,105.3,1.579,65.695,3.102,1.678,1.466,1.598,1.426,1.665,1.099,1.518,0.082,-1.55,1.484,13.418,4.53,0.04,2.325],[5380.5,0.084,9.545,2019.132,612.622,47.205,120.812,108.995,1.699,71.078,2.967,1.632,1.424,1.359,1.17,1.638,1.17,1.402,0.093,-1.296,-2.176,14.195,4.502,0.047,2.302],[5381.0,0.078,9.479,2020.355,635.368,49.076,124.114,109.634,1.788,74.523,2.912,1.574,1.394,1.334,1.126,1.579,1.194,1.322,0.096,-1.109,-10.841,14.407,4.477,0.043,2.291],[5381.5,0.078,9.397,2033.37,658.322,49.945,126.557,109.319,1.828,75.722,2.915,1.519,1.37,1.372,1.111,1.56,1.213,1.286,0.09,-1.024,-13.204,14.563,4.455,0.031,2.294],[5382.0,0.1,9.331,2047.532,673.695,50.143,127.493,108.471,1.849,75.984,2.897,1.484,1.355,1.435,1.16,1.557,1.234,1.262,0.084,-0.946,-4.053,14.697,4.421,0.027,2.302],[5382.5,0.107,9.455,2047.86,677.759,49.957,126.734,105.947,1.848,75.76,2.868,1.475,1.352,1.49,1.222,1.555,1.231,1.263,0.082,-0.739,9.891,14.675,4.423,0.035,2.306],[5383.0,0.172,9.714,2039.878,669.298,49.507,124.601,102.568,1.788,74.722,2.897,1.494,1.372,1.503,1.254,1.607,1.221,1.316,0.082,-0.436,13.964,14.651,4.452,0.048,2.307],[5383.5,0.092,9.82,2040.754,650.294,49.197,121.873,100.525,1.682,72.589,3.002,1.538,1.419,1.44,1.22,1.697,1.194,1.423,0.086,-0.227,3.223,14.558,4.525,0.051,2.304],[5384.0,0.115,9.647,2051.147,629.365,49.123,119.461,100.292,1.613,70.563,3.119,1.589,1.484,1.376,1.161,1.739,1.157,1.504,0.092,-0.174,-10.774,14.319,4.595,0.047,2.299],[5384.5,0.133,9.524,2059.181,615.528,48.76,117.931,100.017,1.59,68.879,3.258,1.625,1.541,1.466,1.197,1.717,1.113,1.543,0.095,-0.245,-12.528,13.938,4.647,0.049,2.297],[5385.0,0.121,9.661,2056.181,609.413,47.903,117.115,99.256,1.601,67.257,3.406,1.641,1.569,1.773,1.319,1.648,1.074,1.534,0.091,-0.442,-5.227,13.458,4.65,0.056,2.306],[5385.5,0.154,9.879,2046.503,604.792,47.01,116.416,98.932,1.649,65.737,3.418,1.653,1.571,2.107,1.621,1.531,1.031,1.485,0.08,-0.72,-3.294,12.877,4.636,0.052,2.326],[5386.0,0.165,9.973,2039.859,597.703,46.482,115.673,98.935,1.698,64.55,3.259,1.673,1.567,2.101,1.795,1.425,0.98,1.453,0.072,-0.996,-11.13,12.343,4.675,0.04,2.343],[5386.5,0.175,10.025,2041.676,590.694,46.334,115.502,98.359,1.735,64.364,3.114,1.693,1.566,1.827,1.531,1.366,0.957,1.427,0.076,-1.194,-18.466,12.098,4.689,0.035,2.337],[5387.0,0.163,10.04,2043.956,589.616,46.89,116.532,98.215,1.786,65.489,3.084,1.696,1.561,1.641,1.27,1.335,0.982,1.362,0.091,-1.365,-17.242,12.173,4.604,0.04,2.31],[5387.5,0.18,9.871,2038.49,595.931,47.839,118.493,98.763,1.846,67.211,3.105,1.678,1.54,1.635,1.236,1.319,1.029,1.283,0.105,-1.482,-8.886,12.431,4.485,0.042,2.285],[5388.0,0.209,9.698,2030.413,605.617,48.413,120.372,98.739,1.888,68.842,3.102,1.651,1.507,1.673,1.349,1.314,1.068,1.232,0.109,-1.428,-0.653,12.668,4.404,0.044,2.277],[5388.5,0.209,9.685,2025.94,615.082,48.386,121.315,98.803,1.917,71.356,3.061,1.626,1.477,1.642,1.353,1.338,1.121,1.195,0.104,-1.229,1.757,13.098,4.338,0.052,2.282],[5389.0,0.181,9.639,2021.265,622.819,48.085,121.33,100.337,1.968,74.971,3.033,1.606,1.462,1.571,1.289,1.365,1.201,1.138,0.1,-1.058,-1.866,13.718,4.241,0.06,2.287],[5389.5,0.184,9.364,2014.009,627.671,48.076,121.041,102.407,2.041,77.468,3.055,1.593,1.463,1.53,1.223,1.345,1.264,1.065,0.101,-0.92,-5.743,14.01,4.113,0.06,2.284],[5390.0,0.164,9.188,2008.39,630.645,48.439,120.886,103.307,2.092,76.817,3.127,1.586,1.469,1.532,1.158,1.288,1.241,1.039,0.102,-0.715,-4.796,13.75,4.117,0.06,2.281],[5390.5,0.146,9.267,2008.464,634.797,48.86,120.812,103.133,2.107,74.237,3.232,1.575,1.465,1.586,1.159,1.223,1.137,1.077,0.099,-0.467,-0.061,13.158,4.303,0.063,2.286],[5391.0,0.146,9.371,2013.594,640.184,49.051,120.615,102.5,2.118,71.762,3.249,1.562,1.45,1.644,1.232,1.154,1.023,1.13,0.092,-0.255,-1.692,12.486,4.538,0.064,2.297],[5391.5,0.128,9.452,2022.354,643.93,48.836,120.309,102.671,2.138,69.694,3.106,1.553,1.43,1.635,1.244,1.077,0.912,1.185,0.084,-0.146,-13.618,11.76,4.799,0.059,2.31],[5392.0,0.124,9.459,2032.6,645.093,48.257,120.047,104.224,2.113,68.004,2.91,1.55,1.412,1.573,1.196,1.035,0.809,1.284,0.08,-0.148,-25.007,11.164,5.135,0.053,2.316],[5392.5,0.112,9.406,2041.764,644.615,48.022,119.944,106.064,2.046,67.227,2.818,1.551,1.401,1.504,1.172,1.049,0.75,1.4,0.082,-0.183,-28.232,10.961,5.424,0.049,2.313],[5393.0,0.087,9.401,2046.398,643.594,48.405,119.946,107.563,1.98,67.425,2.906,1.554,1.403,1.468,1.153,1.117,0.765,1.459,0.089,-0.183,-27.248,11.285,5.475,0.048,2.301],[5393.5,0.088,9.407,2046.903,643.969,48.859,119.89,107.944,1.899,67.917,3.038,1.553,1.415,1.467,1.165,1.25,0.843,1.483,0.094,-0.158,-25.656,12.095,5.335,0.045,2.291],[5394.0,0.093,9.398,2050.277,645.888,49.213,119.711,106.479,1.817,68.898,3.017,1.548,1.427,1.494,1.19,1.409,0.943,1.494,0.096,-0.1,-23.719,13.053,5.144,0.042,2.288],[5394.5,0.112,9.392,2057.937,648.224,49.529,119.533,104.972,1.763,71.19,2.932,1.543,1.429,1.541,1.196,1.558,1.064,1.467,0.095,0.0,-24.314,14.016,4.904,0.041,2.289],[5395.0,0.118,9.437,2059.612,651.948,50.023,119.647,105.703,1.724,73.648,2.927,1.534,1.417,1.597,1.188,1.697,1.183,1.434,0.093,0.039,-26.173,14.924,4.685,0.047,2.292],[5395.5,0.113,9.55,2048.25,658.103,50.559,120.271,107.574,1.708,75.151,2.947,1.52,1.394,1.635,1.214,1.778,1.261,1.411,0.091,-0.063,-24.05,15.513,4.568,0.051,2.293],[5396.0,0.11,9.599,2033.89,664.807,50.672,121.356,108.776,1.74,76.966,2.919,1.504,1.366,1.614,1.236,1.792,1.324,1.355,0.091,-0.187,-20.719,15.925,4.465,0.045,2.292],[5396.5,0.118,9.582,2028.495,671.637,50.715,122.638,108.737,1.803,79.675,2.89,1.489,1.341,1.545,1.201,1.773,1.396,1.271,0.091,-0.183,-22.654,16.323,4.34,0.038,2.29],[5397.0,0.107,9.493,2030.224,678.707,51.2,123.858,109.247,1.857,82.039,2.901,1.473,1.325,1.471,1.127,1.753,1.439,1.218,0.091,-0.045,-26.095,16.624,4.286,0.041,2.289],[5397.5,0.107,9.345,2031.794,684.412,51.857,124.943,112.283,1.878,83.525,2.968,1.461,1.32,1.421,1.117,1.762,1.45,1.215,0.09,0.163,-25.248,16.897,4.323,0.044,2.288],[5398.0,0.108,9.226,2030.466,687.25,52.035,125.877,116.786,1.862,84.339,3.049,1.455,1.327,1.386,1.111,1.811,1.462,1.239,0.091,0.391,-24.118,17.217,4.37,0.039,2.286],[5398.5,0.101,9.164,2027.197,686.178,51.531,126.424,118.536,1.81,84.472,3.117,1.457,1.344,1.368,1.129,1.902,1.481,1.285,0.093,0.505,-24.311,17.565,4.402,0.036,2.282],[5399.0,0.081,9.119,2023.971,678.405,51.217,126.137,115.913,1.732,83.779,3.145,1.474,1.371,1.402,1.134,2.015,1.479,1.363,0.097,0.407,-19.675,17.797,4.466,0.04,2.277],[5399.5,0.081,9.147,2022.925,663.399,51.262,124.678,111.727,1.637,81.521,3.158,1.507,1.41,1.482,1.169,2.114,1.425,1.489,0.101,0.179,-9.127,17.655,4.604,0.044,2.274],[5400.0,0.101,9.216,2028.531,644.27,49.971,122.181,109.473,1.512,77.211,3.156,1.552,1.462,1.543,1.216,2.197,1.307,1.693,0.105,-0.017,-0.069,16.94,4.821,0.044,2.272],[5400.5,0.101,9.424,2039.746,626.585,48.355,119.311,108.843,1.421,73.537,3.11,1.596,1.526,1.557,1.23,2.243,1.218,1.841,0.109,-0.155,1.502,16.269,4.954,0.043,2.271],[5401.0,0.097,9.75,2051.655,614.639,47.548,116.891,106.628,1.427,72.463,3.045,1.627,1.588,1.629,1.233,2.194,1.23,1.787,0.111,-0.35,-4.063,15.969,4.822,0.048,2.272],[5401.5,0.099,9.855,2056.039,610.457,46.485,115.319,102.608,1.505,73.179,3.057,1.638,1.618,1.86,1.32,2.05,1.332,1.552,0.109,-0.604,-11.279,15.711,4.401,0.054,2.276],[5402.0,0.112,9.864,2052.004,611.91,44.437,114.442,99.968,1.602,74.128,3.155,1.634,1.598,2.145,1.553,1.905,1.46,1.31,0.1,-0.742,-14.358,15.566,3.966,0.053,2.29],[5402.5,0.113,9.878,2047.159,615.146,43.055,113.965,99.856,1.682,74.091,3.246,1.626,1.554,2.192,1.657,1.798,1.525,1.18,0.086,-0.72,-14.619,15.432,3.757,0.046,2.314],[5403.0,0.107,9.869,2042.773,618.491,43.304,113.801,101.41,1.763,74.505,3.201,1.617,1.51,1.937,1.496,1.695,1.561,1.086,0.073,-0.688,-16.69,15.245,3.625,0.041,2.336],[5403.5,0.113,9.862,2030.359,624.286,43.999,114.175,103.4,1.835,74.995,3.054,1.602,1.473,1.668,1.334,1.583,1.47,1.083,0.071,-0.626,-21.454,14.822,3.769,0.037,2.341],[5404.0,0.107,9.854,2015.378,635.005,44.583,115.267,105.456,1.864,75.011,2.893,1.575,1.444,1.524,1.238,1.505,1.277,1.186,0.074,-0.462,-26.128,14.33,4.192,0.033,2.331],[5404.5,0.1,9.77,2012.691,649.029,45.167,116.78,107.931,1.85,75.294,2.765,1.541,1.423,1.461,1.174,1.502,1.189,1.263,0.078,-0.299,-28.153,14.186,4.426,0.032,2.32],[5405.0,0.101,9.669,2018.159,661.466,45.805,118.112,110.027,1.788,74.462,2.726,1.512,1.413,1.435,1.166,1.571,1.185,1.325,0.079,-0.307,-27.582,14.315,4.481,0.036,2.316],[5405.5,0.101,9.756,2019.039,669.419,46.357,118.948,109.252,1.663,72.21,2.8,1.494,1.41,1.458,1.172,1.724,1.189,1.45,0.079,-0.615,-26.425,14.579,4.55,0.041,2.312],[5406.0,0.098,9.893,2014.366,673.752,46.852,119.489,105.739,1.549,70.143,2.887,1.484,1.413,1.533,1.193,1.858,1.172,1.586,0.082,-1.133,-26.985,14.689,4.652,0.044,2.306],[5406.5,0.086,10.058,2009.198,674.865,47.317,120.102,102.771,1.485,68.364,2.983,1.482,1.416,1.604,1.241,1.909,1.111,1.723,0.088,-1.479,-25.786,14.47,4.839,0.048,2.294],[5407.0,0.073,10.168,2010.428,673.73,47.839,120.889,102.609,1.447,66.815,3.166,1.484,1.414,1.64,1.226,1.915,1.038,1.847,0.097,-1.458,-23.38,14.147,5.061,0.053,2.279],[5407.5,0.073,10.142,2023.635,671.946,48.14,121.588,104.869,1.462,66.191,3.352,1.488,1.404,1.652,1.198,1.853,0.978,1.896,0.102,-1.257,-23.46,13.828,5.252,0.053,2.27],[5408.0,0.083,10.142,2041.961,670.491,48.017,121.914,107.39,1.563,67.288,3.37,1.491,1.395,1.654,1.256,1.715,0.946,1.813,0.099,-1.061,-22.863,13.64,5.35,0.049,2.277],[5408.5,0.096,10.07,2052.705,669.5,48.32,121.962,108.706,1.687,69.449,3.197,1.494,1.392,1.598,1.238,1.599,0.963,1.662,0.087,-0.881,-19.581,13.755,5.303,0.047,2.297],[5409.0,0.101,10.037,2055.369,670.02,49.087,122.047,108.245,1.775,71.645,2.942,1.492,1.392,1.513,1.153,1.555,1.013,1.538,0.076,-0.71,-15.766,14.092,5.167,0.048,2.32],[5409.5,0.112,10.035,2058.699,673.891,49.506,122.274,107.264,1.864,74.906,2.716,1.484,1.387,1.493,1.15,1.542,1.106,1.401,0.067,-0.5,-11.635,14.68,4.939,0.049,2.338],[5410.0,0.118,10.173,2061.634,679.634,49.843,122.453,107.526,1.912,78.198,2.655,1.471,1.379,1.523,1.193,1.576,1.209,1.305,0.061,-0.275,-5.562,15.381,4.725,0.05,2.349],[5410.5,0.124,10.357,2062.076,683.207,51.522,122.349,108.581,1.883,79.072,2.741,1.464,1.372,1.564,1.223,1.64,1.244,1.319,0.058,-0.132,-0.432,15.764,4.704,0.048,2.355],[5411.0,0.122,10.391,2063.815,681.868,52.69,121.908,108.277,1.824,77.97,2.79,1.467,1.374,1.563,1.221,1.686,1.2,1.407,0.058,-0.07,2.948,15.701,4.859,0.044,2.356],[5411.5,0.138,10.388,2062.126,676.506,52.41,121.273,107.433,1.777,76.038,2.741,1.478,1.388,1.524,1.197,1.679,1.109,1.517,0.059,-0.042,3.916,15.234,5.106,0.042,2.355],[5412.0,0.139,10.416,2058.639,668.252,52.341,120.656,107.657,1.758,73.258,2.69,1.496,1.414,1.52,1.187,1.61,0.996,1.621,0.062,-0.051,2.794,14.452,5.399,0.044,2.35],[5412.5,0.167,10.421,2064.509,659.15,52.432,120.183,108.001,1.733,69.519,2.665,1.517,1.445,1.581,1.194,1.52,0.87,1.756,0.066,-0.121,3.14,13.458,5.762,0.049,2.343],[5413.0,0.164,10.397,2073.243,653.366,51.993,119.894,107.085,1.69,65.887,2.671,1.531,1.47,1.644,1.231,1.449,0.76,1.912,0.068,-0.298,4.334,12.51,6.119,0.051,2.34],[5413.5,0.157,10.413,2065.877,653.043,51.177,119.855,105.94,1.6,62.001,2.715,1.531,1.48,1.674,1.273,1.435,0.676,2.136,0.069,-0.519,4.507,11.711,6.443,0.05,2.338],[5414.0,0.155,10.409,2051.05,653.568,50.244,120.101,105.781,1.428,57.563,2.752,1.53,1.474,1.74,1.322,1.554,0.621,2.509,0.07,-0.559,6.279,11.247,6.728,0.048,2.336],[5414.5,0.147,10.414,2044.364,651.344,48.899,120.613,105.816,1.282,55.521,2.766,1.535,1.454,1.805,1.347,1.766,0.636,2.774,0.07,-0.387,10.056,11.518,6.725,0.046,2.336],[5415.0,0.153,10.464,2043.816,649.105,48.599,121.331,105.904,1.218,56.255,2.729,1.541,1.433,1.748,1.381,1.966,0.7,2.809,0.07,-0.243,9.423,12.218,6.484,0.042,2.335],[5415.5,0.161,10.423,2043.393,648.89,48.953,122.179,106.124,1.244,59.054,2.619,1.541,1.42,1.597,1.306,2.038,0.782,2.624,0.073,-0.275,4.286,12.923,6.154,0.042,2.331],[5416.0,0.185,10.072,2043.02,648.656,49.322,123.066,106.338,1.38,64.581,2.548,1.542,1.414,1.491,1.162,1.955,0.904,2.193,0.077,-0.421,-1.011,13.722,5.666,0.05,2.322],[5416.5,0.186,9.852,2039.557,648.526,49.654,123.874,106.768,1.551,70.195,2.632,1.542,1.412,1.494,1.177,1.815,1.023,1.785,0.084,-0.598,-6.618,14.328,5.211,0.06,2.309],[5417.0,0.181,9.835,2028.893,650.575,49.799,124.454,106.725,1.674,73.163,2.773,1.537,1.412,1.536,1.218,1.701,1.089,1.564,0.09,-0.761,-12.017,14.529,4.952,0.062,2.297],[5417.5,0.166,9.631,2014.664,654.386,49.742,124.567,105.477,1.743,74.075,2.79,1.528,1.422,1.528,1.184,1.644,1.138,1.447,0.093,-0.934,-13.602,14.633,4.775,0.062,2.291],[5418.0,0.215,9.625,2005.979,655.118,49.276,123.966,103.995,1.782,74.47,2.694,1.526,1.435,1.555,1.2,1.645,1.222,1.349,0.092,-1.173,-12.047,14.967,4.554,0.066,2.292],[5418.5,0.256,9.876,2007.785,650.04,49.18,122.683,103.881,1.797,75.409,2.704,1.538,1.44,1.675,1.212,1.698,1.356,1.256,0.089,-1.44,-8.03,15.581,4.276,0.071,2.299],[5419.0,0.227,10.14,2018.574,642.888,49.989,121.197,106.452,1.786,77.244,2.824,1.555,1.441,1.729,1.313,1.784,1.474,1.21,0.086,-1.572,-1.718,16.255,4.094,0.075,2.306],[5419.5,0.2,10.181,2032.698,637.767,50.401,120.207,110.81,1.73,79.908,2.856,1.568,1.448,1.625,1.299,1.949,1.552,1.255,0.083,-1.492,-2.028,17.187,4.109,0.076,2.313],[5420.0,0.227,9.714,2037.087,634.376,50.081,120.136,112.844,1.695,82.202,2.807,1.576,1.458,1.502,1.159,2.054,1.556,1.321,0.081,-1.364,-12.846,17.772,4.241,0.07,2.318],[5420.5,0.243,9.602,2026.479,632.742,49.679,120.791,112.847,1.698,82.299,2.839,1.58,1.457,1.519,1.233,2.014,1.45,1.394,0.078,-1.346,-22.825,17.464,4.484,0.059,2.324],[5421.0,0.232,10.035,2015.293,633.635,49.205,121.553,113.91,1.683,79.646,2.841,1.578,1.447,1.606,1.353,1.939,1.272,1.535,0.074,-1.363,-29.503,16.663,4.891,0.053,2.332],[5421.5,0.192,10.279,2015.838,633.936,48.875,122.033,114.375,1.651,75.118,2.757,1.577,1.443,1.631,1.337,1.843,1.078,1.723,0.072,-1.238,-43.951,15.544,5.383,0.055,2.336],[5422.0,0.187,10.22,2027.161,631.301,49.115,122.462,112.179,1.598,69.743,2.761,1.584,1.451,1.621,1.271,1.755,0.91,1.945,0.077,-1.003,-70.21,14.337,5.876,0.059,2.326],[5422.5,0.169,9.52,2042.322,629.391,49.178,123.289,108.932,1.5,64.226,2.84,1.589,1.457,1.622,1.232,1.731,0.778,2.244,0.086,-0.845,-95.301,13.251,6.349,0.061,2.309],[5423.0,0.146,9.015,2048.806,632.285,48.043,124.582,106.358,1.416,61.039,2.828,1.582,1.448,1.65,1.205,1.767,0.72,2.455,0.093,-0.831,-96.477,12.766,6.579,0.061,2.297],[5423.5,0.153,9.065,2042.285,638.55,47.739,125.921,105.46,1.392,60.951,2.769,1.566,1.428,1.67,1.197,1.822,0.737,2.471,0.092,-0.849,-69.388,12.947,6.518,0.059,2.296],[5424.0,0.152,9.203,2033.642,644.757,49.05,126.773,106.499,1.422,62.924,2.723,1.551,1.406,1.658,1.251,1.846,0.79,2.344,0.089,-0.784,-39.932,13.397,6.304,0.052,2.3],[5424.5,0.153,9.277,2028.853,651.191,49.855,126.997,107.467,1.532,67.774,2.715,1.536,1.381,1.592,1.295,1.813,0.899,2.046,0.087,-0.586,-23.137,14.159,5.886,0.047,2.302],[5425.0,0.135,9.313,2026.271,658.67,50.699,126.898,108.274,1.733,74.681,2.837,1.518,1.361,1.507,1.236,1.712,1.06,1.643,0.09,-0.348,-11.367,15.079,5.32,0.046,2.296],[5425.5,0.13,9.35,2026.921,664.236,51.463,126.783,109.226,1.895,78.874,3.059,1.505,1.353,1.453,1.177,1.598,1.147,1.394,0.094,-0.247,-9.501,15.453,5.0,0.041,2.287],[5426.0,0.078,9.301,2026.658,663.697,51.606,126.605,110.257,1.957,79.693,3.246,1.507,1.357,1.43,1.122,1.541,1.152,1.337,0.095,-0.284,-20.796,15.39,4.956,0.031,2.284],[5426.5,0.078,9.12,2022.903,658.153,52.462,125.951,111.128,1.985,79.974,3.337,1.519,1.375,1.423,1.102,1.517,1.161,1.307,0.093,-0.308,-27.546,15.379,4.914,0.031,2.29],[5427.0,0.107,9.081,2025.217,649.225,53.608,124.306,111.85,1.967,79.749,3.352,1.54,1.409,1.438,1.105,1.535,1.175,1.306,0.089,-0.259,-23.207,15.42,4.869,0.039,2.3],[5427.5,0.112,9.149,2034.503,634.559,53.719,121.545,111.027,1.916,78.562,3.359,1.576,1.461,1.479,1.14,1.571,1.172,1.341,0.087,-0.198,-19.627,15.37,4.868,0.044,2.306],[5428.0,0.124,9.316,2034.83,615.675,53.432,118.268,109.4,1.839,76.39,3.441,1.624,1.523,1.542,1.204,1.606,1.14,1.41,0.089,-0.221,-18.207,15.075,4.908,0.045,2.308],[5428.5,0.118,9.54,2026.513,600.803,52.409,115.485,107.434,1.765,74.111,3.501,1.664,1.586,1.621,1.291,1.64,1.114,1.473,0.09,-0.39,-13.811,14.782,4.923,0.046,2.31],[5429.0,0.158,9.844,2027.464,594.819,51.043,113.967,103.475,1.705,71.13,3.499,1.681,1.625,1.807,1.383,1.658,1.078,1.54,0.088,-0.623,-12.08,14.439,4.972,0.049,2.315],[5429.5,0.162,10.126,2035.446,594.629,49.937,113.908,98.722,1.606,66.505,3.568,1.682,1.62,2.151,1.428,1.667,0.985,1.703,0.083,-0.783,-17.966,13.673,5.164,0.055,2.324],[5430.0,0.162,10.135,2036.837,595.981,49.595,114.931,97.384,1.556,64.986,3.724,1.678,1.574,2.273,1.764,1.682,0.946,1.786,0.08,-0.822,-24.126,13.365,5.255,0.058,2.329],[5430.5,0.164,9.979,2030.846,596.806,50.398,116.44,101.236,1.638,71.387,3.703,1.676,1.52,1.974,1.797,1.762,1.135,1.585,0.084,-0.761,-21.657,14.762,4.887,0.056,2.322],[5431.0,0.151,9.533,2023.815,597.215,51.453,117.981,105.789,1.738,79.438,3.491,1.674,1.485,1.597,1.353,1.838,1.368,1.35,0.095,-0.726,-13.369,16.315,4.44,0.049,2.302],[5431.5,0.143,9.293,2024.526,599.074,51.547,119.267,107.946,1.79,83.091,3.311,1.669,1.477,1.453,1.251,1.835,1.463,1.255,0.106,-0.82,-8.444,16.764,4.253,0.045,2.283],[5432.0,0.086,9.155,2034.704,602.997,51.046,120.1,108.278,1.82,83.373,3.248,1.658,1.498,1.414,1.225,1.804,1.492,1.21,0.112,-0.951,-10.904,16.769,4.172,0.045,2.274],[5432.5,0.076,8.917,2041.329,607.622,50.34,120.399,107.373,1.81,80.585,3.272,1.646,1.538,1.407,1.198,1.763,1.441,1.227,0.109,-1.034,-18.473,16.293,4.201,0.048,2.276],[5433.0,0.072,8.811,2037.371,611.773,49.367,120.169,104.302,1.702,74.326,3.38,1.635,1.571,1.559,1.202,1.743,1.258,1.405,0.098,-1.165,-21.114,15.142,4.499,0.049,2.293],[5433.5,0.101,9.07,2029.914,612.614,47.815,119.473,100.361,1.546,67.126,3.474,1.632,1.58,1.871,1.362,1.757,1.041,1.71,0.082,-1.353,-15.428,13.872,4.978,0.05,2.321],[5434.0,0.118,9.599,2020.614,606.323,45.884,118.497,97.801,1.437,61.929,3.392,1.649,1.573,2.057,1.697,1.763,0.89,1.996,0.073,-1.472,-12.248,12.935,5.418,0.05,2.339],[5434.5,0.106,9.896,2008.811,594.982,45.14,117.627,97.79,1.369,58.72,3.187,1.681,1.569,1.92,1.615,1.745,0.783,2.237,0.076,-1.448,-19.012,12.2,5.794,0.05,2.337],[5435.0,0.078,9.898,2003.386,586.257,45.797,117.334,99.864,1.344,56.986,3.054,1.706,1.571,1.736,1.371,1.68,0.697,2.414,0.082,-1.335,-26.693,11.517,6.147,0.05,2.328],[5435.5,0.078,9.699,2004.997,585.422,46.492,117.912,102.243,1.393,55.949,3.035,1.708,1.571,1.681,1.296,1.507,0.606,2.488,0.083,-1.209,-29.096,10.683,6.569,0.047,2.326],[5436.0,0.077,9.533,2004.851,592.24,46.761,119.174,103.274,1.488,55.829,3.057,1.689,1.565,1.715,1.284,1.335,0.549,2.432,0.081,-1.119,-28.447,10.12,6.844,0.045,2.328],[5436.5,0.073,9.673,2002.462,603.587,46.623,120.42,103.7,1.597,57.103,3.089,1.657,1.552,1.741,1.305,1.255,0.563,2.234,0.08,-1.055,-25.383,10.22,6.738,0.043,2.328],[5437.0,0.09,9.932,2002.978,613.857,46.524,120.979,105.823,1.695,59.197,3.179,1.629,1.538,1.712,1.297,1.238,0.616,2.016,0.079,-1.016,-18.61,10.72,6.466,0.04,2.327],[5437.5,0.094,10.033,2006.651,618.662,46.822,120.687,108.781,1.75,64.015,3.266,1.616,1.534,1.663,1.275,1.337,0.749,1.81,0.077,-0.903,-9.174,11.953,5.997,0.039,2.33],[5438.0,0.093,10.096,2013.848,618.976,47.179,119.863,109.466,1.767,70.97,3.236,1.616,1.537,1.644,1.275,1.523,0.962,1.6,0.072,-0.59,-5.933,13.747,5.358,0.046,2.339],[5438.5,0.106,10.179,2021.981,618.726,47.296,118.95,108.386,1.753,75.136,3.202,1.616,1.54,1.655,1.256,1.671,1.124,1.487,0.067,-0.149,-11.901,14.955,4.943,0.052,2.35],[5439.0,0.106,10.353,2018.947,619.363,47.623,118.25,108.063,1.724,76.197,3.292,1.615,1.546,1.684,1.294,1.769,1.226,1.445,0.064,0.195,-16.492,15.566,4.72,0.048,2.355],[5439.5,0.107,10.44,2006.884,618.873,48.436,117.917,108.448,1.715,76.248,3.348,1.616,1.553,1.718,1.382,1.807,1.284,1.408,0.065,0.315,-15.203,15.83,4.578,0.041,2.354],[5440.0,0.11,10.421,2001.252,616.665,48.907,117.989,108.32,1.732,75.212,3.251,1.622,1.555,1.732,1.366,1.749,1.237,1.416,0.067,0.241,-17.68,15.468,4.647,0.038,2.35],[5440.5,0.106,10.381,2007.024,615.632,49.05,118.377,107.329,1.76,73.519,3.099,1.624,1.551,1.756,1.317,1.641,1.128,1.457,0.069,0.076,-24.849,14.746,4.86,0.039,2.347],[5441.0,0.153,10.347,2014.774,616.245,49.431,118.858,105.755,1.801,72.412,3.004,1.623,1.548,1.813,1.363,1.539,1.043,1.476,0.069,-0.122,-28.201,14.144,5.034,0.042,2.345],[5441.5,0.178,10.337,2017.543,615.038,49.849,119.193,106.635,1.84,72.335,3.028,1.626,1.547,1.806,1.426,1.468,0.996,1.475,0.071,-0.381,-23.987,13.789,5.14,0.047,2.343],[5442.0,0.203,10.302,2015.234,611.929,50.306,119.352,111.12,1.851,72.308,3.183,1.634,1.547,1.734,1.383,1.435,0.959,1.498,0.074,-0.616,-18.537,13.568,5.254,0.049,2.336],[5442.5,0.209,10.239,2013.271,610.805,50.822,119.529,114.472,1.808,71.122,3.312,1.637,1.545,1.686,1.287,1.46,0.924,1.582,0.079,-0.697,-16.893,13.469,5.412,0.049,2.328],[5443.0,0.218,10.213,2011.683,612.202,51.38,119.955,113.183,1.697,68.793,3.315,1.633,1.541,1.689,1.272,1.568,0.894,1.755,0.08,-0.659,-10.471,13.551,5.625,0.048,2.325],[5443.5,0.227,10.331,2007.971,614.184,51.732,120.655,110.33,1.591,66.918,3.321,1.628,1.533,1.726,1.29,1.691,0.875,1.933,0.08,-0.59,-0.976,13.722,5.818,0.04,2.324],[5444.0,0.22,10.378,2008.274,616.9,51.722,121.52,109.824,1.517,66.044,3.404,1.621,1.514,1.743,1.308,1.782,0.855,2.085,0.08,-0.574,-3.074,13.797,5.987,0.031,2.324],[5444.5,0.211,10.284,2017.409,621.797,51.527,122.503,111.336,1.459,66.794,3.461,1.608,1.49,1.733,1.287,1.897,0.859,2.207,0.08,-0.706,-13.8,14.118,6.097,0.031,2.323],[5445.0,0.239,10.157,2026.934,629.624,51.508,123.687,112.158,1.435,68.577,3.443,1.588,1.465,1.693,1.276,1.99,0.886,2.244,0.08,-0.9,-16.962,14.578,6.102,0.039,2.32],[5445.5,0.209,9.981,2029.856,639.306,51.518,125.187,110.866,1.443,68.527,3.461,1.564,1.438,1.64,1.281,1.964,0.88,2.231,0.084,-0.971,-13.844,14.465,6.1,0.047,2.312],[5446.0,0.209,9.608,2026.614,647.905,51.293,126.946,108.797,1.493,66.679,3.503,1.543,1.405,1.596,1.261,1.811,0.841,2.151,0.09,-0.949,-10.06,13.773,6.078,0.053,2.298],[5446.5,0.208,9.321,2018.758,655.24,50.976,128.69,107.245,1.569,65.157,3.437,1.526,1.372,1.583,1.208,1.634,0.807,2.024,0.096,-0.892,-7.677,13.073,6.01,0.058,2.286],[5447.0,0.122,9.131,2010.331,662.538,51.007,130.003,106.141,1.624,64.839,3.281,1.509,1.346,1.573,1.131,1.535,0.801,1.917,0.098,-0.744,-9.714,12.725,5.896,0.06,2.279],[5447.5,0.086,8.985,2006.723,667.93,51.441,130.452,105.204,1.657,65.858,3.164,1.497,1.336,1.509,1.12,1.511,0.838,1.807,0.098,-0.519,-13.191,12.788,5.673,0.057,2.278],[5448.0,0.078,8.987,2007.302,669.304,51.609,129.789,104.369,1.676,68.057,3.147,1.494,1.343,1.44,1.127,1.554,0.931,1.675,0.098,-0.336,-12.404,13.301,5.319,0.052,2.277],[5448.5,0.071,9.017,2007.843,668.805,51.082,128.098,104.077,1.673,70.548,3.213,1.495,1.356,1.414,1.122,1.647,1.068,1.548,0.098,-0.19,-9.532,14.068,4.91,0.047,2.277],[5449.0,0.087,9.119,2010.081,667.748,50.156,125.821,103.594,1.647,72.49,3.286,1.498,1.373,1.435,1.151,1.768,1.22,1.453,0.096,0.01,-6.648,14.863,4.537,0.047,2.282],[5449.5,0.142,9.401,2019.618,666.172,49.554,123.509,102.005,1.608,73.031,3.283,1.501,1.394,1.533,1.173,1.875,1.343,1.397,0.091,0.282,-0.704,15.393,4.258,0.049,2.292],[5450.0,0.158,9.868,2035.736,665.98,49.537,121.583,100.402,1.582,71.666,3.26,1.502,1.408,1.629,1.192,1.89,1.371,1.379,0.086,0.51,7.222,15.261,4.132,0.047,2.301],[5450.5,0.166,10.197,2048.321,668.662,49.757,120.167,99.287,1.588,69.656,3.263,1.496,1.407,1.665,1.152,1.806,1.297,1.393,0.083,0.619,9.855,14.647,4.194,0.045,2.305],[5451.0,0.206,10.568,2045.69,671.548,49.845,119.055,97.673,1.629,68.452,3.24,1.489,1.397,1.612,1.165,1.671,1.18,1.416,0.081,0.608,5.624,13.885,4.373,0.052,2.309],[5451.5,0.316,11.106,2033.203,669.37,49.991,117.938,96.868,1.681,67.912,3.17,1.494,1.389,1.526,1.172,1.531,1.076,1.423,0.079,0.436,0.208,13.13,4.532,0.056,2.313],[5452.0,0.356,11.346,2024.697,659.73,49.437,116.577,97.005,1.708,66.996,3.113,1.516,1.397,1.457,1.151,1.438,1.002,1.435,0.081,0.122,-5.481,12.54,4.646,0.052,2.313],[5452.5,0.369,11.224,2024.819,645.545,50.314,114.91,96.214,1.701,64.88,3.11,1.549,1.432,1.393,1.159,1.374,0.927,1.485,0.085,-0.237,-15.533,11.94,4.786,0.049,2.307],[5453.0,0.361,11.1,2029.771,630.352,51.293,113.123,95.29,1.657,61.981,3.248,1.586,1.499,1.391,1.162,1.338,0.86,1.558,0.091,-0.618,-25.295,11.326,4.89,0.053,2.3],[5453.5,0.358,10.979,2042.828,616.024,50.49,111.638,95.181,1.568,58.758,3.536,1.623,1.587,1.492,1.227,1.353,0.818,1.655,0.094,-1.022,-24.841,10.822,4.907,0.059,2.299],[5454.0,0.342,10.757,2054.228,605.079,49.263,110.891,93.009,1.455,55.298,3.843,1.653,1.668,1.764,1.452,1.404,0.788,1.784,0.086,-1.32,-16.935,10.418,4.907,0.06,2.316],[5454.5,0.324,10.592,2041.627,597.577,47.901,111.086,87.744,1.364,52.18,4.044,1.673,1.713,2.232,1.732,1.461,0.762,1.919,0.07,-1.47,-14.587,10.17,4.954,0.061,2.349],[5455.0,0.268,10.545,2007.98,589.372,46.508,112.225,83.315,1.318,50.778,4.069,1.697,1.707,2.652,2.165,1.498,0.748,2.001,0.055,-1.604,-20.14,10.078,4.997,0.065,2.381],[5455.5,0.227,10.551,1986.267,579.352,45.706,114.157,84.08,1.321,52.178,3.997,1.726,1.665,2.717,1.951,1.512,0.747,2.023,0.053,-1.897,-24.062,10.2,5.065,0.061,2.389],[5456.0,0.208,10.312,1988.062,571.23,46.364,116.723,90.247,1.387,55.898,3.864,1.751,1.62,2.306,1.448,1.553,0.755,2.058,0.064,-2.307,-24.96,11.012,5.412,0.049,2.367],[5456.5,0.18,9.734,1992.2,568.397,47.938,119.746,98.276,1.461,59.872,3.673,1.759,1.594,1.753,1.324,1.595,0.761,2.097,0.082,-2.549,-31.605,11.904,5.808,0.039,2.332],[5457.0,0.116,9.361,1992.666,572.258,48.926,122.761,104.633,1.522,62.994,3.477,1.747,1.582,1.525,1.192,1.577,0.775,2.036,0.097,-2.516,-40.46,12.251,5.867,0.035,2.305],[5457.5,0.101,9.35,1994.17,582.042,49.254,125.056,109.675,1.616,65.653,3.314,1.718,1.573,1.524,1.216,1.522,0.819,1.864,0.102,-2.384,-36.758,12.546,5.692,0.036,2.295],[5458.0,0.129,9.395,1991.16,593.386,49.47,126.052,112.876,1.723,67.997,3.09,1.685,1.571,1.576,1.288,1.463,0.875,1.676,0.097,-2.24,-19.012,12.861,5.46,0.041,2.3],[5458.5,0.121,9.45,1983.056,601.304,49.702,125.664,112.163,1.812,70.287,2.835,1.663,1.578,1.654,1.351,1.42,0.935,1.523,0.09,-2.036,-4.973,13.139,5.224,0.047,2.31],[5459.0,0.126,9.563,1980.0,603.921,49.949,124.33,108.025,1.866,72.665,2.744,1.656,1.582,1.753,1.46,1.42,1.015,1.403,0.086,-1.807,-3.842,13.529,4.957,0.049,2.317],[5459.5,0.147,9.995,1981.126,602.348,49.833,122.738,104.687,1.853,73.958,2.858,1.66,1.572,1.836,1.53,1.478,1.087,1.36,0.086,-1.603,-9.092,13.973,4.773,0.049,2.317],[5460.0,0.134,10.313,1987.209,599.267,49.524,121.408,104.502,1.785,73.116,3.002,1.669,1.559,1.861,1.519,1.548,1.09,1.422,0.087,-1.433,-16.481,14.106,4.804,0.046,2.316],[5460.5,0.126,10.339,1999.417,596.229,49.37,120.55,105.143,1.709,71.076,3.019,1.677,1.55,1.784,1.473,1.598,1.041,1.538,0.086,-1.298,-26.032,13.94,4.973,0.041,2.318],[5461.0,0.135,10.084,2004.61,593.536,49.433,120.278,104.165,1.654,69.423,2.898,1.685,1.544,1.676,1.378,1.621,0.989,1.64,0.084,-1.2,-33.143,13.691,5.139,0.04,2.323],[5461.5,0.108,9.666,1995.63,592.72,49.61,120.663,103.22,1.643,68.361,2.762,1.687,1.541,1.627,1.342,1.583,0.932,1.699,0.082,-1.113,-31.779,13.283,5.291,0.043,2.327],[5462.0,0.122,9.675,1981.875,596.446,49.591,121.616,104.47,1.664,67.366,2.714,1.677,1.539,1.625,1.326,1.514,0.882,1.717,0.079,-1.001,-21.907,12.856,5.412,0.047,2.332],[5462.5,0.111,9.922,1972.567,603.166,49.477,122.838,105.644,1.687,66.765,2.747,1.658,1.535,1.654,1.319,1.486,0.878,1.694,0.075,-0.895,-11.603,12.804,5.416,0.046,2.338],[5463.0,0.124,10.066,1969.117,607.727,49.336,123.952,106.085,1.706,67.747,2.705,1.645,1.524,1.689,1.319,1.518,0.931,1.633,0.072,-0.87,-11.023,13.228,5.277,0.04,2.343],[5463.5,0.113,10.232,1970.698,609.228,49.195,124.755,107.002,1.714,70.917,2.562,1.641,1.509,1.712,1.305,1.608,1.042,1.546,0.072,-0.911,-16.897,14.067,5.022,0.039,2.342],[5464.0,0.084,10.084,1974.24,611.556,49.529,125.352,107.482,1.705,74.034,2.522,1.635,1.495,1.711,1.285,1.714,1.156,1.484,0.076,-0.963,-22.541,14.92,4.793,0.047,2.333],[5464.5,0.077,9.531,1983.337,615.34,50.357,125.95,106.447,1.678,74.369,2.627,1.625,1.487,1.665,1.252,1.785,1.194,1.495,0.08,-0.99,-28.741,15.288,4.753,0.055,2.324],[5465.0,0.076,9.437,2000.713,617.84,50.649,126.512,105.53,1.631,72.318,2.76,1.619,1.48,1.599,1.207,1.818,1.143,1.594,0.083,-0.944,-33.157,15.142,4.923,0.057,2.319],[5465.5,0.073,9.538,2017.47,620.173,50.624,126.713,105.365,1.589,70.556,2.831,1.612,1.472,1.562,1.187,1.83,1.071,1.71,0.084,-0.783,-28.719,14.852,5.148,0.053,2.316],[5466.0,0.084,9.657,2020.577,624.152,50.746,126.218,106.759,1.57,70.512,2.856,1.602,1.469,1.55,1.184,1.829,1.031,1.774,0.086,-0.592,-16.566,14.664,5.278,0.046,2.311],[5466.5,0.084,9.93,2009.794,626.133,51.045,125.066,110.061,1.569,71.154,2.872,1.597,1.476,1.576,1.217,1.813,1.006,1.802,0.089,-0.51,-5.101,14.53,5.359,0.037,2.305],[5467.0,0.079,10.159,1998.727,624.539,51.307,123.653,112.238,1.555,70.642,2.994,1.601,1.495,1.578,1.283,1.807,0.977,1.85,0.093,-0.567,-0.11,14.345,5.451,0.037,2.298],[5467.5,0.091,10.234,2000.039,623.929,50.998,122.399,109.283,1.485,68.457,3.204,1.603,1.511,1.587,1.29,1.871,0.953,1.964,0.096,-0.737,-3.407,14.174,5.519,0.052,2.293],[5468.0,0.087,10.135,2009.096,627.006,50.1,121.434,103.94,1.397,65.856,3.321,1.595,1.511,1.685,1.325,1.985,0.942,2.108,0.096,-1.02,-12.473,14.139,5.571,0.07,2.293],[5468.5,0.084,10.195,2014.199,630.557,49.094,120.651,100.788,1.315,63.481,3.297,1.586,1.5,1.848,1.386,2.097,0.919,2.286,0.092,-1.301,-16.619,14.065,5.685,0.071,2.298],[5469.0,0.135,10.373,2012.056,631.84,49.041,120.07,100.4,1.251,61.899,3.25,1.583,1.488,1.851,1.278,2.176,0.887,2.456,0.088,-1.36,-13.737,13.903,5.82,0.062,2.305],[5469.5,0.139,10.637,2009.924,631.93,50.094,119.884,101.311,1.277,63.119,3.275,1.582,1.481,1.679,1.259,2.142,0.899,2.388,0.086,-1.233,-15.261,13.943,5.76,0.06,2.309],[5470.0,0.178,10.849,2014.322,631.431,51.406,120.106,102.076,1.415,67.115,3.369,1.584,1.479,1.584,1.332,1.971,0.969,2.048,0.086,-1.161,-22.918,14.166,5.437,0.06,2.31],[5470.5,0.133,11.029,2019.923,627.76,51.826,120.277,102.513,1.591,70.615,3.499,1.593,1.484,1.61,1.331,1.753,1.032,1.702,0.087,-1.239,-24.719,14.175,5.1,0.055,2.309],[5471.0,0.127,11.199,2014.26,618.2,51.298,119.868,102.644,1.752,72.42,3.57,1.618,1.501,1.628,1.318,1.568,1.075,1.462,0.089,-1.365,-17.513,13.975,4.829,0.047,2.308],[5471.5,0.119,11.031,1997.022,604.907,50.891,118.929,102.231,1.887,73.246,3.499,1.653,1.542,1.578,1.272,1.427,1.118,1.278,0.092,-1.416,-13.457,13.727,4.559,0.04,2.305],[5472.0,0.126,10.786,1984.089,593.889,50.383,118.119,101.346,1.946,72.21,3.481,1.684,1.602,1.533,1.242,1.336,1.111,1.204,0.094,-1.324,-21.304,13.279,4.438,0.035,2.304],[5472.5,0.141,10.94,1983.218,588.909,49.574,118.082,100.09,1.921,69.043,3.661,1.698,1.647,1.65,1.329,1.282,1.027,1.252,0.089,-1.2,-29.06,12.585,4.555,0.039,2.315],[5473.0,0.158,11.077,1987.15,587.416,48.911,118.731,97.819,1.865,65.851,3.906,1.702,1.645,2.022,1.847,1.266,0.945,1.341,0.076,-1.224,-20.675,12.053,4.735,0.039,2.339],[5473.5,0.158,11.127,1985.584,584.673,48.324,119.44,96.28,1.815,64.62,3.932,1.71,1.611,2.305,1.878,1.287,0.916,1.407,0.067,-1.274,-9.664,11.921,4.834,0.028,2.358],[5474.0,0.141,10.979,1977.832,579.528,47.935,119.92,97.639,1.824,66.597,3.658,1.726,1.58,2.088,1.354,1.313,0.952,1.383,0.07,-1.112,-12.369,12.239,4.775,0.025,2.353],[5474.5,0.152,10.307,1969.468,575.545,48.585,120.624,100.542,1.864,69.858,3.323,1.737,1.564,1.723,1.28,1.342,1.011,1.328,0.083,-0.829,-19.923,12.776,4.69,0.04,2.329],[5475.0,0.147,9.68,1966.021,578.192,49.798,122.11,102.793,1.861,71.785,3.148,1.73,1.554,1.565,1.266,1.397,1.049,1.332,0.095,-0.701,-23.496,13.274,4.695,0.049,2.307],[5475.5,0.156,9.673,1972.346,588.422,50.17,124.221,104.894,1.861,73.903,3.186,1.699,1.546,1.543,1.282,1.476,1.132,1.307,0.099,-0.778,-25.554,14.033,4.606,0.048,2.298],[5476.0,0.161,9.956,1990.058,600.345,49.979,126.094,107.178,1.866,76.46,3.288,1.666,1.538,1.57,1.35,1.573,1.245,1.264,0.095,-0.926,-19.931,14.992,4.471,0.049,2.301],[5476.5,0.173,10.138,2011.056,607.948,49.83,126.876,107.943,1.875,78.472,3.281,1.645,1.532,1.631,1.393,1.639,1.329,1.234,0.09,-1.056,-6.935,15.699,4.383,0.048,2.309],[5477.0,0.18,10.155,2022.759,609.543,49.699,126.492,106.598,1.913,80.684,3.183,1.641,1.535,1.678,1.387,1.66,1.424,1.168,0.087,-1.147,-0.684,16.211,4.231,0.044,2.314],[5477.5,0.225,10.156,2023.483,607.195,49.807,125.546,105.423,1.97,83.565,3.202,1.647,1.547,1.694,1.355,1.671,1.593,1.056,0.088,-1.189,-7.563,16.808,3.933,0.043,2.312],[5478.0,0.186,10.102,2019.389,605.019,50.239,124.555,106.085,2.019,85.839,3.367,1.653,1.56,1.711,1.329,1.683,1.799,0.94,0.088,-1.187,-12.996,17.351,3.595,0.045,2.312],[5478.5,0.209,9.931,2008.533,604.681,50.304,123.533,106.615,2.04,87.322,3.466,1.654,1.565,1.742,1.345,1.706,2.021,0.848,0.083,-1.164,-5.454,17.773,3.277,0.045,2.321],[5479.0,0.202,9.983,1989.093,604.377,49.796,122.17,105.531,2.022,88.068,3.361,1.655,1.566,1.775,1.378,1.745,2.174,0.803,0.073,-1.083,7.491,18.019,3.077,0.044,2.34],[5479.5,0.208,10.433,1974.072,604.693,49.246,120.296,103.453,1.942,86.837,3.166,1.654,1.565,1.784,1.4,1.816,2.174,0.837,0.064,-0.86,11.065,17.977,3.071,0.044,2.358],[5480.0,0.221,10.741,1975.015,607.146,49.134,118.171,101.444,1.779,82.46,3.03,1.647,1.562,1.754,1.371,1.923,1.981,0.984,0.06,-0.49,3.705,17.424,3.285,0.046,2.367],[5480.5,0.262,10.813,1983.302,610.481,49.367,116.408,100.865,1.615,77.12,2.991,1.638,1.555,1.717,1.38,2.023,1.715,1.188,0.06,-0.101,-3.907,16.67,3.619,0.046,2.367],[5481.0,0.302,10.88,1987.455,614.543,49.822,115.48,100.495,1.512,73.543,3.011,1.627,1.541,1.728,1.381,2.096,1.538,1.37,0.06,0.151,-4.102,16.171,3.914,0.043,2.365],[5481.5,0.307,10.986,1989.28,620.824,50.204,115.397,98.863,1.416,70.77,3.029,1.611,1.518,1.765,1.379,2.176,1.382,1.585,0.061,0.192,0.244,15.721,4.236,0.041,2.362],[5482.0,0.278,11.266,1991.073,628.513,50.196,115.879,98.058,1.348,69.409,2.989,1.591,1.493,1.773,1.355,2.249,1.282,1.756,0.062,0.092,-0.988,15.48,4.483,0.041,2.358],[5482.5,0.316,11.513,1994.984,635.246,50.321,116.721,99.663,1.357,70.768,2.906,1.574,1.473,1.698,1.332,2.271,1.304,1.745,0.062,-0.063,-8.63,15.731,4.481,0.049,2.356],[5483.0,0.349,11.187,2004.232,640.091,51.132,118.149,101.265,1.415,73.442,2.86,1.562,1.456,1.604,1.329,2.244,1.394,1.614,0.064,-0.316,-14.631,16.201,4.318,0.064,2.35],[5483.5,0.33,10.769,2011.776,642.863,51.824,120.504,101.675,1.482,76.133,2.921,1.556,1.441,1.566,1.285,2.204,1.508,1.467,0.071,-0.717,-18.006,16.672,4.112,0.072,2.336],[5484.0,0.314,10.676,2012.823,641.638,51.63,123.343,102.069,1.54,78.791,3.086,1.559,1.438,1.565,1.3,2.189,1.654,1.328,0.08,-1.154,-24.056,17.206,3.869,0.066,2.319],[5484.5,0.274,10.462,2012.562,634.077,51.166,125.314,102.774,1.553,80.116,3.241,1.577,1.454,1.565,1.268,2.217,1.752,1.266,0.089,-1.486,-31.329,17.577,3.724,0.057,2.304],[5485.0,0.209,10.439,2015.123,620.332,50.969,125.374,103.017,1.531,79.36,3.324,1.612,1.495,1.534,1.237,2.239,1.732,1.294,0.097,-1.67,-34.394,17.5,3.752,0.053,2.292],[5485.5,0.154,10.203,2016.008,604.319,51.353,123.892,101.957,1.492,76.813,3.383,1.655,1.558,1.506,1.245,2.225,1.613,1.385,0.105,-1.741,-32.359,16.954,3.911,0.047,2.284],[5486.0,0.186,10.383,2007.934,591.433,51.56,122.242,100.052,1.444,72.545,3.598,1.691,1.626,1.527,1.218,2.161,1.421,1.53,0.103,-1.819,-25.66,15.946,4.181,0.05,2.29],[5486.5,0.158,11.019,1996.266,584.681,51.022,121.391,96.841,1.434,66.946,4.117,1.71,1.662,1.754,1.458,1.967,1.185,1.669,0.084,-2.146,-15.825,14.387,4.538,0.065,2.324],[5487.0,0.086,11.041,1994.953,581.349,50.135,121.019,92.39,1.473,61.54,4.608,1.72,1.65,2.257,2.431,1.695,0.971,1.75,0.054,-2.673,-6.267,12.72,4.885,0.055,2.386],[5487.5,0.107,10.592,2004.129,577.166,49.344,120.25,90.298,1.518,58.349,4.142,1.733,1.617,2.483,2.101,1.5,0.845,1.776,0.035,-2.796,-0.039,11.62,5.11,0.01,2.436],[5488.0,0.102,10.31,2013.094,571.757,49.123,119.101,92.836,1.55,58.483,3.308,1.749,1.597,2.081,1.296,1.431,0.806,1.776,0.037,-2.169,1.229,11.328,5.217,-0.011,2.429],[5488.5,0.11,10.27,2017.21,569.061,49.642,118.52,97.48,1.576,62.754,3.084,1.757,1.59,1.638,1.298,1.52,0.883,1.729,0.057,-1.323,1.274,12.241,5.161,0.013,2.381],[5489.0,0.12,10.039,2013.064,573.302,50.641,119.148,102.299,1.583,69.917,3.186,1.744,1.578,1.516,1.331,1.754,1.087,1.624,0.079,-0.89,6.315,14.174,4.873,0.035,2.337],[5489.5,0.112,9.967,2001.716,583.17,51.366,120.674,106.493,1.574,74.682,3.258,1.715,1.559,1.56,1.321,1.923,1.238,1.554,0.089,-0.915,13.37,15.453,4.635,0.038,2.316],[5490.0,0.108,9.918,1993.585,593.234,51.868,122.25,108.978,1.551,74.751,3.26,1.686,1.541,1.616,1.301,1.982,1.264,1.569,0.09,-1.118,10.857,15.693,4.609,0.043,2.311],[5490.5,0.121,9.846,1991.171,600.499,52.147,123.278,109.177,1.515,72.885,3.273,1.665,1.531,1.63,1.315,2.035,1.264,1.61,0.089,-1.235,-0.694,15.737,4.621,0.05,2.312],[5491.0,0.118,10.0,1987.944,605.629,52.071,123.775,107.641,1.493,71.64,3.284,1.651,1.523,1.617,1.306,2.058,1.261,1.633,0.087,-1.244,-8.958,15.689,4.618,0.048,2.313],[5491.5,0.119,10.156,1983.42,608.507,52.017,124.056,104.434,1.491,71.995,3.256,1.643,1.515,1.652,1.324,2.062,1.271,1.623,0.088,-1.293,-9.017,15.693,4.584,0.034,2.312],[5492.0,0.129,10.367,1982.67,607.807,52.096,124.251,101.738,1.489,72.55,3.281,1.645,1.511,1.738,1.388,2.078,1.289,1.612,0.091,-1.398,-6.793,15.805,4.55,0.032,2.306],[5492.5,0.125,10.563,1982.003,604.072,52.148,124.332,102.082,1.502,72.857,3.388,1.655,1.511,1.743,1.384,2.092,1.328,1.576,0.097,-1.454,-6.69,16.039,4.483,0.046,2.297],[5493.0,0.125,10.648,1981.072,597.848,51.973,124.196,104.946,1.538,73.59,3.499,1.673,1.52,1.676,1.313,2.061,1.357,1.519,0.1,-1.396,-9.971,16.177,4.423,0.054,2.293],[5493.5,0.153,10.298,1986.737,587.968,51.488,123.557,107.827,1.601,74.085,3.511,1.701,1.546,1.586,1.299,1.93,1.29,1.495,0.097,-1.217,-14.456,15.75,4.537,0.05,2.301],[5494.0,0.124,9.63,1994.943,574.418,50.914,121.968,109.56,1.658,73.806,3.44,1.741,1.592,1.537,1.3,1.778,1.179,1.509,0.092,-1.04,-17.243,15.051,4.742,0.043,2.314],[5494.5,0.133,9.509,1994.803,559.403,49.88,119.073,109.814,1.653,71.617,3.371,1.788,1.652,1.558,1.366,1.699,1.077,1.587,0.088,-1.017,-15.574,14.348,4.964,0.038,2.324],[5495.0,0.141,9.64,1990.458,544.104,49.087,115.149,108.077,1.555,65.711,3.387,1.838,1.729,1.656,1.508,1.649,0.911,1.843,0.085,-1.129,-9.416,13.102,5.39,0.035,2.334],[5495.5,0.175,10.048,1996.627,530.829,48.509,111.277,105.045,1.439,60.236,3.441,1.884,1.82,1.792,1.597,1.634,0.777,2.107,0.08,-1.297,-0.011,12.002,5.736,0.035,2.346],[5496.0,0.202,10.376,2013.601,523.143,47.643,108.76,101.723,1.383,57.995,3.398,1.912,1.894,1.911,1.567,1.649,0.743,2.22,0.073,-1.535,8.734,11.643,5.813,0.043,2.36],[5496.5,0.218,10.549,2023.013,521.546,46.284,108.411,97.985,1.327,55.976,3.324,1.917,1.909,2.114,1.745,1.685,0.734,2.295,0.063,-1.887,10.7,11.408,5.765,0.058,2.38],[5497.0,0.181,10.694,2017.188,522.287,44.691,110.051,94.088,1.302,54.362,3.373,1.915,1.868,2.419,2.427,1.676,0.757,2.218,0.051,-2.405,7.508,11.138,5.466,0.061,2.406],[5497.5,0.187,10.643,2015.007,520.302,43.838,112.443,92.502,1.311,53.962,3.459,1.922,1.822,2.399,2.262,1.66,0.795,2.091,0.044,-3.157,5.255,11.11,5.191,0.051,2.423],[5498.0,0.194,10.436,2025.936,511.512,44.293,114.116,94.62,1.292,55.49,3.422,1.955,1.827,1.928,1.516,1.744,0.839,2.078,0.05,-4.082,-1.163,11.497,5.086,0.045,2.41],[5498.5,0.165,10.287,2033.261,494.835,45.595,114.338,98.116,1.253,60.477,3.386,2.021,1.911,1.517,1.366,1.964,0.976,2.023,0.073,-4.98,-16.285,12.539,4.804,0.043,2.369],[5499.0,0.167,10.011,2022.756,472.666,46.796,112.928,99.911,1.226,65.415,3.431,2.116,2.086,1.383,1.386,2.182,1.133,1.931,0.1,-5.739,-26.158,13.654,4.486,0.045,2.331],[5499.5,0.179,10.044,2003.147,447.6,46.839,109.752,98.26,1.222,66.212,3.588,2.234,2.351,1.476,1.518,2.275,1.244,1.836,0.116,-6.922,-23.847,14.197,4.25,0.07,2.316],[5500.0,0.078,10.462,1994.07,420.289,45.413,104.677,91.967,1.217,64.39,3.729,2.379,2.691,1.851,2.654,2.277,1.514,1.547,0.12,-9.396,-20.212,14.157,3.569,0.101,2.32],[5500.5,0.067,10.473,1997.487,392.579,42.526,97.988,81.918,1.132,57.958,3.578,2.547,3.07,2.871,3.888,2.166,1.631,1.338,0.112,-12.467,-21.079,12.557,2.864,0.099,2.341],[5501.0,0.078,9.858,2001.923,368.833,37.952,91.008,70.855,0.987,49.043,3.222,2.711,3.406,5.235,5.659,2.108,1.512,1.396,0.086,-13.869,-24.654,10.615,2.603,0.069,2.385],[5501.5,0.09,9.612,2004.917,353.007,31.976,85.785,61.76,0.915,43.644,2.985,2.833,3.556,8.981,7.14,2.109,1.463,1.442,0.049,-12.673,-27.068,9.853,2.499,0.043,2.451],[5502.0,0.097,9.612,2006.535,347.424,26.848,83.665,57.524,0.908,41.685,2.873,2.878,3.399,11.599,7.565,2.081,1.441,1.444,0.024,-10.311,-21.569,9.65,2.485,0.027,2.507],[5502.5,0.128,9.566,2005.916,353.64,23.405,84.604,59.908,0.918,42.805,2.839,2.828,3.024,8.878,5.938,2.051,1.433,1.431,0.023,-8.291,-7.533,9.612,2.488,0.014,2.51],[5503.0,0.169,9.567,2005.766,372.534,24.085,87.83,67.337,1.051,47.288,2.956,2.684,2.637,4.887,3.257,1.878,1.413,1.328,0.04,-6.716,4.876,9.938,2.613,0.023,2.463],[5503.5,0.246,9.634,2003.929,402.542,30.63,92.592,76.834,1.273,53.32,3.163,2.484,2.335,2.582,1.945,1.68,1.344,1.251,0.065,-5.07,10.105,10.854,3.012,0.043,2.408],[5504.0,0.248,9.715,1997.603,440.784,37.494,98.29,86.25,1.349,58.412,3.255,2.269,2.115,1.774,1.604,1.753,1.28,1.371,0.078,-3.34,12.883,12.057,3.499,0.053,2.374],[5504.5,0.253,9.785,1995.159,483.167,40.722,104.128,93.412,1.272,62.341,3.217,2.07,1.945,1.61,1.647,2.089,1.303,1.598,0.077,-1.957,16.074,13.479,3.834,0.049,2.364],[5505.0,0.249,9.865,1998.736,523.749,42.171,108.925,96.658,1.19,65.12,3.121,1.909,1.804,1.672,1.654,2.421,1.357,1.782,0.069,-1.089,15.82,14.669,4.012,0.037,2.368],[5505.5,0.247,9.978,2000.578,556.688,43.385,111.81,96.56,1.143,66.626,3.021,1.796,1.691,1.745,1.566,2.616,1.371,1.908,0.063,-0.589,15.678,15.256,4.128,0.023,2.373],[5506.0,0.249,9.905,1998.302,580.726,46.26,113.115,95.571,1.088,67.554,3.033,1.722,1.608,1.717,1.481,2.861,1.368,2.092,0.062,-0.402,16.576,15.843,4.299,0.019,2.368],[5506.5,0.307,9.764,1997.089,598.773,49.287,114.112,96.711,1.024,68.249,3.129,1.67,1.544,1.677,1.401,3.164,1.357,2.333,0.063,-0.561,14.918,16.514,4.517,0.03,2.362],[5507.0,0.303,9.592,1997.953,613.069,49.506,115.939,100.705,0.996,68.953,3.08,1.631,1.49,1.664,1.352,3.329,1.349,2.468,0.061,-1.011,16.261,16.927,4.657,0.043,2.363],[5507.5,0.319,9.465,1997.668,623.382,48.07,118.802,103.966,1.021,69.984,2.864,1.604,1.446,1.603,1.347,3.296,1.35,2.442,0.058,-1.675,25.143,17.171,4.721,0.047,2.369],[5508.0,0.289,9.493,2000.648,627.036,48.95,121.872,104.424,1.085,70.821,2.8,1.595,1.421,1.507,1.261,3.098,1.324,2.339,0.059,-2.623,27.637,17.126,4.802,0.042,2.365],[5508.5,0.274,9.405,2005.91,620.849,49.917,123.936,104.726,1.157,70.928,3.037,1.611,1.431,1.415,1.174,2.841,1.273,2.231,0.066,-4.089,17.061,16.754,4.887,0.039,2.351],[5509.0,0.271,9.424,2002.561,601.82,50.233,124.246,106.902,1.226,70.525,3.317,1.662,1.491,1.344,1.148,2.599,1.221,2.126,0.076,-6.186,7.8,16.233,4.932,0.043,2.335],[5509.5,0.28,9.483,1991.413,567.985,50.366,122.907,108.485,1.288,70.322,3.377,1.761,1.624,1.315,1.216,2.377,1.181,2.011,0.087,-8.658,6.186,15.616,4.906,0.047,2.323],[5510.0,0.357,9.152,1985.531,522.643,49.944,120.676,106.156,1.335,69.32,3.252,1.913,1.85,1.309,1.318,2.194,1.128,1.944,0.098,-11.372,7.704,14.944,4.916,0.052,2.318],[5510.5,0.368,8.909,1984.733,472.989,48.099,118.077,98.121,1.332,66.05,3.09,2.114,2.175,1.45,2.021,2.103,1.1,1.914,0.104,-15.091,9.604,14.306,4.834,0.064,2.325],[5511.0,0.295,8.925,1981.844,422.729,44.435,114.55,84.836,1.236,60.581,2.967,2.366,2.589,1.993,4.029,2.073,1.157,1.797,0.103,-20.626,9.479,13.094,4.227,0.083,2.342],[5511.5,0.249,8.921,1977.729,371.148,40.349,109.151,70.022,1.119,52.143,2.864,2.694,3.106,3.537,6.118,1.939,1.163,1.667,0.107,-27.22,2.477,11.119,3.537,0.097,2.355],[5512.0,0.217,8.85,1976.585,318.201,37.321,102.279,57.845,0.996,42.519,2.669,3.143,3.745,6.204,7.348,1.709,1.075,1.59,0.135,-33.109,-10.762,8.757,3.01,0.093,2.344],[5512.5,0.215,8.79,1976.999,266.187,34.763,96.064,50.438,0.887,35.673,2.507,3.757,4.544,8.042,7.898,1.522,1.021,1.49,0.194,-37.288,-22.914,6.912,2.512,0.076,2.317],[5513.0,0.228,8.786,1976.755,220.127,31.894,92.677,48.021,0.847,32.194,2.588,4.543,5.567,9.591,8.166,1.451,1.009,1.439,0.251,-39.788,-26.306,6.278,2.309,0.067,2.306],[5513.5,0.239,8.786,1981.394,184.621,27.989,92.388,48.015,0.825,31.073,2.783,5.416,6.847,11.727,13.563,1.46,1.031,1.417,0.245,-40.996,-20.474,6.154,2.217,0.07,2.337],[5514.0,0.249,8.791,1990.624,159.313,23.884,93.46,46.572,0.791,30.923,2.746,6.277,8.401,13.865,19.207,1.505,1.106,1.364,0.198,-41.618,-8.666,6.075,2.049,0.071,2.385],[5514.5,0.212,8.783,1995.824,140.784,22.157,94.308,43.12,0.742,30.836,2.424,7.103,10.431,14.375,13.752,1.609,1.252,1.29,0.22,-42.354,1.478,6.083,1.817,0.064,2.388],[5515.0,0.16,8.773,2002.872,126.683,23.479,95.318,38.467,0.698,30.262,2.188,7.894,13.189,15.405,14.57,1.743,1.464,1.199,0.409,-43.354,-0.399,6.199,1.588,0.056,2.317],[5515.5,0.09,8.768,2017.751,115.66,25.311,97.848,32.981,0.642,28.594,2.107,8.646,16.626,20.958,27.818,1.909,1.784,1.082,0.77,-44.182,-11.181,6.225,1.323,0.047,2.221],[5516.0,0.072,8.759,2026.259,106.875,24.881,102.147,28.309,0.53,24.007,2.067,9.357,20.361,35.621,45.304,1.931,1.676,1.222,1.13,-44.387,-16.527,5.266,1.185,0.037,2.156],[5516.5,0.078,8.74,2021.844,99.855,23.246,106.866,26.017,0.447,18.953,2.046,10.015,23.591,57.966,64.464,1.76,1.229,1.477,1.331,-43.769,-10.744,4.031,1.237,0.035,2.133],[5517.0,0.078,8.733,2018.022,94.572,22.754,110.217,25.748,0.462,17.238,2.048,10.574,25.008,65.146,63.304,1.544,1.038,1.484,1.43,-42.37,2.69,3.618,1.296,0.038,2.129],[5517.5,0.077,8.733,2016.874,91.604,23.741,111.28,27.316,0.497,17.26,2.097,10.917,23.983,52.089,39.232,1.338,0.92,1.455,1.547,-40.381,12.983,3.391,1.371,0.031,2.118],[5518.0,0.075,8.729,2014.427,92.483,25.109,110.353,30.191,0.518,17.922,2.182,10.813,21.14,38.086,26.164,1.225,0.843,1.455,1.637,-37.995,11.635,3.236,1.426,0.023,2.101],[5518.5,0.079,8.732,2015.461,100.193,26.187,108.352,33.093,0.579,20.436,2.249,9.981,17.349,29.676,26.432,1.219,0.949,1.312,1.546,-35.285,5.023,3.615,1.419,0.022,2.095],[5519.0,0.084,8.746,2022.244,118.541,26.802,106.204,35.523,0.694,25.186,2.295,8.436,13.033,23.732,21.742,1.318,1.282,1.058,1.232,-32.113,1.634,4.709,1.37,0.025,2.11],[5519.5,0.084,8.775,2026.779,149.002,26.734,104.638,38.383,0.785,30.232,2.396,6.711,8.994,19.588,16.457,1.485,1.646,0.907,0.807,-28.391,-1.392,5.977,1.347,0.03,2.157],[5520.0,0.108,8.817,2029.265,189.779,25.877,103.949,42.194,0.821,34.564,2.579,5.269,6.023,17.117,18.615,1.696,1.894,0.894,0.427,-24.508,-3.363,7.122,1.392,0.032,2.24],[5520.5,0.141,8.873,2035.253,239.391,25.137,103.932,46.895,0.832,39.13,2.747,4.177,4.13,13.141,17.894,1.957,2.044,0.956,0.197,-20.918,-3.25,8.32,1.508,0.03,2.331],[5521.0,0.141,8.946,2041.756,296.673,25.508,104.41,54.791,0.822,44.252,2.856,3.371,3.004,6.94,11.036,2.4,1.994,1.221,0.108,-16.975,-6.44,10.028,1.89,0.029,2.383],[5521.5,0.213,9.048,2043.82,358.359,26.62,105.731,67.749,0.814,49.33,2.969,2.79,2.346,3.476,4.571,2.876,1.707,1.728,0.09,-11.865,-8.402,11.889,2.667,0.048,2.383],[5522.0,0.413,9.236,2039.552,421.074,27.842,108.451,81.779,0.984,52.709,3.031,2.375,1.949,2.093,1.808,2.566,1.259,2.056,0.091,-6.696,-0.712,12.341,3.848,0.075,2.361],[5522.5,0.394,9.445,2027.353,484.368,29.459,112.49,91.866,1.219,55.268,2.989,2.065,1.695,1.614,1.677,1.942,0.918,2.117,0.091,-3.331,5.482,11.948,4.891,0.079,2.341],[5523.0,0.447,9.814,2010.029,546.882,33.136,116.859,97.492,1.31,59.006,2.889,1.829,1.529,1.556,1.476,1.852,0.867,2.136,0.087,-1.954,-0.984,12.398,5.302,0.065,2.329],[5523.5,0.514,10.162,1995.127,603.137,39.575,120.44,100.6,1.348,62.504,2.819,1.658,1.421,1.652,1.407,1.904,0.91,2.092,0.082,-1.603,-5.544,13.1,5.341,0.062,2.324],[5524.0,0.516,9.75,1987.042,644.948,45.809,122.805,102.45,1.388,65.37,2.835,1.551,1.348,1.734,1.388,1.92,0.966,1.991,0.077,-1.786,1.984,13.61,5.234,0.065,2.323],[5524.5,0.583,9.346,1984.751,665.834,49.423,124.019,104.27,1.444,68.03,2.892,1.502,1.312,1.776,1.389,1.921,1.037,1.857,0.075,-2.329,8.355,14.154,5.072,0.065,2.323],[5525.0,0.516,9.337,1984.171,659.789,51.992,123.991,105.173,1.549,70.092,2.926,1.516,1.321,1.739,1.385,1.825,1.104,1.658,0.077,-3.175,6.723,14.386,4.842,0.063,2.319],[5525.5,0.526,9.407,1984.45,623.862,53.725,122.533,104.698,1.693,71.398,2.935,1.603,1.39,1.628,1.398,1.647,1.147,1.437,0.085,-4.519,1.632,14.19,4.593,0.06,2.312],[5526.0,0.528,9.577,1986.185,564.321,53.925,120.007,104.427,1.8,72.435,2.906,1.772,1.547,1.495,1.421,1.494,1.155,1.293,0.099,-6.51,-0.798,13.719,4.406,0.062,2.305],[5526.5,0.469,9.431,1989.667,494.481,53.292,117.421,104.155,1.884,73.223,2.896,2.022,1.853,1.436,1.51,1.371,1.148,1.195,0.119,-8.956,4.653,13.169,4.258,0.075,2.296],[5527.0,0.5,8.999,1993.735,423.973,51.3,115.587,99.034,1.932,70.77,2.971,2.359,2.371,1.628,1.257,1.285,1.157,1.111,0.143,-11.94,13.468,12.672,4.067,0.107,2.291],[5527.5,0.526,8.816,1992.715,355.282,46.619,114.288,84.997,1.594,58.995,2.908,2.815,3.151,2.417,6.733,1.405,1.066,1.37,0.166,-16.242,14.934,11.016,3.823,0.139,2.296],[5528.0,0.25,8.738,1982.612,287.474,40.724,112.692,64.588,1.011,41.431,2.52,3.479,4.28,5.043,22.841,1.643,0.875,1.932,0.213,-22.119,9.22,8.201,3.445,0.128,2.289],[5528.5,0.215,8.683,1971.818,220.298,36.401,110.642,46.147,0.714,29.774,2.175,4.539,6.097,13.143,32.227,1.624,0.862,1.946,0.357,-28.458,1.385,5.955,2.648,0.073,2.245],[5529.0,0.183,8.656,1968.515,157.98,34.333,108.96,35.323,0.679,25.932,2.13,6.33,9.386,28.248,40.377,1.429,1.11,1.372,0.686,-33.985,-8.166,4.943,1.747,0.032,2.18],[5529.5,0.141,8.641,1969.855,107.934,33.594,108.347,31.193,0.708,24.974,2.152,9.265,15.365,42.858,50.068,1.289,1.418,0.926,1.229,-38.384,-14.365,4.654,1.238,0.023,2.134],[5530.0,0.141,8.637,1972.031,75.154,32.564,108.519,30.232,0.724,24.411,2.129,13.306,24.87,56.342,61.409,1.208,1.61,0.755,1.902,-41.841,-12.11,4.465,1.034,0.02,2.115],[5530.5,0.129,8.634,1975.501,58.339,31.287,108.507,30.293,0.731,23.891,2.111,17.141,36.225,72.76,82.398,1.136,1.664,0.683,2.512,-44.502,-9.146,4.243,0.947,0.016,2.109],[5531.0,0.124,8.633,1985.437,52.168,30.453,107.676,30.757,0.73,23.324,2.105,19.169,44.69,86.622,94.146,1.086,1.677,0.648,2.823,-46.489,-12.53,4.046,0.896,0.015,2.108],[5531.5,0.124,8.632,2000.997,51.131,30.617,106.261,31.292,0.722,23.015,2.067,19.557,47.669,88.59,82.523,1.075,1.662,0.648,2.779,-47.998,-17.327,3.961,0.886,0.021,2.117],[5532.0,0.124,8.634,2017.009,51.076,31.584,104.889,31.974,0.703,23.988,2.007,19.579,47.916,82.482,66.299,1.158,1.84,0.633,2.604,-49.293,-17.808,4.146,0.844,0.03,2.133],[5532.5,0.124,8.631,2028.584,50.22,32.543,103.889,32.223,0.682,25.589,1.986,19.912,48.605,79.564,64.592,1.304,2.196,0.596,2.557,-50.482,-14.384,4.539,0.77,0.029,2.141],[5533.0,0.124,8.627,2032.316,48.947,33.115,103.273,31.548,0.701,26.139,2.025,20.43,49.642,83.822,71.325,1.3,2.468,0.53,2.7,-51.476,-6.854,4.644,0.701,0.023,2.135],[5533.5,0.124,8.629,2028.587,48.212,33.344,103.082,30.332,0.748,25.548,2.033,20.742,48.074,91.267,84.134,1.175,2.563,0.459,2.89,-52.138,-2.469,4.479,0.649,0.022,2.122],[5534.0,0.129,8.628,2022.748,48.284,33.333,103.341,29.563,0.772,24.413,1.974,20.711,42.69,93.696,99.364,1.06,2.29,0.467,2.974,-52.27,-8.487,4.176,0.684,0.027,2.114],[5534.5,0.129,8.619,2014.575,48.891,33.326,103.903,29.692,0.77,23.255,1.931,20.454,36.222,85.506,84.35,0.958,1.782,0.55,2.951,-51.705,-15.803,3.767,0.802,0.029,2.113],[5535.0,0.129,8.618,2004.775,49.768,33.502,104.526,30.842,0.752,21.876,2.017,20.093,31.179,71.021,58.402,0.857,1.26,0.712,2.88,-50.627,-14.225,3.298,1.011,0.029,2.115],[5535.5,0.124,8.625,2000.063,50.937,33.838,105.066,32.631,0.717,20.204,2.169,19.632,27.849,58.033,45.104,0.765,0.824,0.982,2.733,-49.389,-7.611,2.811,1.325,0.031,2.122],[5536.0,0.118,8.626,1997.887,52.672,34.256,105.546,34.269,0.69,19.004,2.267,18.985,25.66,49.857,35.65,0.695,0.559,1.279,2.473,-48.173,-2.649,2.451,1.668,0.035,2.138],[5536.5,0.118,8.621,1988.871,55.074,34.732,105.994,35.627,0.7,19.141,2.31,18.157,24.181,45.871,32.713,0.666,0.478,1.4,2.206,-47.008,-1.022,2.382,1.857,0.033,2.154],[5537.0,0.107,8.621,1978.346,57.7,35.248,106.367,36.854,0.724,20.083,2.354,17.331,23.267,44.673,34.398,0.684,0.51,1.348,2.058,-45.966,-2.02,2.531,1.847,0.029,2.16],[5537.5,0.107,8.629,1981.074,59.915,35.562,106.506,37.268,0.736,20.912,2.321,16.69,22.793,44.551,32.099,0.729,0.576,1.27,2.027,-45.206,-3.924,2.741,1.77,0.031,2.155],[5538.0,0.107,8.631,1997.142,61.462,35.585,106.262,36.284,0.752,21.66,2.175,16.27,22.588,45.09,32.669,0.765,0.656,1.172,2.056,-44.712,-8.235,2.94,1.669,0.035,2.145],[5538.5,0.107,8.635,2012.656,62.87,35.453,105.72,34.844,0.75,21.635,2.083,15.906,22.453,45.156,36.645,0.799,0.707,1.13,2.105,-44.06,-11.835,3.059,1.606,0.034,2.134],[5539.0,0.101,8.654,2020.598,65.924,35.577,105.177,34.245,0.731,20.865,2.146,15.169,21.875,43.422,34.476,0.828,0.71,1.166,2.063,-42.801,-7.05,3.091,1.615,0.033,2.128],[5539.5,0.095,8.659,2022.464,73.16,35.999,104.889,33.446,0.745,20.98,2.254,13.669,19.773,40.179,29.163,0.837,0.743,1.131,1.842,-40.89,3.245,3.185,1.594,0.032,2.13],[5540.0,0.089,8.665,2020.07,86.367,36.054,104.859,31.819,0.804,22.732,2.243,11.579,15.912,36.684,28.216,0.846,0.881,0.985,1.525,-38.4,9.033,3.474,1.485,0.029,2.135],[5540.5,0.078,8.672,2017.428,105.873,35.724,104.806,31.62,0.928,26.336,2.208,9.445,11.804,32.373,32.031,0.877,1.289,0.732,1.221,-35.305,8.704,4.171,1.251,0.027,2.14],[5541.0,0.074,8.691,2030.267,130.68,35.656,104.186,33.77,1.044,30.412,2.289,7.652,8.547,24.063,25.854,0.909,1.753,0.527,0.945,-31.643,2.903,4.85,1.037,0.025,2.151],[5541.5,0.073,8.719,2076.501,157.735,35.686,102.647,38.125,1.125,35.218,2.517,6.34,6.389,14.658,15.028,0.953,2.124,0.453,0.703,-27.572,-6.06,5.496,0.961,0.027,2.175],[5542.0,0.078,8.752,2121.148,183.91,35.313,100.556,46.402,1.289,42.185,2.908,5.437,5.109,8.54,8.239,1.085,2.474,0.439,0.499,-23.294,-11.987,7.216,1.076,0.035,2.215],[5542.5,0.142,8.831,2111.988,209.773,35.439,98.691,57.849,1.388,48.283,3.289,4.767,4.342,5.242,4.86,1.254,2.313,0.563,0.318,-19.121,-12.323,8.876,1.475,0.045,2.275],[5543.0,0.14,8.938,2065.257,238.893,35.405,97.525,67.014,1.306,50.196,3.347,4.186,3.794,3.922,5.395,1.433,1.71,0.894,0.166,-15.093,-11.806,9.499,2.171,0.05,2.355],[5543.5,0.169,9.086,2034.723,273.251,34.289,97.007,70.789,1.203,49.851,3.143,3.66,3.329,3.62,5.192,1.588,1.319,1.22,0.074,-11.17,-12.085,9.737,2.765,0.046,2.435],[5544.0,0.176,9.198,2024.34,311.162,32.924,96.933,70.859,1.144,49.001,3.106,3.214,2.92,3.577,4.922,1.665,1.192,1.401,0.042,-7.816,-8.873,9.726,3.033,0.033,2.474],[5544.5,0.217,9.241,2016.436,349.421,30.948,97.429,69.76,1.089,47.059,3.248,2.862,2.57,3.384,3.809,1.739,1.091,1.603,0.045,-5.695,1.683,9.656,3.295,0.03,2.459],[5545.0,0.785,9.313,2011.871,386.091,30.988,99.074,69.613,1.063,45.942,3.371,2.59,2.286,3.079,2.988,1.771,1.009,1.757,0.06,-4.831,16.822,9.611,3.538,0.07,2.421],[5545.5,0.356,9.44,2012.046,420.601,32.825,102.26,71.207,1.076,46.405,3.339,2.378,2.062,2.827,2.941,1.746,0.96,1.819,0.069,-4.758,25.056,9.588,3.708,0.112,2.396],[5546.0,0.466,9.476,2016.762,450.411,37.95,106.406,74.828,1.105,48.272,3.126,2.22,1.9,2.63,2.552,1.734,0.95,1.825,0.068,-5.119,19.266,9.781,3.819,0.106,2.389],[5546.5,0.495,9.47,2026.739,470.099,43.59,110.071,79.781,1.137,51.605,2.948,2.127,1.816,2.235,1.984,1.815,1.011,1.795,0.069,-5.856,10.131,10.537,3.867,0.072,2.383],[5547.0,0.488,9.445,2033.804,475.183,45.1,111.691,84.215,1.127,54.56,2.965,2.104,1.813,1.87,1.738,1.957,1.081,1.81,0.079,-7.112,8.013,11.251,3.864,0.056,2.363],[5547.5,0.643,9.35,2026.081,464.916,44.406,110.466,87.073,1.097,55.827,3.006,2.151,1.895,1.734,1.79,2.043,1.109,1.843,0.097,-8.906,10.024,11.44,3.83,0.069,2.338],[5548.0,0.856,9.308,2007.915,439.497,43.769,107.017,88.099,1.061,55.189,2.877,2.275,2.078,1.775,1.954,2.135,1.121,1.904,0.11,-11.075,15.384,11.555,3.827,0.1,2.328],[5548.5,0.629,9.289,1995.753,399.158,42.098,103.057,84.736,1.006,51.845,2.631,2.505,2.416,1.959,2.065,2.219,1.145,1.938,0.109,-13.881,27.508,11.396,3.697,0.126,2.342],[5549.0,0.564,9.278,1997.707,347.483,39.202,100.07,75.425,0.94,45.707,2.423,2.878,2.985,2.381,3.345,2.06,1.221,1.709,0.101,-18.301,29.889,9.931,3.064,0.13,2.372],[5549.5,0.385,9.29,2005.909,290.818,34.616,98.225,62.794,0.801,37.434,2.279,3.439,3.876,3.455,5.338,1.812,1.28,1.431,0.095,-24.432,9.994,7.462,2.18,0.119,2.401],[5550.0,0.318,9.105,2009.114,234.485,28.685,96.627,49.547,0.657,29.081,2.052,4.265,5.199,6.592,17.772,1.707,1.232,1.392,0.1,-30.728,-14.914,5.74,1.722,0.107,2.419],[5550.5,0.301,8.773,2007.729,180.913,24.345,94.561,37.455,0.584,23.542,1.697,5.528,7.188,15.04,36.142,1.643,1.304,1.299,0.146,-35.875,-20.754,4.904,1.451,0.091,2.407],[5551.0,0.162,8.701,2010.129,132.977,22.699,92.346,28.787,0.521,21.05,1.471,7.52,10.48,28.379,42.516,1.609,1.814,0.957,0.295,-39.954,-16.973,4.279,0.964,0.066,2.357],[5551.5,0.142,8.709,2016.067,95.323,21.97,90.708,24.508,0.477,19.369,1.511,10.491,16.499,39.244,41.72,1.644,2.228,0.739,0.547,-43.32,-17.458,3.997,0.667,0.048,2.316],[5552.0,0.146,8.714,2020.522,70.748,19.819,89.33,23.101,0.463,18.091,1.727,14.135,27.432,51.892,117.16,1.661,2.208,0.752,0.635,-45.917,-17.608,3.922,0.659,0.04,2.338],[5552.5,0.158,8.72,2027.052,56.928,17.11,86.745,22.823,0.457,17.82,2.015,17.566,44.044,85.648,367.678,1.631,2.148,0.76,0.43,-47.824,-18.114,3.81,0.659,0.026,2.415],[5553.0,0.164,8.721,2037.378,49.673,15.983,82.064,22.478,0.448,17.802,2.16,20.132,63.862,133.702,296.059,1.558,1.958,0.8,0.274,-49.346,-24.473,3.564,0.679,0.014,2.471],[5553.5,0.158,8.72,2042.651,46.176,15.716,76.434,21.743,0.441,17.649,2.095,21.656,85.592,133.215,96.371,1.435,1.641,0.88,0.3,-50.767,-29.503,3.234,0.735,0.027,2.469],[5554.0,0.152,8.718,2038.372,44.904,13.916,72.194,21.256,0.424,16.693,2.071,22.27,102.116,132.721,121.13,1.357,1.393,0.985,0.308,-52.147,-29.237,2.944,0.789,0.056,2.47],[5554.5,0.156,8.721,2032.352,45.263,11.524,70.814,20.722,0.378,14.071,2.079,22.093,96.242,187.974,299.014,1.261,1.05,1.268,0.167,-53.425,-27.473,2.45,0.89,0.06,2.514],[5555.0,0.152,8.723,2025.744,47.045,10.902,72.06,19.363,0.336,11.492,1.928,21.256,73.038,341.981,369.523,1.169,0.753,1.572,0.084,-54.514,-19.377,2.006,0.995,0.031,2.549],[5555.5,0.141,8.717,2014.412,50.103,10.655,74.965,18.151,0.325,10.619,1.646,19.959,51.412,263.122,269.332,1.145,0.682,1.679,0.186,-55.302,-9.707,1.902,1.035,-0.008,2.5],[5556.0,0.129,8.714,2001.2,54.247,12.191,79.026,18.995,0.335,10.823,1.516,18.434,36.605,101.138,111.956,1.11,0.658,1.686,0.59,-55.7,-10.46,1.893,1.067,-0.008,2.384],[5556.5,0.129,8.714,1995.66,59.232,16.431,84.043,21.525,0.37,11.742,1.532,16.883,27.376,48.676,32.41,1.04,0.656,1.587,1.124,-55.717,-20.697,1.958,1.107,0.022,2.275],[5557.0,0.118,8.713,1997.435,64.973,20.779,89.249,24.425,0.407,12.979,1.631,15.391,21.907,29.348,23.207,0.99,0.668,1.482,1.428,-55.461,-31.617,2.055,1.141,0.037,2.213],[5557.5,0.108,8.714,2005.259,72.11,23.28,93.412,27.303,0.442,14.272,1.764,13.868,18.702,20.658,17.979,0.985,0.669,1.471,1.422,-55.054,-38.982,2.225,1.234,0.028,2.192],[5558.0,0.107,8.726,2020.249,81.99,24.205,95.842,29.363,0.477,15.622,1.86,12.197,16.476,17.173,14.022,1.05,0.692,1.515,1.243,-54.614,-40.819,2.565,1.372,0.02,2.193],[5558.5,0.113,8.755,2036.551,95.604,24.086,96.872,29.561,0.492,16.659,1.952,10.46,14.199,17.724,12.126,1.143,0.736,1.554,0.976,-54.21,-39.763,2.875,1.45,0.03,2.212],[5559.0,0.129,8.783,2039.139,112.519,23.094,97.321,29.561,0.494,17.318,2.123,8.887,11.628,22.547,29.918,1.207,0.762,1.584,0.621,-53.778,-36.409,3.042,1.481,0.044,2.267],[5559.5,0.128,8.788,2025.895,130.791,20.848,97.658,31.921,0.5,18.848,2.264,7.646,9.207,25.682,94.864,1.34,0.833,1.611,0.284,-53.181,-25.461,3.432,1.525,0.033,2.365],[5560.0,0.137,8.79,2011.93,149.226,18.051,97.882,36.286,0.558,22.367,2.124,6.701,7.293,19.582,65.083,1.539,1.023,1.52,0.122,-52.34,-8.61,4.413,1.594,0.011,2.445],[5560.5,0.112,8.799,2010.291,169.153,17.024,98.064,40.166,0.649,26.13,1.973,5.912,5.927,13.65,9.583,1.562,1.143,1.368,0.125,-51.312,1.822,5.155,1.675,0.01,2.43],[5561.0,0.118,8.803,2016.17,192.283,19.441,98.464,42.666,0.732,28.269,2.066,5.201,4.989,10.858,9.212,1.383,1.082,1.275,0.213,-50.113,-1.65,5.135,1.764,0.031,2.351],[5561.5,0.118,8.787,2015.335,217.643,23.99,99.054,43.868,0.781,29.197,2.111,4.595,4.343,9.825,9.105,1.251,1.017,1.23,0.277,-48.597,-12.026,4.989,1.821,0.042,2.292],[5562.0,0.124,8.786,2012.436,242.485,26.941,99.416,42.871,0.797,29.14,2.026,4.124,3.898,8.937,8.609,1.202,0.983,1.223,0.268,-46.681,-15.665,4.894,1.848,0.041,2.28],[5562.5,0.129,8.796,2017.877,264.686,27.373,99.285,40.517,0.8,27.928,1.906,3.778,3.597,7.526,7.038,1.174,0.936,1.255,0.225,-44.451,-9.285,4.792,1.901,0.042,2.294],[5563.0,0.129,8.798,2026.691,283.341,27.021,98.885,38.694,0.764,26.166,1.888,3.529,3.393,6.299,6.142,1.232,0.911,1.352,0.19,-42.078,-2.26,4.793,1.952,0.043,2.311],[5563.5,0.131,8.81,2030.005,298.058,26.783,98.709,38.038,0.726,25.71,1.964,3.355,3.245,5.827,5.739,1.33,0.93,1.429,0.168,-39.874,-1.436,4.927,1.965,0.047,2.322],[5564.0,0.137,8.835,2026.342,308.752,26.568,99.039,39.581,0.731,28.354,2.086,3.239,3.143,5.769,5.852,1.444,1.003,1.44,0.148,-37.473,-4.645,5.406,1.995,0.054,2.335],[5564.5,0.145,8.855,2015.327,316.254,25.832,99.728,45.469,0.781,32.995,2.255,3.162,3.099,5.125,5.69,1.683,1.078,1.564,0.122,-33.683,-2.957,6.75,2.325,0.061,2.359],[5565.0,0.147,8.861,2000.17,321.438,25.619,100.548,54.2,0.816,36.425,2.351,3.111,3.101,4.221,3.75,1.915,1.042,1.854,0.098,-29.17,4.744,7.985,2.868,0.063,2.385],[5565.5,0.106,8.882,1987.552,323.28,27.362,101.412,61.026,0.811,38.417,2.261,3.093,3.115,4.161,4.7,2.007,0.978,2.053,0.093,-26.824,7.373,8.308,3.153,0.063,2.391],[5566.0,0.115,8.838,1984.746,318.696,30.104,102.165,65.282,0.794,40.253,2.267,3.138,3.146,4.836,6.13,2.078,0.944,2.219,0.106,-27.373,-3.728,8.399,3.317,0.061,2.377],[5566.5,0.112,8.744,1994.638,305.439,31.857,102.479,71.139,0.644,39.307,2.549,3.274,3.279,4.843,5.317,2.887,0.807,3.848,0.121,-28.24,-17.671,8.727,4.139,0.058,2.366],[5567.0,0.137,8.731,2006.625,283.943,32.729,102.055,78.298,0.454,37.611,2.812,3.522,3.632,4.156,4.173,4.08,0.677,6.114,0.127,-27.788,-22.48,9.132,5.035,0.054,2.368],[5567.5,0.123,8.743,2010.416,256.053,32.336,100.849,79.157,0.408,37.036,2.729,3.905,4.306,3.968,3.431,4.425,0.656,6.749,0.138,-26.9,-21.352,9.204,5.209,0.055,2.371],[5568.0,0.112,8.732,2009.442,224.677,32.717,99.044,69.314,0.458,34.248,2.323,4.451,5.378,4.99,8.513,3.731,0.656,5.686,0.174,-26.558,-21.69,8.476,4.795,0.06,2.358],[5568.5,0.089,8.725,2013.137,193.774,33.431,96.947,53.391,0.535,29.083,1.873,5.161,6.969,8.559,25.741,2.514,0.656,3.83,0.247,-26.516,-26.339,6.664,3.769,0.056,2.329],[5569.0,0.074,8.724,2021.897,167.969,31.446,94.95,39.049,0.556,22.468,1.644,5.953,9.205,18.148,35.869,1.603,0.623,2.556,0.354,-26.042,-34.287,4.564,2.695,0.042,2.294],[5569.5,0.072,8.725,2027.346,152.67,26.967,93.667,30.171,0.566,17.277,1.75,6.55,11.42,40.302,25.596,1.086,0.592,1.835,0.416,-24.632,-41.537,3.104,1.943,0.032,2.284],[5570.0,0.072,8.755,2024.949,153.661,22.453,93.822,26.164,0.599,15.229,2.016,6.508,11.219,63.937,130.402,0.873,0.619,1.413,0.313,-22.268,-42.922,2.668,1.601,0.023,2.328],[5570.5,0.084,8.776,2020.82,173.175,19.468,95.706,25.233,0.626,16.678,2.12,5.775,8.103,73.021,664.185,0.907,0.7,1.31,0.14,-19.344,-33.677,2.907,1.549,0.004,2.416],[5571.0,0.078,8.78,2017.67,208.204,17.913,98.945,28.59,0.753,25.888,2.003,4.803,5.203,32.819,572.449,1.155,1.455,0.985,0.065,-16.248,-14.997,4.639,1.327,-0.003,2.472],[5571.5,0.073,8.769,2014.001,252.741,18.017,102.706,39.656,1.011,41.637,2.182,3.957,3.595,9.292,62.859,1.563,2.623,0.646,0.08,-12.678,1.684,8.281,1.201,0.023,2.434],[5572.0,0.154,8.865,2011.043,301.685,21.238,106.25,56.661,1.281,55.271,2.642,3.315,2.8,3.583,-2.304,1.74,2.771,0.64,0.148,-8.38,6.908,11.375,1.577,0.073,2.339],[5572.5,0.419,9.34,2011.529,353.706,27.704,109.354,71.783,1.439,62.987,2.961,2.827,2.39,2.14,1.517,1.699,2.304,0.741,0.21,-4.354,4.861,12.48,2.023,0.114,2.256],[5573.0,0.795,10.089,2014.193,408.597,34.115,112.103,79.192,1.471,65.468,3.067,2.447,2.148,1.744,1.687,1.661,2.096,0.794,0.216,-1.834,-0.712,12.479,2.213,0.137,2.222],[5573.5,0.457,10.548,2013.303,462.18,38.347,114.305,79.954,1.43,61.918,3.075,2.164,1.986,1.693,1.731,1.62,1.763,0.951,0.17,-0.691,-12.883,11.834,2.561,0.137,2.245],[5574.0,0.726,10.387,2006.522,505.868,40.915,115.352,78.047,1.359,55.654,3.048,1.977,1.873,1.827,1.847,1.562,1.331,1.201,0.11,-0.112,-28.065,10.842,3.08,0.112,2.306],[5574.5,0.439,10.174,1999.693,530.63,42.448,114.715,76.226,1.264,52.203,2.973,1.885,1.807,2.025,1.889,1.621,1.145,1.42,0.069,0.22,-34.715,10.425,3.383,0.082,2.365],[5575.0,0.41,10.109,1994.93,533.568,43.122,112.328,75.08,1.137,51.061,2.916,1.874,1.8,2.128,1.831,1.82,1.104,1.651,0.058,0.41,-30.463,10.51,3.534,0.065,2.388],[5575.5,0.36,10.165,1989.857,520.99,43.644,108.849,74.854,1.016,49.937,2.983,1.919,1.855,2.09,1.647,2.049,1.071,1.917,0.066,0.59,-24.911,10.581,3.669,0.069,2.375],[5576.0,0.316,10.326,1985.046,504.131,43.87,105.488,75.745,0.925,48.082,3.175,1.984,1.953,2.032,1.589,2.231,1.027,2.178,0.077,0.782,-22.904,10.523,3.805,0.095,2.358],[5576.5,0.306,10.371,1984.396,493.29,42.542,103.291,76.523,0.849,44.394,3.36,2.027,2.05,2.126,1.902,2.307,0.919,2.543,0.075,0.907,-22.087,10.002,4.066,0.116,2.364],[5577.0,0.244,9.941,1991.242,494.404,40.379,102.653,75.942,0.796,39.994,3.402,2.023,2.093,2.345,2.266,2.244,0.771,2.93,0.059,1.003,-20.34,9.125,4.41,0.113,2.394],[5577.5,0.274,9.557,1999.881,506.659,38.721,103.363,74.723,0.823,39.133,3.371,1.974,2.042,2.591,2.359,2.099,0.724,2.908,0.041,1.222,-14.39,8.799,4.515,0.097,2.432],[5578.0,0.243,9.535,1999.309,526.006,38.853,104.891,74.107,0.934,41.975,3.429,1.901,1.902,2.775,2.624,1.902,0.761,2.513,0.03,1.503,-7.178,9.005,4.396,0.074,2.457],[5578.5,0.28,9.58,1991.288,550.633,40.914,106.783,75.303,1.096,46.946,3.447,1.816,1.731,2.592,2.618,1.713,0.863,2.017,0.031,1.672,-4.967,9.542,4.127,0.052,2.451],[5579.0,0.427,9.687,1984.179,580.399,43.703,109.035,78.642,1.261,52.908,3.17,1.723,1.589,2.074,1.988,1.622,1.036,1.59,0.045,1.76,-5.672,10.433,3.762,0.051,2.408],[5579.5,0.538,10.024,1978.744,614.219,46.158,111.761,81.857,1.391,58.179,2.899,1.628,1.487,1.593,1.422,1.602,1.227,1.32,0.071,1.875,-4.469,11.382,3.46,0.068,2.343],[5580.0,0.667,10.403,1975.521,649.641,48.135,114.688,83.277,1.456,61.177,2.878,1.539,1.408,1.302,1.307,1.628,1.35,1.206,0.096,1.956,-1.745,12.1,3.327,0.103,2.287],[5580.5,1.289,10.614,1977.839,686.071,48.525,117.252,84.336,1.451,61.42,2.988,1.458,1.345,1.217,1.287,1.676,1.357,1.236,0.108,1.945,-4.116,12.414,3.396,0.138,2.256],[5581.0,0.824,10.594,1982.494,722.843,48.249,118.895,85.464,1.405,60.715,3.056,1.383,1.296,1.248,1.305,1.757,1.308,1.345,0.106,1.839,-13.511,12.59,3.575,0.152,2.25],[5581.5,0.82,10.333,1983.885,753.525,47.668,119.351,85.097,1.316,60.134,3.092,1.327,1.264,1.287,1.344,1.905,1.241,1.543,0.095,1.565,-20.6,12.755,3.822,0.147,2.263],[5582.0,0.795,10.169,1984.436,767.012,46.609,118.744,84.316,1.192,58.949,3.102,1.304,1.251,1.289,1.318,2.089,1.137,1.854,0.081,1.028,-17.384,12.674,4.154,0.128,2.288],[5582.5,0.574,9.914,1990.711,760.676,46.562,117.426,85.423,1.098,57.375,3.042,1.315,1.265,1.227,1.244,2.208,1.042,2.123,0.067,0.074,-7.83,12.373,4.41,0.097,2.319],[5583.0,0.618,9.737,1997.959,745.761,46.904,115.829,88.16,1.032,55.667,2.971,1.341,1.304,1.138,1.164,2.295,0.996,2.309,0.057,-1.394,1.111,12.084,4.505,0.077,2.346],[5583.5,0.55,9.814,1997.166,735.726,46.448,114.494,90.488,0.962,53.409,3.015,1.359,1.355,1.104,1.108,2.399,0.967,2.483,0.054,-3.187,3.181,11.779,4.521,0.078,2.356],[5584.0,0.495,9.596,1994.141,733.609,44.592,113.753,90.259,0.9,50.27,3.134,1.363,1.397,1.198,1.202,2.479,0.936,2.651,0.055,-5.263,-5.248,11.378,4.51,0.086,2.354],[5584.5,0.438,9.292,1999.375,731.761,42.156,113.194,85.257,0.776,45.268,3.166,1.367,1.416,1.404,1.575,2.72,0.898,3.038,0.059,-8.204,-16.442,10.633,4.395,0.101,2.344],[5585.0,0.332,9.166,2004.896,720.465,39.888,111.783,76.03,0.647,40.267,2.989,1.388,1.421,1.729,1.982,2.919,0.913,3.202,0.071,-12.316,-21.485,9.608,3.92,0.112,2.318],[5585.5,0.235,9.039,2004.433,693.128,38.526,108.813,66.57,0.611,36.228,2.687,1.443,1.445,2.169,2.393,2.694,0.933,2.885,0.097,-16.751,-23.339,8.404,3.339,0.1,2.274],[5586.0,0.101,9.009,2003.953,650.976,37.066,104.659,60.065,0.619,32.945,2.483,1.536,1.519,2.527,2.754,2.356,0.903,2.608,0.132,-20.29,-27.355,7.439,3.056,0.068,2.229],[5586.5,0.101,9.004,1998.757,601.905,34.348,100.433,55.795,0.643,31.35,2.496,1.661,1.66,2.742,2.96,2.156,0.88,2.45,0.162,-22.213,-29.779,7.069,2.981,0.038,2.203],[5587.0,0.107,9.008,1986.131,553.778,30.182,96.829,52.592,0.681,31.062,2.725,1.806,1.881,2.982,3.06,1.975,0.873,2.263,0.159,-21.837,-29.342,6.851,2.913,0.023,2.224],[5587.5,0.112,9.062,1982.97,509.374,26.082,93.639,51.254,0.73,31.709,3.084,1.963,2.179,3.462,3.434,1.8,0.89,2.026,0.108,-18.63,-31.892,6.698,2.795,0.015,2.307],[5588.0,0.132,9.178,1992.982,468.917,23.524,90.537,51.751,0.781,32.855,3.399,2.133,2.506,4.213,4.386,1.691,0.932,1.817,0.047,-13.558,-32.363,6.734,2.682,0.011,2.426],[5588.5,0.135,9.254,1997.564,433.059,22.887,87.82,51.929,0.814,33.697,3.444,2.309,2.792,4.833,4.474,1.653,0.981,1.687,0.018,-8.912,-25.299,6.868,2.6,0.015,2.512],[5589.0,0.142,9.271,1989.036,404.366,23.505,86.135,51.638,0.838,34.848,3.434,2.473,2.962,4.841,4.144,1.654,1.041,1.591,0.015,-6.14,-20.112,7.081,2.525,0.015,2.527],[5589.5,0.141,9.274,1978.249,386.855,25.347,85.763,52.349,0.896,37.404,3.5,2.585,2.963,4.33,3.574,1.631,1.105,1.479,0.025,-4.852,-22.826,7.452,2.502,0.007,2.498],[5590.0,0.141,9.277,1975.967,383.842,28.153,86.535,53.795,0.993,41.074,3.333,2.605,2.835,3.624,2.651,1.555,1.196,1.308,0.042,-4.212,-22.806,7.864,2.445,0.003,2.457],[5590.5,0.147,9.282,1980.796,395.585,29.959,88.319,54.934,1.077,43.822,3.026,2.528,2.661,2.789,2.079,1.472,1.292,1.142,0.062,-4.113,-10.665,8.088,2.326,0.017,2.415],[5591.0,0.152,9.263,1983.189,418.098,29.557,90.994,56.375,1.146,44.697,3.013,2.392,2.481,2.331,2.044,1.397,1.347,1.038,0.075,-4.623,3.37,8.157,2.248,0.043,2.386],[5591.5,0.14,9.241,1981.693,443.556,28.868,93.902,58.37,1.212,44.951,3.226,2.255,2.303,2.356,2.734,1.327,1.385,0.959,0.071,-5.023,2.767,8.207,2.2,0.05,2.386],[5592.0,0.123,9.239,1983.647,462.942,29.385,95.747,60.483,1.203,44.478,3.255,2.16,2.159,2.618,2.741,1.331,1.314,1.032,0.059,-4.832,-5.751,8.153,2.327,0.033,2.402],[5592.5,0.114,9.233,1989.739,470.546,29.757,95.676,63.137,1.072,42.142,3.0,2.125,2.097,2.657,2.162,1.451,1.079,1.398,0.061,-4.831,-2.783,7.88,2.772,0.017,2.396],[5593.0,0.107,9.236,1992.889,467.588,30.716,94.501,65.979,0.936,39.434,2.848,2.139,2.126,2.38,1.984,1.59,0.884,1.81,0.08,-5.627,8.177,7.587,3.194,0.023,2.363],[5593.5,0.107,9.234,1992.45,461.488,31.85,94.014,66.704,0.838,36.888,2.937,2.167,2.205,2.316,2.542,1.759,0.808,2.189,0.093,-6.262,8.624,7.472,3.438,0.036,2.344],[5594.0,0.107,9.229,1994.474,458.38,31.617,95.031,63.893,0.732,34.183,3.169,2.182,2.267,2.692,3.247,2.025,0.769,2.638,0.077,-5.405,-6.471,7.512,3.626,0.032,2.372],[5594.5,0.107,9.217,1998.455,456.228,29.811,96.564,59.542,0.67,33.351,3.366,2.192,2.278,3.403,3.142,2.242,0.769,2.914,0.048,-3.078,-20.812,7.665,3.698,0.015,2.427],[5595.0,0.107,9.206,2000.572,449.679,27.854,97.063,57.401,0.679,36.679,3.282,2.224,2.264,3.61,2.724,2.46,0.893,2.797,0.036,-0.514,-21.853,8.558,3.585,-0.003,2.456],[5595.5,0.108,9.225,2001.511,439.029,27.556,96.169,59.64,0.763,43.732,3.078,2.278,2.267,2.932,2.125,2.616,1.171,2.3,0.046,1.147,-16.318,10.194,3.272,0.005,2.434],[5596.0,0.158,9.223,2000.257,430.222,29.431,95.049,64.512,0.872,49.717,2.949,2.324,2.292,2.201,1.646,2.524,1.392,1.825,0.065,1.726,-12.323,11.206,2.993,0.041,2.4],[5596.5,0.129,9.173,1993.963,428.751,30.926,95.025,68.645,0.955,52.297,2.959,2.332,2.316,1.938,1.701,2.337,1.477,1.585,0.069,1.674,-10.126,11.367,2.858,0.064,2.393],[5597.0,0.121,9.13,1985.06,436.138,30.31,96.07,70.477,1.055,52.721,3.11,2.293,2.313,2.055,2.048,2.076,1.498,1.385,0.055,1.525,-10.819,11.108,2.752,0.057,2.417],[5597.5,0.122,9.144,1980.819,448.662,29.165,97.078,71.064,1.175,51.844,3.331,2.229,2.275,2.349,2.014,1.769,1.469,1.202,0.04,1.557,-12.948,10.549,2.663,0.049,2.447],[5598.0,0.101,9.094,1981.75,459.199,29.117,97.184,70.556,1.28,50.713,3.396,2.178,2.214,2.62,2.139,1.52,1.411,1.075,0.034,1.757,-13.675,9.888,2.6,0.047,2.46],[5598.5,0.089,9.013,1982.977,463.092,30.349,96.431,69.706,1.328,50.394,3.265,2.159,2.155,2.667,2.138,1.408,1.401,1.006,0.037,1.953,-16.418,9.55,2.531,0.049,2.45],[5599.0,0.458,9.004,1982.265,461.677,31.699,95.512,69.363,1.333,50.496,3.121,2.166,2.125,2.467,1.795,1.387,1.413,0.981,0.047,1.979,-19.783,9.437,2.478,0.064,2.429],[5599.5,0.077,9.008,1984.685,459.562,32.811,95.164,69.588,1.311,49.696,3.153,2.176,2.124,2.161,1.784,1.382,1.368,1.012,0.055,1.839,-17.897,9.249,2.51,0.085,2.412],[5600.0,0.148,9.032,1987.868,461.016,34.224,95.598,69.67,1.24,47.953,3.324,2.169,2.131,1.976,1.641,1.458,1.309,1.115,0.055,1.776,-13.335,9.212,2.612,0.09,2.411],[5600.5,0.063,9.053,1986.951,468.595,34.834,96.544,69.211,1.168,46.645,3.472,2.134,2.12,2.002,1.802,1.571,1.294,1.214,0.049,1.893,-11.521,9.355,2.683,0.084,2.421],[5601.0,0.067,9.069,1987.9,481.693,35.418,97.621,67.618,1.117,46.727,3.463,2.076,2.075,2.164,1.872,1.693,1.328,1.274,0.044,1.945,-12.855,9.649,2.697,0.077,2.43],[5601.5,0.079,9.096,1992.577,496.396,36.011,98.562,65.968,1.055,49.229,3.327,2.015,2.002,2.266,1.915,1.98,1.536,1.294,0.043,1.778,-14.081,10.597,2.589,0.071,2.43],[5602.0,0.098,9.152,1989.712,509.495,35.832,99.306,66.134,1.001,54.27,3.166,1.963,1.916,2.251,2.037,2.427,2.07,1.194,0.048,1.534,-15.801,12.368,2.268,0.079,2.416],[5602.5,0.234,9.366,1978.541,520.854,35.032,100.059,68.275,0.997,58.363,3.084,1.92,1.841,2.036,1.628,2.674,2.554,1.052,0.058,1.339,-16.431,13.621,1.986,0.094,2.391],[5603.0,0.18,9.61,1971.718,531.627,34.206,101.171,70.278,1.038,59.714,3.112,1.881,1.783,1.811,1.423,2.595,2.618,0.991,0.068,1.098,-12.792,13.729,1.949,0.095,2.367],[5603.5,0.248,9.716,1976.084,542.346,34.802,102.713,70.522,1.11,59.562,3.189,1.844,1.737,1.661,1.41,2.327,2.354,0.989,0.074,0.715,-10.573,13.135,2.081,0.087,2.354],[5604.0,0.412,9.815,1987.056,552.106,37.84,104.241,70.133,1.167,57.438,3.251,1.811,1.705,1.621,1.429,2.022,1.918,1.069,0.074,0.321,-16.682,12.029,2.367,0.087,2.351],[5604.5,0.271,9.936,1995.124,558.236,42.416,104.859,70.596,1.176,52.468,3.249,1.791,1.697,1.657,1.439,1.747,1.417,1.271,0.074,0.109,-24.847,10.496,2.83,0.084,2.35],[5605.0,0.446,10.015,1997.663,557.929,45.449,104.056,71.077,1.141,45.704,3.212,1.792,1.725,1.678,1.525,1.477,0.962,1.587,0.076,-0.032,-27.713,8.629,3.418,0.08,2.346],[5605.5,0.378,10.031,2000.092,550.714,45.497,102.494,70.3,1.102,41.172,3.161,1.816,1.788,1.667,1.49,1.317,0.728,1.817,0.08,-0.412,-29.412,7.412,3.789,0.086,2.341],[5606.0,0.355,10.097,2004.384,538.348,44.017,101.286,68.56,1.055,39.528,3.131,1.858,1.875,1.686,1.515,1.325,0.692,1.916,0.079,-1.059,-32.822,7.131,3.828,0.102,2.346],[5606.5,0.341,10.14,2007.001,521.792,42.004,100.701,67.799,0.997,38.952,3.139,1.916,1.96,1.819,1.628,1.425,0.729,1.956,0.072,-1.649,-35.268,7.238,3.695,0.109,2.363],[5607.0,0.302,9.746,2005.418,498.992,40.253,99.888,68.533,0.997,39.284,3.14,2.004,2.036,2.062,1.996,1.461,0.795,1.844,0.065,-1.868,-35.198,7.43,3.475,0.093,2.381],[5607.5,0.165,9.395,2002.609,468.933,39.156,97.799,69.269,1.05,40.082,3.159,2.133,2.138,2.25,2.121,1.391,0.862,1.623,0.065,-1.869,-36.445,7.451,3.218,0.087,2.39],[5608.0,0.161,9.712,2005.021,436.44,38.309,94.447,68.543,1.116,41.197,3.221,2.291,2.309,2.192,1.706,1.298,0.926,1.405,0.065,-2.115,-37.511,7.388,2.964,0.11,2.397],[5608.5,0.478,10.198,2011.532,409.522,36.294,91.29,65.792,1.179,42.144,3.256,2.442,2.534,2.053,2.302,1.218,0.985,1.24,0.054,-2.595,-32.289,7.328,2.765,0.111,2.426],[5609.0,0.107,9.639,2017.698,395.141,33.774,89.831,61.518,1.255,43.608,3.192,2.531,2.712,2.263,3.052,1.178,1.216,1.012,0.031,-2.802,-26.157,7.551,2.384,0.067,2.478],[5609.5,0.146,9.112,2022.412,397.568,31.49,90.167,58.204,1.34,45.197,3.092,2.515,2.713,3.045,2.871,1.17,1.63,0.745,0.015,-2.559,-22.546,8.01,1.878,0.026,2.528],[5610.0,0.228,9.145,2026.697,415.173,30.595,91.558,57.284,1.384,46.242,2.895,2.409,2.51,4.065,3.453,1.174,1.896,0.621,0.014,-2.175,-13.345,8.293,1.628,0.003,2.531],[5610.5,0.244,9.174,2027.986,440.891,32.077,93.597,57.999,1.396,47.188,2.67,2.268,2.208,4.268,3.523,1.166,1.73,0.688,0.036,-1.992,-2.111,8.309,1.821,-0.005,2.457],[5611.0,0.318,9.359,2020.603,467.37,35.343,96.505,61.027,1.426,48.522,2.692,2.14,1.935,3.344,2.288,1.139,1.36,0.851,0.1,-2.152,-1.909,8.288,2.305,0.043,2.332],[5611.5,1.307,9.847,2008.263,488.281,39.425,100.441,67.098,1.454,51.017,2.898,2.048,1.758,2.149,1.437,1.158,1.166,0.997,0.194,-2.566,-7.075,8.598,2.749,0.111,2.209],[5612.0,0.791,10.267,2001.21,498.501,41.98,104.455,74.317,1.431,54.609,3.027,2.006,1.691,1.5,1.248,1.297,1.172,1.105,0.285,-2.995,-8.202,9.456,2.992,0.145,2.117],[5612.5,1.387,10.561,2001.021,496.851,47.655,106.661,79.49,1.388,56.686,3.122,2.013,1.73,1.215,1.136,1.42,1.198,1.189,0.359,-3.515,-9.447,10.062,3.123,0.172,2.058],[5613.0,0.853,10.322,1999.279,484.948,51.788,105.992,80.233,1.378,56.656,3.342,2.062,1.884,1.14,1.18,1.472,1.287,1.152,0.375,-4.585,-11.664,10.355,3.007,0.223,2.052],[5613.5,1.701,10.304,1991.471,464.756,50.271,103.115,76.193,1.316,54.19,3.623,2.152,2.155,1.29,1.696,1.542,1.449,1.069,0.286,-6.431,-9.325,10.347,2.661,0.233,2.133],[5614.0,0.835,10.197,1978.887,438.317,44.082,99.34,69.944,1.161,47.896,3.724,2.281,2.51,1.674,3.088,1.578,1.378,1.158,0.136,-8.655,-4.555,9.357,2.525,0.164,2.294],[5614.5,0.373,9.427,1968.835,406.825,37.503,95.48,64.452,1.023,40.813,3.349,2.458,2.87,2.522,5.286,1.521,1.135,1.356,0.039,-10.92,-2.835,7.968,2.614,0.072,2.459],[5615.0,0.25,9.13,1970.277,371.817,32.107,91.829,59.597,0.937,36.428,2.745,2.689,3.145,4.441,6.263,1.479,0.996,1.492,0.013,-13.695,0.076,7.079,2.641,0.029,2.538],[5615.5,0.243,9.136,1980.121,338.356,28.772,88.834,54.953,0.907,34.957,2.401,2.955,3.284,6.798,6.593,1.483,1.002,1.488,0.02,-17.593,6.826,6.866,2.555,0.017,2.52],[5616.0,0.244,9.166,1986.419,314.014,28.316,87.541,51.665,0.911,35.83,2.243,3.185,3.297,6.631,4.901,1.574,1.216,1.321,0.046,-22.069,14.429,7.318,2.281,0.024,2.467],[5616.5,0.248,9.232,1982.582,304.738,29.551,88.966,50.458,0.916,37.524,2.169,3.282,3.219,5.168,3.693,1.7,1.505,1.136,0.075,-25.812,18.482,7.953,1.972,0.037,2.422],[5617.0,0.228,9.294,1976.79,311.452,30.908,92.912,49.96,0.939,40.207,2.301,3.211,3.078,4.293,3.838,1.767,1.782,1.005,0.097,-28.194,14.798,8.483,1.785,0.044,2.391],[5617.5,0.265,9.341,1976.105,328.796,32.052,97.78,50.538,0.966,45.554,2.586,3.041,2.907,4.156,4.167,1.921,2.096,0.929,0.109,-29.606,2.883,9.484,1.7,0.052,2.37],[5618.0,0.277,9.417,1978.456,346.661,32.752,101.779,54.148,0.947,48.647,2.843,2.885,2.758,4.247,3.795,2.197,1.813,1.341,0.113,-30.541,-10.474,10.594,2.373,0.064,2.358],[5618.5,0.286,9.626,1980.006,354.076,32.876,104.109,62.207,0.929,49.055,3.033,2.824,2.692,3.739,2.517,2.248,1.21,1.92,0.113,-30.611,-20.128,10.647,3.375,0.071,2.354],[5619.0,0.286,9.765,1978.655,345.254,33.827,104.994,72.125,0.956,50.179,3.087,2.896,2.767,2.902,1.963,2.088,0.994,2.103,0.116,-29.05,-24.177,10.18,3.811,0.08,2.354],[5619.5,0.299,9.645,1974.095,322.742,35.984,105.053,78.299,0.969,49.268,2.853,3.098,3.024,2.289,2.307,2.019,0.953,2.118,0.126,-26.976,-24.744,9.994,3.89,0.102,2.352],[5620.0,0.191,9.543,1966.695,293.944,37.441,104.826,76.03,0.952,44.359,2.435,3.402,3.477,2.193,3.267,1.891,0.953,1.988,0.143,-27.667,-25.825,9.195,3.594,0.122,2.347],[5620.5,0.092,9.463,1960.719,265.23,36.557,104.563,65.666,0.808,35.692,2.036,3.77,4.099,2.753,5.115,1.792,0.925,1.957,0.174,-32.83,-27.005,7.387,2.961,0.109,2.334],[5621.0,0.072,9.194,1958.549,238.449,34.274,104.339,51.679,0.609,26.188,1.823,4.194,4.828,4.138,8.242,1.76,0.817,2.18,0.234,-40.559,-25.738,5.497,2.481,0.065,2.305],[5621.5,0.078,8.84,1958.373,211.707,33.131,104.306,40.233,0.517,20.536,1.832,4.724,5.614,7.201,10.763,1.716,0.722,2.383,0.335,-48.018,-22.978,4.526,2.326,0.02,2.264],[5622.0,0.096,8.77,1957.612,182.832,32.973,104.745,36.082,0.506,20.53,1.99,5.47,6.43,12.548,9.635,1.759,0.73,2.411,0.498,-53.997,-22.478,4.542,2.309,-0.004,2.217],[5622.5,0.213,8.751,1958.611,153.675,33.293,105.742,38.786,0.571,25.066,2.204,6.507,7.233,15.47,7.78,1.863,0.838,2.24,0.747,-58.532,-27.671,5.432,2.397,-0.001,2.167],[5623.0,0.269,8.733,1961.537,130.61,34.572,106.843,43.97,0.657,30.504,2.391,7.656,7.946,13.852,7.724,1.923,0.951,2.029,1.054,-61.949,-37.282,6.43,2.507,0.014,2.125],[5623.5,0.263,8.737,1965.387,117.361,36.542,107.102,46.894,0.671,33.205,2.47,8.521,8.56,12.043,7.645,1.993,0.997,1.999,1.305,-64.398,-43.665,6.822,2.54,0.027,2.098],[5624.0,0.336,8.739,1968.372,110.448,37.804,105.594,47.34,0.642,33.259,2.444,9.054,9.24,11.008,7.338,2.125,1.007,2.111,1.437,-65.793,-45.334,6.953,2.562,0.034,2.089],[5624.5,0.367,8.725,1965.577,104.77,36.733,102.118,48.339,0.625,32.648,2.453,9.545,10.281,11.069,7.12,2.209,1.021,2.166,1.472,-66.3,-45.91,7.053,2.565,0.043,2.096],[5625.0,0.244,8.715,1957.172,98.866,34.705,97.359,49.96,0.609,31.41,2.595,10.115,11.988,11.959,7.791,2.268,1.019,2.227,1.368,-66.409,-46.496,7.051,2.568,0.057,2.129],[5625.5,0.203,8.718,1954.839,92.362,35.424,92.319,49.023,0.615,28.6,2.797,10.827,14.639,14.944,12.567,2.075,0.92,2.25,1.077,-66.438,-42.007,6.477,2.618,0.068,2.198],[5626.0,0.187,8.719,1962.938,85.069,34.604,87.619,45.321,0.651,26.076,2.953,11.755,18.051,23.745,26.414,1.718,0.796,2.153,0.677,-66.464,-30.475,5.686,2.654,0.069,2.3],[5626.5,0.187,8.735,1971.242,77.534,30.8,83.435,43.538,0.667,26.203,3.085,12.898,20.867,41.52,46.442,1.587,0.733,2.177,0.347,-66.507,-16.479,5.404,2.75,0.068,2.404],[5627.0,0.118,8.778,1973.091,71.196,26.34,80.18,46.573,0.696,28.053,3.076,14.046,21.596,56.368,40.477,1.58,0.648,2.469,0.193,-66.527,-4.431,5.616,3.27,0.068,2.47],[5627.5,0.101,8.802,1970.762,68.357,24.858,78.773,53.28,0.761,30.945,2.805,14.629,20.352,47.596,26.98,1.55,0.594,2.612,0.186,-66.479,2.589,6.016,3.76,0.064,2.476],[5628.0,0.09,8.81,1967.191,70.742,26.636,80.103,59.609,0.793,33.374,2.59,14.136,17.701,34.398,26.321,1.538,0.607,2.535,0.272,-66.367,3.739,6.23,3.809,0.056,2.44],[5628.5,0.078,8.813,1963.969,78.143,28.152,84.199,62.883,0.794,33.682,2.527,12.797,14.457,27.407,26.333,1.559,0.619,2.52,0.386,-66.277,1.537,6.323,3.793,0.047,2.391],[5629.0,0.073,8.811,1967.278,89.0,29.178,89.916,63.234,0.759,33.034,2.425,11.236,11.541,23.395,18.765,1.735,0.664,2.606,0.516,-66.298,-3.588,6.671,3.735,0.033,2.335],[5629.5,0.075,8.819,1978.697,100.95,31.346,95.634,62.242,0.696,32.427,2.261,9.906,9.429,18.835,9.548,2.017,0.729,2.762,0.712,-66.422,-13.031,7.137,3.636,0.019,2.262],[5630.0,0.089,8.841,1989.485,110.657,33.709,100.204,61.159,0.662,32.839,2.195,9.037,8.169,14.225,7.336,2.205,0.773,2.851,0.963,-66.595,-20.701,7.447,3.576,0.019,2.184],[5630.5,0.103,8.84,1996.586,115.329,35.249,103.138,61.3,0.639,34.281,2.224,8.671,7.655,10.97,6.293,2.37,0.81,2.925,1.172,-66.779,-23.839,7.726,3.541,0.026,2.129],[5631.0,0.101,8.801,2002.937,115.139,37.888,104.084,62.214,0.612,34.883,2.243,8.685,7.77,9.139,6.779,2.505,0.825,3.036,1.287,-66.905,-29.454,7.826,3.52,0.03,2.107],[5631.5,0.084,8.779,2003.852,112.558,40.217,102.66,62.046,0.58,34.136,2.29,8.884,8.401,8.769,7.329,2.608,0.83,3.142,1.297,-66.833,-39.87,7.721,3.453,0.032,2.11],[5632.0,0.084,8.776,1996.448,109.878,40.266,99.042,60.039,0.563,33.647,2.471,9.101,9.387,10.088,9.182,2.688,0.882,3.057,1.145,-66.513,-44.569,7.726,3.261,0.035,2.146],[5632.5,0.084,8.772,1987.841,108.704,37.519,94.378,56.603,0.571,34.466,2.766,9.199,10.4,13.635,10.767,2.734,1.022,2.707,0.789,-65.996,-36.951,7.966,2.922,0.039,2.229],[5633.0,0.089,8.784,1979.502,110.455,33.018,90.335,52.684,0.584,35.397,3.061,9.053,10.781,21.091,19.913,2.75,1.183,2.342,0.376,-65.36,-24.144,8.205,2.588,0.042,2.349],[5633.5,0.095,8.779,1970.695,115.797,28.564,88.101,49.98,0.639,36.831,3.143,8.636,9.96,34.847,29.646,2.604,1.469,1.851,0.14,-64.706,-15.05,8.451,2.195,0.031,2.456],[5634.0,0.101,8.786,1965.795,123.862,25.175,87.856,48.925,0.72,38.636,2.823,8.073,8.422,38.795,13.642,2.38,1.851,1.319,0.114,-64.16,-10.512,8.716,1.774,0.001,2.468],[5634.5,0.112,8.793,1967.67,132.588,23.922,88.883,48.398,0.761,38.909,2.535,7.542,7.017,22.253,4.59,2.261,2.004,1.129,0.258,-63.792,-10.281,8.783,1.627,-0.014,2.375],[5635.0,0.125,8.786,1976.221,139.632,26.719,90.234,49.016,0.757,38.406,2.442,7.162,6.099,11.219,5.012,2.239,1.916,1.178,0.527,-63.651,-12.558,8.658,1.686,0.0,2.257],[5635.5,0.192,8.784,1985.316,143.786,31.918,91.619,52.014,0.695,36.635,2.328,6.955,5.694,7.334,4.926,2.326,1.56,1.569,0.757,-63.755,-11.23,8.227,2.023,0.012,2.179],[5636.0,0.208,8.784,1987.93,145.19,34.217,93.333,54.878,0.611,34.121,2.235,6.888,5.774,6.285,5.177,2.455,1.168,2.147,0.844,-64.066,11.916,7.645,2.46,0.019,2.152],[5636.5,0.209,8.781,1987.794,143.826,35.826,95.508,54.46,0.553,31.84,2.392,6.953,6.316,6.643,9.986,2.527,0.98,2.605,0.764,-64.527,55.515,7.127,2.71,0.04,2.177],[5637.0,0.243,8.764,1992.778,138.231,36.116,97.73,51.484,0.503,28.781,2.791,7.234,7.313,9.035,39.444,2.589,0.854,3.059,0.537,-65.082,74.831,6.642,2.898,0.064,2.256],[5637.5,0.164,8.745,2001.755,126.801,33.419,99.214,48.271,0.503,26.759,2.866,7.886,8.788,15.07,48.539,2.426,0.844,2.93,0.335,-65.732,48.858,6.182,2.752,0.059,2.346],[5638.0,0.112,8.757,2010.75,109.873,29.493,99.573,45.629,0.56,26.647,2.195,9.101,10.833,23.741,19.103,2.067,0.964,2.199,0.37,-66.587,16.832,5.86,2.288,0.023,2.352],[5638.5,0.112,8.791,2016.465,90.127,27.33,99.649,43.157,0.589,26.76,1.846,11.095,13.567,27.725,11.603,1.917,1.046,1.833,0.798,-67.609,7.9,5.769,2.047,-0.003,2.262],[5639.0,0.279,8.817,2019.532,72.143,28.095,100.94,41.042,0.589,27.366,1.911,13.861,16.683,23.576,12.689,1.925,1.091,1.768,1.643,-68.602,5.506,5.792,1.974,0.006,2.16],[5639.5,0.43,8.823,2021.929,60.3,31.554,103.876,40.194,0.601,27.774,2.084,16.584,19.063,19.88,11.818,1.862,1.11,1.679,2.572,-69.445,1.561,5.711,1.911,0.033,2.095],[5640.0,0.65,8.823,2018.772,55.724,35.137,107.347,40.675,0.601,26.417,2.26,17.946,19.634,18.18,10.292,1.751,1.017,1.731,3.185,-70.128,0.223,5.381,1.971,0.064,2.059],[5640.5,0.537,8.808,2009.169,56.271,37.219,109.688,40.234,0.584,24.385,2.405,17.771,18.658,17.099,10.747,1.664,0.872,1.921,3.399,-70.636,2.098,4.96,2.121,0.089,2.038],[5641.0,1.149,8.78,2001.093,59.992,38.3,109.9,38.589,0.566,22.887,2.483,16.669,17.148,16.273,9.697,1.594,0.754,2.124,3.285,-70.971,7.508,4.607,2.276,0.113,2.029],[5641.5,0.852,8.822,1999.149,65.976,37.585,108.391,38.23,0.556,21.945,2.471,15.157,15.628,15.715,9.365,1.53,0.672,2.282,2.909,-71.141,15.274,4.346,2.404,0.123,2.037],[5642.0,0.37,8.816,2002.566,72.925,37.56,106.493,39.286,0.543,21.595,2.433,13.713,14.013,15.858,10.572,1.515,0.62,2.45,2.343,-71.097,15.244,4.202,2.519,0.099,2.069],[5642.5,0.422,8.725,2010.701,79.61,38.811,105.118,40.161,0.528,21.815,2.427,12.561,12.385,16.49,13.247,1.546,0.583,2.653,1.677,-70.806,1.598,4.166,2.652,0.066,2.132],[5643.0,0.446,8.71,2016.035,85.382,38.556,104.153,41.023,0.52,22.065,2.373,11.712,11.104,16.444,17.88,1.57,0.565,2.778,1.126,-70.325,-14.846,4.166,2.736,0.048,2.206],[5643.5,0.396,8.727,2010.805,89.32,36.418,103.056,42.11,0.496,21.666,2.294,11.196,10.345,14.81,13.484,1.672,0.585,2.852,0.927,-69.784,-20.932,4.214,2.676,0.032,2.235],[5644.0,0.437,8.725,1998.577,90.597,36.025,101.607,42.472,0.444,20.568,2.414,11.038,10.089,12.664,9.346,1.881,0.633,2.967,1.106,-69.342,-15.32,4.23,2.486,0.023,2.197],[5644.5,0.371,8.719,1986.995,89.943,36.749,100.176,40.688,0.428,19.68,2.619,11.118,10.266,11.459,10.066,1.892,0.679,2.798,1.453,-69.039,-8.394,4.124,2.26,0.04,2.137],[5645.0,0.322,8.719,1978.152,88.959,36.117,99.366,36.77,0.472,19.886,2.697,11.241,10.768,11.393,10.321,1.681,0.754,2.261,1.641,-68.685,-7.837,4.029,1.999,0.054,2.11],[5645.5,0.221,8.711,1972.27,88.898,35.054,99.234,33.309,0.511,20.263,2.628,11.249,11.368,12.375,10.515,1.53,0.833,1.85,1.565,-68.042,-7.616,3.982,1.781,0.052,2.122],[5646.0,0.209,8.715,1969.482,90.785,33.514,99.163,31.272,0.563,21.011,2.56,11.015,11.679,14.05,10.271,1.409,1.004,1.46,1.353,-67.261,-2.889,4.031,1.529,0.053,2.152],[5646.5,0.2,8.773,1970.677,95.809,31.399,98.706,29.315,0.627,21.951,2.642,10.437,11.293,15.884,13.713,1.282,1.205,1.08,1.106,-66.612,0.473,4.097,1.273,0.061,2.185],[5647.0,0.207,8.87,1972.329,105.161,29.518,98.303,28.182,0.663,22.073,2.71,9.509,10.125,17.151,21.261,1.18,1.199,0.985,0.871,-65.994,2.285,3.988,1.238,0.052,2.215],[5647.5,0.192,8.916,1968.718,118.616,29.82,98.798,29.286,0.7,22.099,2.621,8.431,8.565,16.87,15.071,1.037,1.017,1.024,0.726,-65.111,8.721,3.69,1.36,0.023,2.228],[5648.0,0.164,8.918,1960.948,133.895,31.505,100.25,31.744,0.772,22.947,2.542,7.469,7.144,14.706,6.003,0.88,0.871,1.012,0.714,-63.935,16.484,3.444,1.473,0.004,2.207],[5648.5,0.17,8.918,1958.291,148.215,33.432,101.651,34.536,0.868,25.276,2.532,6.747,6.131,11.185,4.876,0.812,0.948,0.872,0.78,-62.543,17.757,3.594,1.419,0.006,2.166],[5649.0,0.188,8.919,1961.455,160.054,34.474,101.806,37.427,0.925,28.27,2.467,6.248,5.544,8.173,5.026,0.875,1.274,0.705,0.835,-60.927,15.195,4.138,1.235,0.017,2.132],[5649.5,0.175,8.916,1964.656,169.524,33.243,100.343,39.566,0.933,30.887,2.493,5.899,5.315,6.665,4.879,1.015,1.796,0.58,0.799,-59.142,11.538,4.833,1.025,0.041,2.129],[5650.0,0.154,8.954,1967.548,177.58,32.317,97.908,40.277,0.902,32.104,2.753,5.631,5.373,6.253,7.328,1.198,2.296,0.522,0.629,-57.312,4.084,5.492,0.892,0.074,2.173],[5650.5,0.163,8.992,1967.483,185.394,28.597,95.478,39.537,0.813,31.419,3.209,5.394,5.599,6.812,19.915,1.489,2.579,0.575,0.371,-55.398,-8.648,6.091,0.876,0.088,2.27],[5651.0,0.124,9.01,1961.524,194.611,21.373,93.579,38.229,0.693,30.129,3.65,5.138,5.774,8.878,25.497,1.881,2.591,0.731,0.157,-53.217,-18.462,6.557,0.943,0.073,2.389],[5651.5,0.118,9.012,1958.944,206.258,19.133,92.218,38.066,0.601,28.463,3.801,4.848,5.658,12.477,18.661,2.114,2.049,1.141,0.056,-50.721,-19.563,6.483,1.271,0.048,2.483],[5652.0,0.118,8.996,1963.209,219.906,19.119,91.401,39.591,0.547,26.564,3.54,4.547,5.198,13.977,20.218,2.165,1.306,1.785,0.032,-48.156,-15.697,6.04,1.822,0.027,2.518],[5652.5,0.113,9.0,1963.355,235.062,19.432,91.393,41.896,0.531,26.687,3.068,4.254,4.559,10.833,12.482,2.187,1.036,2.113,0.048,-45.937,-6.904,5.929,2.125,0.014,2.487],[5653.0,0.107,9.008,1957.11,252.204,22.688,92.494,44.354,0.574,29.421,2.774,3.965,3.959,6.958,4.675,2.169,1.109,1.971,0.097,-44.277,8.148,6.347,2.127,0.021,2.415],[5653.5,0.111,9.002,1955.268,271.486,27.404,94.621,46.884,0.67,33.696,2.76,3.683,3.514,5.038,4.403,2.082,1.288,1.64,0.157,-43.089,18.779,7.095,2.05,0.033,2.345],[5654.0,0.112,9.008,1963.516,291.799,30.243,97.097,49.139,0.759,37.08,2.827,3.427,3.233,4.276,3.805,1.946,1.402,1.389,0.19,-42.14,12.87,7.531,1.994,0.037,2.306],[5654.5,0.124,9.015,1974.339,311.048,31.4,98.966,50.614,0.797,37.382,2.858,3.215,3.082,4.036,3.646,1.809,1.351,1.339,0.189,-41.281,-1.972,7.35,2.021,0.038,2.296],[5655.0,0.129,9.003,1976.277,327.537,32.051,99.518,50.566,0.827,34.855,2.871,3.053,3.026,4.104,3.702,1.57,1.149,1.372,0.171,-40.489,-10.357,6.611,2.152,0.042,2.304],[5655.5,0.129,9.001,1968.359,339.959,31.6,98.69,49.416,0.839,30.918,2.928,2.942,3.029,4.429,4.034,1.299,0.875,1.506,0.149,-39.61,-6.718,5.572,2.393,0.04,2.32],[5656.0,0.166,9.01,1961.127,347.523,30.023,97.046,48.912,0.821,27.293,2.988,2.878,3.05,4.881,4.464,1.1,0.65,1.718,0.131,-38.373,-2.069,4.622,2.671,0.03,2.337],[5656.5,0.161,9.011,1958.323,351.413,28.55,95.444,48.983,0.797,24.524,3.097,2.846,3.056,5.061,3.863,0.94,0.489,1.946,0.12,-37.011,-3.676,3.834,2.935,0.027,2.347],[5657.0,0.158,9.008,1956.385,354.771,28.33,94.544,48.023,0.769,22.412,3.199,2.819,3.019,4.843,4.796,0.831,0.385,2.177,0.109,-36.063,-2.132,3.268,3.169,0.035,2.359],[5657.5,0.151,9.002,1954.925,359.573,28.591,94.344,45.806,0.741,21.266,3.128,2.781,2.927,4.644,7.098,0.811,0.344,2.359,0.094,-35.488,5.866,3.068,3.312,0.041,2.377],[5658.0,0.092,8.995,1958.367,364.912,28.543,94.261,44.337,0.716,21.995,2.958,2.74,2.812,4.59,5.437,0.946,0.392,2.415,0.093,-34.872,12.223,3.447,3.275,0.033,2.377],[5658.5,0.086,8.996,1967.257,368.566,29.467,93.805,45.211,0.686,26.123,2.927,2.713,2.733,4.42,3.762,1.402,0.652,2.23,0.117,-34.243,13.233,4.867,2.908,0.022,2.344],[5659.0,0.091,9.005,1975.634,370.008,30.504,93.087,47.471,0.655,31.273,2.996,2.703,2.724,4.069,3.709,1.985,1.037,1.95,0.141,-33.713,8.841,6.6,2.424,0.023,2.316],[5659.5,0.101,9.006,1979.416,370.649,31.565,92.453,49.104,0.629,34.425,3.031,2.698,2.769,3.958,5.255,2.441,1.408,1.752,0.125,-32.752,-1.686,7.816,2.09,0.025,2.334],[5660.0,0.101,9.039,1980.73,370.86,32.406,91.931,49.414,0.61,36.178,2.951,2.696,2.827,4.277,6.185,2.799,1.747,1.609,0.083,-30.513,-14.026,8.707,1.86,0.016,2.388],[5660.5,0.101,9.141,1985.824,368.456,29.32,91.168,48.538,0.604,36.375,2.811,2.714,2.882,4.607,4.206,2.89,1.777,1.635,0.055,-27.212,-21.625,8.916,1.872,0.012,2.435],[5661.0,0.125,9.227,1991.457,361.032,25.699,89.951,47.642,0.606,35.972,2.88,2.77,2.95,4.466,4.633,2.798,1.569,1.795,0.044,-24.286,-17.756,8.651,2.061,0.027,2.458],[5661.5,0.135,9.281,1988.157,349.391,24.623,88.729,47.998,0.615,35.197,3.114,2.862,3.027,4.119,5.254,2.598,1.3,2.026,0.039,-22.398,-6.386,8.151,2.365,0.035,2.47],[5662.0,0.153,9.347,1977.365,337.762,24.595,88.312,49.68,0.635,33.949,3.218,2.961,3.092,3.945,4.146,2.321,1.026,2.278,0.039,-20.59,-0.324,7.516,2.744,0.017,2.475],[5662.5,0.165,9.408,1972.13,329.693,25.063,88.975,52.326,0.658,33.553,3.14,3.033,3.143,3.844,3.625,2.148,0.905,2.375,0.043,-17.965,-2.655,7.213,2.962,0.006,2.467],[5663.0,0.201,9.413,1975.769,325.061,26.697,90.124,54.955,0.685,33.763,3.058,3.076,3.199,3.636,2.983,2.053,0.881,2.33,0.047,-14.835,-3.488,7.18,3.024,0.02,2.461],[5663.5,0.2,9.398,1982.827,321.533,28.101,90.919,56.004,0.714,33.718,3.032,3.11,3.264,3.435,2.997,1.962,0.865,2.268,0.045,-11.971,2.724,7.148,3.068,0.03,2.466],[5664.0,0.179,9.39,1989.333,317.711,28.493,91.097,56.092,0.737,33.689,3.077,3.148,3.325,3.336,3.281,1.893,0.848,2.233,0.038,-9.809,10.725,7.122,3.118,0.03,2.48],[5664.5,0.152,9.391,1990.361,314.566,28.611,91.11,56.768,0.763,34.285,3.102,3.179,3.372,3.347,3.034,1.835,0.829,2.214,0.031,-8.348,12.294,7.14,3.199,0.04,2.497],[5665.0,0.138,9.404,1984.666,314.436,28.567,91.494,57.82,0.809,34.854,2.928,3.18,3.381,3.41,3.274,1.715,0.773,2.219,0.023,-7.434,8.524,7.073,3.404,0.051,2.518],[5665.5,0.157,9.375,1978.252,318.528,28.345,92.39,58.562,0.824,34.439,2.664,3.139,3.324,3.544,3.627,1.64,0.704,2.334,0.019,-6.971,4.292,6.898,3.641,0.049,2.527],[5666.0,0.206,9.37,1978.137,325.757,28.928,93.624,59.056,0.824,34.025,2.56,3.07,3.203,3.694,3.187,1.61,0.658,2.448,0.027,-7.021,0.809,6.755,3.815,0.045,2.504],[5666.5,0.314,9.445,1983.661,334.404,29.954,94.998,59.57,0.907,35.076,2.606,2.99,3.064,3.533,2.735,1.461,0.626,2.33,0.052,-7.683,-1.262,6.709,3.979,0.063,2.45],[5667.0,0.407,9.619,1989.004,343.259,33.05,96.474,60.619,0.986,36.906,2.674,2.913,2.953,3.067,2.417,1.367,0.631,2.167,0.094,-8.919,-0.391,6.884,4.047,0.103,2.383],[5667.5,0.403,9.928,1993.071,352.324,37.756,98.007,63.106,0.995,38.493,2.683,2.838,2.876,2.644,2.288,1.436,0.677,2.121,0.14,-10.591,2.702,7.288,3.997,0.144,2.324],[5668.0,0.414,10.0,1999.964,363.278,40.223,99.349,66.28,0.965,41.903,2.631,2.753,2.821,2.443,2.253,1.728,0.84,2.065,0.164,-12.653,5.016,8.456,3.777,0.169,2.294],[5668.5,1.484,9.878,2005.799,378.124,41.378,100.156,69.149,0.894,46.271,2.617,2.645,2.776,2.341,2.177,2.209,1.068,2.064,0.155,-15.183,2.852,10.007,3.494,0.174,2.297],[5669.0,0.902,9.889,2002.279,396.351,40.961,100.348,70.893,0.829,47.9,2.649,2.523,2.73,2.285,2.231,2.517,1.163,2.165,0.127,-18.255,-1.412,10.634,3.394,0.166,2.321],[5669.5,0.668,10.016,1991.714,415.157,39.341,100.109,70.24,0.777,47.324,2.568,2.409,2.666,2.291,2.44,2.719,1.174,2.317,0.1,-21.939,-4.101,10.758,3.402,0.164,2.349],[5670.0,0.43,9.922,1981.535,431.544,36.971,99.546,67.052,0.701,44.654,2.298,2.317,2.572,2.388,2.989,2.974,1.134,2.64,0.083,-26.533,-6.875,10.605,3.477,0.18,2.369],[5670.5,0.256,9.415,1975.4,443.268,34.615,98.473,63.059,0.63,38.02,2.038,2.256,2.469,2.671,3.682,2.805,0.921,3.095,0.077,-32.472,-12.055,9.048,3.675,0.165,2.375],[5671.0,0.259,8.936,1976.247,449.792,32.817,96.668,59.094,0.682,32.275,1.955,2.223,2.392,3.181,4.103,2.015,0.737,2.75,0.079,-39.488,-17.042,6.769,3.427,0.101,2.371],[5671.5,0.311,8.747,1981.573,453.544,31.717,94.507,54.489,0.816,31.762,1.963,2.205,2.365,3.81,4.592,1.387,0.837,1.748,0.087,-46.43,-15.95,5.697,2.615,0.047,2.357],[5672.0,0.402,8.699,1983.1,460.094,30.969,92.885,49.52,0.873,31.509,1.962,2.173,2.381,4.392,4.809,1.258,0.986,1.282,0.096,-52.21,-7.557,5.606,2.12,0.047,2.341],[5672.5,0.443,8.671,1979.228,476.959,29.589,92.367,44.648,0.829,29.631,1.984,2.097,2.393,5.02,5.406,1.31,1.086,1.211,0.094,-56.353,-2.328,5.53,1.903,0.07,2.339],[5673.0,0.341,8.661,1979.417,508.382,27.315,92.654,39.965,0.663,25.031,1.998,1.967,2.318,6.094,7.383,1.413,0.974,1.579,0.08,-58.996,-7.027,4.741,1.841,0.075,2.351],[5673.5,0.345,8.642,1988.056,549.943,25.486,93.015,35.241,0.456,17.9,1.849,1.818,2.118,7.663,7.057,1.436,0.625,2.538,0.076,-61.064,-15.554,3.357,2.051,0.068,2.348],[5674.0,0.353,8.615,1998.645,593.263,25.411,93.152,30.054,0.346,12.87,1.635,1.686,1.86,7.302,4.419,1.32,0.408,3.282,0.095,-63.434,-21.894,2.348,2.133,0.067,2.303],[5674.5,0.243,8.594,2001.042,636.552,26.817,93.437,25.099,0.304,11.035,1.537,1.571,1.646,3.861,2.212,1.319,0.405,3.324,0.129,-65.481,-25.695,2.034,1.923,0.069,2.236],[5675.0,0.238,8.612,1995.908,685.53,28.269,94.336,22.135,0.278,11.825,1.53,1.459,1.507,1.887,1.964,1.681,0.688,2.673,0.152,-65.999,-25.338,2.361,1.422,0.064,2.189],[5675.5,0.215,8.641,1994.333,741.842,28.558,95.761,21.604,0.268,12.718,1.518,1.348,1.408,1.353,3.385,1.979,0.986,2.04,0.15,-65.806,-21.756,2.71,1.036,0.06,2.174],[5676.0,0.213,8.641,2001.666,800.269,27.618,97.159,21.843,0.284,12.715,1.449,1.25,1.317,1.48,3.376,1.906,1.119,1.721,0.135,-66.988,-19.749,2.755,0.92,0.06,2.181],[5676.5,0.209,8.644,2009.852,853.729,26.46,97.997,21.472,0.323,13.147,1.313,1.171,1.24,2.026,2.905,1.68,1.275,1.336,0.12,-69.631,-21.224,2.747,0.805,0.061,2.192],[5677.0,0.204,8.651,2009.115,898.233,26.009,98.131,20.818,0.383,13.699,1.211,1.113,1.188,2.471,3.29,1.379,1.379,1.006,0.11,-71.867,-20.069,2.655,0.716,0.064,2.2],[5677.5,0.203,8.652,2001.007,932.36,25.852,97.8,20.295,0.446,14.103,1.216,1.073,1.148,2.571,3.539,1.095,1.375,0.794,0.104,-72.775,-13.085,2.468,0.666,0.061,2.203],[5678.0,0.203,8.653,1992.203,959.372,25.507,97.35,19.659,0.494,14.349,1.262,1.042,1.101,2.821,3.383,0.921,1.358,0.678,0.101,-72.983,-8.072,2.314,0.632,0.05,2.204],[5678.5,0.209,8.662,1986.508,984.528,25.374,97.042,19.239,0.523,14.53,1.263,1.016,1.046,3.238,2.565,0.864,1.468,0.593,0.102,-73.085,-9.232,2.308,0.587,0.042,2.197],[5679.0,0.209,8.655,1984.17,1009.719,26.048,96.976,19.255,0.527,14.546,1.276,0.99,0.997,3.035,1.752,0.919,1.723,0.536,0.107,-72.952,-8.167,2.469,0.535,0.044,2.18],[5679.5,0.209,8.652,1986.542,1031.765,26.993,97.097,19.456,0.504,14.416,1.31,0.969,0.963,2.189,1.29,1.034,1.92,0.539,0.114,-72.18,-2.737,2.654,0.513,0.051,2.163],[5680.0,0.209,8.661,1992.979,1047.465,27.562,97.243,19.545,0.476,14.519,1.309,0.955,0.945,1.518,1.037,1.104,1.832,0.609,0.116,-70.824,-1.135,2.682,0.547,0.053,2.155],[5680.5,0.192,8.659,1998.609,1059.501,27.804,97.331,19.195,0.465,14.617,1.289,0.944,0.937,1.23,1.024,1.023,1.414,0.747,0.113,-69.371,-0.882,2.428,0.659,0.054,2.158],[5681.0,0.187,8.652,1998.763,1069.804,27.88,97.399,18.609,0.468,14.392,1.288,0.935,0.936,1.145,1.049,0.878,1.013,0.879,0.109,-68.068,-1.254,2.098,0.78,0.056,2.165],[5681.5,0.181,8.653,1994.962,1072.961,27.774,97.482,18.693,0.47,14.181,1.301,0.932,0.943,1.166,1.308,0.832,0.912,0.914,0.105,-66.896,-8.292,1.996,0.814,0.051,2.171],[5682.0,0.194,8.657,1993.831,1065.349,27.742,97.549,18.966,0.469,13.503,1.313,0.939,0.956,1.259,1.411,0.791,0.828,0.958,0.104,-65.931,-16.238,1.896,0.852,0.041,2.176],[5682.5,0.227,8.651,1996.403,1053.299,28.075,97.499,18.426,0.464,12.703,1.332,0.949,0.971,1.366,1.29,0.768,0.76,1.012,0.103,-65.044,-13.253,1.819,0.889,0.041,2.18],[5683.0,0.23,8.635,1997.012,1044.751,28.213,97.297,18.002,0.457,12.382,1.358,0.957,0.985,1.357,1.391,0.763,0.725,1.053,0.101,-63.757,-6.161,1.781,0.911,0.05,2.186],[5683.5,0.227,8.647,1994.541,1040.51,27.718,97.045,18.426,0.457,12.548,1.373,0.961,0.996,1.352,1.714,0.775,0.736,1.054,0.098,-62.14,-8.381,1.808,0.912,0.054,2.192],[5684.0,0.24,8.666,1993.656,1035.452,27.088,96.856,19.404,0.457,12.884,1.395,0.966,0.997,1.582,1.56,0.791,0.753,1.05,0.096,-61.315,-18.292,1.846,0.91,0.051,2.198],[5684.5,0.249,8.662,1994.452,1022.454,27.19,96.673,20.253,0.464,13.455,1.436,0.978,0.989,1.81,1.211,0.804,0.789,1.023,0.098,-61.883,-24.71,1.907,0.898,0.053,2.197],[5685.0,0.209,8.649,1993.589,994.583,27.329,96.391,20.584,0.462,13.714,1.453,1.005,0.996,1.605,1.235,0.871,0.849,1.027,0.104,-62.6,-22.834,2.045,0.895,0.055,2.19],[5685.5,0.209,8.654,1989.507,951.056,26.178,96.192,20.613,0.426,13.067,1.443,1.051,1.037,1.186,1.123,1.011,0.873,1.16,0.111,-61.896,-19.219,2.18,0.927,0.055,2.185],[5686.0,0.207,8.659,1985.171,905.008,24.792,96.554,20.683,0.385,12.7,1.473,1.105,1.119,0.899,-3.657,1.183,0.867,1.371,0.109,-60.196,-20.531,2.312,0.992,0.067,2.201],[5686.5,0.184,8.658,1980.845,876.609,24.154,97.811,20.866,0.345,12.933,1.616,1.141,1.222,0.818,33.374,1.463,0.788,1.906,0.086,-59.606,-22.215,2.544,1.221,0.098,2.254],[5687.0,0.119,8.656,1975.503,870.372,24.33,99.684,22.178,0.336,13.834,1.786,1.149,1.303,1.121,99.605,1.625,0.702,2.326,0.053,-61.425,-18.772,2.782,1.477,0.112,2.335],[5687.5,0.072,8.659,1973.599,869.499,24.139,101.386,25.12,0.389,15.868,1.679,1.15,1.323,2.512,35.211,1.522,0.732,2.1,0.04,-64.894,-14.581,3.006,1.526,0.081,2.373],[5688.0,0.072,8.673,1979.074,856.549,23.494,102.332,28.022,0.455,18.173,1.449,1.167,1.292,4.95,-1.207,1.46,0.848,1.738,0.065,-68.447,-14.091,3.389,1.485,0.029,2.305],[5688.5,0.073,8.686,1987.319,832.643,24.549,102.693,29.696,0.492,19.678,1.521,1.201,1.257,4.4,2.933,1.475,0.937,1.576,0.12,-71.249,-12.076,3.703,1.466,0.02,2.197],[5689.0,0.084,8.688,1994.742,814.662,28.275,103.067,31.323,0.51,21.124,1.754,1.228,1.247,3.087,2.431,1.519,1.011,1.505,0.156,-72.998,-5.914,3.957,1.453,0.036,2.145],[5689.5,0.101,8.682,1998.45,816.294,31.57,103.7,33.651,0.489,23.258,1.912,1.225,1.252,2.67,5.099,1.961,1.134,1.716,0.14,-73.486,-1.675,4.806,1.567,0.039,2.168],[5690.0,0.09,8.681,1996.66,836.663,31.925,104.194,36.704,0.442,24.581,1.846,1.195,1.251,2.903,5.507,2.433,1.098,2.249,0.109,-73.126,-0.509,5.462,1.867,0.032,2.217],[5690.5,0.084,8.669,1993.195,860.759,31.646,103.982,39.993,0.421,24.623,1.72,1.162,1.24,3.464,3.22,2.477,0.932,2.674,0.103,-72.864,-0.888,5.33,2.132,0.025,2.223],[5691.0,0.078,8.657,1996.499,871.804,32.009,103.02,41.605,0.41,23.524,1.678,1.147,1.234,3.909,3.44,2.37,0.811,2.943,0.123,-73.203,-4.77,4.969,2.283,0.025,2.183],[5691.5,0.078,8.652,2008.502,865.459,32.092,101.857,40.466,0.389,20.414,1.602,1.155,1.239,4.098,3.451,2.109,0.641,3.358,0.146,-73.839,-12.424,4.204,2.461,0.029,2.145],[5692.0,0.078,8.659,2019.648,848.22,31.373,101.071,37.9,0.374,17.38,1.496,1.179,1.256,4.24,3.459,1.833,0.507,3.624,0.154,-74.294,-18.017,3.502,2.566,0.032,2.137],[5692.5,0.104,8.676,2015.093,824.044,29.751,100.756,34.997,0.406,17.542,1.424,1.214,1.289,4.255,3.471,1.677,0.587,3.02,0.147,-74.425,-18.192,3.461,2.269,0.037,2.155],[5693.0,0.106,8.678,1999.642,791.418,27.775,100.53,31.836,0.459,19.31,1.404,1.264,1.353,4.231,3.214,1.605,0.794,2.111,0.132,-74.274,-15.368,3.759,1.805,0.042,2.188],[5693.5,0.111,8.661,1990.984,748.208,25.962,99.909,29.833,0.49,20.296,1.408,1.337,1.454,4.249,3.24,1.611,0.967,1.678,0.116,-73.887,-14.912,4.032,1.555,0.045,2.226],[5694.0,0.112,8.662,1993.579,698.454,24.419,98.711,29.839,0.508,20.975,1.403,1.432,1.596,4.405,3.909,1.634,1.106,1.485,0.104,-73.348,-15.092,4.237,1.427,0.048,2.259],[5694.5,0.118,8.667,1999.608,650.596,23.345,97.165,30.802,0.505,21.329,1.428,1.537,1.783,4.78,3.98,1.684,1.138,1.495,0.1,-72.864,-12.752,4.334,1.422,0.046,2.28],[5695.0,0.118,8.671,2003.07,608.869,22.478,95.657,31.02,0.468,20.697,1.549,1.642,2.002,5.359,3.844,1.765,0.982,1.841,0.104,-72.551,-12.896,4.211,1.616,0.042,2.284],[5695.5,0.118,8.676,2003.74,573.491,21.607,94.412,30.813,0.432,19.821,1.71,1.744,2.216,6.079,3.97,1.816,0.819,2.229,0.115,-72.316,-15.309,4.002,1.818,0.044,2.278],[5696.0,0.114,8.674,2000.971,546.966,20.99,93.339,30.884,0.41,19.018,1.759,1.828,2.369,6.725,4.591,1.832,0.745,2.467,0.124,-72.045,-13.248,3.84,1.915,0.055,2.273],[5696.5,0.112,8.66,1993.548,531.412,20.639,92.206,31.136,0.386,17.762,1.703,1.882,2.423,6.745,4.625,1.846,0.679,2.729,0.127,-71.809,-11.747,3.637,1.989,0.071,2.273],[5697.0,0.106,8.644,1988.521,525.658,20.266,91.072,31.345,0.356,16.498,1.659,1.902,2.384,6.05,4.615,1.913,0.636,3.009,0.126,-71.724,-18.28,3.474,2.026,0.077,2.276],[5697.5,0.101,8.646,1998.294,529.965,20.023,90.289,31.523,0.306,15.15,1.65,1.887,2.296,5.372,4.544,2.175,0.616,3.542,0.125,-71.779,-26.686,3.351,2.018,0.065,2.276],[5698.0,0.096,8.668,2017.0,547.658,20.008,90.057,31.274,0.246,13.374,1.617,1.826,2.192,4.984,4.352,2.496,0.571,4.424,0.122,-71.941,-27.696,3.11,2.026,0.049,2.275],[5698.5,0.091,8.671,2028.981,579.584,20.043,90.203,30.361,0.216,11.961,1.556,1.725,2.088,4.737,4.625,2.568,0.515,5.02,0.117,-72.238,-19.457,2.832,2.044,0.041,2.272],[5699.0,0.084,8.657,2025.975,619.836,20.17,90.408,28.796,0.215,11.254,1.527,1.613,1.975,4.545,5.032,2.374,0.476,4.976,0.114,-72.67,-9.013,2.595,2.022,0.034,2.265],[5699.5,0.078,8.661,2008.786,661.146,20.538,90.557,26.877,0.249,10.771,1.538,1.513,1.84,4.56,5.073,1.824,0.428,4.201,0.113,-73.228,-4.417,2.24,1.936,0.026,2.254],[5700.0,0.078,8.659,1992.092,701.721,21.143,90.831,25.118,0.304,10.363,1.539,1.425,1.686,4.847,4.714,1.25,0.388,3.206,0.113,-73.915,-3.6,1.89,1.807,0.021,2.243],[5700.5,0.089,8.66,1990.032,740.809,21.64,91.391,23.73,0.368,10.394,1.503,1.35,1.534,4.992,4.177,0.885,0.383,2.316,0.113,-74.582,-0.615,1.612,1.565,0.019,2.233],[5701.0,0.097,8.684,1993.976,776.801,21.921,91.998,22.655,0.419,10.633,1.476,1.287,1.402,4.344,3.436,0.662,0.4,1.667,0.114,-75.011,3.4,1.411,1.316,0.018,2.222],[5701.5,0.112,8.691,1988.64,811.753,22.272,92.249,21.817,0.445,11.233,1.471,1.232,1.299,3.384,2.908,0.627,0.459,1.386,0.116,-75.097,4.325,1.425,1.163,0.023,2.21],[5702.0,0.123,8.68,1977.55,850.4,22.745,92.01,20.961,0.466,12.344,1.45,1.176,1.224,2.673,2.767,0.637,0.556,1.161,0.116,-74.764,1.928,1.517,1.023,0.033,2.2],[5702.5,0.124,8.677,1970.073,892.452,23.103,91.445,20.035,0.499,13.51,1.419,1.121,1.17,2.326,2.967,0.659,0.739,0.927,0.114,-74.176,2.042,1.683,0.868,0.045,2.194],[5703.0,0.118,8.664,1969.436,930.887,23.364,90.832,19.496,0.554,14.585,1.445,1.074,1.129,2.329,3.086,0.686,1.087,0.666,0.112,-73.793,7.194,1.943,0.689,0.053,2.189],[5703.5,0.118,8.654,1974.111,959.072,23.725,90.355,19.225,0.597,15.207,1.509,1.043,1.095,2.431,2.661,0.704,1.426,0.502,0.111,-73.772,13.051,2.145,0.565,0.054,2.185],[5704.0,0.118,8.651,1978.015,976.814,24.175,90.029,18.689,0.611,15.503,1.529,1.024,1.07,2.356,2.116,0.714,1.6,0.447,0.109,-73.728,16.921,2.228,0.517,0.05,2.184],[5704.5,0.112,8.658,1976.774,986.335,24.022,89.784,18.328,0.605,15.768,1.447,1.014,1.054,2.05,1.788,0.734,1.663,0.442,0.104,-73.267,20.799,2.267,0.506,0.045,2.192],[5705.0,0.112,8.666,1975.686,987.982,23.6,89.581,18.89,0.582,15.705,1.349,1.012,1.048,1.701,1.541,0.762,1.609,0.476,0.097,-72.474,25.448,2.266,0.524,0.04,2.206],[5705.5,0.113,8.677,1980.69,981.95,23.661,89.412,19.806,0.53,14.857,1.363,1.018,1.054,1.452,1.47,0.843,1.468,0.58,0.094,-71.884,24.705,2.261,0.574,0.037,2.215],[5706.0,0.118,8.677,1987.991,970.15,24.116,89.247,20.172,0.456,13.564,1.472,1.031,1.071,1.288,1.54,1.021,1.38,0.741,0.095,-72.002,15.62,2.347,0.632,0.037,2.215],[5706.5,0.123,8.675,1990.551,955.961,24.518,89.068,20.095,0.412,12.899,1.51,1.046,1.101,1.25,1.81,1.186,1.413,0.839,0.099,-72.862,5.504,2.489,0.654,0.036,2.208],[5707.0,0.135,8.675,1989.842,941.904,24.865,88.85,20.037,0.405,13.473,1.459,1.062,1.138,1.42,2.381,1.302,1.593,0.823,0.104,-74.186,2.058,2.695,0.631,0.032,2.201],[5707.5,0.145,8.673,1991.456,930.265,25.108,88.619,20.062,0.427,15.266,1.45,1.075,1.167,1.894,3.042,1.393,1.959,0.723,0.108,-75.589,3.108,3.043,0.582,0.03,2.196],[5708.0,0.153,8.681,1999.334,924.665,24.925,88.466,20.139,0.469,17.268,1.49,1.081,1.178,2.661,2.45,1.362,2.192,0.622,0.109,-76.579,0.866,3.252,0.551,0.035,2.195],[5708.5,0.152,8.678,2013.227,926.005,24.407,88.447,19.884,0.496,17.944,1.488,1.08,1.171,3.146,2.337,1.252,2.085,0.602,0.107,-76.844,-7.803,3.168,0.566,0.042,2.2],[5709.0,0.152,8.674,2026.707,929.283,23.895,88.523,19.434,0.474,16.244,1.426,1.076,1.15,2.993,2.824,1.133,1.616,0.75,0.103,-76.641,-18.212,2.758,0.661,0.044,2.206],[5709.5,0.152,8.667,2031.062,930.69,23.691,88.617,19.332,0.406,13.074,1.393,1.074,1.122,2.654,2.11,1.041,1.011,1.116,0.101,-76.471,-21.501,2.169,0.834,0.047,2.21],[5710.0,0.152,8.665,2026.321,934.256,23.79,88.783,19.317,0.355,11.134,1.439,1.07,1.094,2.337,1.673,1.047,0.746,1.409,0.101,-76.428,-16.583,1.898,0.946,0.053,2.21],[5710.5,0.154,8.671,2021.699,941.712,24.079,89.18,18.881,0.339,10.661,1.504,1.062,1.076,2.016,1.744,1.058,0.688,1.546,0.102,-76.262,-10.751,1.834,0.992,0.052,2.206],[5711.0,0.158,8.688,2022.312,948.181,24.381,89.809,18.829,0.327,10.428,1.517,1.055,1.075,1.717,1.718,1.057,0.622,1.707,0.103,-75.859,-11.338,1.763,1.055,0.046,2.202],[5711.5,0.17,8.683,2023.339,954.809,24.815,90.393,19.311,0.313,10.387,1.503,1.047,1.089,1.46,1.738,1.113,0.599,1.86,0.104,-75.292,-17.035,1.779,1.103,0.043,2.199],[5712.0,0.175,8.682,2018.375,970.0,25.401,90.632,19.262,0.307,10.773,1.511,1.031,1.102,1.365,2.385,1.192,0.622,1.917,0.103,-74.807,-19.402,1.867,1.114,0.043,2.198],[5712.5,0.169,8.655,2012.304,993.145,25.792,90.526,19.017,0.297,11.291,1.543,1.007,1.09,1.726,3.549,1.306,0.645,2.026,0.101,-74.708,-17.796,1.972,1.135,0.04,2.196],[5713.0,0.169,8.641,2012.059,1013.507,25.73,90.399,19.471,0.278,11.53,1.594,0.987,1.044,2.7,3.506,1.475,0.681,2.168,0.103,-74.844,-16.2,2.089,1.142,0.036,2.189],[5713.5,0.169,8.654,2014.864,1027.857,25.599,90.607,19.605,0.259,11.884,1.643,0.973,0.986,3.152,2.252,1.829,0.912,2.053,0.105,-74.646,-16.819,2.395,1.011,0.035,2.18],[5714.0,0.172,8.657,2017.87,1041.603,25.831,91.211,18.633,0.252,12.159,1.609,0.96,0.939,2.193,1.625,2.133,1.257,1.73,0.106,-73.956,-18.747,2.739,0.824,0.034,2.175],[5714.5,0.182,8.664,2025.373,1061.99,25.959,91.998,17.442,0.28,12.231,1.48,0.942,0.91,1.394,1.552,1.988,1.424,1.399,0.105,-73.153,-22.853,2.808,0.732,0.038,2.174],[5715.0,0.175,8.656,2034.549,1093.024,25.813,92.818,17.184,0.345,12.907,1.368,0.915,0.888,1.137,1.563,1.59,1.471,1.083,0.103,-72.634,-27.838,2.715,0.685,0.045,2.173],[5715.5,0.172,8.657,2037.266,1131.051,25.959,93.691,17.737,0.444,14.095,1.327,0.884,0.864,1.152,1.84,1.153,1.426,0.801,0.101,-72.677,-31.161,2.516,0.654,0.048,2.168],[5716.0,0.164,8.661,2034.705,1160.655,26.373,94.65,18.524,0.524,15.048,1.296,0.862,0.844,1.268,2.019,0.873,1.303,0.669,0.102,-73.203,-32.396,2.326,0.663,0.045,2.161],[5716.5,0.163,8.654,2033.141,1164.053,26.838,95.601,19.191,0.56,15.518,1.297,0.859,0.84,1.3,1.73,0.794,1.215,0.654,0.106,-73.585,-34.061,2.267,0.693,0.039,2.152],[5717.0,0.158,8.654,2034.621,1135.505,27.624,96.346,19.593,0.572,15.864,1.41,0.881,0.862,1.118,1.526,0.777,1.161,0.672,0.114,-73.44,-35.705,2.269,0.727,0.032,2.139],[5717.5,0.153,8.664,2042.101,1090.442,28.521,96.713,20.404,0.527,15.897,1.554,0.917,0.911,0.918,1.649,0.883,1.04,0.877,0.125,-73.199,-35.261,2.343,0.852,0.031,2.127],[5718.0,0.153,8.659,2050.89,1054.589,28.444,96.696,22.462,0.45,15.385,1.599,0.948,0.97,1.036,1.958,1.07,0.871,1.251,0.131,-73.176,-32.868,2.434,1.048,0.038,2.124],[5718.5,0.158,8.652,2052.79,1038.754,27.736,96.442,24.921,0.397,14.528,1.554,0.963,1.009,1.912,3.039,1.195,0.77,1.562,0.128,-73.026,-29.544,2.417,1.167,0.044,2.133],[5719.0,0.158,8.654,2050.121,1031.664,26.853,96.175,25.242,0.358,13.525,1.494,0.969,1.008,4.478,3.374,1.313,0.731,1.799,0.118,-72.412,-28.687,2.393,1.216,0.042,2.153],[5719.5,0.158,8.655,2047.628,1016.55,25.746,96.122,23.228,0.329,12.787,1.468,0.984,0.98,6.265,2.15,1.44,0.734,1.961,0.11,-71.262,-34.327,2.416,1.221,0.04,2.173],[5720.0,0.158,8.658,2042.186,983.063,25.398,96.073,21.087,0.307,12.337,1.511,1.017,0.957,3.489,1.208,1.558,0.768,2.029,0.109,-69.386,-41.011,2.441,1.181,0.039,2.183],[5720.5,0.175,8.657,2035.507,930.659,25.457,95.234,19.969,0.292,12.162,1.541,1.075,0.961,1.517,0.942,1.697,0.838,2.027,0.114,-66.551,-38.586,2.524,1.12,0.041,2.185],[5721.0,0.191,8.656,2037.258,865.006,25.318,92.85,19.336,0.28,12.457,1.497,1.156,1.008,0.97,0.889,1.879,0.969,1.946,0.122,-63.284,-30.306,2.681,1.033,0.043,2.186],[5721.5,0.209,8.666,2045.984,788.801,25.369,88.816,18.809,0.271,12.989,1.473,1.268,1.13,0.887,0.93,2.078,1.15,1.816,0.133,-60.732,-25.312,2.876,0.935,0.043,2.188],[5722.0,0.237,8.674,2053.379,702.815,24.553,83.684,18.409,0.256,13.25,1.578,1.423,1.385,0.907,-160.361,2.372,1.354,1.751,0.136,-59.177,-25.602,3.084,0.85,0.043,2.207],[5722.5,0.255,8.672,2056.728,607.907,21.996,78.146,18.388,0.238,13.181,1.812,1.645,1.854,1.007,994.905,2.669,1.523,1.752,0.107,-58.288,-28.381,3.235,0.789,0.044,2.28],[5723.0,0.352,8.677,2055.565,504.689,16.819,72.491,18.593,0.229,13.079,2.102,1.981,2.622,1.622,4017.103,2.805,1.588,1.767,0.049,-58.382,-32.997,3.273,0.765,0.043,2.414],[5723.5,0.301,8.691,2048.335,398.485,12.348,66.78,17.942,0.22,13.158,2.245,2.51,3.82,5.199,1985.006,2.939,1.609,1.826,0.008,-59.714,-35.412,3.3,0.761,0.042,2.56],[5724.0,0.266,8.706,2044.028,303.487,9.118,61.736,16.782,0.216,13.269,2.098,3.295,5.746,32.902,2610.006,3.022,1.611,1.876,0.0,-61.627,-29.872,3.327,0.766,0.036,2.649],[5724.5,0.249,8.711,2047.945,237.47,7.12,58.751,16.563,0.22,13.275,2.003,4.211,8.28,454.64,2884.967,2.922,1.546,1.891,0.0,-63.459,-20.434,3.28,0.788,0.034,2.673],[5725.0,0.238,8.711,2053.767,216.31,6.47,58.855,17.358,0.228,13.305,2.07,4.623,8.795,1985.1,440.283,2.721,1.419,1.919,0.0,-65.033,-12.713,3.167,0.83,0.036,2.658],[5725.5,0.227,8.706,2054.501,246.751,6.799,62.028,18.538,0.239,13.432,2.034,4.053,6.091,326.086,268.396,2.481,1.274,1.948,0.003,-66.333,-5.451,3.024,0.883,0.034,2.606],[5726.0,0.276,8.694,2049.41,320.798,8.031,67.256,19.878,0.253,13.559,1.687,3.117,3.606,69.637,103.917,2.253,1.173,1.918,0.026,-67.57,-1.208,2.905,0.919,0.023,2.506],[5726.5,0.156,8.683,2043.334,420.148,10.391,73.245,20.9,0.28,13.477,1.446,2.38,2.291,16.467,7.754,1.987,1.117,1.775,0.084,-68.879,-4.761,2.819,0.937,0.015,2.371],[5727.0,0.135,8.683,2044.837,525.484,15.014,79.191,21.3,0.316,13.255,1.42,1.903,1.67,5.463,2.824,1.716,1.076,1.593,0.148,-69.894,-12.077,2.756,0.951,0.024,2.247],[5727.5,0.119,8.686,2056.393,622.898,21.192,84.746,20.989,0.342,13.578,1.471,1.605,1.391,2.599,1.707,1.583,1.072,1.478,0.18,-70.145,-18.974,2.757,0.955,0.038,2.173],[5728.0,0.105,8.679,2072.254,707.892,25.831,89.483,20.771,0.401,15.708,1.551,1.413,1.279,1.6,1.542,1.433,1.119,1.288,0.173,-70.093,-26.82,2.886,0.956,0.041,2.152],[5728.5,0.096,8.683,2088.687,784.113,27.028,92.965,22.879,0.506,19.526,1.691,1.275,1.239,1.323,1.654,1.364,1.158,1.187,0.145,-70.533,-33.503,3.526,1.136,0.036,2.17],[5729.0,0.106,8.685,2099.403,853.794,26.854,95.106,28.415,0.55,22.466,1.785,1.171,1.218,1.544,2.541,1.529,1.037,1.519,0.113,-71.2,-36.942,4.286,1.575,0.025,2.205],[5729.5,0.129,8.681,2094.548,906.53,26.503,96.133,35.279,0.533,23.82,1.715,1.103,1.197,2.347,3.106,1.664,0.893,1.869,0.091,-71.429,-33.675,4.529,1.887,0.016,2.237],[5730.0,0.125,8.683,2071.306,925.772,26.237,96.461,40.264,0.52,23.984,1.586,1.08,1.181,3.522,2.963,1.711,0.87,1.967,0.084,-71.462,-24.457,4.538,1.937,0.022,2.248],[5730.5,0.127,8.686,2044.581,910.222,26.118,96.532,40.643,0.5,23.053,1.519,1.099,1.177,4.22,3.294,1.796,0.879,2.042,0.088,-71.719,-18.168,4.579,1.933,0.036,2.242],[5731.0,0.119,8.681,2036.134,876.845,25.771,96.623,37.113,0.476,22.13,1.458,1.14,1.186,4.297,3.329,1.929,0.908,2.123,0.095,-71.898,-17.969,4.679,1.913,0.041,2.233],[5731.5,0.135,8.68,2042.609,842.514,25.123,96.883,33.699,0.449,21.888,1.406,1.187,1.213,4.094,2.91,2.154,1.006,2.143,0.101,-71.567,-15.614,4.927,1.825,0.04,2.231],[5732.0,0.158,8.689,2038.669,809.456,25.402,97.419,32.962,0.439,21.958,1.407,1.235,1.264,3.691,2.793,2.308,1.114,2.074,0.102,-70.774,-7.644,5.173,1.726,0.043,2.237],[5732.5,0.184,8.696,2017.611,776.624,25.794,98.155,33.995,0.432,22.654,1.475,1.288,1.341,3.232,2.928,2.429,1.194,2.033,0.1,-69.785,2.233,5.351,1.664,0.048,2.248],[5733.0,0.175,8.693,2000.517,748.009,26.733,98.769,37.244,0.437,24.685,1.645,1.337,1.44,2.94,2.434,2.478,1.189,2.087,0.097,-68.656,10.483,5.511,1.731,0.054,2.26],[5733.5,0.21,8.696,1996.147,727.693,28.672,98.954,43.542,0.493,27.113,1.822,1.374,1.542,2.824,2.214,2.251,1.053,2.139,0.096,-67.053,15.109,5.619,2.001,0.066,2.267],[5734.0,0.187,8.699,1995.402,716.94,30.05,98.639,48.09,0.537,27.834,1.747,1.395,1.618,2.836,3.083,2.043,0.956,2.138,0.098,-65.095,17.734,5.596,2.173,0.075,2.266],[5734.5,0.147,8.697,1993.998,720.689,29.42,98.022,44.875,0.551,24.708,1.501,1.388,1.627,3.205,4.225,1.702,0.843,1.989,0.104,-64.265,16.79,4.74,2.066,0.063,2.253],[5735.0,0.178,8.71,1994.496,744.095,28.269,97.403,35.044,0.572,19.662,1.372,1.344,1.552,3.833,4.363,1.111,0.688,1.576,0.115,-65.846,9.27,3.204,1.698,0.043,2.229],[5735.5,0.255,8.715,1999.78,783.204,27.097,96.867,25.614,0.554,16.153,1.379,1.277,1.419,4.252,4.072,0.817,0.704,1.179,0.122,-68.988,-0.458,2.31,1.248,0.039,2.206],[5736.0,0.266,8.707,2011.535,827.994,25.057,96.249,21.046,0.506,14.483,1.409,1.208,1.277,3.999,3.48,0.866,0.872,1.003,0.123,-71.884,-5.202,2.23,0.965,0.044,2.193],[5736.5,0.265,8.693,2024.984,874.063,25.269,95.464,20.489,0.475,13.994,1.476,1.144,1.161,3.108,2.271,0.969,1.028,0.943,0.12,-73.724,-3.158,2.347,0.849,0.048,2.187],[5737.0,0.283,8.678,2032.228,922.865,26.031,94.684,20.686,0.457,14.2,1.575,1.084,1.085,2.098,1.367,1.07,1.161,0.923,0.115,-74.445,4.579,2.494,0.799,0.05,2.185],[5737.5,0.228,8.67,2034.302,973.319,26.176,94.147,20.281,0.446,14.466,1.608,1.027,1.046,1.416,1.018,1.146,1.275,0.899,0.108,-74.29,12.211,2.61,0.761,0.05,2.187],[5738.0,0.209,8.68,2034.01,1023.357,26.399,93.935,20.617,0.453,14.853,1.566,0.977,1.025,1.16,1.381,1.158,1.331,0.87,0.101,-74.141,11.838,2.667,0.745,0.045,2.19],[5738.5,0.227,8.702,2029.73,1071.894,26.593,93.93,22.304,0.513,15.438,1.539,0.933,0.996,1.373,2.269,0.937,1.119,0.834,0.097,-74.848,3.787,2.382,0.808,0.041,2.188],[5739.0,0.224,8.705,2021.532,1114.32,26.341,93.955,23.414,0.584,15.616,1.523,0.897,0.955,2.194,2.666,0.64,0.764,0.843,0.097,-76.121,-6.267,1.888,0.934,0.044,2.181],[5739.5,0.209,8.699,2011.218,1141.45,25.633,93.971,22.86,0.59,14.731,1.488,0.876,0.915,3.136,2.497,0.518,0.553,0.955,0.097,-77.084,-15.082,1.562,1.064,0.043,2.176],[5740.0,0.209,8.698,1998.993,1150.297,25.051,94.035,22.159,0.577,13.515,1.478,0.869,0.888,2.782,1.986,0.457,0.435,1.054,0.094,-77.406,-21.597,1.347,1.153,0.038,2.181],[5740.5,0.209,8.707,1986.646,1153.117,24.986,94.163,22.043,0.574,13.051,1.529,0.867,0.874,1.825,1.725,0.442,0.411,1.073,0.091,-77.296,-25.405,1.293,1.167,0.038,2.188],[5741.0,0.215,8.707,1977.665,1171.512,25.116,94.417,21.793,0.562,13.344,1.539,0.854,0.861,1.389,1.978,0.529,0.517,1.036,0.088,-77.01,-25.59,1.514,1.103,0.046,2.191],[5741.5,0.208,8.694,1975.097,1217.746,25.127,94.977,21.118,0.561,14.182,1.47,0.821,0.837,1.541,2.248,0.663,0.735,0.921,0.084,-76.637,-23.918,1.903,0.978,0.057,2.192],[5742.0,0.175,8.686,1976.669,1281.971,25.268,96.011,20.742,0.59,15.571,1.429,0.78,0.801,2.077,2.013,0.766,1.024,0.768,0.08,-76.002,-26.13,2.312,0.853,0.064,2.192],[5742.5,0.152,8.702,1979.607,1340.746,25.843,97.6,21.29,0.648,17.879,1.458,0.746,0.762,2.211,1.706,0.839,1.425,0.607,0.08,-74.875,-33.859,2.786,0.738,0.058,2.183],[5743.0,0.141,8.711,1985.667,1375.159,26.963,99.678,22.824,0.738,20.91,1.485,0.727,0.732,1.836,1.731,0.895,1.89,0.483,0.086,-73.119,-39.357,3.381,0.667,0.048,2.16],[5743.5,0.129,8.707,1994.102,1376.874,28.483,101.927,25.074,0.848,24.31,1.491,0.726,0.718,1.619,1.872,0.959,2.32,0.416,0.098,-70.621,-37.33,4.169,0.665,0.045,2.13],[5744.0,0.118,8.713,2002.518,1345.611,30.198,103.881,29.125,0.916,28.604,1.548,0.743,0.724,1.62,1.94,1.026,2.558,0.401,0.109,-67.674,-33.441,4.801,0.696,0.047,2.108],[5744.5,0.095,8.729,2012.703,1292.304,31.664,105.21,37.701,0.926,34.473,1.746,0.774,0.761,1.445,1.634,1.261,2.469,0.545,0.112,-64.632,-31.464,5.953,0.95,0.045,2.112],[5745.0,0.09,8.731,2025.386,1232.906,33.056,106.009,50.706,0.91,40.063,2.101,0.811,0.832,1.034,1.125,1.689,1.959,0.949,0.102,-59.877,-31.825,7.824,1.623,0.04,2.147],[5745.5,0.135,8.735,2038.702,1172.17,34.197,106.79,61.754,0.89,43.582,2.404,0.853,0.92,0.86,1.497,1.942,1.553,1.255,0.086,-51.115,-36.217,8.825,2.115,0.039,2.195],[5746.0,0.089,8.744,2048.742,1099.513,35.389,107.951,63.997,0.853,40.393,2.428,0.909,1.0,1.194,2.584,1.859,1.241,1.601,0.08,-41.254,-40.279,8.122,2.557,0.037,2.223],[5746.5,0.091,8.734,2048.059,1004.518,36.833,109.22,59.895,0.797,33.373,2.384,0.996,1.076,2.437,2.8,1.689,0.822,2.128,0.091,-36.401,-37.558,6.89,3.195,0.029,2.216],[5747.0,0.101,8.733,2039.867,891.136,37.749,109.89,59.607,0.789,32.031,2.529,1.122,1.178,3.524,2.501,1.626,0.673,2.422,0.111,-36.813,-34.71,6.543,3.626,0.029,2.2],[5747.5,0.114,8.747,2037.615,779.089,39.476,109.668,67.643,0.792,34.041,2.763,1.284,1.331,2.916,1.774,1.742,0.597,2.973,0.122,-35.841,-39.707,7.008,4.434,0.044,2.207],[5748.0,0.182,8.843,2042.741,689.541,41.26,108.981,76.938,0.77,36.444,2.914,1.45,1.533,2.115,1.452,1.952,0.549,3.559,0.119,-29.348,-46.714,7.645,5.169,0.064,2.237],[5748.5,0.171,8.964,2047.099,627.694,41.937,108.49,80.126,0.788,38.831,2.931,1.593,1.758,1.878,2.046,1.971,0.565,3.493,0.114,-20.51,-45.123,7.918,5.202,0.072,2.263],[5749.0,0.101,9.009,2048.501,583.022,41.546,108.427,77.994,0.828,39.191,2.959,1.715,1.968,2.292,2.832,1.884,0.575,3.274,0.115,-14.876,-35.006,7.957,5.131,0.062,2.274],[5749.5,0.084,9.008,2048.055,546.527,40.845,108.484,74.828,0.855,38.247,2.985,1.83,2.135,3.174,2.993,1.81,0.573,3.157,0.119,-13.733,-27.78,7.899,5.113,0.048,2.279],[5750.0,0.076,9.008,2047.329,517.925,40.464,108.33,73.39,0.89,37.811,2.991,1.931,2.238,3.858,2.904,1.736,0.582,2.983,0.122,-14.4,-32.194,7.886,5.026,0.042,2.285],[5750.5,0.09,9.012,2047.131,498.703,40.369,108.035,73.936,0.92,38.483,3.024,2.005,2.267,3.989,3.006,1.708,0.606,2.821,0.124,-14.777,-41.909,8.023,4.915,0.035,2.288],[5751.0,0.154,9.014,2044.758,489.344,40.564,108.013,75.565,0.941,41.027,3.107,2.044,2.237,3.829,2.896,1.774,0.658,2.705,0.126,-14.384,-47.294,8.529,4.819,0.025,2.289],[5751.5,0.232,9.035,2045.988,490.361,41.545,108.696,77.657,1.031,48.791,3.211,2.039,2.182,3.378,2.112,1.978,0.914,2.29,0.123,-12.897,-48.417,10.497,4.395,0.025,2.292],[5752.0,0.237,9.052,2056.163,500.762,42.875,110.249,80.589,1.147,58.08,3.269,1.997,2.119,2.654,1.515,2.201,1.279,1.766,0.114,-10.297,-48.26,12.919,3.804,0.04,2.301],[5752.5,0.235,9.078,2063.392,516.878,43.449,112.227,85.029,1.195,62.796,3.221,1.935,2.039,2.193,1.854,2.304,1.482,1.556,0.102,-8.063,-46.505,14.06,3.523,0.054,2.314],[5753.0,0.238,9.103,2057.866,534.085,43.697,113.578,90.629,1.199,65.812,3.127,1.872,1.947,2.072,1.925,2.405,1.613,1.492,0.089,-7.817,-40.907,14.711,3.389,0.058,2.33],[5753.5,0.243,9.104,2048.225,550.082,44.318,113.326,95.07,1.137,68.838,3.126,1.818,1.87,2.013,1.392,2.664,1.693,1.574,0.079,-9.456,-34.04,15.409,3.38,0.066,2.343],[5754.0,0.245,9.078,2042.084,567.692,43.689,111.433,96.128,1.065,69.792,3.179,1.762,1.832,1.722,1.146,2.912,1.681,1.735,0.073,-11.974,-29.6,15.825,3.495,0.079,2.35],[5754.5,0.244,9.016,2037.959,592.321,41.899,108.809,91.226,0.913,63.852,3.024,1.688,1.821,1.399,1.098,3.443,1.531,2.365,0.069,-15.177,-29.448,15.23,3.721,0.093,2.351],[5755.0,0.23,8.976,2039.448,626.079,38.422,106.402,79.013,0.582,49.907,2.605,1.597,1.773,1.343,1.701,4.88,1.244,4.148,0.071,-20.198,-28.679,12.978,3.883,0.105,2.339],[5755.5,0.186,8.899,2045.693,667.05,33.655,104.384,62.29,0.385,36.748,2.274,1.499,1.654,1.691,2.079,5.034,1.065,4.706,0.082,-28.333,-23.107,9.846,3.413,0.103,2.309],[5756.0,0.076,8.788,2054.036,713.349,31.254,102.397,47.342,0.407,28.35,2.212,1.402,1.5,2.56,2.683,3.415,0.974,3.457,0.097,-38.755,-20.06,6.977,2.632,0.077,2.269],[5756.5,0.103,8.719,2064.82,763.946,30.61,100.499,38.516,0.439,23.1,2.253,1.309,1.36,3.466,3.013,2.389,0.867,2.74,0.107,-48.677,-26.513,5.331,2.276,0.044,2.238],[5757.0,0.084,8.691,2071.304,815.728,30.604,99.424,35.923,0.464,21.307,2.275,1.226,1.244,3.78,2.86,2.067,0.811,2.545,0.109,-56.065,-34.101,4.889,2.236,0.025,2.221],[5757.5,0.102,8.666,2068.568,866.083,31.045,99.768,35.708,0.507,21.505,2.267,1.155,1.151,3.73,2.784,1.769,0.752,2.343,0.109,-60.542,-33.561,4.535,2.24,0.021,2.21],[5758.0,0.091,8.633,2064.281,912.238,31.885,101.249,35.919,0.548,21.835,2.195,1.096,1.084,3.446,2.473,1.476,0.679,2.172,0.112,-62.592,-29.045,4.123,2.256,0.024,2.194],[5758.5,0.088,8.589,2064.314,948.177,32.877,102.74,36.967,0.563,21.127,2.132,1.055,1.045,3.026,2.379,1.324,0.615,2.156,0.12,-63.267,-24.366,3.803,2.299,0.026,2.168],[5759.0,0.084,8.575,2064.231,967.291,33.441,102.926,37.921,0.557,19.165,2.162,1.034,1.03,2.711,2.391,1.183,0.525,2.264,0.132,-63.57,-21.705,3.37,2.388,0.029,2.143],[5759.5,0.084,8.578,2061.376,963.558,33.38,101.319,37.104,0.546,17.564,2.212,1.038,1.037,2.602,2.499,1.073,0.453,2.369,0.139,-63.619,-23.931,2.994,2.452,0.03,2.132],[5760.0,0.096,8.597,2058.029,934.05,33.011,98.636,35.546,0.542,17.224,2.217,1.071,1.072,2.671,2.715,1.03,0.43,2.392,0.139,-63.107,-29.262,2.85,2.457,0.026,2.139],[5760.5,0.15,8.65,2055.124,884.891,32.473,96.116,34.581,0.543,17.512,2.236,1.13,1.143,2.752,2.834,1.043,0.447,2.335,0.136,-62.01,-32.618,2.893,2.403,0.021,2.157],[5761.0,0.217,8.675,2055.103,833.589,31.489,94.55,34.065,0.542,17.924,2.298,1.2,1.247,2.798,2.539,1.126,0.501,2.251,0.13,-60.533,-31.85,3.115,2.312,0.024,2.181],[5761.5,0.296,8.678,2056.199,793.895,29.793,93.849,33.751,0.535,18.508,2.37,1.26,1.355,3.032,2.41,1.243,0.562,2.213,0.118,-58.666,-29.378,3.395,2.243,0.03,2.211],[5762.0,0.312,8.689,2058.427,764.414,28.023,93.212,33.868,0.529,19.54,2.433,1.308,1.427,3.835,4.311,1.355,0.614,2.205,0.104,-56.085,-27.138,3.66,2.212,0.018,2.242],[5762.5,0.342,8.693,2065.192,733.159,27.499,92.019,34.94,0.53,20.402,2.429,1.364,1.457,4.867,4.005,1.414,0.64,2.209,0.1,-52.867,-21.454,3.83,2.221,-0.002,2.258],[5763.0,0.28,8.681,2071.771,693.104,30.033,90.675,36.647,0.542,20.328,2.387,1.443,1.479,4.437,2.778,1.37,0.619,2.214,0.114,-49.739,-12.495,3.788,2.273,0.001,2.244],[5763.5,0.306,8.669,2066.11,647.686,29.563,89.987,37.731,0.575,20.111,2.415,1.544,1.531,3.303,2.921,1.24,0.58,2.136,0.133,-47.136,-10.828,3.626,2.323,0.025,2.228],[5764.0,0.295,8.693,2048.515,604.953,29.088,90.12,37.818,0.647,21.058,2.535,1.653,1.639,2.779,3.073,1.098,0.587,1.888,0.122,-44.655,-20.136,3.605,2.286,0.043,2.257],[5764.5,0.292,8.786,2043.119,569.78,25.174,90.392,37.699,0.724,22.419,2.735,1.755,1.814,2.856,4.595,1.011,0.634,1.612,0.075,-41.645,-31.977,3.731,2.192,0.051,2.346],[5765.0,0.25,8.95,2058.484,539.03,21.04,90.008,38.572,0.765,23.277,2.875,1.855,2.046,3.518,5.817,0.979,0.664,1.486,0.034,-37.268,-40.699,3.829,2.147,0.049,2.445],[5765.5,0.203,9.045,2066.746,504.661,21.189,89.01,42.016,0.843,26.665,2.758,1.982,2.308,4.817,5.635,1.049,0.848,1.29,0.02,-31.083,-44.104,4.553,2.027,0.037,2.495],[5766.0,0.186,9.05,2050.424,463.118,21.081,88.396,47.631,0.941,32.391,2.746,2.159,2.59,5.978,4.189,1.235,1.153,1.08,0.022,-24.514,-41.24,5.96,1.92,0.027,2.492],[5766.5,0.172,9.06,2028.564,418.571,21.574,89.18,53.497,0.97,37.153,3.091,2.389,2.876,5.581,3.39,1.451,1.327,1.092,0.032,-19.355,-37.137,7.192,2.008,0.023,2.471],[5767.0,0.152,9.088,2022.036,379.074,24.223,91.348,59.684,1.01,41.968,3.511,2.638,3.107,4.7,3.278,1.606,1.429,1.124,0.047,-15.16,-38.118,8.293,2.151,0.015,2.448],[5767.5,0.152,9.105,2023.885,351.202,28.207,93.931,66.843,1.096,44.902,3.697,2.847,3.221,4.103,2.885,1.587,1.274,1.299,0.073,-10.821,-41.13,8.867,2.718,0.004,2.41],[5768.0,0.164,9.096,2026.788,336.238,33.205,95.795,74.222,1.14,45.138,3.791,2.974,3.239,3.666,2.649,1.503,0.925,1.688,0.117,-6.949,-37.958,8.751,3.646,0.007,2.357],[5768.5,0.164,9.096,2031.518,330.929,38.515,96.51,79.522,1.114,46.475,3.864,3.022,3.218,3.34,2.732,1.578,0.791,1.995,0.153,-4.528,-28.098,8.958,4.203,0.025,2.32],[5769.0,0.152,9.101,2039.166,331.023,41.641,96.601,80.478,1.082,48.138,3.808,3.021,3.191,3.172,2.865,1.695,0.799,2.123,0.145,-3.258,-20.116,9.363,4.351,0.037,2.329],[5769.5,0.153,9.101,2042.667,333.72,41.434,96.697,77.939,0.998,46.947,3.651,2.997,3.164,3.163,2.709,1.909,0.802,2.38,0.104,-2.419,-21.306,9.565,4.425,0.034,2.374],[5770.0,0.151,9.099,2035.238,336.99,38.883,96.747,75.667,0.792,43.219,3.575,2.967,3.137,3.211,2.583,2.557,0.809,3.156,0.068,-1.784,-25.237,9.904,4.544,0.032,2.422],[5770.5,0.152,9.099,2028.567,338.692,36.232,96.23,75.266,0.637,40.427,3.644,2.953,3.104,3.368,2.891,3.131,0.809,3.871,0.053,-1.336,-22.719,10.139,4.651,0.031,2.447],[5771.0,0.153,9.106,2030.469,337.54,34.841,95.045,76.442,0.597,40.084,3.699,2.963,3.069,3.558,2.835,3.321,0.784,4.247,0.053,-1.006,-14.904,10.116,4.795,0.025,2.446],[5771.5,0.155,9.102,2032.492,334.365,34.79,93.879,79.524,0.571,40.724,3.762,2.991,3.057,3.438,2.359,3.461,0.737,4.712,0.065,-0.93,-10.866,10.089,5.091,0.027,2.427],[5772.0,0.163,9.088,2033.643,330.896,36.958,93.51,82.516,0.564,41.748,3.842,3.022,3.085,3.06,2.196,3.506,0.707,4.956,0.08,-1.212,-14.112,10.09,5.294,0.038,2.407],[5772.5,0.136,9.101,2043.735,328.068,38.646,93.865,82.063,0.572,42.234,3.832,3.048,3.132,2.874,2.772,3.459,0.711,4.87,0.083,-1.536,-20.827,10.108,5.28,0.04,2.403],[5773.0,0.124,9.111,2059.431,325.755,38.076,93.98,78.733,0.627,42.476,3.695,3.07,3.169,3.029,2.917,3.176,0.758,4.24,0.072,-1.638,-26.663,10.068,4.956,0.029,2.42],[5773.5,0.118,9.107,2067.064,322.941,36.592,93.113,75.519,0.743,43.013,3.581,3.097,3.202,3.369,2.877,2.631,0.829,3.203,0.058,-1.653,-30.258,9.827,4.413,0.021,2.442],[5774.0,0.118,9.108,2065.508,318.996,35.659,91.7,73.889,0.862,42.997,3.553,3.135,3.255,3.529,2.854,2.177,0.866,2.519,0.05,-1.8,-31.518,9.511,4.079,0.029,2.457],[5774.5,0.126,9.102,2065.384,316.168,35.641,91.13,73.294,0.986,43.62,3.596,3.163,3.327,3.432,2.602,1.872,0.905,2.08,0.045,-2.098,-29.291,9.348,3.841,0.042,2.468],[5775.0,0.101,9.097,2082.166,318.772,35.592,92.367,72.168,1.098,45.178,3.699,3.137,3.352,3.383,2.95,1.676,0.967,1.738,0.038,-2.348,-24.75,9.375,3.602,0.039,2.48],[5775.5,0.094,9.102,2112.611,328.587,34.816,94.941,70.868,1.177,47.839,3.79,3.043,3.239,3.682,3.423,1.612,1.052,1.54,0.033,-2.339,-22.12,9.679,3.424,0.025,2.489],[5776.0,0.084,9.103,2131.624,342.434,34.74,97.545,74.154,1.205,53.305,3.713,2.92,2.995,4.355,3.769,1.792,1.271,1.416,0.036,-2.162,-24.16,11.015,3.235,0.016,2.478],[5776.5,0.087,9.096,2124.068,355.681,36.519,99.383,84.615,1.166,61.101,3.575,2.812,2.755,4.521,3.031,2.281,1.62,1.404,0.051,-2.016,-26.833,13.51,3.099,0.03,2.445],[5777.0,0.143,9.114,2099.841,366.768,38.578,100.806,94.81,1.15,68.788,3.523,2.727,2.61,3.324,1.753,2.694,1.915,1.408,0.068,-1.907,-26.209,15.816,3.065,0.057,2.413],[5777.5,0.13,9.169,2078.168,377.96,39.699,102.616,97.043,1.168,73.302,3.517,2.646,2.547,2.287,1.604,2.822,2.031,1.39,0.073,-1.766,-25.574,16.832,3.076,0.073,2.401],[5778.0,0.125,9.233,2063.943,393.633,42.031,105.007,93.159,1.185,72.361,3.487,2.54,2.495,2.006,2.086,2.716,1.929,1.409,0.068,-1.653,-27.833,16.425,3.166,0.062,2.406],[5778.5,0.107,9.259,2054.271,416.511,45.335,107.311,89.343,1.228,67.062,3.435,2.401,2.409,2.207,2.097,2.413,1.643,1.475,0.059,-1.518,-28.832,15.091,3.436,0.045,2.415],[5779.0,0.118,9.289,2049.224,443.746,45.945,108.809,87.601,1.268,61.99,3.327,2.254,2.288,2.521,2.109,2.156,1.394,1.55,0.052,-1.204,-27.245,13.956,3.726,0.04,2.421],[5779.5,0.124,9.326,2047.742,469.319,43.729,109.506,88.26,1.288,60.046,3.101,2.131,2.148,2.696,2.265,2.044,1.284,1.593,0.048,-0.717,-26.48,13.441,3.888,0.043,2.423],[5780.0,0.147,9.34,2048.783,489.917,41.234,110.087,91.0,1.343,60.077,2.849,2.041,2.006,2.699,1.923,1.894,1.16,1.635,0.048,-0.302,-29.347,12.937,4.162,0.047,2.418],[5780.5,0.141,9.341,2058.662,506.722,39.848,111.271,94.107,1.484,62.208,2.779,1.973,1.882,2.426,1.53,1.683,1.042,1.615,0.053,-0.067,-34.49,12.684,4.531,0.055,2.404],[5781.0,0.158,9.363,2073.036,523.057,39.687,113.102,96.009,1.66,67.658,2.912,1.912,1.786,2.039,1.361,1.578,1.083,1.467,0.059,0.095,-39.605,13.376,4.593,0.063,2.388],[5781.5,0.207,9.397,2073.408,542.135,40.497,115.04,96.211,1.769,73.861,3.01,1.845,1.71,1.768,1.279,1.615,1.239,1.31,0.061,0.265,-44.661,14.599,4.386,0.067,2.379],[5782.0,0.203,9.41,2056.219,565.271,42.275,116.623,96.482,1.803,77.552,2.958,1.769,1.646,1.667,1.271,1.687,1.364,1.238,0.058,0.425,-48.7,15.531,4.228,0.063,2.38],[5782.5,0.209,9.433,2041.789,591.425,45.63,117.771,99.211,1.813,79.236,2.952,1.691,1.593,1.646,1.263,1.752,1.452,1.207,0.053,0.498,-48.696,16.221,4.146,0.056,2.386],[5783.0,0.227,9.446,2045.464,615.552,48.916,118.526,103.316,1.808,80.638,3.111,1.625,1.545,1.667,1.287,1.825,1.521,1.199,0.047,0.34,-44.235,16.849,4.11,0.051,2.395],[5783.5,0.232,9.452,2055.732,632.018,49.899,118.897,106.147,1.814,83.913,3.226,1.582,1.5,1.731,1.302,1.909,1.619,1.181,0.045,-0.171,-38.09,17.689,4.058,0.046,2.398],[5784.0,0.316,9.456,2055.063,641.865,49.577,119.041,108.038,1.838,90.479,3.149,1.558,1.464,1.765,1.146,2.056,1.885,1.099,0.047,-0.969,-34.023,19.302,3.825,0.044,2.391],[5784.5,0.414,9.454,2048.657,650.04,50.041,119.301,111.64,1.815,96.398,3.18,1.538,1.451,1.694,1.039,2.255,2.175,1.038,0.051,-1.851,-34.898,20.871,3.569,0.05,2.379],[5785.0,0.479,9.456,2053.375,656.704,50.441,119.844,116.52,1.701,97.401,3.472,1.523,1.461,1.514,1.076,2.493,2.25,1.109,0.049,-2.723,-39.119,21.554,3.556,0.057,2.382],[5785.5,0.467,9.457,2062.062,659.668,50.679,120.131,119.725,1.52,94.886,3.788,1.516,1.478,1.387,1.284,2.838,2.196,1.296,0.039,-3.714,-39.588,21.877,3.7,0.055,2.408],[5786.0,0.468,9.461,2060.168,658.21,51.694,119.095,120.234,1.308,89.742,3.828,1.519,1.488,1.455,1.42,3.254,1.991,1.668,0.03,-5.089,-30.369,21.549,4.051,0.041,2.438],[5786.5,0.459,9.415,2051.407,649.823,52.528,116.114,119.345,1.112,83.386,3.623,1.539,1.509,1.569,1.196,3.639,1.689,2.185,0.03,-7.068,-15.354,20.546,4.541,0.032,2.439],[5787.0,0.435,9.274,2045.894,631.737,52.426,111.733,116.81,0.993,78.314,3.622,1.583,1.578,1.503,1.067,3.896,1.502,2.604,0.042,-9.731,-9.65,19.73,4.882,0.049,2.405],[5787.5,0.421,9.149,2041.754,608.649,51.173,107.182,111.02,0.924,71.682,3.916,1.643,1.706,1.332,1.25,3.87,1.318,2.964,0.06,-13.024,-20.104,18.281,5.171,0.085,2.368],[5788.0,0.421,9.131,2035.651,593.266,49.007,103.187,101.603,0.869,62.312,4.101,1.686,1.847,1.31,1.98,3.527,1.103,3.206,0.069,-16.925,-32.592,15.687,5.275,0.124,2.351],[5788.5,0.363,9.123,2030.52,598.799,46.364,99.628,87.625,0.799,52.848,3.841,1.67,1.91,1.653,3.018,3.19,0.996,3.208,0.07,-21.973,-36.13,13.048,4.866,0.155,2.348],[5789.0,0.126,8.951,2029.4,631.878,42.963,96.357,68.583,0.723,45.879,3.329,1.583,1.83,2.442,3.339,3.023,1.173,2.69,0.071,-29.179,-32.81,11.159,3.717,0.155,2.339],[5789.5,0.084,8.815,2032.923,692.345,39.242,94.02,49.062,0.633,38.886,2.699,1.444,1.618,3.605,3.982,2.874,1.467,1.999,0.077,-38.476,-29.445,9.325,2.431,0.117,2.313],[5790.0,0.073,8.804,2039.889,771.941,35.925,93.731,36.148,0.517,30.641,2.096,1.295,1.361,4.883,4.133,2.647,1.641,1.623,0.086,-47.982,-28.52,7.047,1.614,0.073,2.276],[5790.5,0.067,8.8,2049.154,855.931,33.476,95.998,31.027,0.467,25.167,1.763,1.168,1.145,4.842,2.904,2.327,1.774,1.319,0.096,-55.386,-29.303,5.544,1.166,0.047,2.238],[5791.0,0.084,8.797,2056.902,930.862,32.3,100.174,28.653,0.489,22.463,1.715,1.074,0.997,2.79,1.839,2.027,1.815,1.116,0.103,-59.468,-33.029,5.029,1.028,0.028,2.207],[5791.5,0.126,8.784,2060.783,992.794,32.158,105.111,26.353,0.546,21.006,1.789,1.007,0.911,1.281,1.089,1.672,1.741,0.956,0.107,-60.292,-40.349,4.596,0.978,0.008,2.185],[5792.0,0.161,8.776,2063.499,1041.085,32.331,110.094,25.868,0.608,20.405,1.895,0.961,0.872,0.832,1.108,1.338,1.643,0.813,0.111,-59.018,-41.556,4.14,0.935,0.002,2.167],[5792.5,0.175,8.762,2065.273,1067.895,33.017,114.642,30.088,0.653,21.541,2.066,0.936,0.876,0.825,1.304,1.196,1.525,0.789,0.116,-57.109,-29.976,3.978,0.98,0.012,2.15],[5793.0,0.243,8.766,2059.865,1060.329,34.945,117.919,40.161,0.758,25.404,2.336,0.943,0.915,0.998,1.715,1.143,1.274,0.919,0.123,-55.146,-15.722,4.42,1.343,0.028,2.138],[5793.5,0.209,8.782,2052.203,1015.352,36.942,119.065,54.677,0.93,33.176,2.665,0.985,0.978,1.249,1.908,1.249,1.121,1.112,0.131,-52.48,-10.233,5.99,1.975,0.038,2.134],[5794.0,0.248,8.777,2054.358,943.937,38.647,117.925,69.47,1.014,43.117,2.929,1.059,1.063,1.5,1.432,1.649,1.226,1.335,0.134,-48.144,-19.372,8.535,2.562,0.043,2.144],[5794.5,0.338,8.775,2062.069,865.157,42.257,115.288,77.672,1.017,50.433,3.019,1.156,1.178,1.523,1.377,1.963,1.355,1.45,0.13,-42.479,-36.252,10.189,2.793,0.055,2.171],[5795.0,0.308,8.778,2063.23,800.305,44.548,112.253,74.008,0.978,51.21,2.923,1.25,1.308,1.359,2.237,2.081,1.394,1.493,0.116,-37.896,-46.04,10.343,2.755,0.074,2.213],[5795.5,0.232,8.782,2060.907,764.753,43.881,109.365,60.807,0.821,45.33,2.577,1.308,1.383,1.512,6.711,2.452,1.418,1.737,0.098,-37.489,-46.065,10.069,2.638,0.077,2.254],[5796.0,0.16,8.757,2059.538,756.358,40.898,106.439,45.387,0.652,35.833,2.101,1.322,1.347,2.473,6.508,2.493,1.406,1.804,0.092,-42.384,-48.272,8.359,2.208,0.051,2.267],[5796.5,0.158,8.723,2052.159,756.184,37.114,103.293,33.452,0.555,25.233,1.833,1.322,1.255,4.232,2.668,1.831,1.101,1.692,0.11,-50.3,-56.197,5.289,1.77,0.031,2.234],[5797.0,0.157,8.732,2040.716,748.138,34.29,100.355,26.976,0.506,18.194,1.848,1.337,1.189,3.454,2.214,1.372,0.793,1.736,0.145,-57.26,-64.947,3.553,1.664,0.038,2.181],[5797.5,0.141,8.746,2036.406,735.203,33.757,98.318,25.696,0.484,16.404,2.034,1.36,1.183,2.096,2.012,1.351,0.737,1.834,0.174,-60.712,-69.347,3.34,1.682,0.05,2.143],[5798.0,0.107,8.761,2038.791,732.372,34.788,97.724,27.626,0.469,16.673,2.34,1.365,1.231,1.715,2.776,1.385,0.734,1.887,0.184,-60.645,-66.562,3.32,1.678,0.045,2.13],[5798.5,0.101,8.781,2039.591,742.983,34.714,98.642,30.618,0.467,17.208,2.645,1.346,1.303,2.046,3.971,1.449,0.774,1.874,0.175,-58.084,-60.607,3.455,1.658,0.033,2.138],[5799.0,0.101,8.793,2038.924,749.962,33.098,100.522,32.511,0.462,17.731,2.831,1.333,1.371,3.189,4.43,1.532,0.799,1.92,0.151,-53.699,-59.22,3.613,1.679,0.031,2.17],[5799.5,0.124,8.81,2040.927,733.658,31.923,102.362,33.55,0.479,20.948,2.787,1.363,1.423,5.022,7.986,1.7,0.928,1.875,0.123,-48.059,-61.789,4.199,1.685,0.031,2.219],[5800.0,0.115,8.819,2044.824,692.27,32.978,103.163,38.124,0.538,28.467,2.629,1.445,1.463,6.325,6.87,2.285,1.223,1.882,0.108,-41.822,-60.687,6.377,1.911,0.025,2.255],[5800.5,0.116,8.83,2050.415,638.929,35.755,102.667,50.212,0.599,36.982,2.713,1.565,1.512,5.167,2.761,2.89,1.329,2.201,0.112,-34.229,-54.812,8.841,2.501,0.027,2.263],[5801.0,0.232,8.89,2053.016,588.516,37.468,101.596,66.669,0.561,42.87,2.978,1.699,1.602,2.985,1.682,3.932,1.2,3.351,0.116,-24.027,-51.814,10.845,3.405,0.046,2.272],[5801.5,0.237,8.988,2049.422,552.069,39.32,101.059,78.83,0.48,46.489,3.168,1.811,1.756,1.761,1.246,5.035,1.109,4.545,0.101,-12.97,-54.583,12.192,4.081,0.061,2.305],[5802.0,0.212,9.052,2044.697,533.431,42.393,101.432,81.508,0.456,48.155,3.333,1.875,1.939,1.536,1.933,5.333,1.095,4.87,0.072,-4.857,-56.548,12.426,4.211,0.057,2.36],[5802.5,0.17,9.085,2042.569,528.398,43.011,102.035,76.953,0.471,46.384,3.582,1.893,2.06,2.013,3.062,5.058,1.062,4.756,0.047,-0.927,-53.248,12.115,4.235,0.045,2.414],[5803.0,0.179,9.068,2038.248,529.002,40.963,102.194,72.095,0.489,42.638,3.612,1.89,2.065,3.127,3.197,4.546,0.981,4.637,0.032,0.529,-43.894,11.348,4.297,0.03,2.452],[5803.5,0.164,9.078,2032.557,530.35,38.738,102.215,72.112,0.514,41.849,3.412,1.886,1.993,3.936,2.358,4.278,0.972,4.424,0.029,1.243,-31.221,11.197,4.281,0.022,2.461],[5804.0,0.176,9.093,2033.431,533.528,39.211,103.118,76.263,0.59,45.526,3.247,1.874,1.893,3.228,1.799,3.977,1.063,3.77,0.035,1.855,-22.425,11.907,4.161,0.03,2.442],[5804.5,0.183,9.074,2044.937,544.111,42.182,105.452,81.17,0.662,49.507,3.209,1.838,1.784,2.348,1.521,3.688,1.117,3.304,0.05,2.433,-24.491,12.449,4.137,0.047,2.404],[5805.0,0.198,9.047,2060.523,567.041,43.626,108.618,85.925,0.728,51.849,3.233,1.764,1.674,1.81,1.395,3.379,1.088,3.103,0.069,2.938,-31.953,12.494,4.266,0.059,2.358],[5805.5,0.183,9.043,2066.307,600.691,44.539,111.397,89.665,0.819,53.411,3.241,1.665,1.574,1.632,1.303,2.994,1.047,2.857,0.084,3.239,-34.798,12.459,4.417,0.068,2.321],[5806.0,0.166,9.046,2057.496,638.086,45.722,112.985,90.151,0.894,54.54,3.202,1.567,1.494,1.614,1.268,2.757,1.058,2.61,0.088,3.38,-31.975,12.566,4.411,0.073,2.303],[5806.5,0.166,9.044,2046.677,671.749,45.86,113.484,88.322,0.93,55.392,3.187,1.489,1.432,1.54,1.174,2.705,1.113,2.434,0.085,3.575,-32.59,12.851,4.288,0.072,2.301],[5807.0,0.204,9.062,2043.828,697.304,46.833,113.466,86.462,0.956,56.804,3.268,1.434,1.384,1.528,1.137,2.724,1.202,2.272,0.08,3.886,-39.592,13.292,4.11,0.072,2.304],[5807.5,0.2,9.077,2043.888,716.305,49.494,113.369,84.952,0.978,58.582,3.367,1.396,1.345,1.565,1.25,2.753,1.302,2.119,0.075,4.155,-44.542,13.749,3.923,0.073,2.311],[5808.0,0.282,9.054,2039.689,731.89,48.776,113.4,84.2,1.0,60.502,3.327,1.366,1.313,1.63,1.394,2.772,1.392,1.995,0.069,4.336,-43.305,14.16,3.778,0.072,2.321],[5808.5,0.318,9.117,2031.257,742.606,47.638,113.716,85.435,1.052,63.875,3.187,1.347,1.29,1.568,1.191,2.75,1.543,1.797,0.063,4.498,-40.3,14.759,3.565,0.07,2.333],[5809.0,0.324,9.281,2021.426,746.656,47.572,114.326,88.928,1.126,67.538,3.119,1.339,1.274,1.493,1.137,2.677,1.726,1.561,0.058,4.681,-34.452,15.373,3.315,0.067,2.344],[5809.5,0.291,9.372,2013.814,746.606,48.018,114.854,92.075,1.193,69.092,3.181,1.339,1.265,1.475,1.172,2.553,1.802,1.416,0.055,4.839,-24.687,15.54,3.2,0.06,2.351],[5810.0,0.291,9.372,2011.527,743.544,48.617,114.877,92.313,1.246,69.106,3.17,1.345,1.266,1.474,1.155,2.417,1.789,1.351,0.055,4.911,-18.885,15.363,3.187,0.056,2.35],[5810.5,0.293,9.377,2018.439,738.45,48.643,114.438,91.831,1.265,68.234,3.002,1.354,1.285,1.442,1.141,2.322,1.724,1.348,0.06,4.943,-20.257,15.006,3.232,0.062,2.341],[5811.0,0.237,9.391,2032.096,736.5,47.794,114.001,92.518,1.253,66.021,2.893,1.358,1.319,1.431,1.121,2.256,1.599,1.415,0.063,4.988,-23.064,14.433,3.357,0.072,2.333],[5811.5,0.201,9.39,2047.12,741.84,46.209,113.946,90.7,1.2,62.293,2.944,1.348,1.336,1.563,1.059,2.228,1.42,1.581,0.061,5.02,-23.591,13.651,3.581,0.076,2.337],[5812.0,0.207,9.375,2063.104,752.625,44.649,114.187,85.116,1.103,58.369,3.068,1.329,1.315,1.826,1.314,2.364,1.299,1.823,0.055,5.047,-20.183,13.209,3.775,0.066,2.35],[5812.5,0.206,9.378,2076.48,763.089,44.571,114.337,81.959,0.941,55.087,3.117,1.31,1.273,1.987,1.693,2.828,1.276,2.218,0.05,5.18,-13.304,13.342,3.88,0.059,2.36],[5813.0,0.202,9.456,2080.921,767.25,45.961,114.32,84.874,0.798,53.567,3.044,1.303,1.239,1.74,1.27,3.377,1.278,2.641,0.051,5.394,-9.534,13.698,3.978,0.07,2.357],[5813.5,0.132,9.619,2083.511,766.67,47.595,114.567,89.911,0.711,55.331,3.024,1.304,1.225,1.422,1.026,4.077,1.333,3.047,0.055,5.587,-10.624,14.622,4.069,0.082,2.348],[5814.0,0.172,9.685,2085.529,769.925,48.394,115.416,93.367,0.652,58.715,3.143,1.299,1.222,1.256,1.002,4.773,1.415,3.369,0.054,5.763,-12.28,15.837,4.153,0.078,2.348],[5814.5,0.236,9.703,2077.993,780.35,48.664,116.679,94.104,0.657,61.701,3.203,1.281,1.224,1.254,1.025,4.911,1.46,3.366,0.048,5.873,-15.338,16.465,4.185,0.068,2.362],[5815.0,0.259,9.844,2068.327,791.821,48.886,117.822,93.196,0.722,64.862,3.07,1.263,1.225,1.322,1.082,4.607,1.517,3.048,0.041,5.789,-21.139,16.91,4.138,0.063,2.38],[5815.5,0.299,9.956,2065.701,799.154,49.447,118.429,93.119,0.803,68.289,2.951,1.251,1.222,1.46,1.16,4.297,1.623,2.656,0.038,5.564,-21.925,17.578,4.022,0.061,2.391],[5816.0,0.332,9.95,2064.099,799.138,49.981,118.502,95.653,0.844,70.662,2.955,1.251,1.217,1.582,1.247,4.226,1.74,2.433,0.037,5.443,-15.291,18.212,3.889,0.063,2.393],[5816.5,0.463,9.921,2060.396,790.02,50.45,118.372,99.149,0.859,73.235,2.991,1.266,1.22,1.592,1.23,4.355,1.916,2.28,0.039,5.517,-8.601,19.103,3.711,0.069,2.389],[5817.0,0.5,9.819,2055.271,773.579,51.627,118.314,102.068,0.84,76.531,3.026,1.293,1.236,1.497,1.139,4.765,2.127,2.24,0.043,5.647,-5.415,20.397,3.563,0.082,2.379],[5817.5,0.493,9.671,2050.221,755.155,51.192,118.328,104.311,0.806,78.699,3.093,1.324,1.263,1.402,1.098,5.175,2.222,2.33,0.049,5.692,-3.669,21.29,3.556,0.094,2.366],[5818.0,0.474,9.51,2052.857,739.821,51.482,118.277,106.616,0.767,77.96,3.209,1.352,1.297,1.393,1.125,5.42,2.055,2.686,0.052,5.653,-5.422,21.191,3.873,0.093,2.36],[5818.5,0.396,9.432,2064.429,729.943,51.901,118.043,108.643,0.717,73.801,3.281,1.37,1.333,1.46,1.188,5.485,1.634,3.473,0.049,5.524,-10.853,20.08,4.686,0.075,2.369],[5819.0,0.338,9.381,2071.348,722.001,51.711,117.631,109.831,0.709,68.885,3.225,1.385,1.366,1.548,1.26,5.094,1.24,4.145,0.044,5.267,-12.699,18.422,5.577,0.058,2.384],[5819.5,0.269,9.356,2064.605,711.603,51.014,117.161,110.925,0.743,65.881,3.035,1.405,1.393,1.635,1.312,4.531,1.048,4.325,0.041,5.038,-8.772,17.151,6.088,0.057,2.395],[5820.0,0.21,9.358,2051.572,699.768,50.023,116.698,112.119,0.799,63.34,2.827,1.429,1.414,1.708,1.394,3.976,0.938,4.232,0.041,5.049,-3.392,16.123,6.395,0.066,2.396],[5820.5,0.158,9.365,2046.632,689.761,48.996,116.169,111.627,0.966,61.174,2.771,1.45,1.432,1.742,1.394,3.121,0.848,3.656,0.044,5.255,-3.94,14.897,6.52,0.072,2.391],[5821.0,0.069,9.319,2054.881,680.767,48.366,115.463,109.427,1.181,60.966,2.888,1.469,1.448,1.726,1.333,2.374,0.823,2.884,0.047,5.464,-10.254,14.11,6.364,0.069,2.384],[5821.5,0.054,9.132,2062.891,670.541,48.378,114.621,107.969,1.306,62.49,3.009,1.491,1.465,1.683,1.301,2.109,0.853,2.48,0.046,5.637,-13.374,14.045,6.118,0.064,2.387],[5822.0,0.05,8.924,2059.359,661.025,48.505,113.825,107.207,1.403,67.065,3.031,1.513,1.479,1.653,1.292,2.047,0.969,2.14,0.04,5.821,-6.476,14.663,5.658,0.069,2.407],[5822.5,0.044,8.961,2055.865,657.054,48.286,113.261,105.968,1.486,72.13,2.962,1.522,1.49,1.668,1.641,2.043,1.123,1.829,0.031,6.037,4.802,15.501,5.139,0.074,2.436],[5823.0,0.044,9.154,2060.67,659.805,47.696,113.04,105.145,1.478,73.265,2.865,1.516,1.493,1.696,1.758,2.107,1.2,1.756,0.025,6.25,17.014,15.891,4.914,0.064,2.453],[5823.5,0.044,9.224,2064.035,666.161,47.164,113.192,105.407,1.402,71.463,2.956,1.501,1.479,1.755,1.383,2.223,1.201,1.853,0.027,6.423,28.723,15.898,4.914,0.058,2.447],[5824.0,0.044,9.258,2061.709,673.146,46.904,113.617,105.533,1.343,69.704,3.233,1.486,1.447,1.875,1.43,2.291,1.168,1.962,0.032,6.52,31.606,15.712,4.991,0.059,2.427],[5824.5,0.044,9.25,2061.713,679.139,46.697,114.083,105.018,1.336,69.559,3.319,1.472,1.41,1.918,1.427,2.281,1.153,1.978,0.038,6.574,18.891,15.562,5.009,0.053,2.41],[5825.0,0.044,9.187,2059.798,683.005,46.536,114.42,104.195,1.367,70.365,3.11,1.464,1.386,1.766,1.273,2.213,1.153,1.919,0.041,6.606,-4.879,15.441,4.97,0.044,2.4],[5825.5,0.044,9.246,2052.992,686.602,46.815,114.655,103.074,1.394,70.775,2.898,1.456,1.375,1.538,1.335,2.136,1.13,1.89,0.044,6.519,-19.867,15.204,4.993,0.048,2.392],[5826.0,0.043,9.34,2051.061,694.379,47.577,114.967,102.236,1.39,70.752,2.892,1.44,1.365,1.444,1.189,2.118,1.108,1.911,0.044,6.259,-15.333,15.036,5.035,0.062,2.388],[5826.5,0.038,9.349,2056.896,705.606,48.081,115.5,102.321,1.355,69.395,3.06,1.417,1.351,1.502,1.14,2.145,1.078,1.997,0.041,5.938,-8.714,14.834,5.112,0.078,2.396],[5827.0,0.08,9.336,2061.44,713.77,48.087,116.166,103.374,1.257,65.532,3.203,1.401,1.338,1.571,1.187,2.235,0.994,2.266,0.036,5.659,-20.371,14.31,5.359,0.085,2.408],[5827.5,0.333,9.17,2062.604,715.587,48.29,116.727,105.846,1.169,62.279,3.154,1.397,1.331,1.575,1.144,2.333,0.929,2.513,0.038,5.441,-40.654,13.924,5.564,0.09,2.403],[5828.0,0.287,8.976,2060.365,714.012,48.878,117.034,108.38,1.159,61.998,3.084,1.401,1.328,1.55,1.132,2.386,0.951,2.512,0.046,5.224,-50.494,14.116,5.512,0.108,2.379],[5828.5,0.091,8.854,2054.28,712.547,49.287,117.215,109.806,1.172,62.975,3.148,1.403,1.332,1.553,1.089,2.424,1.001,2.424,0.057,4.862,-52.057,14.51,5.382,0.119,2.353],[5829.0,0.078,8.627,2049.812,712.049,49.396,117.509,110.262,1.243,67.759,3.227,1.404,1.34,1.551,1.077,2.477,1.181,2.152,0.063,4.261,-52.969,15.743,5.02,0.108,2.339],[5829.5,0.095,8.377,2048.198,710.265,49.449,117.76,110.935,1.372,76.722,3.329,1.408,1.346,1.541,1.115,2.547,1.508,1.726,0.065,3.435,-53.192,17.865,4.447,0.089,2.334],[5830.0,0.118,8.277,2043.085,701.795,49.291,117.145,112.688,1.405,82.69,3.497,1.425,1.352,1.541,1.173,2.693,1.649,1.639,0.066,2.379,-50.648,19.28,4.347,0.079,2.333],[5830.5,0.112,8.299,2033.459,681.139,49.3,114.722,115.61,1.308,82.911,3.54,1.468,1.378,1.46,1.187,2.923,1.51,1.96,0.066,1.026,-44.193,19.471,4.82,0.076,2.338],[5831.0,0.112,8.378,2029.353,651.358,49.922,110.34,117.723,1.217,80.209,3.346,1.535,1.448,1.328,1.075,3.023,1.312,2.318,0.064,-0.801,-33.676,18.782,5.334,0.072,2.349],[5831.5,0.107,8.486,2042.573,625.937,50.283,104.975,117.424,1.173,76.702,3.084,1.598,1.58,1.219,1.048,2.975,1.17,2.549,0.061,-3.37,-25.014,17.821,5.66,0.064,2.362],[5832.0,0.107,8.499,2073.336,619.282,49.18,100.127,112.718,1.135,72.589,2.923,1.615,1.742,1.266,0.46,2.965,1.084,2.74,0.056,-6.823,-20.681,17.173,5.885,0.065,2.374],[5832.5,0.107,8.524,2099.28,636.909,46.787,96.665,100.814,1.133,65.031,2.89,1.57,1.819,1.71,4.96,2.66,0.949,2.793,0.046,-11.205,-18.526,15.329,5.992,0.073,2.394],[5833.0,0.106,8.556,2102.706,673.393,44.122,94.407,84.285,1.232,57.413,2.923,1.485,1.721,2.968,16.999,1.978,0.9,2.26,0.036,-16.785,-20.331,12.259,5.179,0.07,2.415],[5833.5,0.101,8.559,2089.543,711.931,41.268,92.932,69.7,1.267,53.874,2.908,1.405,1.526,4.66,9.378,1.67,1.263,1.445,0.034,-24.225,-28.119,10.755,3.488,0.08,2.417],[5834.0,0.101,8.586,2072.411,733.016,38.768,92.122,58.519,1.102,50.469,2.647,1.364,1.36,4.088,2.973,1.93,2.133,0.962,0.045,-33.741,-37.717,10.676,2.062,0.116,2.381],[5834.5,0.101,8.578,2057.319,732.662,37.513,91.892,48.397,0.881,43.335,2.289,1.365,1.28,2.535,3.676,2.178,2.461,0.919,0.071,-43.805,-47.45,9.768,1.503,0.123,2.316],[5835.0,0.095,8.555,2048.44,723.616,36.574,91.864,41.765,0.734,34.579,2.085,1.382,1.292,1.824,2.411,2.122,1.855,1.203,0.107,-51.129,-58.068,7.982,1.644,0.076,2.249],[5835.5,0.095,8.562,2044.263,715.594,35.711,91.229,42.699,0.684,30.974,2.084,1.397,1.391,1.665,1.799,2.035,1.474,1.381,0.131,-53.116,-68.389,7.111,1.791,0.034,2.211],[5836.0,0.095,8.577,2042.801,704.748,35.43,89.164,48.618,0.716,32.97,2.126,1.419,1.578,1.901,2.336,1.957,1.434,1.363,0.124,-51.114,-74.62,7.135,1.848,0.029,2.224],[5836.5,0.095,8.572,2045.324,680.594,35.095,85.443,51.412,0.766,36.414,2.12,1.469,1.853,2.703,1.739,1.862,1.423,1.309,0.093,-49.444,-75.963,7.277,1.899,0.04,2.284],[5837.0,0.095,8.563,2048.74,636.743,32.864,80.452,46.588,0.784,36.971,2.147,1.57,2.184,4.85,13.343,1.827,1.428,1.28,0.056,-50.496,-75.164,7.315,1.901,0.053,2.37],[5837.5,0.095,8.574,2045.98,580.371,28.054,74.747,37.17,0.74,31.809,2.316,1.723,2.497,11.797,84.876,1.649,1.253,1.325,0.024,-53.268,-73.405,6.286,1.851,0.058,2.469],[5838.0,0.095,8.592,2038.772,529.194,23.953,68.989,29.307,0.602,23.898,2.528,1.89,2.724,29.306,83.851,1.466,1.01,1.463,0.006,-56.356,-69.818,4.532,1.647,0.05,2.561],[5838.5,0.095,8.594,2036.51,498.109,21.559,64.101,25.403,0.458,17.819,2.466,2.008,2.837,30.072,21.077,1.459,0.896,1.632,0.002,-59.275,-64.704,3.398,1.403,0.037,2.602],[5839.0,0.095,8.579,2040.092,493.962,18.698,61.201,23.586,0.387,14.583,2.368,2.024,2.828,23.224,16.985,1.453,0.86,1.69,0.003,-61.691,-57.309,2.876,1.241,0.047,2.583],[5839.5,0.095,8.564,2046.347,516.614,18.363,61.164,21.815,0.368,13.281,2.521,1.936,2.697,19.71,34.173,1.389,0.876,1.588,0.006,-63.368,-47.009,2.613,1.109,0.075,2.559],[5840.0,0.095,8.569,2058.385,554.089,17.173,63.611,20.381,0.381,13.024,2.674,1.805,2.48,16.398,36.225,1.325,0.985,1.364,0.004,-64.307,-35.988,2.576,0.98,0.084,2.571],[5840.5,0.091,8.594,2072.148,583.546,14.609,66.872,20.223,0.406,13.263,2.539,1.714,2.263,12.71,41.229,1.28,1.155,1.12,0.001,-64.832,-26.299,2.649,0.857,0.073,2.608],[5841.0,0.09,8.596,2079.835,588.234,14.025,69.307,21.203,0.429,14.111,2.174,1.7,2.142,8.737,26.583,1.261,1.327,0.958,0.0,-65.532,-22.758,2.761,0.776,0.063,2.621],[5841.5,0.09,8.593,2086.19,573.599,14.712,70.631,22.953,0.465,16.983,1.861,1.743,2.174,4.663,1.578,1.397,1.89,0.779,0.006,-66.093,-27.813,3.344,0.679,0.044,2.559],[5842.0,0.09,8.601,2091.107,561.256,15.064,72.021,25.983,0.502,20.661,1.843,1.782,2.332,2.872,0.65,1.61,2.269,0.727,0.036,-65.167,-32.605,4.135,0.691,0.03,2.436],[5842.5,0.09,8.582,2089.738,569.868,17.16,74.65,28.707,0.519,22.88,2.107,1.755,2.44,3.043,5.19,1.664,2.023,0.835,0.076,-63.119,-30.144,4.41,0.822,0.055,2.344],[5843.0,0.09,8.588,2090.62,608.063,21.416,78.202,28.577,0.517,22.736,2.53,1.645,2.283,5.696,91.743,1.604,1.746,0.934,0.067,-62.303,-25.865,4.244,0.913,0.079,2.353],[5843.5,0.09,8.611,2098.112,672.41,24.447,81.285,26.265,0.484,18.525,2.98,1.487,1.919,12.954,124.023,1.31,1.167,1.269,0.025,-63.759,-23.348,3.284,1.147,0.07,2.451],[5844.0,0.09,8.621,2106.866,746.864,23.37,83.308,24.506,0.451,13.798,2.929,1.339,1.545,12.159,31.38,0.952,0.601,1.639,0.006,-66.413,-21.753,2.202,1.397,0.042,2.545],[5844.5,0.078,8.618,2119.262,818.304,20.86,85.245,23.683,0.437,12.557,2.2,1.222,1.262,4.769,3.203,0.908,0.526,1.727,0.007,-68.995,-20.852,2.024,1.43,0.018,2.531],[5845.0,0.078,8.518,2131.366,898.182,20.517,88.487,22.935,0.433,13.073,1.769,1.113,1.071,1.907,2.942,0.967,0.571,1.695,0.025,-70.547,-20.932,2.136,1.389,0.022,2.423],[5845.5,0.094,8.403,2128.597,1012.838,23.266,93.216,22.731,0.429,14.196,1.685,0.987,0.942,1.136,1.413,1.042,0.632,1.649,0.054,-70.544,-23.335,2.278,1.341,0.038,2.309],[5846.0,0.108,8.369,2112.8,1155.574,27.164,98.087,23.937,0.408,15.677,1.677,0.865,0.846,1.029,1.043,1.3,0.778,1.666,0.071,-69.73,-25.925,2.689,1.287,0.044,2.238],[5846.5,0.118,8.368,2098.275,1286.037,29.96,101.628,25.73,0.4,16.824,1.655,0.778,0.778,1.146,1.63,1.524,1.005,1.548,0.076,-69.543,-21.469,3.113,1.169,0.047,2.203],[5847.0,0.118,8.329,2085.999,1378.218,32.134,103.524,26.177,0.425,17.32,1.648,0.726,0.736,1.267,1.432,1.543,1.273,1.24,0.079,-70.36,-13.389,3.345,0.99,0.053,2.178],[5847.5,0.118,8.285,2077.435,1434.504,34.59,104.351,25.002,0.47,17.767,1.639,0.697,0.714,1.27,1.01,1.451,1.515,0.969,0.087,-71.161,-12.43,3.465,0.854,0.057,2.148],[5848.0,0.118,8.266,2081.288,1460.743,36.673,104.638,23.256,0.517,18.645,1.624,0.685,0.704,1.352,1.33,1.343,1.733,0.783,0.096,-71.243,-18.587,3.541,0.762,0.059,2.121],[5848.5,0.118,8.252,2093.675,1461.587,37.742,104.59,22.357,0.56,19.31,1.651,0.684,0.706,1.645,1.577,1.258,1.919,0.659,0.101,-71.129,-24.206,3.59,0.696,0.056,2.107],[5849.0,0.117,8.254,2097.655,1449.652,38.018,104.33,23.063,0.601,19.528,1.758,0.69,0.717,1.737,1.436,1.159,1.972,0.587,0.103,-71.25,-29.128,3.547,0.668,0.05,2.104],[5849.5,0.113,8.255,2093.256,1440.261,37.741,104.03,24.872,0.635,19.777,1.88,0.694,0.73,1.486,1.669,1.036,1.789,0.582,0.103,-71.331,-31.634,3.352,0.701,0.048,2.107],[5850.0,0.113,8.249,2088.391,1440.256,36.909,103.848,26.426,0.658,19.92,1.907,0.694,0.732,1.495,2.095,0.898,1.41,0.648,0.1,-71.151,-28.593,3.012,0.808,0.051,2.113],[5850.5,0.107,8.22,2087.754,1438.605,35.716,103.804,26.911,0.661,19.649,1.868,0.695,0.722,1.913,2.39,0.796,1.081,0.744,0.098,-70.926,-22.531,2.686,0.931,0.053,2.119],[5851.0,0.107,8.21,2088.867,1424.721,34.942,103.738,26.023,0.649,18.707,1.852,0.702,0.712,2.299,2.333,0.732,0.872,0.85,0.097,-70.67,-13.353,2.427,1.045,0.052,2.123],[5851.5,0.107,8.231,2085.827,1403.646,34.672,103.488,24.884,0.627,16.897,1.825,0.712,0.713,1.718,1.766,0.664,0.675,1.005,0.098,-69.993,-4.47,2.13,1.19,0.047,2.125],[5852.0,0.107,8.261,2078.491,1389.258,34.803,103.08,24.723,0.597,15.141,1.779,0.72,0.724,0.992,1.208,0.615,0.529,1.173,0.1,-68.844,-3.182,1.877,1.323,0.039,2.123],[5852.5,0.101,8.267,2078.408,1386.635,35.106,102.64,24.948,0.593,15.291,1.78,0.721,0.734,0.807,1.424,0.639,0.554,1.177,0.102,-68.011,-7.162,1.94,1.319,0.035,2.118],[5853.0,0.101,8.254,2087.287,1384.895,35.196,102.272,24.491,0.639,18.264,1.807,0.722,0.737,1.097,2.331,0.775,0.925,0.922,0.104,-68.07,-8.801,2.552,1.1,0.039,2.114],[5853.5,0.097,8.243,2089.208,1364.794,35.184,101.904,23.618,0.688,21.672,1.81,0.733,0.746,1.554,2.368,0.955,1.551,0.646,0.107,-68.561,-7.735,3.363,0.837,0.043,2.111],[5854.0,0.097,8.252,2080.446,1315.676,34.991,101.081,23.194,0.706,23.257,1.821,0.76,0.772,1.439,1.834,1.087,2.119,0.519,0.111,-68.65,-8.471,3.924,0.695,0.042,2.111],[5854.5,0.096,8.254,2069.721,1245.789,34.828,99.366,23.883,0.71,23.728,1.832,0.803,0.813,1.337,1.632,1.187,2.553,0.467,0.114,-68.217,-11.35,4.299,0.628,0.041,2.116],[5855.0,0.096,8.24,2068.375,1167.76,35.073,96.999,25.776,0.7,23.916,1.812,0.856,0.871,1.443,2.197,1.279,2.81,0.455,0.118,-67.625,-14.668,4.568,0.603,0.039,2.124],[5855.5,0.096,8.232,2082.072,1082.985,35.119,94.707,28.086,0.677,23.807,1.79,0.923,0.956,1.435,2.399,1.341,2.675,0.507,0.124,-67.202,-15.787,4.636,0.65,0.033,2.132],[5856.0,0.095,8.239,2100.271,990.016,34.279,92.885,29.856,0.653,23.341,1.855,1.01,1.068,1.496,1.79,1.334,2.163,0.633,0.13,-67.038,-13.883,4.451,0.781,0.031,2.14],[5856.5,0.096,8.243,2103.19,899.419,33.054,91.176,31.09,0.643,23.077,2.096,1.112,1.187,2.1,2.529,1.296,1.701,0.77,0.131,-67.166,-11.438,4.255,0.938,0.042,2.162],[5857.0,0.096,8.252,2093.737,820.531,31.839,88.935,31.999,0.652,23.865,2.447,1.219,1.304,3.393,6.172,1.292,1.556,0.834,0.118,-67.59,-6.722,4.308,1.031,0.047,2.203],[5857.5,0.112,8.255,2089.074,754.125,30.088,86.302,32.277,0.679,26.733,2.57,1.326,1.439,3.949,5.92,1.411,1.96,0.746,0.104,-68.218,0.372,4.907,0.956,0.037,2.246],[5858.0,0.118,8.174,2093.919,704.864,29.654,84.413,32.014,0.699,30.174,2.453,1.419,1.59,3.869,2.822,1.607,2.692,0.606,0.098,-68.807,1.932,5.734,0.802,0.041,2.268],[5858.5,0.118,8.1,2101.726,680.331,28.178,84.0,31.669,0.692,31.123,2.574,1.47,1.696,4.819,3.38,1.715,3.037,0.565,0.089,-69.144,-10.02,6.057,0.741,0.07,2.292],[5859.0,0.118,8.09,2103.266,676.369,25.215,84.513,31.01,0.655,29.157,2.844,1.478,1.699,6.762,15.526,1.771,2.991,0.599,0.063,-69.294,-22.967,5.917,0.738,0.085,2.346],[5859.5,0.12,8.094,2092.009,680.994,23.69,84.995,29.917,0.551,24.372,2.795,1.468,1.633,7.606,21.574,1.791,2.296,0.876,0.039,-69.431,-17.739,5.045,0.864,0.072,2.404],[5860.0,0.107,8.116,2077.901,691.74,24.707,85.23,28.404,0.444,19.366,2.468,1.446,1.55,6.017,9.383,1.739,1.433,1.268,0.036,-69.602,-5.092,3.948,1.046,0.052,2.414],[5860.5,0.118,8.12,2072.668,715.495,26.806,85.86,26.736,0.42,17.645,2.315,1.398,1.465,4.504,4.465,1.7,1.303,1.315,0.05,-69.754,-7.622,3.648,1.045,0.041,2.371],[5861.0,0.118,8.115,2075.19,757.604,27.916,87.46,25.379,0.431,18.043,2.312,1.32,1.372,4.063,3.357,1.688,1.462,1.159,0.068,-69.785,-25.321,3.715,0.946,0.043,2.319],[5861.5,0.118,8.128,2078.832,811.492,27.953,89.64,24.483,0.441,18.781,2.364,1.232,1.272,4.092,4.188,1.7,1.689,1.018,0.075,-69.61,-39.331,3.826,0.849,0.051,2.291],[5862.0,0.124,8.135,2082.565,861.523,28.024,91.445,23.989,0.453,19.417,2.348,1.161,1.18,3.93,4.592,1.764,2.097,0.855,0.073,-69.242,-43.376,4.079,0.732,0.057,2.284],[5862.5,0.124,8.139,2084.989,894.965,28.837,92.539,24.099,0.47,19.848,2.252,1.117,1.115,3.303,3.393,1.806,2.57,0.711,0.076,-68.644,-43.561,4.333,0.631,0.056,2.271],[5863.0,0.129,8.134,2084.305,914.166,30.238,93.377,24.44,0.485,20.129,2.201,1.094,1.084,2.57,2.85,1.741,2.597,0.676,0.089,-67.584,-41.039,4.311,0.622,0.05,2.239],[5863.5,0.141,8.131,2085.835,931.096,31.509,94.475,24.649,0.478,20.078,2.2,1.074,1.075,2.27,2.301,1.671,2.156,0.79,0.103,-65.712,-35.171,4.08,0.713,0.047,2.205],[5864.0,0.146,8.135,2095.132,950.044,32.17,95.984,24.704,0.462,19.521,2.201,1.053,1.072,2.496,2.088,1.674,1.763,0.96,0.107,-62.591,-29.202,3.95,0.84,0.055,2.194],[5864.5,0.141,8.131,2105.327,961.591,31.989,97.803,24.676,0.453,18.738,2.201,1.04,1.065,3.158,3.161,1.688,1.525,1.113,0.099,-57.968,-32.483,3.904,0.954,0.066,2.208],[5865.0,0.152,8.127,2108.589,954.689,31.265,99.846,25.624,0.444,18.746,2.152,1.047,1.061,3.958,4.154,1.75,1.364,1.294,0.089,-52.442,-40.806,3.964,1.086,0.064,2.23],[5865.5,0.174,8.131,2102.854,927.545,30.454,102.171,29.987,0.434,20.976,2.017,1.078,1.072,3.693,3.869,2.068,1.259,1.652,0.087,-46.895,-42.908,4.579,1.356,0.054,2.241],[5866.0,0.222,8.132,2094.38,889.992,30.373,104.869,40.429,0.435,24.331,2.036,1.124,1.101,2.516,2.564,2.544,1.033,2.698,0.093,-40.14,-40.456,5.641,2.234,0.048,2.236],[5866.5,0.268,8.146,2093.366,858.484,32.561,107.955,57.28,0.505,30.356,2.357,1.165,1.139,1.652,1.213,2.828,0.802,3.656,0.095,-29.291,-38.196,7.326,3.499,0.055,2.239],[5867.0,0.32,8.263,2094.729,841.165,36.312,111.367,74.404,0.671,42.396,2.822,1.189,1.172,1.237,1.022,3.041,0.919,3.401,0.082,-15.429,-30.886,10.532,4.217,0.068,2.269],[5867.5,0.333,8.35,2077.338,831.469,40.516,114.89,84.789,0.804,53.585,3.137,1.203,1.187,1.109,1.059,3.227,1.122,2.889,0.064,-4.085,-17.067,13.249,4.381,0.083,2.311],[5868.0,0.381,8.369,2048.518,822.182,43.541,118.117,88.611,0.829,58.536,3.164,1.216,1.188,1.108,1.012,3.346,1.198,2.793,0.057,1.71,-7.798,14.159,4.386,0.091,2.332],[5868.5,0.333,8.372,2034.601,815.649,45.114,120.62,90.487,0.811,60.118,3.072,1.226,1.179,1.138,0.944,3.544,1.244,2.846,0.061,3.846,-7.574,14.672,4.375,0.087,2.322],[5869.0,0.274,8.336,2041.162,815.624,46.604,122.233,92.201,0.789,60.743,3.059,1.226,1.165,1.137,0.897,3.771,1.288,2.926,0.073,4.664,-10.521,15.178,4.372,0.086,2.293],[5869.5,0.394,8.316,2053.597,820.812,48.549,123.073,94.286,0.771,61.059,3.201,1.218,1.153,1.13,0.867,3.952,1.324,2.984,0.085,5.003,-3.695,15.555,4.359,0.099,2.268],[5870.0,0.323,8.348,2062.22,829.45,51.072,123.379,96.723,0.768,61.05,3.297,1.206,1.145,1.131,0.807,4.011,1.34,2.993,0.087,5.228,19.351,15.734,4.357,0.118,2.262],[5870.5,0.352,8.515,2060.372,841.718,52.633,123.332,98.589,0.777,60.491,3.18,1.188,1.136,1.153,0.828,3.924,1.296,3.032,0.078,5.392,42.544,15.571,4.466,0.122,2.277],[5871.0,0.339,8.747,2052.382,853.255,51.357,122.995,98.842,0.794,59.082,3.03,1.172,1.128,1.203,0.875,3.7,1.179,3.144,0.068,5.432,45.325,14.989,4.731,0.113,2.299],[5871.5,0.243,8.866,2051.308,855.857,50.348,122.39,97.812,0.808,56.653,3.167,1.168,1.125,1.23,0.907,3.454,1.043,3.321,0.062,5.374,28.065,14.244,5.083,0.102,2.313],[5872.0,0.236,8.942,2059.095,846.157,50.084,121.563,97.12,0.808,54.123,3.544,1.182,1.132,1.251,0.941,3.306,0.936,3.539,0.059,5.297,8.526,13.637,5.416,0.093,2.321],[5872.5,0.216,8.973,2063.274,827.98,49.307,120.612,97.891,0.787,52.966,3.677,1.208,1.152,1.266,0.958,3.318,0.866,3.848,0.058,5.248,-4.399,13.324,5.723,0.083,2.328],[5873.0,0.183,8.969,2063.187,805.127,49.149,119.639,100.192,0.763,52.905,3.332,1.242,1.184,1.285,0.977,3.403,0.807,4.235,0.057,5.204,-14.096,13.236,6.104,0.071,2.334],[5873.5,0.172,8.969,2066.968,780.906,49.349,118.744,102.89,0.782,54.632,3.007,1.281,1.226,1.311,1.016,3.378,0.804,4.211,0.057,5.11,-20.928,13.469,6.221,0.065,2.34],[5874.0,0.164,9.012,2072.723,760.708,49.34,118.02,103.903,0.837,57.475,3.135,1.315,1.273,1.355,1.053,3.259,0.859,3.803,0.055,5.008,-24.072,13.915,6.015,0.068,2.349],[5874.5,0.146,9.055,2075.888,745.708,49.123,117.518,101.772,0.896,60.021,3.533,1.341,1.311,1.466,1.093,3.167,0.938,3.392,0.049,4.937,-25.515,14.487,5.744,0.074,2.366],[5875.0,0.143,9.089,2079.178,732.196,48.591,117.229,98.254,0.96,62.277,3.591,1.366,1.333,1.662,1.189,3.103,1.041,2.997,0.042,4.8,-21.317,15.195,5.431,0.067,2.388],[5875.5,0.203,9.104,2078.016,719.459,47.807,117.12,97.622,1.025,64.415,3.241,1.39,1.345,1.757,1.337,3.015,1.137,2.66,0.04,4.686,-8.054,15.762,5.151,0.048,2.396],[5876.0,0.165,9.147,2068.568,709.566,47.369,117.22,102.218,1.057,67.425,2.909,1.409,1.358,1.65,1.281,3.024,1.196,2.53,0.047,4.781,0.256,16.325,5.067,0.039,2.379],[5876.5,0.152,9.233,2060.96,700.953,47.565,117.587,108.83,1.099,73.621,2.762,1.427,1.372,1.487,1.24,3.184,1.322,2.424,0.06,5.001,-8.651,17.897,5.034,0.047,2.346],[5877.0,0.163,9.284,2063.596,692.603,48.206,118.203,111.735,1.163,80.395,2.744,1.444,1.388,1.435,1.201,3.281,1.486,2.217,0.074,5.052,-24.69,19.494,4.877,0.056,2.318],[5877.5,0.161,9.327,2070.278,688.927,49.065,118.899,110.517,1.203,83.244,2.818,1.452,1.405,1.457,1.198,3.251,1.575,2.067,0.083,4.863,-29.298,19.965,4.708,0.059,2.3],[5878.0,0.171,9.403,2075.503,690.537,49.852,119.41,108.351,1.247,83.334,2.913,1.448,1.422,1.513,1.21,3.15,1.654,1.909,0.087,4.586,-20.746,20.042,4.502,0.061,2.292],[5878.5,0.123,9.466,2076.229,690.301,50.305,119.559,106.258,1.312,81.972,2.976,1.449,1.439,1.587,1.225,2.939,1.706,1.725,0.088,4.334,-15.242,19.655,4.279,0.061,2.291],[5879.0,0.146,9.495,2066.407,684.898,50.4,119.379,103.777,1.373,78.984,3.014,1.46,1.452,1.656,1.283,2.65,1.637,1.619,0.086,4.096,-19.949,18.555,4.21,0.061,2.297],[5879.5,0.144,9.538,2050.142,678.623,50.448,119.076,102.37,1.403,75.646,3.044,1.474,1.457,1.694,1.316,2.415,1.491,1.622,0.08,3.931,-21.45,17.294,4.31,0.065,2.31],[5880.0,0.174,9.578,2040.809,675.445,50.741,118.845,102.149,1.393,73.107,3.071,1.481,1.455,1.704,1.318,2.313,1.373,1.687,0.072,3.921,-14.733,16.459,4.451,0.068,2.325],[5880.5,0.152,9.6,2043.892,675.174,51.068,118.726,101.5,1.324,69.841,3.122,1.481,1.444,1.717,1.323,2.344,1.27,1.853,0.066,3.996,-12.555,15.83,4.632,0.068,2.339],[5881.0,0.163,9.574,2054.543,676.625,51.118,118.599,100.88,1.229,66.268,3.22,1.478,1.425,1.72,1.339,2.446,1.184,2.071,0.061,4.068,-18.771,15.337,4.811,0.061,2.35],[5881.5,0.169,9.562,2064.877,679.219,51.1,118.397,100.846,1.171,65.296,3.309,1.472,1.407,1.662,1.338,2.565,1.171,2.19,0.058,4.142,-24.568,15.33,4.859,0.053,2.357],[5882.0,0.174,9.577,2076.535,682.256,51.141,118.277,100.262,1.129,67.034,3.274,1.466,1.403,1.577,1.245,2.745,1.234,2.224,0.055,4.239,-27.377,15.811,4.758,0.052,2.362],[5882.5,0.179,9.525,2095.155,685.973,51.126,118.461,98.808,1.102,69.086,3.185,1.458,1.41,1.545,1.189,2.906,1.32,2.202,0.053,4.401,-32.085,16.342,4.598,0.057,2.366],[5883.0,0.209,9.416,2105.361,691.789,50.741,118.913,98.402,1.1,70.281,3.216,1.446,1.408,1.603,1.24,2.972,1.395,2.132,0.051,4.698,-37.803,16.693,4.443,0.057,2.37],[5883.5,0.213,9.352,2090.452,699.12,50.036,119.24,98.349,1.099,70.839,3.348,1.43,1.389,1.686,1.325,3.013,1.458,2.067,0.05,5.117,-41.702,16.913,4.306,0.05,2.373],[5884.0,0.173,9.327,2067.322,705.333,49.652,119.078,97.657,1.077,70.346,3.373,1.418,1.364,1.694,1.354,3.088,1.484,2.081,0.049,5.5,-41.054,16.972,4.245,0.051,2.373],[5884.5,0.191,9.296,2055.634,710.465,50.028,118.625,97.919,0.997,67.257,3.265,1.408,1.34,1.642,1.268,3.284,1.403,2.362,0.052,5.702,-36.056,16.651,4.419,0.063,2.366],[5885.0,0.198,9.266,2053.756,717.068,50.546,118.537,99.438,0.887,63.002,3.19,1.395,1.318,1.582,1.196,3.563,1.259,2.853,0.055,5.781,-28.26,16.074,4.753,0.073,2.355],[5885.5,0.233,9.205,2054.022,725.349,50.755,119.274,100.311,0.814,60.889,3.106,1.379,1.299,1.526,1.168,3.789,1.157,3.281,0.058,5.896,-20.889,15.74,5.053,0.08,2.347],[5886.0,0.174,8.919,2051.716,734.767,51.191,120.674,100.992,0.8,60.475,3.007,1.361,1.283,1.46,1.137,3.8,1.073,3.546,0.058,6.081,-16.58,15.519,5.378,0.084,2.345],[5886.5,0.185,8.718,2042.835,745.342,51.687,122.188,102.859,0.82,60.254,2.96,1.342,1.265,1.422,1.085,3.622,0.993,3.647,0.057,6.252,-14.472,15.163,5.671,0.084,2.346],[5887.0,0.165,8.656,2034.904,756.19,51.701,123.378,104.711,0.853,59.497,2.928,1.322,1.245,1.447,1.092,3.369,0.912,3.698,0.057,6.358,-11.296,14.642,5.976,0.081,2.345],[5887.5,0.105,8.555,2037.951,763.836,51.28,124.196,106.774,0.888,57.453,2.901,1.309,1.227,1.477,1.125,3.047,0.794,3.852,0.06,6.412,-11.351,13.795,6.485,0.08,2.336],[5888.0,0.078,8.48,2050.278,767.544,50.965,124.779,110.698,0.907,56.092,2.95,1.303,1.215,1.461,1.103,2.854,0.714,3.998,0.066,6.477,-17.727,13.21,6.868,0.073,2.321],[5888.5,0.072,8.457,2066.754,771.64,51.081,125.072,114.614,0.917,56.043,3.062,1.296,1.206,1.427,1.092,2.81,0.702,4.004,0.071,6.629,-19.932,13.158,6.957,0.063,2.307],[5889.0,0.078,8.451,2089.804,780.19,51.444,124.869,115.652,0.958,56.994,3.166,1.282,1.197,1.415,1.114,2.753,0.733,3.77,0.072,6.896,-12.164,13.459,6.825,0.055,2.305],[5889.5,0.11,8.471,2114.858,792.328,51.672,124.094,113.55,1.055,59.938,3.178,1.262,1.183,1.407,1.084,2.613,0.813,3.243,0.064,7.188,-5.624,14.036,6.432,0.057,2.319],[5890.0,0.156,8.727,2124.987,807.145,51.569,122.901,109.835,1.179,64.181,3.061,1.239,1.166,1.388,1.059,2.438,0.921,2.673,0.053,7.425,-6.356,14.63,5.922,0.067,2.344],[5890.5,0.147,9.0,2119.756,823.167,51.402,121.56,106.601,1.277,67.536,2.938,1.215,1.147,1.348,0.993,2.306,1.013,2.286,0.045,7.648,-8.726,15.012,5.511,0.07,2.365],[5891.0,0.129,9.053,2108.68,834.728,51.247,120.342,105.241,1.319,68.037,2.946,1.198,1.131,1.304,0.964,2.222,1.037,2.144,0.042,7.927,-13.768,14.957,5.354,0.06,2.372],[5891.5,0.129,9.054,2099.274,841.177,51.282,119.449,105.161,1.282,65.712,3.017,1.189,1.12,1.266,0.984,2.209,0.981,2.26,0.043,8.135,-24.59,14.456,5.475,0.053,2.367],[5892.0,0.124,9.046,2096.34,847.77,51.495,118.984,104.808,1.206,62.411,3.064,1.18,1.113,1.255,0.982,2.242,0.895,2.515,0.046,8.09,-32.808,13.809,5.734,0.06,2.358],[5892.5,0.109,9.04,2094.958,854.752,51.673,118.901,104.073,1.137,58.366,3.069,1.17,1.111,1.265,0.989,2.222,0.788,2.842,0.048,7.813,-33.8,12.913,6.107,0.068,2.349],[5893.0,0.119,9.044,2087.88,855.177,51.942,118.971,104.039,1.096,54.702,3.022,1.169,1.117,1.263,0.965,2.144,0.691,3.109,0.049,7.528,-33.57,12.005,6.458,0.071,2.347],[5893.5,0.115,9.059,2081.047,846.928,52.364,118.884,104.892,1.078,53.646,2.879,1.181,1.128,1.255,0.975,2.135,0.664,3.216,0.049,7.261,-35.564,11.744,6.566,0.07,2.35],[5894.0,0.09,9.06,2075.939,833.939,52.534,118.418,104.898,1.039,54.59,2.726,1.199,1.143,1.282,0.992,2.284,0.692,3.3,0.048,6.892,-35.452,12.101,6.496,0.069,2.355],[5894.5,0.078,9.105,2068.258,817.951,52.043,117.548,103.74,0.992,56.404,2.73,1.223,1.17,1.334,1.026,2.535,0.754,3.363,0.048,6.468,-33.49,12.818,6.321,0.072,2.357],[5895.0,0.084,9.186,2062.218,797.45,51.157,116.518,103.386,0.965,58.875,2.919,1.254,1.212,1.368,1.061,2.795,0.85,3.297,0.049,6.113,-30.204,13.76,6.027,0.08,2.358],[5895.5,0.101,9.226,2062.691,775.32,50.478,115.773,103.802,0.965,61.435,3.141,1.29,1.261,1.384,1.088,2.964,0.954,3.114,0.049,5.837,-26.402,14.607,5.694,0.085,2.362],[5896.0,0.097,9.255,2075.004,755.189,50.09,115.657,103.479,0.991,62.473,3.237,1.324,1.307,1.455,1.123,2.944,1.0,2.943,0.048,5.649,-25.876,14.886,5.523,0.084,2.368],[5896.5,0.104,9.259,2095.539,735.527,49.665,116.117,102.49,1.059,62.478,3.247,1.36,1.343,1.595,1.21,2.725,0.997,2.733,0.047,5.574,-27.788,14.684,5.468,0.08,2.375],[5897.0,0.129,9.238,2102.832,715.969,49.128,116.763,102.295,1.133,62.237,3.291,1.397,1.371,1.729,1.278,2.467,0.96,2.569,0.046,5.476,-28.527,14.25,5.513,0.074,2.381],[5897.5,0.143,9.235,2092.092,699.658,48.556,117.258,104.028,1.18,61.719,3.461,1.429,1.394,1.79,1.263,2.282,0.9,2.534,0.046,5.209,-30.392,13.74,5.666,0.068,2.381],[5898.0,0.165,9.258,2085.078,686.928,47.883,117.655,105.447,1.197,61.709,3.644,1.456,1.42,1.731,1.219,2.232,0.886,2.52,0.05,4.843,-34.409,13.641,5.718,0.06,2.376],[5898.5,0.206,9.265,2083.66,676.581,48.26,118.223,104.119,1.181,62.86,3.661,1.478,1.447,1.671,1.184,2.356,0.946,2.491,0.054,4.66,-36.315,14.201,5.577,0.054,2.366],[5899.0,0.197,9.25,2073.665,671.087,49.376,118.993,102.453,1.17,64.885,3.486,1.49,1.464,1.675,1.247,2.502,1.041,2.407,0.057,4.729,-32.669,14.954,5.338,0.053,2.36],[5899.5,0.216,9.249,2062.688,671.316,49.06,119.621,103.206,1.177,67.099,3.336,1.49,1.464,1.737,1.335,2.599,1.141,2.281,0.058,4.818,-23.998,15.615,5.086,0.061,2.358],[5900.0,0.274,9.278,2064.024,672.582,47.956,119.699,104.468,1.177,69.053,3.318,1.487,1.458,1.791,1.345,2.695,1.234,2.186,0.057,4.816,-15.536,16.192,4.874,0.071,2.36],[5900.5,0.189,9.305,2072.926,671.92,48.165,119.214,104.433,1.167,69.475,3.337,1.488,1.453,1.776,1.323,2.739,1.255,2.185,0.055,4.783,-16.504,16.323,4.831,0.074,2.364],[5901.0,0.162,9.306,2081.179,671.872,49.283,118.618,104.292,1.149,67.47,3.28,1.488,1.446,1.719,1.327,2.698,1.172,2.31,0.053,4.728,-25.618,15.827,5.023,0.071,2.37],[5901.5,0.146,9.305,2083.267,673.66,48.806,118.398,104.159,1.123,64.39,3.23,1.484,1.442,1.685,1.311,2.627,1.05,2.509,0.052,4.743,-31.136,15.068,5.336,0.071,2.373],[5902.0,0.145,9.299,2076.952,675.436,48.018,118.671,102.84,1.103,62.171,3.245,1.481,1.439,1.668,1.283,2.57,0.962,2.675,0.052,4.935,-30.862,14.475,5.589,0.075,2.372],[5902.5,0.152,9.274,2072.043,678.082,51.845,119.204,101.477,1.1,61.3,3.232,1.475,1.433,1.672,1.302,2.523,0.921,2.741,0.052,5.18,-32.818,14.174,5.713,0.082,2.371],[5903.0,0.208,9.285,2076.485,681.839,55.644,119.702,100.774,1.153,62.623,3.168,1.467,1.422,1.686,1.306,2.427,0.945,2.579,0.053,5.26,-40.43,14.27,5.611,0.089,2.369],[5903.5,0.181,9.292,2083.147,683.365,55.401,120.034,100.325,1.265,66.529,3.152,1.463,1.415,1.708,1.289,2.303,1.056,2.203,0.054,5.158,-50.362,14.847,5.243,0.097,2.366],[5904.0,0.209,9.278,2082.503,683.918,54.414,120.266,101.054,1.37,70.572,3.114,1.462,1.413,1.714,1.29,2.222,1.19,1.876,0.054,5.052,-58.929,15.535,4.855,0.104,2.365],[5904.5,0.137,9.281,2073.75,686.848,53.695,120.525,103.479,1.437,73.337,3.03,1.456,1.409,1.692,1.243,2.197,1.304,1.689,0.053,5.09,-63.219,16.117,4.592,0.104,2.367],[5905.0,0.138,9.296,2057.068,691.334,52.789,120.844,105.484,1.474,75.132,3.006,1.446,1.401,1.663,1.224,2.2,1.381,1.594,0.052,5.168,-60.056,16.556,4.45,0.098,2.369],[5905.5,0.218,9.32,2043.538,695.628,52.217,121.117,105.067,1.433,75.777,3.037,1.438,1.39,1.669,1.244,2.317,1.391,1.668,0.05,5.175,-50.348,16.9,4.51,0.094,2.372],[5906.0,0.225,9.332,2042.794,701.12,51.836,121.2,103.518,1.27,74.149,3.063,1.426,1.374,1.672,1.251,2.664,1.301,2.072,0.048,5.208,-39.509,17.081,4.895,0.092,2.377],[5906.5,0.416,9.331,2050.427,708.013,52.075,121.079,102.748,1.084,71.271,3.117,1.412,1.353,1.665,1.237,3.107,1.186,2.636,0.047,5.373,-26.818,17.076,5.356,0.09,2.378],[5907.0,0.316,9.296,2051.638,716.063,52.39,120.901,104.51,0.948,68.795,3.133,1.397,1.334,1.627,1.192,3.542,1.119,3.176,0.049,5.554,-12.803,17.047,5.657,0.09,2.372],[5907.5,0.395,9.255,2040.555,726.888,52.395,120.843,107.175,0.86,67.134,3.006,1.376,1.314,1.583,1.155,3.897,1.091,3.575,0.052,5.588,-4.616,17.07,5.807,0.092,2.362],[5908.0,0.293,9.247,2028.483,737.2,52.264,120.827,107.997,0.849,66.974,2.875,1.356,1.294,1.558,1.142,3.959,1.109,3.573,0.055,5.456,-1.235,17.16,5.744,0.098,2.353],[5908.5,0.462,9.266,2028.896,740.878,52.238,120.449,107.485,0.897,67.818,2.839,1.35,1.281,1.526,1.155,3.771,1.149,3.287,0.056,5.313,3.733,17.236,5.57,0.109,2.348],[5909.0,0.381,9.258,2040.675,738.231,52.407,119.521,107.416,0.974,69.466,2.808,1.355,1.281,1.464,1.102,3.493,1.2,2.918,0.057,5.366,1.443,17.328,5.363,0.112,2.348],[5909.5,0.381,9.246,2049.316,734.612,52.715,118.507,108.264,1.08,72.335,2.751,1.361,1.297,1.407,1.065,3.196,1.288,2.497,0.057,5.65,-8.894,17.562,5.071,0.104,2.348],[5910.0,0.402,9.136,2046.634,732.456,52.756,118.144,109.358,1.198,75.738,2.799,1.365,1.324,1.38,1.065,2.929,1.397,2.107,0.058,5.985,-15.261,17.87,4.754,0.098,2.345],[5910.5,0.381,9.054,2034.56,732.622,52.089,118.627,109.413,1.291,78.165,2.997,1.365,1.339,1.456,1.107,2.757,1.491,1.853,0.057,6.174,-16.522,18.165,4.526,0.112,2.347],[5911.0,0.169,9.017,2025.604,735.172,51.159,119.39,106.497,1.345,79.166,3.327,1.36,1.326,1.715,1.344,2.692,1.57,1.717,0.05,6.118,-18.22,18.493,4.373,0.131,2.365],[5911.5,0.141,8.884,2034.29,739.007,50.742,119.697,103.262,1.374,79.892,3.529,1.353,1.302,1.956,1.573,2.686,1.635,1.643,0.04,5.897,-20.1,18.851,4.278,0.121,2.392],[5912.0,0.128,8.737,2059.969,741.881,50.739,119.514,104.383,1.37,80.996,3.365,1.348,1.288,1.855,1.283,2.75,1.652,1.666,0.038,5.771,-23.866,19.228,4.322,0.085,2.399],[5912.5,0.1,8.635,2082.015,743.853,50.684,119.509,108.404,1.337,80.738,3.101,1.344,1.28,1.564,1.096,2.806,1.549,1.821,0.045,5.938,-26.479,19.157,4.608,0.062,2.379],[5913.0,0.124,8.579,2082.468,748.999,50.479,119.973,110.253,1.306,78.874,3.014,1.335,1.268,1.412,1.132,2.793,1.389,2.018,0.051,6.325,-20.783,18.622,4.99,0.06,2.36],[5913.5,0.147,8.617,2067.741,755.944,50.431,120.312,110.002,1.285,77.132,2.997,1.323,1.254,1.382,1.131,2.763,1.275,2.169,0.051,6.728,-12.39,18.124,5.279,0.06,2.359],[5914.0,0.137,8.663,2059.473,754.794,50.65,119.694,110.47,1.287,75.791,2.935,1.325,1.254,1.358,1.095,2.677,1.188,2.254,0.049,7.007,-13.951,17.58,5.495,0.061,2.365],[5914.5,0.133,8.675,2059.82,742.646,50.711,117.939,111.279,1.327,74.758,2.902,1.347,1.275,1.333,1.081,2.509,1.127,2.225,0.05,7.098,-25.441,16.98,5.591,0.064,2.366],[5915.0,0.123,8.674,2057.15,726.832,50.525,115.734,111.634,1.397,74.782,3.028,1.376,1.317,1.334,1.088,2.341,1.14,2.057,0.052,7.002,-36.132,16.67,5.432,0.066,2.362],[5915.5,0.127,8.681,2051.127,710.305,50.406,114.119,111.871,1.464,75.414,3.246,1.408,1.377,1.417,1.108,2.223,1.194,1.865,0.054,6.748,-35.533,16.608,5.167,0.069,2.359],[5916.0,0.125,8.678,2046.049,693.617,50.257,113.65,110.881,1.51,75.361,3.326,1.442,1.435,1.635,1.248,2.137,1.22,1.753,0.056,6.327,-26.16,16.474,5.012,0.069,2.357],[5916.5,0.127,8.681,2039.577,681.703,49.73,114.158,108.733,1.518,73.683,3.354,1.467,1.459,1.958,1.41,2.074,1.184,1.754,0.058,5.825,-21.739,16.078,5.045,0.067,2.356],[5917.0,0.121,8.664,2039.575,676.227,49.217,115.271,108.063,1.477,71.622,3.522,1.479,1.441,2.226,1.538,2.091,1.125,1.861,0.059,5.43,-24.169,15.752,5.2,0.062,2.354],[5917.5,0.128,8.624,2049.871,673.197,49.355,116.714,109.499,1.399,71.801,3.7,1.485,1.404,2.188,1.528,2.255,1.12,2.011,0.062,5.312,-22.434,16.054,5.318,0.055,2.35],[5918.0,0.112,8.627,2057.969,671.841,50.053,118.346,111.084,1.288,74.83,3.653,1.488,1.369,1.887,1.268,2.659,1.21,2.191,0.066,5.541,-17.299,17.363,5.329,0.049,2.341],[5918.5,0.101,8.56,2054.971,677.037,50.915,120.13,112.165,1.219,78.56,3.503,1.477,1.339,1.575,1.114,3.005,1.32,2.275,0.072,5.936,-17.341,18.69,5.255,0.045,2.326],[5919.0,0.095,8.379,2047.471,691.576,51.533,121.96,113.687,1.23,80.934,3.464,1.446,1.316,1.424,1.096,3.049,1.369,2.227,0.077,6.176,-18.965,19.148,5.191,0.044,2.312],[5919.5,0.101,8.317,2046.705,713.723,51.751,123.458,115.477,1.278,81.876,3.463,1.401,1.294,1.402,1.104,2.932,1.369,2.141,0.078,6.254,-19.262,19.108,5.178,0.044,2.305],[5920.0,0.088,8.32,2059.341,739.093,52.073,124.155,116.858,1.318,81.129,3.429,1.353,1.271,1.408,1.089,2.78,1.324,2.101,0.075,6.413,-23.852,18.704,5.245,0.047,2.307],[5920.5,0.099,8.315,2073.76,760.58,52.568,123.874,117.682,1.323,78.905,3.429,1.315,1.249,1.412,1.09,2.686,1.254,2.144,0.07,6.671,-28.255,18.145,5.375,0.052,2.313],[5921.0,0.129,8.321,2072.637,769.626,52.81,122.889,118.011,1.233,74.083,3.497,1.299,1.236,1.398,1.084,2.766,1.129,2.486,0.066,6.911,-25.183,17.322,5.723,0.056,2.319],[5921.5,0.156,8.323,2062.78,764.514,52.844,121.609,117.745,1.046,67.331,3.524,1.308,1.238,1.367,1.066,3.093,0.969,3.247,0.066,7.075,-19.237,16.352,6.302,0.058,2.322],[5922.0,0.132,8.346,2061.749,752.477,53.045,120.122,117.248,0.878,62.459,3.428,1.329,1.25,1.348,1.078,3.534,0.858,4.152,0.067,7.074,-16.003,15.715,6.813,0.063,2.322],[5922.5,0.144,8.384,2069.628,739.672,53.269,118.468,116.945,0.747,59.425,3.307,1.352,1.276,1.347,1.081,4.023,0.777,5.221,0.068,6.897,-15.656,15.233,7.292,0.069,2.322],[5923.0,0.15,8.411,2069.523,726.462,53.209,117.165,117.114,0.671,57.691,3.247,1.377,1.315,1.361,1.091,4.316,0.716,6.038,0.068,6.719,-20.709,14.779,7.664,0.072,2.324],[5923.5,0.138,8.418,2061.547,714.011,53.059,116.855,118.142,0.677,57.279,3.282,1.401,1.356,1.39,1.132,4.214,0.693,6.08,0.066,6.617,-27.559,14.527,7.776,0.075,2.33],[5924.0,0.16,8.413,2054.725,700.975,52.925,117.319,118.813,0.8,58.303,3.4,1.427,1.38,1.601,1.167,3.626,0.715,5.112,0.06,6.378,-30.029,14.473,7.523,0.087,2.347],[5924.5,0.149,8.417,2048.829,680.612,52.526,117.447,117.156,0.98,60.661,3.567,1.469,1.391,1.947,1.47,2.951,0.776,3.836,0.049,5.916,-31.431,14.606,7.006,0.099,2.377],[5925.0,0.153,8.426,2042.412,650.161,51.799,116.369,114.639,1.074,63.896,3.588,1.538,1.426,1.944,1.602,2.768,0.867,3.208,0.041,5.525,-37.986,15.176,6.516,0.097,2.405],[5925.5,0.108,8.362,2043.422,618.55,51.172,114.466,114.261,1.103,67.902,3.29,1.617,1.514,1.451,1.264,2.838,0.981,2.904,0.042,5.437,-45.503,15.985,6.065,0.082,2.407],[5926.0,0.084,8.144,2058.183,596.606,50.942,112.893,113.539,1.129,70.756,2.967,1.676,1.637,1.199,1.073,2.876,1.081,2.666,0.051,5.397,-43.687,16.581,5.7,0.069,2.389],[5926.5,0.082,8.063,2080.807,586.793,50.488,112.065,108.648,1.123,71.839,2.976,1.704,1.732,1.552,1.199,2.995,1.191,2.518,0.057,4.999,-24.153,17.149,5.358,0.064,2.378],[5927.0,0.145,8.239,2099.762,582.781,49.296,111.244,102.201,1.082,71.998,3.256,1.716,1.767,2.453,1.851,3.208,1.321,2.431,0.054,4.233,3.162,17.694,4.985,0.063,2.386],[5927.5,0.135,8.486,2109.153,577.88,48.245,109.719,99.299,1.024,71.91,3.502,1.73,1.774,2.681,2.063,3.465,1.408,2.46,0.049,3.455,15.715,18.09,4.768,0.067,2.398],[5928.0,0.209,8.585,2109.253,571.79,48.665,107.951,100.153,0.951,72.503,3.697,1.749,1.772,2.24,1.632,3.833,1.469,2.606,0.048,3.053,7.515,18.546,4.686,0.08,2.402],[5928.5,0.244,8.617,2101.355,570.457,50.094,107.091,102.11,0.914,73.132,3.94,1.753,1.738,2.074,1.512,4.056,1.531,2.649,0.046,3.186,-8.913,18.921,4.587,0.094,2.406],[5929.0,0.278,8.625,2090.746,579.428,51.098,107.712,103.731,0.914,72.475,4.018,1.726,1.665,2.202,1.604,4.082,1.558,2.621,0.04,3.67,-23.124,19.039,4.535,0.092,2.421],[5929.5,0.24,8.623,2086.584,596.739,51.328,109.381,104.969,0.961,72.496,3.848,1.676,1.587,2.232,1.618,3.888,1.56,2.492,0.035,4.197,-32.597,18.952,4.509,0.073,2.431],[5930.0,0.243,8.63,2081.347,617.807,51.843,111.463,106.01,1.176,75.077,3.592,1.619,1.523,1.989,1.479,3.157,1.537,2.049,0.037,4.75,-37.663,18.362,4.433,0.06,2.422],[5930.5,0.277,8.629,2067.344,642.504,52.837,113.91,107.331,1.44,77.994,3.252,1.556,1.464,1.695,1.343,2.422,1.5,1.613,0.043,5.428,-40.218,17.614,4.357,0.064,2.4],[5931.0,0.281,8.62,2055.041,669.84,53.632,116.919,108.252,1.562,78.603,2.965,1.493,1.404,1.534,1.306,2.169,1.474,1.472,0.049,6.082,-39.537,17.284,4.353,0.075,2.379],[5931.5,0.206,8.544,2054.667,694.051,54.41,120.214,107.621,1.608,78.198,2.877,1.441,1.342,1.515,1.261,2.095,1.454,1.441,0.054,6.497,-36.341,17.194,4.39,0.083,2.364],[5932.0,0.247,8.462,2064.929,712.278,54.88,123.023,106.877,1.594,79.853,2.938,1.404,1.283,1.513,1.252,2.188,1.517,1.441,0.058,6.658,-36.579,17.762,4.348,0.091,2.351],[5932.5,0.247,8.444,2079.345,722.749,54.771,124.721,108.539,1.501,83.224,3.034,1.384,1.248,1.458,1.171,2.487,1.669,1.488,0.062,6.626,-43.477,18.993,4.228,0.088,2.339],[5933.0,0.074,8.311,2081.208,719.847,54.64,125.149,111.793,1.42,85.445,3.074,1.389,1.252,1.344,1.067,2.741,1.773,1.545,0.066,6.535,-49.633,19.853,4.156,0.067,2.329],[5933.5,0.072,8.019,2069.318,705.053,54.693,124.495,114.122,1.357,86.073,2.958,1.418,1.296,1.233,0.996,2.922,1.809,1.615,0.07,6.554,-48.95,20.22,4.148,0.044,2.323],[5934.0,0.083,7.852,2064.36,690.889,54.475,123.135,114.538,1.288,85.755,2.843,1.447,1.362,1.189,0.973,3.105,1.811,1.715,0.071,6.63,-44.917,20.397,4.181,0.038,2.325],[5934.5,0.127,7.95,2070.736,683.131,53.832,121.478,112.243,1.24,83.19,3.012,1.464,1.421,1.328,1.026,3.155,1.726,1.832,0.068,6.563,-38.287,19.971,4.301,0.045,2.334],[5935.0,0.185,8.173,2070.535,676.314,53.005,119.834,107.853,1.198,77.436,3.363,1.479,1.455,1.7,1.223,3.033,1.518,2.013,0.062,6.389,-27.527,18.564,4.558,0.056,2.349],[5935.5,0.217,8.323,2067.564,667.001,51.91,118.372,104.006,1.144,71.457,3.417,1.499,1.462,2.061,1.612,2.936,1.31,2.253,0.055,6.353,-18.964,17.161,4.876,0.072,2.367],[5936.0,0.211,8.357,2069.159,656.368,50.873,117.323,102.693,1.105,67.495,3.135,1.524,1.456,2.071,1.603,2.885,1.17,2.474,0.051,6.451,-13.062,16.281,5.175,0.087,2.377],[5936.5,0.224,8.31,2064.61,645.309,50.456,116.996,103.468,1.09,64.508,2.987,1.55,1.452,1.811,1.42,2.758,1.041,2.657,0.053,6.38,-6.852,15.359,5.49,0.093,2.374],[5937.0,0.153,8.178,2051.912,636.172,50.76,117.486,105.175,1.095,62.092,3.013,1.572,1.455,1.634,1.274,2.572,0.921,2.797,0.058,6.042,-5.347,14.381,5.806,0.09,2.365],[5937.5,0.127,8.105,2046.219,631.743,51.11,118.37,105.733,1.098,60.336,2.906,1.583,1.464,1.537,1.23,2.423,0.832,2.917,0.061,5.711,-7.206,13.591,6.07,0.082,2.36],[5938.0,0.108,8.136,2054.411,630.369,51.197,118.816,102.463,1.093,59.272,2.711,1.586,1.493,1.493,1.226,2.354,0.785,3.002,0.063,5.567,-2.878,13.142,6.218,0.07,2.356],[5938.5,0.084,8.145,2061.273,628.131,50.737,118.242,102.736,1.087,58.991,2.757,1.592,1.545,1.488,1.226,2.342,0.774,3.026,0.067,5.511,5.74,13.001,6.233,0.057,2.347],[5939.0,0.092,8.143,2057.673,623.237,50.098,116.777,108.524,1.071,59.174,3.013,1.605,1.602,1.573,1.295,2.401,0.792,3.031,0.071,5.381,16.405,13.122,6.15,0.05,2.341],[5939.5,0.209,8.246,2045.546,616.035,50.03,115.049,110.728,0.997,59.063,3.22,1.623,1.643,1.795,1.534,2.686,0.846,3.166,0.069,5.096,28.204,13.578,5.962,0.051,2.346],[5940.0,0.209,8.436,2031.38,606.865,49.951,113.524,105.191,0.907,58.703,3.311,1.648,1.669,2.017,1.65,3.059,0.92,3.321,0.063,4.773,32.051,14.117,5.701,0.06,2.36],[5940.5,0.21,8.55,2019.641,596.054,49.306,112.245,98.901,0.914,59.137,3.351,1.678,1.684,2.145,1.596,3.075,0.974,3.161,0.059,4.533,22.873,14.32,5.457,0.069,2.372],[5941.0,0.209,8.561,2015.862,585.569,48.939,111.17,95.118,0.976,60.45,3.233,1.708,1.687,2.155,1.83,2.878,1.01,2.852,0.056,4.281,11.609,14.314,5.26,0.069,2.38],[5941.5,0.19,8.579,2016.661,578.666,49.117,110.494,94.875,1.05,61.45,2.992,1.728,1.685,2.092,1.803,2.67,1.044,2.562,0.055,3.942,2.473,14.26,5.072,0.063,2.384],[5942.0,0.171,8.563,2012.333,578.016,49.138,110.538,97.69,1.189,62.251,2.845,1.73,1.685,1.965,1.538,2.344,1.095,2.152,0.058,3.744,-6.867,14.104,4.79,0.057,2.379],[5942.5,0.118,8.518,2005.959,584.746,49.529,111.353,99.424,1.31,63.094,2.887,1.71,1.675,1.858,1.425,2.09,1.127,1.858,0.062,3.783,-10.856,13.946,4.598,0.054,2.368],[5943.0,0.155,8.535,2012.861,597.469,49.656,112.593,99.149,1.333,63.28,2.971,1.674,1.643,1.899,1.457,2.017,1.072,1.887,0.062,3.894,-2.467,13.728,4.765,0.054,2.364],[5943.5,0.195,8.549,2031.03,612.265,49.37,113.789,99.912,1.324,64.009,3.001,1.633,1.596,1.979,1.544,2.023,1.031,1.963,0.059,3.966,6.692,13.673,4.924,0.06,2.369],[5944.0,0.199,8.555,2044.786,626.166,49.628,114.722,102.714,1.291,64.746,3.069,1.597,1.546,1.959,1.581,2.122,1.056,2.008,0.055,4.013,7.399,13.971,4.909,0.066,2.374],[5944.5,0.181,8.558,2051.728,638.281,49.748,115.482,104.055,1.223,65.203,3.196,1.567,1.503,1.838,1.483,2.353,1.131,2.079,0.055,3.97,3.488,14.656,4.816,0.071,2.372],[5945.0,0.188,8.499,2055.165,647.82,49.908,116.255,102.633,1.138,66.41,3.198,1.544,1.471,1.679,1.344,2.706,1.25,2.16,0.057,3.803,0.12,15.646,4.649,0.081,2.366],[5945.5,0.165,8.451,2047.629,654.66,50.196,117.149,101.851,1.031,67.665,3.086,1.528,1.451,1.581,1.229,3.182,1.358,2.336,0.06,3.545,-0.616,16.629,4.544,0.087,2.356],[5946.0,0.128,8.455,2028.621,660.82,50.338,118.143,103.274,0.931,68.894,3.087,1.513,1.442,1.549,1.188,3.682,1.436,2.56,0.066,3.282,-1.022,17.432,4.504,0.084,2.342],[5946.5,0.118,8.447,2015.298,666.245,50.419,119.121,105.048,0.896,70.203,3.192,1.501,1.446,1.579,1.216,3.898,1.459,2.671,0.071,3.091,-4.998,17.809,4.53,0.08,2.331],[5947.0,0.209,8.44,2017.363,667.181,50.571,119.896,105.189,0.943,71.279,3.2,1.499,1.457,1.636,1.244,3.687,1.424,2.587,0.072,3.034,-8.224,17.689,4.61,0.079,2.327],[5947.5,0.16,8.456,2028.283,663.452,50.809,120.273,103.313,1.035,71.092,3.14,1.507,1.47,1.691,1.267,3.263,1.356,2.402,0.07,3.046,-2.418,17.157,4.697,0.074,2.332],[5948.0,0.148,8.456,2039.312,659.906,50.871,120.112,101.225,1.159,69.841,3.178,1.515,1.478,1.735,1.277,2.788,1.28,2.173,0.066,2.953,4.265,16.367,4.745,0.061,2.342],[5948.5,0.141,8.454,2042.441,656.742,50.689,119.441,101.547,1.298,69.484,3.231,1.523,1.484,1.747,1.274,2.375,1.233,1.924,0.061,2.7,1.254,15.656,4.711,0.056,2.353],[5949.0,0.175,8.439,2037.303,651.487,50.881,118.523,103.457,1.426,70.499,3.181,1.535,1.49,1.709,1.266,2.094,1.225,1.709,0.057,2.567,-7.661,15.191,4.601,0.07,2.364],[5949.5,0.138,8.435,2042.658,648.233,51.009,117.669,104.024,1.553,72.514,3.185,1.543,1.489,1.677,1.275,1.915,1.306,1.48,0.056,2.773,-14.096,15.158,4.329,0.088,2.368],[5950.0,0.067,8.458,2060.628,649.9,50.418,116.991,102.666,1.69,75.657,3.278,1.539,1.477,1.782,1.259,1.832,1.56,1.198,0.057,3.151,-11.296,15.796,3.808,0.089,2.365],[5950.5,0.091,8.483,2062.051,653.322,49.776,116.439,100.413,1.798,78.801,3.261,1.531,1.462,1.915,1.289,1.804,1.869,0.974,0.06,3.555,1.749,16.557,3.308,0.072,2.358],[5951.0,0.221,8.506,2043.858,657.584,49.742,115.982,99.77,1.798,81.967,3.141,1.521,1.449,1.845,1.262,1.899,2.132,0.891,0.064,4.017,11.798,17.404,3.036,0.062,2.347],[5951.5,0.162,8.392,2034.031,663.914,50.118,115.683,101.642,1.658,84.616,3.002,1.506,1.439,1.625,1.217,2.194,2.247,0.978,0.068,4.463,8.868,18.443,3.049,0.072,2.338],[5952.0,0.35,8.292,2032.98,668.595,50.4,115.72,104.085,1.501,84.798,2.854,1.496,1.437,1.523,1.245,2.477,2.152,1.157,0.068,4.698,3.598,18.94,3.275,0.082,2.336],[5952.5,0.481,8.305,2026.024,668.467,50.554,116.184,105.037,1.369,80.31,2.823,1.496,1.445,1.545,1.268,2.641,1.852,1.467,0.068,4.665,5.986,18.424,3.755,0.082,2.336],[5953.0,0.78,8.313,2024.1,664.182,50.717,116.733,105.274,1.227,72.701,2.959,1.506,1.458,1.547,1.254,2.756,1.454,1.943,0.069,4.482,12.512,17.238,4.468,0.086,2.335],[5953.5,0.675,8.29,2033.389,657.865,50.809,116.762,106.145,1.169,68.975,3.085,1.52,1.477,1.56,1.27,2.765,1.257,2.201,0.07,4.327,17.477,16.505,4.874,0.087,2.335],[5954.0,0.648,8.285,2037.018,651.077,50.664,115.993,107.389,1.22,69.727,3.062,1.536,1.507,1.621,1.296,2.611,1.221,2.138,0.068,4.3,13.814,16.233,4.934,0.077,2.339],[5954.5,0.85,8.293,2030.775,645.35,50.392,114.781,107.903,1.286,70.852,3.032,1.55,1.535,1.703,1.304,2.431,1.196,2.032,0.065,4.229,1.675,15.941,4.946,0.066,2.349],[5955.0,0.728,8.278,2027.775,643.086,50.197,113.815,106.604,1.327,71.123,3.221,1.555,1.539,1.822,1.357,2.324,1.184,1.962,0.057,3.988,-7.149,15.744,4.933,0.067,2.366],[5955.5,0.661,8.274,2029.464,646.06,49.936,113.58,103.509,1.348,70.049,3.673,1.548,1.514,2.023,1.649,2.249,1.165,1.931,0.044,3.759,-6.426,15.479,4.931,0.077,2.397],[5956.0,0.62,8.285,2030.198,653.901,49.533,114.03,99.85,1.313,67.449,3.911,1.529,1.468,2.13,1.894,2.261,1.129,2.004,0.03,3.816,-1.641,15.148,4.98,0.073,2.439],[5956.5,0.645,8.281,2038.673,664.863,49.275,114.87,97.603,1.224,64.718,3.518,1.504,1.411,1.997,1.489,2.399,1.101,2.18,0.024,4.158,2.598,14.958,5.042,0.061,2.458],[5957.0,0.472,8.247,2055.224,678.6,49.39,116.131,97.523,1.13,63.53,3.086,1.474,1.347,1.728,1.28,2.594,1.089,2.381,0.031,4.605,5.964,14.941,5.089,0.065,2.432],[5957.5,0.263,8.233,2065.45,696.83,49.584,118.051,98.367,1.054,63.551,3.019,1.435,1.29,1.488,1.144,2.805,1.092,2.569,0.045,4.965,6.534,15.076,5.124,0.08,2.386],[5958.0,0.164,8.219,2063.293,719.159,49.752,120.579,99.253,1.008,63.543,3.084,1.391,1.251,1.334,1.083,2.976,1.109,2.684,0.054,5.183,5.597,15.317,5.128,0.08,2.359],[5958.5,0.16,8.191,2061.498,739.095,49.572,123.19,100.939,0.988,64.018,3.099,1.353,1.23,1.255,1.117,3.11,1.144,2.718,0.056,5.298,9.974,15.689,5.089,0.074,2.35],[5959.0,0.084,8.18,2065.622,748.658,49.709,124.927,103.8,1.003,67.303,3.056,1.336,1.227,1.227,1.102,3.225,1.246,2.599,0.058,5.237,18.348,16.525,4.933,0.073,2.344],[5959.5,0.097,8.171,2065.628,745.366,50.575,124.777,105.88,1.045,72.138,3.026,1.342,1.247,1.247,1.09,3.262,1.373,2.382,0.062,4.967,24.29,17.41,4.712,0.067,2.334],[5960.0,0.118,8.154,2058.449,731.295,50.645,122.492,105.353,1.069,74.445,3.055,1.367,1.294,1.288,1.135,3.262,1.437,2.27,0.068,4.664,24.158,17.799,4.596,0.055,2.322],[5960.5,0.105,8.199,2059.992,709.905,50.802,118.973,102.638,1.08,73.553,3.117,1.409,1.368,1.318,1.105,3.222,1.483,2.174,0.078,4.54,19.05,17.76,4.447,0.044,2.306],[5961.0,0.137,8.415,2072.153,686.823,50.936,115.568,98.459,1.084,69.952,3.151,1.456,1.456,1.377,1.161,3.052,1.494,2.042,0.09,4.549,15.014,16.894,4.193,0.042,2.288],[5961.5,0.404,8.716,2076.039,667.746,50.969,113.223,92.999,1.088,64.547,3.152,1.498,1.521,1.567,1.323,2.758,1.419,1.941,0.095,4.495,15.835,15.317,4.001,0.075,2.284],[5962.0,0.778,8.952,2067.808,653.247,49.953,112.094,86.581,1.109,59.862,3.22,1.531,1.537,2.069,1.674,2.483,1.365,1.818,0.089,4.153,8.501,14.045,3.817,0.118,2.298],[5962.5,0.327,8.913,2062.017,640.157,48.25,111.741,81.433,1.147,57.125,3.19,1.562,1.526,2.674,2.255,2.268,1.378,1.647,0.085,3.625,-13.154,13.268,3.574,0.115,2.309],[5963.0,0.508,8.724,2062.749,629.77,47.643,111.938,81.315,1.147,56.308,2.889,1.588,1.519,2.459,1.734,2.242,1.408,1.592,0.106,3.376,-31.552,13.113,3.455,0.071,2.275],[5963.5,0.399,9.01,2062.35,626.646,48.279,112.979,86.998,1.087,57.757,2.759,1.596,1.517,1.843,1.349,2.463,1.454,1.692,0.158,3.438,-32.914,13.634,3.479,0.077,2.199],[5964.0,1.772,9.329,2061.385,630.687,50.037,114.909,93.314,1.004,60.779,2.916,1.586,1.51,1.6,1.384,2.844,1.496,1.899,0.219,3.326,-24.307,14.499,3.595,0.147,2.124],[5964.5,1.65,9.415,2064.284,637.535,50.911,117.099,95.883,0.933,63.642,3.099,1.569,1.499,1.656,1.379,3.273,1.482,2.213,0.255,2.968,-15.33,15.552,3.898,0.209,2.082],[5965.0,1.665,9.586,2068.111,645.27,50.133,118.823,95.691,0.919,66.368,3.229,1.55,1.494,1.652,1.327,3.483,1.47,2.37,0.25,2.689,-13.173,16.349,4.127,0.234,2.085],[5965.5,1.207,9.857,2063.71,655.548,48.981,119.782,95.522,0.936,68.597,3.376,1.525,1.49,1.604,1.314,3.519,1.503,2.342,0.202,2.571,-19.05,16.82,4.154,0.213,2.134],[5966.0,0.366,9.429,2053.642,667.846,48.771,120.129,96.154,0.951,69.573,3.488,1.497,1.47,1.601,1.299,3.543,1.536,2.308,0.135,2.622,-22.793,17.212,4.16,0.155,2.219],[5966.5,0.314,8.889,2052.179,679.395,48.103,120.156,97.194,0.96,70.113,3.477,1.472,1.424,1.696,1.31,3.579,1.555,2.302,0.081,2.994,-17.65,17.555,4.19,0.102,2.308],[5967.0,0.308,8.905,2061.329,687.875,46.902,119.94,98.303,0.904,69.727,3.409,1.454,1.369,1.81,1.501,3.862,1.485,2.634,0.052,3.665,-12.028,17.684,4.442,0.099,2.37],[5967.5,0.218,8.817,2068.95,691.891,49.985,119.466,100.186,0.754,66.813,3.295,1.445,1.334,1.743,1.622,4.589,1.285,3.678,0.039,4.384,-12.723,17.358,5.076,0.113,2.403],[5968.0,0.189,8.549,2072.966,691.449,52.287,118.931,103.243,0.642,63.75,3.144,1.446,1.329,1.547,1.359,5.173,1.12,4.639,0.037,4.978,-11.092,16.916,5.615,0.101,2.411],[5968.5,0.084,8.238,2077.899,689.985,50.867,118.529,105.888,0.636,62.822,3.086,1.449,1.345,1.389,1.143,5.156,1.072,4.806,0.041,5.451,-5.233,16.709,5.782,0.069,2.4],[5969.0,0.072,8.201,2079.981,689.063,50.381,118.075,104.94,0.711,63.186,3.099,1.451,1.37,1.335,1.124,4.569,1.047,4.358,0.046,5.678,-2.602,16.385,5.808,0.046,2.385],[5969.5,0.131,8.362,2077.614,685.531,51.131,117.266,101.234,0.849,64.451,3.086,1.459,1.404,1.364,1.156,3.739,1.041,3.595,0.047,5.539,-1.253,15.98,5.697,0.048,2.382],[5970.0,0.145,8.532,2076.5,677.623,51.524,116.057,99.018,1.027,65.666,3.042,1.476,1.446,1.463,1.183,3.024,1.064,2.85,0.045,5.233,1.243,15.598,5.442,0.067,2.389],[5970.5,0.159,8.709,2073.562,668.132,51.113,114.68,99.05,1.214,66.482,3.059,1.497,1.478,1.626,1.298,2.48,1.105,2.252,0.041,4.921,-3.128,15.236,5.122,0.083,2.402],[5971.0,0.199,8.778,2063.575,657.532,50.359,113.595,99.057,1.318,66.689,3.116,1.521,1.496,1.835,1.531,2.24,1.138,1.969,0.037,4.444,-12.518,15.07,4.914,0.082,2.415],[5971.5,0.136,8.708,2048.822,643.099,49.921,113.283,99.253,1.318,66.657,3.165,1.555,1.511,1.908,1.589,2.246,1.149,1.954,0.038,3.871,-14.961,15.11,4.878,0.071,2.414],[5972.0,0.146,8.766,2036.976,627.619,49.812,113.9,100.573,1.243,67.667,3.208,1.593,1.53,1.778,1.464,2.459,1.181,2.079,0.05,3.571,-10.663,15.541,4.884,0.066,2.387],[5972.5,0.394,8.948,2034.966,615.079,50.174,115.029,101.112,1.153,69.547,3.227,1.626,1.55,1.685,1.587,2.766,1.244,2.221,0.072,3.579,-5.977,16.242,4.847,0.077,2.34],[5973.0,0.22,9.13,2047.159,605.059,50.834,115.861,100.697,1.086,70.539,3.156,1.653,1.574,1.719,1.614,3.034,1.296,2.34,0.098,3.53,-1.845,16.798,4.811,0.106,2.296],[5973.5,0.466,9.31,2065.859,595.883,51.136,115.715,101.134,1.029,70.812,3.094,1.678,1.606,1.814,1.573,3.306,1.355,2.439,0.116,3.222,-5.062,17.339,4.752,0.14,2.269],[5974.0,0.768,9.111,2075.374,588.308,50.915,114.578,102.197,0.988,71.859,3.131,1.7,1.646,1.87,1.615,3.546,1.436,2.469,0.12,2.764,-17.649,17.886,4.625,0.166,2.264],[5974.5,0.306,8.946,2072.48,582.153,50.264,113.255,102.489,0.966,73.536,3.136,1.718,1.688,1.815,1.519,3.731,1.536,2.431,0.111,2.346,-28.391,18.404,4.452,0.167,2.281],[5975.0,0.16,9.096,2059.516,577.589,49.588,112.507,102.789,0.977,75.327,3.091,1.731,1.712,1.781,1.496,3.819,1.703,2.255,0.09,2.041,-30.145,19.055,4.17,0.144,2.317],[5975.5,0.257,9.019,2038.1,573.619,49.284,112.144,103.892,1.022,76.606,3.165,1.743,1.698,1.949,1.626,3.742,1.859,2.018,0.064,1.761,-24.109,19.511,3.902,0.117,2.367],[5976.0,0.224,8.906,2017.946,566.772,48.453,111.354,104.973,1.091,76.478,3.348,1.764,1.673,2.218,2.019,3.424,1.787,1.916,0.043,1.388,-14.892,19.011,3.962,0.097,2.417],[5976.5,0.179,8.822,2009.814,554.396,47.214,109.904,105.265,1.168,75.361,3.32,1.804,1.683,2.129,1.86,2.992,1.555,1.926,0.035,0.874,-6.116,17.801,4.265,0.07,2.44],[5977.0,0.254,8.879,2007.615,538.273,46.897,108.631,104.893,1.222,73.633,3.039,1.858,1.745,1.751,1.367,2.688,1.38,1.949,0.043,0.023,6.412,16.755,4.516,0.049,2.421],[5977.5,0.373,9.171,2001.611,522.461,47.865,108.4,103.314,1.265,68.946,2.88,1.914,1.832,1.567,1.346,2.366,1.177,2.02,0.064,-1.624,19.972,15.25,4.847,0.065,2.378],[5978.0,0.38,9.322,1993.39,505.724,48.989,108.969,99.644,1.285,61.876,2.989,1.977,1.916,1.782,1.493,2.016,0.943,2.15,0.083,-4.289,16.912,13.229,5.233,0.097,2.347],[5978.5,0.329,9.104,1988.344,479.833,49.14,109.271,95.772,1.248,56.265,3.171,2.084,2.02,2.13,1.865,1.854,0.811,2.291,0.089,-7.432,-0.76,11.823,5.414,0.105,2.346],[5979.0,0.366,8.771,1987.182,440.115,48.583,108.754,94.667,1.191,54.545,3.223,2.272,2.221,2.04,1.921,1.886,0.794,2.374,0.088,-10.495,-14.383,11.452,5.356,0.089,2.358],[5979.5,0.336,8.673,1988.737,392.273,48.032,107.975,94.645,1.107,56.429,3.195,2.549,2.589,1.877,1.605,2.118,0.867,2.438,0.098,-14.326,-18.848,11.908,5.108,0.09,2.359],[5980.0,0.359,8.694,1994.547,344.471,47.509,107.651,89.599,1.025,57.747,3.125,2.903,3.151,2.153,4.401,2.386,0.952,2.504,0.124,-20.527,-18.848,12.465,4.861,0.114,2.345],[5980.5,0.144,8.753,2002.99,298.728,45.896,107.57,77.023,0.939,56.119,2.882,3.348,3.915,3.419,9.154,2.736,1.144,2.423,0.166,-28.822,-18.841,12.943,4.338,0.12,2.323],[5981.0,0.156,8.639,2010.413,253.307,43.656,106.897,62.448,0.793,51.984,2.492,3.948,4.916,6.391,9.597,3.058,1.531,2.053,0.265,-37.097,-24.977,12.259,3.144,0.076,2.274],[5981.5,0.239,8.296,2019.394,208.305,42.855,105.376,52.568,0.708,43.703,2.105,4.801,6.261,10.552,12.174,2.726,1.564,1.739,0.518,-43.948,-34.542,9.862,2.336,0.012,2.181],[5982.0,0.146,8.143,2029.422,166.799,43.033,103.574,47.385,0.717,34.558,1.934,5.995,8.153,15.091,17.396,2.103,1.237,1.7,0.968,-49.474,-37.012,7.685,2.31,0.0,2.084],[5982.5,0.152,8.072,2033.206,132.835,41.999,102.02,43.711,0.691,28.105,1.945,7.528,10.695,18.164,19.478,1.653,0.977,1.69,1.415,-53.964,-31.02,5.879,2.214,0.026,2.043],[5983.0,0.159,7.974,2023.723,108.256,39.107,100.574,41.173,0.632,23.981,2.001,9.237,13.613,19.204,21.076,1.431,0.855,1.675,1.573,-57.46,-19.825,4.621,2.004,0.042,2.07],[5983.5,0.146,7.933,2006.6,92.448,35.962,98.885,40.257,0.574,22.034,1.953,10.817,16.502,19.452,19.264,1.503,0.828,1.823,1.485,-59.93,-2.037,4.389,1.97,0.049,2.125],[5984.0,0.117,7.927,1994.992,84.06,34.218,97.228,40.082,0.501,20.863,1.867,11.896,18.825,19.914,18.798,1.679,0.754,2.251,1.417,-61.573,21.776,4.256,2.101,0.06,2.159],[5984.5,0.129,7.926,1997.872,81.054,33.677,96.412,39.119,0.427,19.782,1.88,12.337,19.537,21.595,19.038,1.879,0.676,2.803,1.438,-62.654,37.206,4.07,2.239,0.06,2.164],[5985.0,0.122,7.916,2006.875,80.518,33.434,96.727,36.835,0.364,17.534,1.954,12.42,18.528,25.322,23.606,1.966,0.563,3.591,1.459,-63.31,37.985,3.659,2.437,0.048,2.162],[5985.5,0.154,7.927,2010.094,80.47,33.082,97.573,34.51,0.316,15.256,2.081,12.427,17.033,28.902,26.713,2.045,0.472,4.368,1.45,-63.663,31.428,3.29,2.594,0.042,2.164],[5986.0,0.158,8.004,2008.954,80.778,32.869,98.246,34.4,0.303,14.913,2.234,12.38,15.904,27.107,24.225,2.142,0.482,4.464,1.441,-63.793,24.038,3.317,2.561,0.042,2.164],[5986.5,0.158,8.057,2010.227,82.589,33.186,98.514,35.832,0.326,17.119,2.323,12.108,15.027,22.104,21.243,2.284,0.636,3.756,1.431,-63.567,21.287,3.816,2.291,0.041,2.161],[5987.0,0.164,8.151,2013.475,87.095,33.412,98.545,35.729,0.371,21.159,2.287,11.482,13.852,19.075,16.345,2.458,0.975,2.687,1.38,-62.979,22.416,4.669,1.849,0.04,2.157],[5987.5,0.169,8.219,2017.127,93.92,33.162,98.485,33.017,0.429,24.595,2.162,10.647,12.23,18.953,18.398,2.481,1.487,1.805,1.267,-62.23,18.982,5.424,1.429,0.042,2.159],[5988.0,0.175,8.225,2024.827,100.193,32.545,98.137,30.749,0.529,27.219,2.018,9.981,10.708,20.36,21.323,2.283,2.54,1.057,1.138,-61.452,12.007,6.085,0.988,0.043,2.169],[5988.5,0.169,8.228,2040.442,102.573,31.343,97.263,31.548,0.621,28.726,1.938,9.749,9.832,18.562,19.268,2.052,3.556,0.591,1.054,-60.516,8.443,6.489,0.688,0.037,2.181],[5989.0,0.208,8.247,2054.048,100.43,30.413,96.054,34.773,0.641,29.513,1.988,9.957,9.771,13.349,14.099,1.99,3.623,0.555,1.003,-59.24,4.573,6.513,0.673,0.037,2.196],[5989.5,0.271,8.272,2047.198,96.314,30.529,95.028,38.367,0.604,30.292,2.136,10.383,10.479,10.256,7.622,2.056,2.9,0.749,0.915,-57.954,-2.007,6.327,0.843,0.048,2.223],[5990.0,0.226,8.263,2027.323,93.104,30.606,94.668,39.562,0.551,30.249,2.219,10.741,11.758,10.305,8.269,2.176,2.173,1.021,0.813,-57.399,-6.568,6.115,1.059,0.057,2.252],[5990.5,0.187,8.25,2018.677,91.513,30.039,95.188,36.362,0.516,28.724,2.095,10.927,13.202,13.925,16.975,2.278,1.924,1.19,0.817,-57.635,-2.095,5.981,1.155,0.058,2.255],[5991.0,0.181,8.232,2020.651,90.418,28.981,96.089,31.297,0.426,24.716,1.89,11.06,14.334,21.762,21.174,2.603,1.756,1.507,0.949,-57.852,14.447,5.533,1.17,0.054,2.228],[5991.5,0.209,8.241,2016.387,89.137,28.091,96.319,27.85,0.345,20.578,1.789,11.219,14.7,27.01,18.81,2.846,1.625,1.754,1.094,-57.327,27.833,5.004,1.143,0.055,2.203],[5992.0,0.168,8.258,2008.544,88.92,28.26,95.247,26.895,0.344,19.243,1.776,11.246,13.828,25.384,12.919,2.804,1.665,1.686,1.142,-55.969,22.827,4.926,1.098,0.061,2.194],[5992.5,0.163,8.255,2008.69,92.867,28.724,93.032,28.511,0.379,21.076,1.815,10.768,11.997,21.737,11.611,2.681,1.831,1.487,1.098,-54.136,13.539,5.15,1.048,0.063,2.193],[5993.0,0.152,8.246,2016.496,103.387,28.758,90.327,33.206,0.509,26.345,2.003,9.672,10.045,17.71,13.12,2.324,2.201,1.097,1.0,-51.888,13.57,5.86,0.992,0.061,2.19],[5993.5,0.103,8.248,2020.718,119.65,29.819,87.827,39.599,0.675,33.348,2.392,8.358,8.357,14.185,13.044,2.152,2.538,0.849,0.8,-48.545,11.396,7.433,1.082,0.063,2.206],[5994.0,0.101,8.238,2019.637,138.535,30.855,85.804,46.092,0.787,39.451,2.809,7.218,6.961,11.357,24.255,2.263,2.582,0.877,0.488,-43.943,1.563,9.106,1.31,0.065,2.273],[5994.5,0.095,8.235,2023.204,157.337,29.951,84.357,52.592,0.857,43.501,3.039,6.356,5.886,8.975,26.37,2.27,2.446,0.93,0.215,-39.571,-3.031,9.93,1.512,0.067,2.377],[5995.0,0.133,8.264,2031.575,173.636,27.316,84.103,58.637,0.901,45.91,3.113,5.759,5.157,7.365,16.557,2.223,2.299,0.967,0.089,-37.368,1.473,10.223,1.652,0.069,2.46],[5995.5,0.121,8.29,2042.452,185.493,24.482,85.734,63.702,0.913,48.081,3.107,5.391,4.754,6.381,10.187,2.243,2.176,1.037,0.059,-37.404,8.305,10.444,1.788,0.07,2.488],[5996.0,0.102,8.301,2057.807,193.437,24.301,88.979,67.344,0.853,49.773,2.927,5.17,4.6,5.668,5.91,2.512,1.945,1.317,0.065,-38.26,13.437,10.85,2.096,0.073,2.478],[5996.5,0.091,8.314,2070.333,198.628,26.446,92.795,68.614,0.784,50.23,2.587,5.035,4.577,5.609,5.207,2.802,1.73,1.626,0.094,-39.382,11.897,11.208,2.411,0.067,2.443],[5997.0,0.082,8.318,2068.898,199.479,30.075,96.23,66.266,0.727,48.489,2.1,5.013,4.604,6.702,6.152,3.015,1.718,1.753,0.195,-41.372,5.424,11.1,2.401,0.024,2.358],[5997.5,0.103,8.312,2059.193,192.51,33.971,98.885,60.639,0.63,44.437,1.921,5.195,4.746,8.27,9.41,3.277,1.87,1.759,0.455,-43.97,4.96,10.43,2.087,-0.017,2.224],[5998.0,0.102,8.319,2061.279,176.467,37.116,100.827,53.217,0.601,40.282,2.047,5.667,5.184,8.554,8.194,3.128,2.067,1.53,0.847,-46.458,12.739,9.527,1.723,0.005,2.104],[5998.5,0.095,8.328,2076.884,155.019,39.904,101.926,45.637,0.675,37.406,2.148,6.451,6.066,7.958,6.423,2.472,2.237,1.117,1.131,-49.098,17.507,8.388,1.401,0.047,2.062],[5999.0,0.09,8.32,2086.355,134.092,39.837,101.66,39.144,0.729,34.406,2.037,7.458,7.389,8.29,7.684,1.99,2.285,0.871,1.126,-52.215,14.123,7.398,1.202,0.055,2.102],[5999.5,0.084,8.321,2084.119,116.481,35.676,100.0,33.896,0.677,28.744,1.789,8.585,9.087,10.624,8.971,1.751,1.842,1.005,0.998,-55.226,2.134,6.098,1.25,0.039,2.165],[6000.0,0.089,8.328,2084.591,100.347,31.148,97.9,29.97,0.524,22.182,1.65,9.965,11.303,15.803,11.04,1.712,1.28,1.391,1.038,-57.474,-12.933,4.516,1.317,0.036,2.189],[6000.5,0.146,8.323,2087.538,83.854,29.42,96.472,27.326,0.409,18.497,1.66,11.926,14.348,20.967,13.828,1.906,1.123,1.699,1.316,-58.912,-18.111,3.965,1.311,0.05,2.176],[6001.0,0.137,8.311,2084.045,68.112,29.11,96.049,25.77,0.372,17.268,1.657,14.682,18.339,23.368,18.286,2.068,1.085,1.911,1.689,-59.859,-12.198,3.914,1.34,0.062,2.167],[6001.5,0.101,8.313,2077.842,55.945,28.208,96.143,25.39,0.334,16.632,1.586,17.875,23.085,25.053,21.324,2.266,1.027,2.213,1.98,-60.621,-7.707,3.843,1.389,0.059,2.175],[6002.0,0.09,8.314,2071.597,48.65,27.394,96.107,26.095,0.291,15.942,1.557,20.555,28.104,27.69,26.6,2.516,0.944,2.699,2.136,-61.243,-14.192,3.717,1.468,0.048,2.189],[6002.5,0.096,8.323,2065.61,45.43,27.484,95.83,26.43,0.252,14.981,1.626,22.012,33.005,30.147,27.38,2.717,0.805,3.436,2.216,-61.513,-25.186,3.492,1.623,0.047,2.196],[6003.0,0.113,8.323,2064.094,44.615,27.701,95.661,25.56,0.23,13.918,1.696,22.414,37.378,32.293,23.535,2.738,0.672,4.104,2.27,-61.416,-29.625,3.215,1.783,0.054,2.195],[6003.5,0.122,8.315,2070.546,44.814,27.692,95.798,24.372,0.229,12.987,1.682,22.315,40.368,35.34,25.967,2.549,0.59,4.319,2.28,-61.201,-22.005,2.977,1.874,0.06,2.193],[6004.0,0.119,8.315,2079.534,45.366,27.58,96.058,23.879,0.245,12.5,1.664,22.043,41.615,39.194,26.747,2.279,0.556,4.097,2.226,-60.934,-11.705,2.839,1.895,0.066,2.195],[6004.5,0.107,8.328,2084.983,46.275,27.354,96.199,24.295,0.271,12.674,1.738,21.61,41.444,43.291,25.867,2.036,0.554,3.681,2.155,-60.448,-10.225,2.799,1.877,0.066,2.198],[6005.0,0.116,8.327,2087.612,47.907,27.353,96.233,25.643,0.325,13.739,1.843,20.874,39.933,45.348,28.194,1.734,0.571,3.053,2.106,-59.755,-20.499,2.826,1.838,0.054,2.195],[6005.5,0.146,8.318,2092.484,50.704,27.84,96.41,27.51,0.401,15.027,1.914,19.722,36.833,43.913,32.784,1.389,0.563,2.464,2.063,-58.918,-34.188,2.796,1.848,0.039,2.188],[6006.0,0.18,8.31,2093.437,55.236,28.585,96.956,29.882,0.454,15.447,1.991,18.104,32.103,40.14,31.532,1.135,0.495,2.293,2.001,-57.902,-36.747,2.619,1.974,0.031,2.176],[6006.5,0.217,8.312,2077.681,61.633,29.365,97.8,32.2,0.476,15.025,2.052,16.225,26.543,35.891,24.485,0.983,0.411,2.41,1.897,-56.715,-19.372,2.389,2.177,0.035,2.163],[6007.0,0.209,8.321,2049.311,69.218,30.041,98.594,33.331,0.484,14.422,2.084,14.447,21.594,33.17,24.08,0.892,0.348,2.574,1.73,-55.252,5.827,2.206,2.363,0.051,2.158],[6007.5,0.193,8.383,2026.987,77.464,30.398,98.994,33.884,0.492,14.146,2.1,12.909,17.986,28.605,25.216,0.829,0.309,2.689,1.511,-53.22,17.125,2.084,2.509,0.062,2.163],[6008.0,0.169,8.429,2018.871,87.74,30.674,98.972,36.097,0.525,15.288,2.117,11.397,15.433,20.539,22.389,0.808,0.312,2.618,1.271,-50.389,9.272,2.173,2.585,0.059,2.174],[6008.5,0.169,8.425,2017.207,102.757,31.172,98.862,39.433,0.649,19.402,2.163,9.732,13.3,14.926,16.338,0.846,0.419,2.121,1.044,-47.055,-0.637,2.819,2.516,0.05,2.182],[6009.0,0.158,8.423,2013.145,124.586,31.664,99.029,41.107,0.775,23.686,2.205,8.027,10.912,13.027,10.811,0.88,0.542,1.64,0.846,-43.715,-2.154,3.486,2.392,0.047,2.186],[6009.5,0.152,8.423,2009.398,153.019,31.73,99.528,40.484,0.827,25.836,2.175,6.535,8.229,13.475,11.862,0.928,0.643,1.453,0.679,-40.438,-3.223,3.925,2.273,0.05,2.189],[6010.0,0.158,8.423,2009.513,186.795,31.828,100.064,40.114,0.871,27.121,2.113,5.353,5.938,12.893,11.745,0.995,0.77,1.303,0.538,-36.926,-8.373,4.432,2.144,0.05,2.196],[6010.5,0.164,8.42,2010.491,223.834,32.626,100.299,43.142,0.916,29.292,2.156,4.468,4.422,9.129,12.026,1.051,0.917,1.156,0.419,-32.689,-12.93,4.919,2.001,0.05,2.211],[6011.0,0.159,8.415,2011.735,261.24,33.799,100.259,50.884,0.926,34.415,2.388,3.828,3.524,5.212,9.272,1.336,1.155,1.142,0.316,-27.016,-18.346,6.262,1.994,0.054,2.236],[6011.5,0.152,8.417,2016.731,299.504,34.637,100.445,60.965,0.883,40.449,2.798,3.339,2.98,3.226,3.505,1.955,1.306,1.495,0.228,-20.441,-22.329,8.726,2.479,0.064,2.271],[6012.0,0.209,8.416,2024.15,342.587,36.36,101.319,69.627,0.856,44.89,3.153,2.919,2.612,2.53,2.629,2.369,1.253,1.9,0.16,-15.145,-16.095,10.362,3.085,0.076,2.307],[6012.5,0.212,8.428,2032.446,389.888,38.367,102.666,75.287,0.875,47.809,3.212,2.565,2.335,2.518,2.525,2.416,1.171,2.066,0.115,-13.255,-5.111,10.802,3.429,0.084,2.338],[6013.0,0.148,8.459,2042.299,435.916,38.861,103.844,78.862,0.906,49.962,3.018,2.294,2.126,2.709,2.627,2.397,1.119,2.142,0.089,-13.986,-2.544,11.074,3.677,0.08,2.358],[6013.5,0.122,8.461,2050.279,475.579,38.668,104.482,82.703,0.985,52.327,2.87,2.103,1.981,2.664,2.954,2.272,1.074,2.114,0.078,-13.915,-11.031,11.383,3.939,0.067,2.365],[6014.0,0.165,8.458,2046.73,505.227,39.54,104.729,86.989,1.074,55.133,2.962,1.979,1.882,2.311,2.007,2.156,1.08,1.998,0.077,-11.633,-18.885,11.813,4.061,0.057,2.359],[6014.5,0.119,8.469,2026.328,523.141,41.269,104.976,88.719,1.088,56.88,3.162,1.912,1.813,2.185,1.908,2.199,1.138,1.933,0.081,-9.646,-13.985,12.215,3.986,0.06,2.346],[6015.0,0.146,8.494,2001.923,531.664,42.198,105.318,86.84,1.053,57.402,3.239,1.881,1.772,2.429,2.341,2.401,1.255,1.915,0.088,-9.296,1.856,12.881,3.822,0.067,2.333],[6015.5,0.135,8.482,1990.902,534.699,42.074,105.309,85.408,1.021,57.684,3.181,1.87,1.766,2.393,2.581,2.594,1.377,1.885,0.091,-8.379,8.898,13.515,3.646,0.063,2.327],[6016.0,0.126,8.459,1993.832,535.649,41.949,104.612,86.949,0.989,57.172,3.107,1.867,1.789,2.059,2.242,2.68,1.347,2.002,0.084,-5.854,1.875,13.523,3.738,0.054,2.338],[6016.5,0.105,8.461,2002.708,538.96,42.208,103.634,88.949,0.926,55.626,3.087,1.855,1.812,1.97,1.727,2.765,1.18,2.375,0.068,-3.024,-3.041,13.061,4.141,0.056,2.366],[6017.0,0.154,8.467,2010.934,548.346,42.474,103.179,89.324,0.903,55.131,3.139,1.824,1.807,2.162,2.534,2.785,1.077,2.589,0.053,-0.585,3.356,12.832,4.424,0.059,2.396],[6017.5,0.207,8.534,2010.513,563.992,42.373,103.677,88.858,0.944,56.778,3.34,1.773,1.757,2.345,2.61,2.741,1.133,2.43,0.042,1.628,8.108,13.207,4.335,0.06,2.418],[6018.0,0.222,8.625,2002.271,583.179,41.909,104.875,89.538,1.015,59.45,3.648,1.715,1.664,2.4,1.924,2.691,1.279,2.124,0.035,3.306,3.763,13.935,4.063,0.067,2.433],[6018.5,0.189,8.663,1995.812,603.861,41.639,106.188,92.077,1.093,62.959,3.697,1.656,1.558,2.223,2.838,2.676,1.514,1.791,0.033,4.36,-1.415,14.935,3.69,0.069,2.437],[6019.0,0.166,8.671,1997.241,625.396,42.622,107.388,95.386,1.159,66.837,3.417,1.599,1.472,1.871,2.41,2.71,1.831,1.498,0.036,5.142,-4.195,16.049,3.281,0.064,2.423],[6019.5,0.191,8.686,2006.768,647.464,44.553,108.854,97.692,1.206,71.07,3.192,1.544,1.416,1.529,1.438,2.783,2.191,1.281,0.046,5.74,-7.49,17.146,2.924,0.069,2.393],[6020.0,0.13,8.709,2021.01,667.182,46.756,110.682,98.247,1.199,76.367,3.129,1.499,1.379,1.363,1.342,3.051,2.641,1.16,0.053,6.121,-10.857,18.636,2.638,0.088,2.371],[6020.5,0.18,8.724,2031.525,679.32,48.316,112.047,97.611,1.149,81.375,3.114,1.472,1.358,1.413,1.47,3.446,3.047,1.131,0.051,6.304,-7.476,20.183,2.464,0.102,2.374],[6021.0,0.143,8.705,2032.717,681.668,48.992,111.91,97.071,1.114,82.393,3.076,1.467,1.367,1.487,1.707,3.595,2.982,1.214,0.044,6.358,6.212,20.45,2.559,0.092,2.392],[6021.5,0.116,8.775,2027.8,676.058,48.968,110.166,96.959,1.083,78.593,2.96,1.479,1.42,1.449,1.476,3.512,2.473,1.449,0.041,6.517,23.978,19.43,2.964,0.078,2.4],[6022.0,0.174,9.101,2020.096,667.415,48.549,107.943,97.364,1.054,73.641,2.856,1.498,1.495,1.428,1.392,3.358,1.973,1.721,0.043,6.79,36.787,18.082,3.436,0.089,2.397],[6022.5,0.214,9.25,2011.838,661.014,47.804,106.576,97.247,1.041,69.76,2.886,1.513,1.537,1.639,1.592,3.182,1.664,1.923,0.041,6.874,41.396,16.92,3.792,0.106,2.404],[6023.0,0.217,9.006,2005.41,659.648,46.659,106.421,95.048,1.039,63.744,2.999,1.516,1.515,2.217,2.193,2.861,1.339,2.175,0.031,6.808,38.615,15.178,4.284,0.099,2.434],[6023.5,0.28,8.643,2000.495,660.885,45.668,106.775,91.939,1.04,56.031,3.041,1.513,1.464,2.647,3.316,2.438,0.997,2.475,0.02,6.968,30.493,12.956,4.879,0.078,2.471],[6024.0,0.322,8.443,1995.428,660.374,45.654,106.996,91.768,1.049,53.092,2.859,1.514,1.425,2.162,2.711,2.227,0.86,2.592,0.017,7.4,25.682,11.931,5.149,0.075,2.485],[6024.5,0.309,8.4,1993.05,659.477,46.564,107.348,95.166,1.154,59.001,2.633,1.516,1.402,1.577,1.502,2.209,0.997,2.277,0.023,7.839,24.248,13.028,4.906,0.097,2.462],[6025.0,0.311,8.371,1995.276,665.157,47.878,108.387,98.823,1.293,68.189,2.648,1.503,1.385,1.302,1.358,2.256,1.24,1.846,0.033,8.097,15.761,14.9,4.489,0.115,2.426],[6025.5,0.295,8.376,2005.986,679.163,49.06,110.008,100.354,1.331,71.861,2.856,1.472,1.375,1.27,1.366,2.294,1.32,1.739,0.039,8.121,2.144,15.598,4.385,0.112,2.406],[6026.0,0.325,8.387,2021.713,693.722,49.608,111.42,98.643,1.316,70.691,3.066,1.442,1.375,1.342,1.372,2.322,1.284,1.809,0.039,7.868,-3.829,15.605,4.513,0.103,2.405],[6026.5,0.355,8.378,2029.628,702.696,49.365,112.031,95.622,1.299,69.109,3.139,1.423,1.378,1.463,1.431,2.372,1.26,1.882,0.035,7.314,-0.43,15.734,4.633,0.102,2.413],[6027.0,0.277,8.376,2029.914,706.181,48.77,112.034,95.531,1.274,68.629,3.081,1.416,1.375,1.623,1.542,2.437,1.238,1.97,0.031,6.636,10.191,15.846,4.753,0.097,2.425],[6027.5,0.27,8.401,2034.865,705.529,48.696,112.063,99.557,1.214,69.59,2.922,1.417,1.367,1.745,1.625,2.629,1.226,2.144,0.028,6.114,18.458,16.25,4.919,0.085,2.438],[6028.0,0.242,8.424,2044.465,699.195,49.015,112.476,104.133,1.156,72.336,2.775,1.43,1.363,1.727,1.544,2.916,1.289,2.263,0.026,5.685,13.843,17.197,4.955,0.077,2.445],[6028.5,0.23,8.392,2047.432,686.709,49.376,112.962,107.069,1.136,75.329,2.78,1.456,1.369,1.596,1.443,3.149,1.427,2.211,0.029,4.994,6.58,18.269,4.762,0.08,2.438],[6029.0,0.209,8.371,2045.003,668.501,49.995,112.995,108.547,1.126,77.384,2.977,1.496,1.392,1.48,1.417,3.36,1.614,2.089,0.036,3.635,10.554,19.309,4.455,0.093,2.416],[6029.5,0.214,8.331,2052.99,640.692,50.603,112.55,109.183,1.115,79.854,3.193,1.561,1.449,1.418,1.408,3.566,1.826,1.959,0.047,1.384,14.081,20.298,4.139,0.106,2.39],[6030.0,0.115,8.282,2071.086,595.87,50.802,112.1,109.868,1.082,82.992,3.278,1.678,1.566,1.348,1.374,3.807,2.0,1.903,0.058,-1.416,10.694,21.019,3.904,0.104,2.375],[6030.5,0.082,8.281,2076.129,532.544,50.357,111.949,109.556,1.04,85.012,3.153,1.878,1.788,1.324,1.349,4.011,2.087,1.921,0.062,-4.443,11.676,21.296,3.786,0.094,2.38],[6031.0,0.114,8.331,2062.237,461.9,49.32,111.753,105.369,1.018,84.331,2.881,2.165,2.17,1.509,1.704,4.104,2.109,1.946,0.063,-8.779,13.119,21.322,3.752,0.09,2.396],[6031.5,0.106,8.416,2049.243,395.571,47.937,110.804,96.592,0.95,77.83,2.519,2.528,2.765,2.011,2.258,4.15,1.933,2.199,0.078,-15.724,5.25,20.135,3.9,0.08,2.388],[6032.0,0.1,8.468,2042.935,332.001,46.889,108.762,84.502,0.778,62.961,2.124,3.012,3.644,3.499,5.403,4.088,1.427,3.066,0.159,-24.218,-3.425,16.296,4.33,0.042,2.314],[6032.5,0.072,8.504,2035.182,266.668,46.661,106.173,71.018,0.541,43.742,1.965,3.75,4.941,8.875,14.449,4.112,0.883,5.224,0.402,-32.216,0.264,11.29,4.822,0.018,2.182],[6033.0,0.083,8.437,2029.342,205.617,46.052,103.982,59.394,0.319,26.712,2.04,4.863,6.861,21.57,20.303,4.248,0.518,9.262,0.814,-39.073,9.736,6.886,4.919,0.031,2.075],[6033.5,0.094,8.305,2028.089,155.334,43.803,102.419,51.88,0.207,17.438,2.128,6.438,9.677,33.725,22.957,4.154,0.35,12.047,1.195,-44.676,11.584,4.39,4.632,0.039,2.046],[6034.0,0.157,8.266,2026.732,116.412,39.479,100.836,47.939,0.187,15.144,2.071,8.59,13.656,37.398,27.951,4.179,0.332,12.583,1.398,-48.688,2.656,3.991,4.459,0.036,2.082],[6034.5,0.209,8.183,2026.562,89.812,34.219,98.616,44.803,0.194,15.784,1.9,11.134,18.365,32.326,29.513,4.436,0.39,11.688,1.513,-51.575,-12.947,4.389,4.237,0.048,2.128],[6035.0,0.209,8.103,2029.194,75.611,31.807,95.955,41.686,0.218,18.118,1.81,13.226,22.437,27.118,24.439,4.615,0.526,9.133,1.634,-54.022,-26.335,5.157,3.712,0.066,2.151],[6035.5,0.209,8.097,2028.642,69.554,31.089,93.688,40.995,0.238,20.924,1.803,14.377,25.363,24.894,21.74,4.802,0.662,7.31,1.712,-56.11,-31.373,5.85,3.293,0.072,2.159],[6036.0,0.187,8.09,2022.835,66.679,30.442,92.42,46.051,0.2,27.387,1.792,14.997,27.45,26.872,27.727,9.849,0.94,9.522,1.697,-57.786,-29.354,7.931,3.153,0.072,2.171],[6036.5,0.181,8.089,2015.995,64.071,29.855,91.956,56.659,0.107,35.476,1.795,15.608,29.371,30.992,30.408,30.933,1.152,27.445,1.617,-59.18,-31.148,10.817,3.513,0.068,2.19],[6037.0,0.175,8.101,2013.487,60.255,29.927,91.692,66.881,0.041,39.158,1.877,16.596,32.053,34.389,29.579,75.503,1.089,71.293,1.553,-60.365,-38.954,12.187,4.178,0.058,2.211],[6037.5,0.175,8.103,2023.558,55.079,29.907,91.386,68.351,0.043,37.637,1.92,18.156,35.517,37.798,31.811,76.329,1.026,73.522,1.582,-61.245,-41.354,11.826,4.27,0.046,2.225],[6038.0,0.163,8.099,2038.136,49.593,29.186,91.234,58.294,0.095,30.216,1.862,20.164,38.869,43.825,39.786,30.358,0.903,31.082,1.762,-61.674,-29.709,9.312,3.78,0.036,2.225],[6038.5,0.161,8.097,2037.686,44.746,28.795,91.432,43.469,0.184,21.981,1.779,22.348,41.774,54.299,52.197,8.351,0.773,10.608,2.114,-61.549,-14.309,6.43,3.068,0.029,2.209],[6039.0,0.169,8.098,2027.019,40.647,29.432,91.878,31.717,0.304,17.968,1.67,24.602,44.837,70.117,63.416,3.289,0.773,4.319,2.581,-60.886,-5.329,4.393,2.132,0.028,2.187],[6039.5,0.168,8.095,2023.468,37.062,28.799,92.3,25.448,0.393,16.084,1.547,26.982,48.31,82.629,69.577,1.606,0.809,2.017,3.057,-59.91,-0.996,3.131,1.45,0.037,2.17],[6040.0,0.166,8.095,2026.406,34.088,28.318,92.651,22.909,0.439,14.965,1.522,29.336,51.958,81.969,72.492,1.236,0.821,1.508,3.482,-59.095,-1.587,2.749,1.244,0.048,2.16],[6040.5,0.175,8.099,2027.631,31.919,28.078,93.139,22.262,0.479,14.776,1.549,31.33,55.81,76.595,63.473,1.045,0.844,1.243,3.829,-58.722,-13.29,2.539,1.119,0.049,2.153],[6041.0,0.187,8.1,2033.458,30.427,27.613,93.943,21.947,0.499,14.836,1.545,32.866,60.085,75.133,60.056,0.951,0.902,1.057,4.066,-58.73,-27.148,2.423,0.999,0.043,2.15],[6041.5,0.204,8.094,2044.195,29.299,27.428,94.97,21.733,0.508,14.834,1.57,34.131,64.641,79.29,73.511,0.918,0.958,0.959,4.229,-59.084,-30.048,2.381,0.923,0.046,2.15],[6042.0,0.239,8.092,2050.724,28.363,27.429,95.921,22.321,0.501,15.487,1.681,35.257,69.128,85.655,81.55,1.011,1.112,0.912,4.392,-59.994,-22.853,2.571,0.866,0.052,2.149],[6042.5,0.25,8.091,2047.844,27.553,27.24,96.551,23.038,0.464,17.212,1.766,36.294,73.635,90.229,83.027,1.366,1.543,0.89,4.517,-61.456,-13.714,3.198,0.784,0.053,2.149],[6043.0,0.255,8.015,2043.302,26.861,27.279,96.844,22.904,0.447,18.818,1.739,37.228,78.648,93.509,82.165,1.665,2.014,0.832,4.554,-62.959,-8.787,3.804,0.704,0.05,2.153],[6043.5,0.251,7.943,2042.457,26.38,27.716,96.953,22.961,0.462,19.426,1.676,37.908,83.596,95.75,74.632,1.736,2.302,0.757,4.555,-63.993,-12.098,4.1,0.662,0.043,2.157],[6044.0,0.278,7.923,2040.362,26.204,28.26,96.936,24.359,0.482,20.006,1.684,38.163,85.721,97.098,66.422,1.745,2.483,0.704,4.55,-64.548,-12.916,4.297,0.642,0.035,2.159],[6044.5,0.318,7.922,2032.814,26.357,28.384,96.786,26.881,0.512,21.036,1.738,37.941,83.434,98.992,70.479,1.687,2.35,0.725,4.43,-64.88,1.703,4.409,0.706,0.038,2.163],[6045.0,0.325,7.923,2023.814,26.824,28.362,96.536,29.427,0.539,21.934,1.758,37.281,79.451,102.172,75.927,1.607,1.881,0.878,4.134,-65.06,13.693,4.418,0.899,0.053,2.175],[6045.5,0.324,7.89,2016.715,27.599,28.805,96.272,31.294,0.55,22.662,1.744,36.233,76.929,104.135,79.656,1.546,1.453,1.079,3.805,-65.012,3.486,4.338,1.124,0.061,2.187],[6046.0,0.335,7.895,2011.366,28.688,29.347,96.103,31.972,0.556,23.29,1.726,34.858,76.221,102.913,73.928,1.488,1.207,1.241,3.611,-64.67,-11.686,4.22,1.307,0.057,2.19],[6046.5,0.335,7.911,2007.787,30.128,28.563,96.07,30.982,0.563,23.289,1.697,33.192,75.381,100.003,65.898,1.441,1.09,1.323,3.542,-64.033,-16.22,4.14,1.412,0.054,2.183],[6047.0,0.331,7.889,2005.354,32.06,28.126,96.121,28.371,0.55,22.102,1.664,31.192,72.619,96.382,62.665,1.434,1.11,1.296,3.479,-63.154,-20.616,4.025,1.354,0.058,2.174],[6047.5,0.329,7.882,2004.369,34.593,28.223,96.174,25.446,0.498,20.079,1.65,28.907,68.343,92.059,58.123,1.473,1.27,1.171,3.353,-62.101,-25.393,3.738,1.112,0.063,2.165],[6048.0,0.329,7.912,2003.756,37.528,28.741,96.227,23.774,0.454,17.982,1.653,26.647,63.962,87.857,51.946,1.493,1.387,1.077,3.157,-60.886,-20.492,3.463,0.928,0.064,2.16],[6048.5,0.345,7.944,2002.997,40.46,29.472,96.283,23.877,0.449,16.829,1.609,24.715,60.008,83.466,52.062,1.463,1.396,1.049,2.9,-59.408,-15.443,3.349,0.891,0.068,2.162],[6049.0,0.346,7.953,2012.442,43.238,29.714,96.288,24.803,0.479,17.159,1.541,23.128,56.238,75.794,54.283,1.324,1.382,0.957,2.629,-57.56,-20.247,3.217,0.864,0.076,2.17],[6049.5,0.346,7.953,2031.573,46.32,30.089,96.262,25.682,0.529,18.19,1.551,21.589,52.217,65.994,48.498,1.154,1.4,0.827,2.407,-55.323,-26.189,3.111,0.826,0.081,2.174],[6050.0,0.31,7.949,2039.292,51.002,30.649,96.405,26.217,0.565,19.13,1.647,19.607,46.421,57.612,44.091,1.105,1.517,0.732,2.205,-52.742,-26.111,3.189,0.782,0.075,2.172],[6050.5,0.303,7.949,2028.454,59.326,31.016,96.999,26.425,0.581,19.714,1.688,16.856,36.53,51.002,44.076,1.146,1.679,0.683,1.913,-49.758,-26.503,3.399,0.752,0.063,2.17],[6051.0,0.302,7.944,2016.933,73.206,31.286,98.076,26.806,0.583,20.418,1.657,13.66,24.807,45.812,42.287,1.246,1.887,0.661,1.54,-46.189,-24.24,3.706,0.73,0.057,2.171],[6051.5,0.294,7.969,2014.918,94.175,31.242,99.292,27.531,0.539,21.412,1.632,10.618,15.483,40.138,40.403,1.623,2.159,0.743,1.193,-42.073,-11.061,4.363,0.747,0.054,2.172],[6052.0,0.312,8.032,2019.672,122.932,30.994,100.275,28.171,0.469,22.396,1.664,8.135,9.553,30.762,36.971,2.189,2.359,0.924,0.929,-37.841,5.009,5.174,0.813,0.053,2.168],[6052.5,0.278,8.059,2024.647,159.023,30.264,101.121,30.299,0.43,23.579,1.817,6.288,6.213,17.13,27.231,2.587,2.434,1.062,0.724,-33.379,10.317,5.66,0.863,0.058,2.167],[6053.0,0.284,8.094,2020.398,202.69,30.092,102.376,36.426,0.417,26.129,2.131,4.934,4.391,8.803,19.174,2.938,2.478,1.185,0.54,-27.742,8.405,6.256,0.936,0.07,2.178],[6053.5,0.203,8.205,2011.114,255.168,30.674,104.436,46.916,0.476,32.243,2.524,3.919,3.333,4.982,10.253,3.173,2.536,1.251,0.369,-21.136,9.904,7.689,1.123,0.086,2.21],[6054.0,0.167,8.369,2014.225,314.821,31.398,107.044,60.099,0.654,41.666,2.806,3.176,2.66,3.427,3.897,3.047,2.401,1.282,0.228,-15.361,12.119,10.05,1.604,0.094,2.262],[6054.5,0.095,8.491,2031.349,374.814,32.16,109.495,74.137,0.843,49.913,2.837,2.668,2.235,2.793,2.992,2.823,1.909,1.53,0.136,-10.819,7.538,12.124,2.485,0.085,2.319],[6055.0,0.119,8.523,2046.203,427.461,33.515,111.354,87.802,0.954,56.469,2.715,2.339,1.989,2.266,2.57,2.697,1.529,1.769,0.091,-6.415,3.089,13.112,3.197,0.074,2.359],[6055.5,0.163,8.562,2051.674,469.012,35.759,112.798,99.054,1.064,63.69,2.696,2.132,1.857,1.807,1.786,2.725,1.519,1.794,0.074,-2.335,9.955,14.837,3.615,0.074,2.373],[6056.0,0.208,8.612,2054.101,499.664,38.641,114.204,105.578,1.127,69.659,2.848,2.001,1.786,1.662,1.423,2.929,1.568,1.867,0.07,0.221,18.195,16.852,3.986,0.072,2.371],[6056.5,0.209,8.623,2054.777,521.883,41.053,115.566,107.193,1.093,72.422,3.033,1.916,1.744,1.763,1.471,3.149,1.52,2.078,0.068,1.323,15.897,17.552,4.294,0.061,2.371],[6057.0,0.242,8.623,2051.855,537.567,41.961,116.413,106.898,1.03,72.395,3.093,1.86,1.717,1.965,1.518,3.373,1.396,2.435,0.062,1.777,10.755,17.701,4.729,0.051,2.379],[6057.5,0.252,8.623,2047.845,546.733,42.684,116.294,106.48,0.965,70.829,2.952,1.829,1.701,2.177,1.578,3.562,1.249,2.872,0.054,1.979,10.348,17.543,5.236,0.049,2.393],[6058.0,0.247,8.623,2045.457,548.975,44.337,115.311,105.163,0.932,69.093,2.759,1.822,1.7,2.209,1.768,3.599,1.132,3.186,0.049,2.079,9.793,17.123,5.622,0.049,2.405],[6058.5,0.253,8.623,2042.5,545.309,45.223,113.974,103.123,0.925,68.061,2.76,1.834,1.714,2.084,1.762,3.561,1.076,3.311,0.047,2.138,7.417,16.821,5.803,0.053,2.41],[6059.0,0.308,8.615,2041.269,536.606,44.592,112.522,102.084,0.919,67.241,2.902,1.864,1.74,2.008,1.58,3.556,1.063,3.346,0.049,2.037,8.081,16.691,5.828,0.061,2.408],[6059.5,0.263,8.608,2047.647,522.956,44.387,110.735,101.6,0.92,67.552,3.004,1.912,1.782,2.01,1.737,3.593,1.13,3.191,0.053,1.683,11.836,16.881,5.562,0.064,2.401],[6060.0,0.232,8.603,2055.659,503.984,44.252,108.366,100.911,0.93,68.295,3.035,1.984,1.861,1.965,1.73,3.615,1.228,2.948,0.059,1.205,11.504,17.177,5.196,0.064,2.393],[6060.5,0.182,8.626,2056.271,480.364,43.314,105.669,100.3,0.942,68.21,2.994,2.082,1.999,1.85,1.721,3.587,1.309,2.747,0.062,0.754,3.627,17.243,4.899,0.071,2.393],[6061.0,0.175,8.662,2053.729,454.414,44.179,103.541,97.814,0.954,67.591,2.884,2.201,2.19,1.874,1.802,3.471,1.401,2.484,0.058,0.222,-1.688,16.908,4.489,0.085,2.406],[6061.5,0.147,8.696,2055.691,427.379,46.345,102.93,92.132,0.945,64.239,2.821,2.34,2.399,2.218,1.999,3.307,1.382,2.406,0.049,-0.65,4.065,15.975,4.299,0.091,2.431],[6062.0,0.104,8.746,2055.595,397.707,46.293,103.922,86.683,0.877,56.869,2.853,2.514,2.61,2.921,3.455,3.142,1.156,2.779,0.034,-1.786,12.838,14.115,4.578,0.074,2.473],[6062.5,0.104,8.777,2052.231,365.275,43.239,105.715,84.42,0.811,50.311,2.897,2.738,2.855,3.528,5.286,2.963,0.93,3.2,0.021,-2.969,16.888,12.285,4.913,0.036,2.514],[6063.0,0.089,8.821,2055.217,334.111,41.226,107.54,84.659,0.796,47.321,2.522,2.993,3.156,3.676,4.466,2.768,0.813,3.418,0.033,-4.34,20.393,11.26,5.156,-0.012,2.487],[6063.5,0.093,8.908,2062.45,309.246,42.543,109.131,84.756,0.787,44.629,1.989,3.234,3.462,3.914,3.441,2.531,0.695,3.659,0.14,-6.083,24.812,10.171,5.449,-0.047,2.343],[6064.0,0.08,8.965,2063.543,292.042,44.956,110.521,82.188,0.823,43.191,1.946,3.424,3.711,4.519,4.806,2.238,0.64,3.503,0.406,-7.888,25.16,9.352,5.428,-0.013,2.16],[6064.5,0.12,9.006,2062.237,279.812,47.777,111.48,76.614,0.888,43.072,2.129,3.574,3.908,4.941,4.551,2.004,0.647,3.101,0.666,-9.492,24.46,9.064,5.202,0.05,2.045],[6065.0,0.079,9.041,2066.059,270.25,49.915,111.224,71.323,0.931,42.554,2.259,3.7,4.081,4.951,3.93,1.871,0.656,2.855,0.715,-11.106,24.869,8.884,5.029,0.069,2.035],[6065.5,0.089,9.046,2070.316,263.616,47.964,109.048,68.936,0.964,41.853,2.205,3.793,4.222,4.991,4.357,1.75,0.682,2.573,0.589,-12.909,21.129,8.603,4.695,0.045,2.094],[6066.0,0.095,9.038,2067.647,260.443,45.656,105.153,68.271,0.951,41.29,2.034,3.84,4.317,5.124,4.642,1.754,0.753,2.339,0.443,-14.682,10.445,8.506,4.218,0.031,2.166],[6066.5,0.101,9.039,2060.255,259.653,40.903,100.622,67.905,0.921,41.688,1.858,3.851,4.381,5.099,4.242,1.875,0.859,2.19,0.332,-16.325,-2.462,8.816,3.823,0.04,2.228],[6067.0,0.104,8.951,2055.896,260.095,37.362,96.713,67.702,0.915,43.071,1.747,3.845,4.425,4.974,4.115,1.964,0.96,2.051,0.24,-18.233,-10.377,9.175,3.557,0.05,2.286],[6067.5,0.11,8.787,2058.319,261.229,35.053,94.253,67.146,0.9,44.72,1.707,3.828,4.447,5.182,4.493,2.082,1.078,1.936,0.177,-21.026,-7.241,9.558,3.304,0.059,2.334],[6068.0,0.116,8.653,2054.732,262.895,32.86,93.356,64.923,0.896,45.803,1.697,3.804,4.449,5.52,5.22,2.184,1.248,1.766,0.148,-24.842,7.612,9.987,2.992,0.065,2.358],[6068.5,0.128,8.448,2040.118,265.521,31.69,93.566,62.065,0.961,46.48,1.699,3.766,4.438,5.608,5.003,2.046,1.524,1.386,0.143,-29.12,22.143,9.984,2.488,0.063,2.362],[6069.0,0.133,8.232,2031.455,270.155,32.323,94.191,60.123,1.06,46.949,1.693,3.702,4.409,5.601,4.775,1.811,1.881,0.987,0.157,-33.114,27.117,9.768,1.96,0.055,2.346],[6069.5,0.111,8.125,2037.489,277.779,33.528,94.566,58.934,1.132,47.314,1.66,3.6,4.345,5.742,5.006,1.681,2.15,0.786,0.193,-36.643,22.326,9.713,1.682,0.052,2.311],[6070.0,0.138,8.099,2041.323,288.117,34.273,94.348,57.273,1.189,48.666,1.669,3.471,4.217,5.861,4.904,1.596,2.382,0.675,0.243,-39.926,7.249,9.68,1.516,0.061,2.266],[6070.5,0.124,8.097,2030.304,301.383,34.315,93.628,55.315,1.257,50.444,1.787,3.318,4.001,5.772,4.96,1.495,2.667,0.565,0.288,-43.05,-12.71,9.58,1.34,0.075,2.226],[6071.0,0.104,8.094,2010.18,320.496,33.435,92.656,52.886,1.269,49.754,1.921,3.12,3.689,5.604,5.22,1.422,2.628,0.547,0.292,-45.778,-24.684,9.217,1.311,0.077,2.211],[6071.5,0.123,8.098,1991.062,348.405,31.793,91.544,49.966,1.211,46.527,1.964,2.87,3.293,5.546,5.583,1.4,2.245,0.632,0.243,-48.278,-26.763,8.657,1.444,0.069,2.231],[6072.0,0.123,8.103,1980.707,385.025,29.952,90.272,47.901,1.143,42.915,1.87,2.597,2.857,5.525,5.583,1.393,1.898,0.741,0.171,-51.237,-29.38,8.128,1.598,0.062,2.277],[6072.5,0.141,8.095,1982.642,429.731,28.707,88.94,47.203,1.048,38.694,1.709,2.327,2.429,5.368,4.936,1.364,1.533,0.914,0.118,-54.901,-36.288,7.311,1.795,0.06,2.32],[6073.0,0.133,8.093,1990.805,483.879,27.549,87.922,46.115,0.946,34.407,1.679,2.067,2.059,4.799,4.639,1.318,1.203,1.108,0.095,-58.682,-42.77,6.373,1.976,0.065,2.335],[6073.5,0.083,8.1,1995.977,547.866,25.864,87.573,43.77,0.882,31.423,1.803,1.825,1.772,4.167,4.371,1.293,1.061,1.22,0.091,-62.156,-45.432,5.828,2.038,0.072,2.322],[6074.0,0.073,8.092,1994.368,615.948,24.995,87.878,40.608,0.807,28.806,1.894,1.624,1.554,3.673,4.044,1.297,0.985,1.32,0.098,-65.395,-44.816,5.337,2.008,0.073,2.292],[6074.5,0.078,8.084,1990.235,680.107,25.207,88.473,37.277,0.714,25.74,1.829,1.47,1.388,3.305,3.727,1.317,0.901,1.474,0.108,-68.183,-41.992,4.794,1.976,0.068,2.258],[6075.0,0.078,8.087,1987.95,739.782,25.697,88.922,34.442,0.578,21.549,1.68,1.352,1.267,2.931,3.096,1.431,0.75,1.993,0.115,-70.003,-36.353,4.153,2.075,0.061,2.229],[6075.5,0.078,8.092,1986.767,796.488,25.764,89.017,33.425,0.449,18.024,1.553,1.256,1.188,2.52,2.491,1.586,0.609,2.634,0.118,-70.842,-30.598,3.614,2.208,0.053,2.211],[6076.0,0.095,8.089,1990.838,846.003,25.885,88.869,33.583,0.406,16.729,1.553,1.182,1.133,2.226,2.142,1.621,0.547,2.973,0.117,-71.257,-24.113,3.36,2.282,0.046,2.199],[6076.5,0.101,8.06,2010.674,885.481,26.494,88.734,33.05,0.383,16.143,1.657,1.129,1.091,2.162,2.241,1.61,0.498,3.238,0.118,-71.673,-15.286,3.145,2.343,0.043,2.189],[6077.0,0.107,8.047,2035.211,917.994,27.161,88.793,31.66,0.376,15.781,1.724,1.089,1.058,2.278,2.547,1.583,0.481,3.293,0.117,-72.126,-8.806,3.04,2.347,0.046,2.182],[6077.5,0.124,8.053,2041.487,942.336,27.334,89.104,30.624,0.399,15.758,1.706,1.061,1.037,2.316,2.449,1.514,0.508,2.998,0.115,-72.465,-12.348,3.079,2.256,0.05,2.18],[6078.0,0.124,8.051,2035.917,954.265,27.047,89.642,30.911,0.429,16.09,1.694,1.048,1.026,2.139,1.929,1.449,0.552,2.633,0.114,-72.561,-24.891,3.171,2.134,0.051,2.18],[6078.5,0.107,8.035,2038.889,953.278,27.011,90.342,32.401,0.455,17.376,1.726,1.049,1.025,1.975,1.45,1.463,0.618,2.381,0.113,-72.421,-37.371,3.403,2.047,0.051,2.181],[6079.0,0.107,8.031,2045.613,939.566,27.241,91.089,34.24,0.473,19.615,1.765,1.064,1.034,1.988,1.737,1.607,0.718,2.247,0.115,-72.172,-41.457,3.886,2.012,0.05,2.18],[6079.5,0.11,8.046,2046.451,914.913,27.449,91.711,35.263,0.47,20.827,1.778,1.093,1.063,2.082,2.017,1.723,0.746,2.316,0.12,-72.004,-36.176,4.136,2.061,0.046,2.177],[6080.0,0.141,8.048,2051.073,887.072,28.589,92.024,35.337,0.448,20.481,1.736,1.127,1.115,2.171,2.003,1.769,0.693,2.56,0.124,-72.01,-31.049,4.042,2.167,0.045,2.176],[6080.5,0.158,8.037,2067.609,862.96,29.098,91.92,34.913,0.424,19.788,1.642,1.159,1.177,2.36,1.984,1.826,0.65,2.815,0.125,-72.071,-31.881,3.953,2.259,0.053,2.18],[6081.0,0.146,8.022,2081.978,841.123,28.902,91.414,34.518,0.408,19.386,1.547,1.189,1.232,2.813,2.304,1.908,0.654,2.917,0.122,-72.048,-34.48,3.974,2.256,0.068,2.191],[6081.5,0.12,8.012,2083.932,819.96,28.482,90.626,34.266,0.405,19.521,1.488,1.22,1.27,3.317,2.798,1.981,0.714,2.784,0.115,-71.983,-32.835,4.101,2.137,0.081,2.209],[6082.0,0.124,8.006,2082.057,803.857,27.923,89.707,33.958,0.42,20.081,1.439,1.244,1.289,3.619,3.029,1.971,0.787,2.515,0.106,-72.008,-28.157,4.218,1.994,0.084,2.23],[6082.5,0.118,8.008,2079.301,795.536,27.159,88.764,33.551,0.451,20.98,1.395,1.257,1.287,3.62,3.261,1.879,0.882,2.153,0.098,-72.109,-20.938,4.319,1.827,0.073,2.246],[6083.0,0.113,8.011,2069.581,793.007,26.497,87.899,32.738,0.5,21.766,1.395,1.261,1.273,3.323,2.946,1.708,1.004,1.719,0.095,-72.181,-9.038,4.335,1.611,0.059,2.254],[6083.5,0.121,8.027,2056.847,795.501,26.266,87.263,31.469,0.539,21.716,1.461,1.257,1.256,3.035,2.468,1.536,1.083,1.423,0.095,-72.185,5.6,4.218,1.448,0.05,2.253],[6084.0,0.183,8.047,2041.906,804.91,26.546,87.004,30.362,0.545,21.205,1.534,1.242,1.239,2.922,2.748,1.47,1.114,1.32,0.098,-72.178,7.976,4.093,1.364,0.045,2.246],[6084.5,0.293,8.042,2025.962,817.92,27.213,87.151,29.764,0.536,21.213,1.576,1.223,1.225,2.911,3.3,1.5,1.193,1.261,0.1,-72.184,-7.733,4.106,1.281,0.043,2.237],[6085.0,0.38,8.023,2021.366,829.013,28.047,87.611,29.39,0.527,21.822,1.588,1.206,1.215,2.945,2.993,1.593,1.376,1.167,0.102,-72.182,-22.755,4.287,1.166,0.049,2.231],[6085.5,0.45,8.02,2027.293,837.274,28.697,88.208,29.366,0.522,22.697,1.595,1.194,1.208,3.006,3.126,1.747,1.776,1.007,0.104,-72.149,-24.676,4.657,0.996,0.058,2.225],[6086.0,0.503,8.004,2030.294,843.433,29.457,88.71,29.591,0.517,23.523,1.592,1.186,1.203,3.035,3.476,1.939,2.297,0.853,0.106,-72.075,-25.445,5.112,0.835,0.07,2.221],[6086.5,0.518,7.997,2038.75,843.695,30.515,88.973,30.218,0.481,23.976,1.601,1.185,1.204,2.979,3.233,2.205,2.461,0.903,0.106,-72.014,-31.738,5.379,0.814,0.08,2.22],[6087.0,0.505,7.993,2056.068,837.319,29.865,89.056,30.937,0.424,24.03,1.646,1.194,1.212,2.932,3.128,2.501,2.215,1.147,0.104,-71.998,-35.336,5.39,0.911,0.077,2.225],[6087.5,0.535,7.996,2059.032,830.312,27.873,89.127,30.779,0.389,23.378,1.673,1.204,1.219,2.948,3.089,2.616,1.873,1.414,0.102,-71.938,-32.101,5.194,1.038,0.068,2.231],[6088.0,0.547,8.0,2044.867,826.297,26.996,89.256,30.082,0.368,21.3,1.6,1.21,1.225,2.922,3.023,2.517,1.47,1.763,0.098,-71.699,-28.282,4.739,1.224,0.068,2.24],[6088.5,0.545,8.019,2036.632,823.079,26.624,89.385,30.34,0.366,18.468,1.487,1.215,1.229,2.874,3.117,2.182,1.063,2.08,0.093,-71.249,-28.463,4.059,1.442,0.079,2.251],[6089.0,0.538,8.143,2034.762,821.973,26.499,89.401,30.93,0.387,16.76,1.509,1.217,1.227,2.894,3.044,1.785,0.83,2.148,0.09,-70.674,-26.41,3.51,1.574,0.091,2.257],[6089.5,0.4,8.223,2027.532,825.979,27.261,89.313,30.463,0.408,16.986,1.621,1.211,1.219,2.964,2.684,1.622,0.792,2.051,0.091,-70.257,-20.495,3.378,1.584,0.095,2.254],[6090.0,0.328,8.253,2021.16,832.079,28.887,89.345,30.041,0.422,18.482,1.698,1.202,1.209,2.963,2.734,1.659,0.884,1.886,0.094,-70.261,-14.293,3.574,1.506,0.089,2.247],[6090.5,0.241,8.286,2020.281,837.938,30.378,89.668,30.521,0.426,19.81,1.753,1.193,1.198,2.893,2.613,1.784,1.029,1.743,0.096,-70.576,-3.39,3.879,1.406,0.077,2.241],[6091.0,0.204,8.321,2017.207,848.062,30.322,90.14,31.086,0.426,20.783,1.84,1.179,1.182,2.796,2.476,1.974,1.269,1.576,0.096,-70.899,10.031,4.297,1.273,0.067,2.239],[6091.5,0.209,8.328,2010.409,865.122,29.692,90.438,30.793,0.429,21.818,1.853,1.156,1.159,2.69,2.615,2.166,1.601,1.368,0.094,-71.103,13.27,4.752,1.112,0.067,2.238],[6092.0,0.172,8.318,2006.125,882.576,29.526,90.393,29.875,0.442,22.727,1.729,1.133,1.135,2.608,2.676,2.21,1.861,1.194,0.092,-71.271,4.68,4.987,0.998,0.07,2.24],[6092.5,0.152,8.325,2008.642,896.675,29.151,90.136,29.09,0.46,23.19,1.574,1.115,1.112,2.548,2.794,2.129,1.965,1.085,0.09,-71.557,0.547,5.004,0.946,0.065,2.24],[6093.0,0.122,8.328,2012.529,910.772,28.323,89.987,28.67,0.453,22.829,1.504,1.098,1.093,2.534,2.78,2.131,1.882,1.137,0.091,-71.978,10.476,4.923,0.973,0.053,2.236],[6093.5,0.124,8.329,2009.773,925.688,27.77,90.142,28.539,0.423,22.097,1.511,1.08,1.08,2.439,2.597,2.242,1.741,1.3,0.092,-72.232,23.235,4.829,1.033,0.047,2.229],[6094.0,0.123,8.331,1999.351,937.351,27.701,90.508,28.603,0.368,20.457,1.535,1.067,1.074,2.294,2.315,2.471,1.479,1.744,0.093,-72.1,25.916,4.6,1.179,0.049,2.224],[6094.5,0.129,8.33,1992.781,944.912,27.604,90.914,28.541,0.3,17.9,1.586,1.058,1.069,2.264,2.069,2.786,1.128,2.554,0.095,-71.818,24.685,4.23,1.413,0.049,2.22],[6095.0,0.131,8.324,1997.074,948.516,27.47,91.347,28.275,0.259,15.766,1.672,1.054,1.067,2.321,2.24,2.912,0.888,3.34,0.097,-71.631,27.518,3.852,1.627,0.048,2.215],[6095.5,0.148,8.317,2005.112,943.26,27.482,91.877,28.442,0.253,14.535,1.739,1.06,1.074,2.365,2.088,2.731,0.734,3.723,0.101,-71.464,27.423,3.521,1.785,0.05,2.208],[6096.0,0.135,8.321,2008.769,926.394,27.616,92.451,29.501,0.282,14.216,1.763,1.079,1.096,2.366,1.784,2.279,0.636,3.565,0.105,-71.234,23.76,3.222,1.886,0.048,2.204],[6096.5,0.145,8.318,2007.937,904.567,27.786,92.852,30.458,0.38,15.286,1.707,1.106,1.131,2.34,1.955,1.647,0.586,2.812,0.107,-70.98,21.188,3.021,1.916,0.042,2.205],[6097.0,0.135,8.315,2005.696,886.506,27.985,92.831,30.506,0.496,17.448,1.601,1.128,1.17,2.281,2.019,1.238,0.618,2.024,0.107,-70.63,17.989,3.093,1.861,0.038,2.208],[6097.5,0.142,8.321,2004.915,874.236,28.537,92.323,30.671,0.536,18.94,1.555,1.144,1.196,2.348,1.796,1.198,0.673,1.782,0.111,-70.115,13.662,3.28,1.811,0.035,2.204],[6098.0,0.152,8.324,2010.872,866.122,29.525,91.526,32.208,0.54,19.772,1.624,1.155,1.197,2.718,2.381,1.258,0.736,1.715,0.117,-69.607,15.494,3.472,1.755,0.035,2.195],[6098.5,0.164,8.313,2020.901,861.809,29.952,90.714,33.497,0.548,20.601,1.745,1.16,1.183,3.027,3.035,1.331,0.848,1.579,0.121,-69.319,24.039,3.729,1.64,0.038,2.189],[6099.0,0.163,8.316,2023.77,861.502,29.327,90.059,32.396,0.547,20.896,1.86,1.161,1.165,2.793,2.517,1.407,0.96,1.469,0.12,-69.17,31.664,3.929,1.522,0.033,2.19],[6099.5,0.176,8.322,2016.431,868.43,28.752,89.659,29.921,0.572,21.104,1.912,1.152,1.141,2.411,2.064,1.366,1.094,1.269,0.117,-69.008,35.665,3.971,1.361,0.027,2.195],[6100.0,0.293,8.32,2012.05,887.049,28.533,89.606,28.292,0.627,21.54,1.871,1.127,1.106,2.325,2.171,1.221,1.248,0.991,0.112,-68.778,40.08,3.894,1.167,0.041,2.199],[6100.5,0.27,8.3,2019.566,915.176,28.141,89.932,27.9,0.673,21.919,1.823,1.093,1.068,2.426,2.322,1.107,1.378,0.808,0.109,-68.438,43.96,3.799,1.027,0.065,2.198],[6101.0,0.209,8.314,2031.339,942.165,27.827,90.547,27.696,0.699,22.12,1.841,1.061,1.036,2.481,2.23,1.03,1.424,0.724,0.107,-67.868,38.18,3.674,0.957,0.07,2.195],[6101.5,0.139,8.341,2033.736,956.22,27.761,91.309,26.995,0.704,21.824,1.871,1.046,1.019,2.414,2.047,0.979,1.365,0.718,0.107,-66.927,18.596,3.521,0.958,0.063,2.193],[6102.0,0.129,8.338,2026.791,951.273,27.611,92.083,26.247,0.689,21.051,1.881,1.051,1.023,2.347,2.045,0.985,1.275,0.775,0.107,-65.477,-3.277,3.462,1.009,0.058,2.193],[6102.5,0.097,8.345,2016.197,928.071,27.69,92.817,26.819,0.659,20.175,1.912,1.078,1.053,2.316,2.083,1.034,1.183,0.877,0.109,-63.266,-14.839,3.475,1.092,0.053,2.195],[6103.0,0.078,8.358,2003.952,888.447,28.588,93.537,30.255,0.627,20.198,1.983,1.126,1.113,2.239,2.066,1.121,1.123,0.998,0.11,-59.789,-16.956,3.569,1.179,0.047,2.202],[6103.5,0.08,8.369,1993.293,833.195,29.868,94.177,37.072,0.538,21.808,2.075,1.2,1.213,2.221,1.858,1.829,1.222,1.442,0.11,-54.864,-18.182,4.668,1.394,0.045,2.215],[6104.0,0.091,8.367,1987.314,765.116,31.282,94.382,46.68,0.424,25.876,2.204,1.307,1.362,2.354,1.899,3.156,1.435,2.158,0.105,-49.053,-20.419,6.581,1.686,0.047,2.24],[6104.5,0.129,8.337,1981.521,686.476,33.012,93.506,58.279,0.361,32.177,2.441,1.457,1.572,2.492,2.319,4.727,1.451,3.335,0.097,-42.483,-20.341,8.519,2.229,0.05,2.276],[6105.0,0.15,8.316,1976.582,597.371,35.381,91.018,70.387,0.428,41.114,2.774,1.674,1.87,2.491,2.12,5.227,1.334,3.961,0.087,-34.901,-19.175,11.131,3.135,0.057,2.316],[6105.5,0.162,8.338,1981.122,503.306,37.607,87.156,80.322,0.547,50.587,3.054,1.987,2.308,2.427,2.051,4.804,1.306,3.692,0.079,-27.285,-19.009,13.364,3.792,0.062,2.355],[6106.0,0.163,8.373,1989.526,416.099,39.283,83.072,85.138,0.575,55.363,3.13,2.403,2.95,2.579,2.035,4.759,1.312,3.628,0.069,-21.544,-17.156,13.973,3.953,0.063,2.396],[6106.5,0.304,8.394,1990.212,345.066,38.52,79.942,82.632,0.571,54.322,3.143,2.898,3.805,3.201,1.994,4.714,1.295,3.642,0.051,-18.863,-13.062,13.735,3.937,0.07,2.449],[6107.0,0.263,8.401,1989.331,293.89,34.806,77.997,72.9,0.522,48.352,3.21,3.403,4.706,4.813,7.794,4.808,1.203,4.047,0.024,-19.306,-7.694,12.766,3.939,0.078,2.519],[6107.5,0.281,8.366,1998.897,260.183,29.491,76.577,60.36,0.425,39.481,3.324,3.843,5.29,9.237,28.53,4.955,1.042,4.874,0.004,-21.857,0.57,10.66,3.804,0.074,2.599],[6108.0,0.229,8.334,2011.214,237.482,24.526,74.959,51.134,0.435,34.958,3.378,4.211,5.394,17.85,40.204,4.291,1.154,4.018,0.0,-25.073,9.687,9.225,3.095,0.064,2.651],[6108.5,0.207,8.298,2015.755,220.756,21.475,73.235,47.583,0.517,34.621,3.063,4.53,5.274,12.191,27.693,3.458,1.403,2.496,0.003,-28.543,11.671,9.085,2.418,0.047,2.606],[6109.0,0.071,8.251,2014.573,210.604,21.123,72.246,48.851,0.553,35.546,2.691,4.748,5.205,6.128,7.643,3.292,1.516,2.188,0.075,-31.86,-0.852,9.277,2.278,0.025,2.458],[6109.5,0.09,8.279,2011.268,210.081,24.868,72.614,53.842,0.635,39.985,2.592,4.76,5.25,4.384,2.528,3.071,1.645,1.898,0.265,-33.118,-22.664,9.863,2.248,0.028,2.305],[6110.0,0.147,8.355,2008.706,220.761,29.592,74.123,61.914,0.7,42.704,2.689,4.53,5.263,4.216,2.838,2.805,1.465,1.981,0.382,-30.817,-38.374,10.025,2.628,0.043,2.232],[6110.5,0.243,8.374,2008.725,240.373,31.135,75.998,69.16,0.694,41.814,2.994,4.16,5.009,5.144,2.793,2.708,1.119,2.444,0.311,-25.543,-40.797,9.603,3.212,0.055,2.255],[6111.0,0.265,8.367,2009.745,261.414,30.201,77.486,70.674,0.693,40.184,3.459,3.825,4.484,7.386,5.783,2.601,0.968,2.695,0.165,-18.625,-36.13,9.196,3.538,0.061,2.344],[6111.5,0.362,8.359,2012.284,274.866,29.009,78.297,69.313,0.712,38.19,3.608,3.638,3.957,8.525,8.887,2.419,0.853,2.846,0.07,-11.684,-29.638,8.779,3.839,0.056,2.44],[6112.0,0.39,8.363,2021.954,277.723,28.883,78.788,72.334,0.76,37.725,3.095,3.601,3.637,5.674,5.625,2.186,0.762,2.865,0.041,-6.534,-26.064,8.448,4.118,0.045,2.486],[6112.5,0.439,8.369,2034.137,276.204,29.241,79.596,79.773,0.846,40.204,2.672,3.621,3.548,3.482,3.352,1.962,0.731,2.686,0.047,-3.449,-28.955,8.437,4.286,0.047,2.475],[6113.0,0.414,8.373,2033.262,279.428,31.438,81.162,85.428,0.936,44.635,2.658,3.579,3.578,2.797,2.647,1.902,0.754,2.527,0.062,-1.418,-35.264,9.095,4.473,0.059,2.45],[6113.5,0.409,8.388,2021.521,292.295,35.36,83.478,85.508,0.97,47.63,2.859,3.421,3.55,2.954,2.181,1.976,0.784,2.519,0.061,-0.093,-38.42,9.783,4.628,0.068,2.446],[6114.0,0.406,8.42,2017.026,314.067,37.298,86.11,80.847,0.937,47.048,3.052,3.184,3.364,3.881,3.169,2.076,0.799,2.598,0.046,0.534,-35.242,9.912,4.604,0.066,2.467],[6114.5,0.38,8.427,2023.311,339.856,35.292,88.683,75.593,0.891,44.739,3.135,2.942,3.079,5.088,4.215,2.18,0.807,2.705,0.029,1.022,-25.72,9.896,4.552,0.05,2.495],[6115.0,0.383,8.426,2024.532,364.376,34.093,91.326,73.639,0.932,44.52,3.216,2.744,2.782,5.145,4.275,2.107,0.829,2.55,0.022,1.848,-14.819,9.984,4.472,0.037,2.51],[6115.5,0.362,8.438,2017.716,386.415,35.067,94.362,74.806,1.013,46.224,3.291,2.588,2.503,4.054,3.582,1.952,0.858,2.276,0.024,2.894,-10.219,10.092,4.365,0.043,2.501],[6116.0,0.332,8.441,2013.594,408.292,37.777,97.726,78.586,1.073,48.209,3.208,2.449,2.264,2.939,2.7,1.864,0.879,2.125,0.032,3.914,-12.959,10.194,4.307,0.059,2.475],[6116.5,0.272,8.413,2016.103,431.988,40.61,101.04,84.676,1.211,52.048,2.931,2.315,2.091,2.277,2.052,1.733,0.938,1.858,0.042,4.813,-18.681,10.653,4.216,0.07,2.445],[6117.0,0.261,8.38,2014.056,458.56,42.995,104.021,90.08,1.334,56.434,2.705,2.181,1.983,1.816,1.635,1.668,1.018,1.64,0.053,5.57,-25.258,11.363,4.142,0.075,2.415],[6117.5,0.252,8.4,2004.035,488.284,44.608,106.554,91.373,1.334,58.833,2.746,2.048,1.911,1.597,1.449,1.744,1.061,1.644,0.063,6.1,-31.774,11.878,4.154,0.086,2.389],[6118.0,0.374,8.429,1998.878,519.035,45.209,108.49,89.783,1.285,59.99,3.039,1.927,1.839,1.676,1.358,1.879,1.094,1.716,0.063,6.35,-36.832,12.313,4.177,0.095,2.38],[6118.5,0.25,8.433,2010.62,546.049,45.232,109.568,88.463,1.233,60.756,3.377,1.831,1.763,1.949,1.621,2.047,1.156,1.77,0.052,6.457,-40.692,12.877,4.134,0.085,2.398],[6119.0,0.251,8.425,2033.065,564.489,45.231,109.679,88.292,1.19,61.29,3.506,1.772,1.702,2.164,2.006,2.235,1.258,1.778,0.039,6.563,-43.955,13.567,4.009,0.068,2.425],[6119.5,0.283,8.414,2047.815,573.922,44.954,109.265,89.23,1.165,62.512,3.319,1.742,1.661,2.088,1.686,2.409,1.386,1.74,0.036,6.535,-45.187,14.326,3.842,0.072,2.433],[6120.0,0.274,8.383,2042.917,579.994,45.284,109.153,91.817,1.173,66.268,3.041,1.724,1.63,1.899,1.463,2.561,1.612,1.608,0.039,6.344,-40.535,15.349,3.568,0.094,2.424],[6120.5,0.36,8.382,2029.507,589.989,45.818,109.777,95.706,1.213,72.472,2.911,1.695,1.604,1.85,1.442,2.715,2.007,1.374,0.042,6.23,-28.177,16.83,3.148,0.103,2.414],[6121.0,0.475,8.413,2020.691,606.639,46.014,110.869,99.149,1.247,76.761,2.952,1.648,1.574,1.924,1.385,2.804,2.284,1.229,0.042,6.229,-18.824,17.855,2.903,0.089,2.411],[6121.5,0.494,8.424,2016.424,625.742,46.387,111.897,100.124,1.248,77.6,3.095,1.598,1.534,1.95,1.389,2.851,2.302,1.239,0.038,6.18,-19.286,18.162,2.928,0.076,2.417],[6122.0,0.635,8.418,2014.646,637.794,47.02,112.383,99.282,1.244,77.533,3.266,1.568,1.497,1.932,1.431,2.881,2.26,1.276,0.035,6.007,-18.826,18.311,3.008,0.079,2.426],[6122.5,0.619,8.425,2018.366,633.947,47.25,111.73,99.423,1.237,76.35,3.31,1.577,1.486,1.849,1.427,2.839,2.044,1.407,0.034,5.549,-10.216,17.94,3.294,0.091,2.429],[6123.0,0.774,8.426,2021.408,611.345,47.041,109.289,100.878,1.205,73.295,3.128,1.636,1.53,1.701,1.351,2.745,1.642,1.703,0.037,4.766,-3.025,16.901,3.878,0.097,2.422],[6123.5,0.719,8.389,2021.413,573.966,47.025,105.064,101.755,1.146,70.176,2.9,1.742,1.663,1.514,1.28,2.723,1.357,2.021,0.044,3.701,-4.17,15.931,4.375,0.1,2.411],[6124.0,0.746,8.369,2028.834,531.068,47.323,100.169,100.248,1.088,67.574,2.789,1.883,1.913,1.422,1.274,2.761,1.231,2.247,0.05,2.225,-1.802,15.341,4.629,0.104,2.405],[6124.5,0.709,8.39,2042.664,494.648,46.96,96.186,94.354,1.041,63.864,2.815,2.022,2.256,1.65,1.336,2.798,1.147,2.446,0.049,0.098,13.607,14.867,4.816,0.116,2.416],[6125.0,0.601,8.419,2047.092,472.828,44.581,93.78,82.712,0.962,58.437,3.022,2.115,2.554,2.543,1.454,2.899,1.126,2.58,0.034,-2.941,25.679,14.212,4.702,0.128,2.457],[6125.5,0.497,8.44,2039.863,467.132,39.827,92.313,68.999,0.911,54.886,3.351,2.141,2.616,5.202,11.664,2.907,1.488,2.124,0.013,-7.251,17.769,13.514,3.655,0.12,2.528],[6126.0,0.34,8.433,2029.791,477.04,34.423,91.027,62.396,0.94,54.286,3.496,2.096,2.427,7.565,33.306,2.81,2.258,1.344,0.002,-12.486,3.167,13.478,2.36,0.081,2.598],[6126.5,0.258,8.418,2022.329,503.986,31.929,90.285,66.554,0.958,54.416,3.274,1.984,2.11,4.561,22.586,2.756,2.517,1.107,0.001,-17.701,-2.591,13.484,2.006,0.038,2.615],[6127.0,0.206,8.423,2024.272,549.958,33.626,91.148,72.071,0.994,57.41,2.91,1.818,1.771,2.695,2.038,2.62,2.41,1.098,0.006,-23.623,-5.257,13.261,2.056,0.039,2.559],[6127.5,0.165,8.395,2034.861,614.506,36.812,93.725,68.259,1.017,58.988,2.467,1.627,1.477,2.23,2.301,2.525,2.422,1.05,0.026,-32.641,-9.33,13.09,2.013,0.063,2.458],[6128.0,0.096,8.35,2041.437,692.119,37.993,96.634,55.759,0.877,50.567,2.097,1.445,1.266,2.371,2.654,2.538,1.956,1.393,0.059,-44.277,-11.202,11.373,2.215,0.07,2.351],[6128.5,0.158,8.33,2035.442,774.383,36.867,98.158,42.616,0.675,37.083,1.929,1.291,1.139,2.484,2.577,2.417,1.32,1.901,0.091,-54.609,-12.265,8.404,2.375,0.061,2.266],[6129.0,0.134,8.328,2028.411,852.138,35.037,97.611,34.106,0.507,26.664,1.945,1.174,1.07,2.185,2.418,2.287,1.002,2.316,0.11,-60.711,-13.663,5.918,2.166,0.054,2.211],[6129.5,0.185,8.323,2033.132,912.887,33.453,95.533,29.451,0.373,20.234,2.011,1.095,1.034,1.976,1.989,2.315,0.855,2.736,0.116,-62.37,-12.697,4.391,1.896,0.049,2.184],[6130.0,0.208,8.326,2043.39,936.751,32.523,92.785,25.829,0.323,17.189,1.994,1.068,1.022,2.356,2.825,2.352,0.808,2.913,0.115,-60.758,-11.547,3.886,1.786,0.045,2.182],[6130.5,0.253,8.335,2041.377,910.563,31.893,89.895,22.692,0.324,16.257,1.909,1.098,1.046,3.46,3.93,2.331,0.896,2.681,0.111,-57.812,-15.189,3.857,1.641,0.05,2.195],[6131.0,0.278,8.335,2024.884,843.994,30.858,87.199,20.974,0.341,16.97,1.864,1.185,1.123,3.994,3.024,2.311,1.384,1.895,0.112,-54.883,-22.671,4.026,1.203,0.061,2.209],[6131.5,0.295,8.328,2008.348,762.801,29.27,85.163,20.81,0.367,17.553,1.93,1.311,1.27,2.993,1.136,2.298,1.875,1.241,0.115,-51.902,-29.894,4.307,0.86,0.072,2.224],[6132.0,0.299,8.325,2004.494,686.749,27.231,84.231,23.503,0.381,17.553,2.085,1.456,1.488,2.157,6.355,2.288,2.004,1.142,0.105,-47.923,-28.737,4.446,0.824,0.091,2.261],[6132.5,0.218,8.329,2008.484,622.146,25.341,84.538,32.18,0.402,19.749,2.314,1.607,1.753,1.906,22.951,2.258,2.1,1.077,0.071,-41.06,-15.358,4.632,0.819,0.112,2.339],[6133.0,0.15,8.354,2008.537,567.931,24.426,85.924,47.811,0.493,25.267,2.626,1.761,2.004,2.224,13.817,2.193,1.95,1.143,0.034,-30.224,-6.667,5.499,1.092,0.12,2.439],[6133.5,0.253,8.379,2005.778,523.832,24.237,88.105,65.539,0.618,32.166,2.998,1.909,2.167,3.084,2.854,2.293,1.457,1.718,0.015,-17.828,-10.966,7.271,2.065,0.123,2.512],[6134.0,0.325,8.403,2006.978,491.015,25.443,90.803,79.23,0.675,37.46,3.217,2.037,2.223,3.498,3.496,2.564,1.005,2.698,0.009,-7.965,-15.992,8.831,3.448,0.13,2.545],[6134.5,0.167,8.416,2008.003,470.946,27.713,93.649,87.111,0.739,42.309,3.245,2.123,2.21,2.994,4.932,2.612,0.825,3.166,0.007,-2.234,-14.789,9.817,4.423,0.117,2.557],[6135.0,0.156,8.435,2003.177,463.911,32.543,96.174,91.767,0.823,48.086,3.178,2.156,2.18,2.318,2.621,2.639,0.847,3.113,0.008,0.473,-15.36,11.097,4.851,0.082,2.554],[6135.5,0.132,8.447,1997.851,465.727,38.17,97.861,95.23,0.836,53.675,3.082,2.147,2.166,1.963,1.428,3.0,0.946,3.167,0.01,1.4,-14.774,12.788,5.014,0.07,2.539],[6136.0,0.166,8.361,1994.05,467.531,42.65,98.163,96.697,0.827,57.507,3.013,2.139,2.194,1.899,1.613,3.288,1.036,3.175,0.014,1.027,-7.029,13.879,4.974,0.081,2.523],[6136.5,0.232,8.293,1989.92,460.988,44.561,96.848,95.94,0.83,59.156,2.904,2.169,2.289,1.98,1.734,3.363,1.106,3.045,0.017,-0.525,3.07,14.248,4.787,0.085,2.513],[6137.0,0.228,8.275,1988.056,444.553,43.509,94.328,93.397,0.828,59.683,2.761,2.249,2.461,2.141,1.779,3.467,1.198,2.9,0.019,-3.309,5.964,14.651,4.547,0.078,2.508],[6137.5,0.371,8.274,1994.947,425.426,41.411,91.453,88.017,0.837,59.51,2.719,2.351,2.694,2.492,1.891,3.453,1.308,2.651,0.02,-7.422,2.124,14.745,4.197,0.081,2.505],[6138.0,0.3,8.289,2001.539,414.86,39.329,88.998,79.32,0.864,57.908,2.792,2.41,2.917,3.066,2.371,3.213,1.398,2.306,0.02,-12.471,-0.867,14.163,3.769,0.095,2.508],[6138.5,0.316,8.294,2001.381,420.887,36.442,87.365,69.834,0.881,53.897,2.825,2.376,2.964,3.906,3.909,2.897,1.434,2.021,0.017,-18.289,-2.561,13.025,3.371,0.1,2.517],[6139.0,0.33,8.275,2000.45,447.331,33.1,86.524,61.49,0.834,45.967,2.683,2.235,2.706,4.955,6.856,2.484,1.275,1.975,0.015,-25.772,-7.261,10.682,3.096,0.095,2.519],[6139.5,0.367,8.249,2003.465,495.05,30.275,86.294,53.778,0.705,35.424,2.461,2.02,2.254,4.945,4.98,2.075,0.94,2.272,0.018,-35.214,-14.357,7.58,2.985,0.094,2.503],[6140.0,0.291,8.226,2001.773,561.008,28.415,86.696,45.088,0.599,27.429,2.252,1.783,1.811,4.04,3.711,1.905,0.741,2.578,0.024,-45.134,-21.181,5.834,2.919,0.102,2.471],[6140.5,0.205,8.227,1991.491,640.184,27.545,87.806,35.508,0.51,22.023,1.92,1.562,1.482,3.068,4.842,1.875,0.686,2.757,0.035,-53.988,-26.372,4.888,2.656,0.102,2.424],[6141.0,0.135,8.261,1981.302,727.46,27.376,89.459,28.004,0.433,20.257,1.561,1.375,1.268,2.344,3.103,1.998,1.167,2.153,0.056,-61.073,-28.971,4.403,1.779,0.091,2.352],[6141.5,0.136,8.28,1979.608,819.313,27.661,91.323,24.543,0.423,20.449,1.422,1.221,1.133,2.002,1.917,2.172,1.816,1.29,0.082,-65.792,-30.337,4.694,1.02,0.082,2.273],[6142.0,0.082,8.28,1981.712,913.883,27.971,93.042,24.076,0.427,19.634,1.424,1.094,1.041,2.108,2.412,2.219,1.965,1.129,0.1,-68.197,-33.131,4.838,0.914,0.073,2.217],[6142.5,0.111,8.274,1980.49,1004.469,28.206,94.304,25.619,0.429,20.543,1.436,0.996,0.972,2.475,2.249,2.273,2.063,1.105,0.102,-69.37,-36.085,4.971,0.896,0.057,2.193],[6143.0,0.123,8.278,1982.226,1078.087,28.331,94.994,29.154,0.416,22.65,1.431,0.928,0.922,2.513,1.688,2.531,2.077,1.226,0.096,-69.978,-38.909,5.348,0.96,0.044,2.189],[6143.5,0.13,8.235,1992.186,1128.249,27.857,95.251,33.901,0.404,26.311,1.46,0.886,0.886,2.036,1.189,2.974,2.288,1.327,0.09,-69.849,-35.91,6.153,1.014,0.043,2.193],[6144.0,0.17,8.15,2003.677,1158.969,27.504,95.324,37.232,0.428,31.936,1.572,0.863,0.859,1.575,0.898,3.416,3.24,1.107,0.087,-69.302,-21.498,7.505,0.889,0.043,2.196],[6144.5,0.248,8.137,2016.223,1177.906,27.696,95.363,36.668,0.449,35.451,1.677,0.849,0.837,1.492,1.244,3.652,3.972,0.92,0.087,-69.327,-7.936,8.37,0.783,0.036,2.193],[6145.0,0.358,8.14,2040.867,1188.853,28.163,95.341,32.336,0.433,34.236,1.597,0.841,0.822,1.678,1.763,3.786,4.071,0.929,0.09,-70.052,-10.243,8.328,0.76,0.032,2.184],[6145.5,0.397,8.129,2067.571,1192.741,28.183,95.084,27.406,0.348,29.36,1.416,0.838,0.817,1.737,1.675,4.324,3.732,1.215,0.095,-70.644,-20.678,7.476,0.751,0.041,2.171],[6146.0,0.415,8.131,2066.858,1191.259,27.756,94.512,24.17,0.256,22.967,1.366,0.839,0.823,1.493,1.365,4.556,2.718,1.793,0.101,-70.609,-25.516,5.953,0.843,0.056,2.157],[6146.5,0.4,8.134,2041.105,1187.949,27.96,93.788,22.602,0.256,17.733,1.459,0.842,0.837,1.263,1.165,3.599,1.719,2.086,0.106,-70.2,-24.034,4.553,1.01,0.067,2.146],[6147.0,0.381,8.129,2022.548,1183.624,27.988,93.155,21.946,0.311,14.314,1.563,0.845,0.853,1.132,1.024,2.185,1.089,1.973,0.106,-69.643,-19.479,3.296,1.154,0.075,2.147],[6147.5,0.377,8.127,2019.257,1177.986,28.336,92.756,22.189,0.35,12.793,1.611,0.849,0.863,1.171,1.092,1.438,0.776,1.851,0.102,-69.096,-19.552,2.566,1.228,0.076,2.157],[6148.0,0.358,8.133,2013.025,1172.264,27.672,92.584,22.752,0.383,13.281,1.613,0.853,0.864,1.427,1.381,1.263,0.77,1.654,0.097,-68.894,-28.671,2.458,1.189,0.071,2.169],[6148.5,0.36,8.133,2000.714,1166.705,27.393,92.611,22.876,0.441,14.931,1.582,0.857,0.857,1.749,2.0,1.15,0.927,1.278,0.093,-69.187,-33.26,2.581,1.049,0.067,2.179],[6149.0,0.343,8.125,1989.192,1160.61,27.885,92.831,23.022,0.494,16.462,1.579,0.862,0.847,1.996,1.792,1.107,1.135,0.989,0.092,-69.715,-21.802,2.794,0.92,0.066,2.183],[6149.5,0.329,8.128,1985.745,1159.231,28.32,93.219,23.644,0.532,17.654,1.64,0.863,0.841,1.916,1.876,1.128,1.376,0.832,0.094,-69.977,-5.612,3.064,0.834,0.063,2.179],[6150.0,0.295,8.139,1995.512,1167.979,28.602,93.663,24.487,0.568,19.007,1.691,0.856,0.841,1.339,1.89,1.16,1.676,0.699,0.097,-69.239,1.197,3.365,0.75,0.063,2.17],[6150.5,0.386,8.145,2009.115,1180.101,28.899,94.001,25.006,0.605,20.166,1.694,0.847,0.841,0.937,1.043,1.135,1.804,0.63,0.1,-67.456,1.905,3.5,0.721,0.066,2.161],[6151.0,0.411,8.148,2011.269,1185.231,29.207,94.125,25.17,0.63,20.988,1.705,0.844,0.839,1.043,1.098,1.108,1.8,0.616,0.103,-66.028,2.359,3.562,0.735,0.063,2.153],[6151.5,0.419,8.136,2002.929,1180.994,30.284,94.069,25.137,0.64,21.729,1.766,0.847,0.836,1.845,1.951,1.152,1.957,0.591,0.107,-65.898,1.829,3.762,0.716,0.057,2.146],[6152.0,0.387,8.136,1996.442,1165.689,32.159,94.033,25.107,0.648,22.511,1.842,0.858,0.845,2.601,2.094,1.234,2.325,0.536,0.11,-66.163,0.413,4.082,0.657,0.054,2.142],[6152.5,0.329,8.147,1998.813,1136.06,33.233,94.183,24.916,0.668,23.319,1.91,0.88,0.87,1.902,1.527,1.311,2.927,0.456,0.109,-65.305,-0.913,4.478,0.576,0.055,2.15],[6153.0,0.308,8.143,2010.266,1094.564,33.596,94.496,24.45,0.706,23.983,1.923,0.914,0.914,1.147,0.93,1.358,3.731,0.371,0.103,-62.694,-3.644,4.898,0.494,0.055,2.172],[6153.5,0.272,8.137,2030.206,1041.997,33.983,94.817,24.785,0.745,24.763,1.873,0.96,0.971,0.886,0.783,1.379,4.563,0.306,0.095,-59.112,-7.998,5.251,0.43,0.056,2.199],[6154.0,0.272,8.126,2050.669,976.79,34.18,95.072,27.422,0.789,26.538,1.819,1.024,1.039,1.025,1.105,1.363,4.985,0.274,0.093,-56.214,-6.306,5.483,0.409,0.058,2.217],[6154.5,0.244,8.129,2055.106,905.169,32.948,95.383,32.829,0.871,29.786,1.866,1.105,1.12,1.553,1.57,1.285,4.353,0.305,0.101,-54.8,2.49,5.697,0.509,0.058,2.216],[6155.0,0.278,8.136,2039.44,837.822,31.178,95.922,40.503,0.938,34.152,1.982,1.194,1.224,2.218,1.803,1.333,3.101,0.467,0.118,-53.914,8.242,6.379,0.832,0.06,2.2],[6155.5,0.299,8.112,2023.204,777.39,32.171,96.661,49.512,0.932,38.123,2.101,1.286,1.347,2.588,1.912,1.639,2.089,0.854,0.133,-51.686,6.192,7.779,1.497,0.071,2.19],[6156.0,0.258,8.088,2015.302,723.062,34.24,97.3,59.329,0.908,42.244,2.188,1.383,1.476,2.841,2.29,1.937,1.558,1.267,0.128,-47.03,5.461,8.976,2.176,0.083,2.213],[6156.5,0.354,8.091,2010.689,673.582,34.83,97.551,70.072,0.908,47.643,2.255,1.485,1.602,3.031,2.683,2.262,1.425,1.59,0.102,-41.162,7.604,10.499,2.737,0.093,2.27],[6157.0,0.385,8.115,2012.995,623.76,36.366,97.42,79.349,0.954,53.482,2.325,1.603,1.734,2.974,2.35,2.528,1.454,1.738,0.077,-36.639,5.792,12.324,3.14,0.099,2.327],[6157.5,0.346,8.117,2021.626,574.297,37.087,97.17,84.626,1.001,57.405,2.367,1.741,1.876,2.697,2.405,2.523,1.47,1.716,0.069,-34.218,5.793,12.894,3.255,0.098,2.356],[6158.0,0.304,8.119,2025.178,537.11,37.225,97.007,86.254,1.028,58.75,2.37,1.862,2.002,2.543,2.311,2.455,1.465,1.676,0.068,-33.455,13.024,12.88,3.262,0.097,2.367],[6158.5,0.267,8.128,2021.652,519.767,36.974,96.892,85.057,1.078,59.263,2.334,1.924,2.089,2.701,2.309,2.371,1.537,1.547,0.063,-34.758,17.186,13.048,3.155,0.092,2.381],[6159.0,0.209,8.14,2017.729,518.542,36.035,96.597,81.864,1.123,59.615,2.333,1.928,2.133,3.076,3.0,2.315,1.683,1.382,0.058,-37.74,14.96,13.275,2.938,0.084,2.392],[6159.5,0.209,8.116,2013.947,528.904,35.212,95.902,78.979,1.113,59.802,2.365,1.891,2.124,3.201,3.433,2.369,1.823,1.301,0.053,-40.398,15.478,13.451,2.742,0.082,2.398],[6160.0,0.207,8.098,2011.19,547.626,34.691,94.831,77.373,1.072,60.007,2.363,1.826,2.044,2.956,2.945,2.474,1.94,1.276,0.048,-42.142,21.163,13.531,2.593,0.093,2.406],[6160.5,0.141,8.098,2010.816,571.337,33.933,93.618,75.07,1.037,59.983,2.26,1.75,1.894,2.725,2.275,2.596,2.124,1.225,0.044,-44.392,24.29,13.739,2.409,0.103,2.411],[6161.0,0.118,8.11,2012.172,600.139,33.098,92.464,68.348,0.93,56.734,2.049,1.666,1.723,2.655,2.075,2.897,2.215,1.316,0.046,-48.735,21.643,13.581,2.276,0.091,2.402],[6161.5,0.106,8.112,2015.445,636.542,31.737,91.468,54.976,0.684,46.832,1.75,1.571,1.581,2.717,2.161,3.422,1.883,1.932,0.055,-55.447,17.206,11.619,2.304,0.063,2.371],[6162.0,0.089,8.098,2022.607,679.57,29.649,90.639,38.982,0.434,33.383,1.532,1.472,1.471,2.906,2.281,3.89,1.383,2.923,0.072,-62.541,17.624,8.461,2.258,0.044,2.325],[6162.5,0.089,8.095,2033.042,721.312,27.618,89.82,27.748,0.319,22.79,1.46,1.386,1.381,3.075,2.452,3.387,1.109,3.03,0.091,-67.459,18.406,5.542,1.822,0.045,2.277],[6163.0,0.089,8.092,2041.469,752.11,26.664,88.904,23.117,0.287,16.556,1.435,1.33,1.317,2.987,2.475,2.542,0.986,2.569,0.108,-69.884,9.525,3.751,1.4,0.052,2.239],[6163.5,0.125,8.084,2039.504,768.427,26.842,88.096,21.616,0.287,13.736,1.408,1.301,1.288,2.727,2.355,2.229,0.94,2.37,0.122,-70.943,-1.916,3.254,1.284,0.056,2.211],[6164.0,0.174,8.085,2030.898,774.381,27.48,87.734,20.565,0.311,12.993,1.415,1.291,1.294,2.496,2.209,1.9,0.905,2.095,0.134,-71.336,-1.561,2.993,1.226,0.06,2.189],[6164.5,0.223,8.103,2035.884,776.748,27.695,87.951,20.147,0.354,13.377,1.427,1.287,1.313,2.65,1.938,1.527,0.898,1.707,0.142,-71.294,5.248,2.725,1.128,0.074,2.176],[6165.0,0.236,8.098,2050.478,781.893,27.102,88.507,20.515,0.413,14.348,1.416,1.279,1.319,3.267,2.826,1.234,0.955,1.301,0.142,-71.045,1.276,2.578,1.004,0.094,2.175],[6165.5,0.249,8.109,2045.278,792.655,26.121,88.978,21.301,0.468,14.944,1.455,1.262,1.302,3.837,4.153,1.055,1.012,1.046,0.134,-70.702,-15.672,2.507,0.921,0.096,2.185],[6166.0,0.209,8.122,2022.722,807.975,25.683,89.206,22.37,0.513,15.053,1.536,1.238,1.267,3.865,4.307,0.944,1.035,0.912,0.123,-70.332,-34.425,2.468,0.885,0.083,2.199],[6166.5,0.188,8.129,2013.064,828.737,25.491,89.473,23.336,0.543,15.454,1.539,1.207,1.219,3.545,3.709,0.889,1.058,0.841,0.115,-70.221,-47.213,2.466,0.865,0.076,2.208],[6167.0,0.188,8.132,2015.01,859.616,25.558,90.064,23.858,0.551,16.104,1.481,1.163,1.158,3.28,3.32,0.866,0.979,0.934,0.111,-70.608,-51.159,2.436,0.969,0.078,2.208],[6167.5,0.267,8.138,2010.389,899.228,26.333,90.851,24.757,0.535,15.641,1.513,1.112,1.094,3.111,2.983,0.827,0.673,1.387,0.107,-71.401,-46.338,2.262,1.392,0.078,2.206],[6168.0,0.283,8.145,2003.907,934.29,26.48,91.185,25.912,0.516,15.282,1.637,1.07,1.042,3.035,3.128,0.812,0.456,1.786,0.102,-72.337,-38.018,2.141,1.746,0.072,2.207],[6168.5,0.295,8.142,2000.835,948.649,26.339,90.303,26.234,0.505,15.565,1.749,1.054,1.018,2.939,3.344,0.841,0.45,1.868,0.1,-73.114,-35.077,2.17,1.789,0.062,2.209],[6169.0,0.317,8.135,1997.356,933.483,26.598,88.072,26.087,0.478,15.347,1.76,1.071,1.036,2.735,3.145,0.97,0.505,1.917,0.102,-73.381,-39.673,2.349,1.733,0.054,2.208],[6169.5,0.325,8.139,1991.055,892.674,27.365,85.227,26.078,0.445,14.96,1.734,1.12,1.109,2.618,2.64,1.167,0.599,1.948,0.107,-72.907,-45.639,2.644,1.643,0.05,2.206],[6170.0,0.329,8.139,1984.574,838.835,25.937,82.789,26.162,0.427,14.825,1.801,1.192,1.239,2.8,1.988,1.354,0.71,1.911,0.109,-71.787,-48.506,2.944,1.546,0.059,2.216],[6170.5,0.356,8.136,1984.156,781.517,24.239,81.299,27.163,0.417,15.258,2.041,1.28,1.421,3.739,13.1,1.481,0.789,1.878,0.095,-70.546,-47.368,3.151,1.483,0.081,2.256],[6171.0,0.352,8.138,1997.539,723.973,23.124,80.636,31.124,0.421,16.827,2.385,1.381,1.641,6.287,29.147,1.628,0.836,1.941,0.064,-69.696,-36.473,3.503,1.549,0.108,2.333],[6171.5,0.381,8.15,2016.16,673.678,21.113,80.482,39.171,0.425,20.437,2.681,1.484,1.84,8.384,26.111,2.006,0.936,2.13,0.036,-68.573,-16.235,4.349,1.714,0.138,2.416],[6172.0,0.195,8.152,2023.666,643.105,18.813,80.974,48.736,0.485,27.348,2.827,1.555,1.922,6.804,12.925,2.396,1.177,2.075,0.024,-66.114,-3.963,5.926,1.858,0.151,2.459],[6172.5,0.095,8.139,2017.779,640.138,18.232,82.574,54.146,0.641,35.568,2.831,1.562,1.834,4.515,4.677,2.312,1.439,1.634,0.026,-63.296,-6.354,7.406,1.909,0.125,2.455],[6173.0,0.089,8.14,2009.135,669.098,21.199,85.324,52.65,0.723,39.24,2.692,1.495,1.618,3.523,2.705,2.11,1.51,1.402,0.032,-62.315,-7.916,7.782,1.914,0.073,2.43],[6173.5,0.106,8.141,2005.409,729.814,27.005,88.541,46.84,0.648,35.755,2.505,1.37,1.375,3.191,2.726,2.169,1.34,1.655,0.038,-63.618,-2.347,7.158,1.995,0.037,2.401],[6174.0,0.107,8.141,2004.45,811.448,30.881,91.245,39.174,0.552,28.966,2.261,1.232,1.177,2.649,2.922,2.205,1.099,2.036,0.048,-65.735,1.488,6.215,2.105,0.026,2.358],[6174.5,0.112,8.165,1999.338,899.867,31.342,92.911,30.966,0.443,21.754,1.931,1.111,1.036,1.924,2.065,2.151,0.863,2.585,0.063,-67.181,3.755,4.89,2.094,0.032,2.302],[6175.0,0.095,8.189,1991.26,992.07,30.175,93.742,24.65,0.345,15.899,1.656,1.008,0.937,1.544,1.167,1.879,0.664,2.829,0.076,-67.663,14.027,3.338,1.844,0.05,2.253],[6175.5,0.085,8.183,1986.269,1086.458,28.353,94.225,21.84,0.333,13.645,1.499,0.92,0.868,1.574,1.051,1.618,0.673,2.431,0.081,-67.903,29.242,2.752,1.529,0.061,2.224],[6176.0,0.085,8.174,1988.435,1166.096,26.887,94.561,21.561,0.353,13.634,1.422,0.858,0.823,1.803,1.151,1.593,0.791,2.032,0.079,-67.918,37.4,2.871,1.355,0.05,2.213],[6176.5,0.084,8.178,1999.292,1208.738,26.599,94.771,22.308,0.378,14.272,1.438,0.827,0.802,1.829,1.131,1.527,0.883,1.738,0.08,-66.795,38.571,2.938,1.238,0.034,2.204],[6177.0,0.094,8.185,2014.396,1204.429,26.862,95.149,23.048,0.413,14.925,1.553,0.83,0.802,1.662,1.061,1.405,0.95,1.487,0.085,-64.255,37.63,2.953,1.156,0.028,2.191],[6177.5,0.112,8.178,2022.812,1156.087,27.0,96.07,23.489,0.446,15.35,1.648,0.865,0.826,1.537,1.068,1.328,0.999,1.33,0.094,-61.009,31.951,3.021,1.122,0.033,2.18],[6178.0,0.124,8.179,2020.128,1075.713,26.93,97.531,25.249,0.469,15.945,1.652,0.93,0.886,1.329,1.104,1.269,0.989,1.285,0.103,-57.872,23.056,3.041,1.143,0.043,2.174],[6178.5,0.12,8.194,2011.908,980.547,26.992,99.092,31.786,0.525,17.674,1.669,1.02,0.993,1.007,0.94,1.193,0.877,1.374,0.113,-54.744,12.483,3.188,1.378,0.054,2.176],[6179.0,0.14,8.192,2004.826,881.118,27.892,100.328,45.129,0.652,23.313,1.839,1.135,1.156,0.857,0.82,1.258,0.883,1.438,0.116,-49.196,-2.135,4.251,1.77,0.065,2.192],[6179.5,0.187,8.184,1998.865,781.037,29.605,101.366,61.653,0.889,36.229,2.117,1.28,1.372,1.044,1.204,1.541,1.415,1.194,0.112,-39.94,-12.194,7.193,1.913,0.077,2.224],[6180.0,0.322,8.181,1993.272,685.067,31.803,102.793,74.087,1.171,51.979,2.27,1.46,1.63,1.724,1.798,1.808,2.222,0.855,0.107,-31.579,-10.214,10.926,1.842,0.088,2.258],[6180.5,0.308,8.189,1992.204,598.294,35.555,104.839,79.166,1.269,61.171,2.269,1.671,1.929,2.604,2.967,1.961,2.527,0.778,0.107,-28.349,-7.005,12.7,1.867,0.095,2.281],[6181.0,0.2,8.209,1998.812,529.621,38.695,106.809,79.132,1.213,63.302,2.328,1.888,2.25,2.874,2.785,2.053,2.405,0.855,0.107,-28.259,-4.736,12.709,1.962,0.095,2.302],[6181.5,0.303,8.235,2005.461,495.876,39.093,107.709,75.536,1.16,62.418,2.267,2.017,2.477,2.972,1.976,2.164,2.298,0.944,0.106,-29.618,6.872,12.795,2.069,0.085,2.314],[6182.0,0.256,8.242,2006.551,505.636,39.528,107.135,68.185,1.067,57.431,1.885,1.978,2.417,3.635,2.535,2.362,2.149,1.119,0.139,-34.786,17.277,12.698,2.2,0.047,2.266],[6182.5,0.23,8.225,2007.565,547.494,41.707,105.502,59.066,0.828,47.071,1.772,1.827,2.09,5.435,5.786,2.577,1.83,1.448,0.239,-43.529,12.909,10.712,2.161,0.004,2.137],[6183.0,0.208,8.191,2011.601,604.218,43.28,103.554,50.445,0.65,37.097,1.945,1.655,1.73,6.476,7.297,2.466,1.58,1.557,0.353,-52.34,5.335,8.208,1.921,0.012,2.006],[6183.5,0.125,8.169,2012.849,665.893,42.251,101.584,42.588,0.64,31.209,2.048,1.502,1.464,4.3,4.449,2.211,1.524,1.45,0.378,-59.411,3.849,7.236,1.76,0.042,1.955],[6184.0,0.107,8.178,2007.461,729.786,38.573,99.341,35.214,0.605,26.183,1.968,1.37,1.291,2.797,2.233,1.859,1.315,1.408,0.305,-64.844,-0.147,5.814,1.616,0.047,1.993],[6184.5,0.106,8.186,2000.943,794.935,34.12,96.641,29.413,0.543,21.671,1.839,1.258,1.179,2.141,1.856,1.427,1.07,1.33,0.212,-68.658,-6.951,3.995,1.372,0.033,2.073],[6185.0,0.103,8.183,1999.698,859.681,30.565,93.788,25.916,0.5,19.018,1.814,1.163,1.11,1.834,1.664,1.308,1.008,1.298,0.152,-70.828,-8.778,3.341,1.231,0.025,2.137],[6185.5,0.163,8.147,2004.654,922.734,28.691,91.387,24.261,0.471,17.599,1.799,1.084,1.065,1.803,1.849,1.359,0.997,1.365,0.125,-71.763,-9.791,3.265,1.216,0.033,2.167],[6186.0,0.164,8.133,2010.378,979.758,27.822,89.983,23.407,0.378,15.62,1.75,1.021,1.03,2.058,2.216,1.763,0.921,1.981,0.114,-72.016,-14.842,3.168,1.285,0.051,2.173],[6186.5,0.119,8.141,2011.078,1023.393,27.398,89.736,22.827,0.258,13.123,1.745,0.977,0.992,2.643,2.304,2.386,0.792,3.09,0.111,-72.04,-20.171,3.007,1.418,0.067,2.169],[6187.0,0.101,8.141,2009.221,1054.336,27.242,90.282,22.546,0.209,11.76,1.766,0.948,0.953,3.369,2.453,2.723,0.709,3.85,0.108,-72.075,-19.973,2.897,1.518,0.077,2.169],[6187.5,0.102,8.157,2009.333,1082.925,26.842,91.098,22.885,0.197,11.604,1.776,0.923,0.918,3.212,2.079,2.873,0.686,4.188,0.103,-71.956,-15.217,2.883,1.559,0.072,2.175],[6188.0,0.141,8.172,2009.206,1116.035,26.311,91.883,23.829,0.201,12.333,1.792,0.896,0.886,2.306,1.43,2.906,0.719,4.065,0.096,-71.558,-13.83,2.98,1.541,0.057,2.183],[6188.5,0.141,8.168,2008.573,1154.168,26.172,92.655,24.956,0.228,13.631,1.867,0.866,0.854,1.933,1.91,2.709,0.803,3.409,0.091,-71.085,-21.635,3.137,1.454,0.051,2.186],[6189.0,0.138,8.177,2010.654,1195.701,26.712,93.596,25.994,0.271,14.995,1.935,0.836,0.82,1.948,2.533,2.401,0.933,2.619,0.09,-70.766,-31.613,3.295,1.319,0.054,2.182],[6189.5,0.136,8.18,2016.651,1231.87,27.861,94.729,26.544,0.314,16.202,1.848,0.812,0.793,1.924,2.046,2.183,1.121,1.994,0.09,-70.543,-32.403,3.483,1.167,0.051,2.174],[6190.0,0.142,8.191,2019.942,1256.193,28.825,95.805,25.895,0.374,17.376,1.684,0.796,0.78,1.63,1.617,1.954,1.419,1.438,0.092,-70.092,-26.603,3.688,0.986,0.047,2.165],[6190.5,0.167,8.183,2016.886,1273.797,29.218,96.571,24.767,0.44,18.169,1.63,0.785,0.782,1.175,1.328,1.709,1.714,1.008,0.095,-68.996,-22.722,3.821,0.832,0.046,2.156],[6191.0,0.22,8.151,2012.093,1287.122,29.547,97.102,24.828,0.473,18.401,1.66,0.777,0.791,0.906,0.946,1.583,1.78,0.89,0.095,-67.329,-22.354,3.823,0.798,0.048,2.152],[6191.5,0.289,8.146,2012.403,1283.533,30.323,97.593,25.971,0.48,18.552,1.682,0.779,0.8,1.055,1.245,1.515,1.612,0.948,0.096,-65.675,-22.284,3.714,0.862,0.049,2.152],[6192.0,0.337,8.165,2016.087,1249.617,31.228,97.949,27.117,0.483,18.362,1.753,0.8,0.81,1.726,1.83,1.444,1.323,1.108,0.098,-64.317,-22.759,3.561,1.015,0.051,2.152],[6192.5,0.415,8.17,2014.411,1182.47,31.966,97.84,27.773,0.493,18.129,1.901,0.846,0.839,2.069,2.056,1.379,1.117,1.238,0.105,-62.777,-21.306,3.468,1.157,0.05,2.148],[6193.0,0.455,8.167,2010.207,1085.98,32.658,97.101,28.672,0.515,18.449,2.071,0.921,0.909,1.529,1.665,1.31,1.026,1.278,0.115,-60.537,-14.265,3.444,1.248,0.05,2.148],[6193.5,0.479,8.181,2018.0,976.372,32.997,96.047,31.183,0.559,20.06,2.143,1.024,1.026,1.243,1.151,1.233,0.96,1.285,0.121,-57.606,-7.434,3.506,1.356,0.053,2.161],[6194.0,0.471,8.174,2040.336,873.733,33.078,95.153,37.902,0.701,24.86,2.133,1.145,1.185,1.495,1.942,1.181,1.016,1.174,0.117,-53.72,-10.192,4.221,1.53,0.059,2.193],[6194.5,0.5,8.156,2054.812,778.277,33.242,94.304,49.402,0.862,31.36,2.234,1.285,1.388,2.204,3.047,1.315,1.101,1.196,0.102,-48.588,-17.532,5.831,1.962,0.071,2.243],[6195.0,0.556,8.157,2048.394,678.059,33.238,92.796,60.899,0.872,36.294,2.564,1.475,1.653,2.883,3.257,1.648,1.06,1.574,0.084,-43.517,-16.586,7.282,2.571,0.083,2.301],[6195.5,0.493,8.172,2035.195,581.668,32.765,90.304,66.867,0.777,39.007,2.846,1.719,1.991,3.228,3.554,2.104,0.97,2.194,0.067,-40.649,-6.111,8.24,3.175,0.091,2.357],[6196.0,0.428,8.17,2032.784,513.827,32.761,87.53,67.698,0.709,40.809,2.748,1.946,2.345,3.387,3.524,2.441,0.931,2.621,0.056,-40.208,1.482,8.818,3.514,0.1,2.397],[6196.5,0.306,8.17,2036.349,489.237,33.001,85.629,65.217,0.696,41.313,2.57,2.044,2.522,3.771,3.587,2.515,0.937,2.686,0.047,-41.069,-1.223,8.945,3.544,0.11,2.421],[6197.0,0.281,8.182,2031.457,506.976,31.302,85.196,57.777,0.688,39.248,2.57,1.972,2.36,5.154,3.169,2.523,0.984,2.574,0.037,-42.716,-9.0,8.871,3.36,0.115,2.442],[6197.5,0.177,8.174,2018.817,551.621,27.241,85.83,46.357,0.645,34.057,2.567,1.813,1.973,8.243,37.146,2.321,0.996,2.329,0.026,-44.798,-17.291,7.695,2.858,0.107,2.467],[6198.0,0.151,8.179,2009.352,595.644,24.312,86.784,36.307,0.596,27.44,2.275,1.679,1.619,9.538,57.735,1.906,0.946,2.012,0.022,-46.618,-25.667,5.824,2.278,0.078,2.474],[6198.5,0.154,8.183,2014.348,618.716,23.563,87.927,31.092,0.574,22.436,1.894,1.616,1.427,4.089,14.759,1.642,0.911,1.802,0.033,-47.79,-33.018,4.818,1.961,0.041,2.435],[6199.0,0.085,8.162,2036.892,623.526,21.644,89.472,31.368,0.562,20.216,1.731,1.604,1.401,1.632,0.478,1.584,0.891,1.777,0.063,-47.689,-39.675,4.544,1.892,0.026,2.358],[6199.5,0.084,8.147,2057.788,621.401,22.319,91.054,39.067,0.547,21.581,1.77,1.609,1.492,1.092,0.708,1.666,0.862,1.967,0.095,-44.514,-46.246,4.618,2.011,0.038,2.296],[6200.0,0.087,8.149,2056.082,611.523,25.877,91.802,55.806,0.492,26.238,1.974,1.635,1.656,1.202,0.884,2.412,0.813,3.007,0.1,-37.501,-46.729,5.876,2.703,0.055,2.289],[6200.5,0.096,8.144,2043.358,584.39,29.129,91.293,74.837,0.457,32.865,2.29,1.711,1.887,1.765,1.861,3.415,0.78,4.412,0.079,-29.744,-40.783,7.948,3.799,0.063,2.333],[6201.0,0.102,8.169,2033.91,535.198,31.309,89.996,82.888,0.455,37.96,2.458,1.868,2.223,2.628,2.27,3.92,0.754,5.236,0.055,-25.41,-37.523,9.083,4.492,0.067,2.393],[6201.5,0.107,8.179,2019.966,470.795,32.767,88.66,75.182,0.443,38.212,2.374,2.124,2.666,3.952,2.609,4.01,0.735,5.456,0.04,-25.313,-38.629,9.079,4.583,0.079,2.442],[6202.0,0.101,8.168,2010.151,408.95,32.836,87.458,60.977,0.508,35.111,2.254,2.445,3.111,6.581,7.014,3.426,0.757,4.58,0.03,-27.348,-34.441,8.436,4.157,0.088,2.479],[6202.5,0.206,8.176,2014.581,367.346,30.72,86.08,52.42,0.708,33.18,2.13,2.722,3.376,8.651,9.797,2.286,0.88,2.738,0.022,-29.9,-24.435,7.576,3.271,0.083,2.509],[6203.0,0.233,8.185,2018.168,355.686,27.589,84.716,53.779,0.845,33.25,2.04,2.811,3.318,6.621,6.873,1.648,0.964,1.728,0.021,-32.193,-17.57,7.108,2.758,0.078,2.515],[6203.5,0.172,8.182,2018.503,377.715,24.909,84.232,60.218,0.907,34.629,2.177,2.647,2.96,4.254,4.088,1.444,0.828,1.754,0.029,-34.576,-15.855,6.596,3.008,0.083,2.488],[6204.0,0.192,8.13,2031.213,431.11,24.069,85.137,62.186,0.988,36.362,2.391,2.32,2.456,3.514,2.623,1.196,0.673,1.775,0.043,-39.112,-17.237,5.985,3.313,0.088,2.444],[6204.5,0.238,8.096,2039.246,509.291,25.68,86.843,54.371,0.984,35.485,2.351,1.964,1.971,3.847,3.272,1.172,0.711,1.661,0.055,-46.748,-19.857,5.873,3.105,0.082,2.399],[6205.0,0.304,8.064,2025.598,602.499,27.233,88.236,41.225,0.788,30.134,2.108,1.66,1.596,3.925,4.167,1.426,0.893,1.593,0.068,-55.069,-21.244,5.488,2.365,0.07,2.35],[6205.5,0.278,7.994,2010.459,698.144,26.34,88.819,30.436,0.539,23.031,1.884,1.432,1.342,3.029,3.434,1.669,1.076,1.554,0.083,-61.057,-21.627,4.525,1.594,0.061,2.298],[6206.0,0.301,7.987,2008.245,788.468,25.644,89.071,24.753,0.46,18.348,1.775,1.268,1.176,2.17,2.365,1.598,1.13,1.414,0.096,-64.223,-21.299,3.75,1.232,0.058,2.253],[6206.5,0.323,7.983,2009.735,874.322,25.828,89.692,21.983,0.477,16.798,1.775,1.144,1.063,1.576,1.587,1.396,1.093,1.275,0.101,-65.493,-15.671,3.383,1.148,0.062,2.223],[6207.0,0.392,7.954,2003.232,955.594,26.5,90.808,21.072,0.514,17.112,1.786,1.046,0.987,1.173,1.249,1.176,1.063,1.105,0.1,-65.425,-3.523,3.069,1.071,0.079,2.208],[6207.5,0.387,7.946,1988.505,1028.769,27.052,92.101,22.183,0.553,18.016,1.768,0.972,0.937,0.92,1.14,1.028,1.152,0.904,0.097,-64.116,1.556,2.9,0.943,0.096,2.199],[6208.0,0.32,7.975,1980.347,1085.006,27.077,93.342,23.868,0.581,18.635,1.766,0.922,0.896,0.952,0.832,1.011,1.378,0.744,0.094,-62.035,-6.624,3.001,0.818,0.096,2.193],[6208.5,0.255,7.994,1988.148,1111.187,27.838,94.56,24.126,0.593,19.08,1.792,0.9,0.868,1.306,1.042,1.127,1.886,0.617,0.093,-60.281,-15.03,3.413,0.693,0.078,2.19],[6209.0,0.266,7.975,1995.601,1103.639,29.932,95.83,23.421,0.589,19.621,1.785,0.906,0.87,1.445,1.768,1.331,2.516,0.532,0.095,-59.117,-12.336,3.995,0.593,0.062,2.188],[6209.5,0.312,7.968,1992.003,1078.583,31.679,97.086,24.839,0.573,20.371,1.781,0.927,0.91,0.994,1.286,1.538,2.957,0.52,0.099,-58.08,-8.263,4.492,0.565,0.064,2.183],[6210.0,0.386,7.985,1987.17,1055.522,32.306,98.166,31.353,0.554,22.751,1.903,0.947,0.966,0.784,0.708,1.737,3.307,0.526,0.101,-56.394,-11.808,4.911,0.552,0.081,2.185],[6210.5,0.368,7.967,1987.147,1038.245,32.785,98.949,41.445,0.51,25.635,2.12,0.963,1.005,1.057,0.977,2.094,2.679,1.039,0.09,-53.434,-17.032,5.367,0.94,0.097,2.211],[6211.0,0.352,7.956,1986.322,1017.258,33.508,99.369,48.363,0.433,26.552,2.198,0.983,1.023,1.728,1.885,2.686,1.403,2.438,0.073,-51.365,-16.757,5.839,1.9,0.096,2.256],[6211.5,0.34,7.983,1982.737,987.071,33.769,99.365,46.878,0.394,26.79,1.984,1.013,1.042,1.986,2.172,3.009,0.916,3.316,0.065,-52.51,-16.072,6.051,2.467,0.083,2.281],[6212.0,0.293,7.986,1979.432,952.489,33.384,99.013,40.774,0.458,28.933,1.777,1.05,1.069,1.713,2.177,2.779,1.475,2.379,0.071,-54.487,-21.551,6.405,1.913,0.078,2.272],[6212.5,0.337,7.982,1980.275,921.79,32.903,98.634,37.385,0.552,30.042,1.708,1.085,1.094,1.62,2.148,2.448,2.335,1.171,0.08,-53.982,-27.961,6.853,1.169,0.089,2.258],[6213.0,0.36,7.983,1986.488,901.249,32.86,98.547,37.577,0.628,29.819,1.62,1.11,1.114,1.755,2.001,2.196,2.82,0.788,0.081,-52.751,-24.301,6.995,0.925,0.097,2.259],[6213.5,0.329,8.024,1995.831,893.36,32.789,98.731,38.272,0.698,31.198,1.513,1.119,1.137,1.743,2.251,1.976,3.14,0.634,0.079,-52.689,-9.268,7.03,0.834,0.094,2.265],[6214.0,0.297,8.089,2000.478,894.545,32.248,98.881,39.574,0.737,33.054,1.554,1.118,1.155,1.704,1.483,1.868,3.31,0.565,0.079,-51.636,6.159,7.02,0.788,0.086,2.266],[6214.5,0.301,8.11,1997.354,897.523,31.717,98.736,41.909,0.769,33.883,1.778,1.114,1.153,1.948,1.914,1.725,2.772,0.66,0.076,-48.358,9.764,6.759,0.967,0.084,2.271],[6215.0,0.289,8.118,1998.956,896.142,31.558,98.362,42.742,0.799,33.791,1.951,1.116,1.141,2.433,2.805,1.564,1.924,0.845,0.07,-46.176,2.782,6.375,1.282,0.085,2.286],[6215.5,0.316,8.132,2012.18,888.641,31.75,98.094,40.752,0.811,33.534,1.87,1.125,1.145,2.228,2.308,1.53,1.703,0.899,0.067,-47.343,-5.101,6.333,1.382,0.079,2.294],[6216.0,0.317,8.139,2024.958,879.297,31.736,98.133,39.547,0.814,33.109,1.757,1.137,1.168,1.664,1.318,1.581,1.908,0.834,0.072,-48.448,-10.554,6.569,1.285,0.074,2.284],[6216.5,0.315,8.13,2030.79,873.857,31.093,98.281,42.575,0.808,33.132,1.873,1.144,1.184,1.597,1.234,1.621,1.946,0.842,0.075,-46.156,-13.824,6.688,1.287,0.083,2.277],[6217.0,0.285,8.111,2036.686,869.503,30.934,98.263,46.71,0.807,34.179,2.054,1.15,1.188,1.986,2.063,1.635,1.751,0.944,0.071,-42.99,-12.197,6.737,1.441,0.09,2.289],[6217.5,0.255,8.096,2048.327,853.381,32.466,98.171,46.705,0.731,33.16,2.0,1.172,1.21,2.215,2.311,1.848,1.461,1.333,0.067,-42.264,-7.298,6.768,1.767,0.084,2.302],[6218.0,0.264,8.094,2062.168,818.082,34.628,98.33,44.274,0.595,29.502,1.884,1.222,1.271,2.089,1.692,2.187,1.133,1.993,0.071,-42.244,-6.567,6.564,2.185,0.077,2.298],[6218.5,0.31,8.1,2067.924,768.309,35.078,98.651,45.956,0.514,27.585,1.97,1.302,1.367,2.089,2.057,2.495,0.967,2.606,0.079,-40.387,-3.53,6.504,2.51,0.075,2.292],[6219.0,0.311,8.105,2056.815,710.954,35.275,98.278,54.057,0.47,28.713,2.147,1.407,1.493,2.292,2.496,2.921,0.903,3.243,0.079,-37.034,8.543,6.99,2.877,0.071,2.304],[6219.5,0.278,8.125,2039.657,648.22,36.563,96.218,64.876,0.522,32.716,2.331,1.543,1.682,2.322,2.533,2.968,0.883,3.358,0.071,-32.823,17.907,7.777,3.271,0.073,2.334],[6220.0,0.271,8.163,2033.683,586.739,37.782,92.517,72.981,0.671,37.559,2.539,1.704,1.962,2.177,2.171,2.436,0.828,2.926,0.062,-28.807,13.547,8.134,3.667,0.082,2.367],[6220.5,0.266,8.225,2038.637,539.816,38.083,88.521,73.413,0.768,40.042,2.539,1.852,2.294,2.327,1.911,2.068,0.753,2.747,0.053,-28.336,3.955,8.102,4.0,0.088,2.398],[6221.0,0.222,8.218,2040.164,517.89,36.363,85.668,64.803,0.742,38.153,2.302,1.931,2.527,3.23,2.128,2.098,0.727,2.902,0.043,-32.905,-3.554,7.873,4.017,0.098,2.426],[6221.5,0.188,8.108,2035.934,520.957,31.632,84.254,51.257,0.579,31.606,2.257,1.92,2.483,6.551,8.984,2.407,0.712,3.4,0.03,-39.527,-5.304,6.893,3.593,0.114,2.459],[6222.0,0.229,8.058,2031.267,541.382,25.001,83.614,39.602,0.393,24.839,2.441,1.847,2.21,14.397,35.046,2.922,0.742,3.93,0.017,-45.066,1.055,5.732,2.899,0.113,2.501],[6222.5,0.27,8.091,2025.105,569.411,18.779,83.317,33.372,0.308,20.879,2.397,1.756,1.912,12.96,37.418,3.233,0.784,4.12,0.012,-49.392,9.537,5.067,2.401,0.085,2.521],[6223.0,0.135,8.085,2022.976,598.208,15.287,83.792,32.252,0.299,19.13,2.035,1.672,1.689,6.258,10.462,3.131,0.787,3.978,0.021,-53.182,8.599,4.772,2.25,0.055,2.477],[6223.5,0.094,8.043,2034.715,629.368,16.361,85.698,33.604,0.318,19.071,1.808,1.589,1.542,3.371,1.244,2.898,0.799,3.63,0.051,-56.086,1.25,4.703,2.184,0.042,2.383],[6224.0,0.085,8.014,2049.985,668.062,21.764,88.884,35.384,0.337,19.417,1.766,1.497,1.454,2.702,1.877,2.693,0.806,3.343,0.085,-57.456,1.53,4.623,2.129,0.037,2.303],[6224.5,0.117,8.026,2052.401,706.449,25.422,92.196,36.826,0.351,19.466,1.815,1.416,1.413,2.76,1.777,2.507,0.768,3.264,0.098,-57.239,11.195,4.495,2.174,0.036,2.268],[6225.0,0.126,8.024,2043.806,725.057,26.866,94.33,37.939,0.371,19.573,1.9,1.379,1.413,2.807,1.936,2.334,0.711,3.279,0.097,-55.616,15.662,4.407,2.303,0.046,2.266],[6225.5,0.128,8.017,2036.445,713.171,27.41,94.825,39.158,0.394,20.185,1.992,1.402,1.449,2.767,2.085,2.198,0.68,3.232,0.095,-52.679,6.2,4.415,2.41,0.055,2.273],[6226.0,0.118,8.039,2035.199,676.638,28.462,94.221,40.587,0.437,21.415,2.046,1.478,1.516,2.837,2.072,2.053,0.675,3.043,0.096,-48.734,-8.483,4.559,2.506,0.055,2.279],[6226.5,0.12,8.043,2038.274,631.359,29.53,93.559,41.169,0.524,23.398,2.029,1.584,1.612,3.143,2.182,1.855,0.708,2.638,0.101,-44.466,-14.996,4.929,2.584,0.055,2.283],[6227.0,0.231,8.033,2042.106,592.831,29.793,93.914,41.045,0.602,25.251,2.018,1.687,1.736,3.477,2.188,1.746,0.771,2.273,0.109,-40.637,-12.044,5.363,2.582,0.069,2.28],[6227.5,0.209,8.027,2047.58,564.94,29.572,95.832,42.738,0.66,27.443,2.078,1.77,1.87,3.534,2.256,1.703,0.846,2.022,0.12,-37.4,-10.463,5.734,2.518,0.08,2.273],[6228.0,0.192,8.005,2051.414,542.317,29.229,98.715,50.07,0.765,33.853,2.104,1.844,1.977,3.64,2.463,1.755,1.01,1.761,0.129,-33.269,-13.646,6.924,2.525,0.08,2.266],[6228.5,0.148,8.016,2043.533,520.031,28.95,101.29,64.902,0.876,45.7,2.066,1.923,2.057,3.735,2.9,2.272,1.266,1.792,0.129,-27.082,-15.181,10.27,2.97,0.086,2.273],[6229.0,0.227,8.054,2026.1,494.92,30.019,102.92,84.41,0.839,55.084,2.028,2.021,2.165,3.257,2.726,3.217,1.232,2.786,0.118,-20.582,-13.482,13.43,4.21,0.104,2.298],[6229.5,0.211,8.055,2010.016,467.189,33.407,103.972,102.003,0.67,57.525,2.06,2.14,2.35,2.441,1.831,4.314,0.95,4.751,0.107,-16.382,-15.109,14.327,5.731,0.113,2.322],[6230.0,0.204,7.99,2005.32,443.521,38.089,104.944,107.562,0.564,57.582,2.17,2.255,2.569,1.924,1.274,5.013,0.821,6.117,0.109,-15.378,-19.902,14.409,6.517,0.106,2.327],[6230.5,0.155,7.94,2015.701,432.056,41.52,105.458,94.67,0.558,55.047,2.169,2.315,2.709,2.014,1.819,4.932,0.812,6.08,0.121,-18.511,-24.621,14.039,6.412,0.099,2.315],[6231.0,0.135,7.92,2037.552,435.175,42.87,104.388,70.542,0.522,46.766,1.894,2.298,2.71,2.941,3.917,4.563,0.813,5.627,0.14,-26.389,-29.878,12.202,5.58,0.089,2.29],[6231.5,0.101,7.923,2055.474,448.82,41.696,101.282,50.603,0.411,35.156,1.579,2.228,2.607,4.935,5.089,4.268,0.782,5.476,0.188,-36.435,-35.158,9.021,4.261,0.061,2.232],[6232.0,0.152,7.94,2058.295,467.5,38.037,97.258,40.922,0.32,25.657,1.576,2.139,2.45,6.505,5.026,3.868,0.721,5.365,0.266,-44.843,-33.59,6.37,3.248,0.055,2.149],[6232.5,0.22,7.948,2049.162,488.328,33.958,93.904,37.953,0.296,20.504,1.755,2.048,2.267,5.365,2.959,3.39,0.65,5.219,0.304,-50.084,-26.452,5.134,2.931,0.085,2.106],[6233.0,0.184,7.944,2037.705,511.878,30.968,91.734,37.389,0.292,18.088,1.898,1.954,2.077,3.979,2.793,3.032,0.577,5.262,0.245,-52.806,-22.74,4.525,2.91,0.105,2.147],[6233.5,0.153,7.931,2026.865,539.229,27.115,90.23,36.675,0.288,16.511,1.873,1.854,1.903,3.251,3.303,2.719,0.518,5.249,0.148,-54.39,-21.053,3.991,2.86,0.096,2.242],[6234.0,0.129,7.929,2014.841,569.579,24.838,88.855,34.64,0.31,14.53,1.727,1.756,1.764,2.897,2.389,2.076,0.428,4.774,0.096,-56.319,-16.58,3.209,2.774,0.082,2.308],[6234.5,0.111,7.934,2003.619,600.847,24.647,87.74,32.186,0.349,12.841,1.642,1.664,1.671,2.857,2.443,1.443,0.353,4.062,0.089,-59.05,-13.929,2.539,2.663,0.075,2.312],[6235.0,0.09,7.927,2001.688,629.098,24.606,87.437,31.349,0.372,12.554,1.619,1.59,1.624,3.035,3.127,1.272,0.347,3.662,0.094,-61.425,-21.405,2.411,2.577,0.069,2.296],[6235.5,0.092,7.929,2008.145,651.952,24.181,88.008,33.041,0.37,13.698,1.601,1.534,1.607,3.243,2.754,1.403,0.379,3.684,0.09,-62.366,-33.607,2.642,2.581,0.068,2.297],[6236.0,0.112,7.93,2010.201,673.967,23.995,88.874,36.158,0.356,15.665,1.614,1.484,1.586,3.503,2.669,1.726,0.428,4.013,0.078,-62.381,-39.933,3.13,2.705,0.073,2.315],[6236.5,0.09,7.909,2004.693,703.239,24.248,89.574,38.273,0.36,17.067,1.713,1.422,1.526,3.69,2.979,1.827,0.449,4.072,0.069,-62.638,-39.761,3.356,2.775,0.072,2.328],[6237.0,0.073,7.899,1998.614,744.166,24.617,90.152,37.258,0.394,17.51,1.721,1.344,1.414,3.482,2.967,1.639,0.453,3.623,0.066,-63.582,-41.497,3.267,2.678,0.059,2.324],[6237.5,0.073,7.909,1993.691,801.42,25.239,90.822,33.486,0.458,17.405,1.573,1.248,1.261,3.114,2.885,1.336,0.465,2.887,0.069,-64.848,-43.678,3.076,2.458,0.048,2.306],[6238.0,0.084,7.912,1991.482,877.561,26.256,91.653,29.484,0.512,17.591,1.497,1.14,1.105,2.612,2.498,1.135,0.565,2.103,0.075,-65.638,-39.438,2.965,2.029,0.047,2.276],[6238.5,0.101,7.906,1995.287,959.867,27.327,92.61,26.609,0.518,18.011,1.54,1.042,0.982,2.022,1.654,1.148,0.774,1.543,0.084,-65.412,-30.952,3.037,1.519,0.053,2.241],[6239.0,0.107,7.902,1998.465,1026.892,28.276,93.588,24.831,0.504,17.535,1.534,0.974,0.903,1.609,1.394,1.217,0.932,1.313,0.091,-64.73,-22.379,3.133,1.257,0.057,2.212],[6239.5,0.111,7.915,1995.268,1068.762,29.191,94.46,24.22,0.486,17.003,1.489,0.936,0.865,1.293,1.245,1.319,1.145,1.177,0.095,-63.985,-6.035,3.271,1.088,0.05,2.194],[6240.0,0.159,7.924,1988.461,1084.428,30.791,95.147,24.106,0.474,16.824,1.501,0.922,0.865,1.014,1.072,1.407,1.352,1.042,0.097,-62.55,20.486,3.402,0.936,0.04,2.187],[6240.5,0.187,7.927,1981.956,1072.468,31.115,95.706,24.502,0.481,16.816,1.569,0.932,0.904,0.823,0.994,1.4,1.393,1.005,0.098,-60.414,33.418,3.437,0.915,0.039,2.186],[6241.0,0.208,7.924,1981.671,1036.905,29.435,96.313,27.939,0.497,17.923,1.68,0.964,0.977,0.792,1.042,1.36,1.373,0.991,0.1,-58.468,18.203,3.449,0.934,0.047,2.19],[6241.5,0.175,7.919,1983.824,987.357,27.939,97.014,35.46,0.528,19.885,1.901,1.013,1.072,1.073,1.135,1.314,1.196,1.134,0.099,-56.477,-3.196,3.543,1.144,0.054,2.202],[6242.0,0.175,7.926,1984.527,925.285,27.558,97.646,44.052,0.573,22.745,2.218,1.081,1.169,1.872,1.966,1.461,0.978,1.535,0.092,-53.606,-9.349,4.308,1.68,0.057,2.23],[6242.5,0.17,7.921,1985.859,845.931,28.238,97.982,49.548,0.619,27.947,2.42,1.182,1.273,2.938,2.968,1.883,1.053,1.789,0.083,-50.599,-11.028,6.003,2.099,0.062,2.267],[6243.0,0.17,7.913,1989.125,754.025,29.563,97.928,50.755,0.666,33.81,2.393,1.326,1.399,3.517,3.075,2.237,1.332,1.693,0.079,-48.682,-19.487,7.628,2.132,0.064,2.295],[6243.5,0.175,7.932,1989.184,665.87,30.339,97.607,51.054,0.691,36.804,2.319,1.502,1.554,3.513,3.233,2.338,1.469,1.597,0.086,-47.414,-31.415,8.255,2.089,0.062,2.3],[6244.0,0.175,7.928,1992.207,596.587,30.983,97.208,54.063,0.715,38.883,2.321,1.676,1.737,3.445,3.311,2.376,1.588,1.512,0.101,-45.426,-39.887,8.686,2.041,0.068,2.293],[6244.5,0.188,7.921,2005.092,549.871,32.232,96.768,59.844,0.789,44.125,2.34,1.819,1.937,3.518,3.54,2.468,2.041,1.253,0.108,-42.207,-36.028,9.962,1.846,0.081,2.295],[6245.0,0.192,7.925,2015.17,518.496,34.42,96.197,65.802,0.894,51.227,2.371,1.929,2.136,3.757,3.72,2.584,2.531,1.033,0.101,-38.318,-22.695,11.821,1.743,0.084,2.316],[6245.5,0.259,7.923,2008.275,492.696,35.729,95.548,70.601,0.917,54.479,2.397,2.03,2.311,4.039,3.966,2.786,2.302,1.271,0.087,-34.48,-15.538,13.023,2.187,0.078,2.345],[6246.0,0.29,7.916,1996.216,469.857,35.705,95.064,74.064,0.877,54.797,2.427,2.128,2.449,4.175,4.075,2.952,1.822,1.637,0.077,-31.36,-20.963,13.206,2.712,0.072,2.368],[6246.5,0.32,7.926,2003.61,453.694,35.934,95.066,76.665,0.85,54.418,2.458,2.204,2.53,4.087,3.826,2.997,1.603,1.89,0.074,-29.472,-30.018,13.009,3.036,0.075,2.377],[6247.0,0.324,7.939,2027.033,446.438,36.607,95.702,77.999,0.822,52.704,2.417,2.24,2.545,3.806,3.826,3.032,1.385,2.211,0.075,-28.695,-32.312,12.718,3.434,0.082,2.377],[6247.5,0.289,7.928,2041.836,447.177,37.115,96.711,78.618,0.807,51.485,2.366,2.236,2.499,3.576,3.569,3.047,1.248,2.445,0.074,-28.438,-25.717,12.561,3.739,0.084,2.379],[6248.0,0.315,7.911,2045.441,454.192,37.411,97.593,80.16,0.822,51.522,2.424,2.202,2.423,3.428,3.366,2.978,1.196,2.49,0.071,-27.962,-11.645,12.495,3.877,0.076,2.383],[6248.5,0.245,7.904,2053.946,466.265,37.991,97.985,83.16,0.892,53.256,2.579,2.145,2.345,3.056,3.209,2.718,1.179,2.307,0.066,-26.537,7.459,12.304,3.874,0.075,2.388],[6249.0,0.269,7.909,2067.716,482.509,39.038,97.924,85.385,1.0,56.339,2.711,2.072,2.268,2.546,2.335,2.408,1.214,1.99,0.063,-24.958,20.482,12.243,3.746,0.09,2.389],[6249.5,0.255,7.888,2074.139,501.483,39.608,97.789,82.31,1.063,56.534,2.663,1.994,2.185,2.447,2.075,2.259,1.253,1.803,0.064,-26.073,13.72,12.264,3.632,0.103,2.383],[6250.0,0.21,7.892,2074.461,520.546,38.449,97.846,71.905,0.998,49.188,2.338,1.921,2.093,2.904,2.926,2.103,1.13,1.875,0.069,-31.207,-3.113,10.753,3.516,0.106,2.369],[6250.5,0.141,7.901,2070.078,534.705,35.6,97.627,59.586,0.744,38.506,1.959,1.87,2.008,3.507,3.764,2.348,1.003,2.344,0.075,-38.088,-12.551,8.487,3.134,0.105,2.355],[6251.0,0.195,7.872,2057.815,536.705,32.525,96.112,52.28,0.526,31.565,1.768,1.863,1.964,3.614,3.885,2.888,1.016,2.84,0.08,-43.787,-16.872,7.642,2.793,0.101,2.344],[6251.5,0.236,7.844,2044.493,522.846,30.525,92.808,50.301,0.492,29.279,1.832,1.913,1.998,3.405,4.02,3.048,1.048,2.91,0.09,-47.446,-17.566,7.654,2.711,0.095,2.331],[6252.0,0.261,7.824,2040.53,500.199,30.071,88.446,51.236,0.496,29.665,2.035,1.999,2.13,3.292,3.877,3.033,1.076,2.822,0.106,-49.571,-10.069,7.68,2.651,0.093,2.313],[6252.5,0.244,7.826,2042.645,483.145,30.177,84.61,54.063,0.543,31.801,2.229,2.07,2.338,3.497,3.592,2.828,1.141,2.497,0.112,-50.407,-2.193,7.768,2.532,0.094,2.31],[6253.0,0.253,7.848,2045.729,481.073,28.873,82.475,56.063,0.605,34.0,2.415,2.079,2.522,4.546,3.759,2.551,1.184,2.155,0.084,-49.857,-3.47,7.859,2.463,0.102,2.353],[6253.5,0.265,7.914,2047.949,491.396,25.006,81.876,54.53,0.614,34.71,2.727,2.035,2.55,8.491,7.333,2.522,1.168,2.16,0.039,-48.337,-8.782,7.9,2.511,0.124,2.44],[6254.0,0.31,7.996,2045.968,504.339,20.672,81.951,51.429,0.603,34.343,3.04,1.983,2.397,17.833,18.342,2.571,1.127,2.284,0.015,-46.626,-12.698,7.909,2.607,0.146,2.515],[6254.5,0.36,8.011,2040.565,512.462,18.965,82.473,50.669,0.583,33.269,2.775,1.951,2.163,15.998,22.723,2.651,1.056,2.517,0.013,-45.141,-15.816,7.882,2.774,0.141,2.52],[6255.0,0.364,7.998,2046.49,516.338,19.617,84.189,52.634,0.56,32.271,2.219,1.937,1.959,8.152,11.584,2.752,0.981,2.813,0.032,-43.978,-17.931,7.868,2.982,0.114,2.453],[6255.5,0.329,7.995,2066.486,522.234,23.682,87.91,56.16,0.547,32.413,1.973,1.915,1.839,4.547,3.679,2.819,0.929,3.036,0.077,-43.031,-18.274,7.869,3.145,0.087,2.353],[6256.0,0.311,8.014,2080.065,532.718,27.844,93.491,62.113,0.589,34.49,2.029,1.877,1.806,3.326,2.524,2.68,0.88,3.043,0.127,-40.98,-16.954,7.994,3.382,0.074,2.272],[6256.5,0.322,8.074,2074.871,540.315,30.961,99.875,71.328,0.728,41.014,2.218,1.851,1.852,2.797,2.166,2.453,0.922,2.692,0.147,-35.232,-13.678,9.074,3.649,0.074,2.244],[6257.0,0.333,8.183,2064.98,530.691,34.121,106.01,81.16,0.813,48.971,2.44,1.884,1.984,2.471,1.927,2.592,0.997,2.609,0.142,-25.863,-7.399,10.755,4.007,0.09,2.253],[6257.5,0.364,8.283,2061.482,495.385,37.314,111.55,85.515,0.8,52.773,2.45,2.019,2.231,2.4,1.873,2.835,0.99,2.865,0.164,-17.892,1.335,11.571,4.338,0.113,2.239],[6258.0,0.392,8.314,2058.781,444.318,41.015,116.511,79.538,0.791,51.179,2.003,2.251,2.622,2.718,2.034,2.787,0.973,2.864,0.315,-15.334,6.275,11.257,4.291,0.094,2.12],[6258.5,0.227,8.305,2047.92,398.143,44.967,120.538,66.559,0.774,45.199,1.515,2.512,3.124,3.687,7.541,2.533,0.986,2.582,0.742,-15.916,0.849,10.024,3.79,0.006,1.901],[6259.0,0.311,8.319,2027.297,368.64,48.152,122.876,54.402,0.722,40.167,1.311,2.713,3.566,6.024,8.927,2.538,1.415,1.985,1.405,-16.478,-2.01,9.306,2.746,-0.08,1.677],[6259.5,0.367,8.338,2005.907,353.13,51.391,122.927,47.084,0.701,37.936,1.425,2.832,3.726,10.496,17.494,2.619,2.271,1.267,2.014,-17.309,10.767,9.332,1.68,-0.1,1.522],[6260.0,0.343,8.349,1991.993,346.188,54.595,120.75,46.453,0.695,37.678,1.763,2.889,3.581,11.417,29.862,2.679,2.727,0.983,2.239,-18.744,19.292,9.449,1.286,-0.043,1.477],[6260.5,0.369,8.33,1983.919,347.791,56.601,116.912,51.584,0.64,41.897,2.139,2.875,3.282,5.873,16.652,3.225,2.87,1.119,1.868,-20.207,5.652,10.41,1.346,0.021,1.569],[6261.0,0.207,8.32,1983.983,364.378,55.331,112.05,55.222,0.574,48.01,2.306,2.744,2.94,3.279,2.963,4.027,3.134,1.281,1.087,-23.83,-9.998,11.722,1.389,0.048,1.791],[6261.5,0.191,8.334,1997.183,405.243,49.45,106.639,51.872,0.57,48.732,2.035,2.468,2.571,2.735,2.216,4.099,3.234,1.269,0.447,-31.661,-11.325,11.823,1.359,0.071,2.053],[6262.0,0.185,8.334,2014.84,475.389,40.714,101.154,45.823,0.636,44.326,1.671,2.104,2.182,2.981,2.697,3.343,3.424,0.991,0.184,-41.187,-12.412,10.706,1.173,0.081,2.224],[6262.5,0.185,8.349,2030.341,567.553,33.076,96.394,44.224,0.651,39.712,1.613,1.762,1.795,3.422,3.004,2.96,3.404,0.885,0.109,-47.432,-18.11,9.817,1.081,0.078,2.289],[6263.0,0.178,8.361,2047.059,664.404,30.827,93.27,46.127,0.58,37.692,1.673,1.505,1.461,3.731,3.313,3.199,2.864,1.147,0.084,-49.024,-21.55,9.405,1.236,0.074,2.305],[6263.5,0.148,8.361,2058.287,748.447,32.161,92.257,44.134,0.518,37.164,1.703,1.336,1.226,3.274,3.036,3.414,2.465,1.388,0.076,-50.177,-23.346,9.03,1.361,0.07,2.302],[6264.0,0.138,8.364,2053.22,812.435,33.401,93.067,36.26,0.435,32.84,1.725,1.231,1.096,2.021,2.277,3.783,2.035,2.045,0.084,-54.658,-28.311,8.169,1.531,0.058,2.271],[6264.5,0.157,8.338,2036.502,863.184,33.816,94.765,29.908,0.367,25.196,1.782,1.159,1.036,1.079,1.19,3.459,1.419,2.538,0.104,-60.624,-36.688,6.183,1.632,0.054,2.22],[6265.0,0.161,8.303,2024.802,912.418,33.449,96.274,30.113,0.409,20.575,1.821,1.096,1.004,0.905,0.961,2.347,1.153,2.034,0.117,-64.2,-43.517,4.618,1.487,0.064,2.184],[6265.5,0.134,8.278,2027.079,959.791,32.818,96.976,33.761,0.503,20.671,1.785,1.042,0.98,1.309,1.896,1.743,1.257,1.425,0.115,-64.79,-37.898,4.383,1.31,0.072,2.177],[6266.0,0.114,8.277,2032.186,992.955,32.37,96.902,35.708,0.676,24.202,1.715,1.007,0.967,1.783,2.365,1.363,1.642,0.907,0.107,-63.855,-20.327,4.507,1.061,0.066,2.184],[6266.5,0.112,8.288,2026.854,1000.619,32.366,96.448,34.562,0.81,27.435,1.689,0.999,0.968,1.766,1.518,1.109,1.976,0.565,0.102,-62.081,-6.174,4.578,0.863,0.057,2.193],[6267.0,0.093,8.3,2018.748,978.771,32.393,96.001,32.101,0.824,28.097,1.659,1.022,0.978,1.906,1.957,1.084,2.015,0.538,0.101,-59.746,-5.393,4.563,0.84,0.058,2.2],[6267.5,0.069,8.301,2016.506,931.992,32.138,95.692,29.454,0.811,27.236,1.578,1.073,1.002,2.366,2.573,1.1,1.937,0.579,0.105,-57.459,-12.665,4.546,0.88,0.056,2.202],[6268.0,0.078,8.307,2018.409,875.214,31.746,95.399,27.437,0.7,23.87,1.545,1.143,1.055,2.047,1.392,1.235,1.525,0.891,0.116,-55.616,-18.427,4.336,1.114,0.049,2.194],[6268.5,0.078,8.318,2022.043,824.447,31.359,95.038,28.6,0.565,20.568,1.599,1.213,1.14,1.39,1.294,1.43,1.102,1.323,0.127,-53.729,-19.493,4.099,1.394,0.048,2.187],[6269.0,0.078,8.318,2021.203,785.102,31.257,94.714,34.792,0.501,20.499,1.667,1.274,1.238,1.231,1.602,1.687,0.994,1.716,0.13,-50.815,-23.009,4.253,1.597,0.053,2.192],[6269.5,0.174,8.31,2011.672,749.12,31.506,94.558,43.839,0.428,22.345,1.766,1.335,1.332,1.374,1.36,2.34,0.925,2.574,0.124,-45.332,-31.861,4.963,2.012,0.057,2.212],[6270.0,0.187,8.304,2002.993,705.788,32.038,94.615,52.185,0.421,27.055,1.961,1.417,1.425,1.682,1.468,3.049,0.968,3.173,0.111,-36.9,-39.379,6.595,2.505,0.068,2.246],[6270.5,0.213,8.326,2001.838,653.128,32.748,94.847,57.162,0.524,34.221,2.24,1.531,1.541,2.014,2.183,3.211,1.167,2.807,0.093,-29.362,-40.452,8.558,2.716,0.078,2.292],[6271.0,0.21,8.33,2002.894,601.511,33.404,95.239,59.668,0.619,39.381,2.486,1.662,1.692,2.192,2.602,3.052,1.339,2.293,0.078,-25.905,-33.885,9.63,2.673,0.072,2.332],[6271.5,0.18,8.343,2006.552,562.627,33.411,95.874,62.955,0.661,41.724,2.673,1.777,1.851,2.415,2.311,2.945,1.426,2.07,0.073,-23.826,-20.827,9.935,2.588,0.055,2.35],[6272.0,0.208,8.34,2015.158,539.469,33.131,96.851,68.583,0.765,46.978,2.891,1.854,1.953,2.939,2.898,2.821,1.724,1.741,0.072,-19.259,-10.25,10.851,2.401,0.05,2.358],[6272.5,0.234,8.333,2021.202,526.713,33.575,98.166,75.775,1.096,58.042,3.162,1.899,1.98,3.487,3.709,2.439,2.29,1.15,0.062,-12.229,-8.813,13.178,2.183,0.061,2.381],[6273.0,0.207,8.353,2017.663,517.807,35.105,99.609,84.011,1.407,67.478,3.34,1.931,1.969,3.185,2.84,2.103,2.373,0.893,0.047,-4.51,-13.968,15.066,2.392,0.074,2.415],[6273.5,0.219,8.369,2009.868,509.393,37.252,100.765,92.316,1.474,72.221,3.303,1.963,1.956,2.379,1.772,2.104,2.081,1.016,0.036,2.104,-19.433,15.825,2.834,0.085,2.444],[6274.0,0.186,8.365,2002.663,503.306,39.388,101.283,98.282,1.399,75.184,3.154,1.987,1.951,1.953,1.705,2.358,1.968,1.201,0.032,6.295,-23.553,16.757,3.165,0.092,2.457],[6274.5,0.253,8.348,1995.474,505.316,41.67,101.224,101.06,1.265,76.395,3.115,1.979,1.965,1.803,1.608,2.746,1.904,1.446,0.034,8.522,-28.144,17.639,3.441,0.097,2.45],[6275.0,0.166,8.33,1989.488,518.374,43.927,101.003,101.945,1.173,76.796,3.164,1.929,1.988,1.796,1.555,3.044,1.892,1.61,0.037,9.88,-32.112,18.194,3.57,0.102,2.439],[6275.5,0.241,8.326,1991.338,536.792,45.965,101.064,101.627,1.111,77.216,3.216,1.863,1.979,1.941,1.511,3.313,1.945,1.702,0.034,10.762,-30.378,18.748,3.578,0.109,2.444],[6276.0,0.253,8.325,2001.813,550.498,47.134,101.446,100.035,1.066,77.793,3.328,1.817,1.912,2.321,2.132,3.539,2.027,1.749,0.024,11.198,-25.427,19.251,3.529,0.121,2.472],[6276.5,0.261,8.342,2009.053,552.847,46.817,101.428,99.381,1.122,79.086,3.451,1.809,1.824,2.626,2.368,3.438,2.221,1.563,0.016,11.375,-23.862,19.626,3.296,0.122,2.506],[6277.0,0.263,8.397,2007.894,542.516,46.057,100.312,101.195,1.286,81.199,3.356,1.843,1.773,2.325,1.622,3.046,2.552,1.214,0.013,11.505,-23.31,19.816,2.906,0.111,2.517],[6277.5,0.259,8.375,2007.901,525.066,46.623,98.499,103.578,1.533,85.454,3.016,1.905,1.795,1.897,1.402,2.549,3.011,0.872,0.017,11.478,-19.193,19.627,2.457,0.108,2.501],[6278.0,0.231,8.329,2013.605,512.5,47.765,97.25,103.547,1.862,91.57,2.734,1.951,1.88,1.599,1.351,2.054,3.684,0.577,0.024,11.171,-16.469,19.219,1.972,0.113,2.479],[6278.5,0.185,8.322,2022.484,512.14,48.054,97.381,99.976,2.013,92.967,2.654,1.953,1.969,1.663,1.331,1.832,4.087,0.45,0.026,10.798,-21.409,18.814,1.712,0.112,2.472],[6279.0,0.107,8.324,2034.51,518.846,47.016,98.558,93.65,1.864,86.733,2.731,1.927,1.994,2.373,1.595,1.932,3.789,0.517,0.019,10.556,-30.712,18.309,1.805,0.118,2.495],[6279.5,0.102,8.327,2044.313,521.829,45.032,99.67,89.618,1.675,80.647,2.818,1.916,1.962,3.538,4.359,2.128,3.532,0.603,0.008,10.271,-35.642,18.137,1.908,0.123,2.546],[6280.0,0.101,8.332,2044.769,513.827,43.438,99.777,92.441,1.611,80.113,2.819,1.946,1.938,3.718,3.832,2.219,3.385,0.659,0.003,9.752,-31.975,18.238,2.011,0.107,2.589],[6280.5,0.101,8.327,2036.326,494.312,43.587,99.01,99.873,1.545,83.291,2.694,2.023,1.959,2.794,1.452,2.457,3.054,0.812,0.002,9.004,-24.195,19.239,2.354,0.089,2.592],[6281.0,0.095,8.323,2029.813,468.041,45.084,97.972,107.085,1.409,87.623,2.491,2.137,2.034,2.134,1.639,2.894,2.943,0.983,0.006,8.155,-19.989,20.718,2.612,0.09,2.565],[6281.5,0.095,8.322,2036.407,440.151,46.429,96.683,111.729,1.33,90.181,2.351,2.272,2.186,1.895,1.546,3.11,2.966,1.048,0.01,7.26,-21.243,21.123,2.643,0.092,2.542],[6282.0,0.095,8.314,2045.036,412.416,46.679,94.605,111.157,1.287,89.478,2.382,2.425,2.463,1.886,1.494,3.241,2.988,1.085,0.013,6.217,-21.951,21.282,2.644,0.088,2.532],[6282.5,0.09,8.323,2036.295,382.154,45.618,91.161,102.751,1.17,83.212,2.563,2.617,2.916,2.189,1.428,3.481,2.813,1.27,0.013,4.848,-21.236,20.729,2.767,0.088,2.537],[6283.0,0.09,8.328,2016.109,345.316,43.005,86.158,88.42,0.952,67.537,2.738,2.896,3.623,3.049,2.895,3.476,2.038,1.87,0.008,3.059,-21.987,17.015,3.229,0.088,2.565],[6283.5,0.09,8.331,2003.227,303.703,39.194,80.172,72.074,0.806,49.688,2.844,3.293,4.666,4.799,7.064,2.802,1.235,2.283,0.002,0.645,-22.161,11.591,3.468,0.087,2.613],[6284.0,0.09,8.283,2009.226,265.073,34.058,74.632,57.206,0.81,38.732,2.932,3.773,5.952,7.158,5.996,1.979,0.946,2.078,0.0,-2.474,-18.577,8.138,3.178,0.105,2.655],[6284.5,0.09,8.162,2029.632,238.447,27.07,71.055,46.135,0.859,33.423,2.999,4.194,6.917,12.315,16.869,1.464,0.883,1.655,0.0,-5.852,-15.051,6.382,2.678,0.131,2.675],[6285.0,0.09,8.093,2042.479,229.712,22.263,70.29,39.794,0.854,29.616,2.917,4.353,6.864,27.815,31.031,1.186,0.834,1.425,0.0,-9.135,-17.466,5.184,2.303,0.142,2.678],[6285.5,0.09,8.091,2040.673,239.366,19.573,72.303,37.708,0.793,25.799,2.664,4.178,5.894,35.926,22.157,1.137,0.743,1.54,0.0,-12.495,-23.262,4.602,2.306,0.145,2.668],[6286.0,0.089,8.097,2043.603,264.973,18.163,76.266,40.01,0.753,23.803,2.434,3.774,4.65,16.285,17.142,1.149,0.683,1.682,0.0,-15.846,-25.579,4.417,2.399,0.157,2.635],[6286.5,0.089,8.096,2062.458,303.889,18.709,81.271,46.166,0.729,24.446,2.388,3.291,3.616,6.725,13.245,1.186,0.661,1.798,0.007,-18.793,-22.463,4.41,2.479,0.163,2.572],[6287.0,0.084,8.099,2085.5,351.506,21.817,86.804,54.561,0.668,27.381,2.416,2.845,2.929,4.155,6.192,1.632,0.673,2.389,0.028,-21.124,-22.212,5.418,2.958,0.148,2.496],[6287.5,0.123,8.095,2095.517,402.122,26.517,92.275,64.539,0.619,33.872,2.373,2.487,2.502,3.14,3.556,2.495,0.775,3.176,0.051,-21.938,-27.677,7.833,3.701,0.115,2.434],[6288.0,0.119,8.097,2084.151,453.669,30.173,96.71,77.379,0.655,45.88,2.392,2.204,2.221,2.35,2.784,3.32,1.049,3.234,0.06,-20.318,-27.582,11.216,3.985,0.087,2.403],[6288.5,0.118,8.081,2063.264,510.738,33.711,99.333,91.693,0.737,60.102,2.564,1.958,2.02,1.867,1.936,3.918,1.442,2.77,0.057,-17.643,-11.032,14.823,3.837,0.089,2.395],[6289.0,0.118,8.021,2054.039,580.339,38.228,100.131,100.04,0.79,68.15,2.764,1.723,1.832,1.681,1.475,4.083,1.648,2.478,0.048,-16.664,9.936,16.466,3.708,0.112,2.4],[6289.5,0.118,7.999,2055.74,662.283,41.561,99.645,94.723,0.818,66.699,2.695,1.51,1.61,1.739,1.512,3.88,1.647,2.355,0.04,-19.864,14.663,16.162,3.64,0.126,2.406],[6290.0,0.118,7.99,2052.052,751.055,42.402,98.465,75.857,0.888,57.18,2.354,1.331,1.378,2.039,1.894,3.048,1.51,1.989,0.037,-28.705,3.761,13.596,3.306,0.118,2.4],[6290.5,0.119,7.949,2039.194,841.94,40.325,97.004,53.092,0.877,42.43,1.992,1.188,1.175,2.721,2.203,2.014,1.252,1.593,0.044,-41.013,-4.181,9.135,2.662,0.092,2.363],[6291.0,0.118,7.887,2031.06,930.14,35.978,95.596,36.0,0.656,28.291,1.781,1.075,1.022,3.271,2.036,1.726,1.048,1.685,0.059,-51.468,-0.953,5.7,1.998,0.071,2.307],[6291.5,0.118,7.863,2033.685,1009.814,31.023,94.638,27.115,0.42,19.065,1.656,0.99,0.922,2.595,1.504,1.874,0.932,2.042,0.073,-56.888,0.391,3.938,1.56,0.066,2.257],[6292.0,0.124,7.864,2041.142,1074.776,28.204,94.383,23.299,0.344,15.544,1.61,0.93,0.87,1.326,0.988,1.942,0.987,1.995,0.081,-58.293,-9.288,3.416,1.302,0.068,2.226],[6292.5,0.227,7.86,2044.987,1118.956,28.728,94.729,22.046,0.346,15.076,1.61,0.894,0.854,0.748,0.842,1.914,1.134,1.698,0.086,-58.191,-22.21,3.387,1.114,0.069,2.206],[6293.0,0.227,7.853,2045.127,1143.06,29.722,95.348,22.55,0.344,14.941,1.607,0.875,0.866,0.647,0.822,1.914,1.175,1.631,0.089,-58.266,-25.653,3.361,1.062,0.073,2.193],[6293.5,0.227,7.856,2045.439,1152.003,30.238,95.993,24.531,0.35,15.616,1.628,0.868,0.892,0.699,0.814,1.91,1.258,1.532,0.09,-58.424,-12.832,3.41,1.012,0.074,2.189],[6294.0,0.232,7.859,2047.593,1146.638,30.257,96.567,27.338,0.347,16.202,1.723,0.872,0.92,0.902,0.928,1.973,1.182,1.753,0.088,-58.141,8.083,3.49,1.139,0.065,2.196],[6294.5,0.244,7.85,2050.444,1128.0,29.813,96.944,30.024,0.332,16.178,1.825,0.887,0.945,1.359,1.504,2.076,0.892,2.423,0.083,-57.962,16.491,3.511,1.516,0.051,2.21],[6295.0,0.287,7.849,2054.727,1101.364,29.654,96.975,32.614,0.346,16.688,1.906,0.908,0.968,1.512,1.447,2.018,0.712,2.859,0.079,-58.232,9.666,3.556,1.88,0.048,2.226],[6295.5,0.306,7.854,2056.61,1074.919,30.531,96.674,35.533,0.365,17.551,1.987,0.93,0.983,1.412,1.075,1.93,0.64,3.017,0.076,-58.356,3.599,3.59,2.082,0.06,2.238],[6296.0,0.308,7.852,2055.303,1059.063,32.064,96.263,37.207,0.378,18.471,2.004,0.944,0.983,1.541,1.683,1.877,0.636,2.952,0.076,-58.589,-4.512,3.619,2.111,0.08,2.24],[6296.5,0.298,7.848,2063.883,1058.494,33.22,95.963,35.444,0.385,18.317,1.973,0.945,0.967,1.96,1.968,1.838,0.633,2.904,0.081,-59.934,-17.426,3.614,2.117,0.088,2.23],[6297.0,0.149,7.852,2080.505,1069.72,33.505,95.843,31.391,0.363,16.866,1.98,0.935,0.946,2.288,1.943,1.866,0.629,2.971,0.089,-61.849,-21.82,3.458,2.043,0.089,2.207],[6297.5,0.263,7.851,2077.197,1090.173,33.202,95.856,28.349,0.348,15.837,1.975,0.917,0.927,2.146,1.902,1.912,0.684,2.813,0.097,-62.671,-18.847,3.393,1.852,0.092,2.184],[6298.0,0.193,7.853,2047.763,1115.104,32.722,96.036,27.177,0.361,15.815,1.927,0.897,0.911,2.044,1.358,1.882,0.754,2.504,0.1,-61.927,-17.769,3.465,1.708,0.093,2.173],[6298.5,0.278,7.878,2023.148,1126.647,32.154,96.524,26.509,0.397,16.408,1.908,0.888,0.899,2.069,1.527,1.744,0.806,2.176,0.099,-60.005,-9.997,3.522,1.625,0.086,2.174],[6299.0,0.306,7.901,2018.932,1109.182,31.594,97.32,26.154,0.449,17.413,1.898,0.902,0.898,1.989,2.18,1.621,0.897,1.823,0.098,-57.401,1.192,3.704,1.537,0.075,2.18],[6299.5,0.446,7.899,2028.52,1064.934,31.325,98.186,27.326,0.489,18.318,1.893,0.939,0.918,1.797,1.89,1.583,0.98,1.618,0.1,-55.271,-2.534,3.953,1.497,0.067,2.183],[6300.0,0.433,7.895,2045.708,1002.284,31.683,98.889,31.974,0.516,20.054,1.913,0.998,0.972,1.656,1.69,1.557,1.003,1.552,0.107,-54.156,-14.531,4.099,1.517,0.067,2.183],[6300.5,0.397,7.887,2054.188,924.931,32.586,99.536,43.7,0.613,26.611,1.987,1.081,1.072,1.471,1.727,1.623,1.158,1.427,0.11,-51.735,-20.54,5.168,1.63,0.072,2.193],[6301.0,0.373,7.925,2043.243,836.094,33.177,100.404,61.945,0.81,39.158,2.209,1.196,1.224,1.309,1.382,1.94,1.424,1.389,0.103,-45.859,-21.419,8.204,2.133,0.079,2.228],[6301.5,0.472,7.953,2027.6,744.942,33.622,101.415,76.911,0.908,50.585,2.594,1.342,1.425,1.43,1.263,2.464,1.394,1.826,0.082,-40.21,-23.56,11.389,3.125,0.086,2.289],[6302.0,0.199,7.957,2020.618,660.035,35.564,101.966,80.416,0.886,56.985,2.87,1.515,1.672,2.009,2.305,2.825,1.267,2.229,0.064,-40.03,-31.249,12.779,3.742,0.096,2.348],[6302.5,0.112,7.872,2018.779,587.154,39.03,101.475,75.795,0.886,57.285,2.887,1.703,1.946,2.932,3.058,2.803,1.241,2.258,0.062,-44.607,-41.61,12.673,3.79,0.119,2.367],[6303.0,0.095,7.658,2021.14,534.084,40.126,100.095,69.723,0.892,51.695,2.959,1.872,2.171,3.606,2.959,2.494,1.158,2.149,0.072,-49.756,-41.696,11.356,3.634,0.16,2.36],[6303.5,0.42,7.582,2025.64,508.823,37.236,98.639,64.581,0.9,46.44,3.101,1.965,2.241,4.052,2.841,2.176,1.193,1.843,0.064,-53.589,-34.438,10.006,3.143,0.197,2.382],[6304.0,0.415,7.696,2026.311,510.262,33.145,97.753,61.056,0.935,45.706,3.171,1.96,2.128,4.684,3.358,2.083,1.685,1.349,0.035,-55.928,-34.888,9.944,2.364,0.213,2.447],[6304.5,0.417,7.802,2025.948,528.272,30.289,97.316,59.87,0.971,47.496,3.094,1.893,1.923,4.973,5.427,2.106,2.515,0.888,0.015,-57.247,-40.499,10.439,1.626,0.201,2.509],[6305.0,0.209,7.861,2029.056,553.349,29.241,96.747,58.228,0.957,48.353,2.707,1.807,1.737,4.279,6.122,2.195,3.039,0.723,0.016,-58.671,-42.37,10.713,1.31,0.158,2.504],[6305.5,0.187,7.916,2033.873,580.567,29.573,95.732,54.996,0.9,48.006,2.253,1.722,1.608,3.251,3.738,2.374,3.231,0.735,0.034,-60.191,-42.595,10.874,1.25,0.107,2.436],[6306.0,0.187,7.927,2040.771,606.198,30.895,94.547,52.854,0.781,45.632,2.132,1.65,1.524,2.648,2.814,2.764,3.042,0.934,0.062,-60.554,-45.597,10.837,1.338,0.079,2.363],[6306.5,0.187,7.93,2045.08,625.455,32.162,93.692,53.062,0.668,43.073,2.267,1.599,1.483,2.513,2.654,3.132,2.556,1.246,0.08,-59.47,-50.142,10.591,1.553,0.068,2.322],[6307.0,0.192,7.934,2038.856,634.3,32.631,93.293,52.896,0.681,43.388,2.371,1.577,1.499,2.527,2.674,2.982,2.216,1.343,0.085,-58.252,-52.555,10.294,1.73,0.058,2.31],[6307.5,0.192,7.94,2024.948,631.845,32.473,93.102,52.04,0.747,44.099,2.382,1.583,1.574,2.622,2.555,2.621,2.002,1.309,0.089,-57.624,-49.847,9.971,1.852,0.051,2.303],[6308.0,0.209,7.934,2014.587,622.917,32.332,92.985,53.103,0.773,43.231,2.535,1.605,1.665,2.893,2.464,2.485,1.919,1.296,0.091,-56.922,-42.496,9.808,1.898,0.061,2.303],[6308.5,0.169,7.914,2017.804,613.788,32.229,92.925,57.504,0.802,43.005,2.819,1.629,1.724,3.428,2.724,2.391,1.91,1.252,0.081,-55.5,-37.6,9.777,1.901,0.072,2.324],[6309.0,0.128,7.941,2034.16,600.922,32.27,92.57,63.64,0.857,44.489,2.978,1.664,1.747,3.709,2.926,2.206,1.73,1.284,0.064,-53.491,-41.923,9.623,2.09,0.069,2.36],[6309.5,0.124,7.982,2047.787,574.573,31.486,91.278,66.933,0.907,46.799,2.843,1.74,1.767,3.329,2.775,2.043,1.469,1.397,0.054,-51.685,-46.173,9.459,2.406,0.061,2.388],[6310.0,0.137,7.996,2057.862,532.607,31.444,88.709,63.318,0.926,48.123,2.46,1.878,1.833,2.817,2.586,1.999,1.368,1.462,0.056,-51.773,-38.099,9.451,2.565,0.055,2.393],[6310.5,0.147,7.999,2078.074,482.728,31.486,85.214,52.549,0.894,44.888,2.062,2.072,2.014,2.489,2.552,1.973,1.287,1.552,0.073,-54.87,-24.704,9.019,2.609,0.051,2.371],[6311.0,0.145,8.006,2099.399,432.302,30.984,81.43,39.678,0.751,35.379,1.965,2.313,2.38,2.518,2.259,1.882,1.027,1.907,0.099,-59.516,-17.059,7.271,2.633,0.062,2.345],[6311.5,0.146,8.02,2103.968,383.016,27.9,77.482,30.231,0.567,24.402,2.155,2.611,2.994,3.427,1.218,1.623,0.718,2.284,0.1,-62.908,-12.705,4.801,2.434,0.084,2.36],[6312.0,0.272,8.058,2094.773,331.763,22.919,72.869,25.156,0.492,18.146,2.422,3.014,3.94,7.535,15.528,1.263,0.631,2.038,0.058,-64.011,-4.856,3.184,1.897,0.091,2.44],[6312.5,0.285,8.117,2082.585,275.345,16.498,67.34,22.351,0.5,16.309,2.583,3.632,5.41,28.565,86.638,1.011,0.756,1.393,0.014,-63.801,7.055,2.572,1.309,0.086,2.55],[6313.0,0.405,8.2,2077.301,217.316,10.492,61.555,20.709,0.505,15.294,2.642,4.602,7.856,164.103,149.273,0.908,0.86,1.06,0.0,-63.332,17.241,2.341,1.014,0.09,2.633],[6313.5,0.383,8.243,2082.631,169.285,6.872,56.756,19.691,0.478,13.855,2.757,5.907,11.963,316.85,125.735,0.872,0.838,1.042,0.0,-62.978,15.613,2.131,0.942,0.105,2.672],[6314.0,0.43,8.225,2095.184,142.927,5.827,54.064,18.677,0.427,12.187,2.853,6.997,16.757,278.025,132.065,0.832,0.731,1.155,0.0,-62.86,2.577,1.816,0.924,0.106,2.684],[6314.5,0.331,8.217,2103.122,143.432,6.276,53.98,17.827,0.378,10.724,2.722,6.972,15.62,311.968,175.529,0.832,0.626,1.34,0.0,-62.861,-7.461,1.603,0.952,0.093,2.686],[6315.0,0.276,8.23,2102.72,169.756,6.177,56.324,17.194,0.351,10.03,2.558,5.891,9.61,395.475,177.14,0.882,0.597,1.479,0.0,-62.582,-5.609,1.58,0.983,0.08,2.682],[6315.5,0.244,8.226,2101.717,216.404,5.607,60.537,16.81,0.333,9.812,2.547,4.621,5.612,253.829,160.493,0.942,0.584,1.617,0.0,-61.771,-0.736,1.601,1.019,0.072,2.681],[6316.0,0.272,8.225,2107.249,275.216,5.971,65.946,17.455,0.326,10.52,2.48,3.634,3.712,54.688,131.297,1.066,0.616,1.724,0.0,-60.682,-1.833,1.776,1.065,0.069,2.671],[6316.5,0.312,8.214,2113.631,337.65,7.1,71.901,21.19,0.339,13.174,2.129,2.962,2.812,13.812,69.046,1.465,0.825,1.773,0.001,-59.614,-5.635,2.555,1.137,0.071,2.616],[6317.0,0.258,8.105,2113.273,401.573,8.88,77.984,29.656,0.39,18.035,1.854,2.49,2.332,5.098,11.622,1.884,1.166,1.647,0.021,-58.134,-11.236,3.784,1.2,0.087,2.507],[6317.5,0.209,7.992,2103.917,473.5,11.92,84.054,41.06,0.465,24.478,1.887,2.112,2.008,2.958,0.496,2.188,1.447,1.514,0.062,-55.833,-18.448,5.237,1.335,0.119,2.393],[6318.0,0.306,7.957,2091.258,555.538,17.489,89.78,51.301,0.473,30.937,2.087,1.8,1.74,2.561,1.933,2.982,1.549,1.921,0.086,-53.197,-15.496,7.057,1.69,0.144,2.33],[6318.5,0.21,7.935,2094.832,637.227,24.871,94.435,56.028,0.439,36.003,2.217,1.569,1.52,2.692,2.28,3.818,1.59,2.394,0.078,-50.999,-1.218,8.476,1.975,0.147,2.322],[6319.0,0.245,7.934,2111.878,704.725,30.787,97.396,52.727,0.423,37.071,2.093,1.419,1.355,2.809,2.322,4.087,1.62,2.523,0.063,-50.054,7.165,8.818,2.02,0.132,2.339],[6319.5,0.272,7.967,2113.222,750.401,33.445,98.714,43.415,0.438,34.733,1.808,1.333,1.246,2.797,2.01,3.832,1.974,2.125,0.057,-50.815,1.595,8.421,1.702,0.103,2.345],[6320.0,0.379,7.997,2097.483,771.182,33.858,99.076,34.557,0.549,32.527,1.603,1.297,1.195,2.741,1.802,2.933,3.032,1.157,0.063,-52.012,-7.577,7.832,1.073,0.078,2.327],[6320.5,0.421,8.004,2092.497,766.504,32.791,99.194,30.894,0.649,30.543,1.527,1.305,1.201,2.753,1.469,2.288,3.792,0.606,0.075,-51.339,-9.503,7.57,0.742,0.072,2.3],[6321.0,0.423,8.007,2098.883,741.373,31.413,99.329,32.331,0.668,30.475,1.581,1.349,1.262,2.901,1.446,2.225,3.927,0.568,0.085,-48.205,-4.481,7.582,0.717,0.073,2.286],[6321.5,0.414,8.006,2097.133,702.57,31.254,99.286,37.578,0.728,34.268,1.833,1.423,1.38,3.045,1.652,2.093,3.818,0.571,0.083,-43.402,2.548,7.736,0.795,0.081,2.299],[6322.0,0.409,7.991,2084.239,650.887,32.319,98.929,46.91,0.813,37.794,2.2,1.536,1.564,3.049,2.206,1.987,2.609,0.956,0.066,-36.508,9.787,8.24,1.5,0.097,2.344],[6322.5,0.335,8.083,2064.739,586.979,33.193,98.472,58.996,0.835,39.835,2.386,1.704,1.826,3.081,2.791,2.057,1.334,1.639,0.043,-27.029,16.135,8.767,2.585,0.111,2.41],[6323.0,0.369,8.286,2044.622,517.125,33.868,98.189,69.409,0.864,43.534,2.295,1.934,2.159,3.212,2.97,2.171,1.071,2.034,0.028,-17.527,13.157,9.587,3.333,0.12,2.465],[6323.5,0.292,8.372,2032.868,448.574,34.607,98.129,76.173,0.951,48.247,2.272,2.229,2.536,3.39,3.11,2.194,1.069,2.054,0.026,-11.499,-1.515,10.649,3.693,0.135,2.483],[6324.0,0.282,8.374,2029.609,389.171,35.245,98.097,80.707,0.991,51.228,2.364,2.57,2.929,3.447,3.023,2.201,1.081,2.036,0.036,-9.301,-19.342,11.138,3.823,0.16,2.468],[6324.5,0.222,8.367,2031.783,348.999,36.421,97.814,82.582,0.969,52.33,2.358,2.865,3.286,3.377,3.296,2.275,1.072,2.122,0.058,-8.8,-29.733,11.248,3.893,0.168,2.435],[6325.0,0.191,8.364,2040.669,331.053,38.606,97.179,79.66,0.933,52.265,2.1,3.021,3.51,3.498,2.622,2.386,1.052,2.273,0.103,-8.502,-26.699,11.351,4.009,0.14,2.376],[6325.5,0.216,8.371,2055.86,328.757,40.938,96.612,73.622,0.841,49.955,1.82,3.042,3.537,4.617,7.52,2.725,1.033,2.637,0.17,-8.202,-10.95,11.494,4.13,0.111,2.304],[6326.0,0.242,8.38,2070.042,331.804,41.922,96.868,70.942,0.67,46.557,1.747,3.014,3.452,10.145,18.091,3.547,1.066,3.31,0.203,-8.066,7.447,11.723,4.084,0.099,2.273],[6326.5,0.289,8.412,2079.122,331.782,41.067,98.434,75.981,0.538,45.282,1.836,3.014,3.386,17.467,8.971,4.501,1.105,4.064,0.164,-8.128,16.485,12.221,4.102,0.089,2.309],[6327.0,0.284,8.426,2079.845,324.514,38.72,101.364,85.852,0.448,48.043,1.89,3.082,3.406,7.687,2.725,6.128,1.171,5.183,0.107,-8.721,13.083,13.538,4.285,0.088,2.373],[6327.5,0.345,8.422,2075.456,312.958,36.415,105.28,96.497,0.382,53.184,1.811,3.195,3.56,4.962,3.522,7.677,1.246,6.152,0.085,-10.394,3.892,14.883,4.432,0.1,2.405],[6328.0,0.345,8.338,2075.502,307.113,36.941,109.461,105.739,0.432,57.858,1.665,3.256,3.891,4.319,3.314,7.098,1.206,5.852,0.115,-13.257,-5.435,15.238,4.705,0.097,2.371],[6328.5,0.266,8.238,2075.97,318.637,39.602,113.114,105.131,0.543,61.148,1.299,3.138,4.262,4.077,1.715,5.604,1.123,4.967,0.25,-17.45,-15.368,15.244,5.045,0.035,2.242],[6329.0,0.238,8.166,2073.054,355.684,41.931,115.575,86.679,0.585,59.52,1.106,2.811,4.198,6.118,8.29,5.02,1.081,4.649,0.581,-23.251,-25.281,14.998,5.148,-0.058,2.016],[6329.5,0.238,8.171,2079.389,417.255,43.49,116.375,58.777,0.572,47.767,1.379,2.397,3.516,30.041,58.776,4.157,0.919,4.455,1.055,-30.795,-30.603,12.126,4.82,-0.095,1.748],[6330.0,0.235,8.208,2089.253,492.599,46.296,115.31,37.827,0.536,33.104,1.846,2.03,2.671,79.317,194.045,2.879,0.86,3.595,1.431,-39.899,-27.752,7.913,3.67,-0.065,1.527],[6330.5,0.236,8.193,2079.817,575.342,50.772,112.5,28.662,0.538,26.65,2.224,1.738,1.992,12.945,177.684,2.132,1.672,1.906,1.455,-49.723,-24.298,5.775,1.85,-0.015,1.434],[6331.0,0.215,8.131,2058.192,669.938,53.143,108.258,25.692,0.581,24.105,2.397,1.493,1.531,4.682,21.136,1.714,2.7,0.677,1.098,-58.384,-27.274,5.078,0.739,0.019,1.505],[6331.5,0.14,8.051,2054.02,777.052,50.933,103.149,23.949,0.59,21.568,2.296,1.287,1.246,3.01,-0.52,1.592,3.11,0.513,0.625,-64.33,-33.114,4.799,0.574,0.032,1.706],[6332.0,0.158,7.997,2064.971,889.842,44.494,98.118,22.819,0.592,20.733,1.952,1.124,1.07,2.719,2.277,1.512,3.221,0.47,0.303,-67.583,-38.424,4.573,0.527,0.033,1.932],[6332.5,0.161,7.961,2063.357,1001.85,36.904,94.241,23.289,0.601,20.532,1.703,0.998,0.96,2.551,2.33,1.428,3.237,0.441,0.165,-69.183,-42.734,4.377,0.502,0.035,2.079],[6333.0,0.15,7.951,2049.309,1101.908,31.922,92.238,24.759,0.607,20.597,1.636,0.908,0.895,2.224,1.718,1.366,3.08,0.445,0.118,-69.883,-42.153,4.233,0.512,0.045,2.139],[6333.5,0.147,7.952,2054.518,1176.306,31.294,92.093,25.949,0.612,20.972,1.632,0.85,0.861,1.993,1.309,1.307,2.644,0.499,0.102,-69.88,-36.823,4.084,0.58,0.051,2.157],[6334.0,0.143,7.956,2081.13,1224.473,32.327,93.127,26.258,0.647,22.203,1.619,0.817,0.841,2.045,1.413,1.232,2.293,0.538,0.097,-69.478,-33.949,4.062,0.66,0.059,2.16],[6334.5,0.124,7.955,2092.896,1255.888,32.505,94.403,25.918,0.692,24.005,1.592,0.796,0.828,2.483,1.808,1.194,2.251,0.531,0.095,-69.408,-35.129,4.217,0.695,0.066,2.159],[6335.0,0.118,7.951,2083.979,1276.368,31.969,95.252,26.272,0.696,24.832,1.605,0.783,0.823,2.58,2.284,1.227,2.277,0.539,0.093,-69.692,-40.004,4.36,0.711,0.066,2.16],[6335.5,0.136,7.945,2075.331,1291.479,31.448,95.567,28.104,0.668,24.279,1.703,0.774,0.824,1.835,2.014,1.302,2.219,0.587,0.09,-69.261,-42.625,4.436,0.742,0.059,2.163],[6336.0,0.169,7.944,2071.047,1309.711,31.13,95.61,29.832,0.627,23.492,1.842,0.764,0.815,1.569,1.396,1.414,2.145,0.66,0.088,-67.85,-38.849,4.513,0.781,0.052,2.166],[6336.5,0.192,7.945,2064.25,1332.129,30.81,95.628,30.041,0.572,23.018,1.863,0.751,0.792,2.049,2.835,1.588,2.067,0.771,0.086,-66.829,-37.031,4.613,0.829,0.049,2.168],[6337.0,0.209,7.951,2061.835,1345.589,30.193,95.718,29.603,0.514,22.364,1.734,0.743,0.769,2.915,3.673,1.787,1.913,0.943,0.085,-66.88,-41.14,4.668,0.91,0.052,2.169],[6337.5,0.209,7.947,2066.585,1337.962,29.636,95.867,29.784,0.472,21.522,1.609,0.747,0.761,2.781,2.511,1.931,1.706,1.139,0.085,-66.899,-43.422,4.648,1.015,0.057,2.169],[6338.0,0.209,7.942,2065.876,1313.376,29.581,95.996,30.695,0.459,21.237,1.593,0.761,0.77,2.094,1.647,1.958,1.53,1.284,0.087,-65.965,-43.637,4.587,1.117,0.057,2.169],[6338.5,0.209,7.947,2052.35,1278.298,30.241,96.084,32.146,0.518,20.855,1.668,0.782,0.793,1.926,1.661,1.65,1.191,1.406,0.089,-64.168,-46.252,4.219,1.391,0.051,2.17],[6339.0,0.232,7.946,2031.698,1225.134,31.154,96.216,34.658,0.616,20.415,1.751,0.816,0.838,1.981,2.463,1.195,0.8,1.507,0.092,-61.697,-42.106,3.664,1.765,0.049,2.17],[6339.5,0.23,7.945,2015.358,1148.234,32.073,96.404,40.111,0.682,21.31,1.774,0.871,0.921,2.016,2.753,0.973,0.634,1.536,0.097,-58.74,-31.862,3.376,1.985,0.056,2.175],[6340.0,0.209,7.943,2012.356,1056.199,33.218,96.499,49.827,0.798,23.808,1.84,0.947,1.043,2.113,2.666,0.841,0.546,1.543,0.099,-55.372,-31.221,3.377,2.336,0.064,2.188],[6340.5,0.209,7.943,2023.04,959.306,34.302,96.341,62.214,0.991,28.862,2.056,1.042,1.19,2.323,3.211,0.808,0.505,1.595,0.095,-50.599,-42.794,4.11,3.004,0.071,2.216],[6341.0,0.209,7.942,2036.662,861.791,34.782,95.97,73.199,1.117,37.66,2.384,1.16,1.36,2.569,3.209,1.158,0.669,1.71,0.083,-43.73,-47.391,6.654,3.621,0.08,2.263],[6341.5,0.233,7.948,2043.071,766.891,34.633,95.749,79.939,1.148,46.93,2.699,1.304,1.549,2.845,3.58,1.628,0.917,1.776,0.064,-36.764,-33.285,9.557,3.864,0.095,2.324],[6342.0,0.216,7.952,2040.612,679.232,34.282,96.161,82.398,1.147,51.631,2.874,1.472,1.733,3.394,4.27,1.836,1.015,1.808,0.05,-32.035,-15.277,10.747,3.928,0.112,2.377],[6342.5,0.194,7.952,2031.479,606.511,33.726,97.347,82.68,1.129,52.433,2.866,1.649,1.873,4.126,4.501,1.916,1.017,1.885,0.045,-28.962,-8.085,11.049,4.032,0.126,2.403],[6343.0,0.192,7.949,2024.682,553.839,33.412,98.844,83.67,1.116,52.874,2.761,1.806,1.952,4.263,4.608,2.002,1.027,1.949,0.046,-24.985,-3.762,11.408,4.12,0.138,2.409],[6343.5,0.192,7.941,2030.451,516.123,34.328,99.857,87.707,1.113,54.352,2.694,1.938,1.997,3.46,3.329,2.079,1.03,2.019,0.048,-18.594,9.04,11.819,4.259,0.15,2.414],[6344.0,0.217,7.941,2043.77,485.205,36.76,99.822,93.399,1.176,58.013,2.632,2.061,2.064,2.42,2.107,2.105,1.061,1.989,0.047,-10.656,20.963,12.64,4.419,0.163,2.422],[6344.5,0.286,8.007,2053.7,459.417,39.245,98.836,96.958,1.255,62.544,2.547,2.177,2.185,1.882,1.602,2.117,1.131,1.874,0.049,-3.981,23.327,13.564,4.45,0.175,2.423],[6345.0,0.535,8.077,2060.657,441.195,40.865,97.558,96.704,1.264,64.843,2.486,2.267,2.358,1.82,1.585,2.152,1.165,1.847,0.054,-0.529,19.143,13.887,4.422,0.182,2.417],[6345.5,0.614,8.085,2069.852,430.712,40.857,96.642,93.441,1.249,65.098,2.53,2.322,2.537,2.181,2.233,2.194,1.179,1.861,0.051,0.427,10.229,13.999,4.406,0.181,2.426],[6346.0,0.494,8.093,2073.704,425.373,39.709,96.226,89.247,1.223,63.416,2.639,2.351,2.645,2.919,2.802,2.235,1.173,1.907,0.032,0.726,-1.314,13.953,4.415,0.166,2.47],[6346.5,0.55,8.149,2065.428,421.674,38.49,95.908,86.92,1.135,58.174,2.788,2.372,2.637,3.704,3.179,2.265,1.057,2.172,0.012,0.914,-7.627,13.121,4.634,0.142,2.537],[6347.0,0.511,8.247,2051.408,417.291,37.096,95.344,87.987,1.057,53.195,2.893,2.396,2.554,4.04,3.527,2.24,0.916,2.457,0.003,0.325,-6.601,12.091,4.908,0.111,2.591],[6347.5,0.402,8.306,2044.72,412.911,37.028,94.758,91.552,1.116,54.492,2.805,2.422,2.471,3.556,2.958,2.12,0.935,2.289,0.002,-1.134,-1.862,12.031,4.794,0.078,2.605],[6348.0,0.362,8.327,2051.794,412.121,38.424,94.743,94.576,1.24,59.175,2.649,2.426,2.422,2.778,2.083,1.952,1.027,1.91,0.004,-2.778,2.642,12.316,4.459,0.055,2.583],[6348.5,0.319,8.315,2063.102,418.356,40.205,95.723,93.115,1.302,61.533,2.573,2.39,2.391,2.346,1.802,1.86,1.072,1.736,0.011,-4.748,2.747,12.371,4.282,0.054,2.54],[6349.0,0.295,8.232,2065.681,430.3,41.04,97.649,86.46,1.332,60.31,2.449,2.324,2.355,2.434,1.651,1.82,1.104,1.65,0.026,-8.138,-3.914,12.369,4.162,0.068,2.488],[6349.5,0.137,8.082,2061.631,441.088,40.143,100.185,77.161,1.289,55.467,2.205,2.267,2.324,2.985,1.964,1.751,1.147,1.528,0.047,-13.306,-7.075,11.541,3.741,0.091,2.433],[6350.0,0.112,7.966,2062.465,443.36,37.831,103.16,67.365,1.187,48.518,2.041,2.256,2.33,3.667,2.57,1.653,1.178,1.404,0.074,-19.623,5.773,10.041,3.166,0.115,2.38],[6350.5,0.106,7.923,2074.417,433.598,36.406,106.631,61.02,1.143,43.695,2.065,2.306,2.405,3.878,2.575,1.585,1.182,1.341,0.108,-25.71,24.455,9.249,2.904,0.12,2.333],[6351.0,0.101,7.894,2088.621,414.015,35.079,110.342,63.943,1.147,44.572,2.152,2.415,2.583,3.583,2.343,1.578,1.168,1.353,0.144,-29.338,28.961,9.247,2.941,0.097,2.294],[6351.5,0.155,7.908,2083.95,393.12,34.67,113.489,74.377,1.135,49.516,2.172,2.544,2.888,3.021,2.206,1.69,1.123,1.51,0.197,-29.041,19.84,9.778,3.24,0.064,2.248],[6352.0,0.211,7.926,2067.424,381.379,36.847,115.229,80.704,1.117,53.64,2.111,2.622,3.255,2.813,2.452,1.829,1.097,1.667,0.317,-27.019,12.211,10.421,3.524,0.035,2.155],[6352.5,0.21,7.934,2065.78,383.69,41.968,115.194,74.316,1.058,51.105,1.949,2.606,3.452,3.801,0.813,1.881,1.077,1.749,0.6,-27.579,9.457,10.14,3.491,-0.008,1.983],[6353.0,0.346,7.933,2077.191,398.397,47.536,113.469,58.152,0.878,41.799,2.013,2.51,3.303,7.875,18.793,1.966,1.117,1.771,1.056,-32.662,4.018,8.735,2.941,-0.038,1.767],[6353.5,0.292,7.941,2081.827,422.501,50.759,110.24,41.404,0.632,31.103,2.294,2.367,2.952,12.537,73.13,2.054,1.247,1.664,1.418,-40.675,-9.952,6.595,2.008,-0.021,1.609],[6354.0,0.293,7.946,2075.444,456.253,50.166,105.614,30.975,0.466,23.062,2.541,2.192,2.582,8.299,64.571,2.038,1.343,1.527,1.343,-48.932,-28.991,4.874,1.364,0.005,1.598],[6354.5,0.228,7.928,2066.775,503.867,45.113,99.98,27.326,0.429,19.411,2.557,1.985,2.235,4.949,12.211,1.968,1.385,1.421,0.868,-55.355,-44.615,4.314,1.156,0.002,1.751],[6355.0,0.264,7.905,2060.217,568.206,36.804,94.374,26.23,0.421,18.077,2.206,1.76,1.911,3.906,2.415,1.911,1.34,1.429,0.397,-59.314,-54.875,4.113,1.139,0.006,1.989],[6355.5,0.226,7.884,2052.21,642.784,29.012,90.124,24.737,0.401,16.601,1.815,1.556,1.634,3.663,2.965,1.765,1.151,1.55,0.186,-61.668,-59.114,3.624,1.175,0.039,2.159],[6356.0,0.266,7.865,2036.637,719.23,25.145,88.012,23.232,0.385,14.551,1.678,1.39,1.424,3.569,2.432,1.495,0.868,1.752,0.137,-63.461,-52.237,2.946,1.276,0.054,2.2],[6356.5,0.324,7.865,2024.426,796.1,26.507,87.84,22.41,0.396,13.496,1.631,1.256,1.275,3.407,2.604,1.246,0.694,1.794,0.13,-64.877,-37.54,2.507,1.342,0.059,2.19],[6357.0,0.338,7.872,2025.123,869.902,28.749,88.739,21.963,0.422,13.483,1.622,1.15,1.17,3.105,2.581,1.074,0.629,1.706,0.123,-65.727,-24.455,2.304,1.361,0.064,2.183],[6357.5,0.323,7.85,2031.605,933.356,29.321,89.799,21.847,0.434,13.682,1.582,1.071,1.095,2.668,2.011,1.0,0.593,1.688,0.115,-65.763,-11.717,2.215,1.388,0.065,2.183],[6358.0,0.343,7.859,2040.12,986.625,29.65,90.542,21.742,0.431,14.576,1.531,1.014,1.042,2.004,1.564,1.106,0.673,1.65,0.108,-64.453,1.333,2.43,1.348,0.06,2.183],[6358.5,0.324,7.876,2049.109,1031.241,29.994,90.972,21.349,0.403,15.694,1.543,0.97,0.998,1.54,1.066,1.429,0.853,1.665,0.103,-62.068,4.276,2.895,1.266,0.056,2.185],[6359.0,0.324,7.889,2055.356,1059.344,29.701,91.264,21.057,0.34,16.175,1.609,0.944,0.962,1.517,1.681,2.046,1.054,1.915,0.098,-60.478,-6.925,3.439,1.214,0.053,2.189],[6359.5,0.346,7.875,2052.257,1067.324,29.31,91.517,21.418,0.283,16.655,1.691,0.937,0.946,1.53,2.555,2.774,1.287,2.149,0.096,-60.145,-20.097,3.97,1.151,0.046,2.192],[6360.0,0.318,7.87,2040.469,1066.156,29.434,91.72,22.665,0.264,17.515,1.785,0.938,0.959,1.107,1.563,3.233,1.508,2.147,0.097,-58.96,-21.142,4.358,1.075,0.034,2.191],[6360.5,0.304,7.89,2029.837,1069.899,29.57,91.847,24.978,0.264,18.407,1.866,0.935,0.98,0.857,0.705,3.409,1.669,2.047,0.096,-56.864,-13.751,4.59,1.022,0.031,2.191],[6361.0,0.295,7.893,2035.091,1083.229,29.886,91.931,27.702,0.308,19.676,1.875,0.923,0.983,1.189,1.23,3.073,1.8,1.717,0.093,-56.701,-14.372,4.693,0.969,0.049,2.196],[6361.5,0.285,7.881,2054.394,1098.153,30.112,92.08,28.983,0.394,20.966,1.863,0.911,0.964,2.215,2.628,2.362,1.822,1.294,0.091,-58.98,-23.661,4.625,0.942,0.077,2.197],[6362.0,0.323,7.869,2067.125,1104.588,30.083,92.355,28.635,0.449,21.491,1.879,0.905,0.94,2.539,2.48,1.977,1.749,1.13,0.093,-61.151,-29.739,4.523,0.96,0.084,2.191],[6362.5,0.329,7.898,2060.489,1101.815,29.756,92.637,28.392,0.471,21.607,1.898,0.908,0.923,2.136,2.145,1.874,1.698,1.104,0.096,-61.402,-30.191,4.502,0.984,0.064,2.185],[6363.0,0.371,7.928,2045.291,1092.127,29.576,92.766,29.235,0.489,21.764,1.905,0.916,0.916,1.846,1.868,1.788,1.62,1.106,0.096,-59.452,-27.729,4.465,1.025,0.043,2.188],[6363.5,0.467,7.96,2034.757,1069.5,30.046,92.754,29.724,0.52,21.964,1.927,0.935,0.926,1.752,1.984,1.649,1.485,1.111,0.096,-55.71,-26.074,4.363,1.093,0.051,2.192],[6364.0,0.478,8.041,2026.902,1028.525,30.955,92.8,29.698,0.559,21.95,1.94,0.972,0.961,1.866,2.0,1.522,1.375,1.108,0.1,-51.622,-31.184,4.34,1.174,0.066,2.191],[6364.5,0.479,8.176,2020.489,971.386,31.437,93.165,32.018,0.583,21.775,1.961,1.029,1.031,1.98,1.782,1.461,1.294,1.129,0.107,-48.243,-39.76,4.348,1.248,0.066,2.189],[6365.0,0.472,8.254,2027.286,898.08,31.296,93.866,39.44,0.612,23.012,2.071,1.113,1.14,1.976,1.71,1.401,1.222,1.146,0.112,-43.781,-42.249,4.378,1.331,0.06,2.196],[6365.5,0.331,8.261,2051.23,812.16,31.206,94.402,51.513,0.68,28.826,2.303,1.231,1.294,2.02,2.065,1.623,1.383,1.174,0.105,-35.4,-39.274,5.717,1.512,0.063,2.229],[6366.0,0.322,8.28,2068.685,722.91,31.799,94.139,65.155,0.776,39.256,2.561,1.383,1.499,2.215,2.543,2.151,1.661,1.298,0.082,-24.45,-35.099,8.621,1.909,0.067,2.295],[6366.5,0.244,8.315,2056.082,632.605,33.123,93.064,77.08,0.895,50.863,2.78,1.581,1.757,2.309,2.96,2.513,1.79,1.413,0.056,-14.645,-29.654,11.514,2.388,0.073,2.37],[6367.0,0.257,8.327,2033.938,543.838,34.686,91.86,84.133,0.958,59.076,3.04,1.839,2.073,2.185,2.636,2.661,1.795,1.488,0.041,-7.156,-25.712,13.02,2.701,0.084,2.424],[6367.5,0.304,8.339,2029.443,472.858,37.349,91.303,83.126,0.948,60.803,3.28,2.115,2.417,2.219,1.973,2.734,1.745,1.567,0.034,-2.028,-28.088,13.239,2.816,0.09,2.457],[6368.0,0.332,8.364,2032.276,433.425,40.244,91.673,74.058,0.801,52.925,3.32,2.307,2.665,2.965,2.155,3.097,1.476,2.311,0.026,-0.098,-34.877,12.238,3.175,0.084,2.487],[6368.5,0.266,8.365,2029.665,422.584,40.206,92.648,64.367,0.558,40.566,3.234,2.366,2.69,4.904,4.503,3.795,1.071,3.755,0.017,-0.018,-41.71,10.453,3.695,0.065,2.518],[6369.0,0.303,8.344,2032.457,431.038,36.608,93.93,63.232,0.459,35.257,2.973,2.32,2.536,5.458,7.242,4.13,0.924,4.471,0.012,0.956,-48.162,9.654,3.877,0.048,2.538],[6369.5,0.386,8.329,2036.965,452.033,32.503,95.85,71.534,0.497,36.624,2.667,2.212,2.296,3.6,4.725,3.874,0.864,4.494,0.013,3.393,-54.506,9.754,4.229,0.055,2.528],[6370.0,0.429,8.321,2030.509,483.417,31.846,98.859,82.787,0.661,41.475,2.708,2.069,2.027,2.59,2.169,3.168,0.786,4.037,0.021,5.947,-57.407,10.248,4.861,0.077,2.495],[6370.5,0.564,8.308,2017.726,522.067,35.118,102.722,91.515,0.915,49.838,3.001,1.915,1.785,2.161,1.999,2.505,0.84,3.05,0.029,7.754,-51.692,11.452,5.073,0.086,2.463],[6371.0,0.659,8.291,2007.731,563.357,39.741,106.587,96.957,1.107,59.092,3.217,1.775,1.607,1.77,1.472,2.32,1.032,2.291,0.033,8.931,-43.318,13.117,4.757,0.082,2.443],[6371.5,0.571,8.284,2007.544,606.791,44.62,109.572,100.085,1.221,65.963,3.202,1.648,1.49,1.497,1.11,2.322,1.244,1.884,0.035,9.801,-41.693,14.47,4.338,0.084,2.43],[6372.0,0.666,8.283,2020.5,652.665,48.684,111.347,101.741,1.325,70.269,3.002,1.532,1.414,1.416,1.068,2.259,1.422,1.601,0.036,10.378,-44.419,15.264,3.999,0.091,2.418],[6372.5,0.613,8.29,2032.831,694.102,48.592,112.285,102.099,1.432,73.994,2.793,1.441,1.361,1.417,1.01,2.192,1.648,1.347,0.037,10.733,-44.697,16.02,3.634,0.094,2.408],[6373.0,0.652,8.29,2032.594,720.245,45.868,112.795,101.732,1.526,77.415,2.651,1.388,1.319,1.443,1.017,2.156,1.923,1.13,0.036,10.988,-41.277,16.792,3.258,0.091,2.407],[6373.5,0.708,8.286,2029.847,723.776,43.77,112.714,100.884,1.58,79.519,2.511,1.382,1.298,1.475,1.035,2.13,2.11,1.011,0.033,11.124,-32.4,17.185,3.025,0.092,2.417],[6374.0,0.768,8.267,2034.171,702.554,46.521,111.703,100.05,1.581,80.164,2.456,1.423,1.313,1.466,1.096,2.137,2.157,0.991,0.03,11.171,-21.264,17.255,2.969,0.099,2.43],[6374.5,0.772,8.257,2040.926,662.489,50.88,109.892,99.289,1.51,79.435,2.521,1.509,1.384,1.372,1.103,2.25,2.158,1.042,0.03,11.291,-16.49,17.302,2.975,0.107,2.438],[6375.0,0.472,8.247,2044.287,619.207,51.709,108.066,98.314,1.381,77.22,2.543,1.615,1.529,1.233,1.041,2.446,2.111,1.162,0.031,11.54,-21.17,17.194,3.025,0.112,2.439],[6375.5,0.482,8.217,2043.199,590.147,49.042,107.233,95.648,1.288,74.803,2.561,1.694,1.718,1.255,0.947,2.579,2.033,1.27,0.031,11.76,-29.134,16.96,3.097,0.121,2.444],[6376.0,0.226,8.213,2039.402,579.597,45.261,107.724,89.202,1.23,71.963,2.8,1.725,1.821,2.016,1.164,2.678,1.997,1.342,0.022,11.772,-29.269,16.797,3.122,0.138,2.476],[6376.5,0.673,8.236,2039.251,578.133,40.754,108.828,79.918,1.134,65.551,3.272,1.73,1.757,5.136,10.778,2.728,1.834,1.504,0.009,11.656,-21.176,15.801,3.209,0.152,2.538],[6377.0,0.659,8.238,2041.877,576.591,36.712,109.687,74.835,1.026,57.853,3.499,1.734,1.628,7.469,18.214,2.705,1.569,1.741,0.004,11.701,-20.663,14.184,3.366,0.155,2.575],[6377.5,0.545,8.237,2042.788,578.166,34.118,110.246,78.982,1.0,56.479,3.131,1.73,1.531,2.805,7.518,2.697,1.505,1.796,0.009,12.037,-33.532,13.771,3.398,0.149,2.537]]}'), orient='split')
core_raw = pd.read_json(io.StringIO('{"columns":["sample","core","core_depth","log_depth","sand","kh800","airh800","kv800","airv800","phi800","kh3500","airh3500","kv3500","airv3500","phi3500","So","Sw","grain_density","comments"],"index":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56],"data":[[1,1,5509.2,5513.2,"A7",119,132,null,null,28,64.7,70.3,null,null,26,4,92.6,2.66,null],["1V",1,5509.2,5513.2,"A7",null,null,52.1,null,null,null,null,12.8,null,null,null,null,null,null],[2,1,5511.6,5515.6,"A7",380,400,null,null,29,269,280,null,null,27.3,31.1,60.7,2.65,null],["2V",1,5511.6,5515.6,"A7",null,null,318,null,null,null,null,251,null,null,null,null,null,null],[3,1,5512.4,5516.4,"A7",2299,2345,null,null,34.2,1911,1935,null,null,33,44.2,42.7,2.64,null],["3V",1,5512.4,5516.4,"A7",null,null,1452,null,null,null,null,1266,null,null,null,null,null,null],[4,1,5513.3,5517.3,"A7",1465,1500,null,null,31.9,1260,1282,null,null,31.1,47.6,15.5,2.67,null],["4V",1,5513.3,5517.3,"A7",null,null,1039,null,null,null,null,627,null,null,null,null,null,null],[5,1,5515.2,5519.2,"A7",977,1006,null,null,33.8,806,830,null,null,32.8,28.7,54.7,2.67,null],[6,1,5517.1,5521.1,"A7",16.9,18.8,null,null,20,8.08,9.37,null,null,18.8,1.8,81.7,2.68,null],["6V",1,5517.1,5521.1,"A7",null,null,7.21,null,null,null,null,1.02,null,null,null,null,null,null],[7,1,5525.7,5529.7,"A8",745,784,null,null,35.2,461,492,null,null,33.1,14,67,2.65,null],["7V",1,5525.7,5529.7,"A8",null,null,589,null,null,null,null,340,null,null,null,null,null,null],[8,1,5526.8,5530.8,"A8",1188,1234,null,null,33.7,971,1004,null,null,32.4,30.4,35,2.67,null],[9,1,5527.7,5531.7,"A8",2037,2101,null,null,35.2,1573,1609,null,null,33.1,11.9,44.2,2.64,null],["9V",1,5527.7,5531.7,"A8",null,null,1000,null,null,null,null,746,null,null,null,null,null,null],[10,1,5529.5,5533.5,"A8",846,880,null,null,32.2,706,734,null,null,31.3,26.3,45,2.68,null],[11,1,5530.4,5534.4,"A8",545,573,null,null,34.9,464,489,null,null,32.8,7.8,78.2,2.68,null],[12,2,5531.1,5538.1,"A8",1140,1187,null,null,32.6,951,982,null,null,31.8,51.1,20.8,2.68,null],["12V",2,5531.1,5538.1,"A8",null,null,660,null,null,null,null,561,null,null,null,null,null,null],[13,2,5532.3,5539.3,"A8",962,993,null,null,33.7,821,846,null,null,31.5,16,73.8,2.64,null],["13V",2,5532.3,5539.3,"A8",null,null,669,null,null,null,null,541,null,null,null,null,null,null],[14,2,5533.4,5540.4,"A8",768,803,null,null,33.1,631,658,null,null,32,24.9,52.6,2.66,null],["14V",2,5533.4,5540.4,"A8",null,null,339,null,null,null,null,264,null,null,null,null,null,null],[15,2,5534.6,5541.6,"A8",1240,1270,null,null,32.1,1077,1102,null,null,31.2,26.8,63,2.63,null],["15V",2,5534.6,5541.6,"A8",null,null,1240,null,null,null,null,1078,null,null,null,null,null,null],[16,2,5535.6,5542.6,"A8",1154,1193,null,null,35.4,951,983,null,null,34.2,41.1,43.2,2.65,null],["16V",2,5535.6,5542.6,"A8",null,null,485,null,null,null,null,389,null,null,null,null,null,null],[17,2,5544.5,5551.5,"A9",19.5,21.2,null,null,10.9,6.19,6.97,null,null,9.9,23.3,55.7,2.67,null],["17V",2,5544.5,5551.5,"A9",null,null,35.3,null,null,null,null,17.2,null,null,null,null,null,null],[18,2,5546.5,5553.5,"A9",2153,2341,null,null,18.3,1536,1544,null,null,16.8,16.1,43.7,2.68,null],["18V",2,5546.5,5553.5,"A9",null,null,22.7,null,null,null,null,21.2,null,null,null,null,null,null],[19,2,5549.9,5556.9,"A9",4885,5321,null,null,29,3260,3497,null,null,27.1,9.7,76.3,2.64,null],[20,3,5561.7,5568.7,"A10",1595,1737,null,null,23.5,1290,1354,null,null,21.6,10.7,47.2,2.63,null],[21,4,5980.9,5984.9,"E2",3672,3911,null,null,34.1,1361,1520,null,null,31.1,32.7,35.3,2.63,null],[22,4,5982.3,5986.3,"E2",3580,3628,null,null,32.2,2982,3024,null,null,32.2,49.2,20,2.64,null],[23,4,5983.8,5987.8,"E2",1274,1299,null,null,31.9,1077,1100,null,null,31.9,49.5,22.7,2.64,null],["23V",4,5983.8,5987.8,"E2",null,null,1220,null,null,null,null,1034,null,null,null,null,null,null],[24,4,5985.3,5989.3,"E2",378,394,null,null,32.3,308,323,null,null,31,33.4,33.9,2.66,null],[25,5,5990.8,5993.8,"E2",1463,1497,null,null,35.4,1194,1223,null,null,34.2,44.1,35.9,2.63,null],["25V",5,5990.8,5993.8,"E2",null,null,502,null,null,null,null,433,null,null,null,null,null,null],[26,5,5991.9,5994.9,"E2",4.7,5.57,null,null,18.6,3.99,4.73,null,null,17.8,5.6,60.7,2.68,null],["26V",5,5991.9,5994.9,"E2",null,null,0.833,null,null,null,null,0.038,null,null,null,null,null,null],[27,5,5992.9,5995.9,"E2",1.54,1.75,null,null,15.6,0.097,0.137,null,null,14.6,10.5,75.6,2.68,null],["27V",5,5992.9,5995.9,"E2",null,null,0.386,null,null,null,null,0.011,null,null,null,null,null,null],[28,5,5999.9,6002.9,"E3",">9999",">9999",null,null,27.9,">9999",">9999",null,null,26.1,16.1,57,2.63,null],["28V",5,5999.9,6002.9,"E3",null,null,">9999",null,null,null,null,">9999",null,null,null,null,null,null],[29,5,6000.8,6003.8,"E3",">9999",">9999",null,null,28.5,">9999",">9999",null,null,27.1,17.2,52.6,2.63,null],["29V",5,6000.8,6003.8,"E3",null,null,">9999",null,null,null,null,2309,null,null,null,null,null,null],[30,5,6002.1,6005.1,"E3",1847,1878,null,null,33,1473,1513,null,null,31.3,64.9,32.5,2.63,null],[31,5,6003.1,6006.1,"E3",2676,2689,null,null,31.8,1649,1689,null,null,27.6,67.2,16.4,2.63,null],["31V",5,6003.1,6006.1,"E3",null,null,2074,null,null,null,null,1706,null,null,null,null,null,null],[32,5,6004.2,6007.2,"E3",1722,1759,null,null,32.8,1614,1653,null,null,32.3,67.8,20.6,2.64,null],["32V",5,6004.2,6007.2,"E3",null,null,1439,null,null,null,null,1140,null,null,null,null,null,null],[33,5,6006.1,6009.1,"E3",907,942,null,null,32.8,868,904,null,null,32.4,48.9,27.3,2.63,null],["33V",5,6006.1,6009.1,"E3",null,null,703,null,null,null,null,683,null,null,null,null,null,null],[34,5,6008.1,6011.1,"E3",0.386,0.528,null,null,15.3,0.051,0.094,null,null,14.2,0,98.6,2.67,null]]}'), orient='split')
core = core_raw.copy()
for col in core.columns:
    if col not in ['sample', 'sand', 'comments']:
        core[col + '_lower_limit'] = core[col].astype(str).str.startswith('>')
        core[col] = pd.to_numeric(core[col].astype(str).str.removeprefix('>'), errors='coerce')
horizontal = core[~core['sample'].astype(str).str.endswith('V')].copy()
vertical = core[core['sample'].astype(str).str.endswith('V')].copy()
# Use the supplied log_depth; do not add the core shifts a second time.


## Inspect the measurements
All source columns are preserved in `logs` and `core_raw`. Censored flags live in `core`.

In [ ]:
display(core_raw)
summary = horizontal[['phi800','phi3500','kh3500','So','Sw','grain_density']].copy()
summary.loc[horizontal.kh3500_lower_limit, 'kh3500'] = np.nan
# kh3500 statistics below include only the 32 uncensored values.
display(summary.describe())


In [ ]:
plt.rcParams.update({'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9,
                     'font.family': 'DejaVu Sans', 'savefig.facecolor': 'white'})
BLUE='#2563eb'; RED='#dc4545'; GREEN='#278346'; PURPLE='#8b46c7'; GOLD='#c98a15'

def panel(top=5400, bottom=6300, title='Project 2 | Basic well logs and core measurements'):
    fig, axes = plt.subplots(1, 8, figsize=(19, 12), sharey=True,
                             gridspec_kw={'width_ratios':[1.05,1.05,1.25,1.2,.75,1,1.2,1]})
    fig.subplots_adjust(left=.055,right=.985,bottom=.095,top=.80,wspace=.12)
    d=logs[(logs.DEPT>=top)&(logs.DEPT<=bottom)]
    h=horizontal[horizontal.log_depth.between(top,bottom)]
    v=vertical[vertical.log_depth.between(top,bottom)]
    major=100 if bottom-top>300 else 10
    for ax in axes:
        ax.set_ylim(bottom,top)
        ax.yaxis.set_major_locator(MultipleLocator(major))
        ax.yaxis.set_minor_locator(MultipleLocator(major/5))
        ax.grid(which='major',color='#bbc4cb',lw=.55)
        ax.grid(which='minor',axis='y',color='#e3e7eb',lw=.4)
        ax.tick_params(axis='both',labelsize=8)
        ax.tick_params(axis='x',top=True,labeltop=True,bottom=False,labelbottom=False)
        ax.set_facecolor('#fcfcfd')
        for sp in ax.spines.values(): sp.set_color('#9aa7b3')
    axes[0].set_ylabel('Measured depth (ft) | supplied log datum')
    def header(ax,title,lines):
        ax.text(.5,1.16,title,ha='center',transform=ax.transAxes,weight='bold',fontsize=10)
        for i,(label,color) in enumerate(lines):
            ax.text(.5,1.12-i*.035,label,color=color,ha='center',transform=ax.transAxes,fontsize=8)
    def norm(ax,key,lo,hi,color):
        ax.plot((d[key]-lo)/(hi-lo),d.DEPT,color=color,lw=.7)
    ax=axes[0];ax.set_xlim(0,1);ax.set_xticks([])
    norm(ax,'TH',0,30,GOLD);norm(ax,'U',0,15,PURPLE);norm(ax,'K',0,3,'#444444')
    header(ax,'Spectral gamma',[('Th: 0 to 30 ppm',GOLD),('U: 0 to 15 ppm',PURPLE),('K: 0 to 3 %','#444444')])
    ax=axes[1];ax.set_xlim(0,1);ax.set_xticks([])
    norm(ax,'GR',0,150,GREEN);norm(ax,'CAL',6,16,'#555555')
    header(ax,'Gamma / caliper',[('GR: 0 to 150 API',GREEN),('CAL: 6 to 16 in','#555555')])
    ax=axes[2];ax.set_xscale('log');ax.set_xlim(.2,2000)
    for key,color in [('RIPD',RED),('RIPM',GOLD),('RSFL',BLUE)]:
        ax.plot(d[key].where(d[key]>0),d.DEPT,c=color,lw=.75)
    ax.xaxis.set_major_locator(LogLocator(base=10,numticks=5))
    header(ax,'Resistivity (ohm m)',[('RIPD: deep',RED),('RIPM: medium',GOLD),('RSFL: shallow',BLUE)])
    ax=axes[3];ax.set_xlim(0,1);ax.set_xticks([])
    norm(ax,'ZDEN',1.4,2.8,RED);norm(ax,'CNCF',60,-10,BLUE)
    header(ax,'Density / neutron',[('ZDEN: 1.4 to 2.8 g/cc',RED),('CNCF: 60 to -10 p.u.',BLUE)])
    ax=axes[4];ax.set_xlim(0,6);ax.set_xticks([0,3,6]);ax.plot(d.PE,d.DEPT,c=PURPLE,lw=.8)
    header(ax,'Photoelectric',[('PE (LAS: BARN)',PURPLE)])
    ax=axes[5];ax.set_xlim(0,40);ax.set_xticks([0,20,40])
    ax.scatter(h.phi800,h.log_depth,s=21,edgecolor='#777',facecolor='none',lw=.8)
    ax.scatter(h.phi3500,h.log_depth,s=13,c=BLUE)
    header(ax,'Core porosity (%)',[('800 psia: open circles','#777'),('3500 psia: filled',BLUE)])
    ax=axes[6];ax.set_xscale('log');ax.set_xlim(.01,20000);ax.set_xticks([.01,1,100,10000])
    for data,key,marker,color,open_marker in [(h,'kh800','o','#888',True),(h,'kh3500','o',BLUE,False),(v,'kv3500','s',GREEN,False)]:
        exact=~data[key+'_lower_limit']
        ax.scatter(data.loc[exact,key],data.loc[exact,'log_depth'],s=18,marker=marker,
                   edgecolor=color,facecolor='none' if open_marker else color,lw=.7,zorder=3)
        ax.scatter(data.loc[~exact,key],data.loc[~exact,'log_depth'],s=38,marker='>',c=color,zorder=4)
    header(ax,'Core perm. (mD)',[('kh 800: open circles','#777'),('kh 3500: circles',BLUE),('kv 3500: squares',GREEN)])
    ax=axes[7];ax.set_xlim(0,100);ax.set_xticks([0,50,100])
    ax.scatter(h.Sw,h.log_depth,c=BLUE,s=16);ax.scatter(h.So,h.log_depth,c=RED,s=16)
    header(ax,'Core saturation (%)',[('Sw: water',BLUE),('So: oil',RED)])
    fig.suptitle(title,x=.055,y=.965,ha='left',fontsize=20,weight='bold',color='#183348')
    fig.text(.055,.927,f'{top:,}-{bottom:,} ft | 0.5-ft log sampling | core depths already shifted in source table',fontsize=11,color='#4b6171')
    fig.text(.055,.051,'Core markers are discrete measurements; blank tracks between sampled depths indicate no core data.  > markers: permeability reported as >9999 mD.',fontsize=9)
    fig.text(.055,.031,'Overlay scales run left to right as written above each track. Density/neutron scales are independent; they are not calibrated for a gas-crossover test.',fontsize=9,color='#4b6171')
    return fig

def crossplots():
    fig,axs=plt.subplots(1,3,figsize=(17,7));fig.subplots_adjust(left=.065,right=.98,bottom=.19,top=.78,wspace=.30)
    colors=dict(zip(['A7','A8','A9','A10','E2','E3'],['#2563eb','#16a085','#d69a24','#8e44ad','#da4a4a','#6c7785']))
    for sand,g in horizontal.groupby('sand',sort=False):
        exact=~g.kh3500_lower_limit
        axs[0].scatter(g.loc[exact,'phi3500'],g.loc[exact,'kh3500'],label=sand,c=colors[sand],s=38)
        axs[0].scatter(g.loc[~exact,'phi3500'],g.loc[~exact,'kh3500'],c=colors[sand],marker='^',s=65)
    axs[0].set(xlabel='Core porosity at 3500 psia (%)',ylabel='Horizontal Klinkenberg permeability (mD)',yscale='log',ylim=(.01,20000),title='Porosity and permeability')
    axs[0].legend(title='Sand ID',fontsize=8)
    axs[1].scatter(horizontal.phi800,horizontal.phi3500,c=BLUE,s=32)
    axs[1].plot([0,40],[0,40],c='#888',ls='--',lw=1)
    axs[1].set(xlim=(0,40),ylim=(0,40),xlabel='Core porosity at 800 psia (%)',ylabel='Core porosity at 3500 psia (%)',title='Porosity decreases with confining pressure')
    pairs=horizontal[['sample','kh3500','kh3500_lower_limit']].copy();pairs['id']=pairs['sample'].astype(str)
    vv=vertical[['sample','kv3500','kv3500_lower_limit']].copy();vv['id']=vv['sample'].astype(str).str.removesuffix('V')
    pairs=pairs.merge(vv,on='id');exact=~(pairs.kh3500_lower_limit|pairs.kv3500_lower_limit)
    axs[2].scatter(pairs.loc[exact,'kh3500'],pairs.loc[exact,'kv3500'],c=GREEN,s=32)
    axs[2].plot([.01,20000],[.01,20000],c='#888',ls='--',lw=1)
    axs[2].set(xscale='log',yscale='log',xlim=(.01,20000),ylim=(.01,20000),xlabel='Horizontal permeability at 3500 psia (mD)',ylabel='Vertical permeability at 3500 psia (mD)',title='Directional permeability')
    for ax in axs: ax.grid(alpha=.20);ax.set_axisbelow(True)
    fig.suptitle('What the core measurements show',x=.065,y=.94,ha='left',fontsize=21,weight='bold',color='#183348')
    fig.text(.065,.87,'34 horizontal samples and 23 vertical companion measurements | supplied log depths: 5513.2-6011.1 ft',fontsize=11,color='#4b6171')
    fig.text(.065,.08,'Triangles in the first plot are lower limits (>9999 mD). Censored pairs are omitted from the directional comparison; dashed lines show equality.',fontsize=10)
    return fig


## Main log panel
Change these depth limits to inspect any interval in the supplied log.

In [ ]:
fig = panel(top=5400, bottom=6300)
plt.show()
# fig.savefig('HA2_basic_log_panel.png', dpi=200, bbox_inches='tight')


## Enlarge the sampled intervals

In [ ]:
fig = panel(5500, 5580, 'Upper core interval | A7-A10')
plt.show()
fig = panel(5975, 6020, 'Lower core interval | E2-E3')
plt.show()


## Core relationships

In [ ]:
fig = crossplots()
plt.show()


## First observations
At 3500 psia, porosity ranges from 9.9 to 34.2%. Horizontal permeability ranges from 0.051 mD to values reported as >9999 mD. Grain densities span 2.63-2.68 g/cc.

Median porosity decreases by 1.2 percentage points from 800 to 3500 psia. Among the 32 uncensored horizontal pairs, the median permeability ratio k3500/k800 is approximately 0.817 (about an 18% decrease). Exact permeability statistics must exclude censored values.

The dataset contains both porous/permeable samples and much tighter samples. At supplied log depths 5993.8 and 5995.9 ft, horizontal permeability at 3500 psia decreases from 1194 to 0.097 mD. The E3 interval includes >9999 mD values at 6002.9 and 6003.8 ft, but only 0.051 mD at 6011.1 ft. Such variation supports inspecting beds individually rather than using one average permeability.

Vertical permeability is generally lower than horizontal permeability, with exceptions. Sample 18 has kh=1536 mD and kv=21.2 mD at 3500 psia, despite porosity of 16.8%; inspect sample notes and bed context before fitting one porosity-permeability relation.

The present plots do not establish shale fractions, clay topology, water salinity or in-situ fluid saturation. Those require the subsequent assignment analyses and a log-quality review.